<a href="https://colab.research.google.com/github/a-memoir/Automated-Medical-Report-NNH/blob/main/19sep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# CELL 1 — BASELINE MODEL SETUP
# GPU: A100 recommended
# ============================================================

import os
import sys
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

print("=" * 70)
print("PROJECTNOISOI — BASELINE MODEL V1")
print("=" * 70)

print("PyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))
    print(
        "GPU memory     :",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("WARNING: CUDA is not available.")

# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

from google.colab import drive

drive.mount("/content/drive")

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/NoiSoi_Matching"
)

MANIFEST_FILE = (
    PROJECT_DIR /
    "final_manifest/final_dataset_manifest.csv"
)

CASE_MANIFEST_FILE = (
    PROJECT_DIR /
    "final_manifest/final_case_manifest.csv"
)

SPLIT_FILE = (
    PROJECT_DIR /
    "patient_split/cases_with_split.csv"
)

OUTPUT_DIR = (
    PROJECT_DIR /
    "baseline_model_v1"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nProject directory:", PROJECT_DIR)
print("Manifest exists   :", MANIFEST_FILE.exists())
print("Case manifest     :", CASE_MANIFEST_FILE.exists())
print("Split file        :", SPLIT_FILE.exists())
print("Output directory  :", OUTPUT_DIR)

assert MANIFEST_FILE.exists()
assert CASE_MANIFEST_FILE.exists()

print("\n" + "=" * 70)
print("CELL 1 — PASS")
print("=" * 70)

PROJECTNOISOI — BASELINE MODEL V1
PyTorch version: 2.11.0+cu128
CUDA available : True
GPU            : NVIDIA A100-SXM4-40GB
GPU memory     : 39.49 GB
Mounted at /content/drive

Project directory: /content/drive/MyDrive/NoiSoi_Matching
Manifest exists   : True
Case manifest     : True
Split file        : True
Output directory  : /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v1

CELL 1 — PASS


In [3]:
# ============================================================
# CELL 2 — LOAD BASELINE DATASET
# Target: KẾT LUẬN
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 70)
print("LOADING BASELINE DATASET")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load manifests
# ------------------------------------------------------------

manifest = pd.read_csv(
    MANIFEST_FILE,
    low_memory=False
)

case_manifest = pd.read_csv(
    CASE_MANIFEST_FILE,
    low_memory=False
)

print(f"Image manifest : {len(manifest):,} rows")
print(f"Case manifest  : {len(case_manifest):,} rows")

# ------------------------------------------------------------
# 2. Keep NORMAL images only
# ------------------------------------------------------------

normal_images = manifest[
    manifest["image_status"].eq("NORMAL")
].copy()

print(
    f"NORMAL images  : {len(normal_images):,}"
)

assert len(normal_images) == 76_216

# ------------------------------------------------------------
# 3. Required case-level columns
# ------------------------------------------------------------

required_case_cols = [
    "case_id",
    "patient_group_id",
    "split",
    "ket_luan",
]

missing_cols = [
    c for c in required_case_cols
    if c not in case_manifest.columns
]

print("\nMissing required columns:", missing_cols)

assert not missing_cols

# ------------------------------------------------------------
# 4. Target cleaning
# ------------------------------------------------------------

cases = case_manifest.copy()

cases["ket_luan"] = (
    cases["ket_luan"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Empty target check
empty_target = cases["ket_luan"].eq("")

print(
    "\nEmpty KẾT LUẬN:",
    empty_target.sum()
)

assert empty_target.sum() <= 1

# ------------------------------------------------------------
# 5. Remove only cases with empty target
# ------------------------------------------------------------

cases_model = cases[
    ~empty_target
].copy()

print(
    "Cases available for model:",
    f"{len(cases_model):,}"
)

# ------------------------------------------------------------
# 6. Verify case ↔ image relationship
# ------------------------------------------------------------

case_ids = set(cases_model["case_id"])

normal_images_model = normal_images[
    normal_images["case_id"].isin(case_ids)
].copy()

print(
    "NORMAL images for model:",
    f"{len(normal_images_model):,}"
)

# Every case must have >=1 NORMAL image
image_counts = (
    normal_images_model
    .groupby("case_id")
    .size()
)

missing_image_cases = (
    set(cases_model["case_id"])
    - set(image_counts.index)
)

print(
    "Cases without NORMAL images:",
    len(missing_image_cases)
)

assert len(missing_image_cases) == 0

# ------------------------------------------------------------
# 7. Split distribution
# ------------------------------------------------------------

print("\nCases by split:")
print(
    cases_model["split"]
    .value_counts()
    .sort_index()
)

print("\nImages by split:")
print(
    normal_images_model["split"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 8. Patient-level split check
# ------------------------------------------------------------

patient_split_counts = (
    cases_model
    .groupby("patient_group_id")["split"]
    .nunique()
)

leaking_patients = patient_split_counts[
    patient_split_counts > 1
]

print(
    "\nPatients appearing in >1 split:",
    len(leaking_patients)
)

assert len(leaking_patients) == 0

# ------------------------------------------------------------
# 9. Target statistics
# ------------------------------------------------------------

target_lengths = cases_model["ket_luan"].str.len()

print("\nKẾT LUẬN statistics:")
print(
    target_lengths.describe()
)

print(
    "\nUnique KẾT LUẬN:",
    cases_model["ket_luan"].nunique()
)

# ------------------------------------------------------------
# 10. Top targets
# ------------------------------------------------------------

print("\nTop 20 KẾT LUẬN:")

print(
    cases_model["ket_luan"]
    .value_counts()
    .head(20)
    .to_string()
)

# ------------------------------------------------------------
# 11. Save clean baseline case table
# ------------------------------------------------------------

BASELINE_CASES_FILE = (
    OUTPUT_DIR /
    "baseline_cases.csv"
)

cases_model.to_csv(
    BASELINE_CASES_FILE,
    index=False
)

print(
    "\nSaved:",
    BASELINE_CASES_FILE
)

# ------------------------------------------------------------
# 12. Final checks
# ------------------------------------------------------------

assert cases_model["case_id"].nunique() == len(cases_model)
assert cases_model["patient_group_id"].notna().all()
assert cases_model["split"].isin(
    ["train", "val", "test"]
).all()
assert cases_model["ket_luan"].ne("").all()

print("\n" + "=" * 70)
print("CELL 2 — PASS")
print("=" * 70)

print(f"""
Cases            : {len(cases_model):,}
Patients         : {cases_model["patient_group_id"].nunique():,}
NORMAL images    : {len(normal_images_model):,}
Unique targets   : {cases_model["ket_luan"].nunique():,}
Patient leakage  : {len(leaking_patients)}
""")

LOADING BASELINE DATASET
Image manifest : 76,405 rows
Case manifest  : 7,607 rows
NORMAL images  : 76,216

Missing required columns: []

Empty KẾT LUẬN: 1
Cases available for model: 7,606
NORMAL images for model: 76,209
Cases without NORMAL images: 0

Cases by split:
split
test      712
train    6138
val       756
Name: count, dtype: int64

Images by split:
split
test      7122
train    61478
val       7609
Name: count, dtype: int64

Patients appearing in >1 split: 0

KẾT LUẬN statistics:
count    7606.000000
mean       29.913095
std        19.553287
min         2.000000
25%        14.000000
50%        26.000000
75%        42.000000
max       124.000000
Name: ket_luan, dtype: float64

Unique KẾT LUẬN: 1673

Top 20 KẾT LUẬN:
ket_luan
VIÊM MŨI                                                             801
VIÊM MŨI MẠN                                                         752
VIÊM MŨI XOANG                                                       362
VIÊM HỌNG MẠN - THEO DÕI TRÀO NGƯỢC DỊ

In [4]:
# ============================================================
# CELL 3 — TARGET TOKENIZER ANALYSIS
# ============================================================

from transformers import AutoTokenizer
import pandas as pd
import numpy as np

TOKENIZER_NAME = "google/mt5-small"

print("=" * 70)
print("TARGET TOKENIZER ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load tokenizer
# ------------------------------------------------------------

print(f"\nLoading tokenizer: {TOKENIZER_NAME}")

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_NAME
)

print("Tokenizer loaded.")
print("Vocab size:", tokenizer.vocab_size)

# ------------------------------------------------------------
# 2. Tokenize all targets
# ------------------------------------------------------------

targets = cases_model["ket_luan"].tolist()

token_lengths = []
tokenized_examples = []

for text in targets:
    ids = tokenizer.encode(
        text,
        add_special_tokens=True
    )

    token_lengths.append(len(ids))

    if len(tokenized_examples) < 10:
        tokenized_examples.append(
            (text, ids)
        )

token_lengths = np.array(token_lengths)

# ------------------------------------------------------------
# 3. Statistics
# ------------------------------------------------------------

print("\nToken length statistics:")

print(
    pd.Series(token_lengths).describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

print(
    "\nMaximum token length:",
    token_lengths.max()
)

# ------------------------------------------------------------
# 4. Suggested max lengths
# ------------------------------------------------------------

for max_len in [32, 48, 64, 96, 128, 160]:

    truncated = np.sum(
        token_lengths > max_len
    )

    pct = (
        truncated /
        len(token_lengths) *
        100
    )

    print(
        f"max_len={max_len:3d} | "
        f"truncated={truncated:4d} "
        f"({pct:.3f}%)"
    )

# ------------------------------------------------------------
# 5. Tokenized examples
# ------------------------------------------------------------

print("\nExample tokenizations:")

for text, ids in tokenized_examples[:10]:

    decoded = tokenizer.decode(
        ids,
        skip_special_tokens=False
    )

    print("\nTEXT:")
    print(text)

    print("TOKENS:")
    print(tokenizer.convert_ids_to_tokens(ids))

    print("DECODED:")
    print(decoded)

# ------------------------------------------------------------
# 6. Save statistics
# ------------------------------------------------------------

tokenizer_stats = pd.DataFrame({
    "case_id": cases_model["case_id"].values,
    "token_length": token_lengths
})

TOKEN_STATS_FILE = (
    OUTPUT_DIR /
    "target_token_lengths.csv"
)

tokenizer_stats.to_csv(
    TOKEN_STATS_FILE,
    index=False
)

print(
    "\nSaved:",
    TOKEN_STATS_FILE
)

# ------------------------------------------------------------
# 7. PASS
# ------------------------------------------------------------

assert len(token_lengths) == len(cases_model)
assert token_lengths.min() > 0

print("\n" + "=" * 70)
print("CELL 3 — PASS")
print("=" * 70)

TARGET TOKENIZER ANALYSIS

Loading tokenizer: google/mt5-small
Tokenizer loaded.
Vocab size: 250100

Token length statistics:
count    7606.000000
mean       19.962793
std        11.569386
min         2.000000
50%        18.000000
75%        27.000000
90%        38.000000
95%        44.000000
99%        51.000000
max        75.000000
dtype: float64

Maximum token length: 75
max_len= 32 | truncated=1194 (15.698%)
max_len= 48 | truncated= 117 (1.538%)
max_len= 64 | truncated=   7 (0.092%)
max_len= 96 | truncated=   0 (0.000%)
max_len=128 | truncated=   0 (0.000%)
max_len=160 | truncated=   0 (0.000%)

Example tokenizations:

TEXT:
VIÊM MŨI MẠN - VA
TOKENS:
['▁VI', 'Ê', 'M', '▁M', 'Ũ', 'I', '▁M', 'Ạ', 'N', '▁', '-', '▁VA', '</s>']
DECODED:
VIÊM MŨI MẠN - VA</s>

TEXT:
VIÊM MŨI
TOKENS:
['▁VI', 'Ê', 'M', '▁M', 'Ũ', 'I', '</s>']
DECODED:
VIÊM MŨI</s>

TEXT:
VIÊM MŨI MẠN.
TOKENS:
['▁VI', 'Ê', 'M', '▁M', 'Ũ', 'I', '▁M', 'Ạ', 'N', '.', '</s>']
DECODED:
VIÊM MŨI MẠN.</s>

TEXT:
VIÊM MŨI MẠN
TOKE

In [7]:
# ============================================================
# CELL 4B — MULTI-IMAGE DATASET / DATALOADER
# FIXED: NO HARDCODED SPLIT COUNTS
# ============================================================

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import pandas as pd
import numpy as np

print("=" * 70)
print("MULTI-IMAGE DATASET / DATALOADER SANITY CHECK")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MAX_IMAGES_PER_CASE = 8
IMAGE_SIZE = 224
BATCH_SIZE = 4

print(f"""
MAX_IMAGES_PER_CASE : {MAX_IMAGES_PER_CASE}
IMAGE_SIZE          : {IMAGE_SIZE}
BATCH_SIZE          : {BATCH_SIZE}
""")

# ------------------------------------------------------------
# 1. Prepare image table
# ------------------------------------------------------------

image_df = normal_images_model.copy()

image_df["image_path"] = image_df["image_path"].astype(str)

if "image_order" in image_df.columns:
    image_df = image_df.sort_values(
        ["case_id", "image_order"]
    )
else:
    image_df = image_df.sort_values(
        ["case_id", "image_path"]
    )

# ------------------------------------------------------------
# 2. Build case -> image list
# ------------------------------------------------------------

case_images = (
    image_df
    .groupby("case_id")["image_path"]
    .apply(list)
    .to_dict()
)

# ------------------------------------------------------------
# 3. Build case-level records
# ------------------------------------------------------------

dataset_records = []

for _, row in cases_model.iterrows():

    case_id = row["case_id"]

    paths = case_images.get(case_id, [])

    assert len(paths) > 0

    dataset_records.append({
        "case_id": case_id,
        "patient_group_id": row["patient_group_id"],
        "split": row["split"],
        "ket_luan": row["ket_luan"],
        "image_paths": paths,
    })

print(
    "Case-level records:",
    f"{len(dataset_records):,}"
)

assert len(dataset_records) == len(cases_model)

# ------------------------------------------------------------
# 4. Image transform
# ------------------------------------------------------------

image_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ------------------------------------------------------------
# 5. Dataset
# ------------------------------------------------------------

class MultiImageEndoscopyDataset(Dataset):

    def __init__(
        self,
        records,
        max_images=8,
        transform=None,
        training=False,
    ):

        self.records = records
        self.max_images = max_images
        self.transform = transform
        self.training = training

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):

        record = self.records[idx]

        paths = record["image_paths"]

        # ----------------------------------------------------
        # Image selection
        # ----------------------------------------------------

        if len(paths) > self.max_images:

            if self.training:

                selected = np.random.choice(
                    len(paths),
                    size=self.max_images,
                    replace=False
                )

                selected = sorted(
                    selected.tolist()
                )

                paths_selected = [
                    paths[i]
                    for i in selected
                ]

            else:

                paths_selected = paths[
                    :self.max_images
                ]

        else:

            paths_selected = paths

        # ----------------------------------------------------
        # Load images
        # ----------------------------------------------------

        images = []

        for path in paths_selected:

            with Image.open(path) as img:

                img = img.convert("RGB")

                if self.transform is not None:
                    img = self.transform(img)

            images.append(img)

        images = torch.stack(images)

        return {
            "case_id": record["case_id"],
            "patient_group_id": record["patient_group_id"],
            "images": images,
            "num_images": len(images),
            "target_text": record["ket_luan"],
        }


# ------------------------------------------------------------
# 6. Split records
# ------------------------------------------------------------

train_records = [
    r for r in dataset_records
    if r["split"] == "train"
]

val_records = [
    r for r in dataset_records
    if r["split"] == "val"
]

test_records = [
    r for r in dataset_records
    if r["split"] == "test"
]

print("\nRecords by split:")
print("Train:", len(train_records))
print("Val  :", len(val_records))
print("Test :", len(test_records))

# ------------------------------------------------------------
# 7. Create datasets
# ------------------------------------------------------------

train_dataset = MultiImageEndoscopyDataset(
    train_records,
    max_images=MAX_IMAGES_PER_CASE,
    transform=image_transform,
    training=True,
)

val_dataset = MultiImageEndoscopyDataset(
    val_records,
    max_images=MAX_IMAGES_PER_CASE,
    transform=image_transform,
    training=False,
)

test_dataset = MultiImageEndoscopyDataset(
    test_records,
    max_images=MAX_IMAGES_PER_CASE,
    transform=image_transform,
    training=False,
)

# ------------------------------------------------------------
# 8. Custom collate
# ------------------------------------------------------------

def multimodal_collate(batch):

    return {
        "case_id": [
            x["case_id"]
            for x in batch
        ],

        "patient_group_id": [
            x["patient_group_id"]
            for x in batch
        ],

        "images": [
            x["images"]
            for x in batch
        ],

        "num_images": [
            x["num_images"]
            for x in batch
        ],

        "target_text": [
            x["target_text"]
            for x in batch
        ],
    }


# ------------------------------------------------------------
# 9. DataLoaders
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=multimodal_collate,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=multimodal_collate,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=multimodal_collate,
)

# ------------------------------------------------------------
# 10. Inspect one sample
# ------------------------------------------------------------

sample = train_dataset[0]

print("\nSingle sample:")

print("case_id      :", sample["case_id"])
print("patient_group:", sample["patient_group_id"])
print("images shape :", tuple(sample["images"].shape))
print("num_images   :", sample["num_images"])
print("target       :", sample["target_text"])

# ------------------------------------------------------------
# 11. Test one batch
# ------------------------------------------------------------

batch = next(iter(train_loader))

print("\nBatch:")
print("Batch size:", len(batch["case_id"]))

print(
    "Image tensor shapes:",
    [
        tuple(x.shape)
        for x in batch["images"]
    ]
)

print(
    "Num images:",
    batch["num_images"]
)

print("\nTargets:")

for i, text in enumerate(batch["target_text"]):
    print(f"{i+1}. {text}")

# ------------------------------------------------------------
# 12. GPU test
# ------------------------------------------------------------

gpu_images = [
    x.to(
        "cuda",
        non_blocking=True
    )
    for x in batch["images"]
]

print("\nGPU test:")
print(
    "First tensor device:",
    gpu_images[0].device
)

print(
    "First tensor shape :",
    tuple(gpu_images[0].shape)
)

del gpu_images
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 13. Dynamic assertions
# ------------------------------------------------------------

expected_train = (
    cases_model["split"]
    .eq("train")
    .sum()
)

expected_val = (
    cases_model["split"]
    .eq("val")
    .sum()
)

expected_test = (
    cases_model["split"]
    .eq("test")
    .sum()
)

assert len(train_dataset) == expected_train
assert len(val_dataset) == expected_val
assert len(test_dataset) == expected_test

assert (
    len(train_dataset)
    + len(val_dataset)
    + len(test_dataset)
    == len(cases_model)
)

assert sample["images"].ndim == 4

assert sample["images"].shape[1:] == (
    3,
    IMAGE_SIZE,
    IMAGE_SIZE
)

assert len(batch["images"]) == BATCH_SIZE

for tensor in batch["images"]:

    assert tensor.ndim == 4

    assert tensor.shape[1:] == (
        3,
        IMAGE_SIZE,
        IMAGE_SIZE
    )

print("\n" + "=" * 70)
print("CELL 4B — PASS")
print("=" * 70)

print(f"""
Cases total       : {len(cases_model):,}
Train cases       : {len(train_dataset):,}
Val cases         : {len(val_dataset):,}
Test cases        : {len(test_dataset):,}
NORMAL images     : {len(normal_images_model):,}
Max images/case   : {MAX_IMAGES_PER_CASE}
Image size        : {IMAGE_SIZE} × {IMAGE_SIZE}
Patient leakage   : 0
""")

MULTI-IMAGE DATASET / DATALOADER SANITY CHECK

MAX_IMAGES_PER_CASE : 8
IMAGE_SIZE          : 224
BATCH_SIZE          : 4

Case-level records: 7,606

Records by split:
Train: 6138
Val  : 756
Test : 712

Single sample:
case_id      : 10000.10000.0.10014
patient_group: NGUYEN TRUONG TAN SANG_2015
images shape : (8, 3, 224, 224)
num_images   : 8
target       : VIÊM MŨI MẠN - VA

Batch:
Batch size: 4
Image tensor shapes: [(8, 3, 224, 224), (7, 3, 224, 224), (8, 3, 224, 224), (8, 3, 224, 224)]
Num images: [8, 7, 8, 8]

Targets:
1. VIÊM MŨI + VA A-MI-ĐAN QUÁ PHÁT
2. VIÊM MŨI MẠN
3. VIÊM MŨI MẠN
4. VIÊM MŨI/ VIÊM XOANG P + VA

GPU test:
First tensor device: cuda:0
First tensor shape : (8, 3, 224, 224)

CELL 4B — PASS

Cases total       : 7,606
Train cases       : 6,138
Val cases         : 756
Test cases        : 712
NORMAL images     : 76,209
Max images/case   : 8
Image size        : 224 × 224
Patient leakage   : 0



In [8]:
# ============================================================
# CELL 5 — LOAD VISION + TEXT BACKBONE
# ============================================================
# GPU: A100
# No training yet
# ============================================================

import torch
import torch.nn as nn

from transformers import (
    ViTModel,
    MT5ForConditionalGeneration,
    AutoTokenizer,
)

print("=" * 70)
print("CELL 5 — VISION + TEXT BACKBONE CHECK")
print("=" * 70)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

VISION_MODEL_NAME = "google/vit-base-patch16-224"
TEXT_MODEL_NAME = "google/mt5-small"

MAX_TARGET_LENGTH = 96

print("\nVision model:")
print(VISION_MODEL_NAME)

print("\nText model:")
print(TEXT_MODEL_NAME)

print("\nDevice:")
print(DEVICE)

# ------------------------------------------------------------
# 1. Load tokenizer
# ------------------------------------------------------------

print("\nLoading mT5 tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL_NAME
)

print(
    "Tokenizer vocab size:",
    tokenizer.vocab_size
)

# ------------------------------------------------------------
# 2. Load ViT
# ------------------------------------------------------------

print("\nLoading ViT...")

vision_encoder = ViTModel.from_pretrained(
    VISION_MODEL_NAME
)

print(
    "ViT hidden size:",
    vision_encoder.config.hidden_size
)

print(
    "ViT image size:",
    vision_encoder.config.image_size
)

# ------------------------------------------------------------
# 3. Load mT5
# ------------------------------------------------------------

print("\nLoading mT5-small...")

text_model = MT5ForConditionalGeneration.from_pretrained(
    TEXT_MODEL_NAME
)

print(
    "mT5 d_model:",
    text_model.config.d_model
)

print(
    "mT5 vocab size:",
    text_model.config.vocab_size
)

# ------------------------------------------------------------
# 4. Move to GPU
# ------------------------------------------------------------

vision_encoder = vision_encoder.to(DEVICE)
text_model = text_model.to(DEVICE)

# ------------------------------------------------------------
# 5. Freeze ViT for baseline
# ------------------------------------------------------------

for param in vision_encoder.parameters():
    param.requires_grad = False

vision_encoder.eval()

trainable_vision = sum(
    p.numel()
    for p in vision_encoder.parameters()
    if p.requires_grad
)

total_vision = sum(
    p.numel()
    for p in vision_encoder.parameters()
)

trainable_text = sum(
    p.numel()
    for p in text_model.parameters()
    if p.requires_grad
)

total_text = sum(
    p.numel()
    for p in text_model.parameters()
)

print("\nParameter summary:")

print(
    f"ViT total       : {total_vision:,}"
)

print(
    f"ViT trainable   : {trainable_vision:,}"
)

print(
    f"mT5 total       : {total_text:,}"
)

print(
    f"mT5 trainable   : {trainable_text:,}"
)

# ------------------------------------------------------------
# 6. Projection layer
# ------------------------------------------------------------

vision_dim = vision_encoder.config.hidden_size
text_dim = text_model.config.d_model

visual_projection = nn.Linear(
    vision_dim,
    text_dim
).to(DEVICE)

print("\nVisual projection:")
print(
    f"{vision_dim} → {text_dim}"
)

# ------------------------------------------------------------
# 7. Test Vision Encoder
# ------------------------------------------------------------

print("\nTesting ViT forward pass...")

sample_images = batch["images"][0].to(
    DEVICE
)

print(
    "Input shape:",
    tuple(sample_images.shape)
)

with torch.no_grad():

    vision_output = vision_encoder(
        pixel_values=sample_images
    )

image_embeddings = vision_output.last_hidden_state

print(
    "ViT output:",
    tuple(image_embeddings.shape)
)

# ViT output:
# [num_images, sequence_length, hidden_size]

# ------------------------------------------------------------
# 8. Extract CLS token
# ------------------------------------------------------------

image_cls = image_embeddings[:, 0, :]

print(
    "CLS embeddings:",
    tuple(image_cls.shape)
)

# Expected:
# [num_images, 768]

# ------------------------------------------------------------
# 9. Aggregate multiple images
# ------------------------------------------------------------

case_visual_embedding = image_cls.mean(
    dim=0,
    keepdim=True
)

print(
    "Aggregated visual embedding:",
    tuple(case_visual_embedding.shape)
)

# Expected:
# [1, 768]

# ------------------------------------------------------------
# 10. Project into mT5 space
# ------------------------------------------------------------

projected_visual = visual_projection(
    case_visual_embedding
)

print(
    "Projected visual embedding:",
    tuple(projected_visual.shape)
)

# Expected:
# [1, 512]

# ------------------------------------------------------------
# 11. Tokenizer test
# ------------------------------------------------------------

target_text = batch["target_text"][0]

tokenized = tokenizer(
    target_text,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_TARGET_LENGTH,
)

input_ids = tokenized["input_ids"].to(
    DEVICE
)

attention_mask = tokenized["attention_mask"].to(
    DEVICE
)

print("\nTarget:")
print(target_text)

print(
    "Tokenized shape:",
    tuple(input_ids.shape)
)

print(
    "Token count:",
    input_ids.shape[1]
)

# ------------------------------------------------------------
# 12. mT5 embedding dimension check
# ------------------------------------------------------------

text_embedding_layer = (
    text_model.get_input_embeddings()
)

token_embeddings = text_embedding_layer(
    input_ids
)

print(
    "mT5 token embeddings:",
    tuple(token_embeddings.shape)
)

# Expected:
# [batch, seq_len, 512]

# ------------------------------------------------------------
# 13. Dimension compatibility
# ------------------------------------------------------------

assert vision_dim == 768
assert text_dim == 512

assert projected_visual.shape[-1] == text_dim
assert token_embeddings.shape[-1] == text_dim

# ------------------------------------------------------------
# 14. GPU memory
# ------------------------------------------------------------

allocated_gb = (
    torch.cuda.memory_allocated()
    / 1024**3
)

reserved_gb = (
    torch.cuda.memory_reserved()
    / 1024**3
)

print("\nGPU memory:")
print(
    f"Allocated: {allocated_gb:.2f} GB"
)

print(
    f"Reserved : {reserved_gb:.2f} GB"
)

# ------------------------------------------------------------
# 15. Save architecture configuration
# ------------------------------------------------------------

architecture_config = {
    "vision_model": VISION_MODEL_NAME,
    "text_model": TEXT_MODEL_NAME,
    "vision_hidden_size": vision_dim,
    "text_hidden_size": text_dim,
    "max_images_per_case": MAX_IMAGES_PER_CASE,
    "image_size": IMAGE_SIZE,
    "max_target_length": MAX_TARGET_LENGTH,
    "vision_frozen": True,
    "aggregation": "mean_cls",
}

ARCH_CONFIG_FILE = (
    OUTPUT_DIR /
    "architecture_config.json"
)

import json

with open(
    ARCH_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        architecture_config,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "\nSaved:",
    ARCH_CONFIG_FILE
)

# ------------------------------------------------------------
# Cleanup temporary tensors
# ------------------------------------------------------------

del (
    sample_images,
    vision_output,
    image_embeddings,
    image_cls,
    case_visual_embedding,
    projected_visual,
    input_ids,
    attention_mask,
    token_embeddings,
)

torch.cuda.empty_cache()

# ------------------------------------------------------------
# FINAL PASS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 5 — PASS")
print("=" * 70)

print(f"""
Vision encoder     : {VISION_MODEL_NAME}
Vision dimension   : {vision_dim}
Text model         : {TEXT_MODEL_NAME}
Text dimension     : {text_dim}
Projection         : {vision_dim} → {text_dim}
Vision frozen      : True
Max images/case    : {MAX_IMAGES_PER_CASE}
Max target length  : {MAX_TARGET_LENGTH}
Device             : {DEVICE}
""")

CELL 5 — VISION + TEXT BACKBONE CHECK

Vision model:
google/vit-base-patch16-224

Text model:
google/mt5-small

Device:
cuda

Loading mT5 tokenizer...
Tokenizer vocab size: 250100

Loading ViT...


config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

mT5 d_model: 512
mT5 vocab size: 250112


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.20GB            

model.safetensors: downloading bytes:           |  0.00B            


Parameter summary:
ViT total       : 86,389,248
ViT trainable   : 0
mT5 total       : 300,176,768
mT5 trainable   : 300,176,768

Visual projection:
768 → 512

Testing ViT forward pass...
Input shape: (8, 3, 224, 224)
ViT output: (8, 197, 768)
CLS embeddings: (8, 768)
Aggregated visual embedding: (1, 768)
Projected visual embedding: (1, 512)

Target:
VIÊM MŨI + VA A-MI-ĐAN QUÁ PHÁT
Tokenized shape: (1, 19)
Token count: 19
mT5 token embeddings: (1, 19, 512)

GPU memory:
Allocated: 1.46 GB
Reserved : 1.55 GB

Saved: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v1/architecture_config.json

CELL 5 — PASS

Vision encoder     : google/vit-base-patch16-224
Vision dimension   : 768
Text model         : google/mt5-small
Text dimension     : 512
Projection         : 768 → 512
Vision frozen      : True
Max images/case    : 8
Max target length  : 96
Device             : cuda



In [11]:
# ============================================================
# CELL 5B — RELOAD VISION + TEXT BACKBONES
# ============================================================

import torch
import torch.nn as nn

from transformers import (
    ViTModel,
    MT5ForConditionalGeneration,
    AutoTokenizer
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VISION_MODEL_NAME = "google/vit-base-patch16-224"
TEXT_MODEL_NAME = "google/mt5-small"

print("=" * 70)
print("CELL 5B — RELOAD BACKBONES")
print("=" * 70)

# ------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

# ------------------------------------------------------------
# ViT
# ------------------------------------------------------------

print("\nLoading ViT...")

vit_model = ViTModel.from_pretrained(
    VISION_MODEL_NAME
).to(DEVICE)

vit_model.eval()

# Freeze ViT
for param in vit_model.parameters():
    param.requires_grad = False

# ------------------------------------------------------------
# mT5
# ------------------------------------------------------------

print("Loading mT5-small...")

mt5_model = MT5ForConditionalGeneration.from_pretrained(
    TEXT_MODEL_NAME
).to(DEVICE)

# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

print("\nParameter summary:")
print(f"ViT hidden size : {vit_model.config.hidden_size}")
print(f"mT5 d_model     : {mt5_model.config.d_model}")
print(f"mT5 vocab size  : {mt5_model.config.vocab_size}")

print("\nViT trainable parameters:",
      sum(p.numel() for p in vit_model.parameters() if p.requires_grad))

print("mT5 trainable parameters:",
      sum(p.numel() for p in mt5_model.parameters() if p.requires_grad))

assert vit_model.config.hidden_size == 768
assert mt5_model.config.d_model == 512

print("\n" + "=" * 70)
print("CELL 5B — PASS")
print("=" * 70)

CELL 5B — RELOAD BACKBONES

Loading ViT...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading mT5-small...


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



Parameter summary:
ViT hidden size : 768
mT5 d_model     : 512
mT5 vocab size  : 250112

ViT trainable parameters: 0
mT5 trainable parameters: 300176768

CELL 5B — PASS


In [13]:
# ============================================================
# CELL 6B — MULTI-IMAGE → mT5 FORWARD / LOSS TEST
# ============================================================

import torch
import torch.nn as nn
from transformers.modeling_outputs import BaseModelOutput

print("=" * 70)
print("CELL 6B — MULTI-IMAGE → mT5 FORWARD / LOSS TEST")
print("=" * 70)


# ------------------------------------------------------------
# 1. Visual projection
# ------------------------------------------------------------

class VisualPrefixProjector(nn.Module):
    def __init__(self, vision_dim=768, text_dim=512):
        super().__init__()
        self.proj = nn.Linear(vision_dim, text_dim)

    def forward(self, x):
        return self.proj(x)


visual_projector = VisualPrefixProjector(
    vision_dim=vit_model.config.hidden_size,
    text_dim=mt5_model.config.d_model
).to(DEVICE)

print("Visual projection: 768 → 512")


# ------------------------------------------------------------
# 2. Encode images
# ------------------------------------------------------------

def encode_images(images_list):

    visual_embeddings = []
    attention_masks = []

    for images in images_list:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        # ViT frozen
        with torch.no_grad():

            vit_output = vit_model(
                pixel_values=images
            )

            # CLS token
            cls_embeddings = (
                vit_output.last_hidden_state[:, 0, :]
            )
            # [N, 768]

        # [N, 768] → [N, 512]
        projected = visual_projector(
            cls_embeddings
        )

        visual_embeddings.append(projected)

        attention_masks.append(
            torch.ones(
                projected.shape[0],
                dtype=torch.long,
                device=DEVICE
            )
        )

    # Variable number of images → padding
    visual_prefix = torch.nn.utils.rnn.pad_sequence(
        visual_embeddings,
        batch_first=True
    )

    visual_attention_mask = torch.nn.utils.rnn.pad_sequence(
        attention_masks,
        batch_first=True,
        padding_value=0
    )

    return visual_prefix, visual_attention_mask


# ------------------------------------------------------------
# 3. Get one real batch
# ------------------------------------------------------------

batch = next(iter(train_loader))

print("\nBatch keys:")
print(batch.keys())

images_list = batch["images"]

print("\nInput batch:")
print("Batch size:", len(images_list))
print(
    "Images/case:",
    [x.shape[0] for x in images_list]
)


# ------------------------------------------------------------
# 4. Recover targets
# ------------------------------------------------------------

# Cell 4B's loader apparently does not return "targets".
# We therefore recover targets from dataset_records.

case_ids = batch["case_id"]

print("Case IDs:", case_ids)

# Build lookup once
case_target_lookup = {
    str(row["case_id"]): str(row["ket_luan"])
    for _, row in cases_model.iterrows()
}

targets = [
    case_target_lookup[str(case_id)]
    for case_id in case_ids
]

print("Targets:", targets)


# ------------------------------------------------------------
# 5. Visual encoding
# ------------------------------------------------------------

visual_prefix, visual_attention_mask = encode_images(
    images_list
)

print("\nVisual prefix:")
print("Shape:", tuple(visual_prefix.shape))

print(
    "Attention mask:",
    tuple(visual_attention_mask.shape)
)


# ------------------------------------------------------------
# 6. Tokenize targets
# ------------------------------------------------------------

tokenized = tokenizer(
    targets,
    padding=True,
    truncation=True,
    max_length=MAX_TARGET_LENGTH,
    return_tensors="pt"
)

labels = tokenized.input_ids.to(DEVICE)

# Ignore padding in loss
labels[
    labels == tokenizer.pad_token_id
] = -100

print("\nLabels:")
print("Shape:", tuple(labels.shape))


# ------------------------------------------------------------
# 7. mT5 encoder
# ------------------------------------------------------------

encoder_outputs = mt5_model.encoder(
    inputs_embeds=visual_prefix,
    attention_mask=visual_attention_mask,
    return_dict=True
)

print("\nmT5 encoder output:")
print(
    tuple(
        encoder_outputs.last_hidden_state.shape
    )
)


# ------------------------------------------------------------
# 8. Decoder + loss
# ------------------------------------------------------------

outputs = mt5_model(
    encoder_outputs=BaseModelOutput(
        last_hidden_state=
        encoder_outputs.last_hidden_state
    ),
    attention_mask=visual_attention_mask,
    labels=labels,
    return_dict=True
)

loss = outputs.loss

print("\nForward result:")
print("Loss:", float(loss.detach().cpu()))
print(
    "Logits:",
    tuple(outputs.logits.shape)
)


# ------------------------------------------------------------
# 9. Sanity checks
# ------------------------------------------------------------

assert visual_prefix.ndim == 3

assert (
    visual_prefix.shape[-1]
    == mt5_model.config.d_model
)

assert (
    encoder_outputs.last_hidden_state.shape
    == visual_prefix.shape
)

assert outputs.logits.shape[0] == len(targets)

assert (
    outputs.logits.shape[-1]
    == mt5_model.config.vocab_size
)

assert torch.isfinite(loss)

print("\n" + "=" * 70)
print("CELL 6B — PASS")
print("=" * 70)

print(
    "Visual prefix :",
    tuple(visual_prefix.shape)
)

print(
    "Encoder output:",
    tuple(
        encoder_outputs.last_hidden_state.shape
    )
)

print(
    "Labels        :",
    tuple(labels.shape)
)

print(
    "Logits        :",
    tuple(outputs.logits.shape)
)

print(
    "Loss          :",
    f"{loss.item():.4f}"
)

print("=" * 70)

CELL 6B — MULTI-IMAGE → mT5 FORWARD / LOSS TEST
Visual projection: 768 → 512

Batch keys:
dict_keys(['case_id', 'patient_group_id', 'images', 'num_images', 'target_text'])

Input batch:
Batch size: 4
Images/case: [6, 8, 8, 8]
Case IDs: ['13444.13444.0.13463', '14727.14727.0.14746', '18122.18122.0.18147', '11420.11420.0.11436']
Targets: ['VIÊM MŨI', 'VIÊM MŨI XOANG MẠN (ĐÃ PT)', 'HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI NGOÀI, TAI GIỮA TRÊN NỘI SOI', 'VIÊM MŨI + VIÊM VA']

Visual prefix:
Shape: (4, 8, 512)
Attention mask: (4, 8)

Labels:
Shape: (4, 39)

mT5 encoder output:
(4, 8, 512)

Forward result:
Loss: 14.95007038116455
Logits: (4, 39, 250112)

CELL 6B — PASS
Visual prefix : (4, 8, 512)
Encoder output: (4, 8, 512)
Labels        : (4, 39)
Logits        : (4, 39, 250112)
Loss          : 14.9501


In [15]:
# ============================================================
# CELL 7B — GENERATION SANITY TEST
# ============================================================

import torch
from transformers.modeling_outputs import BaseModelOutput

print("=" * 70)
print("CELL 7B — GENERATION SANITY TEST")
print("=" * 70)


# ------------------------------------------------------------
# 1. Get ONE batch and use only TWO cases
# ------------------------------------------------------------

batch = next(iter(train_loader))

images_list = batch["images"][:2]
case_ids = batch["case_id"][:2]
ground_truth = batch["target_text"][:2]

print("\nCases selected:")

for i in range(2):
    print(f"\nCase {i + 1}")
    print("-" * 50)
    print("Case ID:", case_ids[i])
    print("Images:", images_list[i].shape[0])
    print("Ground truth:", ground_truth[i])


# ------------------------------------------------------------
# 2. Encode images
# ------------------------------------------------------------

visual_embeddings = []
attention_masks = []

for images in images_list:

    images = images.to(
        DEVICE,
        non_blocking=True
    )

    # Frozen ViT
    with torch.no_grad():

        vit_output = vit_model(
            pixel_values=images
        )

        # CLS token
        cls_embeddings = (
            vit_output.last_hidden_state[:, 0, :]
        )

    # 768 → 512
    projected = visual_projector(
        cls_embeddings
    )

    visual_embeddings.append(projected)

    attention_masks.append(
        torch.ones(
            projected.shape[0],
            dtype=torch.long,
            device=DEVICE
        )
    )


# ------------------------------------------------------------
# 3. Pad variable number of images
# ------------------------------------------------------------

visual_prefix = torch.nn.utils.rnn.pad_sequence(
    visual_embeddings,
    batch_first=True
)

visual_attention_mask = torch.nn.utils.rnn.pad_sequence(
    attention_masks,
    batch_first=True,
    padding_value=0
)

print("\nVisual prefix:")
print("  Shape:", tuple(visual_prefix.shape))
print(
    "  Attention mask:",
    tuple(visual_attention_mask.shape)
)


# ------------------------------------------------------------
# 4. mT5 encoder
# ------------------------------------------------------------

with torch.no_grad():

    encoder_outputs = mt5_model.encoder(
        inputs_embeds=visual_prefix,
        attention_mask=visual_attention_mask,
        return_dict=True
    )

print("\nmT5 encoder:")
print(
    "  Output:",
    tuple(
        encoder_outputs.last_hidden_state.shape
    )
)


# ------------------------------------------------------------
# 5. Generation
# ------------------------------------------------------------

print("\nGenerating...")

with torch.no_grad():

    generated_ids = mt5_model.generate(
        encoder_outputs=BaseModelOutput(
            last_hidden_state=
            encoder_outputs.last_hidden_state
        ),
        attention_mask=visual_attention_mask,
        max_new_tokens=MAX_TARGET_LENGTH,
        num_beams=2,
        do_sample=False
    )


# ------------------------------------------------------------
# 6. Decode
# ------------------------------------------------------------

generated_texts = tokenizer.batch_decode(
    generated_ids,
    skip_special_tokens=True
)


# ------------------------------------------------------------
# 7. Results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GENERATION RESULTS")
print("=" * 70)

for i in range(2):

    print(f"\nCase {i + 1}")
    print("-" * 50)

    print("Case ID:")
    print(case_ids[i])

    print("\nGround truth:")
    print(ground_truth[i])

    print("\nGenerated:")
    print(generated_texts[i])


# ------------------------------------------------------------
# 8. Sanity checks
# ------------------------------------------------------------

assert generated_ids.ndim == 2
assert generated_ids.shape[0] == 2
assert len(generated_texts) == 2

print("\n" + "=" * 70)
print("CELL 7B — PASS")
print("=" * 70)

print("Generation works successfully.")
print("Cases tested:", 2)
print("=" * 70)

CELL 7B — GENERATION SANITY TEST

Cases selected:

Case 1
--------------------------------------------------
Case ID: 10036.10036.0.10050
Images: 8
Ground truth: VIÊM MŨI + VA A-MI-ĐAN QUÁ PHÁT

Case 2
--------------------------------------------------
Case ID: 10389.10389.0.10403
Images: 8
Ground truth: VIÊM MŨI + VA

Visual prefix:
  Shape: (2, 8, 512)
  Attention mask: (2, 8)

mT5 encoder:
  Output: (2, 8, 512)

Generating...

GENERATION RESULTS

Case 1
--------------------------------------------------
Case ID:
10036.10036.0.10050

Ground truth:
VIÊM MŨI + VA A-MI-ĐAN QUÁ PHÁT

Generated:
<extra_id_0> eddy  <extra_id_1>e eddy eddy eddy eddy eddy eddy- <extra_id_18>  <extra_id_19>  <extra_id_20>  <extra_id_21>  <extra_id_22>  <extra_id_23>  <extra_id_24>  <extra_id_25>  <extra_id_26>ẽ <extra_id_27> <extra_id_28>  <extra_id_29>ợ <extra_id_30>ợ <extra_id_31>ợ <extra_id_32>  <extra_id_33> <extra_id_34> <extra_id_55> <extra_id_34>i recycle <extra_id_56>e recycle recycle <extra_id_56>ă e

In [16]:
# ============================================================
# CELL 8A — SINGLE TRAINING STEP + GPU MEMORY CHECK
# ============================================================

import os
import gc
import torch
import torch.nn as nn
from torch.optim import AdamW

print("=" * 70)
print("CELL 8A — SINGLE TRAINING STEP + GPU MEMORY CHECK")
print("=" * 70)


# ------------------------------------------------------------
# 1. BF16 check
# ------------------------------------------------------------

assert torch.cuda.is_available(), "CUDA is not available."

bf16_supported = torch.cuda.is_bf16_supported()

print("\nGPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", bf16_supported)

assert bf16_supported, "A100 should support BF16."


# ------------------------------------------------------------
# 2. Make sure trainable components are in train mode
# ------------------------------------------------------------

mt5_model.train()
visual_projector.train()

# ViT stays frozen + evaluation mode
vit_model.eval()

print("\nModes:")
print("  ViT:", "eval / frozen")
print("  Projector:", "trainable")
print("  mT5:", "trainable")


# ------------------------------------------------------------
# 3. Optimizer
# ------------------------------------------------------------

optimizer = AdamW(
    [
        {
            "params": visual_projector.parameters(),
            "lr": 1e-4,
        },
        {
            "params": mt5_model.parameters(),
            "lr": 5e-5,
        },
    ],
    weight_decay=0.01
)

print("\nOptimizer:")
print("  Projector LR: 1e-4")
print("  mT5 LR      : 5e-5")
print("  Weight decay: 0.01")


# ------------------------------------------------------------
# 4. Get one real batch
# ------------------------------------------------------------

batch = next(iter(train_loader))

images_list = batch["images"]
targets = batch["target_text"]

print("\nBatch:")
print("  Cases:", len(images_list))
print(
    "  Images/case:",
    [x.shape[0] for x in images_list]
)
print("  Targets:", targets)


# ------------------------------------------------------------
# 5. Forward pass
# ------------------------------------------------------------

optimizer.zero_grad(set_to_none=True)

visual_embeddings = []
attention_masks = []

# Frozen ViT
with torch.no_grad():

    for images in images_list:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        vit_output = vit_model(
            pixel_values=images
        )

        cls_embeddings = (
            vit_output.last_hidden_state[:, 0, :]
        )

        # Keep ViT output detached
        visual_embeddings.append(
            cls_embeddings
        )

        attention_masks.append(
            torch.ones(
                cls_embeddings.shape[0],
                dtype=torch.long,
                device=DEVICE
            )
        )


# ------------------------------------------------------------
# 6. Project visual embeddings
# ------------------------------------------------------------

projected_embeddings = [
    visual_projector(x)
    for x in visual_embeddings
]

visual_prefix = torch.nn.utils.rnn.pad_sequence(
    projected_embeddings,
    batch_first=True
)

visual_attention_mask = torch.nn.utils.rnn.pad_sequence(
    attention_masks,
    batch_first=True,
    padding_value=0
)

print("\nVisual prefix:")
print("  Shape:", tuple(visual_prefix.shape))


# ------------------------------------------------------------
# 7. Tokenize targets
# ------------------------------------------------------------

tokenized = tokenizer(
    targets,
    padding=True,
    truncation=True,
    max_length=MAX_TARGET_LENGTH,
    return_tensors="pt"
)

labels = tokenized.input_ids.to(DEVICE)

labels[
    labels == tokenizer.pad_token_id
] = -100

print("Labels:")
print("  Shape:", tuple(labels.shape))


# ------------------------------------------------------------
# 8. mT5 forward with BF16
# ------------------------------------------------------------

with torch.autocast(
    device_type="cuda",
    dtype=torch.bfloat16
):

    encoder_outputs = mt5_model.encoder(
        inputs_embeds=visual_prefix,
        attention_mask=visual_attention_mask,
        return_dict=True
    )

    outputs = mt5_model(
        encoder_outputs=encoder_outputs,
        attention_mask=visual_attention_mask,
        labels=labels,
        return_dict=True
    )

    loss = outputs.loss


print("\nForward:")
print("  Loss:", loss.item())


# ------------------------------------------------------------
# 9. Backward
# ------------------------------------------------------------

loss.backward()

print("\nBackward:")
print("  Completed successfully.")


# ------------------------------------------------------------
# 10. Gradient sanity check
# ------------------------------------------------------------

projector_grad = 0.0
mt5_grad = 0.0

for p in visual_projector.parameters():
    if p.grad is not None:
        projector_grad += p.grad.detach().float().norm().item()

for p in mt5_model.parameters():
    if p.grad is not None:
        mt5_grad += p.grad.detach().float().norm().item()

print("\nGradient check:")
print("  Projector grad norm:", projector_grad)
print("  mT5 grad norm      :", mt5_grad)


assert torch.isfinite(loss)
assert projector_grad > 0
assert mt5_grad > 0


# ------------------------------------------------------------
# 11. GPU memory
# ------------------------------------------------------------

torch.cuda.synchronize()

allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3
peak = torch.cuda.max_memory_allocated() / 1024**3

print("\nGPU memory:")
print(f"  Allocated: {allocated:.2f} GB")
print(f"  Reserved : {reserved:.2f} GB")
print(f"  Peak     : {peak:.2f} GB")


# ------------------------------------------------------------
# 12. Cleanup
# ------------------------------------------------------------

optimizer.zero_grad(set_to_none=True)

gc.collect()
torch.cuda.empty_cache()

print("\n" + "=" * 70)
print("CELL 8A — PASS")
print("=" * 70)

print("Single training step works.")
print("BF16 works.")
print("Backward works.")
print("Gradients are present.")
print("Ready for full training loop.")
print("=" * 70)

CELL 8A — SINGLE TRAINING STEP + GPU MEMORY CHECK

GPU: NVIDIA A100-SXM4-40GB
BF16 supported: True

Modes:
  ViT: eval / frozen
  Projector: trainable
  mT5: trainable

Optimizer:
  Projector LR: 1e-4
  mT5 LR      : 5e-5
  Weight decay: 0.01

Batch:
  Cases: 4
  Images/case: [8, 8, 4, 8]
  Targets: ['VIÊM ỐNG TAI NGOÀI + MÀNG NHĨ T VA A-MI-ĐAN QUÁ PHÁT', 'VIÊM MŨI XOANG CẤP', 'VIÊM ỐNG TAI NGOÀI P CẤP/ NÚT RÁY TAI P (ĐÃ LẤY)', 'VIÊM MŨI XOANG + VIÊM VA']

Visual prefix:
  Shape: (4, 8, 512)
Labels:
  Shape: (4, 33)

Forward:
  Loss: 28.893199920654297

Backward:
  Completed successfully.

Gradient check:
  Projector grad norm: 66840.4736328125
  mT5 grad norm      : 113382.99072006345

GPU memory:
  Allocated: 4.11 GB
  Reserved : 5.25 GB
  Peak     : 4.11 GB

CELL 8A — PASS
Single training step works.
BF16 works.
Backward works.
Gradients are present.
Ready for full training loop.


In [17]:
# ============================================================
# CELL 8B — FULL TRAINING LOOP
# ============================================================

import os
import gc
import json
import random
import numpy as np
import pandas as pd
import torch

from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from transformers.modeling_outputs import BaseModelOutput


# ============================================================
# 1. CONFIG
# ============================================================

NUM_EPOCHS = 5

GRADIENT_ACCUMULATION_STEPS = 4

LR_PROJECTOR = 1e-4
LR_MT5 = 5e-5

WEIGHT_DECAY = 0.01

WARMUP_RATIO = 0.05

CHECKPOINT_DIR = os.path.join(
    OUTPUT_DIR,
    "checkpoints"
)

METRICS_DIR = os.path.join(
    OUTPUT_DIR,
    "metrics"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

LAST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "last.pt"
)

BEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best.pt"
)

HISTORY_FILE = os.path.join(
    METRICS_DIR,
    "training_history.csv"
)

CONFIG_FILE = os.path.join(
    OUTPUT_DIR,
    "training_config.json"
)

print("=" * 70)
print("CELL 8B — FULL TRAINING LOOP")
print("=" * 70)

print("\nConfiguration:")
print("  Epochs:", NUM_EPOCHS)
print(
    "  Gradient accumulation:",
    GRADIENT_ACCUMULATION_STEPS
)
print("  Effective batch:", 4 * GRADIENT_ACCUMULATION_STEPS)
print("  Projector LR:", LR_PROJECTOR)
print("  mT5 LR:", LR_MT5)
print("  Weight decay:", WEIGHT_DECAY)
print("  Warmup ratio:", WARMUP_RATIO)
print("  BF16: True")


# ============================================================
# 2. SAVE CONFIG
# ============================================================

training_config = {
    "epochs": NUM_EPOCHS,
    "gradient_accumulation_steps":
        GRADIENT_ACCUMULATION_STEPS,
    "batch_size": 4,
    "effective_batch_size":
        4 * GRADIENT_ACCUMULATION_STEPS,
    "lr_projector": LR_PROJECTOR,
    "lr_mt5": LR_MT5,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "vision_model": VISION_MODEL_NAME,
    "text_model": TEXT_MODEL_NAME,
    "vision_frozen": True,
    "max_images_per_case": MAX_IMAGES_PER_CASE,
    "image_size": IMAGE_SIZE,
    "max_target_length": MAX_TARGET_LENGTH,
    "precision": "bfloat16",
}

with open(CONFIG_FILE, "w") as f:
    json.dump(
        training_config,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nSaved config:", CONFIG_FILE)


# ============================================================
# 3. MODEL MODES
# ============================================================

vit_model.eval()

for param in vit_model.parameters():
    param.requires_grad = False

mt5_model.train()
visual_projector.train()


# ============================================================
# 4. OPTIMIZER
# ============================================================

optimizer = AdamW(
    [
        {
            "params": visual_projector.parameters(),
            "lr": LR_PROJECTOR,
        },
        {
            "params": mt5_model.parameters(),
            "lr": LR_MT5,
        },
    ],
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 5. NUMBER OF TRAINING STEPS
# ============================================================

steps_per_epoch = len(train_loader)

optimizer_steps_per_epoch = int(
    np.ceil(
        steps_per_epoch /
        GRADIENT_ACCUMULATION_STEPS
    )
)

total_optimizer_steps = (
    optimizer_steps_per_epoch *
    NUM_EPOCHS
)

warmup_steps = max(
    1,
    int(
        total_optimizer_steps *
        WARMUP_RATIO
    )
)

print("\nTraining steps:")
print("  Batches / epoch:", steps_per_epoch)
print(
    "  Optimizer steps / epoch:",
    optimizer_steps_per_epoch
)
print(
    "  Total optimizer steps:",
    total_optimizer_steps
)
print("  Warmup steps:", warmup_steps)


# ============================================================
# 6. SCHEDULER
# ============================================================

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_optimizer_steps
)


# ============================================================
# 7. VISUAL ENCODER FUNCTION
# ============================================================

def build_visual_prefix(images_list):

    projected_embeddings = []
    attention_masks = []

    # ViT is frozen
    with torch.no_grad():

        for images in images_list:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            vit_output = vit_model(
                pixel_values=images
            )

            cls_embeddings = (
                vit_output.last_hidden_state[:, 0, :]
            )

            projected = visual_projector(
                cls_embeddings
            )

            projected_embeddings.append(
                projected
            )

            attention_masks.append(
                torch.ones(
                    projected.shape[0],
                    dtype=torch.long,
                    device=DEVICE
                )
            )

    visual_prefix = torch.nn.utils.rnn.pad_sequence(
        projected_embeddings,
        batch_first=True
    )

    visual_attention_mask = torch.nn.utils.rnn.pad_sequence(
        attention_masks,
        batch_first=True,
        padding_value=0
    )

    return (
        visual_prefix,
        visual_attention_mask
    )


# ============================================================
# 8. TRAINING STEP
# ============================================================

def train_one_batch(batch):

    images_list = batch["images"]
    targets = batch["target_text"]

    visual_prefix, visual_attention_mask = (
        build_visual_prefix(images_list)
    )

    tokenized = tokenizer(
        targets,
        padding=True,
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
        return_tensors="pt"
    )

    labels = tokenized.input_ids.to(DEVICE)

    labels[
        labels == tokenizer.pad_token_id
    ] = -100

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16
    ):

        encoder_outputs = mt5_model.encoder(
            inputs_embeds=visual_prefix,
            attention_mask=visual_attention_mask,
            return_dict=True
        )

        outputs = mt5_model(
            encoder_outputs=encoder_outputs,
            attention_mask=visual_attention_mask,
            labels=labels,
            return_dict=True
        )

        loss = outputs.loss

    return loss


# ============================================================
# 9. VALIDATION
# ============================================================

@torch.no_grad()
def validate():

    mt5_model.eval()
    visual_projector.eval()

    total_loss = 0.0
    num_batches = 0

    for batch in val_loader:

        loss = train_one_batch(batch)

        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / max(
        1,
        num_batches
    )

    mt5_model.train()
    visual_projector.train()

    return avg_loss


# ============================================================
# 10. CHECKPOINT HELPERS
# ============================================================

def save_checkpoint(
    path,
    epoch,
    global_step,
    best_val_loss,
    history
):

    checkpoint = {
        "epoch": epoch,
        "global_step": global_step,
        "best_val_loss": best_val_loss,

        "visual_projector":
            visual_projector.state_dict(),

        "mt5_model":
            mt5_model.state_dict(),

        "optimizer":
            optimizer.state_dict(),

        "scheduler":
            scheduler.state_dict(),

        "history": history,

        "rng_state":
            torch.get_rng_state(),

        "cuda_rng_state":
            torch.cuda.get_rng_state_all(),

        "numpy_rng_state":
            np.random.get_state(),

        "python_rng_state":
            random.getstate(),

        "config":
            training_config,
    }

    torch.save(
        checkpoint,
        path
    )


# ============================================================
# 11. RESUME IF CHECKPOINT EXISTS
# ============================================================

start_epoch = 0
global_step = 0
best_val_loss = float("inf")
history = []

if os.path.exists(LAST_CHECKPOINT):

    print("\nExisting checkpoint found:")
    print(LAST_CHECKPOINT)

    checkpoint = torch.load(
        LAST_CHECKPOINT,
        map_location=DEVICE
    )

    visual_projector.load_state_dict(
        checkpoint["visual_projector"]
    )

    mt5_model.load_state_dict(
        checkpoint["mt5_model"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler"]
    )

    start_epoch = (
        checkpoint["epoch"] + 1
    )

    global_step = checkpoint[
        "global_step"
    ]

    best_val_loss = checkpoint[
        "best_val_loss"
    ]

    history = checkpoint.get(
        "history",
        []
    )

    torch.set_rng_state(
        checkpoint["rng_state"]
    )

    torch.cuda.set_rng_state_all(
        checkpoint["cuda_rng_state"]
    )

    np.random.set_state(
        checkpoint["numpy_rng_state"]
    )

    random.setstate(
        checkpoint["python_rng_state"]
    )

    print(
        f"Resuming from epoch {start_epoch}"
    )

else:

    print("\nNo checkpoint found.")
    print("Starting from epoch 0.")


# ============================================================
# 12. TRAIN
# ============================================================

for epoch in range(
    start_epoch,
    NUM_EPOCHS
):

    print("\n")
    print("=" * 70)
    print(
        f"EPOCH {epoch + 1}/{NUM_EPOCHS}"
    )
    print("=" * 70)

    mt5_model.train()
    visual_projector.train()

    optimizer.zero_grad(
        set_to_none=True
    )

    running_loss = 0.0

    for batch_idx, batch in enumerate(
        train_loader
    ):

        loss = train_one_batch(
            batch
        )

        # Gradient accumulation
        scaled_loss = (
            loss /
            GRADIENT_ACCUMULATION_STEPS
        )

        scaled_loss.backward()

        running_loss += loss.item()

        should_step = (
            (batch_idx + 1)
            % GRADIENT_ACCUMULATION_STEPS
            == 0
            or
            (batch_idx + 1)
            == len(train_loader)
        )

        if should_step:

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(
                list(
                    visual_projector.parameters()
                ) +
                list(
                    mt5_model.parameters()
                ),
                max_norm=1.0
            )

            optimizer.step()
            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )

            global_step += 1

        # Progress
        if (
            (batch_idx + 1) % 100 == 0
            or
            (batch_idx + 1)
            == len(train_loader)
        ):

            avg_so_far = (
                running_loss /
                (batch_idx + 1)
            )

            print(
                f"Batch "
                f"{batch_idx + 1}/"
                f"{len(train_loader)} | "
                f"Loss: {loss.item():.4f} | "
                f"Avg: {avg_so_far:.4f} | "
                f"LR: "
                f"{scheduler.get_last_lr()[0]:.2e}"
            )


    # --------------------------------------------------------
    # Epoch metrics
    # --------------------------------------------------------

    train_loss = (
        running_loss /
        len(train_loader)
    )

    val_loss = validate()

    print("\nEpoch result:")
    print(
        f"  Train loss: {train_loss:.4f}"
    )
    print(
        f"  Val loss  : {val_loss:.4f}"
    )

    # --------------------------------------------------------
    # Save history
    # --------------------------------------------------------

    epoch_record = {
        "epoch": epoch + 1,
        "global_step": global_step,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "lr_projector":
            scheduler.get_last_lr()[0],
        "lr_mt5":
            scheduler.get_last_lr()[1],
    }

    history.append(
        epoch_record
    )

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_FILE,
        index=False
    )

    # --------------------------------------------------------
    # Save last checkpoint
    # --------------------------------------------------------

    save_checkpoint(
        LAST_CHECKPOINT,
        epoch,
        global_step,
        best_val_loss,
        history
    )

    print(
        "\nSaved:",
        LAST_CHECKPOINT
    )

    # --------------------------------------------------------
    # Save best checkpoint
    # --------------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        save_checkpoint(
            BEST_CHECKPOINT,
            epoch,
            global_step,
            best_val_loss,
            history
        )

        print(
            "New best checkpoint:",
            BEST_CHECKPOINT
        )

    # --------------------------------------------------------
    # Epoch checkpoint
    # --------------------------------------------------------

    epoch_checkpoint = os.path.join(
        CHECKPOINT_DIR,
        f"epoch_{epoch + 1:02d}.pt"
    )

    save_checkpoint(
        epoch_checkpoint,
        epoch,
        global_step,
        best_val_loss,
        history
    )

    print(
        "Saved epoch checkpoint:",
        epoch_checkpoint
    )

    # Memory cleanup
    gc.collect()
    torch.cuda.empty_cache()


# ============================================================
# 13. TRAINING FINISHED
# ============================================================

print("\n" + "=" * 70)
print("CELL 8B — TRAINING COMPLETE")
print("=" * 70)

print(
    "Best validation loss:",
    best_val_loss
)

print(
    "History:",
    HISTORY_FILE
)

print(
    "Best checkpoint:",
    BEST_CHECKPOINT
)

print("=" * 70)

CELL 8B — FULL TRAINING LOOP

Configuration:
  Epochs: 5
  Gradient accumulation: 4
  Effective batch: 16
  Projector LR: 0.0001
  mT5 LR: 5e-05
  Weight decay: 0.01
  Warmup ratio: 0.05
  BF16: True

Saved config: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v1/training_config.json

Training steps:
  Batches / epoch: 1535
  Optimizer steps / epoch: 384
  Total optimizer steps: 1920
  Warmup steps: 96

No checkpoint found.
Starting from epoch 0.


EPOCH 1/5
Batch 100/1535 | Loss: 29.2065 | Avg: 28.4519 | LR: 2.60e-05
Batch 200/1535 | Loss: 22.3445 | Avg: 27.5853 | LR: 5.21e-05
Batch 300/1535 | Loss: 28.1846 | Avg: 26.3973 | LR: 7.81e-05
Batch 400/1535 | Loss: 21.7332 | Avg: 25.2672 | LR: 1.00e-04
Batch 500/1535 | Loss: 19.4737 | Avg: 23.9008 | LR: 9.99e-05
Batch 600/1535 | Loss: 12.2131 | Avg: 22.4534 | LR: 9.98e-05
Batch 700/1535 | Loss: 9.8595 | Avg: 21.0654 | LR: 9.95e-05
Batch 800/1535 | Loss: 10.7446 | Avg: 19.8034 | LR: 9.92e-05
Batch 900/1535 | Loss: 8.5327 | Avg: 18.64

In [19]:
# ============================================================
# CELL 9A — LOAD BEST MODEL + TEST GENERATION
# ============================================================

import os
import re
import gc
import unicodedata
import numpy as np
import pandas as pd
import torch

from transformers.modeling_outputs import BaseModelOutput

print("=" * 70)
print("CELL 9A — BEST CHECKPOINT + TEST GENERATION")
print("=" * 70)


# ============================================================
# 1. Load BEST checkpoint
# ============================================================

# ============================================================
# LOAD BEST CHECKPOINT
# ============================================================

BEST_CHECKPOINT = os.path.join(
    OUTPUT_DIR,
    "checkpoints",
    "best.pt"
)

assert os.path.exists(
    BEST_CHECKPOINT
), f"Checkpoint not found: {BEST_CHECKPOINT}"

print("\nLoading:")
print(BEST_CHECKPOINT)

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

visual_projector.load_state_dict(
    checkpoint["visual_projector"]
)

mt5_model.load_state_dict(
    checkpoint["mt5_model"]
)

vit_model.eval()
visual_projector.eval()
mt5_model.eval()

print("\nCheckpoint loaded successfully.")

print(
    "Best checkpoint epoch:",
    checkpoint["epoch"] + 1
)

print(
    "Best validation loss:",
    checkpoint["best_val_loss"]
  )


# ============================================================
# 2. Generation helper
# ============================================================

@torch.no_grad()
def generate_batch(batch):

    images_list = batch["images"]

    # --------------------------------------------------------
    # ViT
    # --------------------------------------------------------

    projected_embeddings = []
    attention_masks = []

    for images in images_list:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        vit_output = vit_model(
            pixel_values=images
        )

        cls_embeddings = (
            vit_output.last_hidden_state[:, 0, :]
        )

        projected = visual_projector(
            cls_embeddings
        )

        projected_embeddings.append(
            projected
        )

        attention_masks.append(
            torch.ones(
                projected.shape[0],
                dtype=torch.long,
                device=DEVICE
            )
        )

    # --------------------------------------------------------
    # Pad visual tokens
    # --------------------------------------------------------

    visual_prefix = torch.nn.utils.rnn.pad_sequence(
        projected_embeddings,
        batch_first=True
    )

    visual_attention_mask = torch.nn.utils.rnn.pad_sequence(
        attention_masks,
        batch_first=True,
        padding_value=0
    )

    # --------------------------------------------------------
    # mT5 encoder
    # --------------------------------------------------------

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16
    ):

        encoder_outputs = mt5_model.encoder(
            inputs_embeds=visual_prefix,
            attention_mask=visual_attention_mask,
            return_dict=True
        )

        generated_ids = mt5_model.generate(
            encoder_outputs=BaseModelOutput(
                last_hidden_state=
                encoder_outputs.last_hidden_state
            ),
            attention_mask=visual_attention_mask,
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=4,
            do_sample=False,
            early_stopping=True
        )

    generated_texts = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )

    return generated_texts


# ============================================================
# 3. Normalization helper
# ============================================================

def normalize_text(text):

    text = str(text).strip().upper()

    text = unicodedata.normalize(
        "NFC",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text


# ============================================================
# 4. Test generation
# ============================================================

test_results = []

print("\nRunning test generation...")
print(
    "Test batches:",
    len(test_loader)
)

for batch_idx, batch in enumerate(
    test_loader
):

    generated = generate_batch(
        batch
    )

    for i in range(
        len(generated)
    ):

        test_results.append({
            "case_id":
                batch["case_id"][i],

            "patient_group_id":
                batch["patient_group_id"][i],

            "ground_truth":
                batch["target_text"][i],

            "prediction":
                generated[i],

            "ground_truth_normalized":
                normalize_text(
                    batch["target_text"][i]
                ),

            "prediction_normalized":
                normalize_text(
                    generated[i]
                )
        })

    if (
        (batch_idx + 1) % 50 == 0
        or
        (batch_idx + 1) == len(test_loader)
    ):

        print(
            f"Batch "
            f"{batch_idx + 1}/"
            f"{len(test_loader)}"
        )


# ============================================================
# 5. Save raw predictions
# ============================================================

TEST_RESULTS_FILE = os.path.join(
    OUTPUT_DIR,
    "test_predictions.csv"
)

test_results_df = pd.DataFrame(
    test_results
)

test_results_df.to_csv(
    TEST_RESULTS_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print(TEST_RESULTS_FILE)

print(
    "Test cases:",
    len(test_results_df)
)


# ============================================================
# 6. Basic exact match
# ============================================================

exact_match = (
    test_results_df[
        "ground_truth_normalized"
    ]
    ==
    test_results_df[
        "prediction_normalized"
    ]
).mean()

print("\nExact match:")
print(
    f"{exact_match:.4%}"
)


# ============================================================
# 7. Show first 20 predictions
# ============================================================

print("\n" + "=" * 70)
print("SAMPLE TEST PREDICTIONS")
print("=" * 70)

for i, row in test_results_df.head(20).iterrows():

    print(f"\n[{i + 1}] Case: {row['case_id']}")

    print(
        "GROUND TRUTH:",
        row["ground_truth"]
    )

    print(
        "PREDICTION  :",
        row["prediction"]
    )

print("\n" + "=" * 70)
print("CELL 9A — PASS")
print("=" * 70)

CELL 9A — BEST CHECKPOINT + TEST GENERATION

Loading:
/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v1/checkpoints/best.pt

Checkpoint loaded successfully.
Best checkpoint epoch: 5
Best validation loss: 0.6355828013290804

Running test generation...
Test batches: 178
Batch 50/178
Batch 100/178
Batch 150/178
Batch 178/178

Saved:
/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v1/test_predictions.csv
Test cases: 712

Exact match:
11.7978%

SAMPLE TEST PREDICTIONS

[1] Case: 10017.10017.0.10031
GROUND TRUTH: VIÊM MÀNG NHĨ CẤP BÓNG NƯỚC P - NHIỀU RÁY TAI T
PREDICTION  : VIÊM MŨI MẠN

[2] Case: 10057.10057.0.10071
GROUND TRUTH: VIÊM ỐNG TAI NGOÀI (T) MẠN / VIÊM MŨI ĐỢT CẤP.
PREDICTION  : VIÊM MŨI MẠN

[3] Case: 10116.10116.0.10130
GROUND TRUTH: VIÊM ỐNG TAI NGOÀI MÀNG NHĨ P CẤP
PREDICTION  : VIÊM MŨI MẠN

[4] Case: 10125.10125.0.10139
GROUND TRUTH: VIÊM MŨI MẠN VIÊM ỐNG TAI NGOÀI + MÀNG NHĨ 2 BÊN MẠN
PREDICTION  : VIÊM MŨI

[5] Case: 10126.10126.0.10140
GROUND TRUTH: VIÊM A-M

In [21]:
# ============================================================
# CELL 9B — TEST METRICS + PREDICTION DISTRIBUTION
# ============================================================

import os
import re
import unicodedata
import collections
import numpy as np
import pandas as pd

from difflib import SequenceMatcher

print("=" * 70)
print("CELL 9B — TEST METRICS + PREDICTION DISTRIBUTION")
print("=" * 70)


# ============================================================
# 1. Locate saved predictions
# ============================================================

PREDICTION_FILE = os.path.join(
    OUTPUT_DIR,
    "test_predictions.csv"
)

assert os.path.exists(
    PREDICTION_FILE
), f"Prediction file not found:\n{PREDICTION_FILE}"

print("\nLoading:")
print(PREDICTION_FILE)

df = pd.read_csv(
    PREDICTION_FILE
)

print("Rows:", len(df))

assert len(df) == 712

print("PASS: all 712 test cases found.")


# ============================================================
# 2. Text normalization
# ============================================================

def normalize_text(text):

    text = str(text).strip().upper()

    text = unicodedata.normalize(
        "NFC",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text


df["gt_norm"] = df["ground_truth"].apply(
    normalize_text
)

df["pred_norm"] = df["prediction"].apply(
    normalize_text
)


# ============================================================
# 3. Exact match
# ============================================================

df["exact_match"] = (
    df["gt_norm"]
    ==
    df["pred_norm"]
)

exact_match = df["exact_match"].mean()


# ============================================================
# 4. Character similarity
# ============================================================

def char_similarity(a, b):

    return SequenceMatcher(
        None,
        a,
        b
    ).ratio()


df["char_similarity"] = [
    char_similarity(
        gt,
        pred
    )
    for gt, pred in zip(
        df["gt_norm"],
        df["pred_norm"]
    )
]

mean_char_similarity = (
    df["char_similarity"].mean()
)

median_char_similarity = (
    df["char_similarity"].median()
)


# ============================================================
# 5. Token overlap
# ============================================================

def token_f1(a, b):

    a_tokens = a.split()
    b_tokens = b.split()

    if len(a_tokens) == 0 and len(b_tokens) == 0:
        return 1.0

    if len(a_tokens) == 0 or len(b_tokens) == 0:
        return 0.0

    a_counter = collections.Counter(
        a_tokens
    )

    b_counter = collections.Counter(
        b_tokens
    )

    overlap = sum(
        (a_counter & b_counter).values()
    )

    precision = (
        overlap /
        len(b_tokens)
    )

    recall = (
        overlap /
        len(a_tokens)
    )

    if precision + recall == 0:
        return 0.0

    return (
        2 *
        precision *
        recall /
        (precision + recall)
    )


df["token_f1"] = [
    token_f1(
        gt,
        pred
    )
    for gt, pred in zip(
        df["gt_norm"],
        df["pred_norm"]
    )
]

mean_token_f1 = df[
    "token_f1"
].mean()

median_token_f1 = df[
    "token_f1"
].median()


# ============================================================
# 6. BLEU
# ============================================================

try:

    from nltk.translate.bleu_score import (
        sentence_bleu,
        SmoothingFunction
    )

    smoothing = SmoothingFunction().method1

    bleu_scores = []

    for gt, pred in zip(
        df["gt_norm"],
        df["pred_norm"]
    ):

        reference = [
            gt.split()
        ]

        hypothesis = pred.split()

        if len(hypothesis) == 0:
            score = 0.0
        else:
            score = sentence_bleu(
                reference,
                hypothesis,
                smoothing_function=smoothing
            )

        bleu_scores.append(score)

    df["bleu"] = bleu_scores

    mean_bleu = np.mean(
        bleu_scores
    )

except Exception as e:

    print(
        "\nBLEU unavailable:",
        e
    )

    df["bleu"] = np.nan
    mean_bleu = np.nan


# ============================================================
# 7. ROUGE-L
# ============================================================

try:

    from rouge_score import rouge_scorer

    scorer = rouge_scorer.RougeScorer(
        ["rougeL"],
        use_stemmer=False
    )

    rouge_scores = []

    for gt, pred in zip(
        df["gt_norm"],
        df["pred_norm"]
    ):

        score = scorer.score(
            gt,
            pred
        )["rougeL"].fmeasure

        rouge_scores.append(score)

    df["rougeL"] = rouge_scores

    mean_rougeL = np.mean(
        rouge_scores
    )

except Exception as e:

    print(
        "\nROUGE-L unavailable:",
        e
    )

    df["rougeL"] = np.nan
    mean_rougeL = np.nan


# ============================================================
# 8. Prediction distribution
# ============================================================

prediction_counts = (
    df["pred_norm"]
    .value_counts()
    .reset_index()
)

prediction_counts.columns = [
    "prediction",
    "count"
]

prediction_counts["percentage"] = (
    prediction_counts["count"]
    /
    len(df)
    *
    100
)


ground_truth_counts = (
    df["gt_norm"]
    .value_counts()
    .reset_index()
)

ground_truth_counts.columns = [
    "ground_truth",
    "count"
]

ground_truth_counts["percentage"] = (
    ground_truth_counts["count"]
    /
    len(df)
    *
    100
)


# ============================================================
# 9. Prediction diversity
# ============================================================

num_unique_predictions = (
    df["pred_norm"].nunique()
)

num_unique_ground_truth = (
    df["gt_norm"].nunique()
)

prediction_coverage = (
    num_unique_predictions /
    num_unique_ground_truth
)


# ============================================================
# 10. Top predictions
# ============================================================

print("\n" + "=" * 70)
print("METRICS")
print("=" * 70)

print(
    f"Exact match       : "
    f"{exact_match:.4%}"
)

print(
    f"Mean char similarity: "
    f"{mean_char_similarity:.4f}"
)

print(
    f"Median char similarity: "
    f"{median_char_similarity:.4f}"
)

print(
    f"Mean token F1     : "
    f"{mean_token_f1:.4f}"
)

print(
    f"Median token F1   : "
    f"{median_token_f1:.4f}"
)

print(
    f"Mean BLEU         : "
    f"{mean_bleu:.4f}"
)

print(
    f"Mean ROUGE-L      : "
    f"{mean_rougeL:.4f}"
)

print(
    f"Unique predictions: "
    f"{num_unique_predictions}"
)

print(
    f"Unique ground truth: "
    f"{num_unique_ground_truth}"
)

print(
    f"Prediction/GT diversity ratio: "
    f"{prediction_coverage:.4f}"
)


# ============================================================
# 11. Top predictions
# ============================================================

print("\n" + "=" * 70)
print("TOP 20 MODEL PREDICTIONS")
print("=" * 70)

print(
    prediction_counts.head(20).to_string(
        index=False
    )
)


# ============================================================
# 12. Top ground truths
# ============================================================

print("\n" + "=" * 70)
print("TOP 20 GROUND-TRUTH CONCLUSIONS")
print("=" * 70)

print(
    ground_truth_counts.head(20).to_string(
        index=False
    )
)


# ============================================================
# 13. Save metrics
# ============================================================

METRICS_FILE = os.path.join(
    OUTPUT_DIR,
    "test_metrics.csv"
)

metrics_df = pd.DataFrame([
    {
        "metric":
            "exact_match",
        "value":
            exact_match
    },
    {
        "metric":
            "mean_char_similarity",
        "value":
            mean_char_similarity
    },
    {
        "metric":
            "median_char_similarity",
        "value":
            median_char_similarity
    },
    {
        "metric":
            "mean_token_f1",
        "value":
            mean_token_f1
    },
    {
        "metric":
            "median_token_f1",
        "value":
            median_token_f1
    },
    {
        "metric":
            "mean_bleu",
        "value":
            mean_bleu
    },
    {
        "metric":
            "mean_rougeL",
        "value":
            mean_rougeL
    },
    {
        "metric":
            "unique_predictions",
        "value":
            num_unique_predictions
    },
    {
        "metric":
            "unique_ground_truth",
        "value":
            num_unique_ground_truth
    }
])

metrics_df.to_csv(
    METRICS_FILE,
    index=False
)


# ============================================================
# 14. Save distributions
# ============================================================

PRED_DIST_FILE = os.path.join(
    OUTPUT_DIR,
    "prediction_distribution.csv"
)

GT_DIST_FILE = os.path.join(
    OUTPUT_DIR,
    "ground_truth_distribution_test.csv"
)

prediction_counts.to_csv(
    PRED_DIST_FILE,
    index=False,
    encoding="utf-8-sig"
)

ground_truth_counts.to_csv(
    GT_DIST_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 15. Save enriched predictions
# ============================================================

ENRICHED_FILE = os.path.join(
    OUTPUT_DIR,
    "test_predictions_with_metrics.csv"
)

df.to_csv(
    ENRICHED_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 16. Final
# ============================================================

print("\n" + "=" * 70)
print("CELL 9B — PASS")
print("=" * 70)

print("Saved:")
print("  ", METRICS_FILE)
print("  ", PRED_DIST_FILE)
print("  ", GT_DIST_FILE)
print("  ", ENRICHED_FILE)

print("=" * 70)

CELL 9B — TEST METRICS + PREDICTION DISTRIBUTION

Loading:
/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v1/test_predictions.csv
Rows: 712
PASS: all 712 test cases found.

ROUGE-L unavailable: No module named 'rouge_score'

METRICS
Exact match       : 11.7978%
Mean char similarity: 0.5242
Median char similarity: 0.4577
Mean token F1     : 0.4747
Median token F1   : 0.4000
Mean BLEU         : 0.1189
Mean ROUGE-L      : nan
Unique predictions: 2
Unique ground truth: 267
Prediction/GT diversity ratio: 0.0075

TOP 20 MODEL PREDICTIONS
  prediction  count  percentage
VIÊM MŨI MẠN    475   66.713483
    VIÊM MŨI    237   33.286517

TOP 20 GROUND-TRUTH CONCLUSIONS
                                                     ground_truth  count  percentage
                                                     VIÊM MŨI MẠN     79   11.095506
                                                         VIÊM MŨI     66    9.269663
                     VIÊM HỌNG MẠN - THEO DÕI TRÀO NGƯỢC DỊCH VỊ.     5

In [22]:
# ============================================================
# V2-1 — RUNTIME-INDEPENDENT SETUP
# ============================================================

import os
import gc
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from transformers import (
    ViTModel,
    MT5ForConditionalGeneration,
    AutoTokenizer
)

print("=" * 70)
print("V2-1 — RUNTIME-INDEPENDENT SETUP")
print("=" * 70)

# ------------------------------------------------------------
# 1. Drive
# ------------------------------------------------------------

from google.colab import drive

drive.mount("/content/drive", force_remount=False)


# ------------------------------------------------------------
# 2. Paths
# ------------------------------------------------------------

PROJECT_DIR = (
    "/content/drive/MyDrive/NoiSoi_Matching"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "baseline_model_v2"
)

MANIFEST_FILE = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

CASE_MANIFEST_FILE = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_case_manifest.csv"
)

SPLIT_FILE = os.path.join(
    PROJECT_DIR,
    "patient_split",
    "cases_with_split.csv"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("\nPaths:")
print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR :", OUTPUT_DIR)


# ------------------------------------------------------------
# 3. Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:", DEVICE)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "VRAM:",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

    print(
        "BF16:",
        torch.cuda.is_bf16_supported()
    )


# ------------------------------------------------------------
# 4. Fixed architecture config
# ------------------------------------------------------------

VISION_MODEL_NAME = (
    "google/vit-base-patch16-224"
)

TEXT_MODEL_NAME = (
    "google/mt5-small"
)

MAX_IMAGES_PER_CASE = 8
IMAGE_SIZE = 224
MAX_TARGET_LENGTH = 96

BATCH_SIZE = 4

print("\nArchitecture:")
print("ViT :", VISION_MODEL_NAME)
print("mT5 :", TEXT_MODEL_NAME)
print("Max images:", MAX_IMAGES_PER_CASE)
print("Image size:", IMAGE_SIZE)
print("Target length:", MAX_TARGET_LENGTH)
print("Batch size:", BATCH_SIZE)


# ------------------------------------------------------------
# 5. Verify source files
# ------------------------------------------------------------

for path in [
    MANIFEST_FILE,
    CASE_MANIFEST_FILE,
    SPLIT_FILE
]:

    assert os.path.exists(path), (
        f"Missing:\n{path}"
    )

print("\nSource files: PASS")


# ------------------------------------------------------------
# 6. Load manifests
# ------------------------------------------------------------

manifest = pd.read_csv(
    MANIFEST_FILE
)

case_manifest = pd.read_csv(
    CASE_MANIFEST_FILE
)

split_df = pd.read_csv(
    SPLIT_FILE
)

print("\nLoaded:")
print("Images:", len(manifest))
print("Cases :", len(case_manifest))
print("Split :", len(split_df))


# ------------------------------------------------------------
# 7. Keep only NORMAL images
# ------------------------------------------------------------

normal_manifest = manifest[
    manifest["image_status"] == "NORMAL"
].copy()

print(
    "\nNORMAL images:",
    len(normal_manifest)
)


# ------------------------------------------------------------
# 8. Model cases
# ------------------------------------------------------------

cases_model = case_manifest.copy()

cases_model["ket_luan"] = (
    cases_model["ket_luan"]
    .fillna("")
    .astype(str)
    .str.strip()
)

cases_model = cases_model[
    cases_model["ket_luan"] != ""
].copy()

print(
    "Cases with non-empty target:",
    len(cases_model)
)


# ------------------------------------------------------------
# 9. Fixed split
# ------------------------------------------------------------

split_lookup = split_df[
    [
        "case_id",
        "patient_group_id",
        "split"
    ]
].drop_duplicates(
    subset=["case_id"]
)

cases_model = cases_model.merge(
    split_lookup,
    on="case_id",
    how="inner",
    suffixes=("", "_split")
)

# Use the fixed split
if "split_split" in cases_model.columns:
    cases_model["split"] = (
        cases_model["split_split"]
    )
    cases_model.drop(
        columns=["split_split"],
        inplace=True
    )

print("\nSplit counts:")

print(
    cases_model["split"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 10. Verify patient leakage
# ------------------------------------------------------------

patient_split_counts = (
    cases_model
    .groupby("patient_group_id")["split"]
    .nunique()
)

leaking_patients = (
    patient_split_counts[
        patient_split_counts > 1
    ]
)

print(
    "\nPatient leakage:",
    len(leaking_patients)
)

assert len(leaking_patients) == 0


# ------------------------------------------------------------
# 11. Case → image paths
# ------------------------------------------------------------

normal_manifest["case_id"] = (
    normal_manifest["case_id"]
    .astype(str)
)

case_images = (
    normal_manifest
    .groupby("case_id")["image_path"]
    .apply(list)
    .to_dict()
)

cases_model["image_paths"] = (
    cases_model["case_id"]
    .astype(str)
    .map(case_images)
)

cases_model = cases_model[
    cases_model["image_paths"].notna()
].copy()

cases_model["image_paths"] = (
    cases_model["image_paths"]
    .apply(list)
)

print(
    "\nCases with images:",
    len(cases_model)
)


# ------------------------------------------------------------
# 12. Dataset records
# ------------------------------------------------------------

dataset_records = []

for _, row in cases_model.iterrows():

    dataset_records.append({
        "case_id":
            str(row["case_id"]),

        "patient_group_id":
            str(row["patient_group_id"]),

        "split":
            str(row["split"]),

        "image_paths":
            row["image_paths"],

        "target_text":
            str(row["ket_luan"])
    })


print(
    "Dataset records:",
    len(dataset_records)
)


# ------------------------------------------------------------
# 13. Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("V2-1 — PASS")
print("=" * 70)

print(
    "Train:",
    sum(
        x["split"] == "train"
        for x in dataset_records
    )
)

print(
    "Val:",
    sum(
        x["split"] == "val"
        for x in dataset_records
    )
)

print(
    "Test:",
    sum(
        x["split"] == "test"
        for x in dataset_records
    )
)

print(
    "Patient leakage:",
    len(leaking_patients)
)

print("=" * 70)

V2-1 — RUNTIME-INDEPENDENT SETUP
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Paths:
PROJECT_DIR: /content/drive/MyDrive/NoiSoi_Matching
OUTPUT_DIR : /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v2

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.49 GB
BF16: True

Architecture:
ViT : google/vit-base-patch16-224
mT5 : google/mt5-small
Max images: 8
Image size: 224
Target length: 96
Batch size: 4

Source files: PASS


/tmp/ipykernel_11258/982605115.py:152: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  manifest = pd.read_csv(



Loaded:
Images: 76405
Cases : 7607
Split : 7607

NORMAL images: 76216
Cases with non-empty target: 7606

Split counts:
split
test      712
train    6138
val       756
Name: count, dtype: int64

Patient leakage: 0

Cases with images: 7606
Dataset records: 7606

V2-1 — PASS
Train: 6138
Val: 756
Test: 712
Patient leakage: 0


In [23]:
# ============================================================
# V2-2 — LOAD MODELS + REBUILD DATALOADERS
# ============================================================

import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

from transformers import (
    ViTModel,
    MT5ForConditionalGeneration,
    AutoTokenizer
)

print("=" * 70)
print("V2-2 — LOAD MODELS + REBUILD DATALOADERS")
print("=" * 70)


# ------------------------------------------------------------
# 1. Save V2 configuration
# ------------------------------------------------------------

V2_CONFIG_FILE = os.path.join(
    OUTPUT_DIR,
    "v2_config.json"
)

v2_config = {
    "vision_model": VISION_MODEL_NAME,
    "text_model": TEXT_MODEL_NAME,
    "vision_frozen": False,
    "max_images_per_case": MAX_IMAGES_PER_CASE,
    "image_size": IMAGE_SIZE,
    "max_target_length": MAX_TARGET_LENGTH,
    "batch_size": BATCH_SIZE,
    "patient_split": True,
    "train_cases": 6138,
    "val_cases": 756,
    "test_cases": 712
}

with open(V2_CONFIG_FILE, "w") as f:
    json.dump(
        v2_config,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nSaved config:")
print(V2_CONFIG_FILE)


# ------------------------------------------------------------
# 2. Tokenizer
# ------------------------------------------------------------

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL_NAME
)

print(
    "Tokenizer vocab:",
    tokenizer.vocab_size
)


# ------------------------------------------------------------
# 3. ViT — TRAINABLE in V2
# ------------------------------------------------------------

print("\nLoading ViT...")

vit_model = ViTModel.from_pretrained(
    VISION_MODEL_NAME
).to(DEVICE)

# IMPORTANT:
# V2 fine-tunes ViT
vit_model.train()

for param in vit_model.parameters():
    param.requires_grad = True

print(
    "ViT hidden size:",
    vit_model.config.hidden_size
)

print(
    "ViT trainable params:",
    sum(
        p.numel()
        for p in vit_model.parameters()
        if p.requires_grad
    )
)


# ------------------------------------------------------------
# 4. mT5
# ------------------------------------------------------------

print("\nLoading mT5-small...")

mt5_model = MT5ForConditionalGeneration.from_pretrained(
    TEXT_MODEL_NAME
).to(DEVICE)

mt5_model.train()

print(
    "mT5 d_model:",
    mt5_model.config.d_model
)

print(
    "mT5 trainable params:",
    sum(
        p.numel()
        for p in mt5_model.parameters()
        if p.requires_grad
    )
)


# ------------------------------------------------------------
# 5. Visual projector
# ------------------------------------------------------------

class VisualPrefixProjector(nn.Module):

    def __init__(
        self,
        vision_dim=768,
        text_dim=512
    ):
        super().__init__()

        self.proj = nn.Linear(
            vision_dim,
            text_dim
        )

    def forward(self, x):
        return self.proj(x)


visual_projector = VisualPrefixProjector(
    vision_dim=vit_model.config.hidden_size,
    text_dim=mt5_model.config.d_model
).to(DEVICE)

print(
    "\nVisual projection:",
    f"{vit_model.config.hidden_size}"
    f" → "
    f"{mt5_model.config.d_model}"
)


# ------------------------------------------------------------
# 6. Image transform
# ------------------------------------------------------------

image_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ------------------------------------------------------------
# 7. Dataset
# ------------------------------------------------------------

class MultiImageEndoscopyDataset(Dataset):

    def __init__(
        self,
        records,
        transform,
        max_images=8,
        training=False
    ):

        self.records = records
        self.transform = transform
        self.max_images = max_images
        self.training = training

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):

        record = self.records[idx]

        paths = list(
            record["image_paths"]
        )

        # Random sampling during training
        if (
            self.training
            and len(paths) > self.max_images
        ):

            selected_paths = random.sample(
                paths,
                self.max_images
            )

        else:

            selected_paths = paths[
                :self.max_images
            ]

        images = []

        for path in selected_paths:

            image = Image.open(
                path
            ).convert("RGB")

            image = self.transform(
                image
            )

            images.append(image)

        images = torch.stack(
            images
        )

        return {
            "case_id":
                record["case_id"],

            "patient_group_id":
                record["patient_group_id"],

            "images":
                images,

            "num_images":
                images.shape[0],

            "target_text":
                record["target_text"]
        }


# ------------------------------------------------------------
# 8. Build records
# ------------------------------------------------------------

train_records = [
    x for x in dataset_records
    if x["split"] == "train"
]

val_records = [
    x for x in dataset_records
    if x["split"] == "val"
]

test_records = [
    x for x in dataset_records
    if x["split"] == "test"
]

print("\nRecords:")
print("Train:", len(train_records))
print("Val  :", len(val_records))
print("Test :", len(test_records))


# ------------------------------------------------------------
# 9. Datasets
# ------------------------------------------------------------

train_dataset = MultiImageEndoscopyDataset(
    train_records,
    transform=image_transform,
    max_images=MAX_IMAGES_PER_CASE,
    training=True
)

val_dataset = MultiImageEndoscopyDataset(
    val_records,
    transform=image_transform,
    max_images=MAX_IMAGES_PER_CASE,
    training=False
)

test_dataset = MultiImageEndoscopyDataset(
    test_records,
    transform=image_transform,
    max_images=MAX_IMAGES_PER_CASE,
    training=False
)


# ------------------------------------------------------------
# 10. Collate
# ------------------------------------------------------------

def collate_cases(batch):

    return {
        "case_id": [
            x["case_id"]
            for x in batch
        ],

        "patient_group_id": [
            x["patient_group_id"]
            for x in batch
        ],

        "images": [
            x["images"]
            for x in batch
        ],

        "num_images": [
            x["num_images"]
            for x in batch
        ],

        "target_text": [
            x["target_text"]
            for x in batch
        ]
    }


# ------------------------------------------------------------
# 11. DataLoaders
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_cases
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_cases
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_cases
)


# ------------------------------------------------------------
# 12. Batch sanity check
# ------------------------------------------------------------

batch = next(iter(train_loader))

print("\nBatch keys:")
print(batch.keys())

print(
    "Batch size:",
    len(batch["images"])
)

print(
    "Images/case:",
    batch["num_images"]
)

print(
    "Image tensor shapes:",
    [
        tuple(x.shape)
        for x in batch["images"]
    ]
)

print(
    "First target:",
    batch["target_text"][0]
)


# ------------------------------------------------------------
# 13. Parameter summary
# ------------------------------------------------------------

print("\nParameter summary:")

print(
    "ViT total:",
    sum(
        p.numel()
        for p in vit_model.parameters()
    )
)

print(
    "ViT trainable:",
    sum(
        p.numel()
        for p in vit_model.parameters()
        if p.requires_grad
    )
)

print(
    "mT5 trainable:",
    sum(
        p.numel()
        for p in mt5_model.parameters()
        if p.requires_grad
    )
)

print(
    "Projector trainable:",
    sum(
        p.numel()
        for p in visual_projector.parameters()
        if p.requires_grad
    )
)


# ------------------------------------------------------------
# 14. Final checks
# ------------------------------------------------------------

assert len(train_dataset) == 6138
assert len(val_dataset) == 756
assert len(test_dataset) == 712

assert all(
    p.requires_grad
    for p in vit_model.parameters()
)

assert all(
    p.requires_grad
    for p in mt5_model.parameters()
)

assert (
    visual_projector.proj.in_features
    == 768
)

assert (
    visual_projector.proj.out_features
    == 512
)


print("\n" + "=" * 70)
print("V2-2 — PASS")
print("=" * 70)

print("ViT: TRAINABLE")
print("mT5: TRAINABLE")
print("Projection: TRAINABLE")
print("Patient split: preserved")
print("Train/Val/Test: 6138 / 756 / 712")
print("=" * 70)

V2-2 — LOAD MODELS + REBUILD DATALOADERS

Saved config:
/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v2/v2_config.json

Loading tokenizer...
Tokenizer vocab: 250100

Loading ViT...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ViT hidden size: 768
ViT trainable params: 86389248

Loading mT5-small...


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


mT5 d_model: 512
mT5 trainable params: 300176768

Visual projection: 768 → 512

Records:
Train: 6138
Val  : 756
Test : 712

Batch keys:
dict_keys(['case_id', 'patient_group_id', 'images', 'num_images', 'target_text'])
Batch size: 4
Images/case: [3, 8, 8, 8]
Image tensor shapes: [(3, 3, 224, 224), (8, 3, 224, 224), (8, 3, 224, 224), (8, 3, 224, 224)]
First target: VIÊM ỐNG TAI NGOÀI + MÀNG NHĨ T

Parameter summary:
ViT total: 86389248
ViT trainable: 86389248
mT5 trainable: 300176768
Projector trainable: 393728

V2-2 — PASS
ViT: TRAINABLE
mT5: TRAINABLE
Projection: TRAINABLE
Patient split: preserved
Train/Val/Test: 6138 / 756 / 712


In [24]:
# ============================================================
# V2-3 — FULL BACKWARD + MEMORY DRY RUN
# ============================================================

import gc
import torch
from torch.optim import AdamW

print("=" * 70)
print("V2-3 — FULL BACKWARD + MEMORY DRY RUN")
print("=" * 70)


# ------------------------------------------------------------
# 1. Modes
# ------------------------------------------------------------

vit_model.train()
visual_projector.train()
mt5_model.train()

for param in vit_model.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# 2. Optimizer — only for dry-run initialization
# ------------------------------------------------------------

optimizer = AdamW(
    [
        {
            "params": vit_model.parameters(),
            "lr": 1e-5,
        },
        {
            "params": visual_projector.parameters(),
            "lr": 1e-4,
        },
        {
            "params": mt5_model.parameters(),
            "lr": 5e-5,
        },
    ],
    weight_decay=0.01
)

print("\nLearning rates:")
print("  ViT       :", 1e-5)
print("  Projector :", 1e-4)
print("  mT5       :", 5e-5)


# ------------------------------------------------------------
# 3. Reset memory statistics
# ------------------------------------------------------------

torch.cuda.empty_cache()
gc.collect()

torch.cuda.reset_peak_memory_stats()

optimizer.zero_grad(
    set_to_none=True
)


# ------------------------------------------------------------
# 4. Get one batch
# ------------------------------------------------------------

batch = next(iter(train_loader))

images_list = batch["images"]
targets = batch["target_text"]

print("\nBatch:")
print("  Cases:", len(images_list))
print(
    "  Images/case:",
    [x.shape[0] for x in images_list]
)


# ------------------------------------------------------------
# 5. ViT forward
# ------------------------------------------------------------

visual_embeddings = []
attention_masks = []

for images in images_list:

    images = images.to(
        DEVICE,
        non_blocking=True
    )

    # IMPORTANT:
    # No torch.no_grad() here.
    # V2 needs gradients through ViT.

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16
    ):

        vit_output = vit_model(
            pixel_values=images
        )

        cls_embeddings = (
            vit_output.last_hidden_state[:, 0, :]
        )

        projected = visual_projector(
            cls_embeddings
        )

    visual_embeddings.append(
        projected
    )

    attention_masks.append(
        torch.ones(
            projected.shape[0],
            dtype=torch.long,
            device=DEVICE
        )
    )


# ------------------------------------------------------------
# 6. Pad visual tokens
# ------------------------------------------------------------

visual_prefix = torch.nn.utils.rnn.pad_sequence(
    visual_embeddings,
    batch_first=True
)

visual_attention_mask = torch.nn.utils.rnn.pad_sequence(
    attention_masks,
    batch_first=True,
    padding_value=0
)

print("\nVisual prefix:")
print(
    "  Shape:",
    tuple(visual_prefix.shape)
)


# ------------------------------------------------------------
# 7. Tokenize target
# ------------------------------------------------------------

tokenized = tokenizer(
    targets,
    padding=True,
    truncation=True,
    max_length=MAX_TARGET_LENGTH,
    return_tensors="pt"
)

labels = tokenized.input_ids.to(
    DEVICE
)

labels[
    labels == tokenizer.pad_token_id
] = -100

print(
    "Labels:",
    tuple(labels.shape)
)


# ------------------------------------------------------------
# 8. mT5 forward
# ------------------------------------------------------------

with torch.autocast(
    device_type="cuda",
    dtype=torch.bfloat16
):

    encoder_outputs = mt5_model.encoder(
        inputs_embeds=visual_prefix,
        attention_mask=visual_attention_mask,
        return_dict=True
    )

    outputs = mt5_model(
        encoder_outputs=encoder_outputs,
        attention_mask=visual_attention_mask,
        labels=labels,
        return_dict=True
    )

    loss = outputs.loss


print("\nForward:")
print(
    "  Loss:",
    loss.item()
)


# ------------------------------------------------------------
# 9. Backward
# ------------------------------------------------------------

loss.backward()

print("\nBackward:")
print("  Completed.")


# ------------------------------------------------------------
# 10. Gradient checks
# ------------------------------------------------------------

def grad_norm(model):

    total = 0.0

    for param in model.parameters():

        if param.grad is not None:

            total += (
                param.grad
                .detach()
                .float()
                .norm()
                .item()
            )

    return total


vit_grad = grad_norm(
    vit_model
)

projector_grad = grad_norm(
    visual_projector
)

mt5_grad = grad_norm(
    mt5_model
)

print("\nGradient norms:")
print(
    "  ViT       :",
    vit_grad
)

print(
    "  Projector :",
    projector_grad
)

print(
    "  mT5       :",
    mt5_grad
)


# ------------------------------------------------------------
# 11. Assertions
# ------------------------------------------------------------

assert torch.isfinite(loss)

assert vit_grad > 0, (
    "No gradient reached ViT."
)

assert projector_grad > 0, (
    "No gradient reached projector."
)

assert mt5_grad > 0, (
    "No gradient reached mT5."
)


# ------------------------------------------------------------
# 12. Memory
# ------------------------------------------------------------

torch.cuda.synchronize()

allocated = (
    torch.cuda.memory_allocated()
    / 1024**3
)

reserved = (
    torch.cuda.memory_reserved()
    / 1024**3
)

peak = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print("\nGPU memory:")
print(
    f"  Allocated: {allocated:.2f} GB"
)

print(
    f"  Reserved : {reserved:.2f} GB"
)

print(
    f"  Peak     : {peak:.2f} GB"
)


# ------------------------------------------------------------
# 13. Cleanup — DO NOT optimizer.step()
# ------------------------------------------------------------

optimizer.zero_grad(
    set_to_none=True
)

del outputs
del encoder_outputs
del visual_prefix
del visual_embeddings

gc.collect()
torch.cuda.empty_cache()


# ------------------------------------------------------------
# 14. Final
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("V2-3 — PASS")
print("=" * 70)

print("ViT backward       : PASS")
print("Projector backward : PASS")
print("mT5 backward       : PASS")
print("BF16               : PASS")
print(
    f"Peak memory        : {peak:.2f} GB"
)

print("=" * 70)

V2-3 — FULL BACKWARD + MEMORY DRY RUN

Learning rates:
  ViT       : 1e-05
  Projector : 0.0001
  mT5       : 5e-05

Batch:
  Cases: 4
  Images/case: [8, 8, 8, 8]

Visual prefix:
  Shape: (4, 8, 512)
Labels: (4, 38)

Forward:
  Loss: 26.479372024536133

Backward:
  Completed.

Gradient norms:
  ViT       : 75281.5946414955
  Projector : 8667.576690673828
  mT5       : 21914.59307101369

GPU memory:
  Allocated: 14.50 GB
  Reserved : 18.20 GB
  Peak     : 17.13 GB

V2-3 — PASS
ViT backward       : PASS
Projector backward : PASS
mT5 backward       : PASS
BF16               : PASS
Peak memory        : 17.13 GB


In [25]:
# ============================================================
# V2-4 — FULL TRAINING
# Resume-safe / Drive checkpoint
# ============================================================

import os
import gc
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup


print("=" * 70)
print("V2-4 — FULL TRAINING")
print("=" * 70)


# ------------------------------------------------------------
# 1. Training configuration
# ------------------------------------------------------------

NUM_EPOCHS = 5
GRAD_ACCUM_STEPS = 4

LR_VIT = 1e-5
LR_PROJECTOR = 1e-4
LR_MT5 = 5e-5

WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05

CHECKPOINT_DIR = os.path.join(
    OUTPUT_DIR,
    "checkpoints"
)

METRICS_DIR = os.path.join(
    OUTPUT_DIR,
    "metrics"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

os.makedirs(
    METRICS_DIR,
    exist_ok=True
)

LAST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "last.pt"
)

BEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best.pt"
)

HISTORY_FILE = os.path.join(
    METRICS_DIR,
    "training_history.csv"
)

TRAINING_CONFIG_FILE = os.path.join(
    OUTPUT_DIR,
    "training_config.json"
)


# ------------------------------------------------------------
# 2. Save training configuration
# ------------------------------------------------------------

training_config = {
    "experiment": "Baseline Model V2",
    "vision_model": VISION_MODEL_NAME,
    "text_model": TEXT_MODEL_NAME,

    "vision_trainable": True,
    "projector_trainable": True,
    "mt5_trainable": True,

    "num_epochs": NUM_EPOCHS,
    "gradient_accumulation": GRAD_ACCUM_STEPS,

    "batch_size": BATCH_SIZE,
    "effective_batch_size":
        BATCH_SIZE * GRAD_ACCUM_STEPS,

    "lr_vit": LR_VIT,
    "lr_projector": LR_PROJECTOR,
    "lr_mt5": LR_MT5,

    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,

    "max_images_per_case":
        MAX_IMAGES_PER_CASE,

    "image_size": IMAGE_SIZE,
    "max_target_length":
        MAX_TARGET_LENGTH,

    "bf16": True,

    "train_cases":
        len(train_dataset),

    "val_cases":
        len(val_dataset),

    "test_cases":
        len(test_dataset)
}

with open(
    TRAINING_CONFIG_FILE,
    "w"
) as f:

    json.dump(
        training_config,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nTraining config saved.")


# ------------------------------------------------------------
# 3. Make sure models are trainable
# ------------------------------------------------------------

vit_model.train()
visual_projector.train()
mt5_model.train()

for p in vit_model.parameters():
    p.requires_grad = True

for p in visual_projector.parameters():
    p.requires_grad = True

for p in mt5_model.parameters():
    p.requires_grad = True


# ------------------------------------------------------------
# 4. Optimizer
# ------------------------------------------------------------

optimizer = AdamW(
    [
        {
            "params": vit_model.parameters(),
            "lr": LR_VIT
        },
        {
            "params":
                visual_projector.parameters(),
            "lr": LR_PROJECTOR
        },
        {
            "params": mt5_model.parameters(),
            "lr": LR_MT5
        }
    ],
    weight_decay=WEIGHT_DECAY
)


# ------------------------------------------------------------
# 5. Scheduler
# ------------------------------------------------------------

steps_per_epoch = (
    len(train_loader)
    // GRAD_ACCUM_STEPS
)

if len(train_loader) % GRAD_ACCUM_STEPS != 0:
    steps_per_epoch += 1

total_optimizer_steps = (
    steps_per_epoch * NUM_EPOCHS
)

warmup_steps = int(
    total_optimizer_steps
    * WARMUP_RATIO
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_optimizer_steps
)

print("\nTraining steps:")
print(
    "  Batches/epoch:",
    len(train_loader)
)

print(
    "  Optimizer steps/epoch:",
    steps_per_epoch
)

print(
    "  Total optimizer steps:",
    total_optimizer_steps
)

print(
    "  Warmup steps:",
    warmup_steps
)


# ------------------------------------------------------------
# 6. BF16 autocast
# ------------------------------------------------------------

bf16_enabled = (
    DEVICE.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

print(
    "\nBF16:",
    bf16_enabled
)


# ------------------------------------------------------------
# 7. Visual prefix function
#
# IMPORTANT:
# ViT is NOT inside no_grad().
# Projector is also OUTSIDE no_grad().
# ------------------------------------------------------------

def build_visual_prefix_train(
    images_list
):

    visual_embeddings = []
    attention_masks = []

    for images in images_list:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=bf16_enabled
        ):

            vit_output = vit_model(
                pixel_values=images
            )

            cls_embeddings = (
                vit_output
                .last_hidden_state[:, 0, :]
            )

            projected = visual_projector(
                cls_embeddings
            )

        visual_embeddings.append(
            projected
        )

        attention_masks.append(
            torch.ones(
                projected.shape[0],
                dtype=torch.long,
                device=DEVICE
            )
        )

    visual_prefix = (
        torch.nn.utils.rnn.pad_sequence(
            visual_embeddings,
            batch_first=True
        )
    )

    visual_attention_mask = (
        torch.nn.utils.rnn.pad_sequence(
            attention_masks,
            batch_first=True,
            padding_value=0
        )
    )

    return (
        visual_prefix,
        visual_attention_mask
    )


# ------------------------------------------------------------
# 8. Training step
# ------------------------------------------------------------

def train_one_batch(batch):

    targets = batch["target_text"]

    visual_prefix, visual_attention_mask = (
        build_visual_prefix_train(
            batch["images"]
        )
    )

    tokenized = tokenizer(
        targets,
        padding=True,
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
        return_tensors="pt"
    )

    labels = tokenized.input_ids.to(
        DEVICE,
        non_blocking=True
    )

    labels[
        labels == tokenizer.pad_token_id
    ] = -100

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
        enabled=bf16_enabled
    ):

        encoder_outputs = (
            mt5_model.encoder(
                inputs_embeds=visual_prefix,
                attention_mask=
                    visual_attention_mask,
                return_dict=True
            )
        )

        outputs = mt5_model(
            encoder_outputs=
                encoder_outputs,
            attention_mask=
                visual_attention_mask,
            labels=labels,
            return_dict=True
        )

        loss = outputs.loss

    return loss


# ------------------------------------------------------------
# 9. Validation
# ------------------------------------------------------------

@torch.no_grad()
def validate():

    vit_model.eval()
    visual_projector.eval()
    mt5_model.eval()

    total_loss = 0.0
    count = 0

    for batch in val_loader:

        targets = batch["target_text"]

        visual_prefix, visual_attention_mask = (
            build_visual_prefix_train(
                batch["images"]
            )
        )

        tokenized = tokenizer(
            targets,
            padding=True,
            truncation=True,
            max_length=MAX_TARGET_LENGTH,
            return_tensors="pt"
        )

        labels = tokenized.input_ids.to(
            DEVICE,
            non_blocking=True
        )

        labels[
            labels == tokenizer.pad_token_id
        ] = -100

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=bf16_enabled
        ):

            encoder_outputs = (
                mt5_model.encoder(
                    inputs_embeds=
                        visual_prefix,
                    attention_mask=
                        visual_attention_mask,
                    return_dict=True
                )
            )

            outputs = mt5_model(
                encoder_outputs=
                    encoder_outputs,
                attention_mask=
                    visual_attention_mask,
                labels=labels,
                return_dict=True
            )

            loss = outputs.loss

        batch_size = len(targets)

        total_loss += (
            loss.item() * batch_size
        )

        count += batch_size

    vit_model.train()
    visual_projector.train()
    mt5_model.train()

    return total_loss / count


# ------------------------------------------------------------
# 10. Checkpoint helpers
# ------------------------------------------------------------

def get_rng_state():

    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        )
    }


def restore_rng_state(state):

    if state is None:
        return

    random.setstate(
        state["python"]
    )

    np.random.set_state(
        state["numpy"]
    )

    torch.set_rng_state(
        state["torch"]
    )

    if (
        torch.cuda.is_available()
        and state.get("cuda") is not None
    ):
        torch.cuda.set_rng_state_all(
            state["cuda"]
        )


def save_checkpoint(
    path,
    epoch,
    global_step,
    best_val_loss,
    history
):

    checkpoint = {

        "epoch": epoch,

        "global_step":
            global_step,

        "best_val_loss":
            best_val_loss,

        "vit_state_dict":
            vit_model.state_dict(),

        "projector_state_dict":
            visual_projector.state_dict(),

        "mt5_state_dict":
            mt5_model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "history":
            history,

        "rng_state":
            get_rng_state(),

        "config":
            training_config
    }

    torch.save(
        checkpoint,
        path
    )


# ------------------------------------------------------------
# 11. Resume if checkpoint exists
# ------------------------------------------------------------

start_epoch = 1
global_step = 0
best_val_loss = float("inf")
history = []

if os.path.exists(LAST_CHECKPOINT):

    print("\n" + "=" * 70)
    print("CHECKPOINT FOUND — RESUMING")
    print("=" * 70)

    checkpoint = torch.load(
        LAST_CHECKPOINT,
        map_location=DEVICE,
        weights_only=False
    )

    vit_model.load_state_dict(
        checkpoint["vit_state_dict"]
    )

    visual_projector.load_state_dict(
        checkpoint["projector_state_dict"]
    )

    mt5_model.load_state_dict(
        checkpoint["mt5_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    completed_epoch = checkpoint[
        "epoch"
    ]

    start_epoch = completed_epoch + 1

    global_step = checkpoint[
        "global_step"
    ]

    best_val_loss = checkpoint[
        "best_val_loss"
    ]

    history = checkpoint.get(
        "history",
        []
    )

    restore_rng_state(
        checkpoint.get(
            "rng_state"
        )
    )

    print(
        "Completed epoch:",
        completed_epoch
    )

    print(
        "Next epoch:",
        start_epoch
    )

    print(
        "Best val loss:",
        best_val_loss
    )

else:

    print(
        "\nNo checkpoint found."
    )

    print(
        "Starting V2 from epoch 1."
    )


# ------------------------------------------------------------
# 12. Training loop
# ------------------------------------------------------------

for epoch in range(
    start_epoch,
    NUM_EPOCHS + 1
):

    print("\n")
    print("=" * 70)
    print(
        f"V2 — EPOCH {epoch}/{NUM_EPOCHS}"
    )
    print("=" * 70)

    vit_model.train()
    visual_projector.train()
    mt5_model.train()

    optimizer.zero_grad(
        set_to_none=True
    )

    running_loss = 0.0
    batch_count = 0

    epoch_start = __import__(
        "time"
    ).time()

    for batch_idx, batch in enumerate(
        train_loader,
        start=1
    ):

        loss = train_one_batch(
            batch
        )

        loss_for_backward = (
            loss / GRAD_ACCUM_STEPS
        )

        loss_for_backward.backward()

        running_loss += loss.item()
        batch_count += 1

        should_step = (
            batch_idx % GRAD_ACCUM_STEPS == 0
            or batch_idx == len(train_loader)
        )

        if should_step:

            torch.nn.utils.clip_grad_norm_(
                list(vit_model.parameters())
                + list(visual_projector.parameters())
                + list(mt5_model.parameters()),
                max_norm=1.0
            )

            optimizer.step()
            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )

            global_step += 1

        if (
            batch_idx == 1
            or batch_idx % 100 == 0
            or batch_idx == len(train_loader)
        ):

            avg_loss = (
                running_loss
                / batch_count
            )

            print(
                f"Batch {batch_idx:4d}/"
                f"{len(train_loader)}"
                f" | loss {loss.item():.4f}"
                f" | avg {avg_loss:.4f}"
                f" | step {global_step}"
            )

    train_loss = (
        running_loss
        / batch_count
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    val_loss = validate()

    epoch_time = (
        __import__("time").time()
        - epoch_start
    )

    current_lr = (
        scheduler.get_last_lr()
    )

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "global_step": global_step,
        "lr_vit": current_lr[0],
        "lr_projector": current_lr[1],
        "lr_mt5": current_lr[2],
        "epoch_time_sec": epoch_time
    }

    history.append(
        epoch_record
    )

    # --------------------------------------------------------
    # Save history
    # --------------------------------------------------------

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_FILE,
        index=False
    )

    # --------------------------------------------------------
    # Save last checkpoint
    # --------------------------------------------------------

    save_checkpoint(
        LAST_CHECKPOINT,
        epoch,
        global_step,
        best_val_loss,
        history
    )

    # --------------------------------------------------------
    # Save best
    # --------------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        save_checkpoint(
            BEST_CHECKPOINT,
            epoch,
            global_step,
            best_val_loss,
            history
        )

        best_marker = " ★ BEST"

    else:

        best_marker = ""

    # --------------------------------------------------------
    # Epoch summary
    # --------------------------------------------------------

    print("\n" + "-" * 70)

    print(
        f"Epoch {epoch} complete"
    )

    print(
        f"Train loss: {train_loss:.6f}"
    )

    print(
        f"Val loss  : {val_loss:.6f}"
    )

    print(
        f"Best val  : {best_val_loss:.6f}"
        f"{best_marker}"
    )

    print(
        f"Time      : "
        f"{epoch_time / 60:.2f} min"
    )

    print(
        "Peak VRAM : "
        f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
    )

    print("-" * 70)

    # Reset peak stats for next epoch
    torch.cuda.reset_peak_memory_stats()


# ------------------------------------------------------------
# 13. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("V2-4 — TRAINING COMPLETE")
print("=" * 70)

print(
    "Best validation loss:",
    best_val_loss
)

print(
    "Last checkpoint:",
    LAST_CHECKPOINT
)

print(
    "Best checkpoint:",
    BEST_CHECKPOINT
)

print(
    "History:",
    HISTORY_FILE
)

print("=" * 70)

V2-4 — FULL TRAINING

Training config saved.

Training steps:
  Batches/epoch: 1535
  Optimizer steps/epoch: 384
  Total optimizer steps: 1920
  Warmup steps: 96

BF16: True

No checkpoint found.
Starting V2 from epoch 1.


V2 — EPOCH 1/5
Batch    1/1535 | loss 49.0636 | avg 49.0636 | step 0
Batch  100/1535 | loss 35.4908 | avg 30.9704 | step 25
Batch  200/1535 | loss 19.1757 | avg 29.1349 | step 50
Batch  300/1535 | loss 16.2625 | avg 26.6503 | step 75
Batch  400/1535 | loss 22.2393 | avg 25.3767 | step 100
Batch  500/1535 | loss 25.0847 | avg 24.4563 | step 125
Batch  600/1535 | loss 12.5547 | avg 23.3089 | step 150
Batch  700/1535 | loss 15.0741 | avg 21.9500 | step 175
Batch  800/1535 | loss 12.5472 | avg 20.7016 | step 200
Batch  900/1535 | loss 7.6019 | avg 19.3760 | step 225
Batch 1000/1535 | loss 5.3792 | avg 18.0647 | step 250
Batch 1100/1535 | loss 4.4803 | avg 16.8869 | step 275
Batch 1200/1535 | loss 2.7653 | avg 15.8277 | step 300
Batch 1300/1535 | loss 1.3364 | avg 14.864

In [5]:
# ============================================================
# V2-5 CONTINUE v3 — FORCE BF16 MODEL INFERENCE
# ============================================================

import gc
import torch
import pandas as pd
from PIL import Image
from torchvision import transforms

print("=" * 70)
print("V2-5 CONTINUE v3 — FORCE BF16 INFERENCE")
print("=" * 70)


# ------------------------------------------------------------
# 1. Convert all inference models to BF16
# ------------------------------------------------------------

vit_model = vit_model.to(
    device=DEVICE,
    dtype=torch.bfloat16
)

visual_projector = visual_projector.to(
    device=DEVICE,
    dtype=torch.bfloat16
)

mt5_model = mt5_model.to(
    device=DEVICE,
    dtype=torch.bfloat16
)

vit_model.eval()
visual_projector.eval()
mt5_model.eval()


print("Model dtypes:")

print(
    "  ViT:",
    next(vit_model.parameters()).dtype
)

print(
    "  Projector:",
    next(
        visual_projector.parameters()
    ).dtype
)

print(
    "  mT5:",
    next(mt5_model.parameters()).dtype
)


# ------------------------------------------------------------
# 2. Image transform
# ------------------------------------------------------------

image_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ------------------------------------------------------------
# 3. Visual prefix
# ------------------------------------------------------------

@torch.no_grad()
def build_visual_prefix_v3(
    image_paths
):

    image_paths = sorted(
        image_paths
    )[:MAX_IMAGES_PER_CASE]

    tensors = []

    for path in image_paths:

        image = Image.open(
            path
        ).convert("RGB")

        tensors.append(
            image_transform(image)
        )

    # Start as FP32
    images = torch.stack(
        tensors,
        dim=0
    )

    # Explicitly convert input to BF16
    images = images.to(
        device=DEVICE,
        dtype=torch.bfloat16,
        non_blocking=True
    )

    with torch.no_grad():

        vit_output = vit_model(
            pixel_values=images
        )

        cls_embeddings = (
            vit_output
            .last_hidden_state[:, 0, :]
        )

        projected = visual_projector(
            cls_embeddings
        )

    visual_prefix = (
        projected.unsqueeze(0)
    )

    visual_attention_mask = torch.ones(
        1,
        projected.shape[0],
        dtype=torch.long,
        device=DEVICE
    )

    return (
        visual_prefix,
        visual_attention_mask
    )


# ------------------------------------------------------------
# 4. Generation
# ------------------------------------------------------------

@torch.no_grad()
def generate_one_v3(
    image_paths
):

    (
        visual_prefix,
        visual_attention_mask
    ) = build_visual_prefix_v3(
        image_paths
    )

    encoder_outputs = (
        mt5_model.encoder(
            inputs_embeds=
                visual_prefix,
            attention_mask=
                visual_attention_mask,
            return_dict=True
        )
    )

    generated_ids = (
        mt5_model.generate(
            encoder_outputs=
                encoder_outputs,
            attention_mask=
                visual_attention_mask,

            max_new_tokens=
                MAX_TARGET_LENGTH,

            num_beams=4,

            early_stopping=True,

            no_repeat_ngram_size=2
        )
    )

    return tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    ).strip()


# ------------------------------------------------------------
# 5. SINGLE-CASE TEST
# ------------------------------------------------------------

print("\nRunning single-case test...")

first_case = test_cases.iloc[0]

first_case_id = first_case[
    "case_id"
]

first_images = case_to_images[
    first_case_id
]

print(
    "Case:",
    first_case_id
)

print(
    "Available images:",
    len(first_images)
)

print(
    "Images used:",
    min(
        len(first_images),
        MAX_IMAGES_PER_CASE
    )
)

test_prediction = generate_one_v3(
    first_images
)

print(
    "\nGround truth:"
)

print(
    first_case["ket_luan"]
)

print(
    "\nPrediction:"
)

print(
    test_prediction
)

print(
    "\nSingle-case generation: PASS"
)


# ------------------------------------------------------------
# 6. Generate entire test set
# ------------------------------------------------------------

results = []

print("\n" + "=" * 70)
print("GENERATING ALL TEST PREDICTIONS")
print("=" * 70)

for idx, row in enumerate(
    test_cases.itertuples(index=False),
    start=1
):

    case_id = row.case_id

    patient_group_id = (
        row.patient_group_id
    )

    target = row.ket_luan

    image_paths = case_to_images[
        case_id
    ]

    prediction = generate_one_v3(
        image_paths
    )

    results.append({

        "case_id":
            case_id,

        "patient_group_id":
            patient_group_id,

        "target":
            target,

        "prediction":
            prediction,

        "num_images_available":
            len(image_paths),

        "num_images_used":
            min(
                len(image_paths),
                MAX_IMAGES_PER_CASE
            )
    })

    if (
        idx <= 10
        or idx % 50 == 0
        or idx == len(test_cases)
    ):

        print(
            f"[{idx:4d}/{len(test_cases)}]"
            f" | GT: {target[:50]}"
            f" | PRED: {prediction[:50]}"
        )


# ------------------------------------------------------------
# 7. Save
# ------------------------------------------------------------

predictions_df = pd.DataFrame(
    results
)

predictions_df.to_csv(
    PREDICTION_FILE,
    index=False
)


# ------------------------------------------------------------
# 8. Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("V2-5 — COMPLETE")
print("=" * 70)

print(
    "Rows:",
    len(predictions_df)
)

print(
    "Unique predictions:",
    predictions_df[
        "prediction"
    ].nunique()
)

print(
    "Saved:",
    PREDICTION_FILE
)

print("=" * 70)


# ------------------------------------------------------------
# 9. Cleanup
# ------------------------------------------------------------

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

V2-5 CONTINUE v3 — FORCE BF16 INFERENCE
Model dtypes:
  ViT: torch.bfloat16
  Projector: torch.bfloat16
  mT5: torch.bfloat16

Running single-case test...
Case: 10017.10017.0.10031
Available images: 16
Images used: 8

Ground truth:
VIÊM MÀNG NHĨ CẤP BÓNG NƯỚC P - NHIỀU RÁY TAI T

Prediction:
<extra_id_0>onaleciationciationstãocetifyteriszódδώ <extra_id_32>

Single-case generation: PASS

GENERATING ALL TEST PREDICTIONS
[   1/712] | GT: VIÊM MÀNG NHĨ CẤP BÓNG NƯỚC P - NHIỀU RÁY TAI T | PRED: <extra_id_0>onaleciationciationstãocetifyteriszódδ
[   2/712] | GT: VIÊM ỐNG TAI NGOÀI (T) MẠN / VIÊM MŨI ĐỢT CẤP. | PRED: <extra_id_0>dientscerttaneous flexbox ктивті <extr
[   3/712] | GT: VIÊM ỐNG TAI NGOÀI MÀNG NHĨ P CẤP | PRED: <extra_id_0>onaleciationciationγόρ <extra_id_51> <
[   4/712] | GT: VIÊM MŨI MẠN VIÊM ỐNG TAI NGOÀI + MÀNG NHĨ 2 BÊN M | PRED: <extra_id_0>ционноurator.uratorbuyu
[   5/712] | GT: VIÊM A-MI-ĐAN. | PRED: <extra_id_0>ктивті angler-dèe  бет πεδ真空tայլρόπο <
[   6/712] | GT: V

In [6]:
# ============================================================
# V2-5 v4 — PURE FP32 INFERENCE
# No autocast / no BF16
# ============================================================

import gc
import torch
import pandas as pd

print("=" * 70)
print("V2-5 v4 — PURE FP32 INFERENCE")
print("=" * 70)


# ------------------------------------------------------------
# 1. Restore models to FP32
# ------------------------------------------------------------

vit_model = vit_model.float()
visual_projector = visual_projector.float()
mt5_model = mt5_model.float()

vit_model.eval()
visual_projector.eval()
mt5_model.eval()

print("Model dtypes:")

print(
    "  ViT:",
    next(vit_model.parameters()).dtype
)

print(
    "  Projector:",
    next(
        visual_projector.parameters()
    ).dtype
)

print(
    "  mT5:",
    next(mt5_model.parameters()).dtype
)


# ------------------------------------------------------------
# 2. Pure FP32 visual prefix
# ------------------------------------------------------------

@torch.no_grad()
def build_visual_prefix_fp32(
    image_paths
):

    image_paths = sorted(
        image_paths
    )[:MAX_IMAGES_PER_CASE]

    tensors = []

    for path in image_paths:

        image = Image.open(
            path
        ).convert("RGB")

        tensors.append(
            image_transform(image).float()
        )

    images = torch.stack(
        tensors,
        dim=0
    ).to(
        DEVICE,
        dtype=torch.float32,
        non_blocking=True
    )

    # NO autocast
    vit_output = vit_model(
        pixel_values=images
    )

    cls_embeddings = (
        vit_output
        .last_hidden_state[:, 0, :]
    )

    projected = visual_projector(
        cls_embeddings
    )

    visual_prefix = (
        projected.unsqueeze(0)
    )

    visual_attention_mask = torch.ones(
        1,
        projected.shape[0],
        dtype=torch.long,
        device=DEVICE
    )

    return (
        visual_prefix,
        visual_attention_mask
    )


# ------------------------------------------------------------
# 3. Pure FP32 generation
# ------------------------------------------------------------

@torch.no_grad()
def generate_one_fp32(
    image_paths
):

    (
        visual_prefix,
        visual_attention_mask
    ) = build_visual_prefix_fp32(
        image_paths
    )

    # NO autocast
    encoder_outputs = (
        mt5_model.encoder(
            inputs_embeds=
                visual_prefix,
            attention_mask=
                visual_attention_mask,
            return_dict=True
        )
    )

    generated_ids = (
        mt5_model.generate(
            encoder_outputs=
                encoder_outputs,

            attention_mask=
                visual_attention_mask,

            max_new_tokens=
                MAX_TARGET_LENGTH,

            num_beams=4,

            early_stopping=True,

            no_repeat_ngram_size=2
        )
    )

    return tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    ).strip()


# ------------------------------------------------------------
# 4. SINGLE CASE TEST
# ------------------------------------------------------------

print("\nRunning FP32 single-case test...")

first_case = test_cases.iloc[0]

first_case_id = first_case[
    "case_id"
]

first_images = case_to_images[
    first_case_id
]

print(
    "Case:",
    first_case_id
)

print(
    "Ground truth:",
    first_case["ket_luan"]
)

prediction_fp32 = generate_one_fp32(
    first_images
)

print(
    "Prediction:",
    prediction_fp32
)

print(
    "\nFP32 generation: PASS"
)


# ------------------------------------------------------------
# 5. Generate all test cases
# ------------------------------------------------------------

results_fp32 = []

print("\n" + "=" * 70)
print("GENERATING ALL TEST PREDICTIONS — FP32")
print("=" * 70)

for idx, row in enumerate(
    test_cases.itertuples(index=False),
    start=1
):

    case_id = row.case_id

    patient_group_id = (
        row.patient_group_id
    )

    target = row.ket_luan

    image_paths = case_to_images[
        case_id
    ]

    prediction = generate_one_fp32(
        image_paths
    )

    results_fp32.append({

        "case_id":
            case_id,

        "patient_group_id":
            patient_group_id,

        "target":
            target,

        "prediction":
            prediction,

        "num_images_available":
            len(image_paths),

        "num_images_used":
            min(
                len(image_paths),
                MAX_IMAGES_PER_CASE
            )
    })

    if (
        idx <= 10
        or idx % 50 == 0
        or idx == len(test_cases)
    ):

        print(
            f"[{idx:4d}/{len(test_cases)}]"
            f" | GT: {target[:50]}"
            f" | PRED: {prediction[:80]}"
        )


# ------------------------------------------------------------
# 6. Save separately
# ------------------------------------------------------------

PREDICTION_FILE_FP32 = os.path.join(
    OUTPUT_DIR,
    "test_predictions_fp32.csv"
)

predictions_fp32_df = pd.DataFrame(
    results_fp32
)

predictions_fp32_df.to_csv(
    PREDICTION_FILE_FP32,
    index=False
)


# ------------------------------------------------------------
# 7. Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("V2-5 FP32 — COMPLETE")
print("=" * 70)

print(
    "Rows:",
    len(predictions_fp32_df)
)

print(
    "Unique predictions:",
    predictions_fp32_df[
        "prediction"
    ].nunique()
)

print(
    "Saved:",
    PREDICTION_FILE_FP32
)

print("=" * 70)


# ------------------------------------------------------------
# 8. Cleanup
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

V2-5 v4 — PURE FP32 INFERENCE
Model dtypes:
  ViT: torch.float32
  Projector: torch.float32
  mT5: torch.float32

Running FP32 single-case test...
Case: 10017.10017.0.10031
Ground truth: VIÊM MÀNG NHĨ CẤP BÓNG NƯỚC P - NHIỀU RÁY TAI T
Prediction: <extra_id_0>onaleciationciationstãocetifyteriszódδώ <extra_id_32>

FP32 generation: PASS

GENERATING ALL TEST PREDICTIONS — FP32
[   1/712] | GT: VIÊM MÀNG NHĨ CẤP BÓNG NƯỚC P - NHIỀU RÁY TAI T | PRED: <extra_id_0>onaleciationciationstãocetifyteriszódδώ <extra_id_32>
[   2/712] | GT: VIÊM ỐNG TAI NGOÀI (T) MẠN / VIÊM MŨI ĐỢT CẤP. | PRED: <extra_id_0>dientscerttaneous flexbox ктивті <extra_id_12>тимеnicznych铢篪- . <ext
[   3/712] | GT: VIÊM ỐNG TAI NGOÀI MÀNG NHĨ P CẤP | PRED: <extra_id_0>;">缡,cartPиaceeeetilis)(ранківськ▍ánicoánico.缡licitud့္ufbtfs.urator
[   4/712] | GT: VIÊM MŨI MẠN VIÊM ỐNG TAI NGOÀI + MÀNG NHĨ 2 BÊN M | PRED: <extra_id_0>ционноurator.uratorbuyu
[   5/712] | GT: VIÊM A-MI-ĐAN. | PRED: <extra_id_0>ктивті angler-dèe  бет πεδ真空

In [8]:
# ============================================================
# V2-DIAGNOSTIC-1A — INSPECT CHECKPOINT STRUCTURE
# ============================================================

import os
import torch

PROJECT_DIR = "/content/drive/MyDrive/NoiSoi_Matching"

V1_CKPT = os.path.join(
    PROJECT_DIR, "baseline_model_v1", "checkpoints", "best.pt"
)

V2_CKPT = os.path.join(
    PROJECT_DIR, "baseline_model_v2", "checkpoints", "best.pt"
)

print("=" * 70)
print("V2-DIAGNOSTIC-1A — CHECKPOINT STRUCTURE")
print("=" * 70)

for name, path in [
    ("V1", V1_CKPT),
    ("V2", V2_CKPT),
]:
    print(f"\n{'-' * 70}")
    print(name, path)
    print("Exists:", os.path.exists(path))

    ckpt = torch.load(path, map_location="cpu", weights_only=False)

    print("\nTop-level keys:")
    for k in ckpt.keys():
        v = ckpt[k]

        if isinstance(v, dict):
            print(f"  {k}: dict ({len(v)} keys)")

            # Show first few nested keys
            nested = list(v.keys())[:10]
            for nk in nested:
                print(f"      {nk}")

        else:
            print(f"  {k}: {type(v).__name__}")

    # Search for mT5-related keys at top level
    print("\nPossible model/state keys:")
    for k in ckpt.keys():
        if any(x in k.lower() for x in ["mt5", "text", "model", "state", "decoder"]):
            print(" ", k)

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

V2-DIAGNOSTIC-1A — CHECKPOINT STRUCTURE

----------------------------------------------------------------------
V1 /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v1/checkpoints/best.pt
Exists: True

Top-level keys:
  epoch: int
  global_step: int
  best_val_loss: float
  visual_projector: dict (2 keys)
      proj.weight
      proj.bias
  mt5_model: dict (192 keys)
      shared.weight
      encoder.embed_tokens.weight
      encoder.block.0.layer.0.SelfAttention.q.weight
      encoder.block.0.layer.0.SelfAttention.k.weight
      encoder.block.0.layer.0.SelfAttention.v.weight
      encoder.block.0.layer.0.SelfAttention.o.weight
      encoder.block.0.layer.0.SelfAttention.relative_attention_bias.weight
      encoder.block.0.layer.0.layer_norm.weight
      encoder.block.0.layer.1.DenseReluDense.wi_0.weight
      encoder.block.0.layer.1.DenseReluDense.wi_1.weight
  optimizer: dict (2 keys)
      state
      param_groups
  scheduler: dict (7 keys)
      base_lrs
      last_epoch
      

In [9]:
# ============================================================
# V2-DIAGNOSTIC-1B
# Compare pretrained vs V1 vs V2 mT5
# No training / no checkpoint modification
# ============================================================

import os
import torch
import numpy as np

from transformers import MT5ForConditionalGeneration

PROJECT_DIR = "/content/drive/MyDrive/NoiSoi_Matching"

V1_CKPT = os.path.join(
    PROJECT_DIR, "baseline_model_v1", "checkpoints", "best.pt"
)

V2_CKPT = os.path.join(
    PROJECT_DIR, "baseline_model_v2", "checkpoints", "best.pt"
)

MODEL_NAME = "google/mt5-small"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("V2-DIAGNOSTIC-1B — mT5 WEIGHT / GENERATION ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load checkpoints
# ------------------------------------------------------------

print("\nLoading checkpoints...")

v1_ckpt = torch.load(
    V1_CKPT,
    map_location="cpu",
    weights_only=False
)

v2_ckpt = torch.load(
    V2_CKPT,
    map_location="cpu",
    weights_only=False
)

v1_state = v1_ckpt["mt5_model"]
v2_state = v2_ckpt["mt5_state_dict"]

print("V1 mT5 parameters:", len(v1_state))
print("V2 mT5 parameters:", len(v2_state))

# ------------------------------------------------------------
# 2. Load pretrained model
# ------------------------------------------------------------

print("\nLoading pretrained mT5...")

base_mt5 = MT5ForConditionalGeneration.from_pretrained(
    MODEL_NAME
)

print("Loaded.")

# ------------------------------------------------------------
# 3. Basic config
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("CONFIG")
print("-" * 70)

print("model_type:", base_mt5.config.model_type)
print("d_model:", base_mt5.config.d_model)
print("vocab_size:", base_mt5.config.vocab_size)
print("pad_token_id:", base_mt5.config.pad_token_id)
print("eos_token_id:", base_mt5.config.eos_token_id)
print("decoder_start_token_id:", base_mt5.config.decoder_start_token_id)
print("tie_word_embeddings:", base_mt5.config.tie_word_embeddings)

# ------------------------------------------------------------
# 4. Weight comparison helper
# ------------------------------------------------------------

def tensor_stats(name, x):
    x = x.float()
    return {
        "name": name,
        "mean": x.mean().item(),
        "std": x.std().item(),
        "abs_mean": x.abs().mean().item(),
        "min": x.min().item(),
        "max": x.max().item(),
    }


def compare_tensors(a, b):
    a = a.float()
    b = b.float()

    diff = (a - b).abs()

    return {
        "mean_abs_diff": diff.mean().item(),
        "max_abs_diff": diff.max().item(),
        "relative_mean_diff": (
            diff.mean() / (a.abs().mean() + 1e-12)
        ).item(),
    }


# ------------------------------------------------------------
# 5. shared.weight vs lm_head.weight
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("SHARED vs LM_HEAD")
print("-" * 70)

base_shared = base_mt5.shared.weight.detach().cpu()
base_lm = base_mt5.lm_head.weight.detach().cpu()

v1_shared = v1_state["shared.weight"].cpu()
v1_lm = v1_state["lm_head.weight"].cpu()

v2_shared = v2_state["shared.weight"].cpu()
v2_lm = v2_state["lm_head.weight"].cpu()

for label, shared, lm in [
    ("PRETRAINED", base_shared, base_lm),
    ("V1", v1_shared, v1_lm),
    ("V2", v2_shared, v2_lm),
]:
    stats = compare_tensors(shared, lm)

    print(f"\n{label}")
    print(f"  shared mean/std : "
          f"{shared.float().mean():.6f} / {shared.float().std():.6f}")
    print(f"  lm_head mean/std: "
          f"{lm.float().mean():.6f} / {lm.float().std():.6f}")
    print(f"  shared↔lm mean abs diff: "
          f"{stats['mean_abs_diff']:.8f}")
    print(f"  shared↔lm max abs diff : "
          f"{stats['max_abs_diff']:.8f}")
    print(f"  relative diff          : "
          f"{stats['relative_mean_diff']:.6f}")

# ------------------------------------------------------------
# 6. Drift from pretrained
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("DRIFT FROM PRETRAINED")
print("-" * 70)

for name, base, v1, v2 in [
    ("shared.weight", base_shared, v1_shared, v2_shared),
    ("lm_head.weight", base_lm, v1_lm, v2_lm),
]:
    d1 = compare_tensors(base, v1)
    d2 = compare_tensors(base, v2)

    print(f"\n{name}")

    print("  V1:")
    print(f"    mean abs diff = {d1['mean_abs_diff']:.8f}")
    print(f"    relative     = {d1['relative_mean_diff']:.6f}")

    print("  V2:")
    print(f"    mean abs diff = {d2['mean_abs_diff']:.8f}")
    print(f"    relative     = {d2['relative_mean_diff']:.6f}")

# ------------------------------------------------------------
# 7. Load V1/V2 models
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("LOADING V1 / V2 mT5")
print("-" * 70)

v1_mt5 = MT5ForConditionalGeneration.from_pretrained(
    MODEL_NAME
)

v2_mt5 = MT5ForConditionalGeneration.from_pretrained(
    MODEL_NAME
)

v1_mt5.load_state_dict(v1_state, strict=True)
v2_mt5.load_state_dict(v2_state, strict=True)

v1_mt5.eval()
v2_mt5.eval()

print("V1 checkpoint loaded: PASS")
print("V2 checkpoint loaded: PASS")

# ------------------------------------------------------------
# 8. Generation configuration
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("GENERATION CONFIG")
print("-" * 70)

for name, model in [
    ("PRETRAINED", base_mt5),
    ("V1", v1_mt5),
    ("V2", v2_mt5),
]:
    gc = model.generation_config

    print(f"\n{name}")
    print("  decoder_start_token_id:",
          gc.decoder_start_token_id)
    print("  pad_token_id:",
          gc.pad_token_id)
    print("  eos_token_id:",
          gc.eos_token_id)
    print("  bos_token_id:",
          getattr(gc, "bos_token_id", None))

# ------------------------------------------------------------
# 9. First-token distribution with dummy encoder input
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("FIRST-TOKEN DISTRIBUTION")
print("-" * 70)

base_mt5 = base_mt5.to(device)
v1_mt5 = v1_mt5.to(device)
v2_mt5 = v2_mt5.to(device)

# mT5 encoder output dimension
d_model = base_mt5.config.d_model

# Dummy encoder representation
encoder_hidden = torch.zeros(
    1,
    8,
    d_model,
    device=device
)

encoder_mask = torch.ones(
    1,
    8,
    dtype=torch.long,
    device=device
)

decoder_input_ids = torch.tensor(
    [[base_mt5.config.decoder_start_token_id]],
    dtype=torch.long,
    device=device
)


@torch.no_grad()
def first_token_info(model):
    outputs = model(
        encoder_outputs=(encoder_hidden,),
        attention_mask=encoder_mask,
        decoder_input_ids=decoder_input_ids,
        return_dict=True,
    )

    logits = outputs.logits[:, -1, :]
    probs = torch.softmax(logits.float(), dim=-1)

    top_probs, top_ids = torch.topk(probs, k=20, dim=-1)

    return [
        (int(tok), float(prob))
        for tok, prob in zip(
            top_ids[0].cpu(),
            top_probs[0].cpu()
        )
    ]


for name, model in [
    ("PRETRAINED", base_mt5),
    ("V1", v1_mt5),
    ("V2", v2_mt5),
]:

    top = first_token_info(model)

    print(f"\n{name} top-20 first tokens:")

    for rank, (tok, prob) in enumerate(top, 1):
        print(
            f"  {rank:2d}. token_id={tok:6d} "
            f"prob={prob:.6f}"
        )

print("\n" + "=" * 70)
print("DIAGNOSTIC COMPLETE")
print("=" * 70)

V2-DIAGNOSTIC-1B — mT5 WEIGHT / GENERATION ANALYSIS

Loading checkpoints...
V1 mT5 parameters: 192
V2 mT5 parameters: 192

Loading pretrained mT5...


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded.

----------------------------------------------------------------------
CONFIG
----------------------------------------------------------------------
model_type: mt5
d_model: 512
vocab_size: 250112
pad_token_id: 0
eos_token_id: 1
decoder_start_token_id: 0
tie_word_embeddings: True

----------------------------------------------------------------------
SHARED vs LM_HEAD
----------------------------------------------------------------------

PRETRAINED
  shared mean/std : 0.077423 / 14.437888
  lm_head mean/std: 0.004152 / 0.647459
  shared↔lm mean abs diff: 11.46798897
  shared↔lm max abs diff : 113.70703125
  relative diff          : 1.000087

V1
  shared mean/std : 0.077386 / 14.430928
  lm_head mean/std: 0.004255 / 0.647202
  shared↔lm mean abs diff: 11.46251583
  shared↔lm max abs diff : 113.65099335
  relative diff          : 1.000092

V2
  shared mean/std : 0.077385 / 14.430861
  lm_head mean/std: 0.004244 / 0.647193
  shared↔lm mean abs diff: 11.46245670
  shared↔lm max a

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


V1 checkpoint loaded: PASS
V2 checkpoint loaded: PASS

----------------------------------------------------------------------
GENERATION CONFIG
----------------------------------------------------------------------

PRETRAINED
  decoder_start_token_id: 0
  pad_token_id: 0
  eos_token_id: 1
  bos_token_id: None

V1
  decoder_start_token_id: 0
  pad_token_id: 0
  eos_token_id: 1
  bos_token_id: None

V2
  decoder_start_token_id: 0
  pad_token_id: 0
  eos_token_id: 1
  bos_token_id: None

----------------------------------------------------------------------
FIRST-TOKEN DISTRIBUTION
----------------------------------------------------------------------

PRETRAINED top-20 first tokens:
   1. token_id=250099 prob=0.999985
   2. token_id= 10761 prob=0.000000
   3. token_id=  1096 prob=0.000000
   4. token_id=176570 prob=0.000000
   5. token_id= 24079 prob=0.000000
   6. token_id= 45188 prob=0.000000
   7. token_id=   969 prob=0.000000
   8. token_id= 95266 prob=0.000000
   9. token_id=   423

In [12]:
# ============================================================
# V2-DIAGNOSTIC-2
# REAL IMAGE -> VISUAL PREFIX -> mT5 DECODER
#
# Compare V1 vs V2 on the EXACT SAME TEST CASE
#
# NO TRAINING
# NO CHECKPOINT MODIFICATION
# ============================================================

import os
import torch
import torch.nn as nn
import pandas as pd

from PIL import Image
from torchvision import transforms

from transformers import (
    ViTModel,
    MT5ForConditionalGeneration,
    AutoTokenizer,
)

print("=" * 70)
print("V2-DIAGNOSTIC-2 — REAL VISUAL PREFIX ANALYSIS")
print("=" * 70)


# ============================================================
# 1. CONFIG
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/NoiSoi_Matching"

V1_CKPT = os.path.join(
    PROJECT_DIR,
    "baseline_model_v1",
    "checkpoints",
    "best.pt"
)

V2_CKPT = os.path.join(
    PROJECT_DIR,
    "baseline_model_v2",
    "checkpoints",
    "best.pt"
)

MANIFEST_FILE = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

CASE_MANIFEST_FILE = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_case_manifest.csv"
)

SPLIT_FILE = os.path.join(
    PROJECT_DIR,
    "patient_split",
    "cases_with_split.csv"
)

VISION_MODEL_NAME = "google/vit-base-patch16-224"
TEXT_MODEL_NAME = "google/mt5-small"

IMAGE_SIZE = 224
MAX_IMAGES = 8
MAX_TARGET_LENGTH = 96

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# 2. CHECK FILES
# ============================================================

print("\n" + "-" * 70)
print("CHECKING FILES")
print("-" * 70)

required_files = {
    "V1 checkpoint": V1_CKPT,
    "V2 checkpoint": V2_CKPT,
    "image manifest": MANIFEST_FILE,
    "case manifest": CASE_MANIFEST_FILE,
    "split file": SPLIT_FILE,
}

for name, path in required_files.items():

    exists = os.path.exists(path)

    print(
        f"{name:20s}: "
        f"{'PASS' if exists else 'MISSING'}"
    )

    if not exists:
        raise FileNotFoundError(path)


# ============================================================
# 3. LOAD DATA
# ============================================================

print("\n" + "-" * 70)
print("LOADING DATA")
print("-" * 70)

manifest = pd.read_csv(
    MANIFEST_FILE,
    low_memory=False
)

case_manifest = pd.read_csv(
    CASE_MANIFEST_FILE,
    low_memory=False
)

split_df = pd.read_csv(
    SPLIT_FILE,
    low_memory=False
)

print("Manifest rows:", len(manifest))
print("Case manifest rows:", len(case_manifest))
print("Split rows:", len(split_df))


# ------------------------------------------------------------
# Keep NORMAL images only
# ------------------------------------------------------------

manifest["image_status"] = (
    manifest["image_status"]
    .fillna("")
    .astype(str)
    .str.upper()
)

manifest = manifest[
    manifest["image_status"] == "NORMAL"
].copy()

print("NORMAL images:", len(manifest))


# ------------------------------------------------------------
# Add split if necessary
# ------------------------------------------------------------

if "split" not in case_manifest.columns:

    case_manifest = case_manifest.merge(
        split_df[["case_id", "split"]],
        on="case_id",
        how="left"
    )


# ------------------------------------------------------------
# Clean target
# ------------------------------------------------------------

case_manifest["ket_luan"] = (
    case_manifest["ket_luan"]
    .fillna("")
    .astype(str)
    .str.strip()
)

case_manifest = case_manifest[
    case_manifest["ket_luan"] != ""
].copy()

print(
    "Cases with non-empty target:",
    len(case_manifest)
)


# ============================================================
# 4. SELECT DETERMINISTIC TEST CASE
# ============================================================

print("\n" + "-" * 70)
print("SELECTING TEST CASE")
print("-" * 70)

test_cases = case_manifest[
    case_manifest["split"].astype(str).str.lower() == "test"
].copy()

if len(test_cases) == 0:
    raise RuntimeError("No test cases found.")

test_cases = test_cases.sort_values(
    by="case_id"
).reset_index(drop=True)

CASE_ID = test_cases.iloc[0]["case_id"]

case_row = test_cases[
    test_cases["case_id"] == CASE_ID
].iloc[0]

case_images = manifest[
    manifest["case_id"] == CASE_ID
].copy()

case_images = case_images.sort_values(
    by="image_path"
).reset_index(drop=True)

if len(case_images) == 0:
    raise RuntimeError(
        f"No NORMAL images found for case {CASE_ID}"
    )

image_paths = case_images[
    "image_path"
].tolist()[:MAX_IMAGES]

target_text = str(
    case_row["ket_luan"]
).strip()

print("Selected case:")
print("  case_id:", CASE_ID)
print("  split:", case_row["split"])
print("  available images:", len(case_images))
print("  images used:", len(image_paths))
print("  target:", target_text)


# ============================================================
# 5. TOKENIZER
# ============================================================

print("\n" + "-" * 70)
print("LOADING TOKENIZER")
print("-" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL_NAME,
    use_fast=False
)

print("Tokenizer loaded.")
print("  vocab size:", tokenizer.vocab_size)
print("  pad:", tokenizer.pad_token_id)
print("  eos:", tokenizer.eos_token_id)


# ============================================================
# 6. LOAD IMAGES
# ============================================================

print("\n" + "-" * 70)
print("LOADING IMAGES")
print("-" * 70)

transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

image_tensors = []

for path in image_paths:

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    img = Image.open(path).convert("RGB")

    image_tensors.append(
        transform(img)
    )

# [N, 3, 224, 224]
images = torch.stack(
    image_tensors,
    dim=0
)

# [1, N, 3, 224, 224]
images = images.unsqueeze(0)

images = images.to(device)

print(
    "Image tensor:",
    tuple(images.shape)
)


# ============================================================
# 7. LOAD CHECKPOINTS
# ============================================================

print("\n" + "-" * 70)
print("LOADING CHECKPOINTS")
print("-" * 70)

v1_ckpt = torch.load(
    V1_CKPT,
    map_location="cpu",
    weights_only=False
)

v2_ckpt = torch.load(
    V2_CKPT,
    map_location="cpu",
    weights_only=False
)

print(
    "V1 epoch:",
    v1_ckpt.get("epoch")
)

print(
    "V1 best val loss:",
    v1_ckpt.get("best_val_loss")
)

print(
    "V2 epoch:",
    v2_ckpt.get("epoch")
)

print(
    "V2 best val loss:",
    v2_ckpt.get("best_val_loss")
)


# ============================================================
# 8. PROJECTOR
# ============================================================

class VisualProjector(nn.Module):

    def __init__(
        self,
        input_dim,
        output_dim
    ):
        super().__init__()

        self.proj = nn.Linear(
            input_dim,
            output_dim
        )

    def forward(self, x):

        return self.proj(x)


# ============================================================
# 9. LOAD ViT
# ============================================================

print("\n" + "-" * 70)
print("LOADING ViT")
print("-" * 70)

# V1:
# ViT was frozen during training.
# Therefore it remains original pretrained ViT.

v1_vit = ViTModel.from_pretrained(
    VISION_MODEL_NAME
)

# V2:
# ViT was trainable.
# Restore trained checkpoint.

v2_vit = ViTModel.from_pretrained(
    VISION_MODEL_NAME
)

v2_vit.load_state_dict(
    v2_ckpt["vit_state_dict"],
    strict=True
)

print("V1 ViT: pretrained")
print("V2 ViT: checkpoint restored")


# ============================================================
# 10. LOAD PROJECTORS
# ============================================================

print("\n" + "-" * 70)
print("LOADING PROJECTORS")
print("-" * 70)

v1_proj = VisualProjector(
    768,
    512
)

v2_proj = VisualProjector(
    768,
    512
)

v1_proj.load_state_dict(
    v1_ckpt["visual_projector"],
    strict=True
)

v2_proj.load_state_dict(
    v2_ckpt["projector_state_dict"],
    strict=True
)

print("V1 projector: restored")
print("V2 projector: restored")


# ============================================================
# 11. LOAD mT5
# ============================================================

print("\n" + "-" * 70)
print("LOADING mT5")
print("-" * 70)

v1_mt5 = MT5ForConditionalGeneration.from_pretrained(
    TEXT_MODEL_NAME
)

v2_mt5 = MT5ForConditionalGeneration.from_pretrained(
    TEXT_MODEL_NAME
)

v1_mt5.load_state_dict(
    v1_ckpt["mt5_model"],
    strict=True
)

v2_mt5.load_state_dict(
    v2_ckpt["mt5_state_dict"],
    strict=True
)

print("V1 mT5: restored")
print("V2 mT5: restored")


# ============================================================
# 12. MOVE TO GPU
# ============================================================

v1_vit = v1_vit.to(device).eval()
v2_vit = v2_vit.to(device).eval()

v1_proj = v1_proj.to(device).eval()
v2_proj = v2_proj.to(device).eval()

v1_mt5 = v1_mt5.to(device).eval()
v2_mt5 = v2_mt5.to(device).eval()

print("\nAll models moved to:", device)


# ============================================================
# 13. VISUAL PREFIX
# ============================================================

print("\n" + "-" * 70)
print("EXTRACTING VISUAL PREFIX")
print("-" * 70)


@torch.no_grad()
def make_visual_prefix(
    vit,
    projector,
    batch_images
):
    """
    Input:
        [B, N, 3, H, W]

    ViT expects:
        [B*N, 3, H, W]

    Output:
        [B, N, 512]
    """

    B, N, C, H, W = batch_images.shape

    print(
        f"    Input to ViT: "
        f"({B*N}, {C}, {H}, {W})"
    )

    # --------------------------------------------------------
    # Flatten case/image dimensions
    # --------------------------------------------------------

    flat_images = batch_images.reshape(
        B * N,
        C,
        H,
        W
    )

    # --------------------------------------------------------
    # ViT
    # --------------------------------------------------------

    outputs = vit(
        pixel_values=flat_images
    )

    # [B*N, 768]
    cls = outputs.last_hidden_state[:, 0, :]

    # --------------------------------------------------------
    # Project 768 -> 512
    # --------------------------------------------------------

    projected = projector(cls)

    # [B, N, 512]
    prefix = projected.reshape(
        B,
        N,
        -1
    )

    return prefix


print("\nV1:")
v1_prefix = make_visual_prefix(
    v1_vit,
    v1_proj,
    images
)

print("\nV2:")
v2_prefix = make_visual_prefix(
    v2_vit,
    v2_proj,
    images
)

print(
    "\nV1 prefix shape:",
    tuple(v1_prefix.shape)
)

print(
    "V2 prefix shape:",
    tuple(v2_prefix.shape)
)


# ============================================================
# 14. PREFIX STATISTICS
# ============================================================

print("\n" + "-" * 70)
print("VISUAL PREFIX STATISTICS")
print("-" * 70)


def print_prefix_stats(
    name,
    prefix
):

    x = prefix.float()

    print(f"\n{name}")

    print(
        "  mean      :",
        x.mean().item()
    )

    print(
        "  std       :",
        x.std().item()
    )

    print(
        "  min       :",
        x.min().item()
    )

    print(
        "  max       :",
        x.max().item()
    )

    print(
        "  abs mean  :",
        x.abs().mean().item()
    )

    print(
        "  L2 norm   :",
        x.norm().item()
    )

    print(
        "  RMS       :",
        torch.sqrt(
            torch.mean(x ** 2)
        ).item()
    )


print_prefix_stats(
    "V1",
    v1_prefix
)

print_prefix_stats(
    "V2",
    v2_prefix
)


# ============================================================
# 15. PREFIX SIMILARITY
# ============================================================

print("\n" + "-" * 70)
print("V1 ↔ V2 PREFIX COMPARISON")
print("-" * 70)

v1_flat = v1_prefix.float().flatten()
v2_flat = v2_prefix.float().flatten()

cosine_similarity = torch.nn.functional.cosine_similarity(
    v1_flat.unsqueeze(0),
    v2_flat.unsqueeze(0)
).item()

mean_abs_diff = (
    v1_flat - v2_flat
).abs().mean().item()

max_abs_diff = (
    v1_flat - v2_flat
).abs().max().item()

print(
    "Cosine similarity:",
    cosine_similarity
)

print(
    "Mean absolute difference:",
    mean_abs_diff
)

print(
    "Max absolute difference:",
    max_abs_diff
)


# ============================================================
# 16. TOKENIZE TARGET
# ============================================================

print("\n" + "-" * 70)
print("TOKENIZING TARGET")
print("-" * 70)

target_encoding = tokenizer(
    target_text,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_TARGET_LENGTH
)

labels = target_encoding.input_ids.to(device)

labels_for_loss = labels.clone()

labels_for_loss[
    labels_for_loss == tokenizer.pad_token_id
] = -100

print(
    "Target token shape:",
    tuple(labels.shape)
)

print(
    "Target token count:",
    labels.shape[1]
)


# ============================================================
# 17. ENCODE VISUAL PREFIX
# ============================================================

def encode_visual_prefix(
    mt5,
    prefix
):

    return mt5.encoder(
        inputs_embeds=prefix,
        return_dict=True
    )


# ============================================================
# 18. TEACHER-FORCED LOSS
# ============================================================

print("\n" + "-" * 70)
print("TEACHER-FORCED LOSS")
print("-" * 70)


@torch.no_grad()
def compute_teacher_forced_loss(
    mt5,
    prefix
):

    encoder_outputs = encode_visual_prefix(
        mt5,
        prefix
    )

    attention_mask = torch.ones(
        prefix.shape[:2],
        dtype=torch.long,
        device=device
    )

    outputs = mt5(
        encoder_outputs=encoder_outputs,
        attention_mask=attention_mask,
        labels=labels_for_loss,
        return_dict=True
    )

    return outputs.loss.item()


v1_loss = compute_teacher_forced_loss(
    v1_mt5,
    v1_prefix
)

v2_loss = compute_teacher_forced_loss(
    v2_mt5,
    v2_prefix
)

print(
    "V1 teacher-forced loss:",
    v1_loss
)

print(
    "V2 teacher-forced loss:",
    v2_loss
)


# ============================================================
# 19. FIRST TOKEN — REAL IMAGE
# ============================================================

print("\n" + "-" * 70)
print("FIRST TOKEN DISTRIBUTION — REAL IMAGE")
print("-" * 70)


def get_first_token_distribution(
    mt5,
    prefix
):

    encoder_outputs = encode_visual_prefix(
        mt5,
        prefix
    )

    attention_mask = torch.ones(
        prefix.shape[:2],
        dtype=torch.long,
        device=device
    )

    decoder_input_ids = torch.tensor(
        [[
            mt5.config.decoder_start_token_id
        ]],
        dtype=torch.long,
        device=device
    )

    outputs = mt5(
        encoder_outputs=encoder_outputs,
        attention_mask=attention_mask,
        decoder_input_ids=decoder_input_ids,
        return_dict=True
    )

    logits = outputs.logits[:, -1, :].float()

    probs = torch.softmax(
        logits,
        dim=-1
    )

    top_probs, top_ids = torch.topk(
        probs,
        k=15,
        dim=-1
    )

    result = []

    for token_id, probability in zip(
        top_ids[0].cpu(),
        top_probs[0].cpu()
    ):

        token_id = int(token_id)

        decoded = tokenizer.decode(
            [token_id],
            skip_special_tokens=False
        )

        result.append(
            (
                token_id,
                float(probability),
                decoded
            )
        )

    return result


for name, mt5, prefix in [
    ("V1", v1_mt5, v1_prefix),
    ("V2", v2_mt5, v2_prefix),
]:

    print(f"\n{name}")

    top_tokens = get_first_token_distribution(
        mt5,
        prefix
    )

    for rank, (
        token_id,
        probability,
        decoded
    ) in enumerate(
        top_tokens,
        start=1
    ):

        print(
            f"{rank:2d}. "
            f"id={token_id:6d} "
            f"prob={probability:.6f} "
            f"text={repr(decoded)}"
        )


# ============================================================
# 20. GENERATION
# ============================================================

print("\n" + "-" * 70)
print("GENERATION — GREEDY vs BEAM")
print("-" * 70)


@torch.no_grad()
def generate_predictions(
    mt5,
    prefix
):

    encoder_outputs = encode_visual_prefix(
        mt5,
        prefix
    )

    attention_mask = torch.ones(
        prefix.shape[:2],
        dtype=torch.long,
        device=device
    )

    results = {}

    # --------------------------------------------------------
    # Greedy
    # --------------------------------------------------------

    greedy_ids = mt5.generate(
        encoder_outputs=encoder_outputs,
        attention_mask=attention_mask,
        max_new_tokens=96,
        num_beams=1,
    )

    results["greedy"] = tokenizer.decode(
        greedy_ids[0],
        skip_special_tokens=False
    )

    # --------------------------------------------------------
    # Beam
    # --------------------------------------------------------

    beam_ids = mt5.generate(
        encoder_outputs=encoder_outputs,
        attention_mask=attention_mask,
        max_new_tokens=96,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=2,
    )

    results["beam"] = tokenizer.decode(
        beam_ids[0],
        skip_special_tokens=False
    )

    return results


v1_generation = generate_predictions(
    v1_mt5,
    v1_prefix
)

v2_generation = generate_predictions(
    v2_mt5,
    v2_prefix
)


# ============================================================
# 21. FINAL RESULT
# ============================================================

print("\n" + "=" * 70)
print("FINAL RESULT")
print("=" * 70)

print("\nCASE ID:")
print(CASE_ID)

print("\nTARGET:")
print(target_text)

print("\n" + "-" * 70)
print("V1")
print("-" * 70)

print(
    "Teacher-forced loss:",
    v1_loss
)

print(
    "Greedy:",
    v1_generation["greedy"]
)

print(
    "Beam:",
    v1_generation["beam"]
)

print("\n" + "-" * 70)
print("V2")
print("-" * 70)

print(
    "Teacher-forced loss:",
    v2_loss
)

print(
    "Greedy:",
    v2_generation["greedy"]
)

print(
    "Beam:",
    v2_generation["beam"]
)

print("\n" + "=" * 70)
print("DIAGNOSTIC-2 COMPLETE")
print("=" * 70)

V2-DIAGNOSTIC-2 — REAL VISUAL PREFIX ANALYSIS

Device: cuda
GPU: NVIDIA A100-SXM4-40GB

----------------------------------------------------------------------
CHECKING FILES
----------------------------------------------------------------------
V1 checkpoint       : PASS
V2 checkpoint       : PASS
image manifest      : PASS
case manifest       : PASS
split file          : PASS

----------------------------------------------------------------------
LOADING DATA
----------------------------------------------------------------------
Manifest rows: 76405
Case manifest rows: 7607
Split rows: 7607
NORMAL images: 76216
Cases with non-empty target: 7606

----------------------------------------------------------------------
SELECTING TEST CASE
----------------------------------------------------------------------
Selected case:
  case_id: 10017.10017.0.10031
  split: test
  available images: 16
  images used: 8
  target: VIÊM MÀNG NHĨ CẤP BÓNG NƯỚC P - NHIỀU RÁY TAI T

------------------------

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


V1 mT5: restored
V2 mT5: restored

All models moved to: cuda

----------------------------------------------------------------------
EXTRACTING VISUAL PREFIX
----------------------------------------------------------------------

V1:
    Input to ViT: (8, 3, 224, 224)

V2:
    Input to ViT: (8, 3, 224, 224)

V1 prefix shape: (1, 8, 512)
V2 prefix shape: (1, 8, 512)

----------------------------------------------------------------------
VISUAL PREFIX STATISTICS
----------------------------------------------------------------------

V1
  mean      : 0.02769576758146286
  std       : 0.5314910411834717
  min       : -1.6951754093170166
  max       : 1.7771296501159668
  abs mean  : 0.4304696023464203
  L2 norm   : 34.05743408203125
  RMS       : 0.5321474075317383

V2
  mean      : 0.007998121902346611
  std       : 0.8034038543701172
  min       : -2.567127227783203
  max       : 2.932431697845459
  abs mean  : 0.6418313980102539
  L2 norm   : 51.41411590576172
  RMS       : 0.8033455610

/tmp/ipykernel_2005/4175539125.py:873: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  float(probability),


 1. id=  8755 prob=0.978884 text='VI'
 2. id= 30318 prob=0.019507 text='HI'
 3. id= 12949 prob=0.001092 text='CH'
 4. id=   447 prob=0.000286 text='H'
 5. id=   441 prob=0.000109 text='N'
 6. id= 33431 prob=0.000051 text='NH'
 7. id= 17764 prob=0.000019 text='TR'
 8. id=   431 prob=0.000011 text='D'
 9. id= 32648 prob=0.000010 text='KH'
10. id=   371 prob=0.000008 text='C'
11. id=   364 prob=0.000004 text='B'
12. id=   458 prob=0.000003 text='L'
13. id=   298 prob=0.000003 text='A'
14. id= 13599 prob=0.000002 text='TH'
15. id=   352 prob=0.000001 text='M'

V2
 1. id=  8755 prob=0.974881 text='VI'
 2. id= 30318 prob=0.024019 text='HI'
 3. id= 12949 prob=0.000682 text='CH'
 4. id=   447 prob=0.000272 text='H'
 5. id=   441 prob=0.000056 text='N'
 6. id= 17764 prob=0.000019 text='TR'
 7. id= 33431 prob=0.000018 text='NH'
 8. id=   431 prob=0.000010 text='D'
 9. id= 32648 prob=0.000009 text='KH'
10. id=   371 prob=0.000008 text='C'
11. id=   364 prob=0.000005 text='B'
12. id=   298 prob=0.

In [13]:
# ============================================================
# V2-DIAGNOSTIC-3
# FULL TEST SET — TEACHER-FORCED LOSS COMPARISON
#
# Compare V1 vs V2 on all 712 test cases.
#
# NO TRAINING
# NO CHECKPOINT MODIFICATION
# NO AUTOREGRESSIVE GENERATION
#
# Output:
#   baseline_model_v2/
#       diagnostic_3/
#           test_teacher_forced_loss.csv
#           summary.txt
# ============================================================

import os
import gc
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

from transformers import (
    ViTModel,
    MT5ForConditionalGeneration,
    AutoTokenizer,
)

print("=" * 70)
print("V2-DIAGNOSTIC-3 — FULL TEST TEACHER-FORCED LOSS")
print("=" * 70)


# ============================================================
# 1. CONFIG
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/NoiSoi_Matching"

V1_CKPT = os.path.join(
    PROJECT_DIR,
    "baseline_model_v1",
    "checkpoints",
    "best.pt"
)

V2_CKPT = os.path.join(
    PROJECT_DIR,
    "baseline_model_v2",
    "checkpoints",
    "best.pt"
)

MANIFEST_FILE = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

CASE_MANIFEST_FILE = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_case_manifest.csv"
)

SPLIT_FILE = os.path.join(
    PROJECT_DIR,
    "patient_split",
    "cases_with_split.csv"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "baseline_model_v2",
    "diagnostic_3"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

OUTPUT_CSV = os.path.join(
    OUTPUT_DIR,
    "test_teacher_forced_loss.csv"
)

SUMMARY_FILE = os.path.join(
    OUTPUT_DIR,
    "summary.txt"
)

VISION_MODEL_NAME = "google/vit-base-patch16-224"
TEXT_MODEL_NAME = "google/mt5-small"

IMAGE_SIZE = 224
MAX_IMAGES = 8
MAX_TARGET_LENGTH = 96

BATCH_SIZE = 4

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# 2. CHECKPOINT / RESUME
# ============================================================

if os.path.exists(OUTPUT_CSV):

    old_df = pd.read_csv(
        OUTPUT_CSV
    )

    print("\nExisting diagnostic found:")
    print(
        "  rows:",
        len(old_df)
    )

    print(
        "  file:",
        OUTPUT_CSV
    )

    print(
        "\nTo avoid accidentally overwriting a completed run, "
        "this cell will stop."
    )

    print(
        "If you want to recompute, delete the CSV first."
    )

    raise SystemExit


# ============================================================
# 3. LOAD DATA
# ============================================================

print("\n" + "-" * 70)
print("LOADING DATA")
print("-" * 70)

manifest = pd.read_csv(
    MANIFEST_FILE,
    low_memory=False
)

case_manifest = pd.read_csv(
    CASE_MANIFEST_FILE,
    low_memory=False
)

split_df = pd.read_csv(
    SPLIT_FILE,
    low_memory=False
)

# NORMAL only
manifest["image_status"] = (
    manifest["image_status"]
    .fillna("")
    .astype(str)
    .str.upper()
)

manifest = manifest[
    manifest["image_status"] == "NORMAL"
].copy()

# Split
if "split" not in case_manifest.columns:

    case_manifest = case_manifest.merge(
        split_df[[
            "case_id",
            "split"
        ]],
        on="case_id",
        how="left"
    )

# Target
case_manifest["ket_luan"] = (
    case_manifest["ket_luan"]
    .fillna("")
    .astype(str)
    .str.strip()
)

case_manifest = case_manifest[
    case_manifest["ket_luan"] != ""
].copy()

# Test only
test_cases = case_manifest[
    case_manifest["split"].astype(str).str.lower() == "test"
].copy()

test_cases = test_cases.sort_values(
    "case_id"
).reset_index(drop=True)

print(
    "NORMAL images:",
    len(manifest)
)

print(
    "Test cases:",
    len(test_cases)
)


# ============================================================
# 4. BUILD CASE IMAGE LIST
# ============================================================

case_records = []

for _, row in test_cases.iterrows():

    case_id = row["case_id"]

    imgs = manifest[
        manifest["case_id"] == case_id
    ].sort_values(
        "image_path"
    )

    image_paths = imgs[
        "image_path"
    ].tolist()[:MAX_IMAGES]

    if len(image_paths) == 0:
        continue

    case_records.append({
        "case_id": case_id,
        "patient_group_id": row["patient_group_id"],
        "target_text": row["ket_luan"],
        "image_paths": image_paths,
        "num_images": len(image_paths),
    })

print(
    "Cases with >=1 NORMAL image:",
    len(case_records)
)


# ============================================================
# 5. TOKENIZER
# ============================================================

print("\n" + "-" * 70)
print("LOADING TOKENIZER")
print("-" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL_NAME,
    use_fast=False
)

print(
    "Vocab:",
    tokenizer.vocab_size
)


# ============================================================
# 6. TRANSFORM
# ============================================================

transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])


# ============================================================
# 7. DATASET
# ============================================================

class TestCaseDataset(Dataset):

    def __init__(
        self,
        records,
        tokenizer,
        transform,
        max_length=96
    ):
        self.records = records
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):

        record = self.records[idx]

        image_tensors = []

        for path in record["image_paths"]:

            img = Image.open(path).convert("RGB")

            image_tensors.append(
                self.transform(img)
            )

        images = torch.stack(
            image_tensors,
            dim=0
        )

        encoded = self.tokenizer(
            record["target_text"],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        labels = encoded.input_ids.squeeze(0)

        labels[
            labels == self.tokenizer.pad_token_id
        ] = -100

        return {
            "case_id": record["case_id"],
            "patient_group_id": record["patient_group_id"],
            "images": images,
            "target_text": record["target_text"],
            "labels": labels,
        }


# ============================================================
# 8. COLLATE
# ============================================================

def collate_fn(batch):

    max_n = max(
        item["images"].shape[0]
        for item in batch
    )

    B = len(batch)

    padded_images = torch.zeros(
        B,
        max_n,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE,
        dtype=torch.float32
    )

    image_mask = torch.zeros(
        B,
        max_n,
        dtype=torch.long
    )

    labels = torch.stack([
        item["labels"]
        for item in batch
    ])

    case_ids = []
    patient_ids = []
    targets = []

    for i, item in enumerate(batch):

        n = item["images"].shape[0]

        padded_images[
            i,
            :n
        ] = item["images"]

        image_mask[
            i,
            :n
        ] = 1

        case_ids.append(
            item["case_id"]
        )

        patient_ids.append(
            item["patient_group_id"]
        )

        targets.append(
            item["target_text"]
        )

    return {
        "case_id": case_ids,
        "patient_group_id": patient_ids,
        "images": padded_images,
        "image_mask": image_mask,
        "labels": labels,
        "target_text": targets,
    }


test_dataset = TestCaseDataset(
    case_records,
    tokenizer,
    transform,
    MAX_TARGET_LENGTH
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn
)

print(
    "Dataset:",
    len(test_dataset)
)

print(
    "Batches:",
    len(test_loader)
)


# ============================================================
# 9. PROJECTOR
# ============================================================

class VisualProjector(nn.Module):

    def __init__(
        self,
        input_dim=768,
        output_dim=512
    ):
        super().__init__()

        self.proj = nn.Linear(
            input_dim,
            output_dim
        )

    def forward(self, x):

        return self.proj(x)


# ============================================================
# 10. LOAD CHECKPOINTS
# ============================================================

print("\n" + "-" * 70)
print("LOADING CHECKPOINTS")
print("-" * 70)

v1_ckpt = torch.load(
    V1_CKPT,
    map_location="cpu",
    weights_only=False
)

v2_ckpt = torch.load(
    V2_CKPT,
    map_location="cpu",
    weights_only=False
)


# ============================================================
# 11. LOAD V1
# ============================================================

print("\nLoading V1...")

v1_vit = ViTModel.from_pretrained(
    VISION_MODEL_NAME
)

v1_proj = VisualProjector()

v1_mt5 = MT5ForConditionalGeneration.from_pretrained(
    TEXT_MODEL_NAME
)

v1_proj.load_state_dict(
    v1_ckpt["visual_projector"],
    strict=True
)

v1_mt5.load_state_dict(
    v1_ckpt["mt5_model"],
    strict=True
)

# V1 ViT was frozen during training.
# Therefore it is still the original pretrained ViT.

v1_vit = v1_vit.to(device).eval()
v1_proj = v1_proj.to(device).eval()
v1_mt5 = v1_mt5.to(device).eval()

print("V1 loaded.")


# ============================================================
# 12. LOAD V2
# ============================================================

print("\nLoading V2...")

v2_vit = ViTModel.from_pretrained(
    VISION_MODEL_NAME
)

v2_vit.load_state_dict(
    v2_ckpt["vit_state_dict"],
    strict=True
)

v2_proj = VisualProjector()

v2_proj.load_state_dict(
    v2_ckpt["projector_state_dict"],
    strict=True
)

v2_mt5 = MT5ForConditionalGeneration.from_pretrained(
    TEXT_MODEL_NAME
)

v2_mt5.load_state_dict(
    v2_ckpt["mt5_state_dict"],
    strict=True
)

v2_vit = v2_vit.to(device).eval()
v2_proj = v2_proj.to(device).eval()
v2_mt5 = v2_mt5.to(device).eval()

print("V2 loaded.")


# ============================================================
# 13. VISUAL PREFIX FUNCTION
# ============================================================

@torch.no_grad()
def get_visual_prefix(
    vit,
    projector,
    images,
    image_mask
):

    B, N, C, H, W = images.shape

    flat_images = images.reshape(
        B * N,
        C,
        H,
        W
    )

    # ViT
    vit_output = vit(
        pixel_values=flat_images
    )

    # CLS
    cls = vit_output.last_hidden_state[
        :,
        0,
        :
    ]

    # Project
    projected = projector(cls)

    # [B, N, 512]
    prefix = projected.reshape(
        B,
        N,
        -1
    )

    return prefix


# ============================================================
# 14. EVALUATION FUNCTION
# ============================================================

@torch.no_grad()
def evaluate_model(
    vit,
    projector,
    mt5,
    loader,
    model_name
):

    results = []

    total_batches = len(loader)

    print(
        f"\nEvaluating {model_name}..."
    )

    for batch_idx, batch in enumerate(
        loader,
        start=1
    ):

        images = batch["images"].to(
            device,
            non_blocking=True
        )

        image_mask = batch["image_mask"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Visual encoder
        # ----------------------------------------------------

        prefix = get_visual_prefix(
            vit,
            projector,
            images,
            image_mask
        )

        # ----------------------------------------------------
        # mT5 encoder
        # ----------------------------------------------------

        encoder_outputs = mt5.encoder(
            inputs_embeds=prefix,
            return_dict=True
        )

        # ----------------------------------------------------
        # IMPORTANT:
        # mask padded image slots
        # ----------------------------------------------------

        attention_mask = image_mask

        # ----------------------------------------------------
        # Teacher-forced loss
        # ----------------------------------------------------

        outputs = mt5(
            encoder_outputs=encoder_outputs,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=True
        )

        loss = outputs.loss

        # ----------------------------------------------------
        # Per-case loss
        #
        # We calculate token-level CE manually so that
        # every case gets its own loss.
        # ----------------------------------------------------

        logits = outputs.logits.float()

        # Shift exactly as T5 does
        shift_logits = logits[:, :-1, :]
        shift_labels = labels[:, 1:]

        vocab_size = shift_logits.shape[-1]

        token_loss = torch.nn.functional.cross_entropy(
            shift_logits.reshape(
                -1,
                vocab_size
            ),
            shift_labels.reshape(-1),
            ignore_index=-100,
            reduction="none"
        )

        token_loss = token_loss.reshape(
            shift_labels.shape
        )

        valid_tokens = (
            shift_labels != -100
        )

        per_case_loss = (
            token_loss.sum(dim=1)
            /
            valid_tokens.sum(dim=1).clamp(min=1)
        )

        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------

        for i in range(len(batch["case_id"])):

            results.append({
                "case_id":
                    batch["case_id"][i],

                "patient_group_id":
                    batch["patient_group_id"][i],

                "target_text":
                    batch["target_text"][i],

                f"{model_name}_loss":
                    float(
                        per_case_loss[i].item()
                    ),

                "num_images":
                    int(
                        image_mask[i].sum().item()
                    ),

                "target_tokens":
                    int(
                        valid_tokens[i].sum().item()
                    ),
            })

        if (
            batch_idx == 1
            or batch_idx % 25 == 0
            or batch_idx == total_batches
        ):

            print(
                f"  Batch "
                f"{batch_idx}/{total_batches} "
                f"| batch loss={loss.item():.4f}"
            )

    return pd.DataFrame(results)


# ============================================================
# 15. RUN V1
# ============================================================

v1_results = evaluate_model(
    v1_vit,
    v1_proj,
    v1_mt5,
    test_loader,
    "v1"
)


# ============================================================
# 16. CLEAN GPU MEMORY
# ============================================================

torch.cuda.empty_cache()
gc.collect()


# ============================================================
# 17. RUN V2
# ============================================================

v2_results = evaluate_model(
    v2_vit,
    v2_proj,
    v2_mt5,
    test_loader,
    "v2"
)


# ============================================================
# 18. MERGE
# ============================================================

results = v1_results.merge(
    v2_results[
        [
            "case_id",
            "v2_loss"
        ]
    ],
    on="case_id",
    how="inner"
)

results["loss_difference_v2_minus_v1"] = (
    results["v2_loss"]
    -
    results["v1_loss"]
)

results["v2_better"] = (
    results["v2_loss"]
    <
    results["v1_loss"]
)

results["v1_better"] = (
    results["v1_loss"]
    <
    results["v2_loss"]
)

results["loss_tie"] = (
    results["v1_loss"]
    ==
    results["v2_loss"]
)


# ============================================================
# 19. SAVE RAW RESULTS
# ============================================================

results.to_csv(
    OUTPUT_CSV,
    index=False
)

print(
    "\nSaved:",
    OUTPUT_CSV
)


# ============================================================
# 20. SUMMARY STATISTICS
# ============================================================

v1_mean = results["v1_loss"].mean()
v1_median = results["v1_loss"].median()
v1_std = results["v1_loss"].std()

v2_mean = results["v2_loss"].mean()
v2_median = results["v2_loss"].median()
v2_std = results["v2_loss"].std()

n_v2_better = int(
    results["v2_better"].sum()
)

n_v1_better = int(
    results["v1_better"].sum()
)

n_tie = int(
    results["loss_tie"].sum()
)

n_total = len(results)

pct_v2_better = (
    n_v2_better
    /
    n_total
    *
    100
)

pct_v1_better = (
    n_v1_better
    /
    n_total
    *
    100
)

mean_difference = (
    results[
        "loss_difference_v2_minus_v1"
    ].mean()
)

median_difference = (
    results[
        "loss_difference_v2_minus_v1"
    ].median()
)

# Relative improvement:
# negative difference means V2 has lower loss.

relative_improvement = (
    (v1_mean - v2_mean)
    /
    v1_mean
    *
    100
)


# ============================================================
# 21. TARGET LENGTH ANALYSIS
# ============================================================

results["loss_gap_abs"] = (
    results["loss_difference_v2_minus_v1"]
    .abs()
)

results["target_length"] = (
    results["target_text"]
    .astype(str)
    .str.len()
)

length_corr_v1 = results[
    ["target_length", "v1_loss"]
].corr().iloc[0, 1]

length_corr_v2 = results[
    ["target_length", "v2_loss"]
].corr().iloc[0, 1]


# ============================================================
# 22. EXTREME CASES
# ============================================================

largest_v2_advantage = results.sort_values(
    "loss_difference_v2_minus_v1"
).head(10)

largest_v1_advantage = results.sort_values(
    "loss_difference_v2_minus_v1",
    ascending=False
).head(10)


# ============================================================
# 23. PRINT SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("V2-DIAGNOSTIC-3 SUMMARY")
print("=" * 70)

print("\nNumber of test cases:", n_total)

print("\nTeacher-forced loss:")

print(
    f"  V1 mean   : {v1_mean:.6f}"
)

print(
    f"  V1 median : {v1_median:.6f}"
)

print(
    f"  V1 std    : {v1_std:.6f}"
)

print()

print(
    f"  V2 mean   : {v2_mean:.6f}"
)

print(
    f"  V2 median : {v2_median:.6f}"
)

print(
    f"  V2 std    : {v2_std:.6f}"
)

print("\nV2 - V1 loss:")

print(
    f"  Mean difference   : {mean_difference:.6f}"
)

print(
    f"  Median difference : {median_difference:.6f}"
)

print(
    f"  Relative mean improvement: "
    f"{relative_improvement:.2f}%"
)

print("\nCase-by-case comparison:")

print(
    f"  V2 lower loss : "
    f"{n_v2_better}/{n_total} "
    f"({pct_v2_better:.2f}%)"
)

print(
    f"  V1 lower loss : "
    f"{n_v1_better}/{n_total} "
    f"({pct_v1_better:.2f}%)"
)

print(
    f"  Exact ties    : "
    f"{n_tie}/{n_total}"
)

print("\nTarget-length correlation:")

print(
    f"  V1 loss ↔ target length: "
    f"{length_corr_v1:.4f}"
)

print(
    f"  V2 loss ↔ target length: "
    f"{length_corr_v2:.4f}"
)


# ============================================================
# 24. SAVE SUMMARY
# ============================================================

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "V2-DIAGNOSTIC-3 — FULL TEST TEACHER-FORCED LOSS\n"
    )

    f.write("=" * 70 + "\n\n")

    f.write(
        f"Test cases: {n_total}\n\n"
    )

    f.write(
        f"V1 mean loss: {v1_mean:.8f}\n"
    )

    f.write(
        f"V1 median loss: {v1_median:.8f}\n"
    )

    f.write(
        f"V1 std: {v1_std:.8f}\n\n"
    )

    f.write(
        f"V2 mean loss: {v2_mean:.8f}\n"
    )

    f.write(
        f"V2 median loss: {v2_median:.8f}\n"
    )

    f.write(
        f"V2 std: {v2_std:.8f}\n\n"
    )

    f.write(
        f"Mean V2-V1 difference: "
        f"{mean_difference:.8f}\n"
    )

    f.write(
        f"Median V2-V1 difference: "
        f"{median_difference:.8f}\n"
    )

    f.write(
        f"Relative mean improvement: "
        f"{relative_improvement:.4f}%\n\n"
    )

    f.write(
        f"V2 lower loss: "
        f"{n_v2_better}/{n_total} "
        f"({pct_v2_better:.2f}%)\n"
    )

    f.write(
        f"V1 lower loss: "
        f"{n_v1_better}/{n_total} "
        f"({pct_v1_better:.2f}%)\n"
    )

    f.write(
        f"Ties: {n_tie}/{n_total}\n\n"
    )

    f.write(
        f"V1 loss-target length correlation: "
        f"{length_corr_v1:.6f}\n"
    )

    f.write(
        f"V2 loss-target length correlation: "
        f"{length_corr_v2:.6f}\n"
    )


# ============================================================
# 25. SHOW EXTREME CASES
# ============================================================

print("\n" + "-" * 70)
print("10 CASES WHERE V2 IMPROVES MOST")
print("-" * 70)

print(
    largest_v2_advantage[
        [
            "case_id",
            "v1_loss",
            "v2_loss",
            "loss_difference_v2_minus_v1",
            "target_text"
        ]
    ].to_string(
        index=False
    )
)

print("\n" + "-" * 70)
print("10 CASES WHERE V1 IMPROVES MOST")
print("-" * 70)

print(
    largest_v1_advantage[
        [
            "case_id",
            "v1_loss",
            "v2_loss",
            "loss_difference_v2_minus_v1",
            "target_text"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 26. CLEANUP
# ============================================================

del v1_vit
del v1_proj
del v1_mt5

del v2_vit
del v2_proj
del v2_mt5

del v1_ckpt
del v2_ckpt

torch.cuda.empty_cache()
gc.collect()

print("\n" + "=" * 70)
print("DIAGNOSTIC-3 COMPLETE")
print("=" * 70)

print("\nResults:")
print(OUTPUT_CSV)

print("\nSummary:")
print(SUMMARY_FILE)

V2-DIAGNOSTIC-3 — FULL TEST TEACHER-FORCED LOSS

Device: cuda
GPU: NVIDIA A100-SXM4-40GB

----------------------------------------------------------------------
LOADING DATA
----------------------------------------------------------------------
NORMAL images: 76216
Test cases: 712
Cases with >=1 NORMAL image: 712

----------------------------------------------------------------------
LOADING TOKENIZER
----------------------------------------------------------------------
Vocab: 250100
Dataset: 712
Batches: 178

----------------------------------------------------------------------
LOADING CHECKPOINTS
----------------------------------------------------------------------

Loading V1...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


...

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


V2 loaded.

Evaluating v1...
  Batch 1/178 | batch loss=1.1910
  Batch 25/178 | batch loss=0.8155
  Batch 50/178 | batch loss=1.0179
  Batch 75/178 | batch loss=0.8904
  Batch 100/178 | batch loss=0.6255
  Batch 125/178 | batch loss=0.3032
  Batch 150/178 | batch loss=0.2320
  Batch 175/178 | batch loss=0.2034
  Batch 178/178 | batch loss=0.4971

Evaluating v2...
  Batch 1/178 | batch loss=1.1404
  Batch 25/178 | batch loss=0.7720
  Batch 50/178 | batch loss=1.0375
  Batch 75/178 | batch loss=0.7242
  Batch 100/178 | batch loss=0.6549
  Batch 125/178 | batch loss=0.2989
  Batch 150/178 | batch loss=0.2191
  Batch 175/178 | batch loss=0.1882
  Batch 178/178 | batch loss=0.4747

Saved: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v2/diagnostic_3/test_teacher_forced_loss.csv


V2-DIAGNOSTIC-3 SUMMARY

Number of test cases: 712

Teacher-forced loss:
  V1 mean   : 20.140630
  V1 median : 20.410821
  V1 std    : 1.883231

  V2 mean   : 19.940646
  V2 median : 19.900160
  V2 std    :

In [15]:
# ============================================================
# V2-DIAGNOSTIC-3B
# VERIFY PER-CASE LOSS CALCULATION
#
# This does NOT train anything.
# It only checks whether our manual loss calculation
# matches HuggingFace mT5's outputs.loss.
# ============================================================

import os
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

from transformers import (
    ViTModel,
    MT5ForConditionalGeneration,
    AutoTokenizer,
)

print("=" * 70)
print("V2-DIAGNOSTIC-3B — LOSS CALCULATION VERIFICATION")
print("=" * 70)


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/NoiSoi_Matching"

V1_CKPT = os.path.join(
    PROJECT_DIR,
    "baseline_model_v1",
    "checkpoints",
    "best.pt"
)

V2_CKPT = os.path.join(
    PROJECT_DIR,
    "baseline_model_v2",
    "checkpoints",
    "best.pt"
)

MANIFEST_FILE = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

CASE_MANIFEST_FILE = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_case_manifest.csv"
)

SPLIT_FILE = os.path.join(
    PROJECT_DIR,
    "patient_split",
    "cases_with_split.csv"
)

VISION_MODEL_NAME = "google/vit-base-patch16-224"
TEXT_MODEL_NAME = "google/mt5-small"

IMAGE_SIZE = 224
MAX_IMAGES = 8
MAX_TARGET_LENGTH = 96
BATCH_SIZE = 4

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:", device)


# ============================================================
# 2. LOAD DATA
# ============================================================

manifest = pd.read_csv(
    MANIFEST_FILE,
    low_memory=False
)

case_manifest = pd.read_csv(
    CASE_MANIFEST_FILE,
    low_memory=False
)

split_df = pd.read_csv(
    SPLIT_FILE,
    low_memory=False
)

manifest["image_status"] = (
    manifest["image_status"]
    .fillna("")
    .astype(str)
    .str.upper()
)

manifest = manifest[
    manifest["image_status"] == "NORMAL"
].copy()

if "split" not in case_manifest.columns:

    case_manifest = case_manifest.merge(
        split_df[["case_id", "split"]],
        on="case_id",
        how="left"
    )

case_manifest["ket_luan"] = (
    case_manifest["ket_luan"]
    .fillna("")
    .astype(str)
    .str.strip()
)

case_manifest = case_manifest[
    case_manifest["ket_luan"] != ""
].copy()

test_cases = case_manifest[
    case_manifest["split"].astype(str).str.lower() == "test"
].copy()

test_cases = test_cases.sort_values(
    "case_id"
).reset_index(drop=True)

print(
    "\nTest cases:",
    len(test_cases)
)


# ============================================================
# 3. TOKENIZER + TRANSFORM
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL_NAME,
    use_fast=False
)

transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])


# ============================================================
# 4. BUILD FIRST 4 TEST CASES
# ============================================================

records = []

for _, row in test_cases.head(4).iterrows():

    case_id = row["case_id"]

    imgs = manifest[
        manifest["case_id"] == case_id
    ].sort_values(
        "image_path"
    )

    paths = imgs[
        "image_path"
    ].tolist()[:MAX_IMAGES]

    if len(paths) == 0:
        continue

    records.append({
        "case_id": case_id,
        "patient_group_id": row["patient_group_id"],
        "target_text": row["ket_luan"],
        "image_paths": paths,
    })

print(
    "Diagnostic cases:",
    len(records)
)


# ============================================================
# 5. LOAD BATCH
# ============================================================

def load_case(record):

    tensors = []

    for path in record["image_paths"]:

        img = Image.open(
            path
        ).convert("RGB")

        tensors.append(
            transform(img)
        )

    images = torch.stack(
        tensors,
        dim=0
    )

    encoded = tokenizer(
        record["target_text"],
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
        padding="max_length",
        return_tensors="pt"
    )

    labels = encoded.input_ids.squeeze(0)

    labels[
        labels == tokenizer.pad_token_id
    ] = -100

    return {
        "case_id": record["case_id"],
        "images": images,
        "labels": labels,
        "target": record["target_text"],
    }


batch_items = [
    load_case(r)
    for r in records
]

max_n = max(
    x["images"].shape[0]
    for x in batch_items
)

images = torch.zeros(
    len(batch_items),
    max_n,
    3,
    IMAGE_SIZE,
    IMAGE_SIZE
)

image_mask = torch.zeros(
    len(batch_items),
    max_n,
    dtype=torch.long
)

labels = torch.stack([
    x["labels"]
    for x in batch_items
])

for i, item in enumerate(batch_items):

    n = item["images"].shape[0]

    images[
        i,
        :n
    ] = item["images"]

    image_mask[
        i,
        :n
    ] = 1

images = images.to(device)
image_mask = image_mask.to(device)
labels = labels.to(device)

print(
    "\nImages:",
    tuple(images.shape)
)

print(
    "Labels:",
    tuple(labels.shape)
)


# ============================================================
# 6. LOAD V1
# ============================================================

print("\nLoading V1...")

v1_ckpt = torch.load(
    V1_CKPT,
    map_location="cpu",
    weights_only=False
)

v1_vit = ViTModel.from_pretrained(
    VISION_MODEL_NAME
).to(device).eval()

v1_mt5 = MT5ForConditionalGeneration.from_pretrained(
    TEXT_MODEL_NAME
).to(device).eval()

v1_vit.load_state_dict(
    # V1 ViT was never trained
    # so pretrained weights are correct
    v1_vit.state_dict(),
    strict=True
)

v1_mt5.load_state_dict(
    v1_ckpt["mt5_model"],
    strict=True
)


# ============================================================
# 7. LOAD V2
# ============================================================

print("Loading V2...")

v2_ckpt = torch.load(
    V2_CKPT,
    map_location="cpu",
    weights_only=False
)

v2_vit = ViTModel.from_pretrained(
    VISION_MODEL_NAME
).to(device).eval()

v2_mt5 = MT5ForConditionalGeneration.from_pretrained(
    TEXT_MODEL_NAME
).to(device).eval()

v2_vit.load_state_dict(
    v2_ckpt["vit_state_dict"],
    strict=True
)

v2_mt5.load_state_dict(
    v2_ckpt["mt5_state_dict"],
    strict=True
)


# ============================================================
# 8. PROJECTOR
# ============================================================

class VisualProjector(torch.nn.Module):

    def __init__(
        self,
        input_dim=768,
        output_dim=512
    ):
        super().__init__()

        self.proj = torch.nn.Linear(
            input_dim,
            output_dim
        )

    def forward(self, x):
        return self.proj(x)


v1_proj = VisualProjector().to(device).eval()
v2_proj = VisualProjector().to(device).eval()

v1_proj.load_state_dict(
    v1_ckpt["visual_projector"],
    strict=True
)

v2_proj.load_state_dict(
    v2_ckpt["projector_state_dict"],
    strict=True
)


# ============================================================
# 9. VISUAL PREFIX
# ============================================================

@torch.no_grad()
def make_prefix(
    vit,
    projector,
    images
):

    B, N, C, H, W = images.shape

    flat = images.reshape(
        B * N,
        C,
        H,
        W
    )

    out = vit(
        pixel_values=flat
    )

    cls = out.last_hidden_state[
        :,
        0,
        :
    ]

    projected = projector(
        cls
    )

    return projected.reshape(
        B,
        N,
        512
    )


# ============================================================
# 10. LOSS DIAGNOSTIC
# ============================================================

@torch.no_grad()
def diagnose_loss(
    name,
    mt5,
    prefix
):

    print("\n")
    print("=" * 70)
    print(name)
    print("=" * 70)

    encoder_outputs = mt5.encoder(
        inputs_embeds=prefix,
        return_dict=True
    )

    outputs = mt5(
        encoder_outputs=encoder_outputs,
        attention_mask=image_mask,
        labels=labels,
        return_dict=True
    )

    logits = outputs.logits.float()

    print(
        "\nLogits shape:",
        tuple(logits.shape)
    )

    print(
        "HF outputs.loss:",
        outputs.loss.item()
    )

    # --------------------------------------------------------
    # METHOD A
    # Correct T5-style shifted comparison
    # --------------------------------------------------------

    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]

    vocab_size = shift_logits.shape[-1]

    flat_loss = F.cross_entropy(
        shift_logits.reshape(
            -1,
            vocab_size
        ),
        shift_labels.reshape(-1),
        ignore_index=-100,
        reduction="none"
    )

    token_loss = flat_loss.reshape(
        shift_labels.shape
    )

    valid = (
        shift_labels != -100
    )

    manual_case_loss = (
        token_loss.sum(dim=1)
        /
        valid.sum(dim=1).clamp(min=1)
    )

    manual_mean = manual_case_loss.mean()

    print(
        "\nManual shifted CE mean:",
        manual_mean.item()
    )

    print(
        "Manual per-case:"
    )

    for i in range(
        len(manual_case_loss)
    ):

        print(
            f"  {i}: "
            f"{manual_case_loss[i].item():.6f} "
            f"| valid tokens="
            f"{int(valid[i].sum().item())}"
        )

    # --------------------------------------------------------
    # METHOD B
    # Compare summed CE / valid token count
    # --------------------------------------------------------

    total_loss_sum = token_loss.sum()

    total_valid = valid.sum()

    manual_global = (
        total_loss_sum
        /
        total_valid
    )

    print(
        "\nManual global CE:",
        manual_global.item()
    )

    print(
        "Total valid tokens:",
        int(total_valid.item())
    )

    # --------------------------------------------------------
    # METHOD C
    # Reproduce HuggingFace exactly using
    # logits and labels without manual assumptions
    # --------------------------------------------------------

    hf_style = F.cross_entropy(
        logits.view(
            -1,
            logits.shape[-1]
        ),
        labels.view(-1),
        ignore_index=-100,
        reduction="mean"
    )

    print(
        "\nUnshifted CE:",
        hf_style.item()
    )

    # --------------------------------------------------------
    # Difference
    # --------------------------------------------------------

    print(
        "\nDifference:"
    )

    print(
        "  HF loss - shifted:",
        (
            outputs.loss
            -
            manual_global
        ).item()
    )

    print(
        "  HF loss - unshifted:",
        (
            outputs.loss
            -
            hf_style
        ).item()
    )

    return {
        "hf_loss": outputs.loss.item(),
        "manual_shifted": manual_global.item(),
        "manual_unshifted": hf_style.item(),
        "per_case": manual_case_loss.cpu().numpy(),
    }


# ============================================================
# 11. RUN V1
# ============================================================

with torch.no_grad():

    v1_prefix = make_prefix(
        v1_vit,
        v1_proj,
        images
    )

v1_diag = diagnose_loss(
    "V1 LOSS CHECK",
    v1_mt5,
    v1_prefix
)


# ============================================================
# 12. RUN V2
# ============================================================

with torch.no_grad():

    v2_prefix = make_prefix(
        v2_vit,
        v2_proj,
        images
    )

v2_diag = diagnose_loss(
    "V2 LOSS CHECK",
    v2_mt5,
    v2_prefix
)


# ============================================================
# 13. FINAL COMPARISON
# ============================================================

print("\n")
print("=" * 70)
print("DIAGNOSTIC-3B FINAL")
print("=" * 70)

print("\nV1:")
print(
    "  HF loss       :",
    v1_diag["hf_loss"]
)

print(
    "  Manual shifted:",
    v1_diag["manual_shifted"]
)

print(
    "  Manual unshift:",
    v1_diag["manual_unshifted"]
)

print("\nV2:")
print(
    "  HF loss       :",
    v2_diag["hf_loss"]
)

print(
    "  Manual shifted:",
    v2_diag["manual_shifted"]
)

print(
    "  Manual unshift:",
    v2_diag["manual_unshifted"]
)

print("\n" + "=" * 70)
print("DIAGNOSTIC-3B COMPLETE")
print("=" * 70)

V2-DIAGNOSTIC-3B — LOSS CALCULATION VERIFICATION

Device: cuda

Test cases: 712
Diagnostic cases: 4

Images: (4, 8, 3, 224, 224)
Labels: (4, 96)

Loading V1...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


...

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.




V1 LOSS CHECK

Logits shape: (4, 96, 250112)
HF outputs.loss: 1.1910157203674316

Manual shifted CE mean: 19.400339126586914
Manual per-case:
  0: 16.642889 | valid tokens=31
  1: 18.997700 | valid tokens=32
  2: 21.234537 | valid tokens=20
  3: 20.726229 | valid tokens=32

Manual global CE: 19.23292350769043
Total valid tokens: 115

Unshifted CE: 1.1910157203674316

Difference:
  HF loss - shifted: -18.041908264160156
  HF loss - unshifted: 0.0


V2 LOSS CHECK

Logits shape: (4, 96, 250112)
HF outputs.loss: 1.140355110168457

Manual shifted CE mean: 18.424110412597656
Manual per-case:
  0: 15.991030 | valid tokens=31
  1: 18.361008 | valid tokens=32
  2: 19.505367 | valid tokens=20
  3: 19.839035 | valid tokens=32

Manual global CE: 18.332439422607422
Total valid tokens: 115

Unshifted CE: 1.140355110168457

Difference:
  HF loss - shifted: -17.19208526611328
  HF loss - unshifted: 0.0


DIAGNOSTIC-3B FINAL

V1:
  HF loss       : 1.1910157203674316
  Manual shifted: 19.2329235076904

In [16]:
# ============================================================
# V2-DIAGNOSTIC-3C
# FULL TEST SET — CORRECT PER-CASE TEACHER-FORCED LOSS
#
# IMPORTANT:
# mT5/HuggingFace already handles the decoder shift internally.
# Therefore DO NOT shift logits/labels again here.
#
# NO TRAINING
# NO CHECKPOINT MODIFICATION
# ============================================================

import os
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

from transformers import (
    ViTModel,
    MT5ForConditionalGeneration,
    AutoTokenizer,
)

print("=" * 70)
print("V2-DIAGNOSTIC-3C — CORRECT FULL TEST LOSS")
print("=" * 70)


# ============================================================
# 1. CONFIG
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/NoiSoi_Matching"

V1_CKPT = os.path.join(
    PROJECT_DIR,
    "baseline_model_v1",
    "checkpoints",
    "best.pt"
)

V2_CKPT = os.path.join(
    PROJECT_DIR,
    "baseline_model_v2",
    "checkpoints",
    "best.pt"
)

MANIFEST_FILE = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

CASE_MANIFEST_FILE = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_case_manifest.csv"
)

SPLIT_FILE = os.path.join(
    PROJECT_DIR,
    "patient_split",
    "cases_with_split.csv"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "baseline_model_v2",
    "diagnostic_3"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

OUTPUT_CSV = os.path.join(
    OUTPUT_DIR,
    "test_teacher_forced_loss_correct.csv"
)

SUMMARY_FILE = os.path.join(
    OUTPUT_DIR,
    "summary_correct.txt"
)

VISION_MODEL_NAME = "google/vit-base-patch16-224"
TEXT_MODEL_NAME = "google/mt5-small"

IMAGE_SIZE = 224
MAX_IMAGES = 8
MAX_TARGET_LENGTH = 96
BATCH_SIZE = 4

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# 2. LOAD DATA
# ============================================================

print("\n" + "-" * 70)
print("LOADING DATA")
print("-" * 70)

manifest = pd.read_csv(
    MANIFEST_FILE,
    low_memory=False
)

case_manifest = pd.read_csv(
    CASE_MANIFEST_FILE,
    low_memory=False
)

split_df = pd.read_csv(
    SPLIT_FILE,
    low_memory=False
)

manifest["image_status"] = (
    manifest["image_status"]
    .fillna("")
    .astype(str)
    .str.upper()
)

manifest = manifest[
    manifest["image_status"] == "NORMAL"
].copy()

if "split" not in case_manifest.columns:

    case_manifest = case_manifest.merge(
        split_df[
            ["case_id", "split"]
        ],
        on="case_id",
        how="left"
    )

case_manifest["ket_luan"] = (
    case_manifest["ket_luan"]
    .fillna("")
    .astype(str)
    .str.strip()
)

case_manifest = case_manifest[
    case_manifest["ket_luan"] != ""
].copy()

test_cases = case_manifest[
    case_manifest["split"].astype(str).str.lower()
    == "test"
].copy()

test_cases = test_cases.sort_values(
    "case_id"
).reset_index(drop=True)

print(
    "NORMAL images:",
    len(manifest)
)

print(
    "Test cases:",
    len(test_cases)
)


# ============================================================
# 3. BUILD TEST RECORDS
# ============================================================

case_records = []

for _, row in test_cases.iterrows():

    case_id = row["case_id"]

    imgs = manifest[
        manifest["case_id"] == case_id
    ].sort_values(
        "image_path"
    )

    image_paths = imgs[
        "image_path"
    ].tolist()[:MAX_IMAGES]

    if len(image_paths) == 0:
        continue

    case_records.append({
        "case_id": case_id,
        "patient_group_id": row[
            "patient_group_id"
        ],
        "target_text": row[
            "ket_luan"
        ],
        "image_paths": image_paths,
    })

print(
    "Cases with NORMAL images:",
    len(case_records)
)


# ============================================================
# 4. TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL_NAME,
    use_fast=False
)


# ============================================================
# 5. TRANSFORM
# ============================================================

transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    ),
])


# ============================================================
# 6. DATASET
# ============================================================

class TestCaseDataset(Dataset):

    def __init__(
        self,
        records,
        tokenizer,
        transform,
        max_length
    ):

        self.records = records
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):

        record = self.records[idx]

        image_tensors = []

        for path in record["image_paths"]:

            img = Image.open(
                path
            ).convert("RGB")

            image_tensors.append(
                self.transform(img)
            )

        images = torch.stack(
            image_tensors,
            dim=0
        )

        encoded = self.tokenizer(
            record["target_text"],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        labels = encoded.input_ids.squeeze(0)

        labels[
            labels == tokenizer.pad_token_id
        ] = -100

        return {
            "case_id":
                record["case_id"],

            "patient_group_id":
                record["patient_group_id"],

            "images":
                images,

            "labels":
                labels,

            "target_text":
                record["target_text"],
        }


# ============================================================
# 7. COLLATE
# ============================================================

def collate_fn(batch):

    max_n = max(
        item["images"].shape[0]
        for item in batch
    )

    B = len(batch)

    padded_images = torch.zeros(
        B,
        max_n,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE,
        dtype=torch.float32
    )

    image_mask = torch.zeros(
        B,
        max_n,
        dtype=torch.long
    )

    labels = torch.stack([
        item["labels"]
        for item in batch
    ])

    case_ids = []
    patient_ids = []
    targets = []

    for i, item in enumerate(batch):

        n = item["images"].shape[0]

        padded_images[
            i,
            :n
        ] = item["images"]

        image_mask[
            i,
            :n
        ] = 1

        case_ids.append(
            item["case_id"]
        )

        patient_ids.append(
            item["patient_group_id"]
        )

        targets.append(
            item["target_text"]
        )

    return {
        "case_id":
            case_ids,

        "patient_group_id":
            patient_ids,

        "images":
            padded_images,

        "image_mask":
            image_mask,

        "labels":
            labels,

        "target_text":
            targets,
    }


test_dataset = TestCaseDataset(
    case_records,
    tokenizer,
    transform,
    MAX_TARGET_LENGTH
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn
)

print(
    "Dataset:",
    len(test_dataset)
)

print(
    "Batches:",
    len(test_loader)
)


# ============================================================
# 8. PROJECTOR
# ============================================================

class VisualProjector(nn.Module):

    def __init__(
        self,
        input_dim=768,
        output_dim=512
    ):

        super().__init__()

        self.proj = nn.Linear(
            input_dim,
            output_dim
        )

    def forward(self, x):

        return self.proj(x)


# ============================================================
# 9. LOAD V1
# ============================================================

print("\n" + "-" * 70)
print("LOADING V1")
print("-" * 70)

v1_ckpt = torch.load(
    V1_CKPT,
    map_location="cpu",
    weights_only=False
)

v1_vit = ViTModel.from_pretrained(
    VISION_MODEL_NAME
)

v1_proj = VisualProjector()

v1_mt5 = MT5ForConditionalGeneration.from_pretrained(
    TEXT_MODEL_NAME
)

v1_proj.load_state_dict(
    v1_ckpt["visual_projector"],
    strict=True
)

v1_mt5.load_state_dict(
    v1_ckpt["mt5_model"],
    strict=True
)

v1_vit = v1_vit.to(
    device
).eval()

v1_proj = v1_proj.to(
    device
).eval()

v1_mt5 = v1_mt5.to(
    device
).eval()

print("V1 loaded.")


# ============================================================
# 10. LOAD V2
# ============================================================

print("\n" + "-" * 70)
print("LOADING V2")
print("-" * 70)

v2_ckpt = torch.load(
    V2_CKPT,
    map_location="cpu",
    weights_only=False
)

v2_vit = ViTModel.from_pretrained(
    VISION_MODEL_NAME
)

v2_vit.load_state_dict(
    v2_ckpt["vit_state_dict"],
    strict=True
)

v2_proj = VisualProjector()

v2_proj.load_state_dict(
    v2_ckpt["projector_state_dict"],
    strict=True
)

v2_mt5 = MT5ForConditionalGeneration.from_pretrained(
    TEXT_MODEL_NAME
)

v2_mt5.load_state_dict(
    v2_ckpt["mt5_state_dict"],
    strict=True
)

v2_vit = v2_vit.to(
    device
).eval()

v2_proj = v2_proj.to(
    device
).eval()

v2_mt5 = v2_mt5.to(
    device
).eval()

print("V2 loaded.")


# ============================================================
# 11. VISUAL PREFIX
# ============================================================

@torch.no_grad()
def make_visual_prefix(
    vit,
    projector,
    images
):

    B, N, C, H, W = images.shape

    flat_images = images.reshape(
        B * N,
        C,
        H,
        W
    )

    vit_output = vit(
        pixel_values=flat_images
    )

    cls = vit_output.last_hidden_state[
        :,
        0,
        :
    ]

    projected = projector(
        cls
    )

    prefix = projected.reshape(
        B,
        N,
        -1
    )

    return prefix


# ============================================================
# 12. EVALUATE MODEL
# ============================================================

@torch.no_grad()
def evaluate_model(
    vit,
    projector,
    mt5,
    loader,
    model_name
):

    results = []

    total_batches = len(loader)

    print(
        f"\nEvaluating {model_name}..."
    )

    for batch_idx, batch in enumerate(
        loader,
        start=1
    ):

        images = batch[
            "images"
        ].to(
            device,
            non_blocking=True
        )

        image_mask = batch[
            "image_mask"
        ].to(
            device,
            non_blocking=True
        )

        labels = batch[
            "labels"
        ].to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Visual prefix
        # ----------------------------------------------------

        prefix = make_visual_prefix(
            vit,
            projector,
            images
        )

        # ----------------------------------------------------
        # mT5
        # ----------------------------------------------------

        encoder_outputs = mt5.encoder(
            inputs_embeds=prefix,
            return_dict=True
        )

        outputs = mt5(
            encoder_outputs=encoder_outputs,
            attention_mask=image_mask,
            labels=labels,
            return_dict=True
        )

        # This is the official HF loss.
        batch_loss = outputs.loss

        # ----------------------------------------------------
        # IMPORTANT:
        #
        # DO NOT SHIFT logits/labels.
        #
        # HF's T5 implementation already handles
        # decoder shifting internally.
        # ----------------------------------------------------

        logits = outputs.logits.float()

        flat_loss = F.cross_entropy(
            logits.reshape(
                -1,
                logits.shape[-1]
            ),
            labels.reshape(-1),
            ignore_index=-100,
            reduction="none"
        )

        token_loss = flat_loss.reshape(
            labels.shape
        )

        valid_tokens = (
            labels != -100
        )

        per_case_loss = (
            token_loss.sum(dim=1)
            /
            valid_tokens.sum(dim=1).clamp(min=1)
        )

        # ----------------------------------------------------
        # Sanity check:
        #
        # weighted average of per-case losses should equal
        # HF loss only if cases have same valid-token count.
        #
        # Therefore also calculate global token average.
        # ----------------------------------------------------

        global_loss = (
            token_loss.sum()
            /
            valid_tokens.sum().clamp(min=1)
        )

        if batch_idx == 1:

            print(
                f"\n{model_name} first batch:"
            )

            print(
                "  HF batch loss:",
                batch_loss.item()
            )

            print(
                "  Manual global loss:",
                global_loss.item()
            )

            print(
                "  Difference:",
                abs(
                    batch_loss.item()
                    -
                    global_loss.item()
                )
            )

            if abs(
                batch_loss.item()
                -
                global_loss.item()
            ) > 1e-5:

                raise RuntimeError(
                    "Loss verification failed."
                )

        # ----------------------------------------------------
        # Save per-case
        # ----------------------------------------------------

        for i in range(
            len(batch["case_id"])
        ):

            results.append({

                "case_id":
                    batch["case_id"][i],

                "patient_group_id":
                    batch["patient_group_id"][i],

                "target_text":
                    batch["target_text"][i],

                f"{model_name}_loss":
                    float(
                        per_case_loss[i].item()
                    ),

                "num_images":
                    int(
                        image_mask[i].sum().item()
                    ),

                "target_tokens":
                    int(
                        valid_tokens[i].sum().item()
                    ),
            })

        if (
            batch_idx == 1
            or batch_idx % 25 == 0
            or batch_idx == total_batches
        ):

            print(
                f"  Batch "
                f"{batch_idx}/{total_batches}"
                f" | loss={batch_loss.item():.4f}"
            )

    return pd.DataFrame(results)


# ============================================================
# 13. V1
# ============================================================

v1_results = evaluate_model(
    v1_vit,
    v1_proj,
    v1_mt5,
    test_loader,
    "v1"
)

torch.cuda.empty_cache()
gc.collect()


# ============================================================
# 14. V2
# ============================================================

v2_results = evaluate_model(
    v2_vit,
    v2_proj,
    v2_mt5,
    test_loader,
    "v2"
)


# ============================================================
# 15. MERGE
# ============================================================

results = v1_results.merge(
    v2_results[
        [
            "case_id",
            "v2_loss"
        ]
    ],
    on="case_id",
    how="inner"
)

results["loss_difference_v2_minus_v1"] = (
    results["v2_loss"]
    -
    results["v1_loss"]
)

results["v2_better"] = (
    results["v2_loss"]
    <
    results["v1_loss"]
)

results["v1_better"] = (
    results["v1_loss"]
    <
    results["v2_loss"]
)

results["target_char_length"] = (
    results["target_text"]
    .astype(str)
    .str.len()
)


# ============================================================
# 16. SUMMARY
# ============================================================

n = len(results)

v1_mean = results[
    "v1_loss"
].mean()

v2_mean = results[
    "v2_loss"
].mean()

v1_median = results[
    "v1_loss"
].median()

v2_median = results[
    "v2_loss"
].median()

v1_std = results[
    "v1_loss"
].std()

v2_std = results[
    "v2_loss"
].std()

mean_diff = results[
    "loss_difference_v2_minus_v1"
].mean()

median_diff = results[
    "loss_difference_v2_minus_v1"
].median()

v2_better = int(
    results["v2_better"].sum()
)

v1_better = int(
    results["v1_better"].sum()
)

v2_pct = (
    v2_better
    /
    n
    *
    100
)

v1_pct = (
    v1_better
    /
    n
    *
    100
)

relative_improvement = (
    (v1_mean - v2_mean)
    /
    v1_mean
    *
    100
)


# ============================================================
# 17. SAVE
# ============================================================

results.to_csv(
    OUTPUT_CSV,
    index=False
)

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "V2-DIAGNOSTIC-3C\n"
    )

    f.write(
        "=" * 70 + "\n\n"
    )

    f.write(
        f"Test cases: {n}\n\n"
    )

    f.write(
        f"V1 mean loss: {v1_mean:.8f}\n"
    )

    f.write(
        f"V1 median loss: {v1_median:.8f}\n"
    )

    f.write(
        f"V1 std: {v1_std:.8f}\n\n"
    )

    f.write(
        f"V2 mean loss: {v2_mean:.8f}\n"
    )

    f.write(
        f"V2 median loss: {v2_median:.8f}\n"
    )

    f.write(
        f"V2 std: {v2_std:.8f}\n\n"
    )

    f.write(
        f"Mean V2-V1: {mean_diff:.8f}\n"
    )

    f.write(
        f"Median V2-V1: {median_diff:.8f}\n"
    )

    f.write(
        f"Relative improvement: "
        f"{relative_improvement:.4f}%\n\n"
    )

    f.write(
        f"V2 lower loss: "
        f"{v2_better}/{n} "
        f"({v2_pct:.2f}%)\n"
    )

    f.write(
        f"V1 lower loss: "
        f"{v1_better}/{n} "
        f"({v1_pct:.2f}%)\n"
    )


# ============================================================
# 18. PRINT
# ============================================================

print("\n")
print("=" * 70)
print("V2-DIAGNOSTIC-3C SUMMARY")
print("=" * 70)

print(
    "\nTest cases:",
    n
)

print("\nTeacher-forced loss:")

print(
    f"  V1 mean   : {v1_mean:.6f}"
)

print(
    f"  V1 median : {v1_median:.6f}"
)

print(
    f"  V1 std    : {v1_std:.6f}"
)

print()

print(
    f"  V2 mean   : {v2_mean:.6f}"
)

print(
    f"  V2 median : {v2_median:.6f}"
)

print(
    f"  V2 std    : {v2_std:.6f}"
)

print("\nV2 - V1:")

print(
    f"  Mean difference   : {mean_diff:.6f}"
)

print(
    f"  Median difference : {median_diff:.6f}"
)

print(
    f"  Relative improvement: "
    f"{relative_improvement:.2f}%"
)

print("\nCase-by-case:")

print(
    f"  V2 lower: "
    f"{v2_better}/{n} "
    f"({v2_pct:.2f}%)"
)

print(
    f"  V1 lower: "
    f"{v1_better}/{n} "
    f"({v1_pct:.2f}%)"
)

print("\nSaved:")
print(
    OUTPUT_CSV
)

print(
    SUMMARY_FILE
)

print("\n" + "=" * 70)
print("DIAGNOSTIC-3C COMPLETE")
print("=" * 70)

V2-DIAGNOSTIC-3C — CORRECT FULL TEST LOSS

Device: cuda
GPU: NVIDIA A100-SXM4-40GB

----------------------------------------------------------------------
LOADING DATA
----------------------------------------------------------------------
NORMAL images: 76216
Test cases: 712
Cases with NORMAL images: 712
Dataset: 712
Batches: 178

----------------------------------------------------------------------
LOADING V1
----------------------------------------------------------------------


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


...

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


V2 loaded.

Evaluating v1...

v1 first batch:
  HF batch loss: 1.1910157203674316
  Manual global loss: 1.191015601158142
  Difference: 1.1920928955078125e-07
  Batch 1/178 | loss=1.1910
  Batch 25/178 | loss=0.8155
  Batch 50/178 | loss=1.0179
  Batch 75/178 | loss=0.8904
  Batch 100/178 | loss=0.6255
  Batch 125/178 | loss=0.3032
  Batch 150/178 | loss=0.2320
  Batch 175/178 | loss=0.2034
  Batch 178/178 | loss=0.4971

Evaluating v2...

v2 first batch:
  HF batch loss: 1.140355110168457
  Manual global loss: 1.140355110168457
  Difference: 0.0
  Batch 1/178 | loss=1.1404
  Batch 25/178 | loss=0.7720
  Batch 50/178 | loss=1.0375
  Batch 75/178 | loss=0.7242
  Batch 100/178 | loss=0.6549
  Batch 125/178 | loss=0.2989
  Batch 150/178 | loss=0.2191
  Batch 175/178 | loss=0.1882
  Batch 178/178 | loss=0.4747


V2-DIAGNOSTIC-3C SUMMARY

Test cases: 712

Teacher-forced loss:
  V1 mean   : 0.554132
  V1 median : 0.339786
  V1 std    : 0.665096

  V2 mean   : 0.528674
  V2 median : 0.328596
 

In [17]:
# ============================================================
# V2-GENERATION-EVAL
# V2 BASELINE — FULL TEST GENERATION
#
# Runtime-independent
# NO TRAINING
# NO CHECKPOINT MODIFICATION
# ============================================================

import os
import gc
import re
import math
import difflib
import collections
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

from transformers import (
    ViTModel,
    MT5ForConditionalGeneration,
    AutoTokenizer,
)

print("=" * 70)
print("V2-GENERATION-EVAL — FULL TEST SET")
print("=" * 70)


# ============================================================
# 1. CONFIG
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/NoiSoi_Matching"

V2_DIR = os.path.join(
    PROJECT_DIR,
    "baseline_model_v2"
)

V2_CKPT = os.path.join(
    V2_DIR,
    "checkpoints",
    "best.pt"
)

MANIFEST_FILE = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

CASE_MANIFEST_FILE = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_case_manifest.csv"
)

SPLIT_FILE = os.path.join(
    PROJECT_DIR,
    "patient_split",
    "cases_with_split.csv"
)

OUTPUT_DIR = os.path.join(
    V2_DIR,
    "generation_eval"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

PREDICTION_FILE = os.path.join(
    OUTPUT_DIR,
    "test_predictions_v2_correct.csv"
)

METRICS_FILE = os.path.join(
    OUTPUT_DIR,
    "generation_metrics_v2.csv"
)

SUMMARY_FILE = os.path.join(
    OUTPUT_DIR,
    "generation_summary_v2.txt"
)

FREQUENCY_FILE = os.path.join(
    OUTPUT_DIR,
    "prediction_frequency_v2.csv"
)

VISION_MODEL_NAME = (
    "google/vit-base-patch16-224"
)

TEXT_MODEL_NAME = (
    "google/mt5-small"
)

IMAGE_SIZE = 224
MAX_IMAGES = 8
MAX_TARGET_LENGTH = 96
BATCH_SIZE = 4

# Generation settings
NUM_BEAMS = 1
DO_SAMPLE = False
MAX_NEW_TOKENS = 96

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# 2. CHECK FILES
# ============================================================

required_files = [
    V2_CKPT,
    MANIFEST_FILE,
    CASE_MANIFEST_FILE,
    SPLIT_FILE,
]

for path in required_files:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"Missing required file:\n{path}"
        )

print("\nAll source files found.")
print("Checkpoint:", V2_CKPT)


# ============================================================
# 3. LOAD DATA
# ============================================================

print("\n" + "-" * 70)
print("LOADING DATA")
print("-" * 70)

manifest = pd.read_csv(
    MANIFEST_FILE,
    low_memory=False
)

case_manifest = pd.read_csv(
    CASE_MANIFEST_FILE,
    low_memory=False
)

split_df = pd.read_csv(
    SPLIT_FILE,
    low_memory=False
)

# ------------------------------------------------------------
# NORMAL images only
# ------------------------------------------------------------

manifest["image_status"] = (
    manifest["image_status"]
    .fillna("")
    .astype(str)
    .str.upper()
)

manifest = manifest[
    manifest["image_status"] == "NORMAL"
].copy()

# ------------------------------------------------------------
# Make sure split exists
# ------------------------------------------------------------

if "split" not in case_manifest.columns:

    case_manifest = case_manifest.merge(
        split_df[
            ["case_id", "split"]
        ],
        on="case_id",
        how="left"
    )

# ------------------------------------------------------------
# Target
# ------------------------------------------------------------

case_manifest["ket_luan"] = (
    case_manifest["ket_luan"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Only non-empty targets
case_manifest = case_manifest[
    case_manifest["ket_luan"] != ""
].copy()

# ------------------------------------------------------------
# Test set
# ------------------------------------------------------------

test_cases = case_manifest[
    case_manifest["split"]
    .astype(str)
    .str.lower()
    == "test"
].copy()

test_cases = (
    test_cases
    .sort_values("case_id")
    .reset_index(drop=True)
)

print(
    "NORMAL images:",
    len(manifest)
)

print(
    "Test cases:",
    len(test_cases)
)


# ============================================================
# 4. BUILD CASE RECORDS
# ============================================================

case_records = []

for _, row in test_cases.iterrows():

    case_id = row["case_id"]

    imgs = manifest[
        manifest["case_id"] == case_id
    ].sort_values(
        "image_path"
    )

    image_paths = imgs[
        "image_path"
    ].tolist()

    if len(image_paths) == 0:
        continue

    # Same deterministic first-8-image policy
    image_paths = image_paths[
        :MAX_IMAGES
    ]

    case_records.append({

        "case_id":
            case_id,

        "patient_group_id":
            row["patient_group_id"],

        "target_text":
            row["ket_luan"],

        "image_paths":
            image_paths,
    })

print(
    "Test cases with images:",
    len(case_records)
)


# ============================================================
# 5. TOKENIZER
# ============================================================

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL_NAME,
    use_fast=False
)

print(
    "Vocab size:",
    tokenizer.vocab_size
)


# ============================================================
# 6. IMAGE TRANSFORM
# ============================================================

transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    ),
])


# ============================================================
# 7. DATASET
# ============================================================

class TestGenerationDataset(Dataset):

    def __init__(
        self,
        records,
        transform
    ):

        self.records = records
        self.transform = transform

    def __len__(self):

        return len(self.records)

    def __getitem__(self, idx):

        record = self.records[idx]

        images = []

        for path in record["image_paths"]:

            img = Image.open(
                path
            ).convert("RGB")

            images.append(
                self.transform(img)
            )

        images = torch.stack(
            images,
            dim=0
        )

        return {

            "case_id":
                record["case_id"],

            "patient_group_id":
                record["patient_group_id"],

            "images":
                images,

            "target_text":
                record["target_text"],
        }


# ============================================================
# 8. COLLATE
# ============================================================

def collate_fn(batch):

    max_n = max(
        x["images"].shape[0]
        for x in batch
    )

    B = len(batch)

    padded_images = torch.zeros(
        B,
        max_n,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE,
        dtype=torch.float32
    )

    image_mask = torch.zeros(
        B,
        max_n,
        dtype=torch.long
    )

    case_ids = []
    patient_ids = []
    targets = []

    for i, item in enumerate(batch):

        n = item["images"].shape[0]

        padded_images[
            i,
            :n
        ] = item["images"]

        image_mask[
            i,
            :n
        ] = 1

        case_ids.append(
            item["case_id"]
        )

        patient_ids.append(
            item["patient_group_id"]
        )

        targets.append(
            item["target_text"]
        )

    return {

        "case_id":
            case_ids,

        "patient_group_id":
            patient_ids,

        "images":
            padded_images,

        "image_mask":
            image_mask,

        "target_text":
            targets,
    }


test_dataset = TestGenerationDataset(
    case_records,
    transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn
)

print(
    "Dataset:",
    len(test_dataset)
)

print(
    "Batches:",
    len(test_loader)
)


# ============================================================
# 9. VISUAL PROJECTOR
# ============================================================

class VisualProjector(nn.Module):

    def __init__(
        self,
        input_dim=768,
        output_dim=512
    ):

        super().__init__()

        self.proj = nn.Linear(
            input_dim,
            output_dim
        )

    def forward(self, x):

        return self.proj(x)


# ============================================================
# 10. LOAD V2
# ============================================================

print("\n" + "-" * 70)
print("LOADING V2 CHECKPOINT")
print("-" * 70)

checkpoint = torch.load(
    V2_CKPT,
    map_location="cpu",
    weights_only=False
)

print(
    "Checkpoint epoch:",
    checkpoint.get(
        "epoch",
        "N/A"
    )
)

print(
    "Best validation loss:",
    checkpoint.get(
        "best_val_loss",
        "N/A"
    )
)


# ------------------------------------------------------------
# ViT
# ------------------------------------------------------------

vit = ViTModel.from_pretrained(
    VISION_MODEL_NAME
)

vit.load_state_dict(
    checkpoint["vit_state_dict"],
    strict=True
)


# ------------------------------------------------------------
# Projector
# ------------------------------------------------------------

projector = VisualProjector()

projector.load_state_dict(
    checkpoint[
        "projector_state_dict"
    ],
    strict=True
)


# ------------------------------------------------------------
# mT5
# ------------------------------------------------------------

mt5 = MT5ForConditionalGeneration.from_pretrained(
    TEXT_MODEL_NAME
)

mt5.load_state_dict(
    checkpoint[
        "mt5_state_dict"
    ],
    strict=True
)


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

vit = vit.to(
    device
).eval()

projector = projector.to(
    device
).eval()

mt5 = mt5.to(
    device
).eval()

print("V2 loaded successfully.")


# ============================================================
# 11. VISUAL PREFIX
# ============================================================

@torch.no_grad()
def make_visual_prefix(
    images
):

    B, N, C, H, W = images.shape

    flat_images = images.reshape(
        B * N,
        C,
        H,
        W
    )

    vit_output = vit(
        pixel_values=flat_images
    )

    # CLS token
    cls = vit_output.last_hidden_state[
        :,
        0,
        :
    ]

    projected = projector(
        cls
    )

    prefix = projected.reshape(
        B,
        N,
        -1
    )

    return prefix


# ============================================================
# 12. GENERATION FUNCTION
# ============================================================

@torch.no_grad()
def generate_batch(
    images,
    image_mask
):

    prefix = make_visual_prefix(
        images
    )

    encoder_outputs = mt5.encoder(
        inputs_embeds=prefix,
        return_dict=True
    )

    generated_ids = mt5.generate(

        encoder_outputs=encoder_outputs,

        attention_mask=image_mask,

        max_new_tokens=MAX_NEW_TOKENS,

        num_beams=NUM_BEAMS,

        do_sample=DO_SAMPLE,

        early_stopping=True,

        pad_token_id=tokenizer.pad_token_id,

        eos_token_id=tokenizer.eos_token_id,
    )

    texts = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )

    texts = [
        x.strip()
        for x in texts
    ]

    return texts


# ============================================================
# 13. GENERATE ALL TEST CASES
# ============================================================

print("\n" + "-" * 70)
print("GENERATING TEST PREDICTIONS")
print("-" * 70)

predictions = []

total_batches = len(
    test_loader
)

for batch_idx, batch in enumerate(
    test_loader,
    start=1
):

    images = batch[
        "images"
    ].to(
        device,
        non_blocking=True
    )

    image_mask = batch[
        "image_mask"
    ].to(
        device,
        non_blocking=True
    )

    texts = generate_batch(
        images,
        image_mask
    )

    for i in range(
        len(texts)
    ):

        predictions.append({

            "case_id":
                batch["case_id"][i],

            "patient_group_id":
                batch["patient_group_id"][i],

            "ground_truth":
                batch["target_text"][i],

            "prediction":
                texts[i],

            "num_images":
                int(
                    image_mask[i].sum().item()
                ),
        })

    if (
        batch_idx == 1
        or batch_idx % 25 == 0
        or batch_idx == total_batches
    ):

        print(
            f"Batch "
            f"{batch_idx}/{total_batches}"
        )

        if batch_idx == 1:

            print(
                "Example GT:",
                batch["target_text"][0]
            )

            print(
                "Example prediction:",
                texts[0]
            )


pred_df = pd.DataFrame(
    predictions
)

print(
    "\nGenerated rows:",
    len(pred_df)
)


# ============================================================
# 14. TEXT NORMALIZATION
# ============================================================

def normalize_text(text):

    text = str(text).upper()

    text = text.strip()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text


pred_df[
    "ground_truth_norm"
] = pred_df[
    "ground_truth"
].map(
    normalize_text
)

pred_df[
    "prediction_norm"
] = pred_df[
    "prediction"
].map(
    normalize_text
)


# ============================================================
# 15. EXACT MATCH
# ============================================================

pred_df[
    "exact_match"
] = (
    pred_df["ground_truth_norm"]
    ==
    pred_df["prediction_norm"]
)

exact_match = (
    pred_df["exact_match"].mean()
    * 100
)


# ============================================================
# 16. CHARACTER SIMILARITY
# ============================================================

def char_similarity(
    a,
    b
):

    return difflib.SequenceMatcher(
        None,
        str(a),
        str(b)
    ).ratio()


pred_df[
    "char_similarity"
] = pred_df.apply(
    lambda row:
        char_similarity(
            row["ground_truth_norm"],
            row["prediction_norm"]
        ),
    axis=1
)


# ============================================================
# 17. TOKEN F1
# ============================================================

def token_f1(
    reference,
    prediction
):

    ref_tokens = normalize_text(
        reference
    ).split()

    pred_tokens = normalize_text(
        prediction
    ).split()

    if (
        len(ref_tokens) == 0
        and len(pred_tokens) == 0
    ):

        return 1.0

    if (
        len(ref_tokens) == 0
        or len(pred_tokens) == 0
    ):

        return 0.0

    ref_counter = collections.Counter(
        ref_tokens
    )

    pred_counter = collections.Counter(
        pred_tokens
    )

    common = sum(
        (
            ref_counter
            &
            pred_counter
        ).values()
    )

    if common == 0:
        return 0.0

    precision = (
        common
        /
        len(pred_tokens)
    )

    recall = (
        common
        /
        len(ref_tokens)
    )

    if (
        precision + recall
        == 0
    ):

        return 0.0

    return (
        2
        *
        precision
        *
        recall
        /
        (
            precision
            +
            recall
        )
    )


pred_df[
    "token_f1"
] = pred_df.apply(
    lambda row:
        token_f1(
            row["ground_truth_norm"],
            row["prediction_norm"]
        ),
    axis=1
)


# ============================================================
# 18. BLEU
# ============================================================

try:

    from nltk.translate.bleu_score import (
        sentence_bleu,
        SmoothingFunction
    )

    smoother = (
        SmoothingFunction()
        .method1
    )

    def calculate_bleu(
        reference,
        prediction
    ):

        ref = normalize_text(
            reference
        ).split()

        pred = normalize_text(
            prediction
        ).split()

        if len(pred) == 0:
            return 0.0

        if len(ref) == 0:
            return 0.0

        return sentence_bleu(
            [ref],
            pred,
            smoothing_function=smoother
        )

    pred_df[
        "bleu"
    ] = pred_df.apply(
        lambda row:
            calculate_bleu(
                row["ground_truth_norm"],
                row["prediction_norm"]
            ),
        axis=1
    )

    bleu_available = True

except Exception as e:

    print(
        "\nBLEU unavailable:",
        e
    )

    pred_df[
        "bleu"
    ] = np.nan

    bleu_available = False


# ============================================================
# 19. ROUGE-L
# ============================================================

try:

    from rouge_score import rouge_scorer

    rouge = rouge_scorer.RougeScorer(
        ["rougeL"],
        use_stemmer=False
    )

    def calculate_rouge_l(
        reference,
        prediction
    ):

        score = rouge.score(
            normalize_text(reference),
            normalize_text(prediction)
        )

        return score[
            "rougeL"
        ].fmeasure

    pred_df[
        "rouge_l"
    ] = pred_df.apply(
        lambda row:
            calculate_rouge_l(
                row["ground_truth_norm"],
                row["prediction_norm"]
            ),
        axis=1
    )

    rouge_available = True

except Exception as e:

    print(
        "\nROUGE-L unavailable:",
        e
    )

    pred_df[
        "rouge_l"
    ] = np.nan

    rouge_available = False


# ============================================================
# 20. SAVE PREDICTIONS
# ============================================================

pred_df.to_csv(
    PREDICTION_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    "\nPredictions saved:",
    PREDICTION_FILE
)


# ============================================================
# 21. METRICS
# ============================================================

unique_predictions = (
    pred_df[
        "prediction_norm"
    ]
    .nunique()
)

diversity_ratio = (
    unique_predictions
    /
    len(pred_df)
)


mean_char_similarity = (
    pred_df[
        "char_similarity"
    ].mean()
)

median_char_similarity = (
    pred_df[
        "char_similarity"
    ].median()
)

mean_token_f1 = (
    pred_df[
        "token_f1"
    ].mean()
)

median_token_f1 = (
    pred_df[
        "token_f1"
    ].median()
)

mean_bleu = (
    pred_df[
        "bleu"
    ].mean()
)

median_bleu = (
    pred_df[
        "bleu"
    ].median()
)

if rouge_available:

    mean_rouge_l = (
        pred_df[
            "rouge_l"
        ].mean()
    )

    median_rouge_l = (
        pred_df[
            "rouge_l"
        ].median()
    )

else:

    mean_rouge_l = np.nan
    median_rouge_l = np.nan


metrics = pd.DataFrame([{

    "model":
        "V2",

    "test_cases":
        len(pred_df),

    "exact_match_pct":
        exact_match,

    "mean_char_similarity":
        mean_char_similarity,

    "median_char_similarity":
        median_char_similarity,

    "mean_token_f1":
        mean_token_f1,

    "median_token_f1":
        median_token_f1,

    "mean_bleu":
        mean_bleu,

    "median_bleu":
        median_bleu,

    "mean_rouge_l":
        mean_rouge_l,

    "median_rouge_l":
        median_rouge_l,

    "unique_predictions":
        unique_predictions,

    "diversity_ratio":
        diversity_ratio,
}])


metrics.to_csv(
    METRICS_FILE,
    index=False
)


# ============================================================
# 22. PREDICTION FREQUENCY
# ============================================================

frequency = (
    pred_df[
        "prediction_norm"
    ]
    .value_counts()
    .reset_index()
)

frequency.columns = [
    "prediction",
    "count"
]

frequency[
    "percentage"
] = (
    frequency["count"]
    /
    len(pred_df)
    *
    100
)

frequency.to_csv(
    FREQUENCY_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 23. SUMMARY
# ============================================================

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "V2 GENERATION EVALUATION\n"
    )

    f.write(
        "=" * 70 + "\n\n"
    )

    f.write(
        f"Checkpoint epoch: "
        f"{checkpoint.get('epoch', 'N/A')}\n"
    )

    f.write(
        f"Best validation loss: "
        f"{checkpoint.get('best_val_loss', 'N/A')}\n\n"
    )

    f.write(
        f"Test cases: "
        f"{len(pred_df)}\n\n"
    )

    f.write(
        f"Exact match: "
        f"{exact_match:.4f}%\n"
    )

    f.write(
        f"Mean char similarity: "
        f"{mean_char_similarity:.6f}\n"
    )

    f.write(
        f"Median char similarity: "
        f"{median_char_similarity:.6f}\n"
    )

    f.write(
        f"Mean token F1: "
        f"{mean_token_f1:.6f}\n"
    )

    f.write(
        f"Median token F1: "
        f"{median_token_f1:.6f}\n"
    )

    f.write(
        f"Mean BLEU: "
        f"{mean_bleu:.6f}\n"
    )

    f.write(
        f"Median BLEU: "
        f"{median_bleu:.6f}\n"
    )

    if rouge_available:

        f.write(
            f"Mean ROUGE-L: "
            f"{mean_rouge_l:.6f}\n"
        )

        f.write(
            f"Median ROUGE-L: "
            f"{median_rouge_l:.6f}\n"
        )

    f.write(
        f"\nUnique predictions: "
        f"{unique_predictions}\n"
    )

    f.write(
        f"Diversity ratio: "
        f"{diversity_ratio:.6f}\n"
    )


# ============================================================
# 24. PRINT RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("V2 GENERATION RESULTS")
print("=" * 70)

print(
    f"\nTest cases: {len(pred_df)}"
)

print(
    f"\nExact match: "
    f"{exact_match:.4f}%"
)

print(
    f"Mean char similarity: "
    f"{mean_char_similarity:.4f}"
)

print(
    f"Median char similarity: "
    f"{median_char_similarity:.4f}"
)

print(
    f"Mean token F1: "
    f"{mean_token_f1:.4f}"
)

print(
    f"Median token F1: "
    f"{median_token_f1:.4f}"
)

print(
    f"Mean BLEU: "
    f"{mean_bleu:.4f}"
)

print(
    f"Median BLEU: "
    f"{median_bleu:.4f}"
)

if rouge_available:

    print(
        f"Mean ROUGE-L: "
        f"{mean_rouge_l:.4f}"
    )

    print(
        f"Median ROUGE-L: "
        f"{median_rouge_l:.4f}"
    )

print(
    f"\nUnique predictions: "
    f"{unique_predictions}"
)

print(
    f"Diversity ratio: "
    f"{diversity_ratio:.4f}"
)


# ============================================================
# 25. TOP PREDICTIONS
# ============================================================

print("\n" + "-" * 70)
print("TOP PREDICTIONS")
print("-" * 70)

print(
    frequency.head(20).to_string(
        index=False
    )
)


# ============================================================
# 26. EXAMPLES
# ============================================================

print("\n" + "-" * 70)
print("EXAMPLE PREDICTIONS")
print("-" * 70)

for i in range(
    min(20, len(pred_df))
):

    row = pred_df.iloc[i]

    print(
        f"\n[{i+1}] "
        f"{row['case_id']}"
    )

    print(
        "GT   :",
        row["ground_truth"]
    )

    print(
        "PRED :",
        row["prediction"]
    )

    print(
        "F1   :",
        f"{row['token_f1']:.4f}"
    )


# ============================================================
# 27. FINAL CHECKS
# ============================================================

assert len(pred_df) == 712, (
    f"Expected 712 predictions, "
    f"got {len(pred_df)}"
)

assert (
    pred_df["case_id"].nunique()
    == 712
), "Duplicate or missing case IDs."

assert (
    pred_df["prediction"].notna().all()
), "Missing predictions."

print("\n" + "=" * 70)
print("V2 GENERATION EVAL COMPLETE — PASS")
print("=" * 70)

print(
    "\nSaved files:"
)

print(
    PREDICTION_FILE
)

print(
    METRICS_FILE
)

print(
    FREQUENCY_FILE
)

print(
    SUMMARY_FILE
)

# Cleanup
gc.collect()
torch.cuda.empty_cache()

V2-GENERATION-EVAL — FULL TEST SET

Device: cuda
GPU: NVIDIA A100-SXM4-40GB

All source files found.
Checkpoint: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v2/checkpoints/best.pt

----------------------------------------------------------------------
LOADING DATA
----------------------------------------------------------------------
NORMAL images: 76216
Test cases: 712
Test cases with images: 712

Loading tokenizer...
Vocab size: 250100
Dataset: 712
Batches: 178

----------------------------------------------------------------------
LOADING V2 CHECKPOINT
----------------------------------------------------------------------
Checkpoint epoch: 5
Best validation loss: 0.601288303772293


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


V2 loaded successfully.

----------------------------------------------------------------------
GENERATING TEST PREDICTIONS
----------------------------------------------------------------------


[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Batch 1/178
Example GT: VIÊM MÀNG NHĨ CẤP BÓNG NƯỚC P - NHIỀU RÁY TAI T
Example prediction: VIÊM MŨI XOANG
Batch 25/178
Batch 50/178
Batch 75/178
Batch 100/178
Batch 125/178
Batch 150/178
Batch 175/178
Batch 178/178

Generated rows: 712

ROUGE-L unavailable: No module named 'rouge_score'

Predictions saved: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v2/generation_eval/test_predictions_v2_correct.csv


V2 GENERATION RESULTS

Test cases: 712

Exact match: 16.7135%
Mean char similarity: 0.5946
Median char similarity: 0.6154
Mean token F1: 0.5340
Median token F1: 0.5657
Mean BLEU: 0.2192
Median BLEU: 0.0978

Unique predictions: 5
Diversity ratio: 0.0070

----------------------------------------------------------------------
TOP PREDICTIONS
----------------------------------------------------------------------
                                  prediction  count  percentage
                                VIÊM MŨI MẠN    281   39.466292
                              VIÊM MŨI XOANG

In [18]:
# ============================================================
# V2-ERROR-ANALYSIS
# FULL TEST SET
#
# NO TRAINING
# NO CHECKPOINT MODIFICATION
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import collections
import difflib

print("=" * 70)
print("V2 ERROR ANALYSIS + LATERALITY AUDIT")
print("=" * 70)


# ============================================================
# 1. CONFIG
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/NoiSoi_Matching"

PRED_FILE = os.path.join(
    PROJECT_DIR,
    "baseline_model_v2",
    "generation_eval",
    "test_predictions_v2_correct.csv"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "baseline_model_v2",
    "error_analysis"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

if not os.path.exists(PRED_FILE):

    raise FileNotFoundError(
        PRED_FILE
    )

print(
    "\nPrediction file:",
    PRED_FILE
)


# ============================================================
# 2. LOAD
# ============================================================

df = pd.read_csv(
    PRED_FILE,
    low_memory=False
)

print(
    "Rows:",
    len(df)
)

assert len(df) == 712


# ============================================================
# 3. TEXT NORMALIZATION
# ============================================================

def norm(text):

    text = str(text).upper().strip()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text


df["gt"] = df[
    "ground_truth"
].map(norm)

df["pred"] = df[
    "prediction"
].map(norm)


# ============================================================
# 4. BASIC METRICS
# ============================================================

def char_sim(a, b):

    return difflib.SequenceMatcher(
        None,
        a,
        b
    ).ratio()


df["char_similarity"] = df.apply(
    lambda x:
        char_sim(
            x["gt"],
            x["pred"]
        ),
    axis=1
)


def token_f1(a, b):

    a_tokens = a.split()
    b_tokens = b.split()

    if not a_tokens and not b_tokens:
        return 1.0

    if not a_tokens or not b_tokens:
        return 0.0

    a_count = collections.Counter(
        a_tokens
    )

    b_count = collections.Counter(
        b_tokens
    )

    common = sum(
        (a_count & b_count).values()
    )

    if common == 0:
        return 0.0

    precision = (
        common /
        len(b_tokens)
    )

    recall = (
        common /
        len(a_tokens)
    )

    return (
        2 * precision * recall /
        (precision + recall)
    )


df["token_f1"] = df.apply(
    lambda x:
        token_f1(
            x["gt"],
            x["pred"]
        ),
    axis=1
)


# ============================================================
# 5. LATERALITY NORMALIZATION
#
# IMPORTANT:
# This is NOT saying left/right are clinically equivalent.
# It is an auxiliary diagnostic metric to identify cases
# where the main pathology is correct but laterality differs.
# ============================================================

def normalize_laterality(text):

    text = norm(text)

    # Vietnamese abbreviations
    replacements = [

        (r"\bP\b", " LATERALITY "),
        (r"\bT\b", " LATERALITY "),

        (r"\bPHẢI\b", " LATERALITY "),
        (r"\bTRÁI\b", " LATERALITY "),

        (r"\b2 BÊN\b", " LATERALITY "),
        (r"\bHAI BÊN\b", " LATERALITY "),
        (r"\bHAI TAI\b", " LATERALITY "),
    ]

    for pattern, replacement in replacements:

        text = re.sub(
            pattern,
            replacement,
            text
        )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


df["gt_no_laterality"] = df[
    "gt"
].map(
    normalize_laterality
)

df["pred_no_laterality"] = df[
    "pred"
].map(
    normalize_laterality
)

df["laterality_insensitive_exact"] = (
    df["gt_no_laterality"]
    ==
    df["pred_no_laterality"]
)


# ============================================================
# 6. DETECT LATERALITY
# ============================================================

def has_laterality(text):

    text = norm(text)

    patterns = [

        r"\bP\b",
        r"\bT\b",
        r"\bPHẢI\b",
        r"\bTRÁI\b",
        r"\b2 BÊN\b",
        r"\bHAI BÊN\b",
        r"\bHAI TAI\b",
    ]

    return any(
        re.search(
            p,
            text
        )
        for p in patterns
    )


df["gt_has_laterality"] = df[
    "gt"
].map(
    has_laterality
)

df["pred_has_laterality"] = df[
    "pred"
].map(
    has_laterality
)


# ============================================================
# 7. TOP GROUND TRUTH
# ============================================================

gt_freq = (
    df["gt"]
    .value_counts()
    .reset_index()
)

gt_freq.columns = [
    "ground_truth",
    "count"
]

gt_freq["percentage"] = (
    gt_freq["count"]
    /
    len(df)
    *
    100
)

gt_freq.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ground_truth_frequency.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 8. TOP PREDICTIONS
# ============================================================

pred_freq = (
    df["pred"]
    .value_counts()
    .reset_index()
)

pred_freq.columns = [
    "prediction",
    "count"
]

pred_freq["percentage"] = (
    pred_freq["count"]
    /
    len(df)
    *
    100
)

pred_freq.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "prediction_frequency.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 9. GT -> PRED CONFUSION
# ============================================================

confusion = (
    df.groupby(
        ["gt", "pred"]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False
    )
)

confusion.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "gt_prediction_confusion.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 10. HOW MUCH OF THE TEST SET IS COVERED
#     BY THE TOP-K PREDICTIONS?
# ============================================================

coverage_rows = []

for k in [
    1,
    2,
    3,
    5,
    10,
    20,
    50,
]:

    top_k = set(
        pred_freq.head(k)[
            "prediction"
        ]
    )

    coverage = (
        df["pred"]
        .isin(top_k)
        .mean()
        *
        100
    )

    coverage_rows.append({

        "top_k_predictions":
            k,

        "test_coverage_pct":
            coverage,
    })

coverage_df = pd.DataFrame(
    coverage_rows
)

coverage_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "prediction_coverage.csv"
    ),
    index=False
)


# ============================================================
# 11. PERFORMANCE BY LATERALITY
# ============================================================

laterality_summary = (
    df.groupby(
        "gt_has_laterality"
    )
    .agg(

        cases=(
            "case_id",
            "count"
        ),

        exact_match=(
            "gt",
            lambda x:
                np.nan
        ),

        mean_char_similarity=(
            "char_similarity",
            "mean"
        ),

        mean_token_f1=(
            "token_f1",
            "mean"
        ),

        laterality_insensitive_exact=(
            "laterality_insensitive_exact",
            "mean"
        ),
    )
    .reset_index()
)

# Correct exact match separately
laterality_summary[
    "exact_match"
] = (
    df.groupby(
        "gt_has_laterality"
    )[
        "gt"
    ]
    .apply(
        lambda x: 0
    )
    .values
)

# Recalculate accurately
for value in [False, True]:

    mask = (
        df["gt_has_laterality"]
        == value
    )

    if mask.sum() > 0:

        idx = (
            laterality_summary[
                "gt_has_laterality"
            ]
            == value
        )

        laterality_summary.loc[
            idx,
            "exact_match"
        ] = (
            df.loc[
                mask,
                "gt"
            ]
            ==
            df.loc[
                mask,
                "pred"
            ]
        ).mean()


laterality_summary[
    "exact_match"
] *= 100

laterality_summary[
    "laterality_insensitive_exact"
] *= 100

laterality_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "laterality_summary.csv"
    ),
    index=False
)


# ============================================================
# 12. LATERALITY CASES WHERE MAIN TEXT IS OTHERWISE CLOSE
# ============================================================

laterality_cases = df[
    df["gt_has_laterality"]
].copy()

laterality_cases[
    "laterality_adjusted_f1"
] = laterality_cases[
    "token_f1"
]

# Strong auxiliary signal:
# if removing laterality makes exact match,
# the difference is likely primarily laterality.
laterality_only_cases = laterality_cases[
    laterality_cases[
        "laterality_insensitive_exact"
    ]
].copy()

laterality_only_cases.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "laterality_only_mismatch_cases.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 13. PERFORMANCE BY TARGET LENGTH
# ============================================================

df["gt_char_length"] = (
    df["gt"].str.len()
)

df["length_bin"] = pd.cut(
    df["gt_char_length"],
    bins=[
        0,
        10,
        20,
        30,
        50,
        75,
        100,
        200
    ],
    right=True
)

length_summary = (
    df.groupby(
        "length_bin",
        observed=True
    )
    .agg(

        cases=(
            "case_id",
            "count"
        ),

        exact_match=(
            "gt",
            lambda x: 0
        ),

        char_similarity=(
            "char_similarity",
            "mean"
        ),

        token_f1=(
            "token_f1",
            "mean"
        ),
    )
    .reset_index()
)

for i, row in length_summary.iterrows():

    mask = (
        df["length_bin"]
        == row["length_bin"]
    )

    length_summary.loc[
        i,
        "exact_match"
    ] = (
        (
            df.loc[
                mask,
                "gt"
            ]
            ==
            df.loc[
                mask,
                "pred"
            ]
        ).mean()
        * 100
    )

length_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "performance_by_target_length.csv"
    ),
    index=False
)


# ============================================================
# 14. BEST CASES
# ============================================================

best_cases = (
    df.sort_values(
        [
            "token_f1",
            "char_similarity"
        ],
        ascending=False
    )
    .head(100)
)

best_cases.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "best_cases.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 15. WORST CASES
# ============================================================

worst_cases = (
    df.sort_values(
        [
            "token_f1",
            "char_similarity"
        ],
        ascending=True
    )
    .head(100)
)

worst_cases.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "worst_cases.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 16. CASES WHERE LATERALITY-INSENSITIVE MATCH HELPS
# ============================================================

laterality_help = df[
    (
        ~(
            df["gt"]
            ==
            df["pred"]
        )
    )
    &
    df[
        "laterality_insensitive_exact"
    ]
].copy()

laterality_help.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "laterality_insensitive_rescued.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 17. PRINT RESULTS
# ============================================================

exact_match = (
    (
        df["gt"]
        ==
        df["pred"]
    )
    .mean()
    *
    100
)

laterality_insensitive = (
    df[
        "laterality_insensitive_exact"
    ]
    .mean()
    *
    100
)

print("\n")
print("=" * 70)
print("BASIC RESULTS")
print("=" * 70)

print(
    f"\nExact match: "
    f"{exact_match:.2f}%"
)

print(
    f"Laterality-insensitive exact: "
    f"{laterality_insensitive:.2f}%"
)

print(
    f"\nCases with laterality in GT: "
    f"{df['gt_has_laterality'].sum()} "
    f"/ {len(df)}"
)

print(
    f"Cases where removing laterality "
    f"makes exact match: "
    f"{len(laterality_only_cases)}"
)


# ============================================================
# TOP GT
# ============================================================

print("\n" + "-" * 70)
print("TOP 20 GROUND-TRUTH CONCLUSIONS")
print("-" * 70)

print(
    gt_freq.head(20).to_string(
        index=False
    )
)


# ============================================================
# TOP PRED
# ============================================================

print("\n" + "-" * 70)
print("ALL PREDICTIONS")
print("-" * 70)

print(
    pred_freq.to_string(
        index=False
    )
)


# ============================================================
# COVERAGE
# ============================================================

print("\n" + "-" * 70)
print("PREDICTION CONCENTRATION")
print("-" * 70)

print(
    coverage_df.to_string(
        index=False
    )
)


# ============================================================
# LATERALITY
# ============================================================

print("\n" + "-" * 70)
print("LATERALITY SUMMARY")
print("-" * 70)

print(
    laterality_summary.to_string(
        index=False
    )
)


# ============================================================
# LENGTH
# ============================================================

print("\n" + "-" * 70)
print("PERFORMANCE BY TARGET LENGTH")
print("-" * 70)

print(
    length_summary.to_string(
        index=False
    )
)


# ============================================================
# LATERALITY RESCUE EXAMPLES
# ============================================================

print("\n" + "-" * 70)
print("LATERALITY-ONLY MISMATCH EXAMPLES")
print("-" * 70)

if len(laterality_help) == 0:

    print(
        "No exact laterality-only mismatches found."
    )

else:

    for _, row in (
        laterality_help.head(20)
        .iterrows()
    ):

        print(
            f"\nCase: {row['case_id']}"
        )

        print(
            "GT  :",
            row["ground_truth"]
        )

        print(
            "PRED:",
            row["prediction"]
        )


# ============================================================
# WORST CASES
# ============================================================

print("\n" + "-" * 70)
print("10 WORST CASES")
print("-" * 70)

for _, row in (
    worst_cases.head(10)
    .iterrows()
):

    print(
        f"\nCase: {row['case_id']}"
    )

    print(
        "GT  :",
        row["ground_truth"]
    )

    print(
        "PRED:",
        row["prediction"]
    )

    print(
        "F1  :",
        f"{row['token_f1']:.4f}"
    )


# ============================================================
# SAVE MASTER
# ============================================================

MASTER_FILE = os.path.join(
    OUTPUT_DIR,
    "v2_error_analysis_master.csv"
)

df.to_csv(
    MASTER_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\n")
print("=" * 70)
print("V2 ERROR ANALYSIS COMPLETE")
print("=" * 70)

print(
    "\nMaster:",
    MASTER_FILE
)

print(
    "\nOutput directory:",
    OUTPUT_DIR
)

V2 ERROR ANALYSIS + LATERALITY AUDIT

Prediction file: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v2/generation_eval/test_predictions_v2_correct.csv
Rows: 712


BASIC RESULTS

Exact match: 16.71%
Laterality-insensitive exact: 17.28%

Cases with laterality in GT: 253 / 712
Cases where removing laterality makes exact match: 4

----------------------------------------------------------------------
TOP 20 GROUND-TRUTH CONCLUSIONS
----------------------------------------------------------------------
                                                     ground_truth  count  percentage
                                                     VIÊM MŨI MẠN     79   11.095506
                                                         VIÊM MŨI     66    9.269663
                     VIÊM HỌNG MẠN - THEO DÕI TRÀO NGƯỢC DỊCH VỊ.     51    7.162921
                                                   VIÊM MŨI XOANG     35    4.915730
                                                 VIÊM MŨI ĐỢT C

/tmp/ipykernel_2005/1889149456.py:497: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.25925925925925924' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  laterality_summary.loc[
/tmp/ipykernel_2005/1889149456.py:623: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '10.294117647058822' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  length_summary.loc[


In [19]:
# ============================================================
# V3-PREPARATION — TARGET CONCEPT AUDIT
# 🟢 CPU ENOUGH
#
# Purpose:
# - Audit clinical concept structure of ket_luan
# - Do NOT modify original ground truth
# - Do NOT train anything
# ============================================================

import os
import re
import unicodedata
import pandas as pd
import numpy as np
from collections import Counter

print("=" * 72)
print("V3 PREPARATION — TARGET CONCEPT AUDIT")
print("=" * 72)

PROJECT_DIR = "/content/drive/MyDrive/NoiSoi_Matching"

CASE_FILE = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_case_manifest.csv"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "v3_target_audit"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(CASE_FILE, low_memory=False)

df["ket_luan"] = (
    df["ket_luan"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df = df[df["ket_luan"] != ""].copy()

print("\nCases with target:", len(df))


# ============================================================
# 1. NORMALIZATION FOR AUDIT ONLY
# ============================================================

def normalize(text):
    text = unicodedata.normalize("NFC", str(text))
    text = text.upper().strip()

    text = re.sub(r"\s+", " ", text)

    # normalize punctuation spacing
    text = re.sub(r"\s*-\s*", " - ", text)
    text = re.sub(r"\s*\+\s*", " + ", text)
    text = re.sub(r"\s*/\s*", " / ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()


df["target_norm"] = df["ket_luan"].map(normalize)


# ============================================================
# 2. CLINICAL CONCEPT DICTIONARY
#
# Intentionally broad.
# This is an AUDIT dictionary, not yet V3 ground truth.
# ============================================================

CONCEPTS = {

    # --------------------------------------------------------
    # EAR
    # --------------------------------------------------------

    "viem_ong_tai_ngoai": [
        r"VIÊM ỐNG TAI NGOÀI",
    ],

    "viem_tai_giua": [
        r"VIÊM TAI GIỮA",
    ],

    "viem_mang_nhi": [
        r"VIÊM MÀNG NHĨ",
    ],

    "mang_nhi": [
        r"MÀNG NHĨ",
    ],

    "ray_tai": [
        r"RÁY TAI",
    ],

    "thủng_mang_nhi": [
        r"THỦNG MÀNG NHĨ",
    ],

    # --------------------------------------------------------
    # NOSE / SINUS
    # --------------------------------------------------------

    "viem_mui": [
        r"VIÊM MŨI(?! XOANG)",
    ],

    "viem_mui_xoang": [
        r"VIÊM MŨI XOANG",
    ],

    "polyp_mui": [
        r"POLYP MŨI",
        r"POLYP.*MŨI",
    ],

    "chay_mau_mui": [
        r"CHẢY MÁU MŨI",
    ],

    "lech_vach_ngan": [
        r"LỆCH VÁCH NGĂN",
        r"VẸO VÁCH NGĂN",
    ],

    "qua_phat_cuon_mui": [
        r"QUÁ PHÁT CUỐN",
        r"CUỐN MŨI.*QUÁ PHÁT",
    ],

    # --------------------------------------------------------
    # NASOPHARYNX / VA
    # --------------------------------------------------------

    "viem_va": [
        r"VIÊM VA",
    ],

    "va": [
        r"\bVA\b",
    ],

    "qua_phat_va": [
        r"VA QUÁ PHÁT",
        r"QUÁ PHÁT VA",
    ],

    # --------------------------------------------------------
    # THROAT
    # --------------------------------------------------------

    "viem_hong": [
        r"VIÊM HỌNG",
    ],

    "viem_amidan": [
        r"VIÊM A[\-\s]?MI[\-\s]?ĐAN",
        r"VIÊM AMIDAN",
    ],

    "amidan_qua_phat": [
        r"AMIDAN QUÁ PHÁT",
        r"A[\-\s]?MI[\-\s]?ĐAN QUÁ PHÁT",
    ],

    # --------------------------------------------------------
    # LARYNX
    # --------------------------------------------------------

    "viem_thanh_quan": [
        r"VIÊM THANH QUẢN",
    ],

    "hat_day_thanh": [
        r"HẠT DÂY THANH",
    ],

    "polyp_day_thanh": [
        r"POLYP DÂY THANH",
    ],

    "liet_day_thanh": [
        r"LIỆT DÂY THANH",
    ],

    # --------------------------------------------------------
    # REFLUX
    # --------------------------------------------------------

    "theo_doi_trao_nguoc": [
        r"THEO DÕI TRÀO NGƯỢC",
        r"TRÀO NGƯỢC DỊCH VỊ",
    ],

    # --------------------------------------------------------
    # NEGATIVE / NORMAL
    # --------------------------------------------------------

    "khong_thay_benh_ly": [
        r"KHÔNG THẤY HÌNH ẢNH BỆNH LÝ",
        r"CHƯA THẤY HÌNH ẢNH BỆNH LÝ",
    ],
}


# ============================================================
# 3. ATTRIBUTES
# ============================================================

ATTRIBUTES = {

    "acute": [
        r"\bCẤP\b",
        r"ĐỢT CẤP",
    ],

    "chronic": [
        r"\bMẠN\b",
        r"MẠN TÍNH",
    ],

    "right": [
        r"\bP\b",
        r"\bPHẢI\b",
    ],

    "left": [
        r"\bT\b",
        r"\bTRÁI\b",
    ],

    "bilateral": [
        r"\b2 BÊN\b",
        r"\bHAI BÊN\b",
    ],

    "post_surgery": [
        r"\bĐÃ PT\b",
        r"\bPTNS\b",
        r"ĐÃ PHẪU THUẬT",
    ],
}


# ============================================================
# 4. DETECTOR
# ============================================================

def detect_patterns(text, dictionary):

    found = []

    for label, patterns in dictionary.items():

        for pattern in patterns:

            if re.search(pattern, text):

                found.append(label)
                break

    return found


df["concepts"] = df["target_norm"].apply(
    lambda x: detect_patterns(x, CONCEPTS)
)

df["attributes"] = df["target_norm"].apply(
    lambda x: detect_patterns(x, ATTRIBUTES)
)

df["num_concepts"] = df["concepts"].map(len)
df["num_attributes"] = df["attributes"].map(len)


# ============================================================
# 5. CONCEPT FREQUENCY
# ============================================================

concept_counter = Counter()

for concepts in df["concepts"]:
    concept_counter.update(concepts)

concept_freq = pd.DataFrame(
    concept_counter.most_common(),
    columns=["concept", "count"]
)

concept_freq["percentage"] = (
    concept_freq["count"] /
    len(df) *
    100
)

concept_freq.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "concept_frequency.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 6. ATTRIBUTE FREQUENCY
# ============================================================

attribute_counter = Counter()

for attrs in df["attributes"]:
    attribute_counter.update(attrs)

attribute_freq = pd.DataFrame(
    attribute_counter.most_common(),
    columns=["attribute", "count"]
)

attribute_freq["percentage"] = (
    attribute_freq["count"] /
    len(df) *
    100
)

attribute_freq.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "attribute_frequency.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 7. COVERAGE
# ============================================================

covered = df["num_concepts"] > 0

print("\n" + "=" * 72)
print("CONCEPT COVERAGE")
print("=" * 72)

print(
    f"\nAt least one concept detected: "
    f"{covered.sum()} / {len(df)} "
    f"({covered.mean()*100:.2f}%)"
)

print(
    f"No concept detected: "
    f"{(~covered).sum()} "
    f"({(~covered).mean()*100:.2f}%)"
)


# ============================================================
# 8. NUMBER OF CONCEPTS PER REPORT
# ============================================================

concept_count_dist = (
    df["num_concepts"]
    .value_counts()
    .sort_index()
    .reset_index()
)

concept_count_dist.columns = [
    "num_concepts",
    "cases"
]

concept_count_dist["percentage"] = (
    concept_count_dist["cases"] /
    len(df) *
    100
)

concept_count_dist.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "concept_count_distribution.csv"
    ),
    index=False
)


# ============================================================
# 9. MOST COMMON CONCEPT COMBINATIONS
# ============================================================

df["concept_signature"] = df["concepts"].apply(
    lambda x: " + ".join(sorted(x))
    if x else "UNMAPPED"
)

combination_freq = (
    df["concept_signature"]
    .value_counts()
    .reset_index()
)

combination_freq.columns = [
    "concept_signature",
    "count"
]

combination_freq["percentage"] = (
    combination_freq["count"] /
    len(df) *
    100
)

combination_freq.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "concept_combination_frequency.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 10. UNMAPPED TARGETS
# ============================================================

unmapped = df[
    df["num_concepts"] == 0
].copy()

unmapped_freq = (
    unmapped["target_norm"]
    .value_counts()
    .reset_index()
)

unmapped_freq.columns = [
    "target",
    "count"
]

unmapped_freq.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "unmapped_targets.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 11. MULTI-CONCEPT CASES
# ============================================================

multi = df[
    df["num_concepts"] >= 2
].copy()

multi.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "multi_concept_cases.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 12. SAVE MASTER
# ============================================================

master_file = os.path.join(
    OUTPUT_DIR,
    "target_concept_audit_master.csv"
)

save_df = df.copy()

save_df["concepts"] = save_df["concepts"].apply(
    lambda x: "|".join(x)
)

save_df["attributes"] = save_df["attributes"].apply(
    lambda x: "|".join(x)
)

save_df.to_csv(
    master_file,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 13. PRINT
# ============================================================

print("\n" + "-" * 72)
print("CONCEPT FREQUENCY")
print("-" * 72)

print(
    concept_freq.to_string(
        index=False
    )
)


print("\n" + "-" * 72)
print("ATTRIBUTE FREQUENCY")
print("-" * 72)

print(
    attribute_freq.to_string(
        index=False
    )
)


print("\n" + "-" * 72)
print("CONCEPTS PER REPORT")
print("-" * 72)

print(
    concept_count_dist.to_string(
        index=False
    )
)


print("\n" + "-" * 72)
print("TOP 30 CONCEPT COMBINATIONS")
print("-" * 72)

print(
    combination_freq.head(30).to_string(
        index=False
    )
)


print("\n" + "-" * 72)
print("TOP 50 UNMAPPED TARGETS")
print("-" * 72)

if len(unmapped_freq):

    print(
        unmapped_freq.head(50).to_string(
            index=False
        )
    )

else:

    print("None.")


# ============================================================
# 14. KEY SUMMARY
# ============================================================

print("\n" + "=" * 72)
print("TARGET STRUCTURE SUMMARY")
print("=" * 72)

print(
    f"\nTotal targets: {len(df)}"
)

print(
    f"Unique raw targets: "
    f"{df['ket_luan'].nunique()}"
)

print(
    f"Unique normalized targets: "
    f"{df['target_norm'].nunique()}"
)

print(
    f"Mapped targets: "
    f"{covered.sum()} "
    f"({covered.mean()*100:.2f}%)"
)

print(
    f"Multi-concept targets: "
    f"{len(multi)} "
    f"({len(multi)/len(df)*100:.2f}%)"
)

print(
    f"Unique concept combinations: "
    f"{df['concept_signature'].nunique()}"
)

print("\nSaved to:")
print(OUTPUT_DIR)

print("\n" + "=" * 72)
print("V3 TARGET CONCEPT AUDIT COMPLETE")
print("=" * 72)

V3 PREPARATION — TARGET CONCEPT AUDIT

Cases with target: 7606

CONCEPT COVERAGE

At least one concept detected: 7290 / 7606 (95.85%)
No concept detected: 316 (4.15%)

------------------------------------------------------------------------
CONCEPT FREQUENCY
------------------------------------------------------------------------
            concept  count  percentage
           viem_mui   3577   47.028662
     viem_mui_xoang   1489   19.576650
                 va   1058   13.910071
 viem_ong_tai_ngoai   1003   13.186958
      viem_tai_giua   1001   13.160663
           mang_nhi    710    9.334736
 khong_thay_benh_ly    610    8.019984
          viem_hong    567    7.454641
theo_doi_trao_nguoc    472    6.205627
            viem_va    270    3.549829
          polyp_mui    264    3.470944
            ray_tai    247    3.247436
       chay_mau_mui    141    1.853800
    viem_thanh_quan     59    0.775703
      hat_day_thanh     34    0.447016
      viem_mang_nhi     26    0.341835
     

In [20]:
# ============================================================
# V3-ONTOLOGY-V1
# 🟢 CPU ONLY
#
# Build structured clinical labels from raw ket_luan
#
# IMPORTANT:
# - Original ket_luan is NEVER modified.
# - This ontology is for structured auxiliary supervision.
# - No model training.
# ============================================================

import os
import re
import unicodedata
import json
import numpy as np
import pandas as pd
from collections import Counter

print("=" * 72)
print("V3 ONTOLOGY V1 — STRUCTURED TARGET CONSTRUCTION")
print("=" * 72)


# ============================================================
# 1. CONFIG
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/NoiSoi_Matching"

CASE_FILE = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_case_manifest.csv"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "v3_ontology_v1"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

MASTER_FILE = os.path.join(
    OUTPUT_DIR,
    "v3_structured_targets.csv"
)

ONTOLOGY_FILE = os.path.join(
    OUTPUT_DIR,
    "v3_ontology.json"
)

CONCEPT_FREQ_FILE = os.path.join(
    OUTPUT_DIR,
    "concept_frequency.csv"
)

ATTRIBUTE_FREQ_FILE = os.path.join(
    OUTPUT_DIR,
    "attribute_frequency.csv"
)

UNMAPPED_FILE = os.path.join(
    OUTPUT_DIR,
    "unmapped_targets.csv"
)

AMBIGUOUS_FILE = os.path.join(
    OUTPUT_DIR,
    "ambiguous_targets.csv"
)

COMBINATION_FILE = os.path.join(
    OUTPUT_DIR,
    "concept_combinations.csv"
)


# ============================================================
# 2. LOAD
# ============================================================

if not os.path.exists(CASE_FILE):

    raise FileNotFoundError(
        CASE_FILE
    )

df = pd.read_csv(
    CASE_FILE,
    low_memory=False
)

df["ket_luan"] = (
    df["ket_luan"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df = df[
    df["ket_luan"] != ""
].copy()

df = df.reset_index(drop=True)

print(
    "\nCases:",
    len(df)
)


# ============================================================
# 3. TEXT NORMALIZATION
# ============================================================

def normalize_text(text):

    text = unicodedata.normalize(
        "NFC",
        str(text)
    )

    text = text.upper().strip()

    # Normalize separators
    text = re.sub(
        r"\s*-\s*",
        " - ",
        text
    )

    text = re.sub(
        r"\s*\+\s*",
        " + ",
        text
    )

    text = re.sub(
        r"\s*/\s*",
        " / ",
        text
    )

    text = re.sub(
        r"\s*>\s*",
        " > ",
        text
    )

    # Normalize repeated whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


df["target_norm"] = df[
    "ket_luan"
].map(normalize_text)


# ============================================================
# 4. ONTOLOGY
#
# Each concept is a CLINICAL FINDING / CONDITION.
#
# Anatomy-only words such as "MÀNG NHĨ" and "VA" are NOT
# treated as independent disease concepts when embedded inside
# a more specific pathology.
# ============================================================

ONTOLOGY = {

    # ========================================================
    # EAR — EXTERNAL EAR
    # ========================================================

    "viem_ong_tai_ngoai": {
        "group": "ear",
        "patterns": [
            r"VIÊM ỐNG TAI NGOÀI",
        ],
    },

    "nhot_ong_tai_ngoai": {
        "group": "ear",
        "patterns": [
            r"NHỌT ỐNG TAI NGOÀI",
        ],
    },

    "chan_thuong_ong_tai_ngoai": {
        "group": "ear",
        "patterns": [
            r"CHẤN THƯƠNG ỐNG TAI NGOÀI",
            r"SÂY SÁT ỐNG TAI NGOÀI",
            r"SÂY SÁT DA.*ỐNG TAI NGOÀI",
        ],
    },

    # ========================================================
    # EAR — MIDDLE EAR / TYMPANIC MEMBRANE
    # ========================================================

    "viem_tai_giua": {
        "group": "ear",
        "patterns": [
            r"VIÊM TAI GIỮA",
        ],
    },

    "viem_mang_nhi": {
        "group": "ear",
        "patterns": [
            r"VIÊM MÀNG NHĨ",
            r"MÀNG NHĨ CẤP BÓNG NƯỚC",
        ],
    },

    "thung_mang_nhi": {
        "group": "ear",
        "patterns": [
            r"THỦNG NHĨ",
            r"THỦNG MÀNG NHĨ",
        ],
    },

    "xep_mang_nhi": {
        "group": "ear",
        "patterns": [
            r"XẸP NHĨ",
            r"XẸP MÀNG NHĨ",
        ],
    },

    "ray_tai": {
        "group": "ear",
        "patterns": [
            r"RÁY TAI",
        ],
    },

    # ========================================================
    # EAR — MASTOID / POST-SURGICAL
    # ========================================================

    "viem_tai_xuong_chum": {
        "group": "ear",
        "patterns": [
            r"VIÊM TAI XƯƠNG CHŨM",
        ],
    },

    "hau_phau_va_nhi": {
        "group": "ear",
        "patterns": [
            r"HẬU PHẪU VÁ NHĨ",
            r"VÁ NHĨ",
        ],
    },

    # ========================================================
    # EAR — FOREIGN BODY
    # ========================================================

    "di_vat_tai": {
        "group": "ear",
        "patterns": [
            r"DỊ VẬT TAI",
        ],
    },

    # ========================================================
    # NOSE
    # ========================================================

    "viem_mui": {
        "group": "nose",
        "patterns": [
            # Exclude explicit VIÊM MŨI XOANG
            r"VIÊM MŨI(?! XOANG)",
        ],
    },

    "viem_mui_xoang": {
        "group": "nose",
        "patterns": [
            r"VIÊM MŨI XOANG",
        ],
    },

    "viem_xoang": {
        "group": "nose",
        "patterns": [
            r"VIÊM XOANG",
        ],
    },

    "polyp_mui": {
        "group": "nose",
        "patterns": [
            r"POLYP MŨI",
            r"POLYP.*MŨI",
        ],
    },

    "chay_mau_mui": {
        "group": "nose",
        "patterns": [
            r"CHẢY MÁU MŨI",
        ],
    },

    "dich_vat_mui": {
        "group": "nose",
        "patterns": [
            r"DỊ VẬT MŨI",
        ],
    },

    "hoc_xuong_ca": {
        "group": "foreign_body",
        "patterns": [
            r"HÓC XƯƠNG CÁ",
        ],
    },

    "di_vat_hong_thanh_quan": {
        "group": "foreign_body",
        "patterns": [
            r"DỊ VẬT HỌNG THANH QUẢN",
            r"DỊ VẬT.*HỌNG.*THANH QUẢN",
        ],
    },

    "di_vat": {
        "group": "foreign_body",
        "patterns": [
            r"KHÔNG THẤY.*DỊ VẬT",
            r"HIỆN KHÔNG THẤY.*DỊ VẬT",
        ],
    },

    "hau_phau_mui_xoang": {
        "group": "nose",
        "patterns": [
            r"HẬU PHẪU NỘI SOI MŨI XOANG",
        ],
    },

    "tien_dinh_mui": {
        "group": "nose",
        "patterns": [
            r"VIÊM TIỀN ĐÌNH MŨI",
        ],
    },

    "lech_vach_ngan": {
        "group": "nose",
        "patterns": [
            r"LỆCH VÁCH NGĂN",
            r"VẸO VÁCH NGĂN",
        ],
    },

    # ========================================================
    # NASOPHARYNX / ADENOID
    # ========================================================

    "viem_va": {
        "group": "nasopharynx",
        "patterns": [
            r"VIÊM VA",
        ],
    },

    "qua_phat_va": {
        "group": "nasopharynx",
        "patterns": [
            r"VA QUÁ PHÁT",
            r"QUÁ PHÁT VA",
            r"A[- ]?MI[- ]?ĐAN QUÁ PHÁT",
        ],
    },

    # ========================================================
    # OROPHARYNX / TONSILS
    # ========================================================

    "viem_amidan": {
        "group": "oropharynx",
        "patterns": [
            r"VIÊM A[- ]?MI[- ]?ĐAN",
            r"VIÊM AMIDAN",
        ],
    },

    "amidan_qua_phat": {
        "group": "oropharynx",
        "patterns": [
            r"A[- ]?MI[- ]?ĐAN QUÁ PHÁT",
            r"AMIDAN QUÁ PHÁT",
        ],
    },

    "viem_hong": {
        "group": "oropharynx",
        "patterns": [
            r"VIÊM HỌNG",
        ],
    },

    "viem_mieng": {
        "group": "oropharynx",
        "patterns": [
            r"VIÊM MIỆNG",
        ],
    },

    "viem_luoi": {
        "group": "oropharynx",
        "patterns": [
            r"VIÊM LƯỠI",
        ],
    },

    "ap_to_mieng": {
        "group": "oropharynx",
        "patterns": [
            r"ÁP[- ]?TƠ",
            r"AP[- ]?TƠ",
        ],
    },

    # ========================================================
    # LARYNX
    # ========================================================

    "viem_thanh_quan": {
        "group": "larynx",
        "patterns": [
            r"VIÊM THANH QUẢN",
        ],
    },

    "hat_day_thanh": {
        "group": "larynx",
        "patterns": [
            r"HẠT DÂY THANH",
        ],
    },

    "polyp_day_thanh": {
        "group": "larynx",
        "patterns": [
            r"POLYP DÂY THANH",
        ],
    },

    "liet_day_thanh": {
        "group": "larynx",
        "patterns": [
            r"LIỆT DÂY THANH",
        ],
    },

    # ========================================================
    # REFLUX
    # ========================================================

    "theo_doi_trao_nguoc": {
        "group": "reflux",
        "patterns": [
            r"THEO DÕI TRÀO NGƯỢC",
            r"TRÀO NGƯỢC DỊCH VỊ",
        ],
    },

    # ========================================================
    # NEGATIVE FINDINGS
    # ========================================================

    "khong_thay_benh_ly_tai": {
        "group": "negative",
        "patterns": [
            r"KHÔNG THẤY.*BỆNH LÝ TAI NGOÀI",
            r"KHÔNG THẤY.*BỆNH LÝ TAI NGOÀI.*TAI GIỮA",
        ],
    },

    "khong_thay_benh_ly": {
        "group": "negative",
        "patterns": [
            r"KHÔNG THẤY HÌNH ẢNH BỆNH LÝ",
            r"CHƯA THẤY HÌNH ẢNH BỆNH LÝ",
        ],
    },

    "khong_thay_diem_chay_mau": {
        "group": "negative",
        "patterns": [
            r"KHÔNG THẤY ĐIỂM CHẢY MÁU",
        ],
    },

    "khong_thay_di_vat": {
        "group": "negative",
        "patterns": [
            r"KHÔNG THẤY.*DỊ VẬT",
        ],
    },
}


# ============================================================
# 5. ATTRIBUTES
#
# These are NOT clinical concepts.
# They are modifiers/context.
# ============================================================

ATTRIBUTES = {

    "acute": [
        r"\bCẤP\b",
        r"ĐỢT CẤP",
    ],

    "chronic": [
        r"\bMẠN\b",
        r"MẠN TÍNH",
    ],

    "right": [
        r"\bP\b",
        r"\bPHẢI\b",
        r"\(P\)",
    ],

    "left": [
        r"\bT\b",
        r"\bTRÁI\b",
        r"\(T\)",
    ],

    "bilateral": [
        r"\b2 BÊN\b",
        r"\bHAI BÊN\b",
        r"\b2 TAI\b",
        r"\bHAI TAI\b",
    ],

    "post_surgery": [
        r"\bĐÃ PT\b",
        r"\bPTNS\b",
        r"ĐÃ PHẪU THUẬT",
        r"HẬU PHẪU",
    ],
}


# ============================================================
# 6. EXPLICIT EXCLUSIONS / PRIORITY RULES
#
# Prevent generic anatomy/pathology labels from creating
# redundant labels.
# ============================================================

def detect_concepts(text):

    found = set()

    for concept, spec in ONTOLOGY.items():

        for pattern in spec["patterns"]:

            if re.search(
                pattern,
                text
            ):

                found.add(
                    concept
                )

                break

    # --------------------------------------------------------
    # Specific concepts suppress generic negative labels
    # only where the wording is explicitly negative.
    # --------------------------------------------------------

    # "VIÊM MÀNG NHĨ" should not create a generic
    # "MÀNG NHĨ" concept because anatomy is not a disease.
    #
    # No generic mang_nhi concept exists in V3 ontology.

    # "VIÊM VA" is pathology; "VA" alone is not a concept.
    #
    # No generic VA concept exists.

    # If explicit positive foreign body exists,
    # do not mark "khong_thay_di_vat".
    if "hoc_xuong_ca" in found:
        found.discard(
            "khong_thay_di_vat"
        )

    if "dich_vat_tai" in found:
        found.discard(
            "khong_thay_di_vat"
        )

    if "dich_vat_mui" in found:
        found.discard(
            "khong_thay_di_vat"
        )

    # Explicit pathology should take priority over
    # generic negative pathology statements only when
    # the negative phrase refers to another anatomical site.
    #
    # We therefore do NOT globally remove
    # khong_thay_benh_ly.

    return sorted(found)


def detect_attributes(text):

    found = set()

    for attr, patterns in ATTRIBUTES.items():

        for pattern in patterns:

            if re.search(
                pattern,
                text
            ):

                found.add(attr)
                break

    # Bilateral overrides left/right as a global attribute.
    if "bilateral" in found:

        found.discard("left")
        found.discard("right")

    return sorted(found)


# ============================================================
# 7. APPLY ONTOLOGY
# ============================================================

df["concepts"] = df[
    "target_norm"
].map(
    detect_concepts
)

df["attributes"] = df[
    "target_norm"
].map(
    detect_attributes
)

df["num_concepts"] = df[
    "concepts"
].map(len)

df["num_attributes"] = df[
    "attributes"
].map(len)


# ============================================================
# 8. MULTI-HOT LABELS
# ============================================================

concept_names = list(
    ONTOLOGY.keys()
)

attribute_names = list(
    ATTRIBUTES.keys()
)

for concept in concept_names:

    df[
        f"concept__{concept}"
    ] = df[
        "concepts"
    ].map(
        lambda x,
        c=concept:
        int(c in x)
    )

for attr in attribute_names:

    df[
        f"attribute__{attr}"
    ] = df[
        "attributes"
    ].map(
        lambda x,
        a=attr:
        int(a in x)
    )


# ============================================================
# 9. COVERAGE
# ============================================================

df["has_concept"] = (
    df["num_concepts"] > 0
)

df["has_attribute"] = (
    df["num_attributes"] > 0
)


# ============================================================
# 10. CONCEPT FREQUENCY
# ============================================================

concept_counter = Counter()

for concepts in df["concepts"]:

    concept_counter.update(
        concepts
    )

concept_freq = pd.DataFrame(
    concept_counter.most_common(),
    columns=[
        "concept",
        "count"
    ]
)

concept_freq["percentage"] = (
    concept_freq["count"]
    /
    len(df)
    *
    100
)

concept_freq.to_csv(
    CONCEPT_FREQ_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 11. ATTRIBUTE FREQUENCY
# ============================================================

attribute_counter = Counter()

for attrs in df["attributes"]:

    attribute_counter.update(
        attrs
    )

attribute_freq = pd.DataFrame(
    attribute_counter.most_common(),
    columns=[
        "attribute",
        "count"
    ]
)

attribute_freq["percentage"] = (
    attribute_freq["count"]
    /
    len(df)
    *
    100
)

attribute_freq.to_csv(
    ATTRIBUTE_FREQ_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 12. UNMAPPED
# ============================================================

unmapped = df[
    ~df["has_concept"]
].copy()

unmapped_freq = (
    unmapped[
        "target_norm"
    ]
    .value_counts()
    .reset_index()
)

unmapped_freq.columns = [
    "target",
    "count"
]

unmapped_freq.to_csv(
    UNMAPPED_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 13. AMBIGUITY AUDIT
#
# Flag:
# - no concept
# - too many concepts
# - negative + positive pathology
# - conflicting laterality
# ============================================================

def detect_ambiguity(row):

    reasons = []

    if row["num_concepts"] == 0:

        reasons.append(
            "NO_CONCEPT"
        )

    if row["num_concepts"] >= 5:

        reasons.append(
            "MANY_CONCEPTS"
        )

    positive = [
        c for c in row["concepts"]
        if ONTOLOGY[c]["group"]
        != "negative"
    ]

    negative = [
        c for c in row["concepts"]
        if ONTOLOGY[c]["group"]
        == "negative"
    ]

    if positive and negative:

        reasons.append(
            "POSITIVE_AND_NEGATIVE"
        )

    text = row["target_norm"]

    if (
        re.search(r"\bP\b", text)
        and
        re.search(r"\bT\b", text)
        and
        "bilateral" not in row["attributes"]
    ):

        reasons.append(
            "MULTIPLE_LATERALITY"
        )

    return "|".join(reasons)


df["ambiguity_reason"] = df.apply(
    detect_ambiguity,
    axis=1
)

ambiguous = df[
    df["ambiguity_reason"] != ""
].copy()

ambiguous.to_csv(
    AMBIGUOUS_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 14. CONCEPT COMBINATIONS
# ============================================================

df["concept_signature"] = df[
    "concepts"
].map(
    lambda x:
    " + ".join(x)
    if x
    else "UNMAPPED"
)

combination_freq = (
    df[
        "concept_signature"
    ]
    .value_counts()
    .reset_index()
)

combination_freq.columns = [
    "concept_signature",
    "count"
]

combination_freq["percentage"] = (
    combination_freq["count"]
    /
    len(df)
    *
    100
)

combination_freq.to_csv(
    COMBINATION_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 15. SAVE ONTOLOGY JSON
# ============================================================

ontology_export = {

    "version": "V3-Ontology-V1",

    "concept_count":
        len(ONTOLOGY),

    "attribute_count":
        len(ATTRIBUTES),

    "concepts": ONTOLOGY,

    "attributes": ATTRIBUTES,
}

with open(
    ONTOLOGY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ontology_export,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 16. SAVE MASTER
# ============================================================

# Human-readable list columns
df["concepts_str"] = df[
    "concepts"
].map(
    lambda x:
    "|".join(x)
)

df["attributes_str"] = df[
    "attributes"
].map(
    lambda x:
    "|".join(x)
)

df.to_csv(
    MASTER_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 17. PRINT SUMMARY
# ============================================================

print("\n" + "=" * 72)
print("ONTOLOGY SUMMARY")
print("=" * 72)

print(
    "\nConcepts:",
    len(ONTOLOGY)
)

print(
    "Attributes:",
    len(ATTRIBUTES)
)

print(
    "Cases:",
    len(df)
)

print(
    f"\nConcept coverage: "
    f"{df['has_concept'].sum()} / {len(df)} "
    f"({df['has_concept'].mean()*100:.2f}%)"
)

print(
    f"Unmapped: "
    f"{(~df['has_concept']).sum()} "
    f"({(~df['has_concept']).mean()*100:.2f}%)"
)

print(
    f"Cases with attributes: "
    f"{df['has_attribute'].sum()} "
    f"({df['has_attribute'].mean()*100:.2f}%)"
)

print(
    f"Ambiguous cases: "
    f"{len(ambiguous)} "
    f"({len(ambiguous)/len(df)*100:.2f}%)"
)


# ============================================================
# 18. CONCEPT FREQUENCY
# ============================================================

print("\n" + "-" * 72)
print("CONCEPT FREQUENCY")
print("-" * 72)

print(
    concept_freq.to_string(
        index=False
    )
)


# ============================================================
# 19. ATTRIBUTE FREQUENCY
# ============================================================

print("\n" + "-" * 72)
print("ATTRIBUTE FREQUENCY")
print("-" * 72)

print(
    attribute_freq.to_string(
        index=False
    )
)


# ============================================================
# 20. TOP UNMAPPED
# ============================================================

print("\n" + "-" * 72)
print("TOP UNMAPPED TARGETS")
print("-" * 72)

if len(unmapped_freq):

    print(
        unmapped_freq
        .head(50)
        .to_string(index=False)
    )

else:

    print(
        "No unmapped targets."
    )


# ============================================================
# 21. TOP AMBIGUOUS
# ============================================================

print("\n" + "-" * 72)
print("TOP AMBIGUOUS CASES")
print("-" * 72)

if len(ambiguous):

    cols = [
        "case_id",
        "ket_luan",
        "concepts_str",
        "attributes_str",
        "ambiguity_reason",
    ]

    print(
        ambiguous[
            cols
        ]
        .head(50)
        .to_string(index=False)
    )

else:

    print(
        "No ambiguous cases."
    )


# ============================================================
# 22. TOP COMBINATIONS
# ============================================================

print("\n" + "-" * 72)
print("TOP CONCEPT COMBINATIONS")
print("-" * 72)

print(
    combination_freq
    .head(30)
    .to_string(index=False)
)


# ============================================================
# 23. FINAL CHECKS
# ============================================================

assert (
    len(df) == 7606
), "Unexpected number of target cases."

assert (
    df["case_id"].nunique()
    == len(df)
), "Duplicate case IDs."

assert (
    df["ket_luan"].notna().all()
), "Missing raw targets."

assert (
    df["target_norm"].notna().all()
), "Missing normalized targets."

print("\n" + "=" * 72)
print("V3 ONTOLOGY V1 COMPLETE")
print("=" * 72)

print("\nSaved:")

print(
    MASTER_FILE
)

print(
    ONTOLOGY_FILE
)

print(
    CONCEPT_FREQ_FILE
)

print(
    ATTRIBUTE_FREQ_FILE
)

print(
    UNMAPPED_FILE
)

print(
    AMBIGUOUS_FILE
)

print(
    COMBINATION_FILE
)

print("\n🟢 CPU-only step — PASS")

V3 ONTOLOGY V1 — STRUCTURED TARGET CONSTRUCTION

Cases: 7606

ONTOLOGY SUMMARY

Concepts: 40
Attributes: 6
Cases: 7606

Concept coverage: 7510 / 7606 (98.74%)
Unmapped: 96 (1.26%)
Cases with attributes: 5268 (69.26%)
Ambiguous cases: 809 (10.64%)

------------------------------------------------------------------------
CONCEPT FREQUENCY
------------------------------------------------------------------------
                  concept  count  percentage
                 viem_mui   3577   47.028662
           viem_mui_xoang   1489   19.576650
       viem_ong_tai_ngoai   1003   13.186958
            viem_tai_giua   1001   13.160663
       khong_thay_benh_ly    610    8.019984
                viem_hong    567    7.454641
   khong_thay_benh_ly_tai    514    6.757823
      theo_doi_trao_nguoc    472    6.205627
                  viem_va    270    3.549829
                polyp_mui    264    3.470944
                  ray_tai    247    3.247436
               viem_xoang    192    2.524323
 kh

KeyError: "['concepts_str', 'attributes_str'] not in index"

In [21]:
# ============================================================
# V3-ONTOLOGY-V1 — FIX AMBIGUOUS AUDIT
# 🟢 CPU ONLY
#
# The previous cell created `ambiguous` before adding
# concepts_str / attributes_str to df.
# This cell reconstructs those columns safely.
# ============================================================

print("=" * 72)
print("FIXING AMBIGUOUS AUDIT")
print("=" * 72)


# ------------------------------------------------------------
# 1. Recreate human-readable label columns
# ------------------------------------------------------------

df["concepts_str"] = df["concepts"].map(
    lambda x: "|".join(x) if isinstance(x, list) else ""
)

df["attributes_str"] = df["attributes"].map(
    lambda x: "|".join(x) if isinstance(x, list) else ""
)


# ------------------------------------------------------------
# 2. Rebuild ambiguity flags
# ------------------------------------------------------------

def detect_ambiguity_v2(row):

    reasons = []

    if row["num_concepts"] == 0:
        reasons.append("NO_CONCEPT")

    if row["num_concepts"] >= 5:
        reasons.append("MANY_CONCEPTS")

    positive = [
        c for c in row["concepts"]
        if ONTOLOGY[c]["group"] != "negative"
    ]

    negative = [
        c for c in row["concepts"]
        if ONTOLOGY[c]["group"] == "negative"
    ]

    if positive and negative:
        reasons.append("POSITIVE_AND_NEGATIVE")

    text = row["target_norm"]

    # Explicit P and T without bilateral marker
    has_p = bool(
        re.search(r"\bP\b", text)
        or re.search(r"\bPHẢI\b", text)
    )

    has_t = bool(
        re.search(r"\bT\b", text)
        or re.search(r"\bTRÁI\b", text)
    )

    if (
        has_p
        and has_t
        and "bilateral" not in row["attributes"]
    ):
        reasons.append("MULTIPLE_LATERALITY")

    return "|".join(reasons)


df["ambiguity_reason"] = df.apply(
    detect_ambiguity_v2,
    axis=1
)

ambiguous = df[
    df["ambiguity_reason"] != ""
].copy()


# ------------------------------------------------------------
# 3. Save
# ------------------------------------------------------------

ambiguous.to_csv(
    AMBIGUOUS_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 4. Print summary
# ------------------------------------------------------------

print(
    f"\nAmbiguous cases: {len(ambiguous):,}"
)

print(
    f"Percentage: "
    f"{len(ambiguous) / len(df) * 100:.2f}%"
)


# ------------------------------------------------------------
# 5. Ambiguity reason distribution
# ------------------------------------------------------------

reason_counter = Counter()

for reasons in ambiguous["ambiguity_reason"]:

    for reason in reasons.split("|"):

        if reason:
            reason_counter[reason] += 1


reason_df = pd.DataFrame(
    reason_counter.most_common(),
    columns=[
        "reason",
        "count"
    ]
)

reason_df["percentage"] = (
    reason_df["count"]
    / len(df)
    * 100
)

print("\n" + "-" * 72)
print("AMBIGUITY REASONS")
print("-" * 72)

print(
    reason_df.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 6. Show actual ambiguous cases
# ------------------------------------------------------------

print("\n" + "-" * 72)
print("TOP AMBIGUOUS CASES")
print("-" * 72)

cols = [
    "case_id",
    "ket_luan",
    "concepts_str",
    "attributes_str",
    "ambiguity_reason",
]

print(
    ambiguous[
        cols
    ]
    .head(100)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 7. Save a compact audit file
# ------------------------------------------------------------

audit_cols = [
    "case_id",
    "patient_group_id",
    "split",
    "ket_luan",
    "target_norm",
    "concepts_str",
    "attributes_str",
    "num_concepts",
    "num_attributes",
    "ambiguity_reason",
]

ambiguous[
    audit_cols
].to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ambiguous_cases_audit.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 8. Update master AFTER adding readable columns
# ------------------------------------------------------------

df.to_csv(
    MASTER_FILE,
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 72)
print("FIX COMPLETE")
print("=" * 72)

print(
    "\nSaved:",
    AMBIGUOUS_FILE
)

print(
    "Saved:",
    os.path.join(
        OUTPUT_DIR,
        "ambiguous_cases_audit.csv"
    )
)

print(
    "Updated:",
    MASTER_FILE
)

print("\n🟢 PASS")

FIXING AMBIGUOUS AUDIT

Ambiguous cases: 814
Percentage: 10.70%

------------------------------------------------------------------------
AMBIGUITY REASONS
------------------------------------------------------------------------
               reason  count  percentage
POSITIVE_AND_NEGATIVE    439    5.771759
  MULTIPLE_LATERALITY    291    3.825927
           NO_CONCEPT     96    1.262161

------------------------------------------------------------------------
TOP AMBIGUOUS CASES
------------------------------------------------------------------------
            case_id                                                                                          ket_luan                                                       concepts_str           attributes_str                          ambiguity_reason
10016.10016.0.10030                                                         VIÊM MŨI XOANG CẤP- VIÊM TAI GIỮA CẤP P>T                                       viem_mui_xoang|viem_tai_giua    

In [22]:
# ============================================================
# V3 ONTOLOGY V1.1 — SEMANTIC AUDIT
# 🟢 CPU ONLY
#
# Goal:
#   Separate legitimate multi-finding reports from TRUE conflicts.
#
# IMPORTANT:
#   No raw ket_luan is modified.
#   No training happens here.
# ============================================================

import os
import re
import pandas as pd
from collections import Counter

print("=" * 72)
print("V3 ONTOLOGY V1.1 — SEMANTIC AUDIT")
print("=" * 72)


# ============================================================
# 1. LOAD CURRENT MASTER
# ============================================================

MASTER_FILE = (
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "v3_ontology_v1/v3_structured_targets.csv"
)

OUT_DIR = (
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "v3_ontology_v1"
)

df = pd.read_csv(
    MASTER_FILE,
    low_memory=False
)

print(
    "\nLoaded:",
    len(df),
    "cases"
)


# ============================================================
# 2. RECONSTRUCT LIST COLUMNS
# ============================================================

def split_pipe(x):

    if pd.isna(x):
        return []

    x = str(x).strip()

    if not x:
        return []

    return [
        v.strip()
        for v in x.split("|")
        if v.strip()
    ]


df["concepts"] = df[
    "concepts_str"
].map(split_pipe)

df["attributes"] = df[
    "attributes_str"
].map(split_pipe)


# ============================================================
# 3. TRUE CONFLICT RULES
#
# These are actual semantic contradictions.
# ============================================================

TRUE_CONFLICT_PATTERNS = [

    # --------------------------------------------------------
    # Foreign body present vs explicitly absent
    # --------------------------------------------------------

    (
        "FOREIGN_BODY_PRESENT_AND_ABSENT",
        [
            "di_vat",
            "khong_thay_di_vat"
        ]
    ),

    (
        "EAR_FOREIGN_BODY_PRESENT_AND_ABSENT",
        [
            "di_vat_tai",
            "khong_thay_di_vat"
        ]
    ),

    (
        "NOSE_FOREIGN_BODY_PRESENT_AND_ABSENT",
        [
            "dich_vat_mui",
            "khong_thay_di_vat"
        ]
    ),

    # --------------------------------------------------------
    # Disease vs explicit global "no pathology"
    #
    # Only flag the broad negative statement here.
    # A report saying "no bleeding + rhinitis" is NOT conflict.
    # --------------------------------------------------------

]


# ============================================================
# 4. TRUE CONFLICT DETECTOR
# ============================================================

def detect_true_conflict(row):

    concepts = set(
        row["concepts"]
    )

    reasons = []

    for name, pair in TRUE_CONFLICT_PATTERNS:

        if set(pair).issubset(concepts):

            reasons.append(
                name
            )

    # --------------------------------------------------------
    # Generic "no pathology" + disease
    #
    # This is NOT automatically a contradiction because
    # the no-pathology statement may refer to another site.
    #
    # Therefore we only flag it for MANUAL REVIEW.
    # --------------------------------------------------------

    negative_global = (
        "khong_thay_benh_ly"
        in concepts
    )

    positive = [
        c for c in concepts
        if (
            c != "khong_thay_benh_ly"
            and
            not c.startswith("khong_thay_")
        )
    ]

    if negative_global and positive:

        reasons.append(
            "GLOBAL_NEGATIVE_WITH_POSITIVE"
        )

    return "|".join(
        sorted(set(reasons))
    )


df["true_conflict"] = df.apply(
    detect_true_conflict,
    axis=1
)


# ============================================================
# 5. LATERALITY IS NOT AMBIGUITY
# ============================================================

df["has_left"] = df[
    "attributes"
].map(
    lambda x: int(
        "left" in x
    )
)

df["has_right"] = df[
    "attributes"
].map(
    lambda x: int(
        "right" in x
    )
)

df["has_bilateral"] = df[
    "attributes"
].map(
    lambda x: int(
        "bilateral" in x
    )
)


# ------------------------------------------------------------
# IMPORTANT:
# P>T / T>P is legitimate.
# We do NOT call it ambiguous.
# ------------------------------------------------------------

df["multiple_laterality"] = (
    (
        df["has_left"] == 1
    )
    &
    (
        df["has_right"] == 1
    )
    &
    (
        df["has_bilateral"] == 0
    )
)


# ============================================================
# 6. VALID MULTI-CONCEPT REPORTS
# ============================================================

df["num_concepts"] = df[
    "concepts"
].map(len)

df["is_multi_concept"] = (
    df["num_concepts"] >= 2
)


# ============================================================
# 7. NO CONCEPT
# ============================================================

df["no_concept"] = (
    df["num_concepts"] == 0
)


# ============================================================
# 8. FINAL AUDIT CATEGORY
# ============================================================

def classify_audit(row):

    if row["no_concept"]:

        return "UNMAPPED"

    if row["true_conflict"]:

        return "TRUE_CONFLICT"

    if row["multiple_laterality"]:

        return "VALID_MULTI_LATERALITY"

    if row["is_multi_concept"]:

        return "VALID_MULTI_CONCEPT"

    return "CLEAN"


df["audit_category"] = df.apply(
    classify_audit,
    axis=1
)


# ============================================================
# 9. SUMMARY
# ============================================================

summary = (
    df[
        "audit_category"
    ]
    .value_counts()
    .reset_index()
)

summary.columns = [
    "category",
    "count"
]

summary["percentage"] = (
    summary["count"]
    /
    len(df)
    *
    100
)

print("\n" + "-" * 72)
print("AUDIT CATEGORY")
print("-" * 72)

print(
    summary.to_string(
        index=False
    )
)


# ============================================================
# 10. TRUE CONFLICT DETAILS
# ============================================================

conflicts = df[
    df["audit_category"]
    ==
    "TRUE_CONFLICT"
].copy()

print("\n" + "-" * 72)
print("TRUE CONFLICTS")
print("-" * 72)

print(
    "Count:",
    len(conflicts)
)

if len(conflicts):

    print(
        conflicts[
            [
                "case_id",
                "ket_luan",
                "concepts_str",
                "true_conflict"
            ]
        ]
        .head(100)
        .to_string(index=False)
    )


# ============================================================
# 11. UNMAPPED DETAILS
# ============================================================

unmapped = df[
    df["audit_category"]
    ==
    "UNMAPPED"
].copy()

unmapped_freq = (
    unmapped[
        "target_norm"
    ]
    .value_counts()
    .reset_index()
)

unmapped_freq.columns = [
    "target",
    "count"
]

print("\n" + "-" * 72)
print("UNMAPPED")
print("-" * 72)

print(
    "Cases:",
    len(unmapped)
)

print(
    "\nTop unmapped:"
)

print(
    unmapped_freq
    .head(100)
    .to_string(index=False)
)


# ============================================================
# 12. VALID MULTI-LATERALITY EXAMPLES
# ============================================================

multi_lat = df[
    df["audit_category"]
    ==
    "VALID_MULTI_LATERALITY"
].copy()

print("\n" + "-" * 72)
print("VALID MULTI-LATERALITY EXAMPLES")
print("-" * 72)

print(
    multi_lat[
        [
            "case_id",
            "ket_luan",
            "concepts_str",
            "attributes_str"
        ]
    ]
    .head(50)
    .to_string(index=False)
)


# ============================================================
# 13. VALID MULTI-CONCEPT EXAMPLES
# ============================================================

multi_concept = df[
    df["audit_category"]
    ==
    "VALID_MULTI_CONCEPT"
].copy()

print("\n" + "-" * 72)
print("VALID MULTI-CONCEPT EXAMPLES")
print("-" * 72)

print(
    multi_concept[
        [
            "case_id",
            "ket_luan",
            "concepts_str",
            "attributes_str"
        ]
    ]
    .head(50)
    .to_string(index=False)
)


# ============================================================
# 14. SAVE
# ============================================================

df.to_csv(
    os.path.join(
        OUT_DIR,
        "v3_structured_targets_v1_1_audit.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

summary.to_csv(
    os.path.join(
        OUT_DIR,
        "ontology_v1_1_audit_summary.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

conflicts.to_csv(
    os.path.join(
        OUT_DIR,
        "true_conflicts_v1_1.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

unmapped.to_csv(
    os.path.join(
        OUT_DIR,
        "unmapped_v1_1.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

multi_lat.to_csv(
    os.path.join(
        OUT_DIR,
        "valid_multi_laterality_v1_1.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 15. CHECKS
# ============================================================

assert len(df) == 7606

assert (
    df["case_id"].nunique()
    == 7606
)

assert (
    df["ket_luan"].notna().all()
)

print("\n" + "=" * 72)
print("V3 ONTOLOGY V1.1 AUDIT COMPLETE")
print("=" * 72)

print(
    "\nOutput:",
    os.path.join(
        OUT_DIR,
        "v3_structured_targets_v1_1_audit.csv"
    )
)

print("\n🟢 CPU PASS")

V3 ONTOLOGY V1.1 — SEMANTIC AUDIT

Loaded: 7606 cases

------------------------------------------------------------------------
AUDIT CATEGORY
------------------------------------------------------------------------
              category  count  percentage
                 CLEAN   4390   57.717591
   VALID_MULTI_CONCEPT   2570   33.789114
VALID_MULTI_LATERALITY    281    3.694452
         TRUE_CONFLICT    269    3.536682
              UNMAPPED     96    1.262161

------------------------------------------------------------------------
TRUE CONFLICTS
------------------------------------------------------------------------
Count: 269
            case_id                                                                                          ket_luan                                                       concepts_str                   true_conflict
10084.10084.0.10098                       HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI NGOÀI TAI GIỮA TRÊN NỘI SOI / VIÊM MŨI                 khong_th

In [23]:
# ============================================================
# V3 ONTOLOGY V1.1 — SEMANTIC AUDIT
# 🟢 CPU ONLY
#
# Goal:
#   Separate legitimate multi-finding reports from TRUE conflicts.
#
# IMPORTANT:
#   No raw ket_luan is modified.
#   No training happens here.
# ============================================================

import os
import re
import pandas as pd
from collections import Counter

print("=" * 72)
print("V3 ONTOLOGY V1.1 — SEMANTIC AUDIT")
print("=" * 72)


# ============================================================
# 1. LOAD CURRENT MASTER
# ============================================================

MASTER_FILE = (
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "v3_ontology_v1/v3_structured_targets.csv"
)

OUT_DIR = (
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "v3_ontology_v1"
)

df = pd.read_csv(
    MASTER_FILE,
    low_memory=False
)

print(
    "\nLoaded:",
    len(df),
    "cases"
)


# ============================================================
# 2. RECONSTRUCT LIST COLUMNS
# ============================================================

def split_pipe(x):

    if pd.isna(x):
        return []

    x = str(x).strip()

    if not x:
        return []

    return [
        v.strip()
        for v in x.split("|")
        if v.strip()
    ]


df["concepts"] = df[
    "concepts_str"
].map(split_pipe)

df["attributes"] = df[
    "attributes_str"
].map(split_pipe)


# ============================================================
# 3. TRUE CONFLICT RULES
#
# These are actual semantic contradictions.
# ============================================================

TRUE_CONFLICT_PATTERNS = [

    # --------------------------------------------------------
    # Foreign body present vs explicitly absent
    # --------------------------------------------------------

    (
        "FOREIGN_BODY_PRESENT_AND_ABSENT",
        [
            "di_vat",
            "khong_thay_di_vat"
        ]
    ),

    (
        "EAR_FOREIGN_BODY_PRESENT_AND_ABSENT",
        [
            "di_vat_tai",
            "khong_thay_di_vat"
        ]
    ),

    (
        "NOSE_FOREIGN_BODY_PRESENT_AND_ABSENT",
        [
            "dich_vat_mui",
            "khong_thay_di_vat"
        ]
    ),

    # --------------------------------------------------------
    # Disease vs explicit global "no pathology"
    #
    # Only flag the broad negative statement here.
    # A report saying "no bleeding + rhinitis" is NOT conflict.
    # --------------------------------------------------------

]


# ============================================================
# 4. TRUE CONFLICT DETECTOR
# ============================================================

def detect_true_conflict(row):

    concepts = set(
        row["concepts"]
    )

    reasons = []

    for name, pair in TRUE_CONFLICT_PATTERNS:

        if set(pair).issubset(concepts):

            reasons.append(
                name
            )

    # --------------------------------------------------------
    # Generic "no pathology" + disease
    #
    # This is NOT automatically a contradiction because
    # the no-pathology statement may refer to another site.
    #
    # Therefore we only flag it for MANUAL REVIEW.
    # --------------------------------------------------------

    negative_global = (
        "khong_thay_benh_ly"
        in concepts
    )

    positive = [
        c for c in concepts
        if (
            c != "khong_thay_benh_ly"
            and
            not c.startswith("khong_thay_")
        )
    ]

    if negative_global and positive:

        reasons.append(
            "GLOBAL_NEGATIVE_WITH_POSITIVE"
        )

    return "|".join(
        sorted(set(reasons))
    )


df["true_conflict"] = df.apply(
    detect_true_conflict,
    axis=1
)


# ============================================================
# 5. LATERALITY IS NOT AMBIGUITY
# ============================================================

df["has_left"] = df[
    "attributes"
].map(
    lambda x: int(
        "left" in x
    )
)

df["has_right"] = df[
    "attributes"
].map(
    lambda x: int(
        "right" in x
    )
)

df["has_bilateral"] = df[
    "attributes"
].map(
    lambda x: int(
        "bilateral" in x
    )
)


# ------------------------------------------------------------
# IMPORTANT:
# P>T / T>P is legitimate.
# We do NOT call it ambiguous.
# ------------------------------------------------------------

df["multiple_laterality"] = (
    (
        df["has_left"] == 1
    )
    &
    (
        df["has_right"] == 1
    )
    &
    (
        df["has_bilateral"] == 0
    )
)


# ============================================================
# 6. VALID MULTI-CONCEPT REPORTS
# ============================================================

df["num_concepts"] = df[
    "concepts"
].map(len)

df["is_multi_concept"] = (
    df["num_concepts"] >= 2
)


# ============================================================
# 7. NO CONCEPT
# ============================================================

df["no_concept"] = (
    df["num_concepts"] == 0
)


# ============================================================
# 8. FINAL AUDIT CATEGORY
# ============================================================

def classify_audit(row):

    if row["no_concept"]:

        return "UNMAPPED"

    if row["true_conflict"]:

        return "TRUE_CONFLICT"

    if row["multiple_laterality"]:

        return "VALID_MULTI_LATERALITY"

    if row["is_multi_concept"]:

        return "VALID_MULTI_CONCEPT"

    return "CLEAN"


df["audit_category"] = df.apply(
    classify_audit,
    axis=1
)


# ============================================================
# 9. SUMMARY
# ============================================================

summary = (
    df[
        "audit_category"
    ]
    .value_counts()
    .reset_index()
)

summary.columns = [
    "category",
    "count"
]

summary["percentage"] = (
    summary["count"]
    /
    len(df)
    *
    100
)

print("\n" + "-" * 72)
print("AUDIT CATEGORY")
print("-" * 72)

print(
    summary.to_string(
        index=False
    )
)


# ============================================================
# 10. TRUE CONFLICT DETAILS
# ============================================================

conflicts = df[
    df["audit_category"]
    ==
    "TRUE_CONFLICT"
].copy()

print("\n" + "-" * 72)
print("TRUE CONFLICTS")
print("-" * 72)

print(
    "Count:",
    len(conflicts)
)

if len(conflicts):

    print(
        conflicts[
            [
                "case_id",
                "ket_luan",
                "concepts_str",
                "true_conflict"
            ]
        ]
        .head(100)
        .to_string(index=False)
    )


# ============================================================
# 11. UNMAPPED DETAILS
# ============================================================

unmapped = df[
    df["audit_category"]
    ==
    "UNMAPPED"
].copy()

unmapped_freq = (
    unmapped[
        "target_norm"
    ]
    .value_counts()
    .reset_index()
)

unmapped_freq.columns = [
    "target",
    "count"
]

print("\n" + "-" * 72)
print("UNMAPPED")
print("-" * 72)

print(
    "Cases:",
    len(unmapped)
)

print(
    "\nTop unmapped:"
)

print(
    unmapped_freq
    .head(100)
    .to_string(index=False)
)


# ============================================================
# 12. VALID MULTI-LATERALITY EXAMPLES
# ============================================================

multi_lat = df[
    df["audit_category"]
    ==
    "VALID_MULTI_LATERALITY"
].copy()

print("\n" + "-" * 72)
print("VALID MULTI-LATERALITY EXAMPLES")
print("-" * 72)

print(
    multi_lat[
        [
            "case_id",
            "ket_luan",
            "concepts_str",
            "attributes_str"
        ]
    ]
    .head(50)
    .to_string(index=False)
)


# ============================================================
# 13. VALID MULTI-CONCEPT EXAMPLES
# ============================================================

multi_concept = df[
    df["audit_category"]
    ==
    "VALID_MULTI_CONCEPT"
].copy()

print("\n" + "-" * 72)
print("VALID MULTI-CONCEPT EXAMPLES")
print("-" * 72)

print(
    multi_concept[
        [
            "case_id",
            "ket_luan",
            "concepts_str",
            "attributes_str"
        ]
    ]
    .head(50)
    .to_string(index=False)
)


# ============================================================
# 14. SAVE
# ============================================================

df.to_csv(
    os.path.join(
        OUT_DIR,
        "v3_structured_targets_v1_1_audit.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

summary.to_csv(
    os.path.join(
        OUT_DIR,
        "ontology_v1_1_audit_summary.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

conflicts.to_csv(
    os.path.join(
        OUT_DIR,
        "true_conflicts_v1_1.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

unmapped.to_csv(
    os.path.join(
        OUT_DIR,
        "unmapped_v1_1.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

multi_lat.to_csv(
    os.path.join(
        OUT_DIR,
        "valid_multi_laterality_v1_1.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 15. CHECKS
# ============================================================

assert len(df) == 7606

assert (
    df["case_id"].nunique()
    == 7606
)

assert (
    df["ket_luan"].notna().all()
)

print("\n" + "=" * 72)
print("V3 ONTOLOGY V1.1 AUDIT COMPLETE")
print("=" * 72)

print(
    "\nOutput:",
    os.path.join(
        OUT_DIR,
        "v3_structured_targets_v1_1_audit.csv"
    )
)

print("\n🟢 CPU PASS")

V3 ONTOLOGY V1.1 — SEMANTIC AUDIT

Loaded: 7606 cases

------------------------------------------------------------------------
AUDIT CATEGORY
------------------------------------------------------------------------
              category  count  percentage
                 CLEAN   4390   57.717591
   VALID_MULTI_CONCEPT   2570   33.789114
VALID_MULTI_LATERALITY    281    3.694452
         TRUE_CONFLICT    269    3.536682
              UNMAPPED     96    1.262161

------------------------------------------------------------------------
TRUE CONFLICTS
------------------------------------------------------------------------
Count: 269
            case_id                                                                                          ket_luan                                                       concepts_str                   true_conflict
10084.10084.0.10098                       HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI NGOÀI TAI GIỮA TRÊN NỘI SOI / VIÊM MŨI                 khong_th

In [24]:
# ============================================================
# V3 ONTOLOGY V2
# 🟢 CPU ONLY
#
# Clean structured supervision:
#
#   1. Positive clinical concepts
#   2. Attributes
#   3. Negative findings
#
# IMPORTANT:
#   - raw ket_luan is preserved
#   - no cases are removed
#   - no model training
# ============================================================

import os
import re
import json
import pandas as pd
import unicodedata
from collections import Counter

print("=" * 72)
print("V3 ONTOLOGY V2 — FINAL STRUCTURED TARGET BUILDER")
print("=" * 72)


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = (
    "/content/drive/MyDrive/NoiSoi_Matching"
)

SOURCE_FILE = os.path.join(
    PROJECT_DIR,
    "v3_ontology_v1",
    "v3_structured_targets.csv"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "v3_ontology_v2"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 2. LOAD
# ============================================================

df = pd.read_csv(
    SOURCE_FILE,
    low_memory=False
)

df["ket_luan"] = (
    df["ket_luan"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print(
    "\nCases:",
    len(df)
)


# ============================================================
# 3. NORMALIZATION
# ============================================================

def normalize_text(text):

    text = unicodedata.normalize(
        "NFC",
        str(text)
    )

    text = text.upper()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


df["target_norm"] = df[
    "ket_luan"
].map(normalize_text)


# ============================================================
# 4. POSITIVE CLINICAL CONCEPTS
# ============================================================

CONCEPT_RULES = {

    # --------------------------------------------------------
    # EAR
    # --------------------------------------------------------

    "viem_ong_tai_ngoai": [
        r"VIÊM\s*ỐNG\s*TAI\s*NGOÀI",
        r"VIÊMỐNG\s*TAI\s*NGOÀI",
        r"VIÊM\s*TAI\s*NGOÀI",
    ],

    "nhot_ong_tai_ngoai": [
        r"NHỌT\s*(?:ỐNG\s*)?TAI\s*NGOÀI",
    ],

    "chan_thuong_ong_tai_ngoai": [
        r"CHẤN THƯƠNG\s*(?:ỐNG\s*)?TAI\s*NGOÀI",
        r"S[ÂÂ]Y\s*SÁT.*TAI\s*NGOÀI",
        r"S[ÂÂ]Y\s*X[ÂÂ]T.*TAI\s*NGOÀI",
    ],

    "hep_ong_tai_ngoai": [
        r"HẸP\s*(?:ỐNG\s*)?TAI\s*NGOÀI",
    ],

    "polyp_ong_tai_ngoai": [
        r"POLYP\s*(?:ỐNG\s*)?TAI\s*NGOÀI",
    ],

    "viem_tai_giua": [
        r"VIÊM\s*TAI\s*GIỮA",
    ],

    "viem_mang_nhi": [
        r"VIÊM\s*MÀNG\s*NHĨ",
        r"MÀNG\s*NHĨ\s*CẤP\s*BÓNG\s*NƯỚC",
    ],

    "thung_mang_nhi": [
        r"THỦNG\s*(?:NHĨ|MÀNG\s*NHĨ)",
    ],

    "xep_mang_nhi": [
        r"XẸP\s*(?:NHĨ|MÀNG\s*NHĨ)",
    ],

    "chan_thuong_mang_nhi": [
        r"CHẤN THƯƠNG\s*MÀNG\s*NHĨ",
        r"S[ÂÂ]Y\s*SÁT\s*MÀNG\s*NHĨ",
        r"S[ÂÂ]Y\s*X[ÂÂ]T\s*MÀNG\s*NHĨ",
    ],

    "ray_tai": [
        r"RÁY\s*TAI",
        r"NÚT\s*RÁY\s*TAI",
        r"NHIỀU\s*RÁY\s*TAI",
    ],

    "di_vat_tai": [
        r"DỊ\s*VẬT\s*(?:TAI|ỐNG\s*TAI)",
        r"HÓC\s*DỊ\s*VẬT\s*(?:TAI|ỐNG\s*TAI)",
    ],

    "viem_tai_xuong_chum": [
        r"VIÊM\s*TAI\s*[- ]?\s*XƯƠNG\s*CHŨM",
    ],

    "hau_phau_va_nhi": [
        r"HẬU\s*PHẪU\s*VÁ\s*NHĨ",
        r"VÁ\s*NHĨ",
    ],

    # --------------------------------------------------------
    # AURICLE
    # --------------------------------------------------------

    "viem_vanh_tai": [
        r"VIÊM\s*VÀNH\s*TAI",
        r"VIÊM\s*DA\s*VÀNH\s*TAI",
    ],

    "tu_dich_vanh_tai": [
        r"TỤ\s*DỊCH\s*VÀNH\s*TAI",
    ],

    "ro_luan_nhi": [
        r"RÒ\s*LUÂN\s*NHĨ",
    ],

    "not_vanh_tai": [
        r"NỐT\s*VÀNH\s*TAI",
    ],

    # --------------------------------------------------------
    # NOSE / SINUS
    # --------------------------------------------------------

    "viem_mui": [
        r"VIÊM\s*MŨI(?!\s*XOANG)",
    ],

    "viem_mui_xoang": [
        r"VIÊM\s*MŨI\s*XOANG",
    ],

    "viem_xoang": [
        r"VIÊM\s*XOANG",
    ],

    "polyp_mui": [
        r"POLYP.*MŨI",
    ],

    "chay_mau_mui": [
        r"CHẢY\s*MÁU\s*MŨI",
    ],

    "dich_vat_mui": [
        r"DỊ\s*VẬT\s*MŨI",
    ],

    "tien_dinh_mui": [
        r"VIÊM\s*TIỀN\s*ĐÌNH\s*MŨI",
        r"NHỌT\s*TIỀN\s*ĐÌNH\s*MŨI",
    ],

    "lech_vach_ngan": [
        r"LỆCH\s*VÁCH\s*NGĂN",
        r"VẸO\s*VÁCH\s*NGĂN",
    ],

    "hau_phau_mui_xoang": [
        r"HẬU\s*PHẪU\s*PTNSMX",
        r"HẬU\s*PHẪU\s*NỘI\s*SOI\s*MŨI\s*XOANG",
    ],

    "u_hoc_mui": [
        r"U\s*HỐC\s*MŨI",
    ],

    "u_nhu_hoc_mui": [
        r"U\s*NHÚ\s*HỐC\s*MŨI",
    ],

    "u_nhu_cuon_mui": [
        r"U\s*NHÚ\s*CUỐN\s*MŨI",
    ],

    "u_xoang": [
        r"U\s*XOANG",
    ],

    "seo_hoc_mui": [
        r"SẸO\s*HỐC\s*MŨI",
    ],

    # --------------------------------------------------------
    # NASOPHARYNX
    # --------------------------------------------------------

    "viem_va": [
        r"VIÊM\s*VA",
    ],

    "qua_phat_va": [
        r"VA\s*\+\s*A[- ]?MI[- ]?ĐAN\s*QUÁ\s*PHÁT",
        r"VA\s*QUÁ\s*PHÁT",
        r"QUÁ\s*PHÁT\s*VA",
    ],

    # --------------------------------------------------------
    # TONSIL
    # --------------------------------------------------------

    "viem_amidan": [
        r"VIÊM\s*A[- ]?MI[- ]?ĐAN",
        r"VIÊM\s*AMIDAN",
    ],

    "amidan_qua_phat": [
        r"A[- ]?MI[- ]?ĐAN\s*QUÁ\s*PHÁT",
        r"A[- ]?MI[- ]?ĐAN\s*ĐỘ\s*[I1VIX]+",
        r"AMIDAN\s*QUÁ\s*PHÁT",
    ],

    # --------------------------------------------------------
    # PHARYNX / LARYNX
    # --------------------------------------------------------

    "viem_hong": [
        r"VIÊM\s*HỌNG",
        r"VIÊM\s*HỌNG\s*MŨI",
    ],

    "viem_mieng": [
        r"VIÊM\s*MIỆNG",
    ],

    "viem_luoi": [
        r"VIÊM\s*LƯỠI",
    ],

    "viem_thanh_quan": [
        r"VIÊM\s*THANH\s*QUẢN",
    ],

    "hat_day_thanh": [
        r"HẠT\s*DÂY\s*THANH",
    ],

    "polyp_day_thanh": [
        r"POLYP\s*DÂY\s*THANH",
    ],

    "liet_day_thanh": [
        r"LIỆT\s*DÂY\s*THANH",
    ],

    "nang_day_thanh": [
        r"NANG\s*DÂY\s*THANH",
    ],

    "viem_amidan": [
        r"VIÊM\s*A[- ]?MI[- ]?ĐAN",
        r"VIÊM\s*AMIDAN",
    ],

    # --------------------------------------------------------
    # FOREIGN BODY
    # --------------------------------------------------------

    "hoc_xuong_ca": [
        r"HÓC\s*XƯƠNG\s*CÁ",
        r"DỊ\s*VẬT\s*\(\s*XƯƠNG\s*CÁ\s*\)",
        r"DỊ\s*VẬT.*XƯƠNG\s*CÁ",
    ],

    "di_vat_hong": [
        r"DỊ\s*VẬT\s*HỌNG",
        r"HÓC\s*DỊ\s*VẬT\s*N[12]",
    ],

    "di_vat_hong_thanh_quan": [
        r"DỊ\s*VẬT\s*HỌNG.*THANH\s*QUẢN",
    ],

    # --------------------------------------------------------
    # REFLUX
    # --------------------------------------------------------

    "theo_doi_trao_nguoc": [
        r"THEO\s*DÕI\s*TRÀO\s*NGƯỢC",
        r"TRÀO\s*NGƯỢC\s*DỊCH\s*VỊ",
    ],
}


# ============================================================
# 5. NEGATIVE FINDINGS
#
# These are separate from disease concepts.
# ============================================================

NEGATIVE_RULES = {

    "no_abnormal_external_middle_ear": [
        r"KHÔNG\s*THẤY.*BỆNH\s*LÝ\s*TAI\s*NGOÀI\s*TAI\s*GIỮA",
        r"NO\s*ABNORMAL\s*IMAGES\s*OF\s*THE\s*EXTERNAL\s*OR\s*MIDDLE\s*EAR",
    ],

    "no_abnormal_nose_sinus": [
        r"KHÔNG\s*PHÁT\s*HIỆN\s*BỆNH\s*LÝ\s*MŨI\s*XOANG",
        r"KHÔNG\s*THẤY.*BỆNH\s*LÝ.*MŨI\s*XOANG",
    ],

    "no_abnormal_ent": [
        r"CHƯA\s*PHÁT\s*HIỆN\s*BỆNH\s*LÝ\s*TAI\s*MŨI\s*HỌNG",
        r"HIỆN\s*KHÔNG\s*PHÁT\s*HIỆN\s*BỆNH\s*LÝ\s*TAI\s*MŨI\s*HỌNG",
    ],

    "no_bleeding": [
        r"KHÔNG\s*THẤY\s*ĐIỂM\s*CHẢY\s*MÁU",
        r"KHÔNG\s*THẤY\s*ĐIỂM\s*CHẢY\s*MÁU.*NỘI\s*SOI",
    ],

    "no_foreign_body": [
        r"KHÔNG\s*THẤY\s*HÌNH\s*ẢNH\s*DỊ\s*VẬT",
        r"KHÔNG\s*PHÁT\s*HIỆN\s*DỊ\s*VẬT",
    ],
}


# ============================================================
# 6. ATTRIBUTES
# ============================================================

ATTRIBUTE_RULES = {

    "acute": [
        r"\bCẤP\b",
        r"ĐỢT\s*CẤP",
    ],

    "chronic": [
        r"\bMẠN\b",
        r"MẠN\s*TÍNH",
    ],

    "bilateral": [
        r"\b2\s*BÊN\b",
        r"\bHAI\s*BÊN\b",
        r"\b2\s*TAI\b",
        r"\bHAI\s*TAI\b",
    ],

    "right": [
        r"\bP\b",
        r"\bPHẢI\b",
        r"\(P\)",
    ],

    "left": [
        r"\bT\b",
        r"\bTRÁI\b",
        r"\(T\)",
    ],

    "post_surgery": [
        r"\bĐÃ\s*PT\b",
        r"\bPTNS\b",
        r"ĐÃ\s*PHẪU\s*THUẬT",
        r"HẬU\s*PHẪU",
    ],
}


# ============================================================
# 7. DETECT FUNCTION
# ============================================================

def detect_rules(
    text,
    rules
):

    found = []

    for label, patterns in rules.items():

        for pattern in patterns:

            if re.search(
                pattern,
                text
            ):

                found.append(
                    label
                )

                break

    return sorted(
        set(found)
    )


def detect_attributes(text):

    attrs = detect_rules(
        text,
        ATTRIBUTE_RULES
    )

    # Bilateral means both sides.
    if "bilateral" in attrs:

        attrs = [
            a
            for a in attrs
            if a not in {
                "left",
                "right"
            }
        ]

    return attrs


# ============================================================
# 8. APPLY
# ============================================================

df["concepts_v2"] = df[
    "target_norm"
].map(
    lambda x:
    detect_rules(
        x,
        CONCEPT_RULES
    )
)

df["negative_findings"] = df[
    "target_norm"
].map(
    lambda x:
    detect_rules(
        x,
        NEGATIVE_RULES
    )
)

df["attributes_v2"] = df[
    "target_norm"
].map(
    detect_attributes
)


# ============================================================
# 9. IMPORTANT CONFLICT CLEANUP
#
# Explicit negative statement suppresses positive
# foreign-body labels when the sentence says "no foreign body".
# ============================================================

def cleanup_concepts(row):

    concepts = set(
        row["concepts_v2"]
    )

    negatives = set(
        row["negative_findings"]
    )

    if "no_foreign_body" in negatives:

        concepts.discard(
            "hoc_xuong_ca"
        )

        concepts.discard(
            "di_vat_hong"
        )

        concepts.discard(
            "di_vat_hong_thanh_quan"
        )

        concepts.discard(
            "di_vat_tai"
        )

        concepts.discard(
            "dich_vat_mui"
        )

    return sorted(
        concepts
    )


df["concepts_v2"] = df.apply(
    cleanup_concepts,
    axis=1
)


# ============================================================
# 10. MULTI-HOT
# ============================================================

concept_names = list(
    CONCEPT_RULES.keys()
)

negative_names = list(
    NEGATIVE_RULES.keys()
)

attribute_names = list(
    ATTRIBUTE_RULES.keys()
)

for c in concept_names:

    df[
        f"concept__{c}"
    ] = df[
        "concepts_v2"
    ].map(
        lambda x,
        c=c:
        int(c in x)
    )

for n in negative_names:

    df[
        f"negative__{n}"
    ] = df[
        "negative_findings"
    ].map(
        lambda x,
        n=n:
        int(n in x)
    )

for a in attribute_names:

    df[
        f"attribute__{a}"
    ] = df[
        "attributes_v2"
    ].map(
        lambda x,
        a=a:
        int(a in x)
    )


# ============================================================
# 11. HUMAN-READABLE COLUMNS
# ============================================================

df["concepts_v2_str"] = df[
    "concepts_v2"
].map(
    lambda x:
    "|".join(x)
)

df["negative_findings_str"] = df[
    "negative_findings"
].map(
    lambda x:
    "|".join(x)
)

df["attributes_v2_str"] = df[
    "attributes_v2"
].map(
    lambda x:
    "|".join(x)
)

df["num_concepts_v2"] = df[
    "concepts_v2"
].map(len)

df["num_negative_findings"] = df[
    "negative_findings"
].map(len)

df["num_attributes_v2"] = df[
    "attributes_v2"
].map(len)


# ============================================================
# 12. COVERAGE
# ============================================================

df["has_structured_label"] = (
    (
        df["num_concepts_v2"] > 0
    )
    |
    (
        df["num_negative_findings"] > 0
    )
)


# ============================================================
# 13. UNMAPPED
# ============================================================

unmapped = df[
    ~df["has_structured_label"]
].copy()

unmapped_freq = (
    unmapped[
        "target_norm"
    ]
    .value_counts()
    .reset_index()
)

unmapped_freq.columns = [
    "target",
    "count"
]


# ============================================================
# 14. CONCEPT FREQUENCY
# ============================================================

counter = Counter()

for labels in df[
    "concepts_v2"
]:

    counter.update(
        labels
    )

concept_freq = pd.DataFrame(
    counter.most_common(),
    columns=[
        "concept",
        "count"
    ]
)

concept_freq["percentage"] = (
    concept_freq["count"]
    /
    len(df)
    *
    100
)


# ============================================================
# 15. NEGATIVE FREQUENCY
# ============================================================

counter = Counter()

for labels in df[
    "negative_findings"
]:

    counter.update(
        labels
    )

negative_freq = pd.DataFrame(
    counter.most_common(),
    columns=[
        "negative_finding",
        "count"
    ]
)

negative_freq["percentage"] = (
    negative_freq["count"]
    /
    len(df)
    *
    100
)


# ============================================================
# 16. ATTRIBUTE FREQUENCY
# ============================================================

counter = Counter()

for labels in df[
    "attributes_v2"
]:

    counter.update(
        labels
    )

attribute_freq = pd.DataFrame(
    counter.most_common(),
    columns=[
        "attribute",
        "count"
    ]
)

attribute_freq["percentage"] = (
    attribute_freq["count"]
    /
    len(df)
    *
    100
)


# ============================================================
# 17. CONFLICT AUDIT
# ============================================================

# A true conflict now means:
# positive foreign body + explicit no foreign body.
#
# Other positive + negative findings are allowed because
# they can refer to different anatomical regions.

conflict_mask = (
    df["negative__no_foreign_body"] == 1
) & (
    (
        df["concept__hoc_xuong_ca"] == 1
    )
    |
    (
        df["concept__di_vat_hong"] == 1
    )
    |
    (
        df["concept__di_vat_hong_thanh_quan"] == 1
    )
    |
    (
        df["concept__di_vat_tai"] == 1
    )
    |
    (
        df["concept__dich_vat_mui"] == 1
    )
)

true_conflicts = df[
    conflict_mask
].copy()


# ============================================================
# 18. SAVE MASTER
# ============================================================

MASTER_OUT = os.path.join(
    OUTPUT_DIR,
    "v3_structured_targets_v2.csv"
)

df.to_csv(
    MASTER_OUT,
    index=False,
    encoding="utf-8-sig"
)

concept_freq.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "concept_frequency_v2.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

negative_freq.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "negative_frequency_v2.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

attribute_freq.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "attribute_frequency_v2.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

unmapped_freq.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "unmapped_v2.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

true_conflicts.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "true_conflicts_v2.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 19. SUMMARY
# ============================================================

print("\n" + "=" * 72)
print("V2 ONTOLOGY SUMMARY")
print("=" * 72)

print(
    f"\nPositive concepts: {len(CONCEPT_RULES)}"
)

print(
    f"Negative findings: {len(NEGATIVE_RULES)}"
)

print(
    f"Attributes: {len(ATTRIBUTE_RULES)}"
)

print(
    f"Cases: {len(df):,}"
)

print(
    f"\nStructured coverage: "
    f"{df['has_structured_label'].sum():,} / {len(df):,} "
    f"({df['has_structured_label'].mean()*100:.2f}%)"
)

print(
    f"Unmapped: "
    f"{len(unmapped):,} "
    f"({len(unmapped)/len(df)*100:.2f}%)"
)

print(
    f"TRUE conflicts: "
    f"{len(true_conflicts):,}"
)


print("\n" + "-" * 72)
print("TOP CONCEPTS")
print("-" * 72)

print(
    concept_freq
    .head(30)
    .to_string(index=False)
)


print("\n" + "-" * 72)
print("NEGATIVE FINDINGS")
print("-" * 72)

print(
    negative_freq
    .to_string(index=False)
)


print("\n" + "-" * 72)
print("ATTRIBUTES")
print("-" * 72)

print(
    attribute_freq
    .to_string(index=False)
)


print("\n" + "-" * 72)
print("UNMAPPED")
print("-" * 72)

print(
    unmapped_freq
    .head(100)
    .to_string(index=False)
)


print("\n" + "-" * 72)
print("TRUE CONFLICTS")
print("-" * 72)

if len(true_conflicts):

    print(
        true_conflicts[
            [
                "case_id",
                "ket_luan",
                "concepts_v2_str",
                "negative_findings_str"
            ]
        ]
        .head(100)
        .to_string(index=False)
    )

else:

    print(
        "No true conflicts."
    )


# ============================================================
# 20. FINAL CHECKS
# ============================================================

assert len(df) == 7606

assert (
    df["case_id"].nunique()
    == 7606
)

assert (
    df["ket_luan"].notna().all()
)

print("\n" + "=" * 72)
print("V3 ONTOLOGY V2 COMPLETE")
print("=" * 72)

print(
    "\nSaved:",
    MASTER_OUT
)

print("\n🟢 CPU PASS")

V3 ONTOLOGY V2 — FINAL STRUCTURED TARGET BUILDER

Cases: 7606

V2 ONTOLOGY SUMMARY

Positive concepts: 48
Negative findings: 5
Attributes: 6
Cases: 7,606

Structured coverage: 7,421 / 7,606 (97.57%)
Unmapped: 185 (2.43%)
TRUE conflicts: 0

------------------------------------------------------------------------
TOP CONCEPTS
------------------------------------------------------------------------
                  concept  count  percentage
                 viem_mui   3577   47.028662
           viem_mui_xoang   1489   19.576650
       viem_ong_tai_ngoai   1009   13.265843
            viem_tai_giua   1001   13.160663
                viem_hong    567    7.454641
      theo_doi_trao_nguoc    472    6.205627
                  viem_va    270    3.549829
                polyp_mui    264    3.470944
                  ray_tai    247    3.247436
               viem_xoang    192    2.524323
             chay_mau_mui    141    1.853800
          amidan_qua_phat    135    1.774915
          hau_ph

In [25]:
# ============================================================
# V3 ONTOLOGY V2.1 — COVERAGE PATCH
# 🟢 CPU
#
# Only fixes missing negative-finding variants.
# Does NOT change raw ket_luan.
# ============================================================

import os
import re
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/NoiSoi_Matching"

INPUT_FILE = os.path.join(
    PROJECT_DIR,
    "v3_ontology_v2",
    "v3_structured_targets_v2.csv"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "v3_ontology_v2"
)

df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

print("=" * 72)
print("V3 ONTOLOGY V2.1 — COVERAGE PATCH")
print("=" * 72)

# ------------------------------------------------------------
# Existing negative labels
# ------------------------------------------------------------

def split_pipe(x):

    if pd.isna(x):
        return []

    x = str(x).strip()

    if not x:
        return []

    return [
        v.strip()
        for v in x.split("|")
        if v.strip()
    ]


df["negative_findings"] = df[
    "negative_findings_str"
].map(split_pipe)

df["concepts_v2"] = df[
    "concepts_v2_str"
].map(split_pipe)

df["attributes_v2"] = df[
    "attributes_v2_str"
].map(split_pipe)


# ------------------------------------------------------------
# Normalize
# ------------------------------------------------------------

def norm(x):

    x = str(x).upper()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x.strip()


df["target_norm"] = df[
    "ket_luan"
].map(norm)


# ------------------------------------------------------------
# Additional variants
# ------------------------------------------------------------

EAR_NEGATIVE_PATTERNS = [

    # Main missing pattern
    r"KHÔNG\s*THẤY.*BỆNH\s*LÝ\s*TAI\s*NGOÀI\s*,\s*TAI\s*GIỮA",

    r"KHÔNG\s*THẤY.*BỆNH\s*LÝ\s*TAI\s*,\s*TAI\s*GIỮA",

    r"KHÔNG\s*THẤY\s*BỆNH\s*LÝ\s*TAI\s*NGOÀI\s*,\s*TAI\s*GIỮA",

    r"HIỆN\s*KHÔNG\s*THẤY.*BỆNH\s*LÝ\s*TAI\s*NGOÀI\s*,\s*TAI\s*GIỮA",

    r"KHÔNG\s*THẤY\s*BỆNH\s*LÝ\s*TAI\s*NGOÀI\s*,\s*TAI\s*GIỮA\s*TRÊN\s*NỘI\s*SOI",
]


def has_ear_negative(text):

    return any(
        re.search(
            pattern,
            text
        )
        for pattern in EAR_NEGATIVE_PATTERNS
    )


# ------------------------------------------------------------
# Apply
# ------------------------------------------------------------

patched = 0

for idx, row in df.iterrows():

    text = row["target_norm"]

    negatives = set(
        row["negative_findings"]
    )

    if (
        "no_abnormal_external_middle_ear"
        not in negatives
        and
        has_ear_negative(text)
    ):

        negatives.add(
            "no_abnormal_external_middle_ear"
        )

        df.at[
            idx,
            "negative_findings"
        ] = sorted(
            negatives
        )

        patched += 1


# ------------------------------------------------------------
# Rebuild strings
# ------------------------------------------------------------

df["negative_findings_str"] = df[
    "negative_findings"
].map(
    lambda x:
    "|".join(x)
)

df["num_negative_findings"] = df[
    "negative_findings"
].map(len)


# ------------------------------------------------------------
# Recompute structured coverage
# ------------------------------------------------------------

df["has_structured_label"] = (
    (
        df["concepts_v2"].map(len)
        > 0
    )
    |
    (
        df["negative_findings"].map(len)
        > 0
    )
)


unmapped = df[
    ~df["has_structured_label"]
].copy()

unmapped_freq = (
    unmapped[
        "target_norm"
    ]
    .value_counts()
    .reset_index()
)

unmapped_freq.columns = [
    "target",
    "count"
]


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    "v3_structured_targets_v2_1.csv"
)

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

unmapped_freq.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "unmapped_v2_1.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print(
    f"\nPatched cases: {patched:,}"
)

print(
    f"Structured coverage: "
    f"{df['has_structured_label'].sum():,} / {len(df):,} "
    f"({df['has_structured_label'].mean()*100:.2f}%)"
)

print(
    f"Unmapped: "
    f"{len(unmapped):,} "
    f"({len(unmapped)/len(df)*100:.2f}%)"
)

print("\nRemaining unmapped:")

print(
    unmapped_freq
    .head(100)
    .to_string(index=False)
)

assert len(df) == 7606
assert df["case_id"].nunique() == 7606

print("\n" + "=" * 72)
print("V2.1 PATCH COMPLETE")
print("=" * 72)

print(
    "\nSaved:",
    OUTPUT_FILE
)

print("\n🟢 CPU PASS")

V3 ONTOLOGY V2.1 — COVERAGE PATCH

Patched cases: 262
Structured coverage: 7,555 / 7,606 (99.33%)
Unmapped: 51 (0.67%)

Remaining unmapped:
                                                                     target  count
                                                                         VA      9
                                       HIỆN KHÔNG THẤY DỊ VẬT TRÊN NỘI SOI.      8
                     HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ MŨI, TAI TRÊN NỘI SOI      6
              HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ HỌNG THANH QUẢN TRÊN NỘI SOI      2
                 HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI MŨI HỌNG TRÊN NỘI SOI      1
                    HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI , MŨI TRÊN NỘI SOI      1
                      HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ MŨI,TAI TRÊN NỘI SOI      1
                     HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ MŨI ,TAI TRÊN NỘI SOI      1
             HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ HỌNG, THANH QUẢN TRÊN NỘI SOI      1
             HIỆN KHÔNG THẤY H

In [26]:
# ============================================================
# V3 ONTOLOGY V2.2 — FINAL COVERAGE CLEANUP
# 🟢 CPU
#
# Goal:
#   - Capture remaining explicit "no foreign body" findings
#   - Preserve genuinely unknown / ambiguous rare findings
#   - Freeze ontology afterwards
# ============================================================

import os
import re
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/NoiSoi_Matching"

INPUT_FILE = os.path.join(
    PROJECT_DIR,
    "v3_ontology_v2",
    "v3_structured_targets_v2_1.csv"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "v3_ontology_v2"
)

df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

print("=" * 72)
print("V3 ONTOLOGY V2.2 — FINAL COVERAGE CLEANUP")
print("=" * 72)

print(
    "\nLoaded:",
    len(df),
    "cases"
)


# ============================================================
# 1. RECONSTRUCT LABEL LISTS
# ============================================================

def split_pipe(x):

    if pd.isna(x):
        return []

    x = str(x).strip()

    if not x:
        return []

    return [
        v.strip()
        for v in x.split("|")
        if v.strip()
    ]


df["concepts_v2"] = df[
    "concepts_v2_str"
].map(split_pipe)

df["negative_findings"] = df[
    "negative_findings_str"
].map(split_pipe)

df["attributes_v2"] = df[
    "attributes_v2_str"
].map(split_pipe)


# ============================================================
# 2. NORMALIZE
# ============================================================

def norm(text):

    text = str(text).upper()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


df["target_norm"] = df[
    "ket_luan"
].map(norm)


# ============================================================
# 3. CATCH ALL EXPLICIT "NO FOREIGN BODY"
# ============================================================

NO_FOREIGN_BODY_PATTERN = re.compile(
    r"""
    KHÔNG\s*
    (?:THẤY|PHÁT\s*HIỆN)
    .*?
    DỊ\s*VẬT
    """,
    re.VERBOSE
)


patched = []

for idx, row in df.iterrows():

    text = row["target_norm"]

    negatives = set(
        row["negative_findings"]
    )

    if (
        NO_FOREIGN_BODY_PATTERN.search(text)
        and
        "no_foreign_body" not in negatives
    ):

        negatives.add(
            "no_foreign_body"
        )

        df.at[
            idx,
            "negative_findings"
        ] = sorted(
            negatives
        )

        patched.append(
            idx
        )


# ============================================================
# 4. REBUILD COLUMNS
# ============================================================

df["negative_findings_str"] = df[
    "negative_findings"
].map(
    lambda x:
    "|".join(x)
)

df["num_negative_findings"] = df[
    "negative_findings"
].map(len)

df["num_concepts_v2"] = df[
    "concepts_v2"
].map(len)

df["num_attributes_v2"] = df[
    "attributes_v2"
].map(len)


df["has_structured_label"] = (
    (
        df["num_concepts_v2"] > 0
    )
    |
    (
        df["num_negative_findings"] > 0
    )
)


# ============================================================
# 5. REMAINING UNMAPPED
# ============================================================

unmapped = df[
    ~df["has_structured_label"]
].copy()

unmapped_freq = (
    unmapped[
        "target_norm"
    ]
    .value_counts()
    .reset_index()
)

unmapped_freq.columns = [
    "target",
    "count"
]


# ============================================================
# 6. SAVE
# ============================================================

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    "v3_structured_targets_v2_2.csv"
)

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

unmapped.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "final_unmapped_cases_v2_2.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

unmapped_freq.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "final_unmapped_frequency_v2_2.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 7. SUMMARY
# ============================================================

coverage = (
    df["has_structured_label"].mean()
    * 100
)

print("\n" + "-" * 72)
print("FINAL COVERAGE")
print("-" * 72)

print(
    f"Structured: "
    f"{df['has_structured_label'].sum():,} / "
    f"{len(df):,} "
    f"({coverage:.2f}%)"
)

print(
    f"Unmapped: "
    f"{len(unmapped):,} "
    f"({100-coverage:.2f}%)"
)

print(
    f"New no-foreign-body patches: "
    f"{len(patched):,}"
)


# ============================================================
# 8. REMAINING UNMAPPED
# ============================================================

print("\n" + "-" * 72)
print("REMAINING UNMAPPED")
print("-" * 72)

print(
    unmapped_freq
    .to_string(index=False)
)


# ============================================================
# 9. SANITY CHECK
# ============================================================

assert len(df) == 7606

assert (
    df["case_id"].nunique()
    == 7606
)

assert (
    df["ket_luan"].notna().all()
)

print("\n" + "=" * 72)
print("V3 ONTOLOGY V2.2 COMPLETE")
print("=" * 72)

print(
    "\nSaved:",
    OUTPUT_FILE
)

print("\n🟢 CPU PASS")

V3 ONTOLOGY V2.2 — FINAL COVERAGE CLEANUP

Loaded: 7606 cases

------------------------------------------------------------------------
FINAL COVERAGE
------------------------------------------------------------------------
Structured: 7,567 / 7,606 (99.49%)
Unmapped: 39 (0.51%)
New no-foreign-body patches: 18

------------------------------------------------------------------------
REMAINING UNMAPPED
------------------------------------------------------------------------
                                                                     target  count
                                                                         VA      9
                     HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ MŨI, TAI TRÊN NỘI SOI      6
              HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ HỌNG THANH QUẢN TRÊN NỘI SOI      2
             HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ HỌNG- THANH QUẢN TRÊN NỘI SOI      1
                 HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI MŨI HỌNG TRÊN NỘI SOI      1
                      HI

In [27]:
# ============================================================
# V3-1 — FREEZE ONTOLOGY + BUILD MULTI-TASK MANIFEST
# 🟢 CPU ONLY
#
# Frozen ontology:
#   V2.2
#
# Raw target:
#   ket_luan
#
# Structured supervision:
#   48 concepts
#   5 negative findings
#   6 attributes
#
# Unmapped cases remain in dataset and receive
# report-generation loss only.
# ============================================================

import os
import json
import pandas as pd
import numpy as np

print("=" * 72)
print("V3-1 — FREEZE ONTOLOGY + BUILD TRAINING MANIFEST")
print("=" * 72)


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = (
    "/content/drive/MyDrive/NoiSoi_Matching"
)

INPUT_FILE = os.path.join(
    PROJECT_DIR,
    "v3_ontology_v2",
    "v3_structured_targets_v2_2.csv"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "baseline_model_v3"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

MANIFEST_OUT = os.path.join(
    OUTPUT_DIR,
    "v3_training_manifest.csv"
)

CONFIG_OUT = os.path.join(
    OUTPUT_DIR,
    "v3_config.json"
)


# ============================================================
# 2. LOAD
# ============================================================

df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

print(
    "\nLoaded cases:",
    len(df)
)


# ============================================================
# 3. REQUIRED COLUMNS
# ============================================================

required = [
    "case_id",
    "patient_group_id",
    "split",
    "ket_luan",
    "concepts_v2_str",
    "negative_findings_str",
    "attributes_v2_str",
    "has_structured_label",
]

missing = [
    c for c in required
    if c not in df.columns
]

if missing:

    raise ValueError(
        f"Missing columns: {missing}"
    )


# ============================================================
# 4. DEFINE ONTOLOGY FROM ACTUAL DATA
#
# We extract label names from the frozen V2.2 manifest.
# This avoids silently changing the ontology.
# ============================================================

def split_labels(x):

    if pd.isna(x):
        return []

    x = str(x).strip()

    if not x:
        return []

    return sorted([
        v.strip()
        for v in x.split("|")
        if v.strip()
    ])


df["concept_labels"] = df[
    "concepts_v2_str"
].map(split_labels)

df["negative_labels"] = df[
    "negative_findings_str"
].map(split_labels)

df["attribute_labels"] = df[
    "attributes_v2_str"
].map(split_labels)


# ============================================================
# 5. FREEZE LABEL VOCABULARIES
# ============================================================

concept_vocab = sorted({
    label
    for labels in df["concept_labels"]
    for label in labels
})

negative_vocab = sorted({
    label
    for labels in df["negative_labels"]
    for label in labels
})

attribute_vocab = sorted({
    label
    for labels in df["attribute_labels"]
    for label in labels
})


print("\nFrozen vocabularies:")

print(
    "Concepts:",
    len(concept_vocab)
)

print(
    "Negative findings:",
    len(negative_vocab)
)

print(
    "Attributes:",
    len(attribute_vocab)
)


# ============================================================
# 6. MULTI-HOT VECTORS
# ============================================================

concept_index = {
    label: i
    for i, label in enumerate(
        concept_vocab
    )
}

negative_index = {
    label: i
    for i, label in enumerate(
        negative_vocab
    )
}

attribute_index = {
    label: i
    for i, label in enumerate(
        attribute_vocab
    )
}


def make_multihot(
    labels,
    index
):

    vector = np.zeros(
        len(index),
        dtype=np.float32
    )

    for label in labels:

        if label in index:

            vector[
                index[label]
            ] = 1.0

    return vector


df["concept_vector"] = df[
    "concept_labels"
].map(
    lambda x:
    make_multihot(
        x,
        concept_index
    )
)

df["negative_vector"] = df[
    "negative_labels"
].map(
    lambda x:
    make_multihot(
        x,
        negative_index
    )
)

df["attribute_vector"] = df[
    "attribute_labels"
].map(
    lambda x:
    make_multihot(
        x,
        attribute_index
    )
)


# ============================================================
# 7. STRUCTURED SUPERVISION MASK
#
# 1 = use auxiliary losses
# 0 = report-only case
# ============================================================

df["structured_loss_mask"] = (
    df["has_structured_label"]
    .astype(np.float32)
)


# ============================================================
# 8. SERIALIZE MULTI-HOT VECTORS
#
# CSV-friendly representation.
# ============================================================

df["concept_vector_str"] = df[
    "concept_vector"
].map(
    lambda x:
    ",".join(
        str(int(v))
        for v in x
    )
)

df["negative_vector_str"] = df[
    "negative_vector"
].map(
    lambda x:
    ",".join(
        str(int(v))
        for v in x
    )
)

df["attribute_vector_str"] = df[
    "attribute_vector"
].map(
    lambda x:
    ",".join(
        str(int(v))
        for v in x
    )
)


# ============================================================
# 9. SANITY CHECKS
# ============================================================

assert len(df) == 7606

assert (
    df["case_id"].nunique()
    == 7606
)

assert (
    df["patient_group_id"]
    .notna()
    .all()
)

assert (
    df["split"]
    .isin([
        "train",
        "val",
        "test"
    ])
    .all()
)

assert (
    df["ket_luan"]
    .notna()
    .all()
)


# ============================================================
# 10. SPLIT SUMMARY
# ============================================================

split_summary = (
    df.groupby(
        "split"
    )
    .agg(
        cases=(
            "case_id",
            "count"
        ),
        structured=(
            "structured_loss_mask",
            "sum"
        )
    )
    .reset_index()
)

split_summary[
    "unmapped"
] = (
    split_summary["cases"]
    -
    split_summary["structured"]
)


print("\n" + "-" * 72)
print("SPLIT SUMMARY")
print("-" * 72)

print(
    split_summary.to_string(
        index=False
    )
)


# ============================================================
# 11. LABEL FREQUENCY SANITY
# ============================================================

concept_counts = np.zeros(
    len(concept_vocab),
    dtype=np.int64
)

negative_counts = np.zeros(
    len(negative_vocab),
    dtype=np.int64
)

attribute_counts = np.zeros(
    len(attribute_vocab),
    dtype=np.int64
)


for labels in df[
    "concept_labels"
]:

    for label in labels:

        concept_counts[
            concept_index[label]
        ] += 1


for labels in df[
    "negative_labels"
]:

    for label in labels:

        negative_counts[
            negative_index[label]
        ] += 1


for labels in df[
    "attribute_labels"
]:

    for label in labels:

        attribute_counts[
            attribute_index[label]
        ] += 1


concept_frequency = pd.DataFrame({
    "label": concept_vocab,
    "count": concept_counts
}).sort_values(
    "count",
    ascending=False
)

negative_frequency = pd.DataFrame({
    "label": negative_vocab,
    "count": negative_counts
}).sort_values(
    "count",
    ascending=False
)

attribute_frequency = pd.DataFrame({
    "label": attribute_vocab,
    "count": attribute_counts
}).sort_values(
    "count",
    ascending=False
)


# ============================================================
# 12. SAVE TRAINING MANIFEST
# ============================================================

# Remove numpy arrays before CSV.
save_columns = [
    "case_id",
    "patient_group_id",
    "split",
    "ket_luan",

    "concepts_v2_str",
    "negative_findings_str",
    "attributes_v2_str",

    "concept_vector_str",
    "negative_vector_str",
    "attribute_vector_str",

    "has_structured_label",
    "structured_loss_mask",

    "num_concepts_v2",
    "num_negative_findings",
    "num_attributes_v2",
]

df[
    save_columns
].to_csv(
    MANIFEST_OUT,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 13. SAVE FROZEN CONFIG
# ============================================================

config = {

    "ontology_version":
        "V3-Ontology-V2.2",

    "num_cases":
        int(len(df)),

    "num_structured_cases":
        int(
            df[
                "has_structured_label"
            ].sum()
        ),

    "num_unmapped_cases":
        int(
            (~df[
                "has_structured_label"
            ]).sum()
        ),

    "concept_vocab":
        concept_vocab,

    "negative_vocab":
        negative_vocab,

    "attribute_vocab":
        attribute_vocab,

    "num_concepts":
        len(concept_vocab),

    "num_negative_findings":
        len(negative_vocab),

    "num_attributes":
        len(attribute_vocab),

    "structured_loss_policy":
        "Apply auxiliary losses only when structured_loss_mask=1",

    "raw_target":
        "ket_luan",

    "raw_target_policy":
        "Always train report generation loss",

    "image_encoder":
        "google/vit-base-patch16-224",

    "text_decoder":
        "google/mt5-small",

    "max_images_per_case":
        8,

    "image_size":
        224,

    "max_target_length":
        96,

    "batch_size":
        4,

    "gradient_accumulation":
        4,

    "bf16":
        True,
}


with open(
    CONFIG_OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 14. SAVE FREQUENCIES
# ============================================================

concept_frequency.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "concept_frequency_frozen.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

negative_frequency.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "negative_frequency_frozen.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

attribute_frequency.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "attribute_frequency_frozen.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 15. FINAL REPORT
# ============================================================

print("\n" + "=" * 72)
print("V3 ONTOLOGY FROZEN")
print("=" * 72)

print(
    "\nOntology:",
    "V3-Ontology-V2.2"
)

print(
    "Cases:",
    len(df)
)

print(
    "Structured:",
    int(
        df["structured_loss_mask"].sum()
    )
)

print(
    "Report-only:",
    int(
        (df["structured_loss_mask"] == 0)
        .sum()
    )
)

print(
    "Concept labels:",
    len(concept_vocab)
)

print(
    "Negative labels:",
    len(negative_vocab)
)

print(
    "Attribute labels:",
    len(attribute_vocab)
)

print("\nSaved:")

print(
    MANIFEST_OUT
)

print(
    CONFIG_OUT
)

print("\n🟢 CPU PASS")

V3-1 — FREEZE ONTOLOGY + BUILD TRAINING MANIFEST

Loaded cases: 7606

Frozen vocabularies:
Concepts: 48
Negative findings: 5
Attributes: 6

------------------------------------------------------------------------
SPLIT SUMMARY
------------------------------------------------------------------------
split  cases  structured  unmapped
 test    712       709.0       3.0
train   6138      6107.0      31.0
  val    756       751.0       5.0

V3 ONTOLOGY FROZEN

Ontology: V3-Ontology-V2.2
Cases: 7606
Structured: 7567
Report-only: 39
Concept labels: 48
Negative labels: 5
Attribute labels: 6

Saved:
/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v3/v3_training_manifest.csv
/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v3/v3_config.json

🟢 CPU PASS


In [28]:
# ============================================================
# V3-2 — MULTI-TASK ARCHITECTURE DRY-RUN
# 🔴 A100
#
# V3:
#   ViT-B/16 (trainable)
#       ├── Concept head: 48
#       ├── Negative head: 5
#       ├── Attribute head: 6
#       └── Visual prefix → mT5-small → raw ket_luan
#
# NO TRAINING.
# One complete forward + backward only.
# ============================================================

import os
import json
import gc
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from torchvision import transforms

from transformers import (
    AutoTokenizer,
    ViTModel,
    MT5ForConditionalGeneration,
)

# ============================================================
# 0. CLEAN / DEVICE
# ============================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

USE_BF16 = (
    DEVICE.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

print("=" * 72)
print("V3-2 — MULTI-TASK ARCHITECTURE DRY-RUN")
print("=" * 72)

print(
    "\nDevice:",
    DEVICE
)

if DEVICE.type == "cuda":

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )

print(
    "BF16:",
    USE_BF16
)


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = (
    "/content/drive/MyDrive/NoiSoi_Matching"
)

V3_DIR = os.path.join(
    PROJECT_DIR,
    "baseline_model_v3"
)

MANIFEST_FILE = os.path.join(
    V3_DIR,
    "v3_training_manifest.csv"
)

CONFIG_FILE = os.path.join(
    V3_DIR,
    "v3_config.json"
)

IMAGE_MANIFEST = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

ARCH_CONFIG = os.path.join(
    V3_DIR,
    "v3_architecture_config.json"
)

os.makedirs(
    V3_DIR,
    exist_ok=True
)


# ============================================================
# 2. LOAD CONFIG + MANIFEST
# ============================================================

with open(
    CONFIG_FILE,
    "r",
    encoding="utf-8"
) as f:

    v3_config = json.load(f)


df = pd.read_csv(
    MANIFEST_FILE,
    low_memory=False
)

image_df = pd.read_csv(
    IMAGE_MANIFEST,
    low_memory=False
)

print("\nManifest cases:", len(df))
print("Image manifest:", len(image_df))


# ============================================================
# 3. VALIDATE FROZEN ONTOLOGY
# ============================================================

CONCEPT_VOCAB = v3_config[
    "concept_vocab"
]

NEGATIVE_VOCAB = v3_config[
    "negative_vocab"
]

ATTRIBUTE_VOCAB = v3_config[
    "attribute_vocab"
]

NUM_CONCEPTS = len(
    CONCEPT_VOCAB
)

NUM_NEGATIVE = len(
    NEGATIVE_VOCAB
)

NUM_ATTRIBUTES = len(
    ATTRIBUTE_VOCAB
)

print("\nFrozen ontology:")
print(
    "Concepts:",
    NUM_CONCEPTS
)
print(
    "Negative:",
    NUM_NEGATIVE
)
print(
    "Attributes:",
    NUM_ATTRIBUTES
)

assert NUM_CONCEPTS == 48
assert NUM_NEGATIVE == 5
assert NUM_ATTRIBUTES == 6


# ============================================================
# 4. MODEL CONFIG
# ============================================================

VISION_MODEL = (
    "google/vit-base-patch16-224"
)

TEXT_MODEL = (
    "google/mt5-small"
)

IMAGE_SIZE = 224
MAX_IMAGES = 8
MAX_TARGET_LENGTH = 96

BATCH_SIZE = 4

VISUAL_DIM = 768
MT5_DIM = 512

print("\nModel:")
print(
    "Vision:",
    VISION_MODEL
)
print(
    "Text:",
    TEXT_MODEL
)
print(
    "Max images:",
    MAX_IMAGES
)


# ============================================================
# 5. TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL
)

print(
    "\nTokenizer vocab:",
    len(tokenizer)
)


# ============================================================
# 6. IMAGE TRANSFORM
# ============================================================

image_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    ),
])


# ============================================================
# 7. CASE DATASET
# ============================================================

normal_images = image_df[
    image_df["image_status"] == "NORMAL"
].copy()

normal_images = normal_images[
    normal_images["case_id"].isin(
        set(df["case_id"])
    )
].copy()


# group once
case_to_images = (
    normal_images
    .groupby("case_id")["image_path"]
    .apply(list)
    .to_dict()
)


def parse_vector(
    value,
    expected_length
):

    values = [
        float(x)
        for x in str(value).split(",")
        if str(x).strip() != ""
    ]

    if len(values) != expected_length:

        raise ValueError(
            f"Expected {expected_length} "
            f"values, got {len(values)}"
        )

    return torch.tensor(
        values,
        dtype=torch.float32
    )


class V3Dataset(Dataset):

    def __init__(
        self,
        frame,
        train=False
    ):

        self.df = frame.reset_index(
            drop=True
        )

        self.train = train

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        case_id = row["case_id"]

        paths = case_to_images.get(
            case_id,
            []
        )

        if len(paths) == 0:

            raise RuntimeError(
                f"No NORMAL images: {case_id}"
            )

        # ----------------------------------------------------
        # Same policy as V1/V2
        # ----------------------------------------------------

        if (
            self.train
            and len(paths) > MAX_IMAGES
        ):

            selected = random.sample(
                paths,
                MAX_IMAGES
            )

        else:

            selected = paths[
                :MAX_IMAGES
            ]

        images = []

        for path in selected:

            img = Image.open(
                path
            ).convert("RGB")

            img = image_transform(
                img
            )

            images.append(img)

        images = torch.stack(
            images
        )

        target = str(
            row["ket_luan"]
        ).strip()

        concept = parse_vector(
            row["concept_vector_str"],
            NUM_CONCEPTS
        )

        negative = parse_vector(
            row["negative_vector_str"],
            NUM_NEGATIVE
        )

        attribute = parse_vector(
            row["attribute_vector_str"],
            NUM_ATTRIBUTES
        )

        structured_mask = torch.tensor(
            float(
                row["structured_loss_mask"]
            ),
            dtype=torch.float32
        )

        return {
            "case_id": case_id,
            "patient_group_id":
                row["patient_group_id"],
            "images": images,
            "num_images":
                images.shape[0],
            "target_text": target,
            "concept_targets": concept,
            "negative_targets": negative,
            "attribute_targets": attribute,
            "structured_mask":
                structured_mask,
        }


# ============================================================
# 8. COLLATE
# ============================================================

def collate_fn(batch):

    return {
        "case_id": [
            x["case_id"]
            for x in batch
        ],

        "patient_group_id": [
            x["patient_group_id"]
            for x in batch
        ],

        "images": [
            x["images"]
            for x in batch
        ],

        "num_images": torch.tensor(
            [
                x["num_images"]
                for x in batch
            ],
            dtype=torch.long
        ),

        "target_text": [
            x["target_text"]
            for x in batch
        ],

        "concept_targets": torch.stack([
            x["concept_targets"]
            for x in batch
        ]),

        "negative_targets": torch.stack([
            x["negative_targets"]
            for x in batch
        ]),

        "attribute_targets": torch.stack([
            x["attribute_targets"]
            for x in batch
        ]),

        "structured_mask": torch.stack([
            x["structured_mask"]
            for x in batch
        ]),
    }


# ============================================================
# 9. BUILD DATASET
# ============================================================

train_df = df[
    df["split"] == "train"
].copy()

dataset = V3Dataset(
    train_df,
    train=True
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn
)

batch = next(
    iter(loader)
)

print("\nBatch:")
print(
    "Cases:",
    batch["case_id"]
)

print(
    "Image counts:",
    batch["num_images"].tolist()
)

print(
    "Concept targets:",
    tuple(
        batch[
            "concept_targets"
        ].shape
    )
)

print(
    "Negative targets:",
    tuple(
        batch[
            "negative_targets"
        ].shape
    )
)

print(
    "Attribute targets:",
    tuple(
        batch[
            "attribute_targets"
        ].shape
    )
)

print(
    "Structured mask:",
    batch[
        "structured_mask"
    ].tolist()
)


# ============================================================
# 10. MODEL CLASSES
# ============================================================

class VisualProjector(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        output_dim
    ):

        super().__init__()

        self.proj = nn.Linear(
            input_dim,
            output_dim
        )

    def forward(self, x):

        return self.proj(x)


class V3MultiTaskModel(
    nn.Module
):

    def __init__(
        self,
        vision,
        mt5,
        num_concepts,
        num_negative,
        num_attributes
    ):

        super().__init__()

        self.vision = vision

        self.mt5 = mt5

        self.projector = VisualProjector(
            VISUAL_DIM,
            MT5_DIM
        )

        # ----------------------------------------------------
        # Case-level structured heads
        #
        # Input = mean pooled projected visual features
        # ----------------------------------------------------

        self.concept_head = nn.Linear(
            MT5_DIM,
            num_concepts
        )

        self.negative_head = nn.Linear(
            MT5_DIM,
            num_negative
        )

        self.attribute_head = nn.Linear(
            MT5_DIM,
            num_attributes
        )

    def encode_images(
        self,
        image_list
    ):

        batch_size = len(
            image_list
        )

        counts = [
            x.shape[0]
            for x in image_list
        ]

        flat_images = torch.cat(
            image_list,
            dim=0
        )

        # [total_images, 3, 224, 224]
        vision_out = self.vision(
            pixel_values=flat_images
        )

        cls = vision_out.last_hidden_state[
            :,
            0,
            :
        ]

        # [total_images, 768]
        projected = self.projector(
            cls
        )

        # ----------------------------------------------------
        # Split back into cases
        # ----------------------------------------------------

        prefix_list = []

        start = 0

        for count in counts:

            prefix_list.append(
                projected[
                    start:start + count
                ]
            )

            start += count

        # ----------------------------------------------------
        # Pad visual prefix
        # ----------------------------------------------------

        max_count = max(counts)

        prefix = projected.new_zeros(
            (
                batch_size,
                max_count,
                MT5_DIM
            )
        )

        attention = torch.zeros(
            (
                batch_size,
                max_count
            ),
            dtype=torch.long,
            device=projected.device
        )

        pooled = []

        for i, item in enumerate(
            prefix_list
        ):

            n = item.shape[0]

            prefix[
                i,
                :n
            ] = item

            attention[
                i,
                :n
            ] = 1

            pooled.append(
                item.mean(
                    dim=0
                )
            )

        pooled = torch.stack(
            pooled
        )

        return (
            prefix,
            attention,
            pooled
        )


# ============================================================
# 11. LOAD MODELS
# ============================================================

print("\nLoading ViT...")

vision = ViTModel.from_pretrained(
    VISION_MODEL
)

print(
    "ViT hidden:",
    vision.config.hidden_size
)

print("\nLoading mT5...")

mt5 = MT5ForConditionalGeneration.from_pretrained(
    TEXT_MODEL
)

print(
    "mT5 d_model:",
    mt5.config.d_model
)


# ============================================================
# 12. BUILD MODEL
# ============================================================

model = V3MultiTaskModel(
    vision=vision,
    mt5=mt5,
    num_concepts=NUM_CONCEPTS,
    num_negative=NUM_NEGATIVE,
    num_attributes=NUM_ATTRIBUTES
)

model = model.to(
    DEVICE
)

# ViT trainable — same as V2
for p in model.vision.parameters():
    p.requires_grad = True

# mT5 trainable
for p in model.mt5.parameters():
    p.requires_grad = True


# ============================================================
# 13. PARAMETER SUMMARY
# ============================================================

def count_params(module):

    total = sum(
        p.numel()
        for p in module.parameters()
    )

    trainable = sum(
        p.numel()
        for p in module.parameters()
        if p.requires_grad
    )

    return total, trainable


print("\n" + "-" * 72)
print("PARAMETERS")
print("-" * 72)

for name, module in [
    ("ViT", model.vision),
    ("mT5", model.mt5),
    ("Projector", model.projector),
    ("Concept head", model.concept_head),
    ("Negative head", model.negative_head),
    ("Attribute head", model.attribute_head),
]:

    total, trainable = count_params(
        module
    )

    print(
        f"{name:16s}"
        f" total={total:,}"
        f" trainable={trainable:,}"
    )


# ============================================================
# 14. MOVE BATCH TO DEVICE
# ============================================================

images = [
    x.to(
        DEVICE,
        non_blocking=True
    )
    for x in batch["images"]
]

concept_targets = batch[
    "concept_targets"
].to(DEVICE)

negative_targets = batch[
    "negative_targets"
].to(DEVICE)

attribute_targets = batch[
    "attribute_targets"
].to(DEVICE)

structured_mask = batch[
    "structured_mask"
].to(DEVICE)


# ============================================================
# 15. TOKENIZE REPORT TARGET
# ============================================================

tokenized = tokenizer(
    batch["target_text"],
    padding=True,
    truncation=True,
    max_length=MAX_TARGET_LENGTH,
    return_tensors="pt"
)

labels = tokenized[
    "input_ids"
].to(
    DEVICE
)

labels[
    labels == tokenizer.pad_token_id
] = -100


# ============================================================
# 16. FORWARD
# ============================================================

print("\nRunning forward...")

if DEVICE.type == "cuda":

    torch.cuda.reset_peak_memory_stats()


with torch.autocast(
    device_type=DEVICE.type,
    dtype=torch.bfloat16,
    enabled=USE_BF16
):

    visual_prefix, visual_attention, pooled = (
        model.encode_images(
            images
        )
    )

    # --------------------------------------------------------
    # Structured heads
    # --------------------------------------------------------

    concept_logits = model.concept_head(
        pooled
    )

    negative_logits = model.negative_head(
        pooled
    )

    attribute_logits = model.attribute_head(
        pooled
    )

    # --------------------------------------------------------
    # Report generation
    # --------------------------------------------------------

    text_embeds = model.mt5.shared(
        labels.clamp_min(0)
    )

    # We only need the encoder-side visual prefix.
    # mT5 encoder receives:
    #   [visual tokens]
    #
    # Report labels are used by the normal
    # MT5 conditional-generation loss.
    # --------------------------------------------------------

    encoder_outputs = model.mt5.encoder(
        inputs_embeds=visual_prefix
    )

    report_output = model.mt5(
        encoder_outputs=encoder_outputs,
        attention_mask=visual_attention,
        labels=labels,
        return_dict=True
    )

    report_loss = report_output.loss

    # --------------------------------------------------------
    # Structured losses
    # --------------------------------------------------------

    concept_loss_all = F.binary_cross_entropy_with_logits(
        concept_logits,
        concept_targets,
        reduction="none"
    ).mean(dim=1)

    negative_loss_all = F.binary_cross_entropy_with_logits(
        negative_logits,
        negative_targets,
        reduction="none"
    ).mean(dim=1)

    attribute_loss_all = F.binary_cross_entropy_with_logits(
        attribute_logits,
        attribute_targets,
        reduction="none"
    ).mean(dim=1)

    # Only structured-labeled cases contribute
    # to auxiliary losses.
    mask_sum = structured_mask.sum().clamp_min(1.0)

    concept_loss = (
        concept_loss_all
        * structured_mask
    ).sum() / mask_sum

    negative_loss = (
        negative_loss_all
        * structured_mask
    ).sum() / mask_sum

    attribute_loss = (
        attribute_loss_all
        * structured_mask
    ).sum() / mask_sum


# ============================================================
# 17. INITIAL LOSS WEIGHTS
# ============================================================

LAMBDA_CONCEPT = 0.5
LAMBDA_NEGATIVE = 0.5
LAMBDA_ATTRIBUTE = 0.25

total_loss = (
    report_loss
    + LAMBDA_CONCEPT * concept_loss
    + LAMBDA_NEGATIVE * negative_loss
    + LAMBDA_ATTRIBUTE * attribute_loss
)


# ============================================================
# 18. PRINT FORWARD RESULTS
# ============================================================

print("\n" + "-" * 72)
print("FORWARD RESULTS")
print("-" * 72)

print(
    "Visual prefix:",
    tuple(
        visual_prefix.shape
    )
)

print(
    "Visual attention:",
    tuple(
        visual_attention.shape
    )
)

print(
    "Pooled:",
    tuple(
        pooled.shape
    )
)

print(
    "Concept logits:",
    tuple(
        concept_logits.shape
    )
)

print(
    "Negative logits:",
    tuple(
        negative_logits.shape
    )
)

print(
    "Attribute logits:",
    tuple(
        attribute_logits.shape
    )
)

print(
    "Report logits:",
    tuple(
        report_output.logits.shape
    )
)

print("\nLosses:")

print(
    f"Report loss:     "
    f"{report_loss.item():.6f}"
)

print(
    f"Concept loss:    "
    f"{concept_loss.item():.6f}"
)

print(
    f"Negative loss:   "
    f"{negative_loss.item():.6f}"
)

print(
    f"Attribute loss:  "
    f"{attribute_loss.item():.6f}"
)

print(
    f"Total loss:      "
    f"{total_loss.item():.6f}"
)


# ============================================================
# 19. BACKWARD DRY-RUN
# ============================================================

print("\nRunning backward...")

total_loss.backward()

print(
    "Backward completed."
)


# ============================================================
# 20. GRADIENT CHECK
# ============================================================

print("\n" + "-" * 72)
print("GRADIENT CHECK")
print("-" * 72)


def grad_norm(module):

    values = []

    for p in module.parameters():

        if (
            p.requires_grad
            and p.grad is not None
        ):

            values.append(
                p.grad.detach()
                .float()
                .norm()
                .item()
            )

    if not values:
        return 0.0

    return float(
        np.sqrt(
            np.sum(
                np.square(values)
            )
        )
    )


modules = [
    ("ViT", model.vision),
    ("Projector", model.projector),
    ("mT5", model.mt5),
    ("Concept", model.concept_head),
    ("Negative", model.negative_head),
    ("Attribute", model.attribute_head),
]

gradient_results = {}

for name, module in modules:

    norm = grad_norm(
        module
    )

    gradient_results[name] = norm

    print(
        f"{name:12s}: "
        f"{norm:.6f}"
    )


# ============================================================
# 21. VRAM
# ============================================================

peak_memory_gb = None

if DEVICE.type == "cuda":

    peak_memory_gb = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

    print(
        "\nPeak GPU memory:",
        f"{peak_memory_gb:.2f} GB"
    )


# ============================================================
# 22. SANITY ASSERTIONS
# ============================================================

assert visual_prefix.shape[0] == BATCH_SIZE

assert visual_prefix.shape[2] == MT5_DIM

assert concept_logits.shape == (
    BATCH_SIZE,
    NUM_CONCEPTS
)

assert negative_logits.shape == (
    BATCH_SIZE,
    NUM_NEGATIVE
)

assert attribute_logits.shape == (
    BATCH_SIZE,
    NUM_ATTRIBUTES
)

assert torch.isfinite(
    total_loss
).item()

for name, norm in gradient_results.items():

    assert np.isfinite(norm), (
        f"Invalid gradient: {name}"
    )

    assert norm > 0, (
        f"No gradient: {name}"
    )


# ============================================================
# 23. SAVE ARCHITECTURE CONFIG
# ============================================================

architecture_config = {

    "version":
        "V3",

    "ontology":
        "V3-Ontology-V2.2",

    "num_concepts":
        NUM_CONCEPTS,

    "num_negative_findings":
        NUM_NEGATIVE,

    "num_attributes":
        NUM_ATTRIBUTES,

    "vision_model":
        VISION_MODEL,

    "text_model":
        TEXT_MODEL,

    "vision_trainable":
        True,

    "text_trainable":
        True,

    "max_images":
        MAX_IMAGES,

    "image_size":
        IMAGE_SIZE,

    "max_target_length":
        MAX_TARGET_LENGTH,

    "batch_size":
        BATCH_SIZE,

    "structured_pooling":
        "mean_projected_image_cls",

    "report_visual_tokens":
        "per-image_projected_cls",

    "lambda_concept":
        LAMBDA_CONCEPT,

    "lambda_negative":
        LAMBDA_NEGATIVE,

    "lambda_attribute":
        LAMBDA_ATTRIBUTE,

    "bf16":
        USE_BF16,

    "peak_memory_gb":
        peak_memory_gb,

    "dry_run_losses": {
        "report":
            float(report_loss.item()),
        "concept":
            float(concept_loss.item()),
        "negative":
            float(negative_loss.item()),
        "attribute":
            float(attribute_loss.item()),
        "total":
            float(total_loss.item()),
    },

    "gradient_norms":
        gradient_results,
}


with open(
    ARCH_CONFIG,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        architecture_config,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 24. CLEANUP
# ============================================================

del total_loss
del report_output
del encoder_outputs
del visual_prefix
del visual_attention
del pooled

gc.collect()

if DEVICE.type == "cuda":

    torch.cuda.empty_cache()


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 72)
print("V3-2 ARCHITECTURE DRY-RUN COMPLETE")
print("=" * 72)

print(
    "\nSaved:",
    ARCH_CONFIG
)

print(
    "\n🔴 A100 PASS"
)

V3-2 — MULTI-TASK ARCHITECTURE DRY-RUN

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.49 GB
BF16: True

Manifest cases: 7606
Image manifest: 76405

Frozen ontology:
Concepts: 48
Negative: 5
Attributes: 6

Model:
Vision: google/vit-base-patch16-224
Text: google/mt5-small
Max images: 8

Tokenizer vocab: 250100

Batch:
Cases: ['13495.13495.0.13514', '12904.12904.0.12922', '18268.18268.0.18293', '13368.13368.0.13387']
Image counts: [5, 6, 8, 8]
Concept targets: (4, 48)
Negative targets: (4, 5)
Attribute targets: (4, 6)
Structured mask: [1.0, 1.0, 1.0, 1.0]

Loading ViT...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ViT hidden: 768

Loading mT5...


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


mT5 d_model: 512

------------------------------------------------------------------------
PARAMETERS
------------------------------------------------------------------------
ViT              total=86,389,248 trainable=86,389,248
mT5              total=300,176,768 trainable=300,176,768
Projector        total=393,728 trainable=393,728
Concept head     total=24,624 trainable=24,624
Negative head    total=2,565 trainable=2,565
Attribute head   total=3,078 trainable=3,078

Running forward...

------------------------------------------------------------------------
FORWARD RESULTS
------------------------------------------------------------------------
Visual prefix: (4, 8, 512)
Visual attention: (4, 8)
Pooled: (4, 512)
Concept logits: (4, 48)
Negative logits: (4, 5)
Attribute logits: (4, 6)
Report logits: (4, 24, 250112)

Losses:
Report loss:     23.098948
Concept loss:    0.698207
Negative loss:   0.804399
Attribute loss:  0.723770
Total loss:      24.031193

Running backward...
Backward 

In [29]:
# ============================================================
# V3-3 — MULTI-TASK TRAINING
# 🔴 A100
#
# Auto-resume:
#   baseline_model_v3/checkpoints/last.pt
#
# Objective:
#   L_report
# + 0.50 * L_concept
# + 0.50 * L_negative
# + 0.25 * L_attribute
#
# Same backbone/training setup as V2.
# ============================================================

import os
import json
import gc
import random
import time
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from transformers import (
    AutoTokenizer,
    ViTModel,
    MT5ForConditionalGeneration,
    get_cosine_schedule_with_warmup,
)

# ============================================================
# 0. DEVICE
# ============================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

USE_BF16 = (
    DEVICE.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

assert DEVICE.type == "cuda", (
    "V3 training requires CUDA/A100."
)

print("=" * 72)
print("V3-3 — MULTI-TASK TRAINING")
print("=" * 72)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3,
        2
    ),
    "GB"
)

print(
    "BF16:",
    USE_BF16
)


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = (
    "/content/drive/MyDrive/NoiSoi_Matching"
)

V3_DIR = os.path.join(
    PROJECT_DIR,
    "baseline_model_v3"
)

MANIFEST_FILE = os.path.join(
    V3_DIR,
    "v3_training_manifest.csv"
)

CONFIG_FILE = os.path.join(
    V3_DIR,
    "v3_config.json"
)

CHECKPOINT_DIR = os.path.join(
    V3_DIR,
    "checkpoints"
)

METRIC_DIR = os.path.join(
    V3_DIR,
    "metrics"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

os.makedirs(
    METRIC_DIR,
    exist_ok=True
)

LAST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "last.pt"
)

BEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best.pt"
)

HISTORY_FILE = os.path.join(
    METRIC_DIR,
    "training_history.csv"
)


# ============================================================
# 2. CONFIG
# ============================================================

VISION_MODEL = (
    "google/vit-base-patch16-224"
)

TEXT_MODEL = (
    "google/mt5-small"
)

IMAGE_SIZE = 224
MAX_IMAGES = 8
MAX_TARGET_LENGTH = 96

BATCH_SIZE = 4
GRAD_ACCUMULATION = 4

EPOCHS = 5

LR_VIT = 1e-5
LR_PROJECTOR = 1e-4
LR_MT5 = 5e-5

WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0

LAMBDA_CONCEPT = 0.50
LAMBDA_NEGATIVE = 0.50
LAMBDA_ATTRIBUTE = 0.25

SEED = 42


# ============================================================
# 3. SEED
# ============================================================

def seed_everything(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)


# ============================================================
# 4. LOAD MANIFEST
# ============================================================

with open(
    CONFIG_FILE,
    "r",
    encoding="utf-8"
) as f:

    v3_config = json.load(f)


df = pd.read_csv(
    MANIFEST_FILE,
    low_memory=False
)

IMAGE_MANIFEST = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

image_df = pd.read_csv(
    IMAGE_MANIFEST,
    low_memory=False
)

normal_images = image_df[
    image_df["image_status"] == "NORMAL"
].copy()

normal_images = normal_images[
    normal_images["case_id"].isin(
        set(df["case_id"])
    )
].copy()

case_to_images = (
    normal_images
    .groupby("case_id")["image_path"]
    .apply(list)
    .to_dict()
)

print("\nCases:", len(df))
print(
    "NORMAL images:",
    len(normal_images)
)


# ============================================================
# 5. ONTOLOGY
# ============================================================

CONCEPT_VOCAB = v3_config[
    "concept_vocab"
]

NEGATIVE_VOCAB = v3_config[
    "negative_vocab"
]

ATTRIBUTE_VOCAB = v3_config[
    "attribute_vocab"
]

NUM_CONCEPTS = len(
    CONCEPT_VOCAB
)

NUM_NEGATIVE = len(
    NEGATIVE_VOCAB
)

NUM_ATTRIBUTES = len(
    ATTRIBUTE_VOCAB
)

assert NUM_CONCEPTS == 48
assert NUM_NEGATIVE == 5
assert NUM_ATTRIBUTES == 6


# ============================================================
# 6. TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL
)


# ============================================================
# 7. IMAGE TRANSFORM
# ============================================================

image_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    ),
])


# ============================================================
# 8. DATASET
# ============================================================

def parse_vector(
    value,
    expected_length
):

    values = [
        float(x)
        for x in str(value).split(",")
        if str(x).strip()
    ]

    if len(values) != expected_length:

        raise ValueError(
            f"Vector length {len(values)} "
            f"!= expected {expected_length}"
        )

    return torch.tensor(
        values,
        dtype=torch.float32
    )


class V3Dataset(Dataset):

    def __init__(
        self,
        frame,
        train=False
    ):

        self.df = frame.reset_index(
            drop=True
        )

        self.train = train

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        case_id = row["case_id"]

        paths = case_to_images.get(
            case_id,
            []
        )

        if not paths:

            raise RuntimeError(
                f"No NORMAL images: {case_id}"
            )

        # Same image policy as V1/V2
        if (
            self.train
            and len(paths) > MAX_IMAGES
        ):

            selected = random.sample(
                paths,
                MAX_IMAGES
            )

        else:

            selected = paths[
                :MAX_IMAGES
            ]

        images = []

        for path in selected:

            image = Image.open(
                path
            ).convert("RGB")

            image = image_transform(
                image
            )

            images.append(image)

        images = torch.stack(
            images
        )

        return {
            "case_id":
                case_id,

            "patient_group_id":
                row["patient_group_id"],

            "images":
                images,

            "num_images":
                images.shape[0],

            "target_text":
                str(
                    row["ket_luan"]
                ).strip(),

            "concept_targets":
                parse_vector(
                    row["concept_vector_str"],
                    NUM_CONCEPTS
                ),

            "negative_targets":
                parse_vector(
                    row["negative_vector_str"],
                    NUM_NEGATIVE
                ),

            "attribute_targets":
                parse_vector(
                    row["attribute_vector_str"],
                    NUM_ATTRIBUTES
                ),

            "structured_mask":
                torch.tensor(
                    float(
                        row[
                            "structured_loss_mask"
                        ]
                    ),
                    dtype=torch.float32
                ),
        }


def collate_fn(batch):

    return {

        "case_id": [
            x["case_id"]
            for x in batch
        ],

        "patient_group_id": [
            x["patient_group_id"]
            for x in batch
        ],

        "images": [
            x["images"]
            for x in batch
        ],

        "num_images": torch.tensor(
            [
                x["num_images"]
                for x in batch
            ],
            dtype=torch.long
        ),

        "target_text": [
            x["target_text"]
            for x in batch
        ],

        "concept_targets": torch.stack([
            x["concept_targets"]
            for x in batch
        ]),

        "negative_targets": torch.stack([
            x["negative_targets"]
            for x in batch
        ]),

        "attribute_targets": torch.stack([
            x["attribute_targets"]
            for x in batch
        ]),

        "structured_mask": torch.stack([
            x["structured_mask"]
            for x in batch
        ]),
    }


# ============================================================
# 9. SPLITS
# ============================================================

train_df = df[
    df["split"] == "train"
].copy()

val_df = df[
    df["split"] == "val"
].copy()

test_df = df[
    df["split"] == "test"
].copy()

train_dataset = V3Dataset(
    train_df,
    train=True
)

val_dataset = V3Dataset(
    val_df,
    train=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn
)

print("\nSplit:")
print(
    "Train:",
    len(train_dataset)
)

print(
    "Val:",
    len(val_dataset)
)

print(
    "Test:",
    len(test_df)
)


# ============================================================
# 10. MODEL
# ============================================================

class VisualProjector(nn.Module):

    def __init__(
        self,
        input_dim,
        output_dim
    ):

        super().__init__()

        self.proj = nn.Linear(
            input_dim,
            output_dim
        )

    def forward(self, x):

        return self.proj(x)


class V3MultiTaskModel(nn.Module):

    def __init__(
        self,
        vision,
        mt5,
        num_concepts,
        num_negative,
        num_attributes
    ):

        super().__init__()

        self.vision = vision

        self.mt5 = mt5

        self.projector = VisualProjector(
            768,
            512
        )

        self.concept_head = nn.Linear(
            512,
            num_concepts
        )

        self.negative_head = nn.Linear(
            512,
            num_negative
        )

        self.attribute_head = nn.Linear(
            512,
            num_attributes
        )

    def encode_images(
        self,
        image_list
    ):

        batch_size = len(
            image_list
        )

        counts = [
            x.shape[0]
            for x in image_list
        ]

        flat_images = torch.cat(
            image_list,
            dim=0
        )

        vision_out = self.vision(
            pixel_values=flat_images
        )

        cls = vision_out.last_hidden_state[
            :,
            0,
            :
        ]

        projected = self.projector(
            cls
        )

        prefix_list = []

        start = 0

        for count in counts:

            prefix_list.append(
                projected[
                    start:start + count
                ]
            )

            start += count

        max_count = max(
            counts
        )

        prefix = projected.new_zeros(
            (
                batch_size,
                max_count,
                512
            )
        )

        attention = torch.zeros(
            (
                batch_size,
                max_count
            ),
            dtype=torch.long,
            device=projected.device
        )

        pooled = []

        for i, item in enumerate(
            prefix_list
        ):

            n = item.shape[0]

            prefix[
                i,
                :n
            ] = item

            attention[
                i,
                :n
            ] = 1

            pooled.append(
                item.mean(
                    dim=0
                )
            )

        pooled = torch.stack(
            pooled
        )

        return (
            prefix,
            attention,
            pooled
        )


# ============================================================
# 11. LOAD BACKBONE
# ============================================================

print("\nLoading ViT...")

vision = ViTModel.from_pretrained(
    VISION_MODEL
)

print("Loading mT5...")

mt5 = MT5ForConditionalGeneration.from_pretrained(
    TEXT_MODEL
)

model = V3MultiTaskModel(
    vision=vision,
    mt5=mt5,
    num_concepts=NUM_CONCEPTS,
    num_negative=NUM_NEGATIVE,
    num_attributes=NUM_ATTRIBUTES
)

model = model.to(
    DEVICE
)

model.train()


# ============================================================
# 12. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    [
        {
            "params":
                model.vision.parameters(),
            "lr":
                LR_VIT,
        },

        {
            "params":
                model.projector.parameters(),
            "lr":
                LR_PROJECTOR,
        },

        {
            "params":
                model.mt5.parameters(),
            "lr":
                LR_MT5,
        },

        {
            "params":
                model.concept_head.parameters(),
            "lr":
                LR_PROJECTOR,
        },

        {
            "params":
                model.negative_head.parameters(),
            "lr":
                LR_PROJECTOR,
        },

        {
            "params":
                model.attribute_head.parameters(),
            "lr":
                LR_PROJECTOR,
        },
    ],
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 13. SCHEDULER
# ============================================================

steps_per_epoch = int(
    np.ceil(
        len(train_loader)
        / GRAD_ACCUMULATION
    )
)

total_optimizer_steps = (
    steps_per_epoch
    * EPOCHS
)

warmup_steps = max(
    1,
    int(
        total_optimizer_steps
        * WARMUP_RATIO
    )
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_optimizer_steps
)

print("\nTraining:")
print(
    "Epochs:",
    EPOCHS
)

print(
    "Batches/epoch:",
    len(train_loader)
)

print(
    "Optimizer steps/epoch:",
    steps_per_epoch
)

print(
    "Total optimizer steps:",
    total_optimizer_steps
)

print(
    "Warmup steps:",
    warmup_steps
)


# ============================================================
# 14. LOSS FUNCTION
# ============================================================

def compute_losses(
    model,
    batch
):

    images = [
        x.to(
            DEVICE,
            non_blocking=True
        )
        for x in batch["images"]
    ]

    concept_targets = batch[
        "concept_targets"
    ].to(
        DEVICE,
        non_blocking=True
    )

    negative_targets = batch[
        "negative_targets"
    ].to(
        DEVICE,
        non_blocking=True
    )

    attribute_targets = batch[
        "attribute_targets"
    ].to(
        DEVICE,
        non_blocking=True
    )

    structured_mask = batch[
        "structured_mask"
    ].to(
        DEVICE,
        non_blocking=True
    )

    tokenized = tokenizer(
        batch["target_text"],
        padding=True,
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
        return_tensors="pt"
    )

    labels = tokenized[
        "input_ids"
    ].to(
        DEVICE,
        non_blocking=True
    )

    labels[
        labels == tokenizer.pad_token_id
    ] = -100

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
        enabled=USE_BF16
    ):

        (
            visual_prefix,
            visual_attention,
            pooled
        ) = model.encode_images(
            images
        )

        concept_logits = (
            model.concept_head(
                pooled
            )
        )

        negative_logits = (
            model.negative_head(
                pooled
            )
        )

        attribute_logits = (
            model.attribute_head(
                pooled
            )
        )

        encoder_outputs = (
            model.mt5.encoder(
                inputs_embeds=visual_prefix
            )
        )

        report_output = model.mt5(
            encoder_outputs=encoder_outputs,
            attention_mask=visual_attention,
            labels=labels,
            return_dict=True
        )

        report_loss = (
            report_output.loss
        )

        concept_loss_all = (
            F.binary_cross_entropy_with_logits(
                concept_logits,
                concept_targets,
                reduction="none"
            ).mean(dim=1)
        )

        negative_loss_all = (
            F.binary_cross_entropy_with_logits(
                negative_logits,
                negative_targets,
                reduction="none"
            ).mean(dim=1)
        )

        attribute_loss_all = (
            F.binary_cross_entropy_with_logits(
                attribute_logits,
                attribute_targets,
                reduction="none"
            ).mean(dim=1)
        )

        mask_sum = (
            structured_mask.sum()
            .clamp_min(1.0)
        )

        concept_loss = (
            concept_loss_all
            * structured_mask
        ).sum() / mask_sum

        negative_loss = (
            negative_loss_all
            * structured_mask
        ).sum() / mask_sum

        attribute_loss = (
            attribute_loss_all
            * structured_mask
        ).sum() / mask_sum

        total_loss = (
            report_loss
            + LAMBDA_CONCEPT
              * concept_loss
            + LAMBDA_NEGATIVE
              * negative_loss
            + LAMBDA_ATTRIBUTE
              * attribute_loss
        )

    return {
        "total": total_loss,
        "report": report_loss,
        "concept": concept_loss,
        "negative": negative_loss,
        "attribute": attribute_loss,
    }


# ============================================================
# 15. CHECKPOINT HELPERS
# ============================================================

def get_rng_state():

    return {
        "python":
            random.getstate(),

        "numpy":
            np.random.get_state(),

        "torch":
            torch.get_rng_state(),

        "cuda":
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None,
    }


def restore_rng_state(
    state
):

    if state is None:
        return

    random.setstate(
        state["python"]
    )

    np.random.set_state(
        state["numpy"]
    )

    torch.set_rng_state(
        state["torch"]
    )

    if (
        torch.cuda.is_available()
        and state.get("cuda") is not None
    ):

        torch.cuda.set_rng_state_all(
            state["cuda"]
        )


def save_checkpoint(
    path,
    epoch,
    global_step,
    best_val_loss,
    history
):

    checkpoint = {

        "epoch":
            epoch,

        "global_step":
            global_step,

        "best_val_loss":
            best_val_loss,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "history":
            history,

        "rng_state":
            get_rng_state(),

        "config": {
            "vision_model":
                VISION_MODEL,

            "text_model":
                TEXT_MODEL,

            "epochs":
                EPOCHS,

            "batch_size":
                BATCH_SIZE,

            "gradient_accumulation":
                GRAD_ACCUMULATION,

            "lr_vit":
                LR_VIT,

            "lr_projector":
                LR_PROJECTOR,

            "lr_mt5":
                LR_MT5,

            "lambda_concept":
                LAMBDA_CONCEPT,

            "lambda_negative":
                LAMBDA_NEGATIVE,

            "lambda_attribute":
                LAMBDA_ATTRIBUTE,

            "ontology":
                "V3-Ontology-V2.2",
        },
    }

    torch.save(
        checkpoint,
        path
    )


# ============================================================
# 16. RESUME
# ============================================================

start_epoch = 0
global_step = 0

best_val_loss = float(
    "inf"
)

history = []

if os.path.exists(
    LAST_CHECKPOINT
):

    print("\n" + "=" * 72)
    print("RESUMING FROM CHECKPOINT")
    print("=" * 72)

    checkpoint = torch.load(
        LAST_CHECKPOINT,
        map_location="cpu",
        weights_only=False
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    optimizer.load_state_dict(
        checkpoint[
            "optimizer_state_dict"
        ]
    )

    scheduler.load_state_dict(
        checkpoint[
            "scheduler_state_dict"
        ]
    )

    start_epoch = (
        checkpoint["epoch"] + 1
    )

    global_step = (
        checkpoint["global_step"]
    )

    best_val_loss = (
        checkpoint["best_val_loss"]
    )

    history = (
        checkpoint.get(
            "history",
            []
        )
    )

    restore_rng_state(
        checkpoint.get(
            "rng_state"
        )
    )

    print(
        "Resume epoch:",
        start_epoch + 1
    )

    print(
        "Global step:",
        global_step
    )

    print(
        "Best val loss:",
        best_val_loss
    )

    del checkpoint
    gc.collect()

else:

    print(
        "\nNo previous checkpoint."
    )

    print(
        "Starting V3 from scratch."
    )


# ============================================================
# 17. VALIDATION
# ============================================================

@torch.no_grad()
def validate():

    model.eval()

    totals = {
        "total": 0.0,
        "report": 0.0,
        "concept": 0.0,
        "negative": 0.0,
        "attribute": 0.0,
    }

    count = 0

    for batch in val_loader:

        losses = compute_losses(
            model,
            batch
        )

        bs = len(
            batch["case_id"]
        )

        for key in totals:

            totals[key] += (
                losses[key].item()
                * bs
            )

        count += bs

    model.train()

    return {
        key:
            totals[key] / count
        for key in totals
    }


# ============================================================
# 18. TRAINING LOOP
# ============================================================

print("\n" + "=" * 72)
print("TRAINING START")
print("=" * 72)

for epoch in range(
    start_epoch,
    EPOCHS
):

    epoch_start = time.time()

    model.train()

    running = {
        "total": 0.0,
        "report": 0.0,
        "concept": 0.0,
        "negative": 0.0,
        "attribute": 0.0,
    }

    optimizer.zero_grad(
        set_to_none=True
    )

    num_batches = len(
        train_loader
    )

    for batch_idx, batch in enumerate(
        train_loader
    ):

        losses = compute_losses(
            model,
            batch
        )

        # Accumulate raw losses
        for key in running:

            running[key] += (
                losses[key]
                .detach()
                .float()
                .item()
            )

        # Gradient accumulation
        scaled_loss = (
            losses["total"]
            / GRAD_ACCUMULATION
        )

        scaled_loss.backward()

        is_last_batch = (
            batch_idx
            == num_batches - 1
        )

        should_step = (
            (
                batch_idx + 1
            )
            % GRAD_ACCUMULATION
            == 0
            or is_last_batch
        )

        if should_step:

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                MAX_GRAD_NORM
            )

            optimizer.step()

            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )

            global_step += 1

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (
            (batch_idx + 1) % 100 == 0
            or batch_idx == 0
        ):

            print(
                f"Epoch {epoch+1}/{EPOCHS} | "
                f"Batch {batch_idx+1}/{num_batches} | "
                f"Loss "
                f"{losses['total'].item():.4f}"
            )

    # ========================================================
    # TRAIN MEANS
    # ========================================================

    train_metrics = {
        key:
            running[key] / num_batches
        for key in running
    }

    # ========================================================
    # VALIDATION
    # ========================================================

    val_metrics = validate()

    epoch_time = (
        time.time()
        - epoch_start
    )

    row = {
        "epoch":
            epoch + 1,

        "global_step":
            global_step,

        "train_total":
            train_metrics["total"],

        "train_report":
            train_metrics["report"],

        "train_concept":
            train_metrics["concept"],

        "train_negative":
            train_metrics["negative"],

        "train_attribute":
            train_metrics["attribute"],

        "val_total":
            val_metrics["total"],

        "val_report":
            val_metrics["report"],

        "val_concept":
            val_metrics["concept"],

        "val_negative":
            val_metrics["negative"],

        "val_attribute":
            val_metrics["attribute"],

        "lr_vit":
            optimizer.param_groups[0]["lr"],

        "lr_projector":
            optimizer.param_groups[1]["lr"],

        "lr_mt5":
            optimizer.param_groups[2]["lr"],

        "epoch_seconds":
            epoch_time,
    }

    history.append(
        row
    )

    # ========================================================
    # SAVE HISTORY
    # ========================================================

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_FILE,
        index=False
    )

    # ========================================================
    # SAVE LAST
    # ========================================================

    save_checkpoint(
        LAST_CHECKPOINT,
        epoch,
        global_step,
        best_val_loss,
        history
    )

    # ========================================================
    # BEST
    # ========================================================

    if (
        val_metrics["total"]
        < best_val_loss
    ):

        best_val_loss = (
            val_metrics["total"]
        )

        save_checkpoint(
            BEST_CHECKPOINT,
            epoch,
            global_step,
            best_val_loss,
            history
        )

        is_best = True

    else:

        is_best = False

    # Save epoch checkpoint too
    epoch_checkpoint = os.path.join(
        CHECKPOINT_DIR,
        f"epoch_{epoch+1:02d}.pt"
    )

    save_checkpoint(
        epoch_checkpoint,
        epoch,
        global_step,
        best_val_loss,
        history
    )

    # ========================================================
    # REPORT
    # ========================================================

    print("\n" + "-" * 72)

    print(
        f"Epoch {epoch+1}/{EPOCHS}"
    )

    print(
        f"Train total:     "
        f"{train_metrics['total']:.6f}"
    )

    print(
        f"Train report:    "
        f"{train_metrics['report']:.6f}"
    )

    print(
        f"Train concept:   "
        f"{train_metrics['concept']:.6f}"
    )

    print(
        f"Train negative:  "
        f"{train_metrics['negative']:.6f}"
    )

    print(
        f"Train attribute: "
        f"{train_metrics['attribute']:.6f}"
    )

    print()

    print(
        f"Val total:       "
        f"{val_metrics['total']:.6f}"
    )

    print(
        f"Val report:      "
        f"{val_metrics['report']:.6f}"
    )

    print(
        f"Val concept:     "
        f"{val_metrics['concept']:.6f}"
    )

    print(
        f"Val negative:    "
        f"{val_metrics['negative']:.6f}"
    )

    print(
        f"Val attribute:   "
        f"{val_metrics['attribute']:.6f}"
    )

    print()

    print(
        "Epoch time:",
        f"{epoch_time:.1f}s"
    )

    print(
        "Best val total:",
        f"{best_val_loss:.6f}"
    )

    print(
        "Best checkpoint:",
        is_best
    )

    print(
        "-" * 72
    )

    # --------------------------------------------------------
    # Free some memory between epochs
    # --------------------------------------------------------

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 19. FINAL
# ============================================================

print("\n" + "=" * 72)
print("V3 TRAINING COMPLETE")
print("=" * 72)

print(
    "\nBest validation loss:",
    f"{best_val_loss:.6f}"
)

print(
    "\nLast checkpoint:",
    LAST_CHECKPOINT
)

print(
    "Best checkpoint:",
    BEST_CHECKPOINT
)

print(
    "History:",
    HISTORY_FILE
)

print("\n🔴 A100 TRAINING PASS")

V3-3 — MULTI-TASK TRAINING
GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.49 GB
BF16: True

Cases: 7606
NORMAL images: 76209

Split:
Train: 6138
Val: 756
Test: 712

Loading ViT...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading mT5...


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



Training:
Epochs: 5
Batches/epoch: 1535
Optimizer steps/epoch: 384
Total optimizer steps: 1920
Warmup steps: 96

No previous checkpoint.
Starting V3 from scratch.

TRAINING START
Epoch 1/5 | Batch 1/1535 | Loss 23.6209
Epoch 1/5 | Batch 100/1535 | Loss 28.3917
Epoch 1/5 | Batch 200/1535 | Loss 32.4319
Epoch 1/5 | Batch 300/1535 | Loss 21.4921
Epoch 1/5 | Batch 400/1535 | Loss 14.8719
Epoch 1/5 | Batch 500/1535 | Loss 9.3995
Epoch 1/5 | Batch 600/1535 | Loss 9.2320
Epoch 1/5 | Batch 700/1535 | Loss 7.3551
Epoch 1/5 | Batch 800/1535 | Loss 5.2610
Epoch 1/5 | Batch 900/1535 | Loss 3.1176
Epoch 1/5 | Batch 1000/1535 | Loss 3.9681
Epoch 1/5 | Batch 1100/1535 | Loss 2.6294
Epoch 1/5 | Batch 1200/1535 | Loss 1.8685
Epoch 1/5 | Batch 1300/1535 | Loss 2.0204
Epoch 1/5 | Batch 1400/1535 | Loss 1.6797
Epoch 1/5 | Batch 1500/1535 | Loss 1.4148

------------------------------------------------------------------------
Epoch 1/5
Train total:     9.880334
Train report:    9.552218
Train concept:   0.

In [2]:
# ============================================================
# V3-4 — TEST GENERATION + STRUCTURED EVALUATION
# 🔴 A100
#
# Loads best.pt only.
# NO TRAINING.
# ============================================================

import os
import gc
import json
import random
import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from transformers import (
    AutoTokenizer,
    ViTModel,
    MT5ForConditionalGeneration,
)

# ============================================================
# 0. DEVICE
# ============================================================

gc.collect()
torch.cuda.empty_cache()

DEVICE = torch.device("cuda")

USE_BF16 = torch.cuda.is_bf16_supported()

print("=" * 72)
print("V3-4 — TEST GENERATION + STRUCTURED EVALUATION")
print("=" * 72)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "BF16:",
    USE_BF16
)


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = (
    "/content/drive/MyDrive/NoiSoi_Matching"
)

V3_DIR = os.path.join(
    PROJECT_DIR,
    "baseline_model_v3"
)

MANIFEST_FILE = os.path.join(
    V3_DIR,
    "v3_training_manifest.csv"
)

CONFIG_FILE = os.path.join(
    V3_DIR,
    "v3_config.json"
)

IMAGE_MANIFEST = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

BEST_CHECKPOINT = os.path.join(
    V3_DIR,
    "checkpoints",
    "best.pt"
)

OUTPUT_DIR = os.path.join(
    V3_DIR,
    "test_evaluation"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

assert os.path.exists(
    BEST_CHECKPOINT
)


# ============================================================
# 2. LOAD CONFIG / DATA
# ============================================================

with open(
    CONFIG_FILE,
    "r",
    encoding="utf-8"
) as f:

    config = json.load(f)

df = pd.read_csv(
    MANIFEST_FILE,
    low_memory=False
)

image_df = pd.read_csv(
    IMAGE_MANIFEST,
    low_memory=False
)

normal_images = image_df[
    image_df["image_status"] == "NORMAL"
].copy()

normal_images = normal_images[
    normal_images["case_id"].isin(
        set(df["case_id"])
    )
].copy()

case_to_images = (
    normal_images
    .groupby("case_id")["image_path"]
    .apply(list)
    .to_dict()
)

print(
    "\nCases:",
    len(df)
)

print(
    "NORMAL images:",
    len(normal_images)
)


# ============================================================
# 3. ONTOLOGY
# ============================================================

CONCEPT_VOCAB = config[
    "concept_vocab"
]

NEGATIVE_VOCAB = config[
    "negative_vocab"
]

ATTRIBUTE_VOCAB = config[
    "attribute_vocab"
]

NUM_CONCEPTS = len(
    CONCEPT_VOCAB
)

NUM_NEGATIVE = len(
    NEGATIVE_VOCAB
)

NUM_ATTRIBUTES = len(
    ATTRIBUTE_VOCAB
)

assert NUM_CONCEPTS == 48
assert NUM_NEGATIVE == 5
assert NUM_ATTRIBUTES == 6


# ============================================================
# 4. MODEL CONFIG
# ============================================================

VISION_MODEL = (
    "google/vit-base-patch16-224"
)

TEXT_MODEL = (
    "google/mt5-small"
)

IMAGE_SIZE = 224
MAX_IMAGES = 8
MAX_TARGET_LENGTH = 96
BATCH_SIZE = 4


# ============================================================
# 5. TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL
)


# ============================================================
# 6. IMAGE TRANSFORM
# ============================================================

image_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    ),
])


# ============================================================
# 7. HELPERS
# ============================================================

def parse_vector(
    value,
    expected_length
):

    values = [
        float(x)
        for x in str(value).split(",")
        if str(x).strip()
    ]

    assert len(values) == expected_length

    return torch.tensor(
        values,
        dtype=torch.float32
    )


# ============================================================
# 8. TEST DATASET
# ============================================================

class V3TestDataset(Dataset):

    def __init__(
        self,
        frame
    ):

        self.df = frame.reset_index(
            drop=True
        )

    def __len__(self):

        return len(self.df)

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        case_id = row["case_id"]

        paths = case_to_images[
            case_id
        ]

        # Same deterministic policy as V2
        selected = paths[
            :MAX_IMAGES
        ]

        images = []

        for path in selected:

            image = Image.open(
                path
            ).convert("RGB")

            image = image_transform(
                image
            )

            images.append(image)

        images = torch.stack(
            images
        )

        return {

            "case_id":
                case_id,

            "patient_group_id":
                row["patient_group_id"],

            "images":
                images,

            "target_text":
                str(
                    row["ket_luan"]
                ).strip(),

            "concept_targets":
                parse_vector(
                    row["concept_vector_str"],
                    NUM_CONCEPTS
                ),

            "negative_targets":
                parse_vector(
                    row["negative_vector_str"],
                    NUM_NEGATIVE
                ),

            "attribute_targets":
                parse_vector(
                    row["attribute_vector_str"],
                    NUM_ATTRIBUTES
                ),

            "structured_mask":
                float(
                    row["structured_loss_mask"]
                ),
        }


def collate_fn(batch):

    return {

        "case_id": [
            x["case_id"]
            for x in batch
        ],

        "patient_group_id": [
            x["patient_group_id"]
            for x in batch
        ],

        "images": [
            x["images"]
            for x in batch
        ],

        "target_text": [
            x["target_text"]
            for x in batch
        ],

        "concept_targets": torch.stack([
            x["concept_targets"]
            for x in batch
        ]),

        "negative_targets": torch.stack([
            x["negative_targets"]
            for x in batch
        ]),

        "attribute_targets": torch.stack([
            x["attribute_targets"]
            for x in batch
        ]),

        "structured_mask": torch.tensor(
            [
                x["structured_mask"]
                for x in batch
            ],
            dtype=torch.float32
        ),
    }


test_df = df[
    df["split"] == "test"
].copy()

test_dataset = V3TestDataset(
    test_df
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn
)

print(
    "\nTest cases:",
    len(test_dataset)
)


# ============================================================
# 9. MODEL
# ============================================================

class VisualProjector(nn.Module):

    def __init__(
        self,
        input_dim,
        output_dim
    ):

        super().__init__()

        self.proj = nn.Linear(
            input_dim,
            output_dim
        )

    def forward(self, x):

        return self.proj(x)


class V3MultiTaskModel(nn.Module):

    def __init__(
        self,
        vision,
        mt5
    ):

        super().__init__()

        self.vision = vision
        self.mt5 = mt5

        self.projector = VisualProjector(
            768,
            512
        )

        self.concept_head = nn.Linear(
            512,
            NUM_CONCEPTS
        )

        self.negative_head = nn.Linear(
            512,
            NUM_NEGATIVE
        )

        self.attribute_head = nn.Linear(
            512,
            NUM_ATTRIBUTES
        )

    def encode_images(
        self,
        image_list
    ):

        counts = [
            x.shape[0]
            for x in image_list
        ]

        flat_images = torch.cat(
            image_list,
            dim=0
        )

        vision_out = self.vision(
            pixel_values=flat_images
        )

        cls = vision_out.last_hidden_state[
            :,
            0,
            :
        ]

        projected = self.projector(
            cls
        )

        prefix_list = []

        start = 0

        for count in counts:

            prefix_list.append(
                projected[
                    start:start + count
                ]
            )

            start += count

        batch_size = len(
            image_list
        )

        max_count = max(
            counts
        )

        prefix = projected.new_zeros(
            (
                batch_size,
                max_count,
                512
            )
        )

        attention = torch.zeros(
            (
                batch_size,
                max_count
            ),
            dtype=torch.long,
            device=projected.device
        )

        pooled = []

        for i, item in enumerate(
            prefix_list
        ):

            n = item.shape[0]

            prefix[
                i,
                :n
            ] = item

            attention[
                i,
                :n
            ] = 1

            pooled.append(
                item.mean(
                    dim=0
                )
            )

        pooled = torch.stack(
            pooled
        )

        return (
            prefix,
            attention,
            pooled
        )


# ============================================================
# 10. LOAD CHECKPOINT
# ============================================================

print("\nLoading checkpoint...")

vision = ViTModel.from_pretrained(
    VISION_MODEL
)

mt5 = MT5ForConditionalGeneration.from_pretrained(
    TEXT_MODEL
)

model = V3MultiTaskModel(
    vision,
    mt5
)

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

best_epoch = (
    checkpoint["epoch"] + 1
)

best_val_loss = (
    checkpoint["best_val_loss"]
)

model = model.to(
    DEVICE
)

model.eval()

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best val total:",
    best_val_loss
)

del checkpoint
gc.collect()

torch.cuda.empty_cache()


# ============================================================
# 11. GENERATION + EVALUATION HELPERS
# ============================================================

def normalize_text(text):

    return (
        str(text)
        .strip()
        .upper()
    )


def char_similarity(
    pred,
    target
):

    pred = normalize_text(pred)
    target = normalize_text(target)

    if not target:
        return 0.0

    # Character-level LCS
    m = len(pred)
    n = len(target)

    prev = [0] * (n + 1)

    for i in range(1, m + 1):

        cur = [0] * (n + 1)

        for j in range(1, n + 1):

            if pred[i - 1] == target[j - 1]:

                cur[j] = prev[j - 1] + 1

            else:

                cur[j] = max(
                    prev[j],
                    cur[j - 1]
                )

        prev = cur

    lcs = prev[n]

    return (
        2.0 * lcs
        / (len(pred) + len(target))
        if pred or target
        else 1.0
    )


def token_f1(
    pred,
    target
):

    pred_tokens = normalize_text(
        pred
    ).split()

    target_tokens = normalize_text(
        target
    ).split()

    if not pred_tokens or not target_tokens:

        return float(
            pred_tokens == target_tokens
        )

    from collections import Counter

    p = Counter(pred_tokens)
    t = Counter(target_tokens)

    overlap = sum(
        (p & t).values()
    )

    if overlap == 0:

        return 0.0

    precision = (
        overlap
        / len(pred_tokens)
    )

    recall = (
        overlap
        / len(target_tokens)
    )

    return (
        2 * precision * recall
        / (precision + recall)
    )


# ============================================================
# 12. TEST LOOP
# ============================================================

all_rows = []

test_loss_sum = 0.0
test_report_loss_sum = 0.0
test_concept_loss_sum = 0.0
test_negative_loss_sum = 0.0
test_attribute_loss_sum = 0.0

num_cases = 0

print("\nRunning test evaluation...")


with torch.no_grad():

    for batch_idx, batch in enumerate(
        test_loader
    ):

        images = [
            x.to(
                DEVICE,
                non_blocking=True
            )
            for x in batch["images"]
        ]

        concept_targets = batch[
            "concept_targets"
        ].to(DEVICE)

        negative_targets = batch[
            "negative_targets"
        ].to(DEVICE)

        attribute_targets = batch[
            "attribute_targets"
        ].to(DEVICE)

        structured_mask = batch[
            "structured_mask"
        ].to(DEVICE)

        tokenized = tokenizer(
            batch["target_text"],
            padding=True,
            truncation=True,
            max_length=MAX_TARGET_LENGTH,
            return_tensors="pt"
        )

        labels = tokenized[
            "input_ids"
        ].to(DEVICE)

        labels[
            labels == tokenizer.pad_token_id
        ] = -100

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=USE_BF16
        ):

            (
                visual_prefix,
                visual_attention,
                pooled
            ) = model.encode_images(
                images
            )

            concept_logits = (
                model.concept_head(
                    pooled
                )
            )

            negative_logits = (
                model.negative_head(
                    pooled
                )
            )

            attribute_logits = (
                model.attribute_head(
                    pooled
                )
            )

            encoder_outputs = (
                model.mt5.encoder(
                    inputs_embeds=visual_prefix
                )
            )

            report_output = model.mt5(
                encoder_outputs=encoder_outputs,
                attention_mask=visual_attention,
                labels=labels,
                return_dict=True
            )

            report_loss = (
                report_output.loss
            )

            concept_loss_all = (
                F.binary_cross_entropy_with_logits(
                    concept_logits,
                    concept_targets,
                    reduction="none"
                ).mean(dim=1)
            )

            negative_loss_all = (
                F.binary_cross_entropy_with_logits(
                    negative_logits,
                    negative_targets,
                    reduction="none"
                ).mean(dim=1)
            )

            attribute_loss_all = (
                F.binary_cross_entropy_with_logits(
                    attribute_logits,
                    attribute_targets,
                    reduction="none"
                ).mean(dim=1)
            )

            mask_sum = (
                structured_mask.sum()
                .clamp_min(1.0)
            )

            concept_loss = (
                concept_loss_all
                * structured_mask
            ).sum() / mask_sum

            negative_loss = (
                negative_loss_all
                * structured_mask
            ).sum() / mask_sum

            attribute_loss = (
                attribute_loss_all
                * structured_mask
            ).sum() / mask_sum

            total_loss = (
                report_loss
                + 0.5 * concept_loss
                + 0.5 * negative_loss
                + 0.25 * attribute_loss
            )

        # ----------------------------------------------------
        # Teacher-forced losses
        # ----------------------------------------------------

        bs = len(
            batch["case_id"]
        )

        test_loss_sum += (
            total_loss.item()
            * bs
        )

        test_report_loss_sum += (
            report_loss.item()
            * bs
        )

        test_concept_loss_sum += (
            concept_loss.item()
            * bs
        )

        test_negative_loss_sum += (
            negative_loss.item()
            * bs
        )

        test_attribute_loss_sum += (
            attribute_loss.item()
            * bs
        )

        num_cases += bs

        # ----------------------------------------------------
        # Greedy generation
        # ----------------------------------------------------

        generated_ids = model.mt5.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=visual_attention,
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        predictions = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        # ----------------------------------------------------
        # Structured predictions
        # ----------------------------------------------------

        concept_probs = torch.sigmoid(
            concept_logits
        )

        negative_probs = torch.sigmoid(
            negative_logits
        )

        attribute_probs = torch.sigmoid(
            attribute_logits
        )

        concept_pred = (
            concept_probs >= 0.5
        ).cpu().numpy()

        negative_pred = (
            negative_probs >= 0.5
        ).cpu().numpy()

        attribute_pred = (
            attribute_probs >= 0.5
        ).cpu().numpy()

        concept_true = (
            concept_targets
            .cpu()
            .numpy()
            .astype(bool)
        )

        negative_true = (
            negative_targets
            .cpu()
            .numpy()
            .astype(bool)
        )

        attribute_true = (
            attribute_targets
            .cpu()
            .numpy()
            .astype(bool)
        )

        # ----------------------------------------------------
        # Save per-case
        # ----------------------------------------------------

        for i in range(bs):

            pred = predictions[i]
            target = batch[
                "target_text"
            ][i]

            concept_labels_pred = [
                CONCEPT_VOCAB[j]
                for j in range(NUM_CONCEPTS)
                if concept_pred[i, j]
            ]

            concept_labels_true = [
                CONCEPT_VOCAB[j]
                for j in range(NUM_CONCEPTS)
                if concept_true[i, j]
            ]

            negative_labels_pred = [
                NEGATIVE_VOCAB[j]
                for j in range(NUM_NEGATIVE)
                if negative_pred[i, j]
            ]

            negative_labels_true = [
                NEGATIVE_VOCAB[j]
                for j in range(NUM_NEGATIVE)
                if negative_true[i, j]
            ]

            attribute_labels_pred = [
                ATTRIBUTE_VOCAB[j]
                for j in range(NUM_ATTRIBUTES)
                if attribute_pred[i, j]
            ]

            attribute_labels_true = [
                ATTRIBUTE_VOCAB[j]
                for j in range(NUM_ATTRIBUTES)
                if attribute_true[i, j]
            ]

            all_rows.append({

                "case_id":
                    batch["case_id"][i],

                "patient_group_id":
                    batch[
                        "patient_group_id"
                    ][i],

                "target":
                    target,

                "prediction":
                    pred,

                "exact_match":
                    int(
                        normalize_text(pred)
                        ==
                        normalize_text(target)
                    ),

                "char_similarity":
                    char_similarity(
                        pred,
                        target
                    ),

                "token_f1":
                    token_f1(
                        pred,
                        target
                    ),

                "concept_true":
                    "|".join(
                        concept_labels_true
                    ),

                "concept_pred":
                    "|".join(
                        concept_labels_pred
                    ),

                "negative_true":
                    "|".join(
                        negative_labels_true
                    ),

                "negative_pred":
                    "|".join(
                        negative_labels_pred
                    ),

                "attribute_true":
                    "|".join(
                        attribute_labels_true
                    ),

                "attribute_pred":
                    "|".join(
                        attribute_labels_pred
                    ),

                "structured_mask":
                    float(
                        structured_mask[i].item()
                    ),
            })

        if (
            (batch_idx + 1) % 25 == 0
            or batch_idx == 0
        ):

            print(
                f"Batch "
                f"{batch_idx+1}/"
                f"{len(test_loader)}"
            )


# ============================================================
# 13. SAVE PREDICTIONS
# ============================================================

results = pd.DataFrame(
    all_rows
)

PREDICTIONS_FILE = os.path.join(
    OUTPUT_DIR,
    "test_predictions_v3.csv"
)

results.to_csv(
    PREDICTIONS_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 14. REPORT METRICS
# ============================================================

exact_match = (
    results["exact_match"]
    .mean()
)

mean_char_similarity = (
    results["char_similarity"]
    .mean()
)

median_char_similarity = (
    results["char_similarity"]
    .median()
)

mean_token_f1 = (
    results["token_f1"]
    .mean()
)

median_token_f1 = (
    results["token_f1"]
    .median()
)

unique_predictions = (
    results["prediction"]
    .nunique()
)

diversity_ratio = (
    unique_predictions
    / len(results)
)

mean_test_total = (
    test_loss_sum
    / num_cases
)

mean_test_report = (
    test_report_loss_sum
    / num_cases
)

mean_test_concept = (
    test_concept_loss_sum
    / num_cases
)

mean_test_negative = (
    test_negative_loss_sum
    / num_cases
)

mean_test_attribute = (
    test_attribute_loss_sum
    / num_cases
)


# ============================================================
# 15. STRUCTURED METRICS
# ============================================================

def multilabel_metrics(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true
    ).astype(int)

    y_pred = np.asarray(
        y_pred
    ).astype(int)

    tp = (
        (y_true == 1)
        &
        (y_pred == 1)
    ).sum()

    fp = (
        (y_true == 0)
        &
        (y_pred == 1)
    ).sum()

    fn = (
        (y_true == 1)
        &
        (y_pred == 0)
    ).sum()

    micro_precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )

    micro_recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0.0
    )

    micro_f1 = (
        2
        * micro_precision
        * micro_recall
        / (
            micro_precision
            + micro_recall
        )
        if (
            micro_precision
            + micro_recall
        ) > 0
        else 0.0
    )

    # Per-label F1
    f1s = []

    for j in range(
        y_true.shape[1]
    ):

        yt = y_true[:, j]
        yp = y_pred[:, j]

        tp_j = (
            (yt == 1)
            &
            (yp == 1)
        ).sum()

        fp_j = (
            (yt == 0)
            &
            (yp == 1)
        ).sum()

        fn_j = (
            (yt == 1)
            &
            (yp == 0)
        ).sum()

        if (
            tp_j + fp_j + fn_j
            == 0
        ):

            continue

        precision_j = (
            tp_j / (tp_j + fp_j)
            if tp_j + fp_j > 0
            else 0
        )

        recall_j = (
            tp_j / (tp_j + fn_j)
            if tp_j + fn_j > 0
            else 0
        )

        f1_j = (
            2 * precision_j * recall_j
            / (precision_j + recall_j)
            if precision_j + recall_j > 0
            else 0
        )

        f1s.append(
            f1_j
        )

    macro_f1 = (
        float(np.mean(f1s))
        if f1s
        else 0.0
    )

    return {
        "micro_precision":
            float(micro_precision),

        "micro_recall":
            float(micro_recall),

        "micro_f1":
            float(micro_f1),

        "macro_f1":
            float(macro_f1),
    }


# Reconstruct arrays from CSV labels
def labels_to_matrix(
    series,
    vocab
):

    index = {
        label: i
        for i, label in enumerate(
            vocab
        )
    }

    matrix = np.zeros(
        (
            len(series),
            len(vocab)
        ),
        dtype=int
    )

    for i, value in enumerate(
        series
    ):

        if not value:
            continue

        for label in str(
            value
        ).split("|"):

            if label in index:

                matrix[
                    i,
                    index[label]
                ] = 1

    return matrix


concept_true = labels_to_matrix(
    results["concept_true"],
    CONCEPT_VOCAB
)

concept_pred = labels_to_matrix(
    results["concept_pred"],
    CONCEPT_VOCAB
)

negative_true = labels_to_matrix(
    results["negative_true"],
    NEGATIVE_VOCAB
)

negative_pred = labels_to_matrix(
    results["negative_pred"],
    NEGATIVE_VOCAB
)

attribute_true = labels_to_matrix(
    results["attribute_true"],
    ATTRIBUTE_VOCAB
)

attribute_pred = labels_to_matrix(
    results["attribute_pred"],
    ATTRIBUTE_VOCAB
)


concept_metrics = multilabel_metrics(
    concept_true,
    concept_pred
)

negative_metrics = multilabel_metrics(
    negative_true,
    negative_pred
)

attribute_metrics = multilabel_metrics(
    attribute_true,
    attribute_pred
)


# ============================================================
# 16. PREDICTION FREQUENCY
# ============================================================

prediction_frequency = (
    results[
        "prediction"
    ]
    .value_counts()
    .reset_index()
)

prediction_frequency.columns = [
    "prediction",
    "count"
]

prediction_frequency[
    "percentage"
] = (
    prediction_frequency["count"]
    / len(results)
    * 100
)

prediction_frequency.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "prediction_frequency_v3.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 17. METRICS SUMMARY
# ============================================================

metrics = {

    "num_test_cases":
        len(results),

    "best_epoch":
        best_epoch,

    "best_val_total":
        best_val_loss,

    "test_total_loss":
        mean_test_total,

    "test_report_loss":
        mean_test_report,

    "test_concept_loss":
        mean_test_concept,

    "test_negative_loss":
        mean_test_negative,

    "test_attribute_loss":
        mean_test_attribute,

    "exact_match":
        exact_match,

    "mean_char_similarity":
        mean_char_similarity,

    "median_char_similarity":
        median_char_similarity,

    "mean_token_f1":
        mean_token_f1,

    "median_token_f1":
        median_token_f1,

    "unique_predictions":
        unique_predictions,

    "diversity_ratio":
        diversity_ratio,

    "concept_micro_f1":
        concept_metrics[
            "micro_f1"
        ],

    "concept_macro_f1":
        concept_metrics[
            "macro_f1"
        ],

    "negative_micro_f1":
        negative_metrics[
            "micro_f1"
        ],

    "negative_macro_f1":
        negative_metrics[
            "macro_f1"
        ],

    "attribute_micro_f1":
        attribute_metrics[
            "micro_f1"
        ],

    "attribute_macro_f1":
        attribute_metrics[
            "macro_f1"
        ],
}

metrics_df = pd.DataFrame([
    metrics
])

METRICS_FILE = os.path.join(
    OUTPUT_DIR,
    "generation_and_structured_metrics_v3.csv"
)

metrics_df.to_csv(
    METRICS_FILE,
    index=False
)


# ============================================================
# 18. PRINT
# ============================================================

print("\n" + "=" * 72)
print("V3 TEST RESULTS")
print("=" * 72)

print(
    f"\nTest report loss: "
    f"{mean_test_report:.6f}"
)

print(
    f"Exact match: "
    f"{exact_match*100:.4f}%"
)

print(
    f"Mean char similarity: "
    f"{mean_char_similarity:.4f}"
)

print(
    f"Median char similarity: "
    f"{median_char_similarity:.4f}"
)

print(
    f"Mean token F1: "
    f"{mean_token_f1:.4f}"
)

print(
    f"Median token F1: "
    f"{median_token_f1:.4f}"
)

print(
    f"Unique predictions: "
    f"{unique_predictions}"
)

print(
    f"Diversity ratio: "
    f"{diversity_ratio:.4f}"
)

print("\nStructured metrics:")

print(
    f"Concept    micro F1: "
    f"{concept_metrics['micro_f1']:.4f} | "
    f"macro F1: "
    f"{concept_metrics['macro_f1']:.4f}"
)

print(
    f"Negative   micro F1: "
    f"{negative_metrics['micro_f1']:.4f} | "
    f"macro F1: "
    f"{negative_metrics['macro_f1']:.4f}"
)

print(
    f"Attribute  micro F1: "
    f"{attribute_metrics['micro_f1']:.4f} | "
    f"macro F1: "
    f"{attribute_metrics['macro_f1']:.4f}"
)

print("\nTop predictions:")

print(
    prediction_frequency
    .head(10)
    .to_string(index=False)
)

print("\nSample predictions:")

print(
    results[
        [
            "case_id",
            "target",
            "prediction",
            "token_f1"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

print("\nSaved:")

print(
    PREDICTIONS_FILE
)

print(
    METRICS_FILE
)

print("\n🔴 A100 TEST PASS")

V3-4 — TEST GENERATION + STRUCTURED EVALUATION
GPU: NVIDIA A100-SXM4-40GB
BF16: True

Cases: 7606
NORMAL images: 76209


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Best epoch: 5
Best val total: 0.7656870074373073

Running test evaluation...
Batch 1/178
Batch 25/178
Batch 50/178
Batch 75/178
Batch 100/178
Batch 125/178
Batch 150/178
Batch 175/178

V3 TEST RESULTS

Test report loss: 0.541319
Exact match: 16.1517%
Mean char similarity: 0.5745
Median char similarity: 0.5401
Mean token F1: 0.5465
Median token F1: 0.5000
Unique predictions: 5
Diversity ratio: 0.0070

Structured metrics:
Concept    micro F1: 0.4847 | macro F1: 0.0999
Negative   micro F1: 0.0000 | macro F1: 0.0000
Attribute  micro F1: 0.2309 | macro F1: 0.1370

Top predictions:
                                  prediction  count  percentage
                                    VIÊM MŨI    580   81.460674
VIÊM HỌNG MẠN - THEO DÕI TRÀO NGƯỢC DỊCH VỊ.     80   11.235955
         VIÊM ỐNG TAI NGOÀI + MÀNG NHĨ T CẤP     49    6.882022
             VIÊM ỐNG TAI NGOÀI + MÀNG NHĨ T      2    0.280899
         VIÊM ỐNG TAI NGOÀI + MÀNG NHĨ P CẤP      1    0.140449

Sample predictions:
            

In [3]:
# ============================================================
# V3-5 — STRUCTURED HEAD DIAGNOSTIC
# 🔴 A100
#
# NO TRAINING
#
# Goals:
#   1. Collect raw probabilities for 48 concepts
#      + 5 negative findings
#      + 6 attributes
#   2. Threshold sweep
#   3. Micro / macro F1 by threshold
#   4. Per-label F1
#   5. Positive prediction rate
#   6. Positive prevalence
#
# This determines whether the poor structured F1 is caused by
# thresholding / class imbalance or by weak learned signal.
# ============================================================

import os
import gc
import json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from transformers import (
    AutoTokenizer,
    ViTModel,
    MT5ForConditionalGeneration,
)

# ============================================================
# 0. DEVICE
# ============================================================

gc.collect()
torch.cuda.empty_cache()

DEVICE = torch.device("cuda")

USE_BF16 = torch.cuda.is_bf16_supported()

print("=" * 72)
print("V3-5 — STRUCTURED HEAD DIAGNOSTIC")
print("=" * 72)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "BF16:",
    USE_BF16
)


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = (
    "/content/drive/MyDrive/NoiSoi_Matching"
)

V3_DIR = os.path.join(
    PROJECT_DIR,
    "baseline_model_v3"
)

MANIFEST_FILE = os.path.join(
    V3_DIR,
    "v3_training_manifest.csv"
)

CONFIG_FILE = os.path.join(
    V3_DIR,
    "v3_config.json"
)

IMAGE_MANIFEST = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

BEST_CHECKPOINT = os.path.join(
    V3_DIR,
    "checkpoints",
    "best.pt"
)

OUTPUT_DIR = os.path.join(
    V3_DIR,
    "structured_diagnostic"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

assert os.path.exists(
    BEST_CHECKPOINT
)


# ============================================================
# 2. LOAD DATA
# ============================================================

with open(
    CONFIG_FILE,
    "r",
    encoding="utf-8"
) as f:

    config = json.load(f)

df = pd.read_csv(
    MANIFEST_FILE,
    low_memory=False
)

image_df = pd.read_csv(
    IMAGE_MANIFEST,
    low_memory=False
)

normal_images = image_df[
    image_df["image_status"] == "NORMAL"
].copy()

normal_images = normal_images[
    normal_images["case_id"].isin(
        set(df["case_id"])
    )
].copy()

case_to_images = (
    normal_images
    .groupby("case_id")["image_path"]
    .apply(list)
    .to_dict()
)


# ============================================================
# 3. ONTOLOGY
# ============================================================

CONCEPT_VOCAB = config[
    "concept_vocab"
]

NEGATIVE_VOCAB = config[
    "negative_vocab"
]

ATTRIBUTE_VOCAB = config[
    "attribute_vocab"
]

NUM_CONCEPTS = len(
    CONCEPT_VOCAB
)

NUM_NEGATIVE = len(
    NEGATIVE_VOCAB
)

NUM_ATTRIBUTES = len(
    ATTRIBUTE_VOCAB
)

assert NUM_CONCEPTS == 48
assert NUM_NEGATIVE == 5
assert NUM_ATTRIBUTES == 6


# ============================================================
# 4. MODEL CONFIG
# ============================================================

VISION_MODEL = (
    "google/vit-base-patch16-224"
)

TEXT_MODEL = (
    "google/mt5-small"
)

IMAGE_SIZE = 224
MAX_IMAGES = 8
BATCH_SIZE = 4


# ============================================================
# 5. TRANSFORM
# ============================================================

image_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    ),
])


# ============================================================
# 6. VECTOR PARSER
# ============================================================

def parse_vector(
    value,
    expected_length
):

    values = [
        float(x)
        for x in str(value).split(",")
        if str(x).strip()
    ]

    assert len(values) == expected_length

    return torch.tensor(
        values,
        dtype=torch.float32
    )


# ============================================================
# 7. DATASET
# ============================================================

class StructuredTestDataset(
    Dataset
):

    def __init__(
        self,
        frame
    ):

        self.df = frame.reset_index(
            drop=True
        )

    def __len__(self):

        return len(self.df)

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        case_id = row["case_id"]

        paths = case_to_images[
            case_id
        ]

        selected = paths[
            :MAX_IMAGES
        ]

        images = []

        for path in selected:

            image = Image.open(
                path
            ).convert("RGB")

            image = image_transform(
                image
            )

            images.append(
                image
            )

        return {

            "case_id":
                case_id,

            "images":
                torch.stack(
                    images
                ),

            "concept_targets":
                parse_vector(
                    row[
                        "concept_vector_str"
                    ],
                    NUM_CONCEPTS
                ),

            "negative_targets":
                parse_vector(
                    row[
                        "negative_vector_str"
                    ],
                    NUM_NEGATIVE
                ),

            "attribute_targets":
                parse_vector(
                    row[
                        "attribute_vector_str"
                    ],
                    NUM_ATTRIBUTES
                ),

            "structured_mask":
                float(
                    row[
                        "structured_loss_mask"
                    ]
                ),
        }


def collate_fn(batch):

    return {

        "case_id": [
            x["case_id"]
            for x in batch
        ],

        "images": [
            x["images"]
            for x in batch
        ],

        "concept_targets": torch.stack([
            x["concept_targets"]
            for x in batch
        ]),

        "negative_targets": torch.stack([
            x["negative_targets"]
            for x in batch
        ]),

        "attribute_targets": torch.stack([
            x["attribute_targets"]
            for x in batch
        ]),

        "structured_mask": torch.tensor(
            [
                x["structured_mask"]
                for x in batch
            ],
            dtype=torch.float32
        ),
    }


test_df = df[
    df["split"] == "test"
].copy()

test_dataset = StructuredTestDataset(
    test_df
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn
)

print(
    "\nTest cases:",
    len(test_dataset)
)


# ============================================================
# 8. MODEL
# ============================================================

class VisualProjector(nn.Module):

    def __init__(
        self,
        input_dim,
        output_dim
    ):

        super().__init__()

        self.proj = nn.Linear(
            input_dim,
            output_dim
        )

    def forward(self, x):

        return self.proj(x)


class V3MultiTaskModel(nn.Module):

    def __init__(
        self,
        vision,
        mt5
    ):

        super().__init__()

        self.vision = vision
        self.mt5 = mt5

        self.projector = VisualProjector(
            768,
            512
        )

        self.concept_head = nn.Linear(
            512,
            NUM_CONCEPTS
        )

        self.negative_head = nn.Linear(
            512,
            NUM_NEGATIVE
        )

        self.attribute_head = nn.Linear(
            512,
            NUM_ATTRIBUTES
        )

    def encode_images(
        self,
        image_list
    ):

        counts = [
            x.shape[0]
            for x in image_list
        ]

        flat_images = torch.cat(
            image_list,
            dim=0
        )

        vision_out = self.vision(
            pixel_values=flat_images
        )

        cls = vision_out.last_hidden_state[
            :,
            0,
            :
        ]

        projected = self.projector(
            cls
        )

        prefix_list = []

        start = 0

        for count in counts:

            prefix_list.append(
                projected[
                    start:start + count
                ]
            )

            start += count

        batch_size = len(
            image_list
        )

        max_count = max(
            counts
        )

        prefix = projected.new_zeros(
            (
                batch_size,
                max_count,
                512
            )
        )

        attention = torch.zeros(
            (
                batch_size,
                max_count
            ),
            dtype=torch.long,
            device=projected.device
        )

        pooled = []

        for i, item in enumerate(
            prefix_list
        ):

            n = item.shape[0]

            prefix[
                i,
                :n
            ] = item

            attention[
                i,
                :n
            ] = 1

            pooled.append(
                item.mean(
                    dim=0
                )
            )

        pooled = torch.stack(
            pooled
        )

        return pooled


# ============================================================
# 9. LOAD CHECKPOINT
# ============================================================

print(
    "\nLoading V3 best checkpoint..."
)

vision = ViTModel.from_pretrained(
    VISION_MODEL
)

mt5 = MT5ForConditionalGeneration.from_pretrained(
    TEXT_MODEL
)

model = V3MultiTaskModel(
    vision,
    mt5
)

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

model = model.to(
    DEVICE
)

model.eval()

print(
    "Best epoch:",
    checkpoint["epoch"] + 1
)

print(
    "Best val total:",
    checkpoint[
        "best_val_loss"
    ]
)

del checkpoint

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# 10. COLLECT RAW PROBABILITIES
# ============================================================

all_concept_true = []
all_concept_prob = []

all_negative_true = []
all_negative_prob = []

all_attribute_true = []
all_attribute_prob = []

all_case_ids = []

print(
    "\nCollecting structured probabilities..."
)


with torch.no_grad():

    for batch_idx, batch in enumerate(
        test_loader
    ):

        images = [
            x.to(
                DEVICE,
                non_blocking=True
            )
            for x in batch["images"]
        ]

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=USE_BF16
        ):

            pooled = model.encode_images(
                images
            )

            concept_logits = (
                model.concept_head(
                    pooled
                )
            )

            negative_logits = (
                model.negative_head(
                    pooled
                )
            )

            attribute_logits = (
                model.attribute_head(
                    pooled
                )
            )

        all_case_ids.extend(
            batch["case_id"]
        )

        all_concept_true.append(
            batch[
                "concept_targets"
            ].numpy()
        )

        all_concept_prob.append(
            torch.sigmoid(
                concept_logits
            )
            .float()
            .cpu()
            .numpy()
        )

        all_negative_true.append(
            batch[
                "negative_targets"
            ].numpy()
        )

        all_negative_prob.append(
            torch.sigmoid(
                negative_logits
            )
            .float()
            .cpu()
            .numpy()
        )

        all_attribute_true.append(
            batch[
                "attribute_targets"
            ].numpy()
        )

        all_attribute_prob.append(
            torch.sigmoid(
                attribute_logits
            )
            .float()
            .cpu()
            .numpy()
        )

        if (
            (batch_idx + 1) % 25 == 0
            or batch_idx == 0
        ):

            print(
                f"Batch "
                f"{batch_idx+1}/"
                f"{len(test_loader)}"
            )


concept_true = np.concatenate(
    all_concept_true,
    axis=0
).astype(int)

concept_prob = np.concatenate(
    all_concept_prob,
    axis=0
)

negative_true = np.concatenate(
    all_negative_true,
    axis=0
).astype(int)

negative_prob = np.concatenate(
    all_negative_prob,
    axis=0
)

attribute_true = np.concatenate(
    all_attribute_true,
    axis=0
).astype(int)

attribute_prob = np.concatenate(
    all_attribute_prob,
    axis=0
)


# ============================================================
# 11. METRICS
# ============================================================

def multilabel_metrics(
    y_true,
    y_prob,
    threshold
):

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    tp = (
        (y_true == 1)
        &
        (y_pred == 1)
    ).sum()

    fp = (
        (y_true == 0)
        &
        (y_pred == 1)
    ).sum()

    fn = (
        (y_true == 1)
        &
        (y_pred == 0)
    ).sum()

    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    per_label_f1 = []

    for j in range(
        y_true.shape[1]
    ):

        yt = y_true[:, j]
        yp = y_pred[:, j]

        tp_j = (
            (yt == 1)
            &
            (yp == 1)
        ).sum()

        fp_j = (
            (yt == 0)
            &
            (yp == 1)
        ).sum()

        fn_j = (
            (yt == 1)
            &
            (yp == 0)
        ).sum()

        if (
            tp_j + fp_j + fn_j
            == 0
        ):

            continue

        p_j = (
            tp_j / (tp_j + fp_j)
            if tp_j + fp_j > 0
            else 0.0
        )

        r_j = (
            tp_j / (tp_j + fn_j)
            if tp_j + fn_j > 0
            else 0.0
        )

        f1_j = (
            2 * p_j * r_j
            / (p_j + r_j)
            if p_j + r_j > 0
            else 0.0
        )

        per_label_f1.append(
            f1_j
        )

    macro_f1 = (
        float(
            np.mean(
                per_label_f1
            )
        )
        if per_label_f1
        else 0.0
    )

    return {
        "micro_precision":
            float(precision),

        "micro_recall":
            float(recall),

        "micro_f1":
            float(f1),

        "macro_f1":
            float(macro_f1),
    }


# ============================================================
# 12. THRESHOLD SWEEP
# ============================================================

THRESHOLDS = [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.95,
]

sweep_rows = []

for threshold in THRESHOLDS:

    for name, y_true, y_prob in [

        (
            "concept",
            concept_true,
            concept_prob
        ),

        (
            "negative",
            negative_true,
            negative_prob
        ),

        (
            "attribute",
            attribute_true,
            attribute_prob
        ),
    ]:

        metrics = multilabel_metrics(
            y_true,
            y_prob,
            threshold
        )

        y_pred = (
            y_prob >= threshold
        ).astype(int)

        sweep_rows.append({

            "head":
                name,

            "threshold":
                threshold,

            "micro_precision":
                metrics[
                    "micro_precision"
                ],

            "micro_recall":
                metrics[
                    "micro_recall"
                ],

            "micro_f1":
                metrics[
                    "micro_f1"
                ],

            "macro_f1":
                metrics[
                    "macro_f1"
                ],

            "positive_prevalence":
                y_true.mean(),

            "prediction_positive_rate":
                y_pred.mean(),
        })


sweep_df = pd.DataFrame(
    sweep_rows
)

SWEEP_FILE = os.path.join(
    OUTPUT_DIR,
    "threshold_sweep.csv"
)

sweep_df.to_csv(
    SWEEP_FILE,
    index=False
)


# ============================================================
# 13. BEST THRESHOLD BY MICRO F1
# ============================================================

best_threshold_rows = []

for head in [
    "concept",
    "negative",
    "attribute"
]:

    sub = sweep_df[
        sweep_df["head"] == head
    ]

    best = sub.loc[
        sub["micro_f1"].idxmax()
    ]

    best_threshold_rows.append(
        best
    )

best_threshold_df = pd.DataFrame(
    best_threshold_rows
)

BEST_THRESHOLD_FILE = os.path.join(
    OUTPUT_DIR,
    "best_thresholds.csv"
)

best_threshold_df.to_csv(
    BEST_THRESHOLD_FILE,
    index=False
)


# ============================================================
# 14. PER-LABEL ANALYSIS
# ============================================================

def per_label_table(
    y_true,
    y_prob,
    vocab,
    threshold
):

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    rows = []

    for j, label in enumerate(
        vocab
    ):

        yt = y_true[:, j]
        yp = y_pred[:, j]

        tp = (
            (yt == 1)
            &
            (yp == 1)
        ).sum()

        fp = (
            (yt == 0)
            &
            (yp == 1)
        ).sum()

        fn = (
            (yt == 1)
            &
            (yp == 0)
        ).sum()

        precision = (
            tp / (tp + fp)
            if tp + fp > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if tp + fn > 0
            else 0.0
        )

        f1 = (
            2 * precision * recall
            / (precision + recall)
            if precision + recall > 0
            else 0.0
        )

        rows.append({

            "label":
                label,

            "positive_cases":
                int(
                    yt.sum()
                ),

            "positive_rate":
                float(
                    yt.mean()
                ),

            "mean_probability":
                float(
                    y_prob[:, j].mean()
                ),

            "median_probability":
                float(
                    np.median(
                        y_prob[:, j]
                    )
                ),

            "prediction_positive_rate":
                float(
                    yp.mean()
                ),

            "tp":
                int(tp),

            "fp":
                int(fp),

            "fn":
                int(fn),

            "precision":
                float(precision),

            "recall":
                float(recall),

            "f1":
                float(f1),

        })

    return pd.DataFrame(
        rows
    ).sort_values(
        "f1",
        ascending=False
    )


# Use threshold 0.5 for baseline
concept_label_df = per_label_table(
    concept_true,
    concept_prob,
    CONCEPT_VOCAB,
    0.50
)

negative_label_df = per_label_table(
    negative_true,
    negative_prob,
    NEGATIVE_VOCAB,
    0.50
)

attribute_label_df = per_label_table(
    attribute_true,
    attribute_prob,
    ATTRIBUTE_VOCAB,
    0.50
)

concept_label_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "concept_per_label_threshold_050.csv"
    ),
    index=False
)

negative_label_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "negative_per_label_threshold_050.csv"
    ),
    index=False
)

attribute_label_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "attribute_per_label_threshold_050.csv"
    ),
    index=False
)


# ============================================================
# 15. BEST-THRESHOLD PER-LABEL TABLES
# ============================================================

def best_threshold_per_label(
    y_true,
    y_prob,
    vocab
):

    rows = []

    for j, label in enumerate(
        vocab
    ):

        best_f1 = -1
        best_threshold = None
        best_precision = 0
        best_recall = 0

        for threshold in THRESHOLDS:

            yt = y_true[:, j]

            yp = (
                y_prob[:, j]
                >= threshold
            ).astype(int)

            tp = (
                (yt == 1)
                &
                (yp == 1)
            ).sum()

            fp = (
                (yt == 0)
                &
                (yp == 1)
            ).sum()

            fn = (
                (yt == 1)
                &
                (yp == 0)
            ).sum()

            precision = (
                tp / (tp + fp)
                if tp + fp > 0
                else 0
            )

            recall = (
                tp / (tp + fn)
                if tp + fn > 0
                else 0
            )

            f1 = (
                2 * precision * recall
                / (precision + recall)
                if precision + recall > 0
                else 0
            )

            if f1 > best_f1:

                best_f1 = f1
                best_threshold = threshold
                best_precision = precision
                best_recall = recall

        rows.append({

            "label":
                label,

            "positive_cases":
                int(
                    y_true[:, j].sum()
                ),

            "best_threshold":
                best_threshold,

            "best_f1":
                best_f1,

            "best_precision":
                best_precision,

            "best_recall":
                best_recall,

            "mean_probability":
                float(
                    y_prob[:, j].mean()
                ),

            "median_probability":
                float(
                    np.median(
                        y_prob[:, j]
                    )
                ),
        })

    return pd.DataFrame(
        rows
    ).sort_values(
        "best_f1",
        ascending=False
    )


concept_best_label = best_threshold_per_label(
    concept_true,
    concept_prob,
    CONCEPT_VOCAB
)

negative_best_label = best_threshold_per_label(
    negative_true,
    negative_prob,
    NEGATIVE_VOCAB
)

attribute_best_label = best_threshold_per_label(
    attribute_true,
    attribute_prob,
    ATTRIBUTE_VOCAB
)

concept_best_label.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "concept_best_threshold_per_label.csv"
    ),
    index=False
)

negative_best_label.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "negative_best_threshold_per_label.csv"
    ),
    index=False
)

attribute_best_label.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "attribute_best_threshold_per_label.csv"
    ),
    index=False
)


# ============================================================
# 16. PROBABILITY SEPARATION
# ============================================================

def probability_separation(
    y_true,
    y_prob,
    vocab
):

    rows = []

    for j, label in enumerate(
        vocab
    ):

        positive_probs = y_prob[
            y_true[:, j] == 1,
            j
        ]

        negative_probs = y_prob[
            y_true[:, j] == 0,
            j
        ]

        rows.append({

            "label":
                label,

            "positive_cases":
                len(positive_probs),

            "positive_mean":
                float(
                    positive_probs.mean()
                )
                if len(positive_probs)
                else 0.0,

            "positive_median":
                float(
                    np.median(
                        positive_probs
                    )
                )
                if len(positive_probs)
                else 0.0,

            "negative_mean":
                float(
                    negative_probs.mean()
                )
                if len(negative_probs)
                else 0.0,

            "negative_median":
                float(
                    np.median(
                        negative_probs
                    )
                )
                if len(negative_probs)
                else 0.0,

            "mean_gap":
                float(
                    positive_probs.mean()
                    -
                    negative_probs.mean()
                )
                if (
                    len(positive_probs)
                    and len(negative_probs)
                )
                else 0.0,
        })

    return pd.DataFrame(
        rows
    ).sort_values(
        "mean_gap",
        ascending=False
    )


concept_separation = probability_separation(
    concept_true,
    concept_prob,
    CONCEPT_VOCAB
)

negative_separation = probability_separation(
    negative_true,
    negative_prob,
    NEGATIVE_VOCAB
)

attribute_separation = probability_separation(
    attribute_true,
    attribute_prob,
    ATTRIBUTE_VOCAB
)

concept_separation.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "concept_probability_separation.csv"
    ),
    index=False
)

negative_separation.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "negative_probability_separation.csv"
    ),
    index=False
)

attribute_separation.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "attribute_probability_separation.csv"
    ),
    index=False
)


# ============================================================
# 17. PRINT SUMMARY
# ============================================================

print("\n" + "=" * 72)
print("STRUCTURED DIAGNOSTIC RESULTS")
print("=" * 72)

print("\nBest thresholds:")

print(
    best_threshold_df[
        [
            "head",
            "threshold",
            "micro_f1",
            "macro_f1",
            "prediction_positive_rate"
        ]
    ].to_string(
        index=False
    )
)


print("\n" + "-" * 72)
print("CONCEPT — TOP LABELS BY BEST F1")
print("-" * 72)

print(
    concept_best_label
    .head(15)
    .to_string(
        index=False
    )
)


print("\n" + "-" * 72)
print("NEGATIVE FINDINGS — BEST F1")
print("-" * 72)

print(
    negative_best_label
    .to_string(
        index=False
    )
)


print("\n" + "-" * 72)
print("ATTRIBUTES — BEST F1")
print("-" * 72)

print(
    attribute_best_label
    .to_string(
        index=False
    )
)


print("\n" + "-" * 72)
print("CONCEPT PROBABILITY SEPARATION")
print("-" * 72)

print(
    concept_separation
    .head(15)
    .to_string(
        index=False
    )
)


print("\n" + "-" * 72)
print("NEGATIVE PROBABILITY SEPARATION")
print("-" * 72)

print(
    negative_separation
    .to_string(
        index=False
    )
)


print("\n" + "-" * 72)
print("ATTRIBUTE PROBABILITY SEPARATION")
print("-" * 72)

print(
    attribute_separation
    .to_string(
        index=False
    )
)


# ============================================================
# 18. FINAL SAVE SUMMARY
# ============================================================

summary = {

    "test_cases":
        int(len(test_df)),

    "concepts":
        NUM_CONCEPTS,

    "negative_findings":
        NUM_NEGATIVE,

    "attributes":
        NUM_ATTRIBUTES,

    "thresholds_tested":
        THRESHOLDS,

    "concept_best_threshold":
        float(
            best_threshold_df.loc[
                best_threshold_df["head"]
                == "concept",
                "threshold"
            ].iloc[0]
        ),

    "negative_best_threshold":
        float(
            best_threshold_df.loc[
                best_threshold_df["head"]
                == "negative",
                "threshold"
            ].iloc[0]
        ),

    "attribute_best_threshold":
        float(
            best_threshold_df.loc[
                best_threshold_df["head"]
                == "attribute",
                "threshold"
            ].iloc[0]
        ),
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "structured_diagnostic_summary.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2
    )


print("\n" + "=" * 72)
print("V3-5 DIAGNOSTIC COMPLETE")
print("=" * 72)

print(
    "\nSaved to:",
    OUTPUT_DIR
)

print("\n🔴 A100 PASS")

V3-5 — STRUCTURED HEAD DIAGNOSTIC
GPU: NVIDIA A100-SXM4-40GB
BF16: True

Test cases: 712

Loading V3 best checkpoint...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Best epoch: 5
Best val total: 0.7656870074373073

Batch 1/178
Batch 25/178
Batch 50/178
Batch 75/178
Batch 100/178
Batch 125/178
Batch 150/178
Batch 175/178

STRUCTURED DIAGNOSTIC RESULTS

Best thresholds:
     head  threshold  micro_f1  macro_f1  prediction_positive_rate
  concept       0.25  0.577378  0.140560                  0.033386
 negative       0.10  0.208178  0.058577                  0.052809
attribute       0.25  0.488693  0.346657                  0.248596

------------------------------------------------------------------------
CONCEPT — TOP LABELS BY BEST F1
------------------------------------------------------------------------
              label  positive_cases  best_threshold  best_f1  best_precision  best_recall  mean_probability  median_probability
theo_doi_trao_nguoc              65            0.25 0.902778        0.822785     1.000000          0.079975            0.002220
          viem_hong              74            0.35 0.888889        0.860759     0.918919  

In [6]:
# ============================================================
# V3-6 FIXED — VALIDATION THRESHOLD CALIBRATION
# Uses VALIDATION ONLY
# No training
# No TEST access
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from transformers import ViTModel, MT5ForConditionalGeneration
from PIL import Image
from torchvision import transforms

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/NoiSoi_Matching"
V3_DIR = os.path.join(PROJECT_DIR, "baseline_model_v3")

TRAIN_MANIFEST = os.path.join(
    V3_DIR,
    "v3_training_manifest.csv"
)

STRUCTURED_TARGET_FILE = os.path.join(
    PROJECT_DIR,
    "v3_structured_targets_v2_2.csv"
)

print("Structured target file:")
print(STRUCTURED_TARGET_FILE)
print("Exists:", os.path.exists(STRUCTURED_TARGET_FILE))

if not os.path.exists(STRUCTURED_TARGET_FILE):
    raise FileNotFoundError(STRUCTURED_TARGET_FILE)

IMAGE_MANIFEST = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

CONFIG_FILE = os.path.join(
    V3_DIR,
    "v3_config.json"
)

CHECKPOINT = os.path.join(
    V3_DIR,
    "checkpoints",
    "best.pt"
)

OUT_DIR = os.path.join(
    V3_DIR,
    "structured_diagnostic"
)

os.makedirs(
    OUT_DIR,
    exist_ok=True
)

THRESHOLD_CSV = os.path.join(
    OUT_DIR,
    "validation_best_thresholds.csv"
)

THRESHOLD_JSON = os.path.join(
    OUT_DIR,
    "validation_best_thresholds.json"
)

# ============================================================
# 2. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("V3-6 — VALIDATION THRESHOLD CALIBRATION")
print("=" * 70)

print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        f"VRAM: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB"
    )

# ============================================================
# 3. LOAD CONFIG
# ============================================================

with open(
    CONFIG_FILE,
    "r",
    encoding="utf-8"
) as f:

    config = json.load(f)

concept_names = config["concept_vocab"]
negative_names = config["negative_vocab"]
attribute_names = config["attribute_vocab"]

N_CONCEPTS = len(concept_names)
N_NEGATIVES = len(negative_names)
N_ATTRIBUTES = len(attribute_names)

print("\nOntology:")
print("  concepts :", N_CONCEPTS)
print("  negatives:", N_NEGATIVES)
print("  attrs    :", N_ATTRIBUTES)

# ============================================================
# 4. LOAD CASE + STRUCTURED TARGETS
# ============================================================

training_manifest = pd.read_csv(
    TRAIN_MANIFEST,
    low_memory=False
)

structured_df = pd.read_csv(
    STRUCTURED_TARGET_FILE,
    low_memory=False
)

image_manifest = pd.read_csv(
    IMAGE_MANIFEST,
    low_memory=False
)

print("\nTraining manifest:", training_manifest.shape)
print("Structured targets:", structured_df.shape)
print("Image manifest:", image_manifest.shape)

print("\nStructured target columns:")
print(
    structured_df.columns.tolist()
)

# ------------------------------------------------------------
# Check case_id
# ------------------------------------------------------------

if "case_id" not in training_manifest.columns:
    raise ValueError(
        "training_manifest has no case_id"
    )

if "case_id" not in structured_df.columns:
    raise ValueError(
        "structured target file has no case_id"
    )

# ------------------------------------------------------------
# Find structured columns
# ------------------------------------------------------------

required_structured = [
    "case_id",
    "concept_targets",
    "negative_targets",
    "attribute_targets"
]

missing_structured = [
    c for c in required_structured
    if c not in structured_df.columns
]

if missing_structured:
    raise ValueError(
        f"Structured target file is missing: "
        f"{missing_structured}"
    )

# ============================================================
# 5. MERGE STRUCTURED TARGETS INTO TRAINING MANIFEST
# ============================================================

training_manifest["case_id"] = (
    training_manifest["case_id"]
    .astype(str)
)

structured_df["case_id"] = (
    structured_df["case_id"]
    .astype(str)
)

# Avoid accidental duplicated columns
merge_cols = [
    "case_id",
    "concept_targets",
    "negative_targets",
    "attribute_targets"
]

structured_subset = structured_df[
    merge_cols
].copy()

# Check duplicate case IDs
dup_structured = (
    structured_subset["case_id"]
    .duplicated()
    .sum()
)

print(
    "\nDuplicate case IDs in structured targets:",
    dup_structured
)

if dup_structured:
    raise ValueError(
        "Structured target file contains duplicate case_id."
    )

cases = training_manifest.merge(
    structured_subset,
    on="case_id",
    how="left",
    validate="one_to_one"
)

# ============================================================
# 6. VALIDATION ONLY
# ============================================================

cases["split"] = (
    cases["split"]
    .astype(str)
    .str.lower()
)

val_df = cases[
    cases["split"] == "val"
].copy()

print(
    "\nValidation cases:",
    len(val_df)
)

missing_targets = val_df[
    [
        "concept_targets",
        "negative_targets",
        "attribute_targets"
    ]
].isna().any(axis=1).sum()

print(
    "Validation cases missing structured targets:",
    missing_targets
)

if missing_targets:
    raise ValueError(
        "Some validation cases have missing structured targets."
    )

# ============================================================
# 7. IMAGE INDEX
# ============================================================

normal_images = image_manifest[
    image_manifest["image_status"]
    .astype(str)
    .str.upper()
    == "NORMAL"
].copy()

normal_images["case_id"] = (
    normal_images["case_id"]
    .astype(str)
)

val_case_ids = set(
    val_df["case_id"].astype(str)
)

val_images = normal_images[
    normal_images["case_id"].isin(val_case_ids)
].copy()

print(
    "\nValidation NORMAL images:",
    len(val_images)
)

case_to_images = (
    val_images
    .groupby("case_id")["image_path"]
    .apply(list)
    .to_dict()
)

# ============================================================
# 8. DATASET
# ============================================================

IMAGE_SIZE = int(
    config.get(
        "image_size",
        224
    )
)

MAX_IMAGES = int(
    config.get(
        "max_images_per_case",
        8
    )
)

image_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


class V3ValDataset(
    torch.utils.data.Dataset
):

    def __init__(
        self,
        df,
        case_to_images
    ):

        self.df = df.reset_index(
            drop=True
        )

        self.case_to_images = (
            case_to_images
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        case_id = str(
            row["case_id"]
        )

        paths = self.case_to_images.get(
            case_id,
            []
        )

        paths = sorted(paths)[
            :MAX_IMAGES
        ]

        images = []

        for path in paths:

            try:

                img = Image.open(
                    path
                ).convert("RGB")

                img = image_transform(
                    img
                )

                images.append(img)

            except Exception as e:

                print(
                    "Image error:",
                    path,
                    e
                )

        if not images:

            raise RuntimeError(
                f"No usable NORMAL images "
                f"for case {case_id}"
            )

        return {
            "case_id": case_id,

            "images": images,

            "num_images": len(
                images
            ),

            "concept_targets":
                np.asarray(
                    json.loads(
                        row[
                            "concept_targets"
                        ]
                    ),
                    dtype=np.float32
                ),

            "negative_targets":
                np.asarray(
                    json.loads(
                        row[
                            "negative_targets"
                        ]
                    ),
                    dtype=np.float32
                ),

            "attribute_targets":
                np.asarray(
                    json.loads(
                        row[
                            "attribute_targets"
                        ]
                    ),
                    dtype=np.float32
                )
        }


def collate_fn(batch):

    max_n = max(
        x["num_images"]
        for x in batch
    )

    B = len(batch)

    images = torch.zeros(
        B,
        max_n,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE
    )

    attention = torch.zeros(
        B,
        max_n,
        dtype=torch.long
    )

    for i, item in enumerate(batch):

        n = item["num_images"]

        images[
            i,
            :n
        ] = torch.stack(
            item["images"]
        )

        attention[
            i,
            :n
        ] = 1

    return {

        "case_id": [
            x["case_id"]
            for x in batch
        ],

        "images": images,

        "image_attention": attention,

        "concept_targets":
            torch.tensor(
                np.stack([
                    x["concept_targets"]
                    for x in batch
                ]),
                dtype=torch.float32
            ),

        "negative_targets":
            torch.tensor(
                np.stack([
                    x["negative_targets"]
                    for x in batch
                ]),
                dtype=torch.float32
            ),

        "attribute_targets":
            torch.tensor(
                np.stack([
                    x["attribute_targets"]
                    for x in batch
                ]),
                dtype=torch.float32
            )
    }


val_dataset = V3ValDataset(
    val_df,
    case_to_images
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn
)

print(
    "\nValidation loader:",
    len(val_dataset),
    "cases"
)

print(
    "Validation batches:",
    len(val_loader)
)

# ============================================================
# 9. MODEL
# ============================================================

class VisualProjector(nn.Module):

    def __init__(
        self,
        in_dim=768,
        out_dim=512
    ):

        super().__init__()

        self.proj = nn.Linear(
            in_dim,
            out_dim
        )

    def forward(self, x):

        return self.proj(x)


class V3Model(nn.Module):

    def __init__(self):

        super().__init__()

        self.vit = ViTModel.from_pretrained(
            "google/vit-base-patch16-224"
        )

        self.mt5 = MT5ForConditionalGeneration.from_pretrained(
            "google/mt5-small"
        )

        self.projector = VisualProjector(
            self.vit.config.hidden_size,
            self.mt5.config.d_model
        )

        self.concept_head = nn.Linear(
            self.mt5.config.d_model,
            N_CONCEPTS
        )

        self.negative_head = nn.Linear(
            self.mt5.config.d_model,
            N_NEGATIVES
        )

        self.attribute_head = nn.Linear(
            self.mt5.config.d_model,
            N_ATTRIBUTES
        )

    def forward_visual(
        self,
        images,
        image_attention
    ):

        B, N, C, H, W = (
            images.shape
        )

        flat = images.reshape(
            B * N,
            C,
            H,
            W
        )

        vit_out = self.vit(
            pixel_values=flat
        )

        cls = (
            vit_out
            .last_hidden_state[:, 0]
        )

        cls = cls.reshape(
            B,
            N,
            -1
        )

        visual_tokens = (
            self.projector(cls)
        )

        attention = (
            image_attention
            .unsqueeze(-1)
            .float()
        )

        pooled = (
            visual_tokens * attention
        ).sum(dim=1) / (
            attention
            .sum(dim=1)
            .clamp(min=1.0)
        )

        concept_logits = (
            self.concept_head(
                pooled
            )
        )

        negative_logits = (
            self.negative_head(
                pooled
            )
        )

        attribute_logits = (
            self.attribute_head(
                pooled
            )
        )

        return (
            concept_logits,
            negative_logits,
            attribute_logits
        )

# ============================================================
# 10. LOAD BEST CHECKPOINT
# ============================================================

print(
    "\nLoading checkpoint:"
)

print(CHECKPOINT)

if not os.path.exists(
    CHECKPOINT
):
    raise FileNotFoundError(
        CHECKPOINT
    )

model = V3Model()

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

missing_keys, unexpected_keys = (
    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ],
        strict=False
    )
)

print(
    "Missing keys:",
    missing_keys
)

print(
    "Unexpected keys:",
    unexpected_keys
)

model = model.to(device)
model.eval()

# ============================================================
# 11. VALIDATION INFERENCE
# ============================================================

all_case_ids = []

concept_probs = []
negative_probs = []
attribute_probs = []

concept_targets = []
negative_targets = []
attribute_targets = []

print(
    "\nRunning validation inference..."
)

with torch.no_grad():

    for step, batch in enumerate(
        val_loader
    ):

        images = batch[
            "images"
        ].to(
            device,
            non_blocking=True
        )

        image_attention = batch[
            "image_attention"
        ].to(
            device,
            non_blocking=True
        )

        if device.type == "cuda":

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16
            ):

                (
                    c_logits,
                    n_logits,
                    a_logits
                ) = model.forward_visual(
                    images,
                    image_attention
                )

        else:

            (
                c_logits,
                n_logits,
                a_logits
            ) = model.forward_visual(
                images,
                image_attention
            )

        all_case_ids.extend(
            batch["case_id"]
        )

        concept_probs.append(
            torch.sigmoid(
                c_logits
            )
            .float()
            .cpu()
            .numpy()
        )

        negative_probs.append(
            torch.sigmoid(
                n_logits
            )
            .float()
            .cpu()
            .numpy()
        )

        attribute_probs.append(
            torch.sigmoid(
                a_logits
            )
            .float()
            .cpu()
            .numpy()
        )

        concept_targets.append(
            batch[
                "concept_targets"
            ].numpy()
        )

        negative_targets.append(
            batch[
                "negative_targets"
            ].numpy()
        )

        attribute_targets.append(
            batch[
                "attribute_targets"
            ].numpy()
        )

        if (
            (step + 1) % 50 == 0
        ):

            print(
                f"  {step+1}/"
                f"{len(val_loader)} batches"
            )

concept_probs = np.concatenate(
    concept_probs,
    axis=0
)

negative_probs = np.concatenate(
    negative_probs,
    axis=0
)

attribute_probs = np.concatenate(
    attribute_probs,
    axis=0
)

concept_targets = np.concatenate(
    concept_targets,
    axis=0
)

negative_targets = np.concatenate(
    negative_targets,
    axis=0
)

attribute_targets = np.concatenate(
    attribute_targets,
    axis=0
)

print("\nShapes:")

print(
    " concept :",
    concept_probs.shape
)

print(
    " negative:",
    negative_probs.shape
)

print(
    " attrs   :",
    attribute_probs.shape
)

# ============================================================
# 12. THRESHOLD SEARCH
# ============================================================

thresholds = np.round(
    np.arange(
        0.05,
        0.951,
        0.05
    ),
    2
)


def evaluate_thresholds(
    probs,
    targets,
    names,
    thresholds
):

    rows = []

    for j, name in enumerate(
        names
    ):

        y_true = (
            targets[:, j]
            .astype(int)
        )

        p = probs[:, j]

        support = int(
            y_true.sum()
        )

        best = None

        for threshold in thresholds:

            y_pred = (
                p >= threshold
            ).astype(int)

            f1 = f1_score(
                y_true,
                y_pred,
                zero_division=0
            )

            precision = precision_score(
                y_true,
                y_pred,
                zero_division=0
            )

            recall = recall_score(
                y_true,
                y_pred,
                zero_division=0
            )

            result = {

                "label": name,

                "support": support,

                "threshold":
                    float(threshold),

                "f1":
                    float(f1),

                "precision":
                    float(precision),

                "recall":
                    float(recall),

                "pred_positive_rate":
                    float(
                        y_pred.mean()
                    ),

                "mean_probability":
                    float(
                        p.mean()
                    )
            }

            if (
                best is None
                or result["f1"]
                > best["f1"]
                or (
                    result["f1"]
                    == best["f1"]
                    and result[
                        "threshold"
                    ]
                    >
                    best[
                        "threshold"
                    ]
                )
            ):

                best = result

        rows.append(best)

    return pd.DataFrame(rows)


concept_best = evaluate_thresholds(
    concept_probs,
    concept_targets,
    concept_names,
    thresholds
)

negative_best = evaluate_thresholds(
    negative_probs,
    negative_targets,
    negative_names,
    thresholds
)

attribute_best = evaluate_thresholds(
    attribute_probs,
    attribute_targets,
    attribute_names,
    thresholds
)

concept_best["head"] = "concept"
negative_best["head"] = "negative"
attribute_best["head"] = "attribute"

threshold_table = pd.concat(
    [
        concept_best,
        negative_best,
        attribute_best
    ],
    ignore_index=True
)

# ============================================================
# 13. HEAD-LEVEL THRESHOLDS
# ============================================================

def micro_f1_at_threshold(
    probs,
    targets,
    threshold
):

    pred = (
        probs >= threshold
    ).astype(int)

    return f1_score(
        targets.reshape(-1),
        pred.reshape(-1),
        zero_division=0
    )


head_summary = []

for (
    head_name,
    probs,
    targets
) in [

    (
        "concept",
        concept_probs,
        concept_targets
    ),

    (
        "negative",
        negative_probs,
        negative_targets
    ),

    (
        "attribute",
        attribute_probs,
        attribute_targets
    )
]:

    best = None

    for threshold in thresholds:

        f1 = micro_f1_at_threshold(
            probs,
            targets,
            threshold
        )

        if (
            best is None
            or f1 > best["micro_f1"]
        ):

            best = {
                "head": head_name,
                "threshold":
                    float(threshold),
                "micro_f1":
                    float(f1)
            }

    head_summary.append(best)

head_summary = pd.DataFrame(
    head_summary
)

# ============================================================
# 14. SAVE
# ============================================================

threshold_table.to_csv(
    THRESHOLD_CSV,
    index=False
)

threshold_json = {

    "source":
        "validation_only",

    "checkpoint":
        CHECKPOINT,

    "threshold_grid":
        thresholds.tolist(),

    "concept": {
        row["label"]:
            float(row["threshold"])
        for _, row
        in concept_best.iterrows()
    },

    "negative": {
        row["label"]:
            float(row["threshold"])
        for _, row
        in negative_best.iterrows()
    },

    "attribute": {
        row["label"]:
            float(row["threshold"])
        for _, row
        in attribute_best.iterrows()
    },

    "head_level":
        head_summary.to_dict(
            orient="records"
        )
}

with open(
    THRESHOLD_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        threshold_json,
        f,
        ensure_ascii=False,
        indent=2
    )

# ============================================================
# 15. RESULTS
# ============================================================

print("\n" + "=" * 70)
print("VALIDATION THRESHOLD RESULTS")
print("=" * 70)

print(
    "\nHead-level thresholds:"
)

print(
    head_summary.to_string(
        index=False
    )
)

for head in [
    "concept",
    "negative",
    "attribute"
]:

    print(
        f"\n--- {head.upper()} ---"
    )

    display(
        threshold_table[
            threshold_table["head"]
            == head
        ][
            [
                "label",
                "support",
                "threshold",
                "f1",
                "precision",
                "recall",
                "pred_positive_rate"
            ]
        ].sort_values(
            [
                "support",
                "f1"
            ],
            ascending=[
                False,
                False
            ]
        )
    )

print("\nSaved:")
print(
    THRESHOLD_CSV
)

print(
    THRESHOLD_JSON
)

print(
    "\nPASS — validation calibration complete."
)

print(
    "TEST SET WAS NOT USED."
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Structured target file:
/content/drive/MyDrive/NoiSoi_Matching/v3_structured_targets_v2_2.csv
Exists: False


FileNotFoundError: /content/drive/MyDrive/NoiSoi_Matching/v3_structured_targets_v2_2.csv

In [7]:
import os

ROOT = "/content/drive/MyDrive/NoiSoi_Matching"

print("Searching for v3_structured_targets_v2_2.csv ...")

matches = []

for root, dirs, files in os.walk(ROOT):
    for f in files:
        if f == "v3_structured_targets_v2_2.csv":
            matches.append(os.path.join(root, f))

print("\nFound:", len(matches))

for p in matches:
    print(p)


Searching for v3_structured_targets_v2_2.csv ...

Found: 1
/content/drive/MyDrive/NoiSoi_Matching/v3_ontology_v2/v3_structured_targets_v2_2.csv


In [1]:
# ============================================================
# V3-6 — VALIDATION THRESHOLD CALIBRATION
# FIXED PATH
#
# IMPORTANT:
# - Validation ONLY
# - No training
# - No test-set access
# - Runtime-independent
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from google.colab import drive
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from transformers import ViTModel, MT5ForConditionalGeneration

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)

# ============================================================
# 1. MOUNT + PATHS
# ============================================================

drive.mount(
    "/content/drive",
    force_remount=False
)

PROJECT_DIR = "/content/drive/MyDrive/NoiSoi_Matching"
V3_DIR = os.path.join(
    PROJECT_DIR,
    "baseline_model_v3"
)

TRAIN_MANIFEST = os.path.join(
    V3_DIR,
    "v3_training_manifest.csv"
)

# CORRECT PATH — FOUND BY YOUR SEARCH
STRUCTURED_TARGET_FILE = os.path.join(
    PROJECT_DIR,
    "v3_ontology_v2",
    "v3_structured_targets_v2_2.csv"
)

IMAGE_MANIFEST = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

CONFIG_FILE = os.path.join(
    V3_DIR,
    "v3_config.json"
)

CHECKPOINT = os.path.join(
    V3_DIR,
    "checkpoints",
    "best.pt"
)

OUT_DIR = os.path.join(
    V3_DIR,
    "structured_diagnostic"
)

os.makedirs(
    OUT_DIR,
    exist_ok=True
)

THRESHOLD_CSV = os.path.join(
    OUT_DIR,
    "validation_best_thresholds.csv"
)

THRESHOLD_JSON = os.path.join(
    OUT_DIR,
    "validation_best_thresholds.json"
)

# ============================================================
# 2. CHECK FILES BEFORE LOADING ANYTHING
# ============================================================

print("=" * 70)
print("V3-6 — VALIDATION THRESHOLD CALIBRATION")
print("=" * 70)

required_files = {
    "training manifest": TRAIN_MANIFEST,
    "structured targets": STRUCTURED_TARGET_FILE,
    "image manifest": IMAGE_MANIFEST,
    "V3 config": CONFIG_FILE,
    "V3 best checkpoint": CHECKPOINT,
}

for name, path in required_files.items():

    exists = os.path.exists(path)

    print(
        f"{name:25s}: "
        f"{'OK' if exists else 'MISSING'}"
    )
    print(
        f"  {path}"
    )

    if not exists:
        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

# ============================================================
# 3. DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:", device)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        f"VRAM: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB"
    )

# ============================================================
# 4. CONFIG
# ============================================================

with open(
    CONFIG_FILE,
    "r",
    encoding="utf-8"
) as f:

    config = json.load(f)

concept_names = config["concept_vocab"]
negative_names = config["negative_vocab"]
attribute_names = config["attribute_vocab"]

N_CONCEPTS = len(concept_names)
N_NEGATIVES = len(negative_names)
N_ATTRIBUTES = len(attribute_names)

print("\nOntology:")
print("  concepts :", N_CONCEPTS)
print("  negatives:", N_NEGATIVES)
print("  attrs    :", N_ATTRIBUTES)

# ============================================================
# 5. LOAD MANIFESTS
# ============================================================

training_manifest = pd.read_csv(
    TRAIN_MANIFEST,
    low_memory=False
)

structured_df = pd.read_csv(
    STRUCTURED_TARGET_FILE,
    low_memory=False
)

image_manifest = pd.read_csv(
    IMAGE_MANIFEST,
    low_memory=False
)

print("\nLoaded:")
print(
    "Training manifest:",
    training_manifest.shape
)

print(
    "Structured targets:",
    structured_df.shape
)

print(
    "Image manifest:",
    image_manifest.shape
)

# ============================================================
# 6. CHECK STRUCTURED TARGET FILE
# ============================================================

required_structured = [
    "case_id",
    "concept_targets",
    "negative_targets",
    "attribute_targets"
]

missing = [
    c
    for c in required_structured
    if c not in structured_df.columns
]

if missing:

    print(
        "\nAvailable columns:"
    )

    print(
        structured_df.columns.tolist()
    )

    raise ValueError(
        f"Missing structured columns: {missing}"
    )

print(
    "\nStructured target columns: PASS"
)

# ============================================================
# 7. MERGE STRUCTURED TARGETS
# ============================================================

training_manifest["case_id"] = (
    training_manifest["case_id"]
    .astype(str)
)

structured_df["case_id"] = (
    structured_df["case_id"]
    .astype(str)
)

structured_subset = structured_df[
    [
        "case_id",
        "concept_targets",
        "negative_targets",
        "attribute_targets"
    ]
].copy()

duplicate_structured = (
    structured_subset["case_id"]
    .duplicated()
    .sum()
)

print(
    "Duplicate structured case IDs:",
    duplicate_structured
)

if duplicate_structured:
    raise ValueError(
        "Structured target file contains duplicate case_id."
    )

cases = training_manifest.merge(
    structured_subset,
    on="case_id",
    how="left",
    validate="one_to_one"
)

# ============================================================
# 8. VALIDATION ONLY
# ============================================================

cases["split"] = (
    cases["split"]
    .astype(str)
    .str.lower()
)

val_df = cases[
    cases["split"] == "val"
].copy()

print(
    "\nValidation cases:",
    len(val_df)
)

assert len(val_df) == 756, (
    f"Expected 756 validation cases, "
    f"got {len(val_df)}"
)

missing_target_rows = val_df[
    [
        "concept_targets",
        "negative_targets",
        "attribute_targets"
    ]
].isna().any(axis=1).sum()

print(
    "Validation cases missing targets:",
    missing_target_rows
)

if missing_target_rows:

    raise ValueError(
        "Validation contains missing structured targets."
    )

# ============================================================
# 9. NORMAL IMAGE INDEX
# ============================================================

normal_images = image_manifest[
    image_manifest["image_status"]
    .astype(str)
    .str.upper()
    == "NORMAL"
].copy()

normal_images["case_id"] = (
    normal_images["case_id"]
    .astype(str)
)

val_case_ids = set(
    val_df["case_id"]
    .astype(str)
)

val_images = normal_images[
    normal_images["case_id"]
    .isin(val_case_ids)
].copy()

print(
    "\nValidation NORMAL images:",
    len(val_images)
)

case_to_images = (
    val_images
    .groupby("case_id")["image_path"]
    .apply(list)
    .to_dict()
)

# ============================================================
# 10. DATASET
# ============================================================

IMAGE_SIZE = int(
    config.get(
        "image_size",
        224
    )
)

MAX_IMAGES = int(
    config.get(
        "max_images_per_case",
        8
    )
)

image_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


class V3ValDataset(Dataset):

    def __init__(
        self,
        df,
        case_to_images
    ):

        self.df = df.reset_index(
            drop=True
        )

        self.case_to_images = (
            case_to_images
        )

    def __len__(self):

        return len(self.df)

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        case_id = str(
            row["case_id"]
        )

        paths = self.case_to_images.get(
            case_id,
            []
        )

        paths = sorted(paths)[
            :MAX_IMAGES
        ]

        images = []

        for path in paths:

            with Image.open(path) as img:

                img = img.convert("RGB")

                img = image_transform(
                    img
                )

                images.append(img)

        if not images:

            raise RuntimeError(
                f"No NORMAL image for {case_id}"
            )

        return {

            "case_id": case_id,

            "images": images,

            "num_images": len(
                images
            ),

            "concept_targets":
                np.asarray(
                    json.loads(
                        row[
                            "concept_targets"
                        ]
                    ),
                    dtype=np.float32
                ),

            "negative_targets":
                np.asarray(
                    json.loads(
                        row[
                            "negative_targets"
                        ]
                    ),
                    dtype=np.float32
                ),

            "attribute_targets":
                np.asarray(
                    json.loads(
                        row[
                            "attribute_targets"
                        ]
                    ),
                    dtype=np.float32
                )
        }


def collate_fn(batch):

    max_n = max(
        x["num_images"]
        for x in batch
    )

    B = len(batch)

    images = torch.zeros(
        B,
        max_n,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE
    )

    attention = torch.zeros(
        B,
        max_n,
        dtype=torch.long
    )

    for i, item in enumerate(batch):

        n = item["num_images"]

        images[
            i,
            :n
        ] = torch.stack(
            item["images"]
        )

        attention[
            i,
            :n
        ] = 1

    return {

        "case_id": [
            x["case_id"]
            for x in batch
        ],

        "images": images,

        "image_attention": attention,

        "concept_targets":
            torch.tensor(
                np.stack([
                    x["concept_targets"]
                    for x in batch
                ]),
                dtype=torch.float32
            ),

        "negative_targets":
            torch.tensor(
                np.stack([
                    x["negative_targets"]
                    for x in batch
                ]),
                dtype=torch.float32
            ),

        "attribute_targets":
            torch.tensor(
                np.stack([
                    x["attribute_targets"]
                    for x in batch
                ]),
                dtype=torch.float32
            )
    }


val_dataset = V3ValDataset(
    val_df,
    case_to_images
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn
)

print(
    "\nValidation dataset:",
    len(val_dataset)
)

print(
    "Validation batches:",
    len(val_loader)
)

# ============================================================
# 11. MODEL
# ============================================================

class VisualProjector(nn.Module):

    def __init__(
        self,
        in_dim=768,
        out_dim=512
    ):

        super().__init__()

        self.proj = nn.Linear(
            in_dim,
            out_dim
        )

    def forward(self, x):

        return self.proj(x)


class V3Model(nn.Module):

    def __init__(self):

        super().__init__()

        self.vit = ViTModel.from_pretrained(
            "google/vit-base-patch16-224"
        )

        self.mt5 = MT5ForConditionalGeneration.from_pretrained(
            "google/mt5-small"
        )

        self.projector = VisualProjector(
            self.vit.config.hidden_size,
            self.mt5.config.d_model
        )

        self.concept_head = nn.Linear(
            self.mt5.config.d_model,
            N_CONCEPTS
        )

        self.negative_head = nn.Linear(
            self.mt5.config.d_model,
            N_NEGATIVES
        )

        self.attribute_head = nn.Linear(
            self.mt5.config.d_model,
            N_ATTRIBUTES
        )

    def forward_visual(
        self,
        images,
        image_attention
    ):

        B, N, C, H, W = (
            images.shape
        )

        flat = images.reshape(
            B * N,
            C,
            H,
            W
        )

        vit_output = self.vit(
            pixel_values=flat
        )

        cls = (
            vit_output
            .last_hidden_state[:, 0, :]
        )

        cls = cls.reshape(
            B,
            N,
            -1
        )

        visual_tokens = (
            self.projector(cls)
        )

        attention = (
            image_attention
            .unsqueeze(-1)
            .float()
        )

        pooled = (
            visual_tokens * attention
        ).sum(dim=1) / (
            attention
            .sum(dim=1)
            .clamp(min=1.0)
        )

        concept_logits = (
            self.concept_head(pooled)
        )

        negative_logits = (
            self.negative_head(pooled)
        )

        attribute_logits = (
            self.attribute_head(pooled)
        )

        return (
            concept_logits,
            negative_logits,
            attribute_logits
        )

# ============================================================
# 12. LOAD BEST V3 CHECKPOINT
# ============================================================

model = V3Model()

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ],
    strict=True
)

model = model.to(device)
model.eval()

print(
    "\nV3 best checkpoint loaded."
)

if "epoch" in checkpoint:
    print(
        "Checkpoint epoch:",
        checkpoint["epoch"]
    )

# ============================================================
# 13. VALIDATION INFERENCE
# ============================================================

concept_probs = []
negative_probs = []
attribute_probs = []

concept_targets = []
negative_targets = []
attribute_targets = []

print(
    "\nRunning validation inference..."
)

with torch.no_grad():

    for step, batch in enumerate(
        val_loader
    ):

        images = batch[
            "images"
        ].to(
            device,
            non_blocking=True
        )

        image_attention = batch[
            "image_attention"
        ].to(
            device,
            non_blocking=True
        )

        if device.type == "cuda":

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16
            ):

                (
                    concept_logits,
                    negative_logits,
                    attribute_logits
                ) = model.forward_visual(
                    images,
                    image_attention
                )

        else:

            (
                concept_logits,
                negative_logits,
                attribute_logits
            ) = model.forward_visual(
                images,
                image_attention
            )

        concept_probs.append(
            torch.sigmoid(
                concept_logits
            )
            .float()
            .cpu()
            .numpy()
        )

        negative_probs.append(
            torch.sigmoid(
                negative_logits
            )
            .float()
            .cpu()
            .numpy()
        )

        attribute_probs.append(
            torch.sigmoid(
                attribute_logits
            )
            .float()
            .cpu()
            .numpy()
        )

        concept_targets.append(
            batch[
                "concept_targets"
            ].numpy()
        )

        negative_targets.append(
            batch[
                "negative_targets"
            ].numpy()
        )

        attribute_targets.append(
            batch[
                "attribute_targets"
            ].numpy()
        )

        if (
            (step + 1) % 50 == 0
        ):

            print(
                f"{step+1}/{len(val_loader)}"
            )

concept_probs = np.concatenate(
    concept_probs,
    axis=0
)

negative_probs = np.concatenate(
    negative_probs,
    axis=0
)

attribute_probs = np.concatenate(
    attribute_probs,
    axis=0
)

concept_targets = np.concatenate(
    concept_targets,
    axis=0
)

negative_targets = np.concatenate(
    negative_targets,
    axis=0
)

attribute_targets = np.concatenate(
    attribute_targets,
    axis=0
)

print(
    "\nProbability shapes:"
)

print(
    "Concept :",
    concept_probs.shape
)

print(
    "Negative:",
    negative_probs.shape
)

print(
    "Attribute:",
    attribute_probs.shape
)

assert concept_probs.shape == (
    756,
    48
)

assert negative_probs.shape == (
    756,
    5
)

assert attribute_probs.shape == (
    756,
    6
)

# ============================================================
# 14. THRESHOLD SEARCH
# ============================================================

thresholds = np.round(
    np.arange(
        0.05,
        0.951,
        0.05
    ),
    2
)


def find_best_thresholds(
    probs,
    targets,
    names
):

    rows = []

    for j, name in enumerate(names):

        y_true = (
            targets[:, j]
            .astype(int)
        )

        probabilities = probs[:, j]

        support = int(
            y_true.sum()
        )

        best = None

        for threshold in thresholds:

            y_pred = (
                probabilities >= threshold
            ).astype(int)

            f1 = f1_score(
                y_true,
                y_pred,
                zero_division=0
            )

            precision = precision_score(
                y_true,
                y_pred,
                zero_division=0
            )

            recall = recall_score(
                y_true,
                y_pred,
                zero_division=0
            )

            result = {

                "label": name,

                "support": support,

                "threshold":
                    float(threshold),

                "f1":
                    float(f1),

                "precision":
                    float(precision),

                "recall":
                    float(recall),

                "pred_positive_rate":
                    float(
                        y_pred.mean()
                    ),

                "mean_probability":
                    float(
                        probabilities.mean()
                    )
            }

            if (
                best is None
                or result["f1"]
                > best["f1"]
                or (
                    result["f1"]
                    == best["f1"]
                    and result["threshold"]
                    > best["threshold"]
                )
            ):

                best = result

        rows.append(best)

    return pd.DataFrame(rows)


concept_best = find_best_thresholds(
    concept_probs,
    concept_targets,
    concept_names
)

negative_best = find_best_thresholds(
    negative_probs,
    negative_targets,
    negative_names
)

attribute_best = find_best_thresholds(
    attribute_probs,
    attribute_targets,
    attribute_names
)

concept_best["head"] = "concept"
negative_best["head"] = "negative"
attribute_best["head"] = "attribute"

threshold_table = pd.concat(
    [
        concept_best,
        negative_best,
        attribute_best
    ],
    ignore_index=True
)

# ============================================================
# 15. HEAD-LEVEL SUMMARY
# ============================================================

def get_head_best(
    probs,
    targets,
    head_name
):

    best = None

    for threshold in thresholds:

        pred = (
            probs >= threshold
        ).astype(int)

        micro_f1 = f1_score(
            targets.reshape(-1),
            pred.reshape(-1),
            zero_division=0
        )

        if (
            best is None
            or micro_f1
            > best["micro_f1"]
        ):

            best = {

                "head": head_name,

                "threshold":
                    float(threshold),

                "micro_f1":
                    float(micro_f1)
            }

    return best


head_summary = pd.DataFrame([
    get_head_best(
        concept_probs,
        concept_targets,
        "concept"
    ),
    get_head_best(
        negative_probs,
        negative_targets,
        "negative"
    ),
    get_head_best(
        attribute_probs,
        attribute_targets,
        "attribute"
    )
])

# ============================================================
# 16. SAVE RESULTS
# ============================================================

threshold_table.to_csv(
    THRESHOLD_CSV,
    index=False
)

threshold_json = {

    "source":
        "validation_only",

    "checkpoint":
        CHECKPOINT,

    "ontology_version":
        config["ontology_version"],

    "threshold_grid":
        thresholds.tolist(),

    "concept": {
        row["label"]:
            float(row["threshold"])
        for _, row
        in concept_best.iterrows()
    },

    "negative": {
        row["label"]:
            float(row["threshold"])
        for _, row
        in negative_best.iterrows()
    },

    "attribute": {
        row["label"]:
            float(row["threshold"])
        for _, row
        in attribute_best.iterrows()
    },

    "head_level":
        head_summary.to_dict(
            orient="records"
        )
}

with open(
    THRESHOLD_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        threshold_json,
        f,
        ensure_ascii=False,
        indent=2
    )

# ============================================================
# 17. PRINT
# ============================================================

print("\n" + "=" * 70)
print("VALIDATION THRESHOLD CALIBRATION — COMPLETE")
print("=" * 70)

print(
    "\nHEAD-LEVEL RESULTS:"
)

print(
    head_summary.to_string(
        index=False
    )
)

for head in [
    "concept",
    "negative",
    "attribute"
]:

    print(
        f"\n--- {head.upper()} ---"
    )

    display(
        threshold_table[
            threshold_table["head"]
            == head
        ][
            [
                "label",
                "support",
                "threshold",
                "f1",
                "precision",
                "recall",
                "pred_positive_rate"
            ]
        ].sort_values(
            [
                "support",
                "f1"
            ],
            ascending=[
                False,
                False
            ]
        )
    )

print(
    "\nSaved:"
)

print(
    THRESHOLD_CSV
)

print(
    THRESHOLD_JSON
)

print(
    "\nTEST SET USED: NO"
)

print(
    "\nPASS — V3-6 validation calibration complete."
)

Mounted at /content/drive
V3-6 — VALIDATION THRESHOLD CALIBRATION
training manifest        : OK
  /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v3/v3_training_manifest.csv
structured targets       : OK
  /content/drive/MyDrive/NoiSoi_Matching/v3_ontology_v2/v3_structured_targets_v2_2.csv
image manifest           : OK
  /content/drive/MyDrive/NoiSoi_Matching/final_manifest/final_dataset_manifest.csv
V3 config                : OK
  /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v3/v3_config.json
V3 best checkpoint       : OK
  /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v3/checkpoints/best.pt

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.41 GB

Ontology:
  concepts : 48
  negatives: 5
  attrs    : 6

Loaded:
Training manifest: (7606, 15)
Structured targets: (7606, 102)
Image manifest: (76405, 26)

Available columns:
['case_id', 'patient_group_id', 'split', 'pdf_path', 'ho_ten', 'nam_sinh', 'gioi_tinh_clean', 'ly_do_noi_soi', 'tai', 'hoc_mui', 'hong_mui', 'hon

ValueError: Missing structured columns: ['concept_targets', 'negative_targets', 'attribute_targets']

In [3]:
# V3-6 DEBUG: inspect actual structured-target schema
import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/NoiSoi_Matching"

STRUCTURED_TARGET_FILE = os.path.join(
    PROJECT_DIR,
    "v3_ontology_v2",
    "v3_structured_targets_v2_2.csv"
)

TRAINING_MANIFEST_FILE = os.path.join(
    PROJECT_DIR,
    "baseline_model_v3",
    "v3_training_manifest.csv"
)

print("Structured file exists:", os.path.exists(STRUCTURED_TARGET_FILE))
print("Training manifest exists:", os.path.exists(TRAINING_MANIFEST_FILE))

# Only inspect schema
structured_df = pd.read_csv(
    STRUCTURED_TARGET_FILE,
    nrows=5
)

manifest_df = pd.read_csv(
    TRAINING_MANIFEST_FILE,
    nrows=5
)

print("\n=== STRUCTURED TARGET FILE ===")
print("Shape sample:", structured_df.shape)
print("Columns:")
for i, c in enumerate(structured_df.columns):
    print(i, repr(c))

print("\nSample:")
display(structured_df.head(3))

print("\n=== TRAINING MANIFEST ===")
print("Columns:")
for i, c in enumerate(manifest_df.columns):
    print(i, repr(c))

print("\nSample:")
display(manifest_df.head(3))

Structured file exists: True
Training manifest exists: True

=== STRUCTURED TARGET FILE ===
Shape sample: (5, 102)
Columns:
0 'case_id'
1 'patient_group_id'
2 'split'
3 'pdf_path'
4 'ho_ten'
5 'nam_sinh'
6 'gioi_tinh_clean'
7 'ly_do_noi_soi'
8 'tai'
9 'hoc_mui'
10 'hong_mui'
11 'hong_thanh_quan'
12 'hong_mieng'
13 'ket_luan'
14 'phan_biet'
15 'de_nghi'
16 'target_norm'
17 'concepts'
18 'attributes'
19 'num_concepts'
20 'num_attributes'
21 'concept__viem_ong_tai_ngoai'
22 'concept__nhot_ong_tai_ngoai'
23 'concept__chan_thuong_ong_tai_ngoai'
24 'concept__viem_tai_giua'
25 'concept__viem_mang_nhi'
26 'concept__thung_mang_nhi'
27 'concept__xep_mang_nhi'
28 'concept__ray_tai'
29 'concept__viem_tai_xuong_chum'
30 'concept__hau_phau_va_nhi'
31 'concept__di_vat_tai'
32 'concept__viem_mui'
33 'concept__viem_mui_xoang'
34 'concept__viem_xoang'
35 'concept__polyp_mui'
36 'concept__chay_mau_mui'
37 'concept__dich_vat_mui'
38 'concept__hoc_xuong_ca'
39 'concept__di_vat_hong_thanh_quan'
40 'concept_

,case_id,patient_group_id,split,pdf_path,ho_ten,nam_sinh,gioi_tinh_clean,ly_do_noi_soi,tai,hoc_mui,...,negative__no_abnormal_ent,negative__no_bleeding,negative__no_foreign_body,concepts_v2_str,negative_findings_str,attributes_v2_str,num_concepts_v2,num_negative_findings,num_attributes_v2,has_structured_label
0,10000.10000.0.10014,NGUYEN TRUONG TAN SANG_2015,train,/content/drive/MyDrive/Hồ sơ bệnh/Hình nội soi...,NGUYỄN TRƯƠNG TẤN SANG,2015,NAM,HỈ MŨI LỎNG,NaN,"Cuốn mũi: hồng, xuất tiết nhầy trong. Khe mũi:...",...,0,0,0,viem_mui,NaN,chronic,1,0,1,True
1,10001.10001.0.10015,INARI ANH TUAN_2014,train,/content/drive/MyDrive/Hồ sơ bệnh/Hình nội soi...,INARI ANH TUẤN,2014,NAM,ĐAU TAI BÊN TRÁI,P: màng nhĩ bóng sáng. T: màng nhĩ bóng sáng.,Cuốn mũi: xuất tiết nhầy trắng Khe mũi: thoáng...,...,0,0,0,viem_mui,NaN,NaN,1,0,0,True
2,10002.10002.0.10016,BUI NGOC BAO CHAU_2016,train,/content/drive/MyDrive/Hồ sơ bệnh/Hình nội soi...,BÙI NGỌC BẢO CHÂU,2016,NỮ,CHẢY MŨI LỎNG,NaN,"Cuốn mũi: nhợt, gồ ghề nhẹ, xuất tiết nhầy trắ...",...,0,0,0,viem_mui,NaN,chronic,1,0,1,True



=== TRAINING MANIFEST ===
Columns:
0 'case_id'
1 'patient_group_id'
2 'split'
3 'ket_luan'
4 'concepts_v2_str'
5 'negative_findings_str'
6 'attributes_v2_str'
7 'concept_vector_str'
8 'negative_vector_str'
9 'attribute_vector_str'
10 'has_structured_label'
11 'structured_loss_mask'
12 'num_concepts_v2'
13 'num_negative_findings'
14 'num_attributes_v2'

Sample:


,case_id,patient_group_id,split,ket_luan,concepts_v2_str,negative_findings_str,attributes_v2_str,concept_vector_str,negative_vector_str,attribute_vector_str,has_structured_label,structured_loss_mask,num_concepts_v2,num_negative_findings,num_attributes_v2
0,10000.10000.0.10014,NGUYEN TRUONG TAN SANG_2015,train,VIÊM MŨI MẠN - VA,viem_mui,NaN,chronic,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0","0,0,1,0,0,0",True,1.0,1,0,1
1,10001.10001.0.10015,INARI ANH TUAN_2014,train,VIÊM MŨI,viem_mui,NaN,NaN,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0","0,0,0,0,0,0",True,1.0,1,0,0
2,10002.10002.0.10016,BUI NGOC BAO CHAU_2016,train,VIÊM MŨI MẠN.,viem_mui,NaN,chronic,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0","0,0,1,0,0,0",True,1.0,1,0,1


In [8]:
# ============================================================
# V3-6 — VALIDATION THRESHOLD CALIBRATION
# Uses frozen v3_training_manifest vector strings directly.
# NO TEST DATA USED.
# ============================================================

import os
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import (
    ViTModel,
    MT5ForConditionalGeneration,
    AutoTokenizer,
)
# ------------------------------------------------------------
# 0. DRIVE + PATHS
# ------------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PROJECT_DIR = "/content/drive/MyDrive/NoiSoi_Matching"
V3_DIR = os.path.join(PROJECT_DIR, "baseline_model_v3")

TRAINING_MANIFEST_FILE = os.path.join(
    V3_DIR,
    "v3_training_manifest.csv"
)

IMAGE_MANIFEST_FILE = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

CONFIG_FILE = os.path.join(
    V3_DIR,
    "v3_config.json"
)

CHECKPOINT_FILE = os.path.join(
    V3_DIR,
    "checkpoints",
    "best.pt"
)

OUTPUT_DIR = os.path.join(
    V3_DIR,
    "structured_diagnostic"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

required_files = {
    "training_manifest": TRAINING_MANIFEST_FILE,
    "image_manifest": IMAGE_MANIFEST_FILE,
    "config": CONFIG_FILE,
    "checkpoint": CHECKPOINT_FILE,
}

for name, path in required_files.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"{name}: {path}")

print("All required files found.")

# ------------------------------------------------------------
# 1. DEVICE
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\nDevice:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        f"VRAM: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

USE_BF16 = (
    device.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

print("BF16:", USE_BF16)

# ------------------------------------------------------------
# 2. LOAD CONFIG + MANIFEST
# ------------------------------------------------------------

with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    config = json.load(f)

print("\nV3 config loaded.")

train_manifest = pd.read_csv(
    TRAINING_MANIFEST_FILE
)

image_manifest = pd.read_csv(
    IMAGE_MANIFEST_FILE
)

print("Training manifest:", train_manifest.shape)
print("Image manifest:", image_manifest.shape)

required_manifest_cols = [
    "case_id",
    "patient_group_id",
    "split",
    "ket_luan",
    "concept_vector_str",
    "negative_vector_str",
    "attribute_vector_str",
    "has_structured_label",
    "structured_loss_mask",
]

missing = [
    c for c in required_manifest_cols
    if c not in train_manifest.columns
]

if missing:
    raise ValueError(
        f"Missing manifest columns: {missing}"
    )

# ------------------------------------------------------------
# 3. FILTER VALIDATION CASES
# ------------------------------------------------------------

# ------------------------------------------------------------
# 3. FILTER STRUCTURED VALIDATION CASES
# ------------------------------------------------------------

val_all_df = train_manifest[
    train_manifest["split"] == "val"
].copy()

val_df = val_all_df[
    val_all_df["structured_loss_mask"] == 1
].copy()

print("\nValidation split total:", len(val_all_df))
print("Validation structured:", len(val_df))
print(
    "Validation report-only:",
    len(val_all_df) - len(val_df)
)

# Frozen V3 split:
# 756 validation cases total
# 751 structured cases
# 5 report-only cases

assert len(val_all_df) == 756
assert len(val_df) == 751

print("Validation split PASS.")
# ------------------------------------------------------------
# 4. PARSE FROZEN VECTOR STRINGS
# ------------------------------------------------------------

def parse_vector(x):
    if pd.isna(x):
        return []

    x = str(x).strip()

    if not x:
        return []

    return [
        int(float(v))
        for v in x.split(",")
    ]


concept_targets = np.stack(
    val_df["concept_vector_str"]
    .apply(parse_vector)
    .values
)

negative_targets = np.stack(
    val_df["negative_vector_str"]
    .apply(parse_vector)
    .values
)

attribute_targets = np.stack(
    val_df["attribute_vector_str"]
    .apply(parse_vector)
    .values
)

print("\nTarget shapes:")
print("concept :", concept_targets.shape)
print("negative:", negative_targets.shape)
print("attribute:", attribute_targets.shape)

assert concept_targets.shape == (751, 48)
assert negative_targets.shape == (751, 5)
assert attribute_targets.shape == (751, 6)

print("Target vector shapes PASS.")

# ------------------------------------------------------------
# 5. IMAGE PATHS
# ------------------------------------------------------------

normal_images = image_manifest[
    image_manifest["image_status"] == "NORMAL"
].copy()

normal_images = normal_images[
    normal_images["case_id"].isin(
        val_df["case_id"]
    )
].copy()

print(
    "\nValidation NORMAL images:",
    len(normal_images)
)

images_by_case = (
    normal_images
    .sort_values(["case_id", "image_path"])
    .groupby("case_id")["image_path"]
    .apply(list)
    .to_dict()
)

missing_cases = [
    cid for cid in val_df["case_id"]
    if cid not in images_by_case
]

if missing_cases:
    raise ValueError(
        f"Validation cases without NORMAL images: "
        f"{len(missing_cases)}"
    )

# ------------------------------------------------------------
# 6. DATASET
# ------------------------------------------------------------

IMAGE_SIZE = 224
MAX_IMAGES_PER_CASE = 8

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406,
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225,
]


class StructuredValDataset(Dataset):

    def __init__(
        self,
        dataframe,
        images_by_case,
    ):
        self.df = dataframe.reset_index(drop=True)
        self.images_by_case = images_by_case

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        case_id = row["case_id"]

        paths = self.images_by_case[case_id]

        # deterministic first 8
        paths = paths[:MAX_IMAGES_PER_CASE]

        imgs = []

        for path in paths:

            img = Image.open(path).convert("RGB")

            img = img.resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                Image.BILINEAR
            )

            arr = np.asarray(
                img,
                dtype=np.float32
            ) / 255.0

            arr = (
                arr - np.array(IMAGENET_MEAN)
            ) / np.array(IMAGENET_STD)

            arr = torch.from_numpy(
                arr
            ).permute(2, 0, 1).float()

            imgs.append(arr)

        return {
            "case_id": case_id,
            "images": torch.stack(imgs),
            "num_images": len(imgs),
        }


def collate_fn(batch):

    max_n = max(
        x["num_images"]
        for x in batch
    )

    B = len(batch)

    images = torch.zeros(
        B,
        max_n,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE,
        dtype=torch.float32,
    )

    attention = torch.zeros(
        B,
        max_n,
        dtype=torch.long,
    )

    case_ids = []

    for i, item in enumerate(batch):

        n = item["num_images"]

        images[
            i,
            :n
        ] = item["images"]

        attention[
            i,
            :n
        ] = 1

        case_ids.append(
            item["case_id"]
        )

    return {
        "case_id": case_ids,
        "images": images,
        "attention": attention,
    }


dataset = StructuredValDataset(
    val_df,
    images_by_case,
)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn,
)

print("\nDataLoader:", len(loader), "batches")

# ------------------------------------------------------------
# 7. MODEL CLASSES — EXACT V3 ARCHITECTURE
# ------------------------------------------------------------

class VisualProjector(nn.Module):

    def __init__(
        self,
        in_dim=768,
        out_dim=512,
    ):
        super().__init__()

        self.proj = nn.Linear(
            in_dim,
            out_dim
        )

    def forward(self, x):
        return self.proj(x)


class V3Model(nn.Module):

    def __init__(
        self,
        vit,
        mt5,
        projector,
        concept_head,
        negative_head,
        attribute_head,
    ):
        super().__init__()

        self.vit = vit
        self.mt5 = mt5
        self.projector = projector

        self.concept_head = concept_head
        self.negative_head = negative_head
        self.attribute_head = attribute_head


# ------------------------------------------------------------
# 8. LOAD MODEL
# ------------------------------------------------------------

print("\nLoading V3 model...")

vit = ViTModel.from_pretrained(
    "google/vit-base-patch16-224"
)

tokenizer = AutoTokenizer.from_pretrained(
    "google/mt5-small"
)

mt5 = MT5ForConditionalGeneration.from_pretrained(
    "google/mt5-small"
)

projector = VisualProjector(
    768,
    512
)

concept_head = nn.Linear(
    512,
    48
)

negative_head = nn.Linear(
    512,
    5
)

attribute_head = nn.Linear(
    512,
    6
)

model = V3Model(
    vit,
    mt5,
    projector,
    concept_head,
    negative_head,
    attribute_head,
)

checkpoint = torch.load(
    CHECKPOINT_FILE,
    map_location="cpu",
    weights_only=False,
)

state_dict = checkpoint["model_state_dict"]

model.load_state_dict(
    state_dict,
    strict=True,
)

model = model.to(device)
model.eval()

print("Checkpoint loaded successfully.")

# ------------------------------------------------------------
# 9. STRUCTURED INFERENCE
# ------------------------------------------------------------

all_concept_probs = []
all_negative_probs = []
all_attribute_probs = []

with torch.no_grad():

    for batch in loader:

        images = batch["images"].to(
            device,
            non_blocking=True
        )

        attention = batch["attention"].to(
            device,
            non_blocking=True
        )

        B, N, C, H, W = images.shape

        flat_images = images.reshape(
            B * N,
            C,
            H,
            W
        )

        if USE_BF16:

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
            ):

                vit_out = model.vit(
                    pixel_values=flat_images
                )

                cls = vit_out.last_hidden_state[:, 0]

                cls = cls.reshape(
                    B,
                    N,
                    768
                )

                visual_tokens = model.projector(
                    cls
                )

                pooled = (
                    visual_tokens * attention.unsqueeze(-1)
                ).sum(dim=1)

                pooled = pooled / (
                    attention.sum(
                        dim=1,
                        keepdim=True
                    ).clamp(min=1)
                )

                concept_logits = model.concept_head(
                    pooled
                )

                negative_logits = model.negative_head(
                    pooled
                )

                attribute_logits = model.attribute_head(
                    pooled
                )

        else:

            vit_out = model.vit(
                pixel_values=flat_images
            )

            cls = vit_out.last_hidden_state[:, 0]

            cls = cls.reshape(
                B,
                N,
                768
            )

            visual_tokens = model.projector(
                cls
            )

            pooled = (
                visual_tokens * attention.unsqueeze(-1)
            ).sum(dim=1)

            pooled = pooled / (
                attention.sum(
                    dim=1,
                    keepdim=True
                ).clamp(min=1)
            )

            concept_logits = model.concept_head(
                pooled
            )

            negative_logits = model.negative_head(
                pooled
            )

            attribute_logits = model.attribute_head(
                pooled
            )

        all_concept_probs.append(
            torch.sigmoid(
                concept_logits
            ).float().cpu().numpy()
        )

        all_negative_probs.append(
            torch.sigmoid(
                negative_logits
            ).float().cpu().numpy()
        )

        all_attribute_probs.append(
            torch.sigmoid(
                attribute_logits
            ).float().cpu().numpy()
        )

concept_probs = np.concatenate(
    all_concept_probs,
    axis=0
)

negative_probs = np.concatenate(
    all_negative_probs,
    axis=0
)

attribute_probs = np.concatenate(
    all_attribute_probs,
    axis=0
)

print("\nProbability shapes:")
print("concept :", concept_probs.shape)
print("negative:", negative_probs.shape)
print("attribute:", attribute_probs.shape)

assert concept_probs.shape == (756, 48)
assert negative_probs.shape == (756, 5)
assert attribute_probs.shape == (756, 6)

print("Inference PASS.")

# ------------------------------------------------------------
# 10. F1 FUNCTION
# ------------------------------------------------------------

def binary_f1(y_true, y_pred):

    y_true = np.asarray(y_true).astype(bool)
    y_pred = np.asarray(y_pred).astype(bool)

    tp = np.logical_and(
        y_true,
        y_pred
    ).sum()

    fp = np.logical_and(
        ~y_true,
        y_pred
    ).sum()

    fn = np.logical_and(
        y_true,
        ~y_pred
    ).sum()

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        / (precision + recall)
    )


# ------------------------------------------------------------
# 11. THRESHOLD SWEEP
# ------------------------------------------------------------

thresholds = np.arange(
    0.05,
    0.951,
    0.05
)

def calibrate_thresholds(
    probs,
    targets,
    prefix,
):

    rows = []

    num_labels = targets.shape[1]

    for j in range(num_labels):

        y_true = targets[:, j]

        best_f1 = -1.0
        best_threshold = 0.5

        for threshold in thresholds:

            y_pred = (
                probs[:, j] >= threshold
            )

            f1 = binary_f1(
                y_true,
                y_pred
            )

            if f1 > best_f1:

                best_f1 = f1
                best_threshold = float(
                    threshold
                )

        rows.append({
            "label": j,
            "best_threshold": best_threshold,
            "best_f1": best_f1,
            "positive_count": int(
                y_true.sum()
            ),
            "mean_probability": float(
                probs[:, j].mean()
            ),
        })

    return pd.DataFrame(rows)


concept_cal = calibrate_thresholds(
    concept_probs,
    concept_targets,
    "concept"
)

negative_cal = calibrate_thresholds(
    negative_probs,
    negative_targets,
    "negative"
)

attribute_cal = calibrate_thresholds(
    attribute_probs,
    attribute_targets,
    "attribute"
)

# ------------------------------------------------------------
# 12. ADD REAL LABEL NAMES
# ------------------------------------------------------------

concept_names = [
    c.replace("concept__", "")
    for c in train_manifest.columns
    if c.startswith("concept__")
]

# The manifest contains the frozen 48-vector.
# Use the ontology file to recover the exact V2 label ordering.
ontology_sample = pd.read_csv(
    os.path.join(
        PROJECT_DIR,
        "v3_ontology_v2",
        "v3_structured_targets_v2_2.csv"
    ),
    nrows=1,
)

concept_cols = [
    c for c in ontology_sample.columns
    if c.startswith("concept__")
]

# Keep only the 48 original frozen V2 concepts.
# Their ordering is the vector ordering used in the manifest.
concept_cols_v2 = [
    c for c in concept_cols
    if c in [
        c for c in train_manifest.columns
        if c.startswith("concept__")
    ]
]

# If the ontology file contains extra V2.2 concepts,
# derive names directly from the frozen vector metadata.
if len(concept_cols_v2) != 48:
    print(
        "Warning: ontology concept-column count differs.",
        len(concept_cols_v2)
    )

negative_names = [
    c.replace("negative__", "")
    for c in ontology_sample.columns
    if c.startswith("negative__")
]

attribute_names = [
    c.replace("attribute__", "")
    for c in ontology_sample.columns
    if c.startswith("attribute__")
]

# Explicit frozen order from V3 training manifest vector width.
# The training vector was created from the V3 frozen ontology.
if len(negative_names) != 5:
    raise ValueError(
        f"Expected 5 negative labels, got {len(negative_names)}"
    )

if len(attribute_names) != 6:
    raise ValueError(
        f"Expected 6 attribute labels, got {len(attribute_names)}"
    )

# For concepts, use the known 48 V3 ontology labels
# from the concept__ columns represented by the frozen vectors.
if len(concept_cols_v2) == 48:
    concept_names_final = [
        c.replace("concept__", "")
        for c in concept_cols_v2
    ]
else:
    concept_names_final = [
        f"concept_{i}"
        for i in range(48)
    ]

concept_cal["label_name"] = [
    concept_names_final[i]
    for i in concept_cal["label"]
]

negative_cal["label_name"] = [
    negative_names[i]
    for i in negative_cal["label"]
]

attribute_cal["label_name"] = [
    attribute_names[i]
    for i in attribute_cal["label"]
]

concept_cal["head"] = "concept"
negative_cal["head"] = "negative"
attribute_cal["head"] = "attribute"

threshold_table = pd.concat(
    [
        concept_cal,
        negative_cal,
        attribute_cal,
    ],
    ignore_index=True
)

threshold_csv = os.path.join(
    OUTPUT_DIR,
    "validation_best_thresholds.csv"
)

threshold_table.to_csv(
    threshold_csv,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 13. JSON THRESHOLDS
# ------------------------------------------------------------

threshold_json = {

    "concept": {
        row["label_name"]:
            float(row["best_threshold"])
        for _, row in concept_cal.iterrows()
    },

    "negative": {
        row["label_name"]:
            float(row["best_threshold"])
        for _, row in negative_cal.iterrows()
    },

    "attribute": {
        row["label_name"]:
            float(row["best_threshold"])
        for _, row in attribute_cal.iterrows()
    },
}

json_path = os.path.join(
    OUTPUT_DIR,
    "validation_best_thresholds.json"
)

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        threshold_json,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 14. SUMMARY
# ------------------------------------------------------------

def micro_f1(
    probs,
    targets,
    threshold,
):

    pred = probs >= threshold

    return binary_f1(
        targets.reshape(-1),
        pred.reshape(-1)
    )


print("\n" + "=" * 70)
print("V3-6 VALIDATION THRESHOLD CALIBRATION")
print("=" * 70)

print(
    "\nConcept macro F1:",
    concept_cal["best_f1"].mean()
)

print(
    "Negative macro F1:",
    negative_cal["best_f1"].mean()
)

print(
    "Attribute macro F1:",
    attribute_cal["best_f1"].mean()
)

print("\nBest concept thresholds:")
display(
    concept_cal.sort_values(
        "best_f1",
        ascending=False
    ).head(15)
)

print("\nBest negative thresholds:")
display(
    negative_cal.sort_values(
        "best_f1",
        ascending=False
    )
)

print("\nBest attribute thresholds:")
display(
    attribute_cal.sort_values(
        "best_f1",
        ascending=False
    )
)

print("\nSaved:")
print(threshold_csv)
print(json_path)

print("\nPASS — validation only; test set was not accessed.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All required files found.

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.49 GB
BF16: True

V3 config loaded.


/tmp/ipykernel_1046/1658825893.py:108: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  image_manifest = pd.read_csv(


Training manifest: (7606, 15)
Image manifest: (76405, 26)

Validation split total: 756
Validation structured: 751
Validation report-only: 5
Validation split PASS.

Target shapes:
concept : (751, 48)
negative: (751, 5)
attribute: (751, 6)
Target vector shapes PASS.

Validation NORMAL images: 7564

DataLoader: 188 batches

Loading V3 model...


config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

...

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.20GB            

model.safetensors: downloading bytes:           |  0.00B            

RuntimeError: Error(s) in loading state_dict for V3Model:
	Missing key(s) in state_dict: "vit.embeddings.cls_token", "vit.embeddings.position_embeddings", "vit.embeddings.patch_embeddings.projection.weight", "vit.embeddings.patch_embeddings.projection.bias", "vit.layers.0.attention.q_proj.weight", "vit.layers.0.attention.q_proj.bias", "vit.layers.0.attention.k_proj.weight", "vit.layers.0.attention.k_proj.bias", "vit.layers.0.attention.v_proj.weight", "vit.layers.0.attention.v_proj.bias", "vit.layers.0.attention.o_proj.weight", "vit.layers.0.attention.o_proj.bias", "vit.layers.0.layernorm_before.weight", "vit.layers.0.layernorm_before.bias", "vit.layers.0.layernorm_after.weight", "vit.layers.0.layernorm_after.bias", "vit.layers.0.mlp.fc1.weight", "vit.layers.0.mlp.fc1.bias", "vit.layers.0.mlp.fc2.weight", "vit.layers.0.mlp.fc2.bias", "vit.layers.1.attention.q_proj.weight", "vit.layers.1.attention.q_proj.bias", "vit.layers.1.attention.k_proj.weight", "vit.layers.1.attention.k_proj.bias",

In [11]:
# ============================================================
# V3-6 — VALIDATION THRESHOLD CALIBRATION
# FINAL CLEAN VERSION
# ============================================================

import os
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from PIL import Image

from transformers import (
    ViTModel,
    MT5ForConditionalGeneration,
)

# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 2. DRIVE + PATHS
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "NoiSoi_Matching"
)

V3_DIR = os.path.join(
    PROJECT_DIR,
    "baseline_model_v3"
)

TRAINING_MANIFEST_FILE = os.path.join(
    V3_DIR,
    "v3_training_manifest.csv"
)

IMAGE_MANIFEST_FILE = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

STRUCTURED_TARGET_FILE = os.path.join(
    PROJECT_DIR,
    "v3_ontology_v2",
    "v3_structured_targets_v2_2.csv"
)

CONFIG_FILE = os.path.join(
    V3_DIR,
    "v3_config.json"
)

CHECKPOINT_FILE = os.path.join(
    V3_DIR,
    "checkpoints",
    "best.pt"
)

OUTPUT_DIR = os.path.join(
    V3_DIR,
    "structured_diagnostic"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 3. FILE CHECK
# ============================================================

required_files = {
    "training_manifest":
        TRAINING_MANIFEST_FILE,

    "image_manifest":
        IMAGE_MANIFEST_FILE,

    "structured_targets":
        STRUCTURED_TARGET_FILE,

    "config":
        CONFIG_FILE,

    "checkpoint":
        CHECKPOINT_FILE,
}

print("=" * 70)
print("FILE CHECK")
print("=" * 70)

for name, path in required_files.items():

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"{name} missing:\n{path}"
        )

    print(
        f"{name}: FOUND"
    )


# ============================================================
# 4. DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

USE_BF16 = (
    device.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

print("\nDevice:", device)

if device.type == "cuda":

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "VRAM:",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

print(
    "BF16:",
    USE_BF16
)


# ============================================================
# 5. LOAD FILES
# ============================================================

with open(
    CONFIG_FILE,
    "r",
    encoding="utf-8"
) as f:

    config = json.load(f)

train_manifest = pd.read_csv(
    TRAINING_MANIFEST_FILE,
    low_memory=False
)

image_manifest = pd.read_csv(
    IMAGE_MANIFEST_FILE,
    low_memory=False
)

structured_all = pd.read_csv(
    STRUCTURED_TARGET_FILE,
    low_memory=False
)

print("\nTraining manifest:", train_manifest.shape)
print("Image manifest:", image_manifest.shape)
print("Structured targets:", structured_all.shape)


# ============================================================
# 6. VALIDATION SPLIT
# ============================================================

val_all = train_manifest[
    train_manifest["split"] == "val"
].copy()

val_df = val_all[
    val_all["structured_loss_mask"] == 1
].copy()

val_df = val_df.reset_index(
    drop=True
)

print("\nValidation total:", len(val_all))
print("Validation structured:", len(val_df))
print(
    "Validation report-only:",
    len(val_all) - len(val_df)
)

assert len(val_all) == 756
assert len(val_df) == 751


# ============================================================
# 7. PARSE TARGET VECTORS
# ============================================================

def parse_vector(x):

    if pd.isna(x):
        return []

    x = str(x).strip()

    if not x:
        return []

    return [
        int(float(v))
        for v in x.split(",")
    ]


concept_targets = np.stack(
    val_df["concept_vector_str"]
    .apply(parse_vector)
    .values
)

negative_targets = np.stack(
    val_df["negative_vector_str"]
    .apply(parse_vector)
    .values
)

attribute_targets = np.stack(
    val_df["attribute_vector_str"]
    .apply(parse_vector)
    .values
)

print("\nTarget shapes:")
print(
    "Concept:",
    concept_targets.shape
)

print(
    "Negative:",
    negative_targets.shape
)

print(
    "Attribute:",
    attribute_targets.shape
)

assert concept_targets.shape == (751, 48)
assert negative_targets.shape == (751, 5)
assert attribute_targets.shape == (751, 6)


# ============================================================
# 8. IMAGE INDEX
# ============================================================

normal_images = image_manifest[
    image_manifest["image_status"] == "NORMAL"
].copy()

normal_images = normal_images[
    normal_images["case_id"].isin(
        val_df["case_id"]
    )
].copy()

images_by_case = (
    normal_images
    .sort_values(
        ["case_id", "image_path"]
    )
    .groupby("case_id")["image_path"]
    .apply(list)
    .to_dict()
)

missing_cases = [
    cid
    for cid in val_df["case_id"]
    if cid not in images_by_case
]

if missing_cases:

    raise ValueError(
        "Cases without NORMAL images: "
        f"{len(missing_cases)}"
    )

print(
    "\nValidation NORMAL images:",
    len(normal_images)
)

print(
    "Image mapping: PASS"
)


# ============================================================
# 9. DATASET
# ============================================================

IMAGE_SIZE = 224
MAX_IMAGES_PER_CASE = 8

MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32
)

STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32
)


class StructuredValDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_dict
    ):

        self.df = (
            dataframe
            .reset_index(drop=True)
        )

        self.image_dict = image_dict

    def __len__(self):

        return len(self.df)

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        case_id = row["case_id"]

        paths = self.image_dict[
            case_id
        ][:MAX_IMAGES_PER_CASE]

        images = []

        for path in paths:

            with Image.open(path) as img:

                img = img.convert("RGB")

                img = img.resize(
                    (
                        IMAGE_SIZE,
                        IMAGE_SIZE
                    ),
                    Image.BILINEAR
                )

                arr = (
                    np.asarray(
                        img,
                        dtype=np.float32
                    ) / 255.0
                )

            arr = (
                arr - MEAN
            ) / STD

            tensor = torch.from_numpy(
                arr
            ).permute(
                2, 0, 1
            ).float()

            images.append(
                tensor
            )

        return {
            "case_id": case_id,
            "images": torch.stack(images),
            "num_images": len(images),
        }


def collate_fn(batch):

    max_n = max(
        x["num_images"]
        for x in batch
    )

    B = len(batch)

    images = torch.zeros(
        B,
        max_n,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE,
        dtype=torch.float32
    )

    attention = torch.zeros(
        B,
        max_n,
        dtype=torch.long
    )

    case_ids = []

    for i, item in enumerate(batch):

        n = item["num_images"]

        images[i, :n] = (
            item["images"]
        )

        attention[i, :n] = 1

        case_ids.append(
            item["case_id"]
        )

    return {
        "case_id": case_ids,
        "images": images,
        "attention": attention,
    }


val_dataset = StructuredValDataset(
    val_df,
    images_by_case
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn
)

print(
    "Validation batches:",
    len(val_loader)
)


# ============================================================
# 10. EXACT V3 ARCHITECTURE
# ============================================================

class VisualProjector(nn.Module):

    def __init__(
        self,
        in_dim=768,
        out_dim=512
    ):

        super().__init__()

        self.proj = nn.Linear(
            in_dim,
            out_dim
        )

    def forward(self, x):

        return self.proj(x)


class V3Model(nn.Module):

    def __init__(
        self,
        vision,
        mt5,
        projector,
        concept_head,
        negative_head,
        attribute_head
    ):

        super().__init__()

        # IMPORTANT:
        # checkpoint uses "vision.*"
        self.vision = vision

        self.mt5 = mt5

        self.projector = projector

        self.concept_head = (
            concept_head
        )

        self.negative_head = (
            negative_head
        )

        self.attribute_head = (
            attribute_head
        )


# ============================================================
# 11. LOAD BASE MODEL
# ============================================================

print("\nLoading base models...")

vision = ViTModel.from_pretrained(
    "google/vit-base-patch16-224"
)

mt5 = MT5ForConditionalGeneration.from_pretrained(
    "google/mt5-small"
)

projector = VisualProjector(
    768,
    512
)

concept_head = nn.Linear(
    512,
    48
)

negative_head = nn.Linear(
    512,
    5
)

attribute_head = nn.Linear(
    512,
    6
)

model = V3Model(
    vision=vision,
    mt5=mt5,
    projector=projector,
    concept_head=concept_head,
    negative_head=negative_head,
    attribute_head=attribute_head
)


# ============================================================
# 12. LOAD V3 CHECKPOINT
# ============================================================

checkpoint = torch.load(
    CHECKPOINT_FILE,
    map_location="cpu",
    weights_only=False
)

state_dict = checkpoint[
    "model_state_dict"
]

print(
    "Checkpoint keys:",
    len(state_dict)
)

checkpoint_prefixes = sorted(
    set(
        k.split(".")[0]
        for k in state_dict.keys()
    )
)

print(
    "Checkpoint prefixes:",
    checkpoint_prefixes
)

model.load_state_dict(
    state_dict,
    strict=True
)

print(
    "\nCHECKPOINT LOAD: PASS"
)

model = model.to(device)
model.eval()


# ============================================================
# 13. STRUCTURED INFERENCE
# ============================================================

print("\n" + "=" * 70)
print("STRUCTURED VALIDATION INFERENCE")
print("=" * 70)

concept_prob_list = []
negative_prob_list = []
attribute_prob_list = []

case_ids = []


with torch.no_grad():

    for batch_idx, batch in enumerate(
        val_loader
    ):

        images = batch[
            "images"
        ].to(
            device,
            non_blocking=True
        )

        attention = batch[
            "attention"
        ].to(
            device,
            non_blocking=True
        )

        B, N, C, H, W = (
            images.shape
        )

        flat_images = images.reshape(
            B * N,
            C,
            H,
            W
        )

        # ----------------------------------------------------
        # ViT
        # ----------------------------------------------------

        if USE_BF16:

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16
            ):

                vision_output = (
                    model.vision(
                        pixel_values=flat_images
                    )
                )

                cls = (
                    vision_output
                    .last_hidden_state[:, 0]
                )

                cls = cls.reshape(
                    B,
                    N,
                    768
                )

                visual_tokens = (
                    model.projector(cls)
                )

                mask = (
                    attention
                    .unsqueeze(-1)
                    .to(
                        visual_tokens.dtype
                    )
                )

                pooled = (
                    visual_tokens * mask
                ).sum(dim=1)

                pooled = (
                    pooled
                    /
                    attention.sum(
                        dim=1,
                        keepdim=True
                    ).clamp(min=1)
                )

                # ------------------------------------------------
                # Structured heads
                # ------------------------------------------------

                concept_logits = (
                    model.concept_head(
                        pooled
                    )
                )

                negative_logits = (
                    model.negative_head(
                        pooled
                    )
                )

                attribute_logits = (
                    model.attribute_head(
                        pooled
                    )
                )

        else:

            vision_output = (
                model.vision(
                    pixel_values=flat_images
                )
            )

            cls = (
                vision_output
                .last_hidden_state[:, 0]
            )

            cls = cls.reshape(
                B,
                N,
                768
            )

            visual_tokens = (
                model.projector(cls)
            )

            mask = (
                attention
                .unsqueeze(-1)
                .to(
                    visual_tokens.dtype
                )
            )

            pooled = (
                visual_tokens * mask
            ).sum(dim=1)

            pooled = (
                pooled
                /
                attention.sum(
                    dim=1,
                    keepdim=True
                ).clamp(min=1)
            )

            concept_logits = (
                model.concept_head(
                    pooled
                )
            )

            negative_logits = (
                model.negative_head(
                    pooled
                )
            )

            attribute_logits = (
                model.attribute_head(
                    pooled
                )
            )

        # ----------------------------------------------------
        # Probabilities
        # ----------------------------------------------------

        concept_prob_list.append(
            torch.sigmoid(
                concept_logits
            )
            .float()
            .cpu()
            .numpy()
        )

        negative_prob_list.append(
            torch.sigmoid(
                negative_logits
            )
            .float()
            .cpu()
            .numpy()
        )

        attribute_prob_list.append(
            torch.sigmoid(
                attribute_logits
            )
            .float()
            .cpu()
            .numpy()
        )

        case_ids.extend(
            batch["case_id"]
        )

        if (
            batch_idx == 0
            or
            (batch_idx + 1) % 50 == 0
            or
            (batch_idx + 1) == len(val_loader)
        ):

            print(
                f"Batch "
                f"{batch_idx + 1}/"
                f"{len(val_loader)}"
            )


# ============================================================
# 14. CONCATENATE
# ============================================================

concept_probs = np.concatenate(
    concept_prob_list,
    axis=0
)

negative_probs = np.concatenate(
    negative_prob_list,
    axis=0
)

attribute_probs = np.concatenate(
    attribute_prob_list,
    axis=0
)

print("\nOutput shapes:")

print(
    "Concept:",
    concept_probs.shape
)

print(
    "Negative:",
    negative_probs.shape
)

print(
    "Attribute:",
    attribute_probs.shape
)

assert concept_probs.shape == (
    751,
    48
)

assert negative_probs.shape == (
    751,
    5
)

assert attribute_probs.shape == (
    751,
    6
)

assert len(case_ids) == 751

print(
    "\nInference: PASS"
)


# ============================================================
# 15. F1
# ============================================================

def binary_f1(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true
    ).astype(bool)

    y_pred = np.asarray(
        y_pred
    ).astype(bool)

    tp = np.logical_and(
        y_true,
        y_pred
    ).sum()

    fp = np.logical_and(
        ~y_true,
        y_pred
    ).sum()

    fn = np.logical_and(
        y_true,
        ~y_pred
    ).sum()

    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0.0
    )

    if precision + recall == 0:

        return 0.0

    return (
        2 * precision * recall
        / (precision + recall)
    )


# ============================================================
# 16. THRESHOLD CALIBRATION
# ============================================================

threshold_grid = np.arange(
    0.05,
    0.951,
    0.05
)


def calibrate_head(
    probs,
    targets
):

    rows = []

    for j in range(
        targets.shape[1]
    ):

        y_true = targets[:, j]

        best_f1 = -1.0
        best_threshold = 0.5

        best_pred_count = 0

        for threshold in (
            threshold_grid
        ):

            y_pred = (
                probs[:, j]
                >= threshold
            )

            f1 = binary_f1(
                y_true,
                y_pred
            )

            if f1 > best_f1:

                best_f1 = f1

                best_threshold = (
                    float(threshold)
                )

                best_pred_count = int(
                    y_pred.sum()
                )

        rows.append({
            "label_index": j,
            "best_threshold":
                best_threshold,
            "best_f1":
                float(best_f1),
            "ground_truth_positive":
                int(y_true.sum()),
            "predicted_positive":
                best_pred_count,
            "mean_probability":
                float(
                    probs[:, j].mean()
                ),
            "median_probability":
                float(
                    np.median(
                        probs[:, j]
                    )
                ),
        })

    return pd.DataFrame(rows)


concept_cal = calibrate_head(
    concept_probs,
    concept_targets
)

negative_cal = calibrate_head(
    negative_probs,
    negative_targets
)

attribute_cal = calibrate_head(
    attribute_probs,
    attribute_targets
)


# ============================================================
# 17. LABEL NAMES
# ============================================================

# 48 frozen concepts:
# these are the concept__ columns already represented
# in the V3 training manifest.

concept_names = [
    c.replace(
        "concept__",
        ""
    )
    for c in structured_all.columns
    if (
        c.startswith("concept__")
        and c in train_manifest.columns
    )
]

if len(concept_names) != 48:

    raise ValueError(
        "Expected 48 concept labels, "
        f"found {len(concept_names)}"
    )


negative_names = [
    c.replace(
        "negative__",
        ""
    )
    for c in structured_all.columns
    if c.startswith("negative__")
]

if len(negative_names) != 5:

    raise ValueError(
        "Expected 5 negative labels, "
        f"found {len(negative_names)}"
    )


attribute_names = [
    c.replace(
        "attribute__",
        ""
    )
    for c in structured_all.columns
    if c.startswith("attribute__")
]

if len(attribute_names) != 6:

    raise ValueError(
        "Expected 6 attribute labels, "
        f"found {len(attribute_names)}"
    )


concept_cal["label_name"] = (
    concept_names
)

negative_cal["label_name"] = (
    negative_names
)

attribute_cal["label_name"] = (
    attribute_names
)

concept_cal["head"] = "concept"
negative_cal["head"] = "negative"
attribute_cal["head"] = "attribute"


# ============================================================
# 18. COMBINE
# ============================================================

threshold_table = pd.concat(
    [
        concept_cal,
        negative_cal,
        attribute_cal
    ],
    ignore_index=True
)


# ============================================================
# 19. MACRO + MICRO F1
# ============================================================

def thresholded_predictions(
    probs,
    calibration
):

    pred = np.zeros_like(
        probs,
        dtype=bool
    )

    for j in range(
        probs.shape[1]
    ):

        threshold = float(
            calibration.iloc[j][
                "best_threshold"
            ]
        )

        pred[:, j] = (
            probs[:, j]
            >= threshold
        )

    return pred


concept_pred = (
    thresholded_predictions(
        concept_probs,
        concept_cal
    )
)

negative_pred = (
    thresholded_predictions(
        negative_probs,
        negative_cal
    )
)

attribute_pred = (
    thresholded_predictions(
        attribute_probs,
        attribute_cal
    )
)


concept_micro = binary_f1(
    concept_targets.reshape(-1),
    concept_pred.reshape(-1)
)

negative_micro = binary_f1(
    negative_targets.reshape(-1),
    negative_pred.reshape(-1)
)

attribute_micro = binary_f1(
    attribute_targets.reshape(-1),
    attribute_pred.reshape(-1)
)

concept_macro = (
    concept_cal["best_f1"].mean()
)

negative_macro = (
    negative_cal["best_f1"].mean()
)

attribute_macro = (
    attribute_cal["best_f1"].mean()
)


# ============================================================
# 20. SAVE CSV
# ============================================================

threshold_csv = os.path.join(
    OUTPUT_DIR,
    "validation_best_thresholds.csv"
)

threshold_table.to_csv(
    threshold_csv,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 21. SAVE JSON
# ============================================================

threshold_json = {

    "ontology":
        "V3-Ontology-V2.2",

    "validation_total_cases":
        756,

    "validation_structured_cases":
        751,

    "test_used":
        False,

    "concept": {
        row["label_name"]:
            float(
                row["best_threshold"]
            )
        for _, row
        in concept_cal.iterrows()
    },

    "negative": {
        row["label_name"]:
            float(
                row["best_threshold"]
            )
        for _, row
        in negative_cal.iterrows()
    },

    "attribute": {
        row["label_name"]:
            float(
                row["best_threshold"]
            )
        for _, row
        in attribute_cal.iterrows()
    }
}

threshold_json_file = os.path.join(
    OUTPUT_DIR,
    "validation_best_thresholds.json"
)

with open(
    threshold_json_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        threshold_json,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 22. SAVE SUMMARY
# ============================================================

summary = {

    "ontology":
        "V3-Ontology-V2.2",

    "validation_total":
        756,

    "validation_structured":
        751,

    "test_used":
        False,

    "concept_macro_f1":
        float(concept_macro),

    "concept_micro_f1":
        float(concept_micro),

    "negative_macro_f1":
        float(negative_macro),

    "negative_micro_f1":
        float(negative_micro),

    "attribute_macro_f1":
        float(attribute_macro),

    "attribute_micro_f1":
        float(attribute_micro),

    "threshold_csv":
        threshold_csv,

    "threshold_json":
        threshold_json_file,
}

summary_file = os.path.join(
    OUTPUT_DIR,
    "validation_threshold_calibration_summary.json"
)

with open(
    summary_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 23. FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 70)
print("V3-6 COMPLETE")
print("=" * 70)

print(
    "\nValidation cases:",
    756
)

print(
    "Structured cases:",
    751
)

print(
    "\nCONCEPT"
)

print(
    f"  Macro F1 = {concept_macro:.6f}"
)

print(
    f"  Micro F1 = {concept_micro:.6f}"
)

print(
    "\nNEGATIVE"
)

print(
    f"  Macro F1 = {negative_macro:.6f}"
)

print(
    f"  Micro F1 = {negative_micro:.6f}"
)

print(
    "\nATTRIBUTE"
)

print(
    f"  Macro F1 = {attribute_macro:.6f}"
)

print(
    f"  Micro F1 = {attribute_micro:.6f}"
)

print(
    "\nTop concept labels:"
)

display(
    concept_cal
    .sort_values(
        "best_f1",
        ascending=False
    )
    .head(15)
)

print(
    "\nNegative labels:"
)

display(
    negative_cal
    .sort_values(
        "best_f1",
        ascending=False
    )
)

print(
    "\nAttribute labels:"
)

display(
    attribute_cal
    .sort_values(
        "best_f1",
        ascending=False
    )
)

print(
    "\nSaved:"
)

print(
    threshold_csv
)

print(
    threshold_json_file
)

print(
    summary_file
)

print(
    "\nTEST SET ACCESSED: NO"
)

print(
    "\nSTATUS: PASS"
)

print("=" * 70)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FILE CHECK
training_manifest: FOUND
image_manifest: FOUND
structured_targets: FOUND
config: FOUND
checkpoint: FOUND

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.49 GB
BF16: True

Training manifest: (7606, 15)
Image manifest: (76405, 26)
Structured targets: (7606, 102)

Validation total: 756
Validation structured: 751
Validation report-only: 5

Target shapes:
Concept: (751, 48)
Negative: (751, 5)
Attribute: (751, 6)

Validation NORMAL images: 7564
Image mapping: PASS
Validation batches: 188

Loading base models...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Checkpoint keys: 400
Checkpoint prefixes: ['attribute_head', 'concept_head', 'mt5', 'negative_head', 'projector', 'vision']

CHECKPOINT LOAD: PASS

STRUCTURED VALIDATION INFERENCE
Batch 1/188
Batch 50/188
Batch 100/188
Batch 150/188
Batch 188/188

Output shapes:
Concept: (751, 48)
Negative: (751, 5)
Attribute: (751, 6)

Inference: PASS


ValueError: Expected 48 concept labels, found 0

In [12]:
# ============================================================
# V3-6 CONTINUE — RECOVER EXACT LABEL ORDER + CALIBRATE
# No model/inference rerun.
# ============================================================

import os
import json
import numpy as np
import pandas as pd


# ============================================================
# 1. SANITY CHECK EXISTING INFERENCE VARIABLES
# ============================================================

required_vars = [
    "concept_probs",
    "negative_probs",
    "attribute_probs",
    "concept_targets",
    "negative_targets",
    "attribute_targets",
    "structured_all",
    "OUTPUT_DIR",
]

missing_vars = [
    v for v in required_vars
    if v not in globals()
]

if missing_vars:
    raise RuntimeError(
        "Missing variables from previous V3-6 run: "
        + str(missing_vars)
        + "\n"
        "Do NOT continue. Rerun the complete V3-6 cell."
    )

print("Existing inference variables: PASS")

print(
    "concept_probs:",
    concept_probs.shape
)

print(
    "negative_probs:",
    negative_probs.shape
)

print(
    "attribute_probs:",
    attribute_probs.shape
)


# ============================================================
# 2. EXACT LABEL RECOVERY
#
# We recover the frozen vector ordering by comparing each
# vector dimension against every binary concept column in
# v3_structured_targets_v2_2.csv.
#
# This avoids guessing the order.
# ============================================================

# Use ALL 7606 cases for matching, not only validation.
# We need a stable case_id alignment.

structured_lookup = (
    structured_all
    .set_index("case_id")
)


# ------------------------------------------------------------
# Recover concept labels
# ------------------------------------------------------------

concept_candidate_cols = [
    c
    for c in structured_all.columns
    if c.startswith("concept__")
]

print(
    "\nConcept candidate columns:",
    len(concept_candidate_cols)
)

# Parse the frozen 48-dimensional vector for ALL cases.
train_vectors = (
    train_manifest[
        [
            "case_id",
            "concept_vector_str",
        ]
    ]
    .copy()
)

def parse_vector(x):

    if pd.isna(x):
        return []

    x = str(x).strip()

    if not x:
        return []

    return [
        int(float(v))
        for v in x.split(",")
    ]


all_concept_vectors = np.stack(
    train_vectors[
        "concept_vector_str"
    ]
    .apply(parse_vector)
    .values
)

assert all_concept_vectors.shape[1] == 48

# Align structured rows to exactly the same case order.
aligned_structured = (
    structured_lookup
    .loc[
        train_vectors["case_id"]
    ]
)

assert len(aligned_structured) == len(
    train_vectors
)


# ------------------------------------------------------------
# Compare every vector dimension against every candidate
# binary column.
# ------------------------------------------------------------

concept_matches = {}

for dim in range(48):

    target_vector = (
        all_concept_vectors[:, dim]
    )

    matches = []

    for col in concept_candidate_cols:

        candidate = (
            pd.to_numeric(
                aligned_structured[col],
                errors="coerce"
            )
            .fillna(0)
            .astype(int)
            .to_numpy()
        )

        if np.array_equal(
            target_vector,
            candidate
        ):
            matches.append(col)

    concept_matches[dim] = matches


# ------------------------------------------------------------
# Check whether every dimension has exactly one match.
# ------------------------------------------------------------

ambiguous = {
    dim: matches
    for dim, matches
    in concept_matches.items()
    if len(matches) != 1
}

if ambiguous:

    print(
        "\nWARNING: Concept dimensions with "
        "non-unique matches:"
    )

    for dim, matches in ambiguous.items():

        print(
            f"  dim {dim}: {matches}"
        )

    raise ValueError(
        "Could not uniquely recover "
        "the frozen concept vocabulary."
    )


concept_names = [
    concept_matches[i][0]
    .replace(
        "concept__",
        ""
    )
    for i in range(48)
]

print(
    "\nRecovered 48 concept labels: PASS"
)

for i, name in enumerate(
    concept_names
):

    print(
        f"{i:2d}: {name}"
    )


# ============================================================
# 3. RECOVER NEGATIVE LABELS
# ============================================================

negative_candidate_cols = [
    c
    for c in structured_all.columns
    if c.startswith("negative__")
]

print(
    "\nNegative candidate columns:",
    len(negative_candidate_cols)
)

if len(negative_candidate_cols) != 5:

    raise ValueError(
        "Expected 5 negative columns, "
        f"found {len(negative_candidate_cols)}"
    )

negative_names = [
    c.replace(
        "negative__",
        ""
    )
    for c in negative_candidate_cols
]

print(
    "Negative labels:"
)

for i, name in enumerate(
    negative_names
):

    print(
        f"{i}: {name}"
    )


# ============================================================
# 4. RECOVER ATTRIBUTE LABELS
# ============================================================

attribute_candidate_cols = [
    c
    for c in structured_all.columns
    if c.startswith("attribute__")
]

print(
    "\nAttribute candidate columns:",
    len(attribute_candidate_cols)
)

if len(attribute_candidate_cols) != 6:

    raise ValueError(
        "Expected 6 attribute columns, "
        f"found {len(attribute_candidate_cols)}"
    )

attribute_names = [
    c.replace(
        "attribute__",
        ""
    )
    for c in attribute_candidate_cols
]

print(
    "Attribute labels:"
)

for i, name in enumerate(
    attribute_names
):

    print(
        f"{i}: {name}"
    )


# ============================================================
# 5. F1 FUNCTION
# ============================================================

def binary_f1(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true
    ).astype(bool)

    y_pred = np.asarray(
        y_pred
    ).astype(bool)

    tp = np.logical_and(
        y_true,
        y_pred
    ).sum()

    fp = np.logical_and(
        ~y_true,
        y_pred
    ).sum()

    fn = np.logical_and(
        y_true,
        ~y_pred
    ).sum()

    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0.0
    )

    if precision + recall == 0:

        return 0.0

    return (
        2 * precision * recall
        / (precision + recall)
    )


# ============================================================
# 6. THRESHOLD GRID
# ============================================================

threshold_grid = np.arange(
    0.05,
    0.951,
    0.05
)


# ============================================================
# 7. PER-LABEL THRESHOLD CALIBRATION
# ============================================================

def calibrate_head(
    probs,
    targets
):

    rows = []

    for j in range(
        targets.shape[1]
    ):

        y_true = targets[:, j]

        best_f1 = -1.0
        best_threshold = 0.5
        best_pred_count = 0

        for threshold in (
            threshold_grid
        ):

            y_pred = (
                probs[:, j]
                >= threshold
            )

            f1 = binary_f1(
                y_true,
                y_pred
            )

            if f1 > best_f1:

                best_f1 = f1

                best_threshold = (
                    float(threshold)
                )

                best_pred_count = int(
                    y_pred.sum()
                )

        rows.append({

            "label_index": j,

            "best_threshold":
                best_threshold,

            "best_f1":
                float(best_f1),

            "ground_truth_positive":
                int(y_true.sum()),

            "predicted_positive":
                best_pred_count,

            "mean_probability":
                float(
                    probs[:, j].mean()
                ),

            "median_probability":
                float(
                    np.median(
                        probs[:, j]
                    )
                ),
        })

    return pd.DataFrame(rows)


concept_cal = calibrate_head(
    concept_probs,
    concept_targets
)

negative_cal = calibrate_head(
    negative_probs,
    negative_targets
)

attribute_cal = calibrate_head(
    attribute_probs,
    attribute_targets
)


# ============================================================
# 8. ATTACH LABEL NAMES
# ============================================================

concept_cal["label_name"] = [
    concept_names[i]
    for i in concept_cal[
        "label_index"
    ]
]

concept_cal["head"] = "concept"


negative_cal["label_name"] = [
    negative_names[i]
    for i in negative_cal[
        "label_index"
    ]
]

negative_cal["head"] = "negative"


attribute_cal["label_name"] = [
    attribute_names[i]
    for i in attribute_cal[
        "label_index"
    ]
]

attribute_cal["head"] = "attribute"


# ============================================================
# 9. COMBINE
# ============================================================

threshold_table = pd.concat(
    [
        concept_cal,
        negative_cal,
        attribute_cal,
    ],
    ignore_index=True
)


# ============================================================
# 10. MICRO F1 USING SELECTED THRESHOLDS
# ============================================================

def make_predictions(
    probs,
    calibration
):

    pred = np.zeros_like(
        probs,
        dtype=bool
    )

    for j in range(
        probs.shape[1]
    ):

        threshold = float(
            calibration.iloc[j][
                "best_threshold"
            ]
        )

        pred[:, j] = (
            probs[:, j]
            >= threshold
        )

    return pred


concept_pred = make_predictions(
    concept_probs,
    concept_cal
)

negative_pred = make_predictions(
    negative_probs,
    negative_cal
)

attribute_pred = make_predictions(
    attribute_probs,
    attribute_cal
)


concept_micro = binary_f1(
    concept_targets.reshape(-1),
    concept_pred.reshape(-1)
)

negative_micro = binary_f1(
    negative_targets.reshape(-1),
    negative_pred.reshape(-1)
)

attribute_micro = binary_f1(
    attribute_targets.reshape(-1),
    attribute_pred.reshape(-1)
)


concept_macro = (
    concept_cal["best_f1"].mean()
)

negative_macro = (
    negative_cal["best_f1"].mean()
)

attribute_macro = (
    attribute_cal["best_f1"].mean()
)


# ============================================================
# 11. SAVE CSV
# ============================================================

threshold_csv = os.path.join(
    OUTPUT_DIR,
    "validation_best_thresholds.csv"
)

threshold_table.to_csv(
    threshold_csv,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 12. SAVE JSON
# ============================================================

threshold_json = {

    "ontology":
        "V3-Ontology-V2.2",

    "calibration_split":
        "validation",

    "validation_total_cases":
        756,

    "validation_structured_cases":
        751,

    "test_used":
        False,

    "concept": {
        row["label_name"]:
            float(
                row["best_threshold"]
            )
        for _, row
        in concept_cal.iterrows()
    },

    "negative": {
        row["label_name"]:
            float(
                row["best_threshold"]
            )
        for _, row
        in negative_cal.iterrows()
    },

    "attribute": {
        row["label_name"]:
            float(
                row["best_threshold"]
            )
        for _, row
        in attribute_cal.iterrows()
    },
}

threshold_json_file = os.path.join(
    OUTPUT_DIR,
    "validation_best_thresholds.json"
)

with open(
    threshold_json_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        threshold_json,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 13. SUMMARY
# ============================================================

summary = {

    "ontology":
        "V3-Ontology-V2.2",

    "validation_total":
        756,

    "validation_structured":
        751,

    "test_used":
        False,

    "concept_macro_f1":
        float(concept_macro),

    "concept_micro_f1":
        float(concept_micro),

    "negative_macro_f1":
        float(negative_macro),

    "negative_micro_f1":
        float(negative_micro),

    "attribute_macro_f1":
        float(attribute_macro),

    "attribute_micro_f1":
        float(attribute_micro),

    "threshold_csv":
        threshold_csv,

    "threshold_json":
        threshold_json_file,
}

summary_file = os.path.join(
    OUTPUT_DIR,
    "validation_threshold_calibration_summary.json"
)

with open(
    summary_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 14. DISPLAY RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("V3-6 VALIDATION CALIBRATION COMPLETE")
print("=" * 70)

print(
    f"\nConcept  — "
    f"Macro F1: {concept_macro:.6f} | "
    f"Micro F1: {concept_micro:.6f}"
)

print(
    f"Negative — "
    f"Macro F1: {negative_macro:.6f} | "
    f"Micro F1: {negative_micro:.6f}"
)

print(
    f"Attribute — "
    f"Macro F1: {attribute_macro:.6f} | "
    f"Micro F1: {attribute_micro:.6f}"
)


print(
    "\n" + "-" * 70
)

print(
    "TOP CONCEPT LABELS"
)

display(
    concept_cal
    .sort_values(
        "best_f1",
        ascending=False
    )
    .head(15)
)


print(
    "\n" + "-" * 70
)

print(
    "NEGATIVE LABELS"
)

display(
    negative_cal
    .sort_values(
        "best_f1",
        ascending=False
    )
)


print(
    "\n" + "-" * 70
)

print(
    "ATTRIBUTE LABELS"
)

display(
    attribute_cal
    .sort_values(
        "best_f1",
        ascending=False
    )
)


print(
    "\nSaved:"
)

print(
    threshold_csv
)

print(
    threshold_json_file
)

print(
    summary_file
)

print(
    "\nTEST SET ACCESSED: NO"
)

print(
    "V3-6 STATUS: PASS"
)

print("=" * 70)

Existing inference variables: PASS
concept_probs: (751, 48)
negative_probs: (751, 5)
attribute_probs: (751, 6)

Concept candidate columns: 54

Recovered 48 concept labels: PASS
 0: amidan_qua_phat
 1: chan_thuong_mang_nhi
 2: chan_thuong_ong_tai_ngoai
 3: chay_mau_mui
 4: di_vat_hong
 5: di_vat_hong_thanh_quan
 6: di_vat_tai
 7: dich_vat_mui
 8: hat_day_thanh
 9: hau_phau_mui_xoang
10: hau_phau_va_nhi
11: hep_ong_tai_ngoai
12: hoc_xuong_ca
13: lech_vach_ngan
14: liet_day_thanh
15: nang_day_thanh
16: nhot_ong_tai_ngoai
17: not_vanh_tai
18: polyp_day_thanh
19: polyp_mui
20: polyp_ong_tai_ngoai
21: qua_phat_va
22: ray_tai
23: ro_luan_nhi
24: seo_hoc_mui
25: theo_doi_trao_nguoc
26: thung_mang_nhi
27: tien_dinh_mui
28: tu_dich_vanh_tai
29: u_hoc_mui
30: u_nhu_cuon_mui
31: u_nhu_hoc_mui
32: u_xoang
33: viem_amidan
34: viem_hong
35: viem_luoi
36: viem_mang_nhi
37: viem_mieng
38: viem_mui
39: viem_mui_xoang
40: viem_ong_tai_ngoai
41: viem_tai_giua
42: viem_tai_xuong_chum
43: viem_thanh_quan
44

,label_index,best_threshold,best_f1,ground_truth_positive,predicted_positive,mean_probability,median_probability,label_name,head
25,25,0.30,0.763636,42,68,0.066396,0.002258,theo_doi_trao_nguoc,concept
34,34,0.55,0.747826,48,67,0.080045,0.008301,viem_hong,concept
38,38,0.35,0.721939,321,463,0.423054,0.472656,viem_mui,concept
40,40,0.25,0.508361,111,188,0.180421,0.087402,viem_ong_tai_ngoai,concept
41,41,0.20,0.453258,140,213,0.156651,0.114258,viem_tai_giua,concept
39,39,0.20,0.427966,146,326,0.172561,0.181641,viem_mui_xoang,concept
22,22,0.20,0.266667,21,24,0.043250,0.019775,ray_tai,concept
43,43,0.10,0.250000,11,21,0.008583,0.000938,viem_thanh_quan,concept
19,19,0.05,0.205405,38,147,0.032237,0.026733,polyp_mui,concept
44,44,0.05,0.196319,24,139,0.033410,0.027954,viem_va,concept



----------------------------------------------------------------------
NEGATIVE LABELS


,label_index,best_threshold,best_f1,ground_truth_positive,predicted_positive,mean_probability,median_probability,label_name,head
1,1,0.10,0.221344,44,209,0.081430,0.061768,no_abnormal_nose_sinus,negative
3,3,0.05,0.153846,15,24,0.018757,0.015442,no_bleeding,negative
0,0,0.05,0.000000,0,0,0.000217,0.000203,no_abnormal_external_middle_ear,negative
2,2,0.05,0.000000,13,9,0.021922,0.021240,no_abnormal_ent,negative
4,4,0.05,0.000000,6,2,0.004097,0.001282,no_foreign_body,negative



----------------------------------------------------------------------
ATTRIBUTE LABELS


,label_index,best_threshold,best_f1,ground_truth_positive,predicted_positive,mean_probability,median_probability,label_name,head
2,2,0.25,0.590501,280,499,0.342386,0.312500,right,attribute
5,5,0.20,0.507692,134,256,0.193104,0.130859,post_surgery,attribute
3,3,0.20,0.463636,171,269,0.199061,0.147461,left,attribute
0,0,0.20,0.441426,185,404,0.241781,0.208984,acute,attribute
1,1,0.10,0.317757,72,249,0.095169,0.068359,chronic,attribute
4,4,0.05,0.242915,56,191,0.040829,0.039062,bilateral,attribute



Saved:
/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v3/structured_diagnostic/validation_best_thresholds.csv
/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v3/structured_diagnostic/validation_best_thresholds.json
/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v3/structured_diagnostic/validation_threshold_calibration_summary.json

TEST SET ACCESSED: NO
V3-6 STATUS: PASS


In [4]:
# ============================================================
# V3-7 — FINAL TEST STRUCTURED EVALUATION — FIXED
# ============================================================

import os
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import ViTModel, MT5ForConditionalGeneration


# ============================================================
# 1. SEED + DRIVE
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)


# ============================================================
# 2. PATHS
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/NoiSoi_Matching"

V3_DIR = os.path.join(
    PROJECT_DIR,
    "baseline_model_v3"
)

TRAINING_MANIFEST_FILE = os.path.join(
    V3_DIR,
    "v3_training_manifest.csv"
)

IMAGE_MANIFEST_FILE = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

STRUCTURED_TARGET_FILE = os.path.join(
    PROJECT_DIR,
    "v3_ontology_v2",
    "v3_structured_targets_v2_2.csv"
)

CONFIG_FILE = os.path.join(
    V3_DIR,
    "v3_config.json"
)

CHECKPOINT_FILE = os.path.join(
    V3_DIR,
    "checkpoints",
    "best.pt"
)

THRESHOLD_FILE = os.path.join(
    V3_DIR,
    "structured_diagnostic",
    "validation_best_thresholds.json"
)

OUTPUT_DIR = os.path.join(
    V3_DIR,
    "structured_test_evaluation"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 3. CHECK FILES
# ============================================================

required = {
    "training_manifest": TRAINING_MANIFEST_FILE,
    "image_manifest": IMAGE_MANIFEST_FILE,
    "structured_targets": STRUCTURED_TARGET_FILE,
    "config": CONFIG_FILE,
    "checkpoint": CHECKPOINT_FILE,
    "validation_thresholds": THRESHOLD_FILE,
}

for name, path in required.items():

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

    print(
        f"{name}: FOUND"
    )


# ============================================================
# 4. DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

USE_BF16 = (
    device.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

print("\nDevice:", device)

if device.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print(
    "BF16:",
    USE_BF16
)


# ============================================================
# 5. LOAD DATA
# ============================================================

train_manifest = pd.read_csv(
    TRAINING_MANIFEST_FILE,
    low_memory=False
)

image_manifest = pd.read_csv(
    IMAGE_MANIFEST_FILE,
    low_memory=False
)

structured_all = pd.read_csv(
    STRUCTURED_TARGET_FILE,
    low_memory=False
)

with open(
    THRESHOLD_FILE,
    "r",
    encoding="utf-8"
) as f:
    frozen_thresholds = json.load(f)

print(
    "\nTraining manifest:",
    train_manifest.shape
)

print(
    "Image manifest:",
    image_manifest.shape
)

print(
    "Structured target:",
    structured_all.shape
)


# ============================================================
# 6. VERIFY THRESHOLDS
# ============================================================

assert (
    frozen_thresholds["ontology"]
    == "V3-Ontology-V2.2"
)

assert (
    frozen_thresholds["calibration_split"]
    == "validation"
)

assert (
    frozen_thresholds["test_used"]
    is False
)

print(
    "Frozen validation thresholds: PASS"
)


# ============================================================
# 7. TEST CASES
# ============================================================

test_all = train_manifest[
    train_manifest["split"] == "test"
].copy()

test_df = test_all[
    test_all["structured_loss_mask"] == 1
].copy()

test_df = test_df.reset_index(
    drop=True
)

print("\nTEST CASES")
print(
    "Total:",
    len(test_all)
)

print(
    "Structured:",
    len(test_df)
)

print(
    "Report-only:",
    len(test_all) - len(test_df)
)

assert len(test_all) == 712
assert len(test_df) == 709


# ============================================================
# 8. PARSE VECTORS
# ============================================================

def parse_vector(x):

    if pd.isna(x):
        return []

    x = str(x).strip()

    if not x:
        return []

    return [
        int(float(v))
        for v in x.split(",")
    ]


concept_targets = np.stack(
    test_df[
        "concept_vector_str"
    ].apply(parse_vector).values
)

negative_targets = np.stack(
    test_df[
        "negative_vector_str"
    ].apply(parse_vector).values
)

attribute_targets = np.stack(
    test_df[
        "attribute_vector_str"
    ].apply(parse_vector).values
)

assert concept_targets.shape == (709, 48)
assert negative_targets.shape == (709, 5)
assert attribute_targets.shape == (709, 6)

print(
    "\nTarget shapes:",
    concept_targets.shape,
    negative_targets.shape,
    attribute_targets.shape
)


# ============================================================
# 9. IMAGE INDEX
# ============================================================

normal_images = image_manifest[
    image_manifest["image_status"] == "NORMAL"
].copy()

normal_images = normal_images[
    normal_images["case_id"].isin(
        test_df["case_id"]
    )
].copy()

images_by_case = (
    normal_images
    .sort_values(
        ["case_id", "image_path"]
    )
    .groupby("case_id")["image_path"]
    .apply(list)
    .to_dict()
)

missing = [
    cid
    for cid in test_df["case_id"]
    if cid not in images_by_case
]

if missing:
    raise ValueError(
        f"Cases without NORMAL images: {missing[:10]}"
    )

print(
    "Test NORMAL images:",
    len(normal_images)
)


# ============================================================
# 10. DATASET
# ============================================================

IMAGE_SIZE = 224
MAX_IMAGES_PER_CASE = 8

MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32
)

STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32
)


class StructuredTestDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_dict
    ):
        self.df = dataframe.reset_index(
            drop=True
        )
        self.image_dict = image_dict

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        case_id = row["case_id"]

        paths = self.image_dict[
            case_id
        ][:MAX_IMAGES_PER_CASE]

        images = []

        for path in paths:

            with Image.open(path) as img:

                img = img.convert("RGB")

                img = img.resize(
                    (IMAGE_SIZE, IMAGE_SIZE),
                    Image.BILINEAR
                )

                arr = (
                    np.asarray(
                        img,
                        dtype=np.float32
                    ) / 255.0
                )

            arr = (
                arr - MEAN
            ) / STD

            tensor = (
                torch.from_numpy(arr)
                .permute(2, 0, 1)
                .float()
            )

            images.append(tensor)

        return {
            "case_id": case_id,
            "images": torch.stack(images),
            "num_images": len(images),
        }


def collate_fn(batch):

    max_n = max(
        item["num_images"]
        for item in batch
    )

    B = len(batch)

    images = torch.zeros(
        B,
        max_n,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE
    )

    attention = torch.zeros(
        B,
        max_n,
        dtype=torch.long
    )

    case_ids = []

    for i, item in enumerate(batch):

        n = item["num_images"]

        images[i, :n] = item["images"]

        attention[i, :n] = 1

        case_ids.append(
            item["case_id"]
        )

    return {
        "case_id": case_ids,
        "images": images,
        "attention": attention,
    }


test_dataset = StructuredTestDataset(
    test_df,
    images_by_case
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn
)

print(
    "Test batches:",
    len(test_loader)
)


# ============================================================
# 11. EXACT V3 ARCHITECTURE
# ============================================================

class VisualProjector(nn.Module):

    def __init__(
        self,
        in_dim=768,
        out_dim=512
    ):
        super().__init__()

        self.proj = nn.Linear(
            in_dim,
            out_dim
        )

    def forward(self, x):

        return self.proj(x)


class V3Model(nn.Module):

    def __init__(self):

        super().__init__()

        # IMPORTANT:
        # checkpoint uses prefix "vision.*"
        self.vision = ViTModel.from_pretrained(
            "google/vit-base-patch16-224"
        )

        self.mt5 = MT5ForConditionalGeneration.from_pretrained(
            "google/mt5-small"
        )

        self.projector = VisualProjector(
            768,
            512
        )

        self.concept_head = nn.Linear(
            512,
            48
        )

        self.negative_head = nn.Linear(
            512,
            5
        )

        self.attribute_head = nn.Linear(
            512,
            6
        )


print(
    "\nLoading V3 model..."
)

model = V3Model()

checkpoint = torch.load(
    CHECKPOINT_FILE,
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.to(device)

model.eval()

print(
    "Checkpoint load: PASS"
)


# ============================================================
# 12. INFERENCE
# ============================================================

concept_prob_list = []
negative_prob_list = []
attribute_prob_list = []
case_ids = []

print("\n" + "=" * 70)
print("RUNNING FINAL TEST INFERENCE")
print("=" * 70)

with torch.no_grad():

    for batch_idx, batch in enumerate(
        test_loader
    ):

        images = batch[
            "images"
        ].to(
            device,
            non_blocking=True
        )

        attention = batch[
            "attention"
        ].to(
            device,
            non_blocking=True
        )

        B, N, C, H, W = images.shape

        flat_images = images.reshape(
            B * N,
            C,
            H,
            W
        )

        # ----------------------------------------------------
        # Vision
        # ----------------------------------------------------

        if USE_BF16:

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16
            ):

                vision_output = model.vision(
                    pixel_values=flat_images
                )

                cls = (
                    vision_output
                    .last_hidden_state[:, 0]
                )

                cls = cls.reshape(
                    B,
                    N,
                    768
                )

                visual_tokens = model.projector(
                    cls
                )

                mask = (
                    attention
                    .unsqueeze(-1)
                    .to(visual_tokens.dtype)
                )

                pooled = (
                    visual_tokens * mask
                ).sum(dim=1)

                denom = (
                    attention
                    .sum(
                        dim=1,
                        keepdim=True
                    )
                    .clamp(min=1)
                )

                pooled = (
                    pooled / denom
                )

                concept_logits = model.concept_head(
                    pooled
                )

                negative_logits = model.negative_head(
                    pooled
                )

                attribute_logits = model.attribute_head(
                    pooled
                )

        else:

            vision_output = model.vision(
                pixel_values=flat_images
            )

            cls = (
                vision_output
                .last_hidden_state[:, 0]
            )

            cls = cls.reshape(
                B,
                N,
                768
            )

            visual_tokens = model.projector(
                cls
            )

            mask = (
                attention
                .unsqueeze(-1)
                .to(visual_tokens.dtype)
            )

            pooled = (
                visual_tokens * mask
            ).sum(dim=1)

            denom = (
                attention
                .sum(
                    dim=1,
                    keepdim=True
                )
                .clamp(min=1)
            )

            pooled = (
                pooled / denom
            )

            concept_logits = model.concept_head(
                pooled
            )

            negative_logits = model.negative_head(
                pooled
            )

            attribute_logits = model.attribute_head(
                pooled
            )

        # ----------------------------------------------------
        # Probabilities
        # ----------------------------------------------------

        concept_prob_list.append(
            torch.sigmoid(
                concept_logits
            )
            .float()
            .cpu()
            .numpy()
        )

        negative_prob_list.append(
            torch.sigmoid(
                negative_logits
            )
            .float()
            .cpu()
            .numpy()
        )

        attribute_prob_list.append(
            torch.sigmoid(
                attribute_logits
            )
            .float()
            .cpu()
            .numpy()
        )

        case_ids.extend(
            batch["case_id"]
        )

        if (
            batch_idx == 0
            or (batch_idx + 1) % 50 == 0
            or batch_idx + 1 == len(test_loader)
        ):

            print(
                f"Batch {batch_idx + 1}/"
                f"{len(test_loader)}"
            )


# ============================================================
# 13. CONCATENATE
# ============================================================

concept_probs = np.concatenate(
    concept_prob_list,
    axis=0
)

negative_probs = np.concatenate(
    negative_prob_list,
    axis=0
)

attribute_probs = np.concatenate(
    attribute_prob_list,
    axis=0
)

assert concept_probs.shape == (709, 48)
assert negative_probs.shape == (709, 5)
assert attribute_probs.shape == (709, 6)

print(
    "\nInference: PASS"
)


# ============================================================
# 14. RECOVER LABEL ORDER
# ============================================================

train_case_order = (
    train_manifest["case_id"].tolist()
)

train_vectors = (
    train_manifest[
        "concept_vector_str"
    ]
    .apply(parse_vector)
)

all_concept_vectors = np.stack(
    train_vectors.values
)

structured_lookup = (
    structured_all
    .set_index("case_id")
)

concept_candidate_cols = [
    c
    for c in structured_all.columns
    if c.startswith("concept__")
]

concept_names = []

for dim in range(48):

    target_vector = (
        all_concept_vectors[:, dim]
    )

    matches = []

    for col in concept_candidate_cols:

        candidate = (
            pd.to_numeric(
                structured_lookup.loc[
                    train_case_order,
                    col
                ],
                errors="coerce"
            )
            .fillna(0)
            .astype(int)
            .to_numpy()
        )

        if np.array_equal(
            target_vector,
            candidate
        ):
            matches.append(
                col.replace(
                    "concept__",
                    ""
                )
            )

    if len(matches) != 1:

        raise ValueError(
            f"Cannot recover label "
            f"for dimension {dim}: "
            f"{matches}"
        )

    concept_names.append(
        matches[0]
    )

negative_names = [
    c.replace(
        "negative__",
        ""
    )
    for c in structured_all.columns
    if c.startswith("negative__")
]

attribute_names = [
    c.replace(
        "attribute__",
        ""
    )
    for c in structured_all.columns
    if c.startswith("attribute__")
]

assert len(concept_names) == 48
assert len(negative_names) == 5
assert len(attribute_names) == 6

print(
    "\nRecovered labels: PASS"
)


# ============================================================
# 15. FROZEN VALIDATION THRESHOLDS
# ============================================================

concept_thresholds = np.array([
    frozen_thresholds[
        "concept"
    ][name]
    for name in concept_names
])

negative_thresholds = np.array([
    frozen_thresholds[
        "negative"
    ][name]
    for name in negative_names
])

attribute_thresholds = np.array([
    frozen_thresholds[
        "attribute"
    ][name]
    for name in attribute_names
])


concept_pred = (
    concept_probs
    >= concept_thresholds[None, :]
)

negative_pred = (
    negative_probs
    >= negative_thresholds[None, :]
)

attribute_pred = (
    attribute_probs
    >= attribute_thresholds[None, :]
)


# ============================================================
# 16. METRICS
# ============================================================

def binary_f1(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true
    ).astype(bool)

    y_pred = np.asarray(
        y_pred
    ).astype(bool)

    tp = np.logical_and(
        y_true,
        y_pred
    ).sum()

    fp = np.logical_and(
        ~y_true,
        y_pred
    ).sum()

    fn = np.logical_and(
        y_true,
        ~y_pred
    ).sum()

    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0.0
    )

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        / (precision + recall)
    )


def per_label_metrics(
    targets,
    predictions,
    names,
    thresholds
):

    rows = []

    for j, name in enumerate(names):

        y_true = targets[:, j]
        y_pred = predictions[:, j]

        tp = np.logical_and(
            y_true == 1,
            y_pred == 1
        ).sum()

        fp = np.logical_and(
            y_true == 0,
            y_pred == 1
        ).sum()

        fn = np.logical_and(
            y_true == 1,
            y_pred == 0
        ).sum()

        precision = (
            tp / (tp + fp)
            if tp + fp > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if tp + fn > 0
            else 0.0
        )

        f1 = binary_f1(
            y_true,
            y_pred
        )

        rows.append({
            "label_name": name,
            "threshold": float(
                thresholds[j]
            ),
            "ground_truth_positive": int(
                y_true.sum()
            ),
            "predicted_positive": int(
                y_pred.sum()
            ),
            "true_positive": int(tp),
            "precision": float(
                precision
            ),
            "recall": float(
                recall
            ),
            "f1": float(f1),
        })

    return pd.DataFrame(rows)


concept_metrics = per_label_metrics(
    concept_targets,
    concept_pred,
    concept_names,
    concept_thresholds
)

negative_metrics = per_label_metrics(
    negative_targets,
    negative_pred,
    negative_names,
    negative_thresholds
)

attribute_metrics = per_label_metrics(
    attribute_targets,
    attribute_pred,
    attribute_names,
    attribute_thresholds
)


concept_micro = binary_f1(
    concept_targets.ravel(),
    concept_pred.ravel()
)

negative_micro = binary_f1(
    negative_targets.ravel(),
    negative_pred.ravel()
)

attribute_micro = binary_f1(
    attribute_targets.ravel(),
    attribute_pred.ravel()
)

concept_macro = float(
    concept_metrics["f1"].mean()
)

negative_macro = float(
    negative_metrics["f1"].mean()
)

attribute_macro = float(
    attribute_metrics["f1"].mean()
)


# ============================================================
# 17. SAVE RESULTS
# ============================================================

concept_file = os.path.join(
    OUTPUT_DIR,
    "test_concept_metrics.csv"
)

negative_file = os.path.join(
    OUTPUT_DIR,
    "test_negative_metrics.csv"
)

attribute_file = os.path.join(
    OUTPUT_DIR,
    "test_attribute_metrics.csv"
)

concept_metrics.to_csv(
    concept_file,
    index=False,
    encoding="utf-8-sig"
)

negative_metrics.to_csv(
    negative_file,
    index=False,
    encoding="utf-8-sig"
)

attribute_metrics.to_csv(
    attribute_file,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 18. CASE-LEVEL PREDICTIONS
# ============================================================

case_predictions = pd.DataFrame({
    "case_id": case_ids
})

for j, name in enumerate(
    concept_names
):

    case_predictions[
        f"gt_concept__{name}"
    ] = concept_targets[:, j]

    case_predictions[
        f"pred_concept__{name}"
    ] = concept_pred[:, j]

for j, name in enumerate(
    negative_names
):

    case_predictions[
        f"gt_negative__{name}"
    ] = negative_targets[:, j]

    case_predictions[
        f"pred_negative__{name}"
    ] = negative_pred[:, j]

for j, name in enumerate(
    attribute_names
):

    case_predictions[
        f"gt_attribute__{name}"
    ] = attribute_targets[:, j]

    case_predictions[
        f"pred_attribute__{name}"
    ] = attribute_pred[:, j]


case_prediction_file = os.path.join(
    OUTPUT_DIR,
    "test_structured_predictions.csv"
)

case_predictions.to_csv(
    case_prediction_file,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 19. SUMMARY JSON
# ============================================================

summary = {

    "ontology":
        "V3-Ontology-V2.2",

    "test_total_cases":
        712,

    "test_structured_cases":
        709,

    "test_report_only_cases":
        3,

    "threshold_source":
        "validation",

    "threshold_tuned_on_test":
        False,

    "concept_macro_f1":
        concept_macro,

    "concept_micro_f1":
        float(concept_micro),

    "negative_macro_f1":
        negative_macro,

    "negative_micro_f1":
        float(negative_micro),

    "attribute_macro_f1":
        attribute_macro,

    "attribute_micro_f1":
        float(attribute_micro),

    "concept_metrics_file":
        concept_file,

    "negative_metrics_file":
        negative_file,

    "attribute_metrics_file":
        attribute_file,

    "case_predictions_file":
        case_prediction_file,
}

summary_file = os.path.join(
    OUTPUT_DIR,
    "final_test_structured_metrics_v3.json"
)

with open(
    summary_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 20. FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 70)
print("V3-7 FINAL TEST STRUCTURED EVALUATION")
print("=" * 70)

print(
    f"\nTest cases: 712"
)

print(
    f"Structured: 709"
)

print(
    f"Report-only: 3"
)

print(
    "\nThreshold source: VALIDATION"
)

print(
    "Test tuning: NO"
)

print("\nCONCEPT")
print(
    f"Macro F1 = {concept_macro:.6f}"
)
print(
    f"Micro F1 = {concept_micro:.6f}"
)

print("\nNEGATIVE")
print(
    f"Macro F1 = {negative_macro:.6f}"
)
print(
    f"Micro F1 = {negative_micro:.6f}"
)

print("\nATTRIBUTE")
print(
    f"Macro F1 = {attribute_macro:.6f}"
)
print(
    f"Micro F1 = {attribute_micro:.6f}"
)

print("\nTOP CONCEPTS")
display(
    concept_metrics
    .sort_values(
        "f1",
        ascending=False
    )
    .head(15)
)

print("\nNEGATIVE")
display(
    negative_metrics
    .sort_values(
        "f1",
        ascending=False
    )
)

print("\nATTRIBUTE")
display(
    attribute_metrics
    .sort_values(
        "f1",
        ascending=False
    )
)

print("\nSaved:")
print(summary_file)
print(concept_file)
print(negative_file)
print(attribute_file)
print(case_prediction_file)

print(
    "\nV3-7 STATUS: PASS"
)

Mounted at /content/drive
training_manifest: FOUND
image_manifest: FOUND
structured_targets: FOUND
config: FOUND
checkpoint: FOUND
validation_thresholds: FOUND

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
BF16: True

Training manifest: (7606, 15)
Image manifest: (76405, 26)
Structured target: (7606, 102)
Frozen validation thresholds: PASS

TEST CASES
Total: 712
Structured: 709
Report-only: 3

Target shapes: (709, 48) (709, 5) (709, 6)
Test NORMAL images: 7086
Test batches: 178

Loading V3 model...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

...


NEGATIVE


,label_name,threshold,ground_truth_positive,predicted_positive,true_positive,precision,recall,f1
1,no_abnormal_nose_sinus,0.10,51,187,28,0.149733,0.549020,0.235294
2,no_abnormal_ent,0.05,13,8,2,0.250000,0.153846,0.190476
3,no_bleeding,0.05,16,22,2,0.090909,0.125000,0.105263
0,no_abnormal_external_middle_ear,0.05,0,0,0,0.000000,0.000000,0.000000
4,no_foreign_body,0.05,1,0,0,0.000000,0.000000,0.000000



ATTRIBUTE


,label_name,threshold,ground_truth_positive,predicted_positive,true_positive,precision,recall,f1
2,right,0.25,286,493,246,0.498986,0.860140,0.631579
3,left,0.20,126,245,82,0.334694,0.650794,0.442049
5,post_surgery,0.20,92,210,64,0.304762,0.695652,0.423841
0,acute,0.20,154,385,112,0.290909,0.727273,0.415584
1,chronic,0.10,55,199,38,0.190955,0.690909,0.299213
4,bilateral,0.05,38,154,21,0.136364,0.552632,0.218750



Saved:
/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v3/structured_test_evaluation/final_test_structured_metrics_v3.json
/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v3/structured_test_evaluation/test_concept_metrics.csv
/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v3/structured_test_evaluation/test_negative_metrics.csv
/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v3/structured_test_evaluation/test_attribute_metrics.csv
/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v3/structured_test_evaluation/test_structured_predictions.csv

V3-7 STATUS: PASS


In [6]:
# ============================================================
# V4-1 PATCH — RECOVER THE FROZEN 48-CONCEPT ONTOLOGY
# ============================================================

# IMPORTANT:
# structured_targets contains 54 concept__ columns because it
# retains legacy ontology columns.
#
# The frozen V3 ontology is defined by:
#   concept_vector_str = 48 dimensions
#
# Therefore we recover the exact 48 labels by matching each
# vector dimension against the structured target columns.

print("=" * 70)
print("RECOVERING FROZEN V3 ONTOLOGY")
print("=" * 70)


# ------------------------------------------------------------
# 1. Parse vectors
# ------------------------------------------------------------

def parse_vector(x):

    if pd.isna(x):
        return []

    x = str(x).strip()

    if not x:
        return []

    return [
        int(float(v))
        for v in x.split(",")
    ]


# ------------------------------------------------------------
# 2. Verify vector dimensions
# ------------------------------------------------------------

concept_vectors = (
    train_manifest[
        "concept_vector_str"
    ]
    .apply(parse_vector)
)

negative_vectors = (
    train_manifest[
        "negative_vector_str"
    ]
    .apply(parse_vector)
)

attribute_vectors = (
    train_manifest[
        "attribute_vector_str"
    ]
    .apply(parse_vector)
)

assert concept_vectors.apply(len).eq(48).all()
assert negative_vectors.apply(len).eq(5).all()
assert attribute_vectors.apply(len).eq(6).all()

print(
    "Frozen vector dimensions: 48 / 5 / 6"
)


# ------------------------------------------------------------
# 3. Structured target lookup
# ------------------------------------------------------------

structured_lookup = (
    structured_targets
    .set_index("case_id")
)

train_case_order = (
    train_manifest["case_id"].tolist()
)

all_concept_vectors = np.stack(
    concept_vectors.values
)


# ------------------------------------------------------------
# 4. ALL legacy concept columns
# ------------------------------------------------------------

all_concept_cols = [
    c
    for c in structured_targets.columns
    if c.startswith("concept__")
]

print(
    "\nStructured target concept columns:",
    len(all_concept_cols)
)

print(
    "These include legacy columns and should NOT all "
    "be treated as the frozen ontology."
)


# ------------------------------------------------------------
# 5. Recover exact 48 columns
# ------------------------------------------------------------

concept_names = []
concept_match_details = []

for dim in range(48):

    target_vector = (
        all_concept_vectors[:, dim]
    )

    matches = []

    for col in all_concept_cols:

        candidate = (
            pd.to_numeric(
                structured_lookup.loc[
                    train_case_order,
                    col
                ],
                errors="coerce"
            )
            .fillna(0)
            .astype(int)
            .to_numpy()
        )

        if np.array_equal(
            target_vector,
            candidate
        ):

            matches.append(
                col
            )

    if len(matches) != 1:

        raise ValueError(
            f"Concept dimension {dim} "
            f"does not have exactly one match.\n"
            f"Matches: {matches}"
        )

    concept_col = matches[0]

    concept_name = concept_col.replace(
        "concept__",
        ""
    )

    concept_names.append(
        concept_name
    )

    concept_match_details.append({
        "dimension": dim,
        "column": concept_col,
        "label": concept_name,
    })


# ------------------------------------------------------------
# 6. Negative + attribute labels
# ------------------------------------------------------------

negative_names = [
    c.replace(
        "negative__",
        ""
    )
    for c in structured_targets.columns
    if c.startswith("negative__")
]

attribute_names = [
    c.replace(
        "attribute__",
        ""
    )
    for c in structured_targets.columns
    if c.startswith("attribute__")
]


# ------------------------------------------------------------
# 7. HARD ASSERTIONS
# ------------------------------------------------------------

assert len(concept_names) == 48
assert len(set(concept_names)) == 48

assert len(negative_names) == 5
assert len(set(negative_names)) == 5

assert len(attribute_names) == 6
assert len(set(attribute_names)) == 6


# ------------------------------------------------------------
# 8. Display recovered ontology
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FROZEN V3 ONTOLOGY")
print("=" * 70)

print("\nCONCEPTS — 48")

for i, name in enumerate(
    concept_names
):

    print(
        f"{i:02d}  {name}"
    )

print("\nNEGATIVE FINDINGS — 5")

for i, name in enumerate(
    negative_names
):

    print(
        f"{i:02d}  {name}"
    )

print("\nATTRIBUTES — 6")

for i, name in enumerate(
    attribute_names
):

    print(
        f"{i:02d}  {name}"
    )


# ------------------------------------------------------------
# 9. Save ontology mapping
# ------------------------------------------------------------

ontology_mapping = pd.DataFrame(
    concept_match_details
)

ontology_mapping_file = os.path.join(
    V4_DIR,
    "diagnostics",
    "frozen_concept_ontology_mapping.csv"
)

ontology_mapping.to_csv(
    ontology_mapping_file,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 10. Save config
# ------------------------------------------------------------

V4_CONFIG = {

    "model_name":
        "V4_ConceptConditionedGeneration",

    "vision_model":
        "google/vit-base-patch16-224",

    "text_model":
        "google/mt5-small",

    "ontology":
        "V3-Ontology-V2.2",

    "num_concepts":
        48,

    "num_negative_findings":
        5,

    "num_attributes":
        6,

    "image_size":
        224,

    "max_images_per_case":
        8,

    "max_target_length":
        96,

    "batch_size":
        4,

    "gradient_accumulation":
        4,

    "effective_batch_size":
        16,

    "vision_lr":
        1e-5,

    "projector_lr":
        1e-4,

    "structured_embedding_lr":
        1e-4,

    "mt5_lr":
        5e-5,

    "weight_decay":
        0.01,

    "epochs":
        5,

    "bf16":
        USE_BF16,

    "seed":
        SEED,

    "source_v3_checkpoint":
        V3_CHECKPOINT_FILE,

    "split":
        "fixed patient-level split",

    "structured_conditioning":
        "direct_mT5_encoder_prefix",
}


with open(
    V4_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        V4_CONFIG,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 11. BUILD V4 MANIFEST
# ------------------------------------------------------------

v4_manifest = train_manifest[
    [
        "case_id",
        "patient_group_id",
        "split",
        "ket_luan",
        "concepts_v2_str",
        "negative_findings_str",
        "attributes_v2_str",
        "concept_vector_str",
        "negative_vector_str",
        "attribute_vector_str",
        "has_structured_label",
        "structured_loss_mask",
        "num_concepts_v2",
        "num_negative_findings",
        "num_attributes_v2",
    ]
].copy()

v4_manifest.to_csv(
    V4_MANIFEST_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 12. PASS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("V4-1 PATCH STATUS: PASS")
print("=" * 70)

print(
    "Frozen concepts:",
    len(concept_names)
)

print(
    "Negative findings:",
    len(negative_names)
)

print(
    "Attributes:",
    len(attribute_names)
)

print(
    "Training cases:",
    len(v4_manifest)
)

print(
    "Ontology mapping saved:",
    ontology_mapping_file
)

print(
    "V4 config saved:",
    V4_CONFIG_FILE
)

print(
    "V4 manifest saved:",
    V4_MANIFEST_FILE
)

print(
    "\nThe 54 legacy concept columns were NOT adopted."
)

print(
    "V4 continues with the frozen 48-concept ontology."
)

RECOVERING FROZEN V3 ONTOLOGY
Frozen vector dimensions: 48 / 5 / 6

Structured target concept columns: 54
These include legacy columns and should NOT all be treated as the frozen ontology.

FROZEN V3 ONTOLOGY

CONCEPTS — 48
00  amidan_qua_phat
01  chan_thuong_mang_nhi
02  chan_thuong_ong_tai_ngoai
03  chay_mau_mui
04  di_vat_hong
05  di_vat_hong_thanh_quan
06  di_vat_tai
07  dich_vat_mui
08  hat_day_thanh
09  hau_phau_mui_xoang
10  hau_phau_va_nhi
11  hep_ong_tai_ngoai
12  hoc_xuong_ca
13  lech_vach_ngan
14  liet_day_thanh
15  nang_day_thanh
16  nhot_ong_tai_ngoai
17  not_vanh_tai
18  polyp_day_thanh
19  polyp_mui
20  polyp_ong_tai_ngoai
21  qua_phat_va
22  ray_tai
23  ro_luan_nhi
24  seo_hoc_mui
25  theo_doi_trao_nguoc
26  thung_mang_nhi
27  tien_dinh_mui
28  tu_dich_vanh_tai
29  u_hoc_mui
30  u_nhu_cuon_mui
31  u_nhu_hoc_mui
32  u_xoang
33  viem_amidan
34  viem_hong
35  viem_luoi
36  viem_mang_nhi
37  viem_mieng
38  viem_mui
39  viem_mui_xoang
40  viem_ong_tai_ngoai
41  viem_tai_giua

In [12]:
# ============================================================
# V4-2A — CLEAN SETUP
# ============================================================

import os
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from PIL import Image

from transformers import (
    ViTModel,
    MT5ForConditionalGeneration,
    AutoTokenizer,
)

# ------------------------------------------------------------
# Seed
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Drive
# ------------------------------------------------------------

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "NoiSoi_Matching"
)

V3_DIR = os.path.join(
    PROJECT_DIR,
    "baseline_model_v3"
)

V4_DIR = os.path.join(
    PROJECT_DIR,
    "baseline_model_v4"
)

TRAINING_MANIFEST_FILE = os.path.join(
    V4_DIR,
    "v4_training_manifest.csv"
)

IMAGE_MANIFEST_FILE = os.path.join(
    PROJECT_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

STRUCTURED_TARGET_FILE = os.path.join(
    PROJECT_DIR,
    "v3_ontology_v2",
    "v3_structured_targets_v2_2.csv"
)

V3_CHECKPOINT_FILE = os.path.join(
    V3_DIR,
    "checkpoints",
    "best.pt"
)

V4_CONFIG_FILE = os.path.join(
    V4_DIR,
    "v4_config.json"
)

ARCH_CONFIG_FILE = os.path.join(
    V4_DIR,
    "diagnostics",
    "v4_architecture_config.json"
)

os.makedirs(
    os.path.dirname(
        ARCH_CONFIG_FILE
    ),
    exist_ok=True
)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

USE_BF16 = (
    device.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

print("=" * 60)
print("V4-2A — CLEAN SETUP")
print("=" * 60)

print(
    "Device:",
    device
)

if device.type == "cuda":

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "VRAM:",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

print(
    "BF16:",
    USE_BF16
)

# ------------------------------------------------------------
# Load manifests
# ------------------------------------------------------------

train_manifest = pd.read_csv(
    TRAINING_MANIFEST_FILE,
    low_memory=False
)

image_manifest = pd.read_csv(
    IMAGE_MANIFEST_FILE,
    low_memory=False
)

structured_targets = pd.read_csv(
    STRUCTURED_TARGET_FILE,
    low_memory=False
)

with open(
    V4_CONFIG_FILE,
    "r",
    encoding="utf-8"
) as f:

    v4_config = json.load(f)

# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------

NUM_CONCEPTS = 48
NUM_NEGATIVE = 5
NUM_ATTRIBUTES = 6

IMAGE_SIZE = 224
MAX_IMAGES_PER_CASE = 8
MAX_TARGET_LENGTH = 96
BATCH_SIZE = 4

# ------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    "google/mt5-small"
)

print(
    "Tokenizer:",
    type(tokenizer).__name__
)

print(
    "Vocab size:",
    len(tokenizer)
)

# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------

assert len(train_manifest) == 7606
assert len(structured_targets) == 7606

assert NUM_CONCEPTS == 48
assert NUM_NEGATIVE == 5
assert NUM_ATTRIBUTES == 6

print("\nSource shapes:")
print(
    "Training:",
    train_manifest.shape
)
print(
    "Images:",
    image_manifest.shape
)
print(
    "Structured:",
    structured_targets.shape
)

print(
    "\nV4-2A STATUS: PASS"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
V4-2A — CLEAN SETUP
Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.49 GB
BF16: True

Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Tokenizer: T5Tokenizer
Vocab size: 250100

Source shapes:
Training: (7606, 15)
Images: (76405, 26)
Structured: (7606, 102)

V4-2A STATUS: PASS


In [13]:
# ============================================================
# V4-2B — MODEL + STRUCTURED CONDITIONER
# ============================================================

import torch
import torch.nn as nn
import os


# ============================================================
# 1. LOAD PRETRAINED COMPONENTS
# ============================================================

print("=" * 60)
print("V4-2B — MODEL COMPONENTS")
print("=" * 60)

print("\nLoading ViT-B/16...")

vision = ViTModel.from_pretrained(
    "google/vit-base-patch16-224"
)

print(
    "ViT hidden size:",
    vision.config.hidden_size
)

print("\nLoading mT5-small...")

mt5 = MT5ForConditionalGeneration.from_pretrained(
    "google/mt5-small"
)

print(
    "mT5 d_model:",
    mt5.config.d_model
)

print(
    "mT5 vocab:",
    mt5.config.vocab_size
)


# ============================================================
# 2. V4 DIMENSIONS
# ============================================================

VISION_DIM = (
    vision.config.hidden_size
)

TEXT_DIM = (
    mt5.config.d_model
)

assert VISION_DIM == 768
assert TEXT_DIM == 512


# ============================================================
# 3. STRUCTURED CONDITIONER
# ============================================================

class StructuredConditioner(
    nn.Module
):

    def __init__(
        self,
        d_model,
        num_concepts=48,
        num_negative=5,
        num_attributes=6
    ):

        super().__init__()

        self.num_concepts = (
            num_concepts
        )

        self.num_negative = (
            num_negative
        )

        self.num_attributes = (
            num_attributes
        )

        # ----------------------------------------------------
        # Binary value embeddings
        #
        # Each label has value 0 or 1.
        # ----------------------------------------------------

        self.concept_value = nn.Embedding(
            2,
            d_model
        )

        self.negative_value = nn.Embedding(
            2,
            d_model
        )

        self.attribute_value = nn.Embedding(
            2,
            d_model
        )

        # ----------------------------------------------------
        # Label identity embeddings
        # ----------------------------------------------------

        self.concept_label = nn.Embedding(
            num_concepts,
            d_model
        )

        self.negative_label = nn.Embedding(
            num_negative,
            d_model
        )

        self.attribute_label = nn.Embedding(
            num_attributes,
            d_model
        )

        # ----------------------------------------------------
        # Type embeddings
        #
        # 0 = concept
        # 1 = negative
        # 2 = attribute
        # ----------------------------------------------------

        self.type_embedding = nn.Embedding(
            3,
            d_model
        )

        self.norm = nn.LayerNorm(
            d_model
        )

    def forward(
        self,
        concept_targets,
        negative_targets,
        attribute_targets
    ):

        device = (
            concept_targets.device
        )

        # ----------------------------------------------------
        # Convert 0/1 float vectors to integer IDs
        # ----------------------------------------------------

        concept_ids = (
            concept_targets
            .long()
            .clamp(0, 1)
        )

        negative_ids = (
            negative_targets
            .long()
            .clamp(0, 1)
        )

        attribute_ids = (
            attribute_targets
            .long()
            .clamp(0, 1)
        )

        # ----------------------------------------------------
        # Concept tokens
        # ----------------------------------------------------

        concept_idx = torch.arange(
            self.num_concepts,
            device=device
        )

        concept_type = self.type_embedding(
            torch.zeros(
                self.num_concepts,
                dtype=torch.long,
                device=device
            )
        )

        concept_tokens = (
            self.concept_value(
                concept_ids
            )
            +
            self.concept_label(
                concept_idx
            )[None, :, :]
            +
            concept_type[None, :, :]
        )

        # ----------------------------------------------------
        # Negative tokens
        # ----------------------------------------------------

        negative_idx = torch.arange(
            self.num_negative,
            device=device
        )

        negative_type = self.type_embedding(
            torch.ones(
                self.num_negative,
                dtype=torch.long,
                device=device
            )
        )

        negative_tokens = (
            self.negative_value(
                negative_ids
            )
            +
            self.negative_label(
                negative_idx
            )[None, :, :]
            +
            negative_type[None, :, :]
        )

        # ----------------------------------------------------
        # Attribute tokens
        # ----------------------------------------------------

        attribute_idx = torch.arange(
            self.num_attributes,
            device=device
        )

        attribute_type = self.type_embedding(
            torch.full(
                (
                    self.num_attributes,
                ),
                2,
                dtype=torch.long,
                device=device
            )
        )

        attribute_tokens = (
            self.attribute_value(
                attribute_ids
            )
            +
            self.attribute_label(
                attribute_idx
            )[None, :, :]
            +
            attribute_type[None, :, :]
        )

        # ----------------------------------------------------
        # [concept | negative | attribute]
        # ----------------------------------------------------

        structured_tokens = torch.cat(
            [
                concept_tokens,
                negative_tokens,
                attribute_tokens,
            ],
            dim=1
        )

        structured_tokens = self.norm(
            structured_tokens
        )

        B = (
            concept_targets.shape[0]
        )

        structured_attention = torch.ones(
            B,
            structured_tokens.shape[1],
            dtype=torch.long,
            device=device
        )

        return (
            structured_tokens,
            structured_attention
        )


conditioner = StructuredConditioner(
    d_model=TEXT_DIM,
    num_concepts=48,
    num_negative=5,
    num_attributes=6
)


# ============================================================
# 4. V4 MODEL
# ============================================================

class V4Model(
    nn.Module
):

    def __init__(
        self,
        vision,
        mt5,
        conditioner
    ):

        super().__init__()

        self.vision = vision

        self.mt5 = mt5

        self.projector = nn.Linear(
            VISION_DIM,
            TEXT_DIM
        )

        self.conditioner = (
            conditioner
        )


model = V4Model(
    vision=vision,
    mt5=mt5,
    conditioner=conditioner
)


# ============================================================
# 5. LOAD V3 CHECKPOINT
# ============================================================

print("\nLoading V3 checkpoint...")

checkpoint = torch.load(
    V3_CHECKPOINT_FILE,
    map_location="cpu",
    weights_only=False
)

state_dict = (
    checkpoint[
        "model_state_dict"
    ]
)

print(
    "Checkpoint keys:",
    len(state_dict)
)


# ------------------------------------------------------------
# IMPORTANT:
#
# V3 checkpoint contains:
#
# vision.*
# mt5.*
# projector.*
# concept_head.*
# negative_head.*
# attribute_head.*
#
# V4 only needs:
#
# vision.*
# mt5.*
# projector.*
#
# Structured heads are intentionally NOT loaded.
# ------------------------------------------------------------

v4_state = {}

for key, value in state_dict.items():

    if (
        key.startswith("vision.")
        or
        key.startswith("mt5.")
        or
        key.startswith("projector.")
    ):

        v4_state[key] = value


missing, unexpected = (
    model.load_state_dict(
        v4_state,
        strict=False
    )
)

print(
    "\nLoaded V3 visual/text weights."
)

print(
    "Missing keys:",
    missing
)

print(
    "Unexpected keys:",
    unexpected
)


# ============================================================
# 6. MOVE TO GPU
# ============================================================

model = model.to(
    device
)


# ============================================================
# 7. PARAMETER COUNTS
# ============================================================

def count_params(
    module
):

    total = sum(
        p.numel()
        for p in module.parameters()
    )

    trainable = sum(
        p.numel()
        for p in module.parameters()
        if p.requires_grad
    )

    return total, trainable


print("\n" + "=" * 60)
print("PARAMETERS")
print("=" * 60)

for name, module in [
    ("ViT", model.vision),
    ("mT5", model.mt5),
    ("Projector", model.projector),
    ("Conditioner", model.conditioner),
]:

    total, trainable = count_params(
        module
    )

    print(
        f"{name}: "
        f"{total:,} total | "
        f"{trainable:,} trainable"
    )


# ============================================================
# 8. CONDITIONER PARAMETER CHECK
# ============================================================

conditioner_params = sum(
    p.numel()
    for p in model.conditioner.parameters()
)

expected_conditioner_params = (
    # value embeddings
    (2 * TEXT_DIM) * 3
    +
    # label embeddings
    (
        48
        + 5
        + 6
    ) * TEXT_DIM
    +
    # type embedding
    3 * TEXT_DIM
    +
    # LayerNorm
    2 * TEXT_DIM
)

print(
    "\nConditioner parameters:",
    conditioner_params
)

print(
    "Expected:",
    expected_conditioner_params
)

assert (
    conditioner_params
    == expected_conditioner_params
)


# ============================================================
# 9. CHECKPOINT INITIALIZATION
# ============================================================

# Verify that V3 weights were actually transferred.

assert torch.equal(
    model.projector.proj.weight
    if hasattr(
        model.projector,
        "proj"
    )
    else model.projector.weight,
    (
        state_dict[
            "projector.proj.weight"
        ]
        if "projector.proj.weight"
        in state_dict
        else state_dict[
            "projector.weight"
        ]
    )
)


# ============================================================
# 10. SAVE ARCHITECTURE INFO
# ============================================================

architecture_info = {

    "model":
        "V4_ConceptConditionedGeneration",

    "vision":
        "google/vit-base-patch16-224",

    "text":
        "google/mt5-small",

    "vision_dim":
        VISION_DIM,

    "text_dim":
        TEXT_DIM,

    "concepts":
        48,

    "negative_findings":
        5,

    "attributes":
        6,

    "structured_tokens":
        59,

    "conditioning":
        "direct_encoder_prefix",

    "prefix_order":
        "visual | concept | negative | attribute",

    "v3_initialization":
        True,

    "v3_checkpoint":
        V3_CHECKPOINT_FILE,

    "structured_heads_loaded":
        False,
}


with open(
    ARCH_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        architecture_info,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 11. PASS
# ============================================================

print("\n" + "=" * 60)
print("V4-2B STATUS: PASS")
print("=" * 60)

print(
    "ViT initialized from V3:",
    True
)

print(
    "mT5 initialized from V3:",
    True
)

print(
    "Projector initialized from V3:",
    True
)

print(
    "Structured conditioner:",
    "NEW"
)

print(
    "Structured tokens:",
    59
)

print(
    "No training performed."
)

print(
    "No optimizer step performed."
)

V4-2B — MODEL COMPONENTS

Loading ViT-B/16...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ViT hidden size: 768

Loading mT5-small...


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


mT5 d_model: 512
mT5 vocab: 250112

Loading V3 checkpoint...
Checkpoint keys: 400

Loaded V3 visual/text weights.
Missing keys: ['projector.weight', 'projector.bias', 'conditioner.concept_value.weight', 'conditioner.negative_value.weight', 'conditioner.attribute_value.weight', 'conditioner.concept_label.weight', 'conditioner.negative_label.weight', 'conditioner.attribute_label.weight', 'conditioner.type_embedding.weight', 'conditioner.norm.weight', 'conditioner.norm.bias']
Unexpected keys: ['projector.proj.weight', 'projector.proj.bias']

PARAMETERS
ViT: 86,389,248 total | 86,389,248 trainable
mT5: 300,176,768 total | 300,176,768 trainable
Projector: 393,728 total | 393,728 trainable
Conditioner: 35,840 total | 35,840 trainable

Conditioner parameters: 35840
Expected: 35840


RuntimeError: Expected all tensors to be on the same device, but got other is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__equal)

In [16]:
# ============================================================
# V4-2B — SAVE ARCHITECTURE STATUS
# ============================================================

import json
from pathlib import Path

architecture_info = {
    "model": "V4_ConceptConditionedGeneration",
    "vision": "google/vit-base-patch16-224",
    "text": "google/mt5-small",

    "vision_dim": 768,
    "text_dim": 512,

    "num_concepts": 48,
    "num_negative": 5,
    "num_attributes": 6,

    "max_images": 8,
    "image_size": 224,
    "max_target_length": 96,

    "projector_structure": "VisualProjector.proj",

    "vision_initialized_from_v3": True,
    "mt5_initialized_from_v3": True,
    "projector_initialized_from_v3": True,

    "conditioner_initialized_new": True,

    "conditioning": "direct_mT5_encoder_prefix"
}

ARCH_CONFIG_FILE = (
    Path(PROJECT_DIR)
    / "baseline_model_v4"
    / "diagnostics"
    / "v4_architecture_config.json"
)

ARCH_CONFIG_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    ARCH_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        architecture_info,
        f,
        ensure_ascii=False,
        indent=2
    )

print("=" * 60)
print("V4-2B STATUS: PASS")
print("=" * 60)
print("Projector V3 weights: EXACT MATCH")
print("Projector device:",
      next(model.projector.parameters()).device)
print("Config saved:", ARCH_CONFIG_FILE)

V4-2B STATUS: PASS
Projector V3 weights: EXACT MATCH
Projector device: cuda:0
Config saved: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/diagnostics/v4_architecture_config.json


In [18]:
# ============================================================
# V4-2C-A — REBUILD TRAIN DATALOADER
# ============================================================

import os
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

print("=" * 60)
print("V4-2C-A — REBUILD DATALOADER")
print("=" * 60)


# ------------------------------------------------------------
# 1. Load manifests from Drive
# ------------------------------------------------------------

v4_manifest = pd.read_csv(
    V4_MANIFEST_FILE
)

image_df = pd.read_csv(
    IMAGE_MANIFEST_FILE
)

print(
    "V4 manifest:",
    v4_manifest.shape
)

print(
    "Image manifest:",
    image_df.shape
)


# ------------------------------------------------------------
# 2. Keep NORMAL images only
# ------------------------------------------------------------

image_df = image_df[
    image_df["image_status"] == "NORMAL"
].copy()

image_df = image_df[
    image_df["case_id"].isin(
        v4_manifest["case_id"]
    )
].copy()

print(
    "NORMAL images:",
    len(image_df)
)


# ------------------------------------------------------------
# 3. Image transform
# ------------------------------------------------------------

image_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ------------------------------------------------------------
# 4. Dataset
# ------------------------------------------------------------

class V4EndoscopyDataset(Dataset):

    def __init__(
        self,
        cases_df,
        images_df,
        split,
        max_images=8,
        transform=None,
        training=False
    ):

        self.cases = (
            cases_df[
                cases_df["split"] == split
            ]
            .reset_index(drop=True)
        )

        self.images = images_df
        self.max_images = max_images
        self.transform = transform
        self.training = training

        self.image_map = {}

        for case_id, group in images_df.groupby(
            "case_id"
        ):

            paths = group[
                "image_path"
            ].tolist()

            self.image_map[
                case_id
            ] = paths


    def __len__(self):

        return len(self.cases)


    def __getitem__(self, idx):

        row = self.cases.iloc[idx]

        case_id = row["case_id"]

        paths = list(
            self.image_map[case_id]
        )

        # Same sampling logic as V1/V2/V3
        if len(paths) > self.max_images:

            if self.training:

                paths = random.sample(
                    paths,
                    self.max_images
                )

            else:

                paths = paths[
                    :self.max_images
                ]

        tensors = []

        for path in paths:

            img = Image.open(
                path
            ).convert("RGB")

            if self.transform is not None:

                img = self.transform(img)

            tensors.append(img)

        return {
            "case_id": case_id,
            "patient_group_id": row[
                "patient_group_id"
            ],
            "images": tensors,
            "num_images": len(tensors),
            "target_text": row[
                "ket_luan"
            ]
        }


# ------------------------------------------------------------
# 5. Collate
# ------------------------------------------------------------

def v4_collate_fn(batch):

    return {
        "case_id": [
            x["case_id"]
            for x in batch
        ],

        "patient_group_id": [
            x["patient_group_id"]
            for x in batch
        ],

        "images": [
            x["images"]
            for x in batch
        ],

        "num_images": [
            x["num_images"]
            for x in batch
        ],

        "target_text": [
            x["target_text"]
            for x in batch
        ]
    }


# ------------------------------------------------------------
# 6. Create train loader
# ------------------------------------------------------------

train_dataset = V4EndoscopyDataset(
    cases_df=v4_manifest,
    images_df=image_df,
    split="train",
    max_images=MAX_IMAGES_PER_CASE,
    transform=image_transform,
    training=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=v4_collate_fn
)


# ------------------------------------------------------------
# 7. Sanity check
# ------------------------------------------------------------

batch = next(
    iter(train_loader)
)

print(
    "Cases:",
    len(train_dataset)
)

print(
    "Batch size:",
    len(batch["case_id"])
)

print(
    "Num images:",
    batch["num_images"]
)

print(
    "Targets:",
    batch["target_text"][:2]
)

print("\n" + "=" * 60)
print("V4-2C-A STATUS: PASS")
print("=" * 60)

V4-2C-A — REBUILD DATALOADER


/tmp/ipykernel_3142/2595670654.py:26: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  image_df = pd.read_csv(


V4 manifest: (7606, 15)
Image manifest: (76405, 26)
NORMAL images: 76209
Cases: 6138
Batch size: 4
Num images: [8, 8, 4, 5]
Targets: ['VIÊM MŨI XOANG', 'VIÊM TAI GIỮA T CẤP VIÊM MŨI XOANG']

V4-2C-A STATUS: PASS


In [20]:
# ============================================================
# V4-2C-B PATCH — LOAD FROZEN 48-D CONCEPT MAPPING
# ============================================================

import pandas as pd

print("=" * 60)
print("V4-2C-B PATCH — FROZEN ONTOLOGY")
print("=" * 60)


# ------------------------------------------------------------
# 1. Load frozen mapping created in V4-1
# ------------------------------------------------------------

MAPPING_FILE = (
    "/content/drive/MyDrive/"
    "NoiSoi_Matching/baseline_model_v4/"
    "diagnostics/frozen_concept_ontology_mapping.csv"
)

mapping_df = pd.read_csv(
    MAPPING_FILE,
    low_memory=False
)

print(
    "Mapping shape:",
    mapping_df.shape
)

print(
    "Mapping columns:",
    list(mapping_df.columns)
)


# ------------------------------------------------------------
# 2. Recover concept columns in frozen order
# ------------------------------------------------------------

# The mapping file contains the exact correspondence
# between frozen ontology dimension and source column.

print("\nMapping preview:")
display(mapping_df.head())


# Try common column names used by V4-1
if "source_column" in mapping_df.columns:

    concept_cols = (
        mapping_df
        .sort_values("dimension")
        ["source_column"]
        .tolist()
    )

elif "concept_column" in mapping_df.columns:

    concept_cols = (
        mapping_df
        .sort_values("dimension")
        ["concept_column"]
        .tolist()
    )

elif "column" in mapping_df.columns:

    concept_cols = (
        mapping_df
        .sort_values("dimension")
        ["column"]
        .tolist()
    )

else:

    raise ValueError(
        "Cannot identify source concept column "
        "in frozen mapping file. "
        f"Available columns: {list(mapping_df.columns)}"
    )


# ------------------------------------------------------------
# 3. Load structured targets
# ------------------------------------------------------------

structured_df = pd.read_csv(
    STRUCTURED_TARGET_FILE,
    low_memory=False
)

print(
    "\nStructured target shape:",
    structured_df.shape
)


# ------------------------------------------------------------
# 4. Validate exact 48 concept columns
# ------------------------------------------------------------

assert len(concept_cols) == 48

missing_concepts = [
    c for c in concept_cols
    if c not in structured_df.columns
]

assert len(missing_concepts) == 0, (
    f"Missing frozen concept columns: "
    f"{missing_concepts}"
)


# ------------------------------------------------------------
# 5. Negative / attribute columns
# ------------------------------------------------------------

negative_cols = [
    c for c in structured_df.columns
    if c.startswith("negative__")
]

attribute_cols = [
    c for c in structured_df.columns
    if c.startswith("attribute__")
]

assert len(negative_cols) == 5
assert len(attribute_cols) == 6


# ------------------------------------------------------------
# 6. Restrict to V4 cases
# ------------------------------------------------------------

structured_df = structured_df[
    structured_df["case_id"].isin(
        v4_manifest["case_id"]
    )
].copy()


# ------------------------------------------------------------
# 7. Index by case_id
# ------------------------------------------------------------

structured_df = (
    structured_df
    .set_index("case_id")
)


# ------------------------------------------------------------
# 8. Coverage check
# ------------------------------------------------------------

missing_cases = set(
    v4_manifest["case_id"]
) - set(
    structured_df.index
)

assert len(missing_cases) == 0, (
    f"Missing structured targets: "
    f"{len(missing_cases)}"
)


# ------------------------------------------------------------
# 9. Final checks
# ------------------------------------------------------------

assert len(concept_cols) == 48
assert len(negative_cols) == 5
assert len(attribute_cols) == 6


print("\n" + "=" * 60)
print("V4-2C-B PATCH STATUS: PASS")
print("=" * 60)

print(
    "Structured cases:",
    len(structured_df)
)

print(
    "Frozen concepts:",
    len(concept_cols)
)

print(
    "Negative:",
    len(negative_cols)
)

print(
    "Attributes:",
    len(attribute_cols)
)

print("\nFirst 10 frozen concepts:")

for i, col in enumerate(concept_cols[:10]):
    print(
        f"{i:02d}: {col}"
    )

V4-2C-B PATCH — FROZEN ONTOLOGY
Mapping shape: (48, 3)
Mapping columns: ['dimension', 'column', 'label']

Mapping preview:


,dimension,column,label
0,0,concept__amidan_qua_phat,amidan_qua_phat
1,1,concept__chan_thuong_mang_nhi,chan_thuong_mang_nhi
2,2,concept__chan_thuong_ong_tai_ngoai,chan_thuong_ong_tai_ngoai
3,3,concept__chay_mau_mui,chay_mau_mui
4,4,concept__di_vat_hong,di_vat_hong



Structured target shape: (7606, 102)

V4-2C-B PATCH STATUS: PASS
Structured cases: 7606
Frozen concepts: 48
Negative: 5
Attributes: 6

First 10 frozen concepts:
00: concept__amidan_qua_phat
01: concept__chan_thuong_mang_nhi
02: concept__chan_thuong_ong_tai_ngoai
03: concept__chay_mau_mui
04: concept__di_vat_hong
05: concept__di_vat_hong_thanh_quan
06: concept__di_vat_tai
07: concept__dich_vat_mui
08: concept__hat_day_thanh
09: concept__hau_phau_mui_xoang


In [22]:
# ============================================================
# V4-2C-C PATCH — INSPECT CONDITIONER OUTPUT
# ============================================================

with torch.autocast(
    device_type="cuda",
    dtype=torch.bfloat16
):

    conditioner_output = model.conditioner(
        concept_values,
        negative_values,
        attribute_values
    )

print(
    "Output type:",
    type(conditioner_output)
)

if isinstance(
    conditioner_output,
    tuple
):

    print(
        "Tuple length:",
        len(conditioner_output)
    )

    for i, item in enumerate(
        conditioner_output
    ):

        if torch.is_tensor(item):

            print(
                f"[{i}] tensor:",
                tuple(item.shape),
                item.dtype
            )

        else:

            print(
                f"[{i}]:",
                type(item)
            )

else:

    print(
        "Output shape:",
        conditioner_output.shape
    )

Output type: <class 'tuple'>
Tuple length: 2
[0] tensor: (4, 59, 512) torch.float32
[1] tensor: (4, 59) torch.int64


In [24]:
# ============================================================
# V4-2C-C PATCH — CORRECT VOCAB CHECK
# ============================================================

model_vocab_size = (
    model.mt5.config.vocab_size
)

lm_head_size = (
    model.mt5.lm_head.out_features
)

print(
    "Tokenizer vocab:",
    tokenizer.vocab_size
)

print(
    "mT5 config vocab:",
    model_vocab_size
)

print(
    "LM head vocab:",
    lm_head_size
)

print(
    "Logits vocab:",
    logits.shape[-1]
)


# ------------------------------------------------------------
# Correct assertions
# ------------------------------------------------------------

assert logits.shape[0] == B

assert logits.shape[-1] == (
    model.mt5.config.vocab_size
)

assert logits.shape[-1] == (
    model.mt5.lm_head.out_features
)

assert torch.isfinite(loss)


print("\n" + "=" * 60)
print("V4-2C STATUS: PASS")
print("=" * 60)

print(
    "Encoder prefix:",
    tuple(encoder_prefix.shape)
)

print(
    "Logits:",
    tuple(logits.shape)
)

print(
    "Loss:",
    float(loss.detach().cpu())
)

Tokenizer vocab: 250100
mT5 config vocab: 250112
LM head vocab: 250112
Logits vocab: 250112

V4-2C STATUS: PASS
Encoder prefix: (4, 67, 512)
Logits: (4, 20, 250112)
Loss: 1.279809594154358


In [25]:
# ============================================================
# V4-2D — DTYPE SANITY CHECK
# ============================================================

print("=" * 60)
print("V4-2D — DTYPE SANITY CHECK")
print("=" * 60)

print(
    "Visual dtype:",
    visual_tokens.dtype
)

print(
    "Structured dtype:",
    structured_tokens.dtype
)

print(
    "Encoder prefix dtype:",
    encoder_prefix.dtype
)

# ------------------------------------------------------------
# Expected behavior under BF16 training:
# visual + structured -> BF16
# ------------------------------------------------------------

structured_tokens_bf16 = (
    structured_tokens.to(
        dtype=visual_tokens.dtype
    )
)

encoder_prefix_bf16 = torch.cat(
    [
        visual_tokens,
        structured_tokens_bf16
    ],
    dim=1
)

print(
    "Structured BF16:",
    structured_tokens_bf16.dtype
)

print(
    "Encoder prefix BF16:",
    encoder_prefix_bf16.dtype
)

assert (
    encoder_prefix_bf16.dtype
    == visual_tokens.dtype
)

assert encoder_prefix_bf16.shape == (
    B,
    67,
    TEXT_DIM
)

print("\n" + "=" * 60)
print("V4-2D STATUS: PASS")
print("=" * 60)

V4-2D — DTYPE SANITY CHECK
Visual dtype: torch.bfloat16
Structured dtype: torch.float32
Encoder prefix dtype: torch.float32
Structured BF16: torch.bfloat16
Encoder prefix BF16: torch.bfloat16

V4-2D STATUS: PASS


In [26]:
# ============================================================
# V4-3A — PREDICTED STRUCTURED CONDITIONING
# ============================================================

import torch
import torch.nn as nn

print("=" * 60)
print("V4-3A — PREDICTED STRUCTURED CONDITIONING")
print("=" * 60)


# ------------------------------------------------------------
# 1. Structured prediction heads
# ------------------------------------------------------------

class StructuredPredictionHeads(nn.Module):

    def __init__(
        self,
        hidden_dim=512,
        num_concepts=48,
        num_negative=5,
        num_attributes=6
    ):
        super().__init__()

        self.concept = nn.Linear(
            hidden_dim,
            num_concepts
        )

        self.negative = nn.Linear(
            hidden_dim,
            num_negative
        )

        self.attribute = nn.Linear(
            hidden_dim,
            num_attributes
        )

    def forward(self, visual_repr):

        concept_logits = self.concept(
            visual_repr
        )

        negative_logits = self.negative(
            visual_repr
        )

        attribute_logits = self.attribute(
            visual_repr
        )

        return (
            concept_logits,
            negative_logits,
            attribute_logits
        )


# ------------------------------------------------------------
# 2. Create heads
# ------------------------------------------------------------

structured_heads = StructuredPredictionHeads(
    hidden_dim=TEXT_DIM,
    num_concepts=48,
    num_negative=5,
    num_attributes=6
).to(device)


# ------------------------------------------------------------
# 3. Parameter count
# ------------------------------------------------------------

head_params = sum(
    p.numel()
    for p in structured_heads.parameters()
)

print(
    "Structured head parameters:",
    head_params
)


# ------------------------------------------------------------
# 4. Use visual tokens from dry-run
# ------------------------------------------------------------

# visual_tokens:
# [B, 8, 512]

visual_repr = (
    visual_tokens.mean(
        dim=1
    )
)

print(
    "Visual representation:",
    tuple(visual_repr.shape)
)


# ------------------------------------------------------------
# 5. Predict structured labels
# ------------------------------------------------------------

with torch.autocast(
    device_type="cuda",
    dtype=torch.bfloat16
):

    (
        concept_logits,
        negative_logits,
        attribute_logits
    ) = structured_heads(
        visual_repr
    )


print(
    "Concept logits:",
    tuple(concept_logits.shape)
)

print(
    "Negative logits:",
    tuple(negative_logits.shape)
)

print(
    "Attribute logits:",
    tuple(attribute_logits.shape)
)


# ------------------------------------------------------------
# 6. Convert predictions to soft probabilities
# ------------------------------------------------------------

concept_probs = torch.sigmoid(
    concept_logits
)

negative_probs = torch.sigmoid(
    negative_logits
)

attribute_probs = torch.sigmoid(
    attribute_logits
)


print(
    "Concept probabilities:",
    tuple(concept_probs.shape)
)

print(
    "Negative probabilities:",
    tuple(negative_probs.shape)
)

print(
    "Attribute probabilities:",
    tuple(attribute_probs.shape)
)


# ------------------------------------------------------------
# 7. Basic sanity checks
# ------------------------------------------------------------

assert concept_logits.shape == (
    B,
    48
)

assert negative_logits.shape == (
    B,
    5
)

assert attribute_logits.shape == (
    B,
    6
)

assert torch.isfinite(
    concept_logits
).all()

assert torch.isfinite(
    negative_logits
).all()

assert torch.isfinite(
    attribute_logits
).all()


print("\n" + "=" * 60)
print("V4-3A STATUS: PASS")
print("=" * 60)

V4-3A — PREDICTED STRUCTURED CONDITIONING
Structured head parameters: 30267
Visual representation: (4, 512)
Concept logits: (4, 48)
Negative logits: (4, 5)
Attribute logits: (4, 6)
Concept probabilities: (4, 48)
Negative probabilities: (4, 5)
Attribute probabilities: (4, 6)

V4-3A STATUS: PASS


In [27]:
# ============================================================
# V4-3B — MASKED VISUAL REPRESENTATION
# ============================================================

print("=" * 60)
print("V4-3B — MASKED VISUAL REPRESENTATION")
print("=" * 60)


# ------------------------------------------------------------
# 1. Build valid-image mask
# ------------------------------------------------------------

num_images_tensor = torch.tensor(
    batch["num_images"],
    device=device
)

image_mask = (
    torch.arange(
        MAX_IMAGES_PER_CASE,
        device=device
    )[None, :]
    < num_images_tensor[:, None]
)

image_mask = image_mask.to(
    visual_tokens.dtype
)

print(
    "Image mask:",
    tuple(image_mask.shape)
)

print(
    "Num valid images:",
    batch["num_images"]
)


# ------------------------------------------------------------
# 2. Masked mean
# ------------------------------------------------------------

visual_repr = (
    visual_tokens * image_mask.unsqueeze(-1)
).sum(dim=1) / (
    image_mask.sum(dim=1, keepdim=True)
    .clamp(min=1)
)

print(
    "Masked visual representation:",
    tuple(visual_repr.shape)
)


# ------------------------------------------------------------
# 3. Predict structured information
# ------------------------------------------------------------

with torch.autocast(
    device_type="cuda",
    dtype=torch.bfloat16
):

    (
        concept_logits,
        negative_logits,
        attribute_logits
    ) = structured_heads(
        visual_repr
    )


# ------------------------------------------------------------
# 4. Convert predictions to probabilities
# ------------------------------------------------------------

concept_probs = torch.sigmoid(
    concept_logits
)

negative_probs = torch.sigmoid(
    negative_logits
)

attribute_probs = torch.sigmoid(
    attribute_logits
)


# ------------------------------------------------------------
# 5. Convert probabilities to structured embeddings
# ------------------------------------------------------------

with torch.autocast(
    device_type="cuda",
    dtype=torch.bfloat16
):

    predicted_structured_tokens = model.conditioner(
        concept_probs,
        negative_probs,
        attribute_probs
    )[0]


print(
    "Predicted structured tokens:",
    tuple(
        predicted_structured_tokens.shape
    ),
    predicted_structured_tokens.dtype
)


# ------------------------------------------------------------
# 6. Combine with visual tokens
# ------------------------------------------------------------

predicted_structured_tokens = (
    predicted_structured_tokens.to(
        visual_tokens.dtype
    )
)

predicted_encoder_prefix = torch.cat(
    [
        visual_tokens,
        predicted_structured_tokens
    ],
    dim=1
)

predicted_encoder_attention = torch.cat(
    [
        (
            image_mask.long()
        ),
        torch.ones(
            B,
            59,
            device=device,
            dtype=torch.long
        )
    ],
    dim=1
)


print(
    "Predicted encoder prefix:",
    tuple(
        predicted_encoder_prefix.shape
    )
)

print(
    "Predicted encoder dtype:",
    predicted_encoder_prefix.dtype
)


# ------------------------------------------------------------
# 7. Sanity checks
# ------------------------------------------------------------

assert visual_repr.shape == (
    B,
    TEXT_DIM
)

assert concept_logits.shape == (
    B,
    48
)

assert negative_logits.shape == (
    B,
    5
)

assert attribute_logits.shape == (
    B,
    6
)

assert predicted_structured_tokens.shape == (
    B,
    59,
    TEXT_DIM
)

assert predicted_encoder_prefix.shape == (
    B,
    MAX_IMAGES_PER_CASE + 59,
    TEXT_DIM
)

assert predicted_encoder_attention.shape == (
    B,
    MAX_IMAGES_PER_CASE + 59
)

assert torch.isfinite(
    predicted_encoder_prefix
).all()


print("\n" + "=" * 60)
print("V4-3B STATUS: PASS")
print("=" * 60)

V4-3B — MASKED VISUAL REPRESENTATION
Image mask: (4, 8)
Num valid images: [5, 8, 8, 5]
Masked visual representation: (4, 512)
Predicted structured tokens: (4, 59, 512) torch.float32
Predicted encoder prefix: (4, 67, 512)
Predicted encoder dtype: torch.bfloat16

V4-3B STATUS: PASS


In [28]:
# ============================================================
# V4-3C — ASSEMBLE V4 TRAINING MODEL
# ============================================================

print("=" * 60)
print("V4-3C — ASSEMBLE V4 TRAINING MODEL")
print("=" * 60)


# ------------------------------------------------------------
# Add structured heads into V4 model
# ------------------------------------------------------------

model.structured_heads = structured_heads


# ------------------------------------------------------------
# Verify all trainable components
# ------------------------------------------------------------

components = {
    "ViT": model.vision,
    "mT5": model.mt5,
    "Projector": model.projector,
    "Conditioner": model.conditioner,
    "StructuredHeads": model.structured_heads,
}


# ------------------------------------------------------------
# Parameter statistics
# ------------------------------------------------------------

total_params = 0
trainable_params = 0

print("\nParameter groups:")

for name, module in components.items():

    total = sum(
        p.numel()
        for p in module.parameters()
    )

    trainable = sum(
        p.numel()
        for p in module.parameters()
        if p.requires_grad
    )

    total_params += total
    trainable_params += trainable

    print(
        f"{name:18s} "
        f"{total:,} total | "
        f"{trainable:,} trainable"
    )


print("\nTotal parameters:")
print(f"{total_params:,}")

print("Trainable parameters:")
print(f"{trainable_params:,}")


# ------------------------------------------------------------
# Sanity check
# ------------------------------------------------------------

assert trainable_params > 0

assert (
    sum(
        p.numel()
        for p in model.parameters()
    )
    >= trainable_params
)


print("\n" + "=" * 60)
print("V4-3C STATUS: PASS")
print("=" * 60)


V4-3C — ASSEMBLE V4 TRAINING MODEL

Parameter groups:
ViT                86,389,248 total | 86,389,248 trainable
mT5                300,176,768 total | 300,176,768 trainable
Projector          393,728 total | 393,728 trainable
Conditioner        35,840 total | 35,840 trainable
StructuredHeads    30,267 total | 30,267 trainable

Total parameters:
387,025,851
Trainable parameters:
387,025,851

V4-3C STATUS: PASS


In [29]:
# ============================================================
# V4-3D — LOSS DRY-RUN
# ============================================================

import torch
import torch.nn.functional as F

print("=" * 60)
print("V4-3D — LOSS DRY-RUN")
print("=" * 60)


# ------------------------------------------------------------
# 1. Structured ground truth
# ------------------------------------------------------------

concept_target = concept_values.float()
negative_target = negative_values.float()
attribute_target = attribute_values.float()


# ------------------------------------------------------------
# 2. Structured prediction loss
# ------------------------------------------------------------

concept_loss = F.binary_cross_entropy_with_logits(
    concept_logits.float(),
    concept_target
)

negative_loss = F.binary_cross_entropy_with_logits(
    negative_logits.float(),
    negative_target
)

attribute_loss = F.binary_cross_entropy_with_logits(
    attribute_logits.float(),
    attribute_target
)


# ------------------------------------------------------------
# 3. Report loss
# ------------------------------------------------------------

report_loss = loss


# ------------------------------------------------------------
# 4. Total loss
# ------------------------------------------------------------

total_loss = (
    report_loss
    + 0.5 * concept_loss
    + 0.5 * negative_loss
    + 0.25 * attribute_loss
)


# ------------------------------------------------------------
# 5. Print
# ------------------------------------------------------------

print(
    f"Report loss:      {report_loss.item():.6f}"
)

print(
    f"Concept loss:     {concept_loss.item():.6f}"
)

print(
    f"Negative loss:    {negative_loss.item():.6f}"
)

print(
    f"Attribute loss:   {attribute_loss.item():.6f}"
)

print(
    f"Total V4 loss:    {total_loss.item():.6f}"
)


# ------------------------------------------------------------
# 6. Sanity checks
# ------------------------------------------------------------

assert torch.isfinite(
    concept_loss
)

assert torch.isfinite(
    negative_loss
)

assert torch.isfinite(
    attribute_loss
)

assert torch.isfinite(
    report_loss
)

assert torch.isfinite(
    total_loss
)


print("\n" + "=" * 60)
print("V4-3D STATUS: PASS")
print("=" * 60)

V4-3D — LOSS DRY-RUN
Report loss:      1.279810
Concept loss:     0.692734
Negative loss:    0.772338
Attribute loss:   0.783259
Total V4 loss:    2.208160

V4-3D STATUS: PASS


In [30]:
# ============================================================
# V4-3E — BACKWARD DRY-RUN
# ============================================================

print("=" * 60)
print("V4-3E — BACKWARD DRY-RUN")
print("=" * 60)


# ------------------------------------------------------------
# 1. Clear old gradients
# ------------------------------------------------------------

model.zero_grad(set_to_none=True)


# ------------------------------------------------------------
# 2. Backward
# ------------------------------------------------------------

total_loss.backward()


# ------------------------------------------------------------
# 3. Gradient statistics
# ------------------------------------------------------------

def grad_norm(module):

    grads = [
        p.grad.detach()
        for p in module.parameters()
        if p.grad is not None
    ]

    if not grads:
        return 0.0

    return torch.sqrt(
        sum(
            (g.float() ** 2).sum()
            for g in grads
        )
    ).item()


print(
    "ViT grad norm:",
    grad_norm(model.vision)
)

print(
    "mT5 grad norm:",
    grad_norm(model.mt5)
)

print(
    "Projector grad norm:",
    grad_norm(model.projector)
)

print(
    "Conditioner grad norm:",
    grad_norm(model.conditioner)
)

print(
    "Structured heads grad norm:",
    grad_norm(model.structured_heads)
)


# ------------------------------------------------------------
# 4. Check all trainable components received gradients
# ------------------------------------------------------------

for name, module in [
    ("vision", model.vision),
    ("mt5", model.mt5),
    ("projector", model.projector),
    ("conditioner", model.conditioner),
    ("structured_heads", model.structured_heads),
]:

    has_grad = any(
        p.grad is not None
        for p in module.parameters()
        if p.requires_grad
    )

    assert has_grad, (
        f"No gradient found for {name}"
    )


# ------------------------------------------------------------
# 5. Check gradient finiteness
# ------------------------------------------------------------

for name, module in [
    ("vision", model.vision),
    ("mt5", model.mt5),
    ("projector", model.projector),
    ("conditioner", model.conditioner),
    ("structured_heads", model.structured_heads),
]:

    for p in module.parameters():

        if p.grad is not None:

            assert torch.isfinite(
                p.grad
            ).all(), (
                f"Non-finite gradient: {name}"
            )


print("\n" + "=" * 60)
print("V4-3E STATUS: PASS")
print("=" * 60)
print("Backward completed.")
print("No optimizer step performed.")

V4-3E — BACKWARD DRY-RUN
ViT grad norm: 1.5232455730438232
mT5 grad norm: 13.879461288452148
Projector grad norm: 2.02043080329895
Conditioner grad norm: 1.2451070547103882
Structured heads grad norm: 2.359485149383545

V4-3E STATUS: PASS
Backward completed.
No optimizer step performed.


In [31]:
# ============================================================
# V4-4 — OFFICIAL TRAINING
# ============================================================

import os
import json
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from pathlib import Path
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR


print("=" * 60)
print("V4-4 — OFFICIAL TRAINING")
print("=" * 60)


# ============================================================
# 1. CONFIG
# ============================================================

V4_DIR = Path(
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "baseline_model_v4"
)

CKPT_DIR = V4_DIR / "checkpoints"
METRIC_DIR = V4_DIR / "metrics"

CKPT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

METRIC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LAST_CKPT = CKPT_DIR / "last.pt"
BEST_CKPT = CKPT_DIR / "best.pt"
HISTORY_FILE = METRIC_DIR / "training_history.csv"

EPOCHS = 5
ACCUM_STEPS = 4
GRAD_CLIP = 1.0

VISION_LR = 1e-5
PROJECTOR_LR = 1e-4
CONDITIONER_LR = 1e-4
STRUCTURED_LR = 1e-4
MT5_LR = 5e-5

WEIGHT_DECAY = 0.01

BEST_VAL = float("inf")
START_EPOCH = 1


# ============================================================
# 2. OPTIMIZER
# ============================================================

optimizer = AdamW(
    [
        {
            "params": model.vision.parameters(),
            "lr": VISION_LR,
        },
        {
            "params": model.projector.parameters(),
            "lr": PROJECTOR_LR,
        },
        {
            "params": model.conditioner.parameters(),
            "lr": CONDITIONER_LR,
        },
        {
            "params": model.structured_heads.parameters(),
            "lr": STRUCTURED_LR,
        },
        {
            "params": model.mt5.parameters(),
            "lr": MT5_LR,
        },
    ],
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 3. SCHEDULER
# ============================================================

steps_per_epoch = math.ceil(
    len(train_loader) / ACCUM_STEPS
)

total_steps = (
    steps_per_epoch * EPOCHS
)

scheduler = CosineAnnealingLR(
    optimizer,
    T_max=total_steps
)


# ============================================================
# 4. RESUME
# ============================================================

history = []

if HISTORY_FILE.exists():

    history_df = pd.read_csv(
        HISTORY_FILE
    )

    if len(history_df) > 0:

        history = history_df.to_dict(
            "records"
        )

        print(
            "Existing history:",
            len(history),
            "epochs"
        )


if LAST_CKPT.exists():

    print(
        "\nResuming from:",
        LAST_CKPT
    )

    checkpoint = torch.load(
        LAST_CKPT,
        map_location="cpu",
        weights_only=False
    )

    model.load_state_dict(
        checkpoint["model_state"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler_state"]
    )

    START_EPOCH = (
        checkpoint["epoch"] + 1
    )

    BEST_VAL = checkpoint.get(
        "best_val",
        BEST_VAL
    )

    if "rng_state" in checkpoint:

        torch.set_rng_state(
            checkpoint["rng_state"]
        )

    if "cuda_rng_state" in checkpoint:

        torch.cuda.set_rng_state_all(
            checkpoint["cuda_rng_state"]
        )

    print(
        "Resume epoch:",
        START_EPOCH
    )

    print(
        "Best validation loss:",
        BEST_VAL
    )

else:

    print(
        "\nNo checkpoint found."
    )

    print(
        "Starting from epoch 1."
    )


# ============================================================
# 5. TRAINING FUNCTION
# ============================================================

def run_training_epoch():

    model.train()

    total_report = 0.0
    total_concept = 0.0
    total_negative = 0.0
    total_attribute = 0.0
    total_loss = 0.0

    optimizer.zero_grad(
        set_to_none=True
    )

    num_batches = len(train_loader)

    for batch_idx, batch in enumerate(
        train_loader
    ):

        case_ids = batch["case_id"]
        images = batch["images"]
        targets = batch["target_text"]

        B = len(case_ids)

        # ----------------------------------------------------
        # Visual encoding
        # ----------------------------------------------------

        visual_list = []

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16
        ):

            for case_images in images:

                x = torch.stack(
                    case_images
                ).to(
                    device,
                    non_blocking=True
                )

                vision_out = model.vision(
                    pixel_values=x
                )

                cls = (
                    vision_out
                    .last_hidden_state[:, 0, :]
                )

                projected = model.projector(
                    cls
                )

                visual_list.append(
                    projected
                )

        # ----------------------------------------------------
        # Pad visual tokens
        # ----------------------------------------------------

        visual_tokens = torch.zeros(
            B,
            MAX_IMAGES_PER_CASE,
            TEXT_DIM,
            device=device,
            dtype=visual_list[0].dtype
        )

        image_mask = torch.zeros(
            B,
            MAX_IMAGES_PER_CASE,
            device=device,
            dtype=torch.long
        )

        for i, tokens in enumerate(
            visual_list
        ):

            n = tokens.shape[0]

            visual_tokens[
                i,
                :n
            ] = tokens

            image_mask[
                i,
                :n
            ] = 1

        # ----------------------------------------------------
        # Masked visual representation
        # ----------------------------------------------------

        mask_float = image_mask.to(
            visual_tokens.dtype
        )

        visual_repr = (
            visual_tokens
            * mask_float.unsqueeze(-1)
        ).sum(dim=1) / (
            mask_float.sum(
                dim=1,
                keepdim=True
            ).clamp(min=1)
        )

        # ----------------------------------------------------
        # Structured predictions
        # ----------------------------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16
        ):

            (
                concept_logits,
                negative_logits,
                attribute_logits
            ) = model.structured_heads(
                visual_repr
            )

        # ----------------------------------------------------
        # Structured ground truth
        # ----------------------------------------------------

        structured_rows = (
            structured_df.loc[
                case_ids
            ]
        )

        concept_target = torch.tensor(
            structured_rows[
                concept_cols
            ].values,
            dtype=torch.float32,
            device=device
        )

        negative_target = torch.tensor(
            structured_rows[
                negative_cols
            ].values,
            dtype=torch.float32,
            device=device
        )

        attribute_target = torch.tensor(
            structured_rows[
                attribute_cols
            ].values,
            dtype=torch.float32,
            device=device
        )

        # ----------------------------------------------------
        # Structured losses
        # ----------------------------------------------------

        concept_loss = (
            F.binary_cross_entropy_with_logits(
                concept_logits.float(),
                concept_target
            )
        )

        negative_loss = (
            F.binary_cross_entropy_with_logits(
                negative_logits.float(),
                negative_target
            )
        )

        attribute_loss = (
            F.binary_cross_entropy_with_logits(
                attribute_logits.float(),
                attribute_target
            )
        )

        # ----------------------------------------------------
        # Predicted soft structured tokens
        # ----------------------------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16
        ):

            concept_probs = torch.sigmoid(
                concept_logits
            )

            negative_probs = torch.sigmoid(
                negative_logits
            )

            attribute_probs = torch.sigmoid(
                attribute_logits
            )

            structured_tokens = (
                model.conditioner(
                    concept_probs,
                    negative_probs,
                    attribute_probs
                )[0]
            )

        structured_tokens = (
            structured_tokens.to(
                visual_tokens.dtype
            )
        )

        # ----------------------------------------------------
        # Encoder prefix
        # ----------------------------------------------------

        encoder_prefix = torch.cat(
            [
                visual_tokens,
                structured_tokens
            ],
            dim=1
        )

        encoder_attention = torch.cat(
            [
                image_mask,
                torch.ones(
                    B,
                    59,
                    device=device,
                    dtype=torch.long
                )
            ],
            dim=1
        )

        # ----------------------------------------------------
        # Report target
        # ----------------------------------------------------

        tokenized = tokenizer(
            targets,
            padding=True,
            truncation=True,
            max_length=MAX_TARGET_LENGTH,
            return_tensors="pt"
        )

        labels = tokenized.input_ids.to(
            device
        )

        labels[
            labels == tokenizer.pad_token_id
        ] = -100

        # ----------------------------------------------------
        # Report generation loss
        # ----------------------------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16
        ):

            report_outputs = model.mt5(
                inputs_embeds=encoder_prefix,
                attention_mask=encoder_attention,
                labels=labels
            )

            report_loss = report_outputs.loss

            batch_loss = (
                report_loss
                + 0.5 * concept_loss
                + 0.5 * negative_loss
                + 0.25 * attribute_loss
            )

        # ----------------------------------------------------
        # Gradient accumulation
        # ----------------------------------------------------

        scaled_loss = (
            batch_loss / ACCUM_STEPS
        )

        scaled_loss.backward()

        should_step = (
            (batch_idx + 1) % ACCUM_STEPS == 0
            or
            (batch_idx + 1) == num_batches
        )

        if should_step:

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP
            )

            optimizer.step()

            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )

        # ----------------------------------------------------
        # Accumulate metrics
        # ----------------------------------------------------

        total_report += (
            report_loss.detach().item()
        )

        total_concept += (
            concept_loss.detach().item()
        )

        total_negative += (
            negative_loss.detach().item()
        )

        total_attribute += (
            attribute_loss.detach().item()
        )

        total_loss += (
            batch_loss.detach().item()
        )

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (
            (batch_idx + 1) % 200 == 0
            or
            (batch_idx + 1) == num_batches
        ):

            print(
                f"  batch "
                f"{batch_idx + 1}/"
                f"{num_batches} | "
                f"loss "
                f"{batch_loss.item():.4f}"
            )

    n = num_batches

    return {
        "train_loss": total_loss / n,
        "train_report_loss": total_report / n,
        "train_concept_loss": total_concept / n,
        "train_negative_loss": total_negative / n,
        "train_attribute_loss": total_attribute / n,
    }


# ============================================================
# 6. VALIDATION FUNCTION
# ============================================================

@torch.no_grad()
def run_validation():

    model.eval()

    total_report = 0.0
    total_concept = 0.0
    total_negative = 0.0
    total_attribute = 0.0
    total_loss = 0.0

    for batch in val_loader:

        case_ids = batch["case_id"]
        images = batch["images"]
        targets = batch["target_text"]

        B = len(case_ids)

        # ----------------------------------------------------
        # Visual encoding
        # ----------------------------------------------------

        visual_list = []

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16
        ):

            for case_images in images:

                x = torch.stack(
                    case_images
                ).to(
                    device,
                    non_blocking=True
                )

                vision_out = model.vision(
                    pixel_values=x
                )

                cls = (
                    vision_out
                    .last_hidden_state[:, 0, :]
                )

                projected = model.projector(
                    cls
                )

                visual_list.append(
                    projected
                )

        visual_tokens = torch.zeros(
            B,
            MAX_IMAGES_PER_CASE,
            TEXT_DIM,
            device=device,
            dtype=visual_list[0].dtype
        )

        image_mask = torch.zeros(
            B,
            MAX_IMAGES_PER_CASE,
            device=device,
            dtype=torch.long
        )

        for i, tokens in enumerate(
            visual_list
        ):

            n = tokens.shape[0]

            visual_tokens[
                i,
                :n
            ] = tokens

            image_mask[
                i,
                :n
            ] = 1

        # ----------------------------------------------------
        # Masked representation
        # ----------------------------------------------------

        mask_float = image_mask.to(
            visual_tokens.dtype
        )

        visual_repr = (
            visual_tokens
            * mask_float.unsqueeze(-1)
        ).sum(dim=1) / (
            mask_float.sum(
                dim=1,
                keepdim=True
            ).clamp(min=1)
        )

        # ----------------------------------------------------
        # Structured predictions
        # ----------------------------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16
        ):

            (
                concept_logits,
                negative_logits,
                attribute_logits
            ) = model.structured_heads(
                visual_repr
            )

        structured_rows = (
            structured_df.loc[
                case_ids
            ]
        )

        concept_target = torch.tensor(
            structured_rows[
                concept_cols
            ].values,
            dtype=torch.float32,
            device=device
        )

        negative_target = torch.tensor(
            structured_rows[
                negative_cols
            ].values,
            dtype=torch.float32,
            device=device
        )

        attribute_target = torch.tensor(
            structured_rows[
                attribute_cols
            ].values,
            dtype=torch.float32,
            device=device
        )

        concept_loss = (
            F.binary_cross_entropy_with_logits(
                concept_logits.float(),
                concept_target
            )
        )

        negative_loss = (
            F.binary_cross_entropy_with_logits(
                negative_logits.float(),
                negative_target
            )
        )

        attribute_loss = (
            F.binary_cross_entropy_with_logits(
                attribute_logits.float(),
                attribute_target
            )
        )

        # ----------------------------------------------------
        # Predicted structured tokens
        # ----------------------------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16
        ):

            structured_tokens = (
                model.conditioner(
                    torch.sigmoid(
                        concept_logits
                    ),
                    torch.sigmoid(
                        negative_logits
                    ),
                    torch.sigmoid(
                        attribute_logits
                    )
                )[0]
            )

        structured_tokens = (
            structured_tokens.to(
                visual_tokens.dtype
            )
        )

        encoder_prefix = torch.cat(
            [
                visual_tokens,
                structured_tokens
            ],
            dim=1
        )

        encoder_attention = torch.cat(
            [
                image_mask,
                torch.ones(
                    B,
                    59,
                    device=device,
                    dtype=torch.long
                )
            ],
            dim=1
        )

        # ----------------------------------------------------
        # Report loss
        # ----------------------------------------------------

        tokenized = tokenizer(
            targets,
            padding=True,
            truncation=True,
            max_length=MAX_TARGET_LENGTH,
            return_tensors="pt"
        )

        labels = tokenized.input_ids.to(
            device
        )

        labels[
            labels == tokenizer.pad_token_id
        ] = -100

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16
        ):

            report_outputs = model.mt5(
                inputs_embeds=encoder_prefix,
                attention_mask=encoder_attention,
                labels=labels
            )

        report_loss = report_outputs.loss

        batch_loss = (
            report_loss
            + 0.5 * concept_loss
            + 0.5 * negative_loss
            + 0.25 * attribute_loss
        )

        total_report += (
            report_loss.item()
        )

        total_concept += (
            concept_loss.item()
        )

        total_negative += (
            negative_loss.item()
        )

        total_attribute += (
            attribute_loss.item()
        )

        total_loss += (
            batch_loss.item()
        )

    n = len(val_loader)

    return {
        "val_loss": total_loss / n,
        "val_report_loss": total_report / n,
        "val_concept_loss": total_concept / n,
        "val_negative_loss": total_negative / n,
        "val_attribute_loss": total_attribute / n,
    }


# ============================================================
# 7. TRAIN
# ============================================================

for epoch in range(
    START_EPOCH,
    EPOCHS + 1
):

    print("\n")
    print("=" * 60)
    print(
        f"EPOCH {epoch}/{EPOCHS}"
    )
    print("=" * 60)

    train_metrics = run_training_epoch()

    val_metrics = run_validation()

    row = {
        "epoch": epoch,
        **train_metrics,
        **val_metrics,
        "lr_vision": optimizer.param_groups[0]["lr"],
        "lr_projector": optimizer.param_groups[1]["lr"],
        "lr_conditioner": optimizer.param_groups[2]["lr"],
        "lr_structured": optimizer.param_groups[3]["lr"],
        "lr_mt5": optimizer.param_groups[4]["lr"],
    }

    history.append(row)

    history_df = pd.DataFrame(
        history
    )

    history_df.to_csv(
        HISTORY_FILE,
        index=False
    )

    # --------------------------------------------------------
    # Save last checkpoint
    # --------------------------------------------------------

    checkpoint = {
        "epoch": epoch,

        "model_state":
            model.state_dict(),

        "optimizer_state":
            optimizer.state_dict(),

        "scheduler_state":
            scheduler.state_dict(),

        "best_val":
            min(
                BEST_VAL,
                val_metrics["val_loss"]
            ),

        "rng_state":
            torch.get_rng_state(),

        "cuda_rng_state":
            torch.cuda.get_rng_state_all(),

        "history":
            history,
    }

    torch.save(
        checkpoint,
        LAST_CKPT
    )

    # --------------------------------------------------------
    # Save epoch checkpoint
    # --------------------------------------------------------

    torch.save(
        checkpoint,
        CKPT_DIR / f"epoch_{epoch:02d}.pt"
    )

    # --------------------------------------------------------
    # Best checkpoint
    # --------------------------------------------------------

    if val_metrics["val_loss"] < BEST_VAL:

        BEST_VAL = val_metrics["val_loss"]

        checkpoint["best_val"] = BEST_VAL

        torch.save(
            checkpoint,
            BEST_CKPT
        )

        best_marker = "  <-- BEST"

    else:

        best_marker = ""

    # --------------------------------------------------------
    # Print metrics
    # --------------------------------------------------------

    print(
        f"\nEpoch {epoch}:"
    )

    print(
        f"  Train total:     "
        f"{train_metrics['train_loss']:.6f}"
    )

    print(
        f"  Train report:    "
        f"{train_metrics['train_report_loss']:.6f}"
    )

    print(
        f"  Val total:       "
        f"{val_metrics['val_loss']:.6f}"
    )

    print(
        f"  Val report:      "
        f"{val_metrics['val_report_loss']:.6f}"
        f"{best_marker}"
    )

    print(
        f"  Val concept:     "
        f"{val_metrics['val_concept_loss']:.6f}"
    )

    print(
        f"  Val negative:    "
        f"{val_metrics['val_negative_loss']:.6f}"
    )

    print(
        f"  Val attribute:   "
        f"{val_metrics['val_attribute_loss']:.6f}"
    )

    print(
        f"  GPU memory:      "
        f"{torch.cuda.max_memory_allocated() / 1e9:.2f} GB"
    )


print("\n" + "=" * 60)
print("V4-4 TRAINING COMPLETE")
print("=" * 60)

print(
    "Best validation loss:",
    BEST_VAL
)

print(
    "History:",
    HISTORY_FILE
)

print(
    "Best checkpoint:",
    BEST_CKPT
)

V4-4 — OFFICIAL TRAINING

No checkpoint found.
Starting from epoch 1.


EPOCH 1/5
  batch 200/1535 | loss 0.5273
  batch 400/1535 | loss 1.1254
  batch 600/1535 | loss 0.7350
  batch 800/1535 | loss 0.5148
  batch 1000/1535 | loss 0.9902
  batch 1200/1535 | loss 0.7139
  batch 1400/1535 | loss 0.6107
  batch 1535/1535 | loss 0.4463


NameError: name 'val_loader' is not defined

In [32]:
# ============================================================
# V4-4A — REBUILD VALIDATION LOADER
# ============================================================

print("=" * 60)
print("V4-4A — REBUILD VALIDATION LOADER")
print("=" * 60)

val_dataset = V4EndoscopyDataset(
    cases_df=v4_manifest,
    images_df=image_df,
    split="val",
    max_images=MAX_IMAGES_PER_CASE,
    transform=image_transform,
    training=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=v4_collate_fn
)

print(
    "Validation cases:",
    len(val_dataset)
)

print(
    "Validation batches:",
    len(val_loader)
)

val_batch = next(iter(val_loader))

print(
    "Sample batch size:",
    len(val_batch["case_id"])
)

print(
    "Sample images:",
    val_batch["num_images"]
)

print("\n" + "=" * 60)
print("V4-4A STATUS: PASS")
print("=" * 60)

V4-4A — REBUILD VALIDATION LOADER
Validation cases: 756
Validation batches: 189
Sample batch size: 4
Sample images: [6, 8, 6, 8]

V4-4A STATUS: PASS


In [33]:
# ============================================================
# V4-4B — VALIDATE COMPLETED EPOCH 1
# ============================================================

print("=" * 60)
print("V4-4B — VALIDATE EPOCH 1")
print("=" * 60)

val_metrics = run_validation()

print("\nEpoch 1 validation:")

for key, value in val_metrics.items():
    print(
        f"{key}: {value:.6f}"
    )

V4-4B — VALIDATE EPOCH 1

Epoch 1 validation:
val_loss: 0.698360
val_report_loss: 0.527215
val_concept_loss: 0.066259
val_negative_loss: 0.058909
val_attribute_loss: 0.434243


In [8]:
# ============================================================
# V4-4D PATCH 3 — CONFIRM CURRENT RESUME POINT
# ============================================================

assert START_EPOCH == 3
assert len(optimizer.state) > 0
assert len(history) >= 2

print("=" * 60)
print("V4 RESUME STATE: PASS")
print("=" * 60)

print("Completed epoch :", v4_checkpoint["epoch"])
print("Next epoch      :", START_EPOCH)
print("Best val loss   :", BEST_VAL)
print("History rows    :", len(history))

print("\nReady to continue from Epoch 3.")

V4 RESUME STATE: PASS
Completed epoch : 2
Next epoch      : 3
Best val loss   : 0.6521643281140656
History rows    : 2

Ready to continue from Epoch 3.


In [9]:
# ============================================================
# V4-5 — REBUILD TRAIN / VALIDATION FUNCTIONS
# ============================================================

print("=" * 60)
print("V4-5 — REBUILD TRAINING FUNCTIONS")
print("=" * 60)

GRAD_ACCUM = 4
MAX_GRAD_NORM = 1.0
USE_BF16 = True


def compute_v4_forward(batch):
    images_list = batch["images"]
    num_images = batch["num_images"]
    targets = batch["target_text"]

    B = len(images_list)

    # --------------------------------------------------------
    # Visual encoding
    # --------------------------------------------------------

    visual_tokens_list = []

    for imgs in images_list:
        imgs = imgs.to(
            device,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=USE_BF16
        ):
            vit_out = model.vision(
                pixel_values=imgs
            )

            cls_tokens = vit_out.last_hidden_state[:, 0, :]

            projected = model.projector(
                cls_tokens
            )

        visual_tokens_list.append(projected)

    N = max(num_images)

    visual_tokens = torch.zeros(
        B,
        N,
        512,
        device=device,
        dtype=torch.bfloat16
    )

    visual_attention = torch.zeros(
        B,
        N,
        device=device,
        dtype=torch.long
    )

    for i, tokens in enumerate(
        visual_tokens_list
    ):
        n = tokens.shape[0]

        visual_tokens[i, :n] = tokens
        visual_attention[i, :n] = 1

    # --------------------------------------------------------
    # Masked mean visual representation
    # --------------------------------------------------------

    mask = visual_attention.unsqueeze(-1).to(
        visual_tokens.dtype
    )

    visual_repr = (
        (visual_tokens * mask).sum(dim=1)
        /
        mask.sum(dim=1).clamp(min=1)
    )

    # --------------------------------------------------------
    # Structured prediction
    # --------------------------------------------------------

    concept_logits = (
        model.structured_heads.concept(
            visual_repr.float()
        )
    )

    negative_logits = (
        model.structured_heads.negative(
            visual_repr.float()
        )
    )

    attribute_logits = (
        model.structured_heads.attribute(
            visual_repr.float()
        )
    )

    concept_probs = torch.sigmoid(
        concept_logits
    )

    negative_probs = torch.sigmoid(
        negative_logits
    )

    attribute_probs = torch.sigmoid(
        attribute_logits
    )

    # --------------------------------------------------------
    # Predicted structured tokens
    # --------------------------------------------------------

    structured_tokens, structured_attention = (
        model.conditioner(
            concept_probs,
            negative_probs,
            attribute_probs
        )
    )

    structured_tokens = structured_tokens.to(
        torch.bfloat16
    )

    structured_attention = structured_attention.to(
        device
    )

    # --------------------------------------------------------
    # Combine visual + predicted structured tokens
    # --------------------------------------------------------

    encoder_prefix = torch.cat(
        [
            visual_tokens,
            structured_tokens
        ],
        dim=1
    )

    encoder_attention = torch.cat(
        [
            visual_attention,
            structured_attention
        ],
        dim=1
    )

    # --------------------------------------------------------
    # Tokenize reports
    # --------------------------------------------------------

    tokenized = tokenizer(
        targets,
        padding=True,
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
        return_tensors="pt"
    )

    labels = tokenized.input_ids.to(
        device
    )

    labels[
        labels == tokenizer.pad_token_id
    ] = -100

    # --------------------------------------------------------
    # mT5
    # --------------------------------------------------------

    outputs = model.mt5(
        inputs_embeds=encoder_prefix,
        attention_mask=encoder_attention,
        labels=labels
    )

    report_loss = outputs.loss

    return {
        "report_loss": report_loss,
        "concept_logits": concept_logits,
        "negative_logits": negative_logits,
        "attribute_logits": attribute_logits,
        "labels": labels,
        "logits": outputs.logits,
    }


def compute_v4_loss(
    outputs,
    concept_targets,
    negative_targets,
    attribute_targets
):

    concept_loss = torch.nn.functional.binary_cross_entropy_with_logits(
        outputs["concept_logits"],
        concept_targets
    )

    negative_loss = torch.nn.functional.binary_cross_entropy_with_logits(
        outputs["negative_logits"],
        negative_targets
    )

    attribute_loss = torch.nn.functional.binary_cross_entropy_with_logits(
        outputs["attribute_logits"],
        attribute_targets
    )

    total_loss = (
        outputs["report_loss"]
        + 0.5 * concept_loss
        + 0.5 * negative_loss
        + 0.25 * attribute_loss
    )

    return (
        total_loss,
        concept_loss,
        negative_loss,
        attribute_loss
    )


print("V4 training functions rebuilt.")
print("STATUS: PASS")

V4-5 — REBUILD TRAINING FUNCTIONS
V4 training functions rebuilt.
STATUS: PASS


In [11]:
# ============================================================
# V4-5 PATCH — FIX VARIABLE-LENGTH IMAGE LIST
# ============================================================

def compute_v4_forward(batch):

    images_list = batch["images"]
    num_images = batch["num_images"]
    targets = batch["target_text"]

    B = len(images_list)

    visual_tokens_list = []

    # --------------------------------------------------------
    # Visual encoding
    # --------------------------------------------------------

    for imgs in images_list:

        # Current collate_fn may return:
        # [tensor, tensor, tensor, ...]
        # instead of one stacked tensor.

        if isinstance(imgs, list):
            imgs = torch.stack(imgs, dim=0)

        imgs = imgs.to(
            device,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=USE_BF16
        ):

            vit_out = model.vision(
                pixel_values=imgs
            )

            cls_tokens = (
                vit_out.last_hidden_state[:, 0, :]
            )

            projected = model.projector(
                cls_tokens
            )

        visual_tokens_list.append(
            projected
        )

    # --------------------------------------------------------
    # Pad variable number of images
    # --------------------------------------------------------

    N = max(num_images)

    visual_tokens = torch.zeros(
        B,
        N,
        512,
        device=device,
        dtype=torch.bfloat16
    )

    visual_attention = torch.zeros(
        B,
        N,
        device=device,
        dtype=torch.long
    )

    for i, tokens in enumerate(
        visual_tokens_list
    ):

        n = tokens.shape[0]

        visual_tokens[
            i,
            :n
        ] = tokens

        visual_attention[
            i,
            :n
        ] = 1

    # --------------------------------------------------------
    # Masked visual representation
    # --------------------------------------------------------

    mask = visual_attention.unsqueeze(-1).to(
        visual_tokens.dtype
    )

    visual_repr = (
        (visual_tokens * mask).sum(dim=1)
        /
        mask.sum(dim=1).clamp(min=1)
    )

    # --------------------------------------------------------
    # Structured prediction heads
    # --------------------------------------------------------

    concept_logits = (
        model.structured_heads.concept(
            visual_repr.float()
        )
    )

    negative_logits = (
        model.structured_heads.negative(
            visual_repr.float()
        )
    )

    attribute_logits = (
        model.structured_heads.attribute(
            visual_repr.float()
        )
    )

    concept_probs = torch.sigmoid(
        concept_logits
    )

    negative_probs = torch.sigmoid(
        negative_logits
    )

    attribute_probs = torch.sigmoid(
        attribute_logits
    )

    # --------------------------------------------------------
    # Predicted structured tokens
    # --------------------------------------------------------

    structured_tokens, structured_attention = (
        model.conditioner(
            concept_probs,
            negative_probs,
            attribute_probs
        )
    )

    structured_tokens = structured_tokens.to(
        torch.bfloat16
    )

    structured_attention = structured_attention.to(
        device
    )

    # --------------------------------------------------------
    # Visual + structured prefix
    # --------------------------------------------------------

    encoder_prefix = torch.cat(
        [
            visual_tokens,
            structured_tokens
        ],
        dim=1
    )

    encoder_attention = torch.cat(
        [
            visual_attention,
            structured_attention
        ],
        dim=1
    )

    # --------------------------------------------------------
    # Target tokenization
    # --------------------------------------------------------

    tokenized = tokenizer(
        targets,
        padding=True,
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
        return_tensors="pt"
    )

    labels = tokenized.input_ids.to(
        device
    )

    labels[
        labels == tokenizer.pad_token_id
    ] = -100

    # --------------------------------------------------------
    # mT5
    # --------------------------------------------------------

    outputs = model.mt5(
        inputs_embeds=encoder_prefix,
        attention_mask=encoder_attention,
        labels=labels
    )

    return {
        "report_loss": outputs.loss,
        "concept_logits": concept_logits,
        "negative_logits": negative_logits,
        "attribute_logits": attribute_logits,
        "labels": labels,
        "logits": outputs.logits,
    }


print("=" * 60)
print("V4-5 PATCH STATUS: PASS")
print("=" * 60)
print("compute_v4_forward() now handles list-of-tensors.")

V4-5 PATCH STATUS: PASS
compute_v4_forward() now handles list-of-tensors.


In [12]:
# ============================================================
# V4 FORWARD TEST — NO TRAINING
# ============================================================

model.eval()

batch = next(iter(train_loader))

with torch.no_grad():

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
        enabled=USE_BF16
    ):

        outputs = compute_v4_forward(
            batch
        )

print("=" * 60)
print("V4 FORWARD TEST")
print("=" * 60)

print(
    "Batch size:",
    len(batch["case_id"])
)

print(
    "Num images:",
    batch["num_images"]
)

print(
    "Report loss:",
    outputs["report_loss"].item()
)

print(
    "Concept logits:",
    outputs["concept_logits"].shape
)

print(
    "Negative logits:",
    outputs["negative_logits"].shape
)

print(
    "Attribute logits:",
    outputs["attribute_logits"].shape
)

print(
    "LM logits:",
    outputs["logits"].shape
)

assert outputs["concept_logits"].shape[1] == 48
assert outputs["negative_logits"].shape[1] == 5
assert outputs["attribute_logits"].shape[1] == 6

print("\nSTATUS: PASS")

V4 FORWARD TEST
Batch size: 4
Num images: [8, 8, 4, 8]
Report loss: 0.28116893768310547
Concept logits: torch.Size([4, 48])
Negative logits: torch.Size([4, 5])
Attribute logits: torch.Size([4, 6])
LM logits: torch.Size([4, 38, 250112])

STATUS: PASS


In [5]:
# ============================================================
# V4-6 PATCH — INSPECT STRUCTURED DF
# ============================================================

print("=" * 60)
print("STRUCTURED DF CHECK")
print("=" * 60)

print("Shape:", structured_df.shape)

print("\nColumns:")
print(
    structured_df.columns.tolist()
)

print("\nIndex name:")
print(
    structured_df.index.name
)

print("\nIndex type:")
print(
    type(structured_df.index)
)

print("\nFirst 3 index values:")
print(
    structured_df.index[:3].tolist()
)

print("\nFirst 3 rows:")
display(
    structured_df.head(3)
)

STRUCTURED DF CHECK


NameError: name 'structured_df' is not defined

In [15]:
# ============================================================
# V4-6 PATCH — STRUCTURED TARGET LOOKUP
# ============================================================

# case_id is already the index of structured_df.
# Keep the frozen V4 ontology columns unchanged.

def get_structured_targets(case_ids):

    target_rows = structured_df.loc[
        case_ids
    ]

    concept_targets = torch.tensor(
        target_rows[
            frozen_concept_columns
        ].values,
        dtype=torch.float32,
        device=device
    )

    negative_targets = torch.tensor(
        target_rows[
            negative_columns
        ].values,
        dtype=torch.float32,
        device=device
    )

    attribute_targets = torch.tensor(
        target_rows[
            attribute_columns
        ].values,
        dtype=torch.float32,
        device=device
    )

    return (
        concept_targets,
        negative_targets,
        attribute_targets
    )


# Test against one batch only
batch = next(iter(train_loader))

(
    concept_targets,
    negative_targets,
    attribute_targets
) = get_structured_targets(
    batch["case_id"]
)

print("=" * 60)
print("STRUCTURED TARGET LOOKUP TEST")
print("=" * 60)

print("Case IDs:", batch["case_id"])
print(
    "Concept targets:",
    concept_targets.shape
)
print(
    "Negative targets:",
    negative_targets.shape
)
print(
    "Attribute targets:",
    attribute_targets.shape
)

assert concept_targets.shape == (4, 48)
assert negative_targets.shape == (4, 5)
assert attribute_targets.shape == (4, 6)

print("\nSTATUS: PASS")

NameError: name 'frozen_concept_columns' is not defined

In [16]:
# ============================================================
# V4-6 PATCH — RESTORE FROZEN ONTOLOGY COLUMNS
# ============================================================

MAPPING_FILE = (
    OUTPUT_DIR.parent
    / "diagnostics"
    / "frozen_concept_ontology_mapping.csv"
)

ontology_map = pd.read_csv(
    MAPPING_FILE
)

print("=" * 60)
print("RESTORING FROZEN ONTOLOGY")
print("=" * 60)

print(
    "Mapping shape:",
    ontology_map.shape
)

print(
    "Dimensions:",
    ontology_map["dimension"].value_counts().to_dict()
)


# ------------------------------------------------------------
# Exact frozen columns
# ------------------------------------------------------------

frozen_concept_columns = (
    ontology_map
    .loc[
        ontology_map["dimension"] == "concept"
    ]
    .sort_values("dimension_index")[
        "column"
    ]
    .tolist()
)

negative_columns = (
    ontology_map
    .loc[
        ontology_map["dimension"] == "negative"
    ]
    .sort_values("dimension_index")[
        "column"
    ]
    .tolist()
)

attribute_columns = (
    ontology_map
    .loc[
        ontology_map["dimension"] == "attribute"
    ]
    .sort_values("dimension_index")[
        "column"
    ]
    .tolist()
)


# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

print(
    "Frozen concepts:",
    len(frozen_concept_columns)
)

print(
    "Frozen negatives:",
    len(negative_columns)
)

print(
    "Frozen attributes:",
    len(attribute_columns)
)

assert len(frozen_concept_columns) == 48
assert len(negative_columns) == 5
assert len(attribute_columns) == 6

missing = (
    set(
        frozen_concept_columns
        + negative_columns
        + attribute_columns
    )
    -
    set(structured_df.columns)
)

assert len(missing) == 0, (
    f"Missing ontology columns: {missing}"
)


print("\nFirst 10 concepts:")
print(
    frozen_concept_columns[:10]
)

print("\nNegative:")
print(
    negative_columns
)

print("\nAttributes:")
print(
    attribute_columns
)

print("\nSTATUS: PASS")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/NoiSoi_Matching/diagnostics/frozen_concept_ontology_mapping.csv'

In [4]:
# ============================================================
# V4-6 PATCH — RESTORE EXACT 48 + 5 + 6 COLUMNS
# ============================================================

# ------------------------------------------------------------
# 48 frozen concepts
# ------------------------------------------------------------

frozen_concept_columns = (
    ontology_map["column"]
    .tolist()
)


# ------------------------------------------------------------
# 5 frozen negative findings
# ------------------------------------------------------------

negative_columns = [
    "negative__no_abnormal_external_middle_ear",
    "negative__no_abnormal_nose_sinus",
    "negative__no_abnormal_ent",
    "negative__no_bleeding",
    "negative__no_foreign_body",
]


# ------------------------------------------------------------
# 6 frozen attributes
# ------------------------------------------------------------

attribute_columns = [
    "attribute__acute",
    "attribute__chronic",
    "attribute__right",
    "attribute__left",
    "attribute__bilateral",
    "attribute__post_surgery",
]


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 60)
print("FROZEN ONTOLOGY RESTORED")
print("=" * 60)

print(
    "Concepts  :",
    len(frozen_concept_columns)
)

print(
    "Negatives :",
    len(negative_columns)
)

print(
    "Attributes:",
    len(attribute_columns)
)

assert len(frozen_concept_columns) == 48
assert len(negative_columns) == 5
assert len(attribute_columns) == 6


all_columns = (
    frozen_concept_columns
    + negative_columns
    + attribute_columns
)

missing = [
    c for c in all_columns
    if c not in structured_df.columns
]

assert not missing, (
    f"Missing columns: {missing}"
)

print("\nFirst 5 concepts:")
print(
    frozen_concept_columns[:5]
)

print("\nNegative:")
print(
    negative_columns
)

print("\nAttributes:")
print(
    attribute_columns
)

print("\nSTATUS: PASS")

NameError: name 'ontology_map' is not defined

In [20]:
# ============================================================
# STRUCTURED TARGET LOOKUP TEST
# ============================================================

def get_structured_targets(case_ids):

    target_rows = structured_df.loc[
        case_ids
    ]

    concept_targets = torch.tensor(
        target_rows[
            frozen_concept_columns
        ].values,
        dtype=torch.float32,
        device=device
    )

    negative_targets = torch.tensor(
        target_rows[
            negative_columns
        ].values,
        dtype=torch.float32,
        device=device
    )

    attribute_targets = torch.tensor(
        target_rows[
            attribute_columns
        ].values,
        dtype=torch.float32,
        device=device
    )

    return (
        concept_targets,
        negative_targets,
        attribute_targets
    )


batch = next(iter(train_loader))

(
    concept_targets,
    negative_targets,
    attribute_targets
) = get_structured_targets(
    batch["case_id"]
)

print("=" * 60)
print("STRUCTURED TARGET LOOKUP TEST")
print("=" * 60)

print(
    "Concept targets  :",
    concept_targets.shape
)

print(
    "Negative targets :",
    negative_targets.shape
)

print(
    "Attribute targets:",
    attribute_targets.shape
)

assert concept_targets.shape == (4, 48)
assert negative_targets.shape == (4, 5)
assert attribute_targets.shape == (4, 6)

print("\nSTATUS: PASS")

STRUCTURED TARGET LOOKUP TEST
Concept targets  : torch.Size([4, 48])
Negative targets : torch.Size([4, 5])
Attribute targets: torch.Size([4, 6])

STATUS: PASS


In [3]:
# ============================================================
# V4-6 FINAL — TRAIN EPOCH 3
# ============================================================

print("=" * 60)
print("V4-6 FINAL — TRAIN EPOCH 3")
print("=" * 60)

EPOCH = 3

model.train()

train_sums = {
    "total": 0.0,
    "report": 0.0,
    "concept": 0.0,
    "negative": 0.0,
    "attribute": 0.0,
}

optimizer.zero_grad(set_to_none=True)

num_train_batches = len(train_loader)

for step, batch in enumerate(train_loader):

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
        enabled=USE_BF16
    ):

        outputs = compute_v4_forward(batch)

        (
            concept_targets,
            negative_targets,
            attribute_targets
        ) = get_structured_targets(
            batch["case_id"]
        )

        (
            loss,
            concept_loss,
            negative_loss,
            attribute_loss
        ) = compute_v4_loss(
            outputs,
            concept_targets,
            negative_targets,
            attribute_targets
        )

        scaled_loss = loss / GRAD_ACCUM

    scaled_loss.backward()

    if (
        (step + 1) % GRAD_ACCUM == 0
        or
        (step + 1) == num_train_batches
    ):

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM
        )

        optimizer.step()
        scheduler.step()

        optimizer.zero_grad(
            set_to_none=True
        )

    train_sums["total"] += loss.item()
    train_sums["report"] += (
        outputs["report_loss"].item()
    )
    train_sums["concept"] += (
        concept_loss.item()
    )
    train_sums["negative"] += (
        negative_loss.item()
    )
    train_sums["attribute"] += (
        attribute_loss.item()
    )

    if (step + 1) % 200 == 0:
        print(
            f"Batch {step+1:4d}/{num_train_batches} "
            f"| loss={loss.item():.4f}"
        )


# ============================================================
# TRAIN METRICS
# ============================================================

train_metrics = {
    k: v / num_train_batches
    for k, v in train_sums.items()
}

print("\nTRAIN")
for k, v in train_metrics.items():
    print(f"{k:10s}: {v:.6f}")


# ============================================================
# VALIDATION
# ============================================================

model.eval()

val_sums = {
    "total": 0.0,
    "report": 0.0,
    "concept": 0.0,
    "negative": 0.0,
    "attribute": 0.0,
}

with torch.no_grad():

    for batch in val_loader:

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=USE_BF16
        ):

            outputs = compute_v4_forward(
                batch
            )

            (
                concept_targets,
                negative_targets,
                attribute_targets
            ) = get_structured_targets(
                batch["case_id"]
            )

            (
                loss,
                concept_loss,
                negative_loss,
                attribute_loss
            ) = compute_v4_loss(
                outputs,
                concept_targets,
                negative_targets,
                attribute_targets
            )

        val_sums["total"] += loss.item()
        val_sums["report"] += (
            outputs["report_loss"].item()
        )
        val_sums["concept"] += (
            concept_loss.item()
        )
        val_sums["negative"] += (
            negative_loss.item()
        )
        val_sums["attribute"] += (
            attribute_loss.item()
        )


num_val_batches = len(val_loader)

val_metrics = {
    k: v / num_val_batches
    for k, v in val_sums.items()
}

print("\nVALIDATION")
for k, v in val_metrics.items():
    print(f"{k:10s}: {v:.6f}")


# ============================================================
# UPDATE HISTORY
# ============================================================

epoch_record = {
    "epoch": EPOCH,
    "train_total": train_metrics["total"],
    "train_report": train_metrics["report"],
    "train_concept": train_metrics["concept"],
    "train_negative": train_metrics["negative"],
    "train_attribute": train_metrics["attribute"],
    "val_total": val_metrics["total"],
    "val_report": val_metrics["report"],
    "val_concept": val_metrics["concept"],
    "val_negative": val_metrics["negative"],
    "val_attribute": val_metrics["attribute"],
}

history = [
    h for h in history
    if h.get("epoch") != EPOCH
]

history.append(epoch_record)

history = sorted(
    history,
    key=lambda x: x["epoch"]
)


# ============================================================
# CHECKPOINT
# ============================================================

is_best = (
    val_metrics["total"] < BEST_VAL
)

if is_best:
    BEST_VAL = val_metrics["total"]

checkpoint = {
    "epoch": EPOCH,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "scheduler_state": scheduler.state_dict(),
    "best_val": BEST_VAL,
    "rng_state": torch.get_rng_state(),
    "cuda_rng_state": torch.cuda.get_rng_state_all(),
    "history": history,
}

torch.save(
    checkpoint,
    CHECKPOINT_DIR / "epoch_03.pt"
)

torch.save(
    checkpoint,
    CHECKPOINT_DIR / "last.pt"
)

if is_best:
    torch.save(
        checkpoint,
        CHECKPOINT_DIR / "best.pt"
    )


# ============================================================
# SAVE HISTORY
# ============================================================

pd.DataFrame(history).to_csv(
    OUTPUT_DIR / "metrics" / "training_history.csv",
    index=False
)


print("\n" + "=" * 60)
print("EPOCH 3 COMPLETE")
print("=" * 60)

print(f"Best val loss : {BEST_VAL:.6f}")
print(f"New best      : {is_best}")
print("Saved         : epoch_03.pt")
print("Saved         : last.pt")

if is_best:
    print("Saved         : best.pt")

print("=" * 60)

V4-6 FINAL — TRAIN EPOCH 3


NameError: name 'model' is not defined

In [7]:
# ============================================================
# V4 — FINAL SELF-CONTAINED RESUME + EPOCH 3
# ============================================================
#
# IMPORTANT:
# The previous cell used the wrong StructuredConditioner.
# This version matches the ACTUAL Epoch-2 checkpoint keys:
#
#   concept_value
#   negative_value
#   attribute_value
#   concept_label
#   negative_label
#   attribute_label
#   type_embedding
#   norm
#
# It resumes ONLY from V4 last.pt.
# ============================================================

import os
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import (
    AutoTokenizer,
    ViTModel,
    MT5ForConditionalGeneration,
)

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/NoiSoi_Matching"
)

V3_DIR = (
    PROJECT_DIR /
    "baseline_model_v3"
)

V4_DIR = (
    PROJECT_DIR /
    "baseline_model_v4"
)

V3_CHECKPOINT = (
    V3_DIR /
    "checkpoints" /
    "best.pt"
)

LAST_CHECKPOINT = (
    V4_DIR /
    "checkpoints" /
    "last.pt"
)

BEST_CHECKPOINT = (
    V4_DIR /
    "checkpoints" /
    "best.pt"
)

CHECKPOINT_DIR = (
    V4_DIR /
    "checkpoints"
)

METRICS_DIR = (
    V4_DIR /
    "metrics"
)

METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. CONFIG
# ============================================================

device = torch.device("cuda")

assert torch.cuda.is_available()

MAX_IMAGES_PER_CASE = 8
IMAGE_SIZE = 224
MAX_TARGET_LENGTH = 96

BATCH_SIZE = 4
GRAD_ACCUM = 4
MAX_GRAD_NORM = 1.0

TOTAL_EPOCHS = 5

LR_VIT = 1e-5
LR_PROJECTOR = 1e-4
LR_CONDITIONER = 1e-4
LR_STRUCTURED = 1e-4
LR_MT5 = 5e-5

WEIGHT_DECAY = 0.01

USE_BF16 = True


print("=" * 60)
print("V4 — EXACT CHECKPOINT RESUME")
print("=" * 60)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)


# ============================================================
# 3. LOAD DATA FILES
# ============================================================

V4_MANIFEST = (
    V4_DIR /
    "v4_training_manifest.csv"
)

IMAGE_MANIFEST = (
    PROJECT_DIR /
    "final_manifest" /
    "final_dataset_manifest.csv"
)

STRUCTURED_FILE = (
    PROJECT_DIR /
    "v3_ontology_v2" /
    "v3_structured_targets_v2_2.csv"
)

ONTOLOGY_FILE = (
    V4_DIR /
    "diagnostics" /
    "frozen_concept_ontology_mapping.csv"
)

for p in [
    V3_CHECKPOINT,
    LAST_CHECKPOINT,
    V4_MANIFEST,
    IMAGE_MANIFEST,
    STRUCTURED_FILE,
    ONTOLOGY_FILE,
]:
    assert p.exists(), str(p)


v4_df = pd.read_csv(
    V4_MANIFEST,
    low_memory=False
)

image_df = pd.read_csv(
    IMAGE_MANIFEST,
    low_memory=False
)

structured_df = pd.read_csv(
    STRUCTURED_FILE,
    low_memory=False
)

if "case_id" in structured_df.columns:

    structured_df = structured_df.set_index(
        "case_id"
    )

print(
    "V4 manifest:",
    v4_df.shape
)

print(
    "Image manifest:",
    image_df.shape
)

print(
    "Structured:",
    structured_df.shape
)


# ============================================================
# 4. FROZEN ONTOLOGY
# ============================================================

ontology_map = pd.read_csv(
    ONTOLOGY_FILE
)

frozen_concept_columns = (
    ontology_map["column"]
    .tolist()
)

negative_columns = [
    "negative__no_abnormal_external_middle_ear",
    "negative__no_abnormal_nose_sinus",
    "negative__no_abnormal_ent",
    "negative__no_bleeding",
    "negative__no_foreign_body",
]

attribute_columns = [
    "attribute__acute",
    "attribute__chronic",
    "attribute__right",
    "attribute__left",
    "attribute__bilateral",
    "attribute__post_surgery",
]

assert len(
    frozen_concept_columns
) == 48

assert len(
    negative_columns
) == 5

assert len(
    attribute_columns
) == 6

for c in (
    frozen_concept_columns
    + negative_columns
    + attribute_columns
):

    assert c in structured_df.columns, c


# ============================================================
# 5. TRANSFORM
# ============================================================

image_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ],
    ),
])


# ============================================================
# 6. DATASET
# ============================================================

class V4Dataset(Dataset):

    def __init__(
        self,
        cases,
        image_manifest,
        training=False,
    ):

        self.cases = cases.reset_index(
            drop=True
        )

        self.training = training

        normal = image_manifest[
            image_manifest[
                "image_status"
            ] == "NORMAL"
        ]

        self.groups = {
            str(k): g[
                "image_path"
            ].tolist()
            for k, g in normal.groupby(
                "case_id"
            )
        }

    def __len__(self):

        return len(self.cases)

    def __getitem__(self, idx):

        row = self.cases.iloc[idx]

        case_id = str(
            row["case_id"]
        )

        paths = self.groups[
            case_id
        ].copy()

        if (
            self.training
            and len(paths)
            > MAX_IMAGES_PER_CASE
        ):

            paths = random.sample(
                paths,
                MAX_IMAGES_PER_CASE
            )

        else:

            paths = paths[
                :MAX_IMAGES_PER_CASE
            ]

        from PIL import Image

        images = []

        for p in paths:

            img = Image.open(
                p
            ).convert("RGB")

            img = image_transform(
                img
            )

            images.append(img)

        return {
            "case_id": case_id,
            "patient_group_id":
                row["patient_group_id"],
            "images": images,
            "num_images":
                len(images),
            "target_text":
                str(row["ket_luan"]),
        }


def collate_fn(batch):

    return {
        "case_id": [
            x["case_id"]
            for x in batch
        ],
        "patient_group_id": [
            x["patient_group_id"]
            for x in batch
        ],
        "images": [
            x["images"]
            for x in batch
        ],
        "num_images": [
            x["num_images"]
            for x in batch
        ],
        "target_text": [
            x["target_text"]
            for x in batch
        ],
    }


# ============================================================
# 7. CASES
# ============================================================

v4_df["ket_luan"] = (
    v4_df["ket_luan"]
    .fillna("")
    .astype(str)
    .str.strip()
)

cases = v4_df[
    v4_df["ket_luan"] != ""
].copy()

cases = cases[
    cases["case_id"].isin(
        structured_df.index
    )
].copy()

train_cases = cases[
    cases["split"] == "train"
].copy()

val_cases = cases[
    cases["split"] == "val"
].copy()

assert len(train_cases) == 6138
assert len(val_cases) == 756


# ============================================================
# 8. LOADERS
# ============================================================

train_dataset = V4Dataset(
    train_cases,
    image_df,
    training=True
)

val_dataset = V4Dataset(
    val_cases,
    image_df,
    training=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn,
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Val batches:",
    len(val_loader)
)


# ============================================================
# 9. TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    "google/mt5-small"
)


# ============================================================
# 10. MODEL COMPONENTS
# ============================================================

class VisualProjector(nn.Module):

    def __init__(
        self,
        in_dim=768,
        out_dim=512
    ):

        super().__init__()

        self.proj = nn.Linear(
            in_dim,
            out_dim
        )

    def forward(self, x):

        return self.proj(x)


class StructuredPredictionHeads(
    nn.Module
):

    def __init__(
        self,
        dim=512
    ):

        super().__init__()

        self.concept = nn.Linear(
            dim,
            48
        )

        self.negative = nn.Linear(
            dim,
            5
        )

        self.attribute = nn.Linear(
            dim,
            6
        )

    def forward(self, x):

        return (
            self.concept(x),
            self.negative(x),
            self.attribute(x)
        )


# ------------------------------------------------------------
# EXACT CONDITIONER FROM EPOCH-2 CHECKPOINT
# ------------------------------------------------------------

class ExactStructuredConditioner(
    nn.Module
):

    def __init__(
        self,
        dim=512,
        n_concepts=48,
        n_negative=5,
        n_attributes=6,
    ):

        super().__init__()

        # Values are scalar probabilities.
        self.concept_value = nn.Linear(
            1,
            dim,
            bias=False
        )

        self.negative_value = nn.Linear(
            1,
            dim,
            bias=False
        )

        self.attribute_value = nn.Linear(
            1,
            dim,
            bias=False
        )

        # One learned embedding for each
        # ontology label.
        self.concept_label = nn.Embedding(
            n_concepts,
            dim
        )

        self.negative_label = nn.Embedding(
            n_negative,
            dim
        )

        self.attribute_label = nn.Embedding(
            n_attributes,
            dim
        )

        # 3 token types:
        # concept / negative / attribute
        self.type_embedding = nn.Embedding(
            3,
            dim
        )

        self.norm = nn.LayerNorm(
            dim
        )

    def forward(
        self,
        concepts,
        negatives,
        attributes
    ):

        B = concepts.shape[0]

        device_ = concepts.device

        concept_ids = torch.arange(
            self.concept_label.num_embeddings,
            device=device_
        )

        negative_ids = torch.arange(
            self.negative_label.num_embeddings,
            device=device_
        )

        attribute_ids = torch.arange(
            self.attribute_label.num_embeddings,
            device=device_
        )

        # ----------------------------------------------------
        # Concept tokens
        # ----------------------------------------------------

        c_value = self.concept_value(
            concepts.unsqueeze(-1)
        )

        c_label = self.concept_label(
            concept_ids
        ).unsqueeze(0)

        c_type = self.type_embedding(
            torch.tensor(
                0,
                device=device_
            )
        ).view(
            1,
            1,
            -1
        )

        c = (
            c_value
            + c_label
            + c_type
        )

        # ----------------------------------------------------
        # Negative tokens
        # ----------------------------------------------------

        n_value = self.negative_value(
            negatives.unsqueeze(-1)
        )

        n_label = self.negative_label(
            negative_ids
        ).unsqueeze(0)

        n_type = self.type_embedding(
            torch.tensor(
                1,
                device=device_
            )
        ).view(
            1,
            1,
            -1
        )

        n = (
            n_value
            + n_label
            + n_type
        )

        # ----------------------------------------------------
        # Attribute tokens
        # ----------------------------------------------------

        a_value = self.attribute_value(
            attributes.unsqueeze(-1)
        )

        a_label = self.attribute_label(
            attribute_ids
        ).unsqueeze(0)

        a_type = self.type_embedding(
            torch.tensor(
                2,
                device=device_
            )
        ).view(
            1,
            1,
            -1
        )

        a = (
            a_value
            + a_label
            + a_type
        )

        tokens = torch.cat(
            [
                c,
                n,
                a
            ],
            dim=1
        )

        tokens = self.norm(
            tokens
        )

        attention = torch.ones(
            B,
            tokens.shape[1],
            dtype=torch.long,
            device=device_
        )

        return (
            tokens,
            attention
        )


class V4Model(nn.Module):

    def __init__(self):

        super().__init__()

        self.vision = ViTModel.from_pretrained(
            "google/vit-base-patch16-224"
        )

        self.mt5 = MT5ForConditionalGeneration.from_pretrained(
            "google/mt5-small"
        )

        self.projector = VisualProjector()

        self.conditioner = (
            ExactStructuredConditioner()
        )

        self.structured_heads = (
            StructuredPredictionHeads()
        )


# ============================================================
# 11. CREATE MODEL
# ============================================================

model = V4Model()


# ============================================================
# 12. LOAD V3 WEIGHTS
# ============================================================

v3 = torch.load(
    V3_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

v3_state = v3[
    "model_state_dict"
]

vision_state = {
    k: v
    for k, v in v3_state.items()
    if k.startswith("vision.")
}

mt5_state = {
    k: v
    for k, v in v3_state.items()
    if k.startswith("mt5.")
}

projector_state = {
    k.replace(
        "projector.",
        "",
        1
    ): v
    for k, v in v3_state.items()
    if k.startswith("projector.")
}

model.vision.load_state_dict(
    vision_state,
    strict=False
)

model.mt5.load_state_dict(
    mt5_state,
    strict=False
)

model.projector.load_state_dict(
    projector_state,
    strict=True
)


# ============================================================
# 13. LOAD V4 EPOCH-2 CHECKPOINT
# ============================================================

v4 = torch.load(
    LAST_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print(
    "\nV4 checkpoint epoch:",
    v4["epoch"]
)

print(
    "Best val:",
    v4["best_val"]
)

assert v4["epoch"] == 2


# ------------------------------------------------------------
# Verify conditioner keys BEFORE loading
# ------------------------------------------------------------

checkpoint_conditioner_keys = sorted([
    k
    for k in v4["model_state"]
    if k.startswith(
        "conditioner."
    )
])

model_conditioner_keys = sorted([
    k
    for k in model.state_dict()
    if k.startswith(
        "conditioner."
    )
])

print(
    "\nCheckpoint conditioner keys:"
)

for k in checkpoint_conditioner_keys:
    print(" ", k)

print(
    "\nModel conditioner keys:"
)

for k in model_conditioner_keys:
    print(" ", k)

assert (
    checkpoint_conditioner_keys
    == model_conditioner_keys
), (
    "Conditioner architecture does not "
    "match checkpoint."
)


# ------------------------------------------------------------
# Load V4
# ------------------------------------------------------------

model.load_state_dict(
    v4["model_state"],
    strict=True
)

print(
    "\nV4 model weights: PASS"
)


# ============================================================
# 14. GPU
# ============================================================

model = model.to(
    device
)


# ============================================================
# 15. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    [
        {
            "params":
                model.vision.parameters(),
            "lr": LR_VIT,
        },
        {
            "params":
                model.projector.parameters(),
            "lr": LR_PROJECTOR,
        },
        {
            "params":
                model.conditioner.parameters(),
            "lr": LR_CONDITIONER,
        },
        {
            "params":
                model.structured_heads.parameters(),
            "lr": LR_STRUCTURED,
        },
        {
            "params":
                model.mt5.parameters(),
            "lr": LR_MT5,
        },
    ],
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 16. SCHEDULER
# ============================================================

steps_per_epoch = math.ceil(
    len(train_loader)
    / GRAD_ACCUM
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=(
        steps_per_epoch
        * TOTAL_EPOCHS
    )
)


# ============================================================
# 17. RESTORE OPTIMIZER + SCHEDULER
# ============================================================

optimizer.load_state_dict(
    v4["optimizer_state"]
)

scheduler.load_state_dict(
    v4["scheduler_state"]
)

BEST_VAL = v4[
    "best_val"
]

history = v4[
    "history"
]

torch.set_rng_state(
    v4["rng_state"]
)

torch.cuda.set_rng_state_all(
    v4["cuda_rng_state"]
)


print(
    "\nOptimizer restored."
)

print(
    "Scheduler restored."
)

print(
    "RNG restored."
)

print(
    "Next epoch:",
    v4["epoch"] + 1
)


# ============================================================
# 18. TARGET HELPER
# ============================================================

def get_structured_targets(
    case_ids
):

    rows = structured_df.loc[
        case_ids
    ]

    concepts = torch.tensor(
        rows[
            frozen_concept_columns
        ].values,
        dtype=torch.float32,
        device=device
    )

    negatives = torch.tensor(
        rows[
            negative_columns
        ].values,
        dtype=torch.float32,
        device=device
    )

    attributes = torch.tensor(
        rows[
            attribute_columns
        ].values,
        dtype=torch.float32,
        device=device
    )

    return (
        concepts,
        negatives,
        attributes
    )


# ============================================================
# 19. FORWARD
# ============================================================

def forward_v4(batch):

    images_list = batch[
        "images"
    ]

    num_images = batch[
        "num_images"
    ]

    targets = batch[
        "target_text"
    ]

    B = len(
        images_list
    )

    visual_list = []

    for imgs in images_list:

        imgs = torch.stack(
            imgs,
            dim=0
        ).to(
            device,
            non_blocking=True
        )

        vit = model.vision(
            pixel_values=imgs
        )

        cls = vit.last_hidden_state[
            :,
            0,
            :
        ]

        visual_list.append(
            model.projector(cls)
        )

    N = max(
        num_images
    )

    visual_tokens = torch.zeros(
        B,
        N,
        512,
        device=device,
        dtype=torch.bfloat16
    )

    visual_attention = torch.zeros(
        B,
        N,
        device=device,
        dtype=torch.long
    )

    for i, x in enumerate(
        visual_list
    ):

        n = x.shape[0]

        visual_tokens[
            i,
            :n
        ] = x

        visual_attention[
            i,
            :n
        ] = 1

    mask = (
        visual_attention
        .unsqueeze(-1)
        .to(
            visual_tokens.dtype
        )
    )

    visual_repr = (
        (
            visual_tokens
            * mask
        ).sum(
            dim=1
        )
        /
        mask.sum(
            dim=1
        ).clamp(
            min=1
        )
    )

    concept_logits = (
        model.structured_heads.concept(
            visual_repr.float()
        )
    )

    negative_logits = (
        model.structured_heads.negative(
            visual_repr.float()
        )
    )

    attribute_logits = (
        model.structured_heads.attribute(
            visual_repr.float()
        )
    )

    concept_probs = torch.sigmoid(
        concept_logits
    )

    negative_probs = torch.sigmoid(
        negative_logits
    )

    attribute_probs = torch.sigmoid(
        attribute_logits
    )

    structured_tokens, structured_attention = (
        model.conditioner(
            concept_probs,
            negative_probs,
            attribute_probs
        )
    )

    structured_tokens = (
        structured_tokens.to(
            torch.bfloat16
        )
    )

    prefix = torch.cat(
        [
            visual_tokens,
            structured_tokens
        ],
        dim=1
    )

    attention = torch.cat(
        [
            visual_attention,
            structured_attention
        ],
        dim=1
    )

    tok = tokenizer(
        targets,
        padding=True,
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
        return_tensors="pt"
    )

    labels = tok.input_ids.to(
        device
    )

    labels[
        labels
        == tokenizer.pad_token_id
    ] = -100

    out = model.mt5(
        inputs_embeds=prefix,
        attention_mask=attention,
        labels=labels
    )

    return {
        "report_loss": out.loss,
        "concept_logits": concept_logits,
        "negative_logits": negative_logits,
        "attribute_logits": attribute_logits,
    }


# ============================================================
# 20. LOSS
# ============================================================

def total_loss(
    out,
    concepts,
    negatives,
    attributes
):

    lc = F.binary_cross_entropy_with_logits(
        out["concept_logits"],
        concepts
    )

    ln = F.binary_cross_entropy_with_logits(
        out["negative_logits"],
        negatives
    )

    la = F.binary_cross_entropy_with_logits(
        out["attribute_logits"],
        attributes
    )

    total = (
        out["report_loss"]
        + 0.5 * lc
        + 0.5 * ln
        + 0.25 * la
    )

    return (
        total,
        lc,
        ln,
        la
    )


# ============================================================
# 21. SANITY CHECK
# ============================================================

model.eval()

sanity_batch = next(
    iter(train_loader)
)

with torch.no_grad():

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
        enabled=USE_BF16
    ):

        sanity = forward_v4(
            sanity_batch
        )

print(
    "\nSanity report loss:",
    sanity[
        "report_loss"
    ].item()
)

assert (
    sanity[
        "concept_logits"
    ].shape
    == (4, 48)
)

assert (
    sanity[
        "negative_logits"
    ].shape
    == (4, 5)
)

assert (
    sanity[
        "attribute_logits"
    ].shape
    == (4, 6)
)

print(
    "Forward sanity: PASS"
)


# ============================================================
# 22. TRAIN EPOCH 3
# ============================================================

EPOCH = 3

model.train()

optimizer.zero_grad(
    set_to_none=True
)

sums = {
    "total": 0.0,
    "report": 0.0,
    "concept": 0.0,
    "negative": 0.0,
    "attribute": 0.0,
}

num_batches = len(
    train_loader
)

print(
    "\n" + "=" * 60
)

print(
    "STARTING EPOCH 3"
)

print(
    "=" * 60
)

for step, batch in enumerate(
    train_loader
):

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
        enabled=USE_BF16
    ):

        out = forward_v4(
            batch
        )

        (
            concepts,
            negatives,
            attributes
        ) = get_structured_targets(
            batch["case_id"]
        )

        (
            loss,
            lc,
            ln,
            la
        ) = total_loss(
            out,
            concepts,
            negatives,
            attributes
        )

        scaled = (
            loss
            / GRAD_ACCUM
        )

    scaled.backward()

    if (
        (step + 1)
        % GRAD_ACCUM
        == 0
        or
        step + 1
        == num_batches
    ):

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM
        )

        optimizer.step()

        scheduler.step()

        optimizer.zero_grad(
            set_to_none=True
        )

    sums["total"] += loss.item()
    sums["report"] += (
        out["report_loss"].item()
    )
    sums["concept"] += lc.item()
    sums["negative"] += ln.item()
    sums["attribute"] += la.item()

    if (
        (step + 1)
        % 200
        == 0
    ):

        print(
            f"Batch {step+1:4d}/"
            f"{num_batches} "
            f"| loss={loss.item():.4f}"
        )


train_metrics = {
    k:
        v / num_batches
    for k, v in sums.items()
}

print(
    "\nTRAIN METRICS"
)

for k, v in train_metrics.items():

    print(
        f"{k:10s}: {v:.6f}"
    )


# ============================================================
# 23. VALIDATION
# ============================================================

model.eval()

val_sums = {
    "total": 0.0,
    "report": 0.0,
    "concept": 0.0,
    "negative": 0.0,
    "attribute": 0.0,
}

with torch.no_grad():

    for batch in val_loader:

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=USE_BF16
        ):

            out = forward_v4(
                batch
            )

            (
                concepts,
                negatives,
                attributes
            ) = get_structured_targets(
                batch["case_id"]
            )

            (
                loss,
                lc,
                ln,
                la
            ) = total_loss(
                out,
                concepts,
                negatives,
                attributes
            )

        val_sums["total"] += (
            loss.item()
        )

        val_sums["report"] += (
            out["report_loss"].item()
        )

        val_sums["concept"] += (
            lc.item()
        )

        val_sums["negative"] += (
            ln.item()
        )

        val_sums["attribute"] += (
            la.item()
        )


val_batches = len(
    val_loader
)

val_metrics = {
    k:
        v / val_batches
    for k, v in val_sums.items()
}

print(
    "\nVALIDATION METRICS"
)

for k, v in val_metrics.items():

    print(
        f"{k:10s}: {v:.6f}"
    )


# ============================================================
# 24. UPDATE HISTORY
# ============================================================

record = {
    "epoch": EPOCH,

    "train_total":
        train_metrics["total"],

    "train_report":
        train_metrics["report"],

    "train_concept":
        train_metrics["concept"],

    "train_negative":
        train_metrics["negative"],

    "train_attribute":
        train_metrics["attribute"],

    "val_total":
        val_metrics["total"],

    "val_report":
        val_metrics["report"],

    "val_concept":
        val_metrics["concept"],

    "val_negative":
        val_metrics["negative"],

    "val_attribute":
        val_metrics["attribute"],
}

history = [
    h
    for h in history
    if h.get("epoch") != EPOCH
]

history.append(record)

history = sorted(
    history,
    key=lambda x: x["epoch"]
)


# ============================================================
# 25. SAVE CHECKPOINT
# ============================================================

is_best = (
    val_metrics["total"]
    < BEST_VAL
)

if is_best:

    BEST_VAL = (
        val_metrics["total"]
    )

checkpoint = {
    "epoch": EPOCH,

    "model_state":
        model.state_dict(),

    "optimizer_state":
        optimizer.state_dict(),

    "scheduler_state":
        scheduler.state_dict(),

    "best_val":
        BEST_VAL,

    "rng_state":
        torch.get_rng_state(),

    "cuda_rng_state":
        torch.cuda.get_rng_state_all(),

    "history":
        history,
}

torch.save(
    checkpoint,
    CHECKPOINT_DIR /
    "epoch_03.pt"
)

torch.save(
    checkpoint,
    LAST_CHECKPOINT
)

if is_best:

    torch.save(
        checkpoint,
        BEST_CHECKPOINT
    )

pd.DataFrame(
    history
).to_csv(
    METRICS_DIR /
    "training_history.csv",
    index=False
)


print(
    "\n" + "=" * 60
)

print(
    "V4 EPOCH 3 COMPLETE"
)

print(
    "=" * 60
)

print(
    "Best val loss:",
    BEST_VAL
)

print(
    "New best:",
    is_best
)

print(
    "Checkpoint:",
    LAST_CHECKPOINT
)

print("=" * 60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
V4 — EXACT CHECKPOINT RESUME
GPU: NVIDIA A100-SXM4-40GB
V4 manifest: (7606, 15)
Image manifest: (76405, 26)
Structured: (7606, 101)
Train batches: 1535
Val batches: 189


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



V4 checkpoint epoch: 2
Best val: 0.6521643281140656

Checkpoint conditioner keys:
  conditioner.attribute_label.weight
  conditioner.attribute_value.weight
  conditioner.concept_label.weight
  conditioner.concept_value.weight
  conditioner.negative_label.weight
  conditioner.negative_value.weight
  conditioner.norm.bias
  conditioner.norm.weight
  conditioner.type_embedding.weight

Model conditioner keys:
  conditioner.attribute_label.weight
  conditioner.attribute_value.weight
  conditioner.concept_label.weight
  conditioner.concept_value.weight
  conditioner.negative_label.weight
  conditioner.negative_value.weight
  conditioner.norm.bias
  conditioner.norm.weight
  conditioner.type_embedding.weight


RuntimeError: Error(s) in loading state_dict for V4Model:
	size mismatch for conditioner.concept_value.weight: copying a param with shape torch.Size([2, 512]) from checkpoint, the shape in current model is torch.Size([512, 1]).
	size mismatch for conditioner.negative_value.weight: copying a param with shape torch.Size([2, 512]) from checkpoint, the shape in current model is torch.Size([512, 1]).
	size mismatch for conditioner.attribute_value.weight: copying a param with shape torch.Size([2, 512]) from checkpoint, the shape in current model is torch.Size([512, 1]).

In [8]:
# ============================================================
# V4 CONDITIONER — EXACT SHAPE INSPECTION
# ============================================================

v4 = torch.load(
    LAST_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

state = v4["model_state"]

print("=" * 60)
print("ACTUAL V4 CONDITIONER SHAPES")
print("=" * 60)

for k, v in state.items():

    if k.startswith("conditioner."):

        print(
            f"{k:50s}",
            tuple(v.shape)
        )

print("=" * 60)

print(
    "\nValue-layer shapes:"
)

print(
    "concept_value   :",
    tuple(
        state[
            "conditioner.concept_value.weight"
        ].shape
    )
)

print(
    "negative_value  :",
    tuple(
        state[
            "conditioner.negative_value.weight"
        ].shape
    )
)

print(
    "attribute_value :",
    tuple(
        state[
            "conditioner.attribute_value.weight"
        ].shape
    )
)

ACTUAL V4 CONDITIONER SHAPES
conditioner.concept_value.weight                   (2, 512)
conditioner.negative_value.weight                  (2, 512)
conditioner.attribute_value.weight                 (2, 512)
conditioner.concept_label.weight                   (48, 512)
conditioner.negative_label.weight                  (5, 512)
conditioner.attribute_label.weight                 (6, 512)
conditioner.type_embedding.weight                  (3, 512)
conditioner.norm.weight                            (512,)
conditioner.norm.bias                              (512,)

Value-layer shapes:
concept_value   : (2, 512)
negative_value  : (2, 512)
attribute_value : (2, 512)


In [9]:
# ============================================================
# V4 — RESUME FROM EPOCH 2 → TRAIN EPOCH 3
# SELF-CONTAINED COLAB CELL
# ============================================================

import os
import gc
import math
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    ViTModel,
)

warnings.filterwarnings("ignore")

# ============================================================
# 0. CONFIG
# ============================================================

DRIVE_ROOT = "/content/drive/MyDrive"

BASE_DIR = os.path.join(
    DRIVE_ROOT,
    "NoiSoi_Matching"
)

V4_DIR = os.path.join(
    BASE_DIR,
    "baseline_model_v4"
)

CHECKPOINT_DIR = os.path.join(
    V4_DIR,
    "checkpoints"
)

LAST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "last.pt"
)

BEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best.pt"
)

EPOCH3_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "epoch_03.pt"
)

V4_MANIFEST = os.path.join(
    V4_DIR,
    "v4_training_manifest.csv"
)

IMAGE_MANIFEST = os.path.join(
    BASE_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

STRUCTURED_TARGETS = os.path.join(
    BASE_DIR,
    "v3_ontology_v2",
    "v3_structured_targets_v2_2.csv"
)

ONTOLOGY_MAPPING = os.path.join(
    V4_DIR,
    "diagnostics",
    "frozen_concept_ontology_mapping.csv"
)

CONFIG_PATH = os.path.join(
    V4_DIR,
    "v4_config.json"
)

HISTORY_CSV = os.path.join(
    V4_DIR,
    "v4_training_history.csv"
)

# ------------------------------------------------------------
# Model configuration
# ------------------------------------------------------------

VIT_NAME = "google/vit-base-patch16-224"
MT5_NAME = "google/mt5-small"

IMAGE_SIZE = 224
MAX_IMAGES = 8
MAX_TARGET_LENGTH = 96

D_MODEL = 512

NUM_CONCEPTS = 48
NUM_NEGATIVE = 5
NUM_ATTRIBUTES = 6

BATCH_SIZE = 4
GRAD_ACCUM = 4
NUM_WORKERS = 2

TOTAL_EPOCHS = 5
RESUME_EPOCH = 2
TARGET_EPOCH = 3

LR_VIT = 1e-5
LR_PROJECTOR = 1e-4
LR_MT5 = 5e-5
WEIGHT_DECAY = 0.01

CONCEPT_WEIGHT = 0.5
NEGATIVE_WEIGHT = 0.5
ATTRIBUTE_WEIGHT = 0.25

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_BF16 = (
    DEVICE.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

print("=" * 70)
print("V4 RESUME — EPOCH 3")
print("=" * 70)
print("Device :", DEVICE)
print("BF16   :", USE_BF16)
print("Last checkpoint:", LAST_CHECKPOINT)


# ============================================================
# 1. MOUNT DRIVE
# ============================================================

from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

assert os.path.exists(LAST_CHECKPOINT), (
    f"Checkpoint not found:\n{LAST_CHECKPOINT}"
)

print("✓ Drive mounted")
print("✓ Epoch-2 checkpoint exists")


# ============================================================
# 2. LOAD DATA
# ============================================================

v4_manifest = pd.read_csv(
    V4_MANIFEST
)

image_manifest = pd.read_csv(
    IMAGE_MANIFEST
)

structured_df = pd.read_csv(
    STRUCTURED_TARGETS,
    index_col="case_id"
)

ontology_map = pd.read_csv(
    ONTOLOGY_MAPPING
)

print("\nDATA")
print("-" * 70)
print("V4 manifest      :", v4_manifest.shape)
print("Image manifest   :", image_manifest.shape)
print("Structured       :", structured_df.shape)
print("Ontology mapping :", ontology_map.shape)


# ============================================================
# 3. FROZEN ONTOLOGY
# ============================================================

frozen_concept_columns = (
    ontology_map["column"].tolist()
)

assert len(frozen_concept_columns) == 48

negative_columns = [
    "negative__no_abnormal_external_middle_ear",
    "negative__no_abnormal_nose_sinus",
    "negative__no_abnormal_ent",
    "negative__no_bleeding",
    "negative__no_foreign_body",
]

attribute_columns = [
    "attribute__acute",
    "attribute__chronic",
    "attribute__right",
    "attribute__left",
    "attribute__bilateral",
    "attribute__post_surgery",
]

print("\nONTOLOGY")
print("-" * 70)
print("Concepts   :", len(frozen_concept_columns))
print("Negatives  :", len(negative_columns))
print("Attributes :", len(attribute_columns))


# ============================================================
# 4. FILTER TO VALID STRUCTURED TARGETS
# ============================================================

structured_case_ids = set(
    structured_df.index.astype(str)
)

v4_manifest["case_id"] = (
    v4_manifest["case_id"].astype(str)
)

v4_manifest = v4_manifest[
    v4_manifest["case_id"].isin(
        structured_case_ids
    )
].copy()

print("\nMANIFEST AFTER STRUCTURED FILTER")
print("-" * 70)
print(v4_manifest.shape)


# ============================================================
# 5. TRAIN / VAL SPLIT
# ============================================================

train_df = v4_manifest[
    v4_manifest["split"] == "train"
].copy()

val_df = v4_manifest[
    v4_manifest["split"] == "val"
].copy()

print("\nSPLIT")
print("-" * 70)
print("Train:", len(train_df))
print("Val  :", len(val_df))


# ============================================================
# 6. IMAGE INDEX
# ============================================================

image_manifest["case_id"] = (
    image_manifest["case_id"].astype(str)
)

image_manifest = image_manifest[
    image_manifest["status"] == "NORMAL"
].copy()

image_groups = {}

for case_id, g in image_manifest.groupby("case_id"):

    paths = g["image_path"].tolist()

    if len(paths) > MAX_IMAGES:
        paths = paths[:MAX_IMAGES]

    image_groups[case_id] = paths

print("\nIMAGE GROUPS")
print("-" * 70)
print("Cases with NORMAL images:", len(image_groups))


# ============================================================
# 7. TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MT5_NAME
)

print("\nTokenizer loaded")
print("Tokenizer vocab:", len(tokenizer))


# ============================================================
# 8. IMAGE PROCESSING
# ============================================================

from torchvision import transforms

image_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.5,
            0.5,
            0.5
        ],
        std=[
            0.5,
            0.5,
            0.5
        ]
    )
])


# ============================================================
# 9. DATASET
# ============================================================

class V4Dataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_groups,
        structured_df,
        tokenizer
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.image_groups = image_groups
        self.structured_df = structured_df
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        case_id = str(
            row["case_id"]
        )

        # ----------------------------------------------------
        # Images
        # ----------------------------------------------------

        paths = self.image_groups.get(
            case_id,
            []
        )

        images = []

        for path in paths:

            try:

                img = Image.open(
                    path
                ).convert("RGB")

                img = image_transform(
                    img
                )

                images.append(img)

            except Exception:
                continue

        # ----------------------------------------------------
        # Safety
        # ----------------------------------------------------

        if len(images) == 0:

            images = [
                torch.zeros(
                    3,
                    IMAGE_SIZE,
                    IMAGE_SIZE
                )
            ]

        # ----------------------------------------------------
        # Report target
        # ----------------------------------------------------

        report = str(
            row["target_text"]
        )

        encoded = self.tokenizer(
            report,
            max_length=MAX_TARGET_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = encoded[
            "input_ids"
        ].squeeze(0)

        labels[
            labels == tokenizer.pad_token_id
        ] = -100

        # ----------------------------------------------------
        # Structured GT
        #
        # Used ONLY for auxiliary supervision.
        # NOT passed directly to decoder.
        # ----------------------------------------------------

        target = self.structured_df.loc[
            case_id
        ]

        concept = torch.tensor(
            target[
                frozen_concept_columns
            ].values.astype(np.float32)
        )

        negative = torch.tensor(
            target[
                negative_columns
            ].values.astype(np.float32)
        )

        attribute = torch.tensor(
            target[
                attribute_columns
            ].values.astype(np.float32)
        )

        return {
            "case_id": case_id,
            "images": images,
            "labels": labels,
            "concept": concept,
            "negative": negative,
            "attribute": attribute,
        }


# ============================================================
# 10. COLLATE
# ============================================================

def collate_fn(batch):

    return {
        "case_id": [
            x["case_id"]
            for x in batch
        ],

        "images": [
            x["images"]
            for x in batch
        ],

        "labels": torch.stack([
            x["labels"]
            for x in batch
        ]),

        "concept": torch.stack([
            x["concept"]
            for x in batch
        ]),

        "negative": torch.stack([
            x["negative"]
            for x in batch
        ]),

        "attribute": torch.stack([
            x["attribute"]
            for x in batch
        ]),
    }


train_dataset = V4Dataset(
    train_df,
    image_groups,
    structured_df,
    tokenizer
)

val_dataset = V4Dataset(
    val_df,
    image_groups,
    structured_df,
    tokenizer
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn,
    drop_last=False
)

print("\nLOADERS")
print("-" * 70)
print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))


# ============================================================
# 11. MODEL COMPONENTS
# ============================================================

class VisualProjector(nn.Module):

    def __init__(
        self,
        in_dim=768,
        out_dim=512
    ):

        super().__init__()

        self.proj = nn.Linear(
            in_dim,
            out_dim
        )

    def forward(self, x):

        return self.proj(x)


# ------------------------------------------------------------
# EXACT CHECKPOINT CONDITIONER
# ------------------------------------------------------------

class StructuredConditioner(nn.Module):

    def __init__(
        self,
        d_model,
        num_concepts=48,
        num_negative=5,
        num_attributes=6
    ):

        super().__init__()

        self.num_concepts = (
            num_concepts
        )

        self.num_negative = (
            num_negative
        )

        self.num_attributes = (
            num_attributes
        )

        self.concept_value = nn.Embedding(
            2,
            d_model
        )

        self.negative_value = nn.Embedding(
            2,
            d_model
        )

        self.attribute_value = nn.Embedding(
            2,
            d_model
        )

        self.concept_label = nn.Embedding(
            num_concepts,
            d_model
        )

        self.negative_label = nn.Embedding(
            num_negative,
            d_model
        )

        self.attribute_label = nn.Embedding(
            num_attributes,
            d_model
        )

        self.type_embedding = nn.Embedding(
            3,
            d_model
        )

        self.norm = nn.LayerNorm(
            d_model
        )

    def forward(
        self,
        concept_targets,
        negative_targets,
        attribute_targets
    ):

        device = (
            concept_targets.device
        )

        concept_ids = (
            concept_targets
            .long()
            .clamp(0, 1)
        )

        negative_ids = (
            negative_targets
            .long()
            .clamp(0, 1)
        )

        attribute_ids = (
            attribute_targets
            .long()
            .clamp(0, 1)
        )

        # ----------------------------------------------------
        # Concepts
        # ----------------------------------------------------

        concept_idx = torch.arange(
            self.num_concepts,
            device=device
        )

        concept_type = self.type_embedding(
            torch.zeros(
                self.num_concepts,
                dtype=torch.long,
                device=device
            )
        )

        concept_tokens = (
            self.concept_value(
                concept_ids
            )
            +
            self.concept_label(
                concept_idx
            )[None, :, :]
            +
            concept_type[None, :, :]
        )

        # ----------------------------------------------------
        # Negatives
        # ----------------------------------------------------

        negative_idx = torch.arange(
            self.num_negative,
            device=device
        )

        negative_type = self.type_embedding(
            torch.ones(
                self.num_negative,
                dtype=torch.long,
                device=device
            )
        )

        negative_tokens = (
            self.negative_value(
                negative_ids
            )
            +
            self.negative_label(
                negative_idx
            )[None, :, :]
            +
            negative_type[None, :, :]
        )

        # ----------------------------------------------------
        # Attributes
        # ----------------------------------------------------

        attribute_idx = torch.arange(
            self.num_attributes,
            device=device
        )

        attribute_type = self.type_embedding(
            torch.full(
                (
                    self.num_attributes,
                ),
                2,
                dtype=torch.long,
                device=device
            )
        )

        attribute_tokens = (
            self.attribute_value(
                attribute_ids
            )
            +
            self.attribute_label(
                attribute_idx
            )[None, :, :]
            +
            attribute_type[None, :, :]
        )

        # ----------------------------------------------------
        # Concatenate
        # ----------------------------------------------------

        structured_tokens = torch.cat(
            [
                concept_tokens,
                negative_tokens,
                attribute_tokens,
            ],
            dim=1
        )

        structured_tokens = self.norm(
            structured_tokens
        )

        B = concept_targets.shape[0]

        structured_attention = torch.ones(
            B,
            structured_tokens.shape[1],
            dtype=torch.long,
            device=device
        )

        return (
            structured_tokens,
            structured_attention
        )


# ============================================================
# 12. FULL V4 MODEL
# ============================================================

class V4Model(nn.Module):

    def __init__(self):

        super().__init__()

        self.vision = ViTModel.from_pretrained(
            VIT_NAME
        )

        self.projector = VisualProjector(
            768,
            D_MODEL
        )

        self.mt5 = AutoModelForSeq2SeqLM.from_pretrained(
            MT5_NAME
        )

        self.concept_head = nn.Linear(
            D_MODEL,
            NUM_CONCEPTS
        )

        self.negative_head = nn.Linear(
            D_MODEL,
            NUM_NEGATIVE
        )

        self.attribute_head = nn.Linear(
            D_MODEL,
            NUM_ATTRIBUTES
        )

        self.conditioner = StructuredConditioner(
            D_MODEL,
            NUM_CONCEPTS,
            NUM_NEGATIVE,
            NUM_ATTRIBUTES
        )

    def encode_images(
        self,
        images
    ):

        visual_tokens = []

        for image_list in images:

            x = torch.stack(
                image_list,
                dim=0
            ).to(
                DEVICE,
                non_blocking=True
            )

            out = self.vision(
                pixel_values=x
            )

            cls = out.last_hidden_state[
                :,
                0,
                :
            ]

            cls = self.projector(
                cls
            )

            visual_tokens.append(
                cls
            )

        max_n = max(
            x.shape[0]
            for x in visual_tokens
        )

        padded = []

        mask = []

        for x in visual_tokens:

            n = x.shape[0]

            if n < max_n:

                pad = torch.zeros(
                    max_n - n,
                    D_MODEL,
                    device=x.device,
                    dtype=x.dtype
                )

                x = torch.cat(
                    [
                        x,
                        pad
                    ],
                    dim=0
                )

            padded.append(x)

            m = torch.zeros(
                max_n,
                dtype=torch.long,
                device=x.device
            )

            m[:n] = 1

            mask.append(m)

        return (
            torch.stack(padded),
            torch.stack(mask)
        )

    def forward(
        self,
        images,
        labels=None,
        concept_targets=None,
        negative_targets=None,
        attribute_targets=None
    ):

        # ----------------------------------------------------
        # Image encoding
        # ----------------------------------------------------

        visual_tokens, visual_mask = (
            self.encode_images(images)
        )

        # ----------------------------------------------------
        # Masked visual representation
        # ----------------------------------------------------

        denom = visual_mask.sum(
            dim=1,
            keepdim=True
        ).clamp(
            min=1
        )

        pooled = (
            visual_tokens
            * visual_mask.unsqueeze(-1)
        ).sum(
            dim=1
        ) / denom

        # ----------------------------------------------------
        # Structured prediction heads
        #
        # GT is NOT used here.
        # ----------------------------------------------------

        concept_logits = (
            self.concept_head(
                pooled
            )
        )

        negative_logits = (
            self.negative_head(
                pooled
            )
        )

        attribute_logits = (
            self.attribute_head(
                pooled
            )
        )

        # ----------------------------------------------------
        # IMPORTANT:
        #
        # The original checkpoint conditioner itself accepts
        # binary IDs. For V4 main, predictions are converted
        # to binary IDs using threshold 0.5.
        #
        # Ground truth remains auxiliary supervision only.
        # ----------------------------------------------------

        concept_pred = (
            torch.sigmoid(
                concept_logits
            ) >= 0.5
        ).float()

        negative_pred = (
            torch.sigmoid(
                negative_logits
            ) >= 0.5
        ).float()

        attribute_pred = (
            torch.sigmoid(
                attribute_logits
            ) >= 0.5
        ).float()

        structured_tokens, structured_mask = (
            self.conditioner(
                concept_pred,
                negative_pred,
                attribute_pred
            )
        )

        # ----------------------------------------------------
        # Prefix
        # ----------------------------------------------------

        prefix = torch.cat(
            [
                visual_tokens,
                structured_tokens
            ],
            dim=1
        )

        prefix_mask = torch.cat(
            [
                visual_mask,
                structured_mask
            ],
            dim=1
        )

        # ----------------------------------------------------
        # mT5 encoder
        # ----------------------------------------------------

        encoder_outputs = (
            self.mt5.encoder(
                inputs_embeds=prefix,
                attention_mask=prefix_mask,
                return_dict=True
            )
        )

        # ----------------------------------------------------
        # Report generation loss
        # ----------------------------------------------------

        lm_outputs = self.mt5(
            encoder_outputs=encoder_outputs,
            attention_mask=prefix_mask,
            labels=labels,
            return_dict=True
        )

        report_loss = lm_outputs.loss

        # ----------------------------------------------------
        # Auxiliary losses
        # ----------------------------------------------------

        concept_loss = F.binary_cross_entropy_with_logits(
            concept_logits,
            concept_targets
        )

        negative_loss = F.binary_cross_entropy_with_logits(
            negative_logits,
            negative_targets
        )

        attribute_loss = F.binary_cross_entropy_with_logits(
            attribute_logits,
            attribute_targets
        )

        total_loss = (
            report_loss
            + CONCEPT_WEIGHT * concept_loss
            + NEGATIVE_WEIGHT * negative_loss
            + ATTRIBUTE_WEIGHT * attribute_loss
        )

        return {
            "loss": total_loss,
            "report_loss": report_loss,
            "concept_loss": concept_loss,
            "negative_loss": negative_loss,
            "attribute_loss": attribute_loss,
            "concept_logits": concept_logits,
            "negative_logits": negative_logits,
            "attribute_logits": attribute_logits,
        }


# ============================================================
# 13. BUILD MODEL
# ============================================================

print("\nBUILDING MODEL...")

model = V4Model()

model = model.to(
    DEVICE
)

print("✓ Model created")


# ============================================================
# 14. LOAD V3 INITIALIZATION
# ============================================================

V3_CHECKPOINT = os.path.join(
    BASE_DIR,
    "baseline_model_v3",
    "checkpoints",
    "best.pt"
)

assert os.path.exists(
    V3_CHECKPOINT
), "V3 checkpoint not found"

v3 = torch.load(
    V3_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

v3_state = v3["model_state_dict"]

# ------------------------------------------------------------
# Load only shared V3 components
# ------------------------------------------------------------

vision_state = {
    k.replace(
        "vision.",
        ""
    ): v
    for k, v in v3_state.items()
    if k.startswith("vision.")
}

mt5_state = {
    k.replace(
        "mt5.",
        ""
    ): v
    for k, v in v3_state.items()
    if k.startswith("mt5.")
}

projector_state = {
    k.replace(
        "projector.",
        ""
    ): v
    for k, v in v3_state.items()
    if k.startswith("projector.")
}

model.vision.load_state_dict(
    vision_state,
    strict=False
)

model.mt5.load_state_dict(
    mt5_state,
    strict=False
)

model.projector.load_state_dict(
    projector_state,
    strict=True
)

del v3
del v3_state

gc.collect()

print("✓ Loaded V3 shared weights")


# ============================================================
# 15. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    [
        {
            "params": model.vision.parameters(),
            "lr": LR_VIT,
        },
        {
            "params": model.projector.parameters(),
            "lr": LR_PROJECTOR,
        },
        {
            "params": model.mt5.parameters(),
            "lr": LR_MT5,
        },
        {
            "params": model.concept_head.parameters(),
            "lr": LR_PROJECTOR,
        },
        {
            "params": model.negative_head.parameters(),
            "lr": LR_PROJECTOR,
        },
        {
            "params": model.attribute_head.parameters(),
            "lr": LR_PROJECTOR,
        },
        {
            "params": model.conditioner.parameters(),
            "lr": LR_PROJECTOR,
        },
    ],
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 16. SCHEDULER
#
# Epoch 2 checkpoint was trained with:
# 1535 batches / epoch
# ceil(1535 / 4) = 384 optimizer steps
# 384 × 5 = 1920 total steps
# ============================================================

steps_per_epoch = math.ceil(
    len(train_loader) / GRAD_ACCUM
)

total_steps = (
    steps_per_epoch
    * TOTAL_EPOCHS
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=total_steps
)

print("\nSCHEDULER")
print("-" * 70)
print("Steps / epoch:", steps_per_epoch)
print("Total steps  :", total_steps)


# ============================================================
# 17. LOAD EXACT V4 EPOCH-2 CHECKPOINT
# ============================================================

print("\nLOADING V4 CHECKPOINT...")

checkpoint = torch.load(
    LAST_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

checkpoint_epoch = checkpoint["epoch"]

assert checkpoint_epoch == 2, (
    f"Expected Epoch 2 checkpoint, "
    f"found epoch {checkpoint_epoch}"
)

model.load_state_dict(
    checkpoint["model_state"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state"]
)

best_val = checkpoint["best_val"]

history = checkpoint.get(
    "history",
    []
)

# ------------------------------------------------------------
# Restore RNG
# ------------------------------------------------------------

if "rng_state" in checkpoint:

    torch.set_rng_state(
        checkpoint["rng_state"]
    )

if (
    DEVICE.type == "cuda"
    and "cuda_rng_state" in checkpoint
):

    torch.cuda.set_rng_state_all(
        checkpoint["cuda_rng_state"]
    )

del checkpoint

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print("✓ Loaded Epoch 2 checkpoint")
print("Completed epoch :", checkpoint_epoch)
print("Next epoch      :", checkpoint_epoch + 1)
print("Best val loss   :", best_val)
print(
    "Current LRs     :",
    [
        g["lr"]
        for g in optimizer.param_groups
    ]
)


# ============================================================
# 18. SANITY CHECK — CONDITIONER SHAPES
# ============================================================

print("\nCONDITIONER SHAPES")
print("-" * 70)

for name, param in model.named_parameters():

    if name.startswith(
        "conditioner."
    ):

        print(
            f"{name:50s}",
            tuple(param.shape)
        )


# ============================================================
# 19. TRAIN / VALIDATION FUNCTIONS
# ============================================================

def run_epoch(
    model,
    loader,
    optimizer=None,
    scheduler=None,
    train=True
):

    model.train(
        train
    )

    total_loss = 0.0
    total_report = 0.0
    total_concept = 0.0
    total_negative = 0.0
    total_attribute = 0.0

    n_batches = 0

    if train:

        optimizer.zero_grad(
            set_to_none=True
        )

    for batch_idx, batch in enumerate(
        loader
    ):

        labels = batch[
            "labels"
        ].to(
            DEVICE,
            non_blocking=True
        )

        concept = batch[
            "concept"
        ].to(
            DEVICE,
            non_blocking=True
        )

        negative = batch[
            "negative"
        ].to(
            DEVICE,
            non_blocking=True
        )

        attribute = batch[
            "attribute"
        ].to(
            DEVICE,
            non_blocking=True
        )

        if train:

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
                enabled=USE_BF16
            ):

                outputs = model(
                    images=batch["images"],
                    labels=labels,
                    concept_targets=concept,
                    negative_targets=negative,
                    attribute_targets=attribute
                )

                loss = (
                    outputs["loss"]
                    / GRAD_ACCUM
                )

            loss.backward()

            if (
                (batch_idx + 1)
                % GRAD_ACCUM == 0
                or
                (batch_idx + 1)
                == len(loader)
            ):

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    1.0
                )

                optimizer.step()

                scheduler.step()

                optimizer.zero_grad(
                    set_to_none=True
                )

        else:

            with torch.no_grad():

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.bfloat16,
                    enabled=USE_BF16
                ):

                    outputs = model(
                        images=batch["images"],
                        labels=labels,
                        concept_targets=concept,
                        negative_targets=negative,
                        attribute_targets=attribute
                    )

        total_loss += (
            outputs["loss"]
            .item()
        )

        total_report += (
            outputs["report_loss"]
            .item()
        )

        total_concept += (
            outputs["concept_loss"]
            .item()
        )

        total_negative += (
            outputs["negative_loss"]
            .item()
        )

        total_attribute += (
            outputs["attribute_loss"]
            .item()
        )

        n_batches += 1

        if train and (
            (batch_idx + 1) % 100 == 0
            or
            (batch_idx + 1) == len(loader)
        ):

            print(
                f"\r  Batch "
                f"{batch_idx+1:4d}/{len(loader)}"
                f" | loss "
                f"{total_loss/n_batches:.4f}",
                end=""
            )

    if train:
        print()

    return {
        "loss": total_loss / n_batches,
        "report_loss": total_report / n_batches,
        "concept_loss": total_concept / n_batches,
        "negative_loss": total_negative / n_batches,
        "attribute_loss": total_attribute / n_batches,
    }


# ============================================================
# 20. RUN EPOCH 3
# ============================================================

assert checkpoint_epoch + 1 == TARGET_EPOCH

print("\n")
print("=" * 70)
print("STARTING EPOCH 3")
print("=" * 70)

train_metrics = run_epoch(
    model,
    train_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    train=True
)

print("\nTRAIN")
print("-" * 70)

for k, v in train_metrics.items():
    print(
        f"{k:20s}: {v:.6f}"
    )


# ============================================================
# 21. VALIDATION
# ============================================================

print("\nVALIDATION")

val_metrics = run_epoch(
    model,
    val_loader,
    train=False
)

print("\nVAL")
print("-" * 70)

for k, v in val_metrics.items():
    print(
        f"{k:20s}: {v:.6f}"
    )


# ============================================================
# 22. UPDATE HISTORY
# ============================================================

epoch_record = {
    "epoch": TARGET_EPOCH,
    "train_loss": train_metrics["loss"],
    "train_report_loss": train_metrics["report_loss"],
    "train_concept_loss": train_metrics["concept_loss"],
    "train_negative_loss": train_metrics["negative_loss"],
    "train_attribute_loss": train_metrics["attribute_loss"],
    "val_loss": val_metrics["loss"],
    "val_report_loss": val_metrics["report_loss"],
    "val_concept_loss": val_metrics["concept_loss"],
    "val_negative_loss": val_metrics["negative_loss"],
    "val_attribute_loss": val_metrics["attribute_loss"],
}

history = list(history)

history = [
    h for h in history
    if int(h["epoch"]) != TARGET_EPOCH
]

history.append(
    epoch_record
)


# ============================================================
# 23. SAVE EPOCH 3 CHECKPOINT
#
# IMPORTANT:
# epoch_03.pt is written FIRST.
# Epoch-2 last.pt is untouched until validation is complete.
# ============================================================

checkpoint_payload = {
    "epoch": TARGET_EPOCH,

    "model_state": model.state_dict(),

    "optimizer_state": optimizer.state_dict(),

    "scheduler_state": scheduler.state_dict(),

    "best_val": min(
        best_val,
        val_metrics["loss"]
    ),

    "rng_state": torch.get_rng_state(),

    "cuda_rng_state": (
        torch.cuda.get_rng_state_all()
        if DEVICE.type == "cuda"
        else None
    ),

    "history": history,
}

torch.save(
    checkpoint_payload,
    EPOCH3_CHECKPOINT
)

print("\n✓ Saved:")
print(EPOCH3_CHECKPOINT)


# ============================================================
# 24. UPDATE BEST
# ============================================================

is_best = (
    val_metrics["loss"]
    < best_val
)

if is_best:

    torch.save(
        checkpoint_payload,
        BEST_CHECKPOINT
    )

    best_val = val_metrics[
        "loss"
    ]

    print(
        "✓ New BEST checkpoint:"
    )

    print(
        BEST_CHECKPOINT
    )

else:

    print(
        "Best checkpoint unchanged."
    )


# ============================================================
# 25. NOW UPDATE last.pt
#
# Only after epoch_03.pt and validation are safely written.
# ============================================================

checkpoint_payload["best_val"] = best_val

torch.save(
    checkpoint_payload,
    LAST_CHECKPOINT
)

print("\n✓ Updated last.pt")


# ============================================================
# 26. SAVE HISTORY CSV
# ============================================================

history_df = pd.DataFrame(
    history
).sort_values(
    "epoch"
)

history_df.to_csv(
    HISTORY_CSV,
    index=False
)

print(
    "✓ History saved:",
    HISTORY_CSV
)


# ============================================================
# 27. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("EPOCH 3 COMPLETE")
print("=" * 70)

print(
    f"Epoch              : {TARGET_EPOCH}"
)

print(
    f"Train total loss   : "
    f"{train_metrics['loss']:.6f}"
)

print(
    f"Val total loss     : "
    f"{val_metrics['loss']:.6f}"
)

print(
    f"Val report loss    : "
    f"{val_metrics['report_loss']:.6f}"
)

print(
    f"Val concept loss   : "
    f"{val_metrics['concept_loss']:.6f}"
)

print(
    f"Val negative loss  : "
    f"{val_metrics['negative_loss']:.6f}"
)

print(
    f"Val attribute loss : "
    f"{val_metrics['attribute_loss']:.6f}"
)

print(
    f"Best val loss      : "
    f"{best_val:.6f}"
)

print(
    "New best?          :",
    is_best
)

print("\nCheckpoint files:")
print("  epoch_03.pt :", os.path.exists(EPOCH3_CHECKPOINT))
print("  last.pt     :", os.path.exists(LAST_CHECKPOINT))
print("  best.pt     :", os.path.exists(BEST_CHECKPOINT))

print("\n✓ V4 Epoch 3 finished safely.")

V4 RESUME — EPOCH 3
Device : cuda
BF16   : True
Last checkpoint: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/checkpoints/last.pt
✓ Drive mounted
✓ Epoch-2 checkpoint exists

DATA
----------------------------------------------------------------------
V4 manifest      : (7606, 15)
Image manifest   : (76405, 26)
Structured       : (7606, 101)
Ontology mapping : (48, 3)

ONTOLOGY
----------------------------------------------------------------------
Concepts   : 48
Negatives  : 5
Attributes : 6

MANIFEST AFTER STRUCTURED FILTER
----------------------------------------------------------------------
(7606, 15)

SPLIT
----------------------------------------------------------------------
Train: 6138
Val  : 756


KeyError: 'status'

In [12]:
# ============================================================
# V4 COMPLETE PRE-TRAIN AUDIT
#
# PURPOSE:
#   Verify EVERYTHING before writing the Epoch-3 training cell.
#
# IMPORTANT:
#   - NO training
#   - NO backward()
#   - NO optimizer.step()
#   - NO scheduler.step()
#   - NO checkpoint overwrite
#   - NO checkpoint save
#
# This cell only inspects and performs read-only sanity tests.
# ============================================================

import os
import gc
import math
import json
import warnings

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    ViTModel,
)

warnings.filterwarnings("ignore")

print("=" * 75)
print("V4 COMPLETE PRE-TRAIN AUDIT")
print("=" * 75)


# ============================================================
# 0. PATHS
# ============================================================

DRIVE_ROOT = "/content/drive/MyDrive"

BASE_DIR = os.path.join(
    DRIVE_ROOT,
    "NoiSoi_Matching"
)

V4_DIR = os.path.join(
    BASE_DIR,
    "baseline_model_v4"
)

CHECKPOINT_DIR = os.path.join(
    V4_DIR,
    "checkpoints"
)

LAST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "last.pt"
)

BEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best.pt"
)

V4_MANIFEST = os.path.join(
    V4_DIR,
    "v4_training_manifest.csv"
)

IMAGE_MANIFEST = os.path.join(
    BASE_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

STRUCTURED_TARGETS = os.path.join(
    BASE_DIR,
    "v3_ontology_v2",
    "v3_structured_targets_v2_2.csv"
)

ONTOLOGY_MAPPING = os.path.join(
    V4_DIR,
    "diagnostics",
    "frozen_concept_ontology_mapping.csv"
)

CONFIG_PATH = os.path.join(
    V4_DIR,
    "v4_config.json"
)

HISTORY_CSV = os.path.join(
    V4_DIR,
    "v4_training_history.csv"
)

V3_CHECKPOINT = os.path.join(
    BASE_DIR,
    "baseline_model_v3",
    "checkpoints",
    "best.pt"
)

VIT_NAME = "google/vit-base-patch16-224"
MT5_NAME = "google/mt5-small"

IMAGE_SIZE = 224
MAX_IMAGES = 8
MAX_TARGET_LENGTH = 96

D_MODEL = 512

NUM_CONCEPTS = 48
NUM_NEGATIVE = 5
NUM_ATTRIBUTES = 6

BATCH_SIZE = 4
GRAD_ACCUM = 4
NUM_WORKERS = 2

TOTAL_EPOCHS = 5

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

USE_BF16 = (
    DEVICE.type == "cuda"
    and torch.cuda.is_bf16_supported()
)


# ============================================================
# 1. DRIVE / FILE EXISTENCE
# ============================================================

print("\n[1] FILE EXISTENCE")
print("-" * 75)

from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

paths_to_check = {
    "V4 manifest": V4_MANIFEST,
    "Image manifest": IMAGE_MANIFEST,
    "Structured targets": STRUCTURED_TARGETS,
    "Ontology mapping": ONTOLOGY_MAPPING,
    "V4 config": CONFIG_PATH,
    "V3 checkpoint": V3_CHECKPOINT,
    "V4 last checkpoint": LAST_CHECKPOINT,
    "V4 best checkpoint": BEST_CHECKPOINT,
}

all_files_ok = True

for name, path in paths_to_check.items():

    exists = os.path.exists(path)

    print(
        f"{'✓' if exists else '✗'} "
        f"{name:25s}: {path}"
    )

    if not exists:
        all_files_ok = False

assert all_files_ok, (
    "One or more required files are missing."
)

print("✓ All required files exist.")


# ============================================================
# 2. LOAD ALL TABULAR FILES
# ============================================================

print("\n[2] LOAD DATA")
print("-" * 75)

v4_manifest = pd.read_csv(
    V4_MANIFEST
)

image_manifest = pd.read_csv(
    IMAGE_MANIFEST
)

structured_df = pd.read_csv(
    STRUCTURED_TARGETS,
    index_col="case_id"
)

ontology_map = pd.read_csv(
    ONTOLOGY_MAPPING
)

print(
    "V4 manifest      :",
    v4_manifest.shape
)

print(
    "Image manifest   :",
    image_manifest.shape
)

print(
    "Structured       :",
    structured_df.shape
)

print(
    "Ontology mapping :",
    ontology_map.shape
)


# ============================================================
# 3. PRINT EXACT SCHEMAS
# ============================================================

print("\n[3] EXACT SCHEMAS")
print("-" * 75)

print("\nV4 MANIFEST")
for i, c in enumerate(
    v4_manifest.columns
):
    print(
        f"{i:3d}: {c}"
    )

print("\nIMAGE MANIFEST")
for i, c in enumerate(
    image_manifest.columns
):
    print(
        f"{i:3d}: {c}"
    )

print("\nSTRUCTURED TARGET COLUMNS")
for i, c in enumerate(
    structured_df.columns
):
    print(
        f"{i:3d}: {c}"
    )

print("\nONTOLOGY MAPPING")
print(
    ontology_map.to_string(
        index=False
    )
)


# ============================================================
# 4. V4 MANIFEST TARGET INVESTIGATION
# ============================================================

print("\n[4] V4 TARGET INVESTIGATION")
print("-" * 75)

target_candidates = [
    "target_text",
    "ket_luan",
    "report",
    "report_text",
    "text",
]

for c in target_candidates:

    if c in v4_manifest.columns:

        s = v4_manifest[c].astype(str)

        print(
            f"FOUND: {c}"
        )

        print(
            "  non-null:",
            v4_manifest[c].notna().sum()
        )

        print(
            "  non-empty:",
            (
                s.str.strip().ne("")
            ).sum()
        )

        print(
            "  unique:",
            s.nunique()
        )

        print(
            "  sample:"
        )

        for x in s.head(3):
            print(
                "   ",
                repr(x[:300])
            )

print(
    "\nExpected V4 report target candidate:"
)

if "ket_luan" in v4_manifest.columns:

    print(
        "✓ ket_luan exists"
    )

    ket = (
        v4_manifest["ket_luan"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    print(
        "  non-empty:",
        ket.ne("").sum(),
        "/",
        len(ket)
    )

    print(
        "  empty:",
        ket.eq("").sum()
    )

    print(
        "  unique:",
        ket.nunique()
    )

else:

    print(
        "✗ ket_luan missing"
    )


# ============================================================
# 5. V4 CONFIG
# ============================================================

print("\n[5] V4 CONFIG")
print("-" * 75)

with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    v4_config = json.load(f)

print(
    json.dumps(
        v4_config,
        indent=2,
        ensure_ascii=False
    )
)


# ============================================================
# 6. SPLIT / CASE / PATIENT AUDIT
# ============================================================

print("\n[6] SPLIT / CASE / PATIENT AUDIT")
print("-" * 75)

required_v4_columns = [
    "case_id",
    "patient_group_id",
    "split",
    "ket_luan",
    "has_structured_label",
    "structured_loss_mask",
]

missing = [
    c for c in required_v4_columns
    if c not in v4_manifest.columns
]

print(
    "Missing required columns:",
    missing
)

assert not missing, (
    f"Missing V4 columns: {missing}"
)

print(
    "\nSplit counts:"
)

print(
    v4_manifest[
        "split"
    ].value_counts()
)

print(
    "\nUnique cases:",
    v4_manifest["case_id"].nunique()
)

print(
    "Unique patients:",
    v4_manifest[
        "patient_group_id"
    ].nunique()
)

print(
    "Duplicate case IDs:",
    v4_manifest[
        "case_id"
    ].duplicated().sum()
)


# ------------------------------------------------------------
# Patient leakage
# ------------------------------------------------------------

patient_split_counts = (
    v4_manifest
    .groupby("patient_group_id")[
        "split"
    ]
    .nunique()
)

patient_leakage = (
    patient_split_counts > 1
).sum()

print(
    "Patients appearing in >1 split:",
    patient_leakage
)

assert patient_leakage == 0


# ============================================================
# 7. STRUCTURED TARGET ALIGNMENT
# ============================================================

print("\n[7] STRUCTURED TARGET ALIGNMENT")
print("-" * 75)

v4_case_ids = set(
    v4_manifest[
        "case_id"
    ].astype(str)
)

structured_case_ids = set(
    structured_df.index.astype(str)
)

print(
    "V4 cases:",
    len(v4_case_ids)
)

print(
    "Structured cases:",
    len(structured_case_ids)
)

print(
    "V4 not in structured:",
    len(
        v4_case_ids
        - structured_case_ids
    )
)

print(
    "Structured not in V4:",
    len(
        structured_case_ids
        - v4_case_ids
    )
)

assert (
    v4_case_ids
    <= structured_case_ids
)


# ============================================================
# 8. ONTOLOGY AUDIT
# ============================================================

print("\n[8] ONTOLOGY AUDIT")
print("-" * 75)

assert list(
    ontology_map.columns
) == [
    "dimension",
    "column",
    "label"
]

concept_columns = (
    ontology_map["column"]
    .tolist()
)

negative_columns = [
    "negative__no_abnormal_external_middle_ear",
    "negative__no_abnormal_nose_sinus",
    "negative__no_abnormal_ent",
    "negative__no_bleeding",
    "negative__no_foreign_body",
]

attribute_columns = [
    "attribute__acute",
    "attribute__chronic",
    "attribute__right",
    "attribute__left",
    "attribute__bilateral",
    "attribute__post_surgery",
]

print(
    "Concepts:",
    len(concept_columns)
)

print(
    "Negatives:",
    len(negative_columns)
)

print(
    "Attributes:",
    len(attribute_columns)
)

assert len(concept_columns) == 48
assert len(negative_columns) == 5
assert len(attribute_columns) == 6

missing_structured = [
    c for c in (
        concept_columns
        + negative_columns
        + attribute_columns
    )
    if c not in structured_df.columns
]

print(
    "Missing structured columns:",
    missing_structured
)

assert not missing_structured


# ============================================================
# 9. STRUCTURED VALUE AUDIT
# ============================================================

print("\n[9] STRUCTURED VALUE AUDIT")
print("-" * 75)

all_structured_columns = (
    concept_columns
    + negative_columns
    + attribute_columns
)

bad_values = {}

for c in all_structured_columns:

    vals = (
        structured_df[c]
        .dropna()
        .unique()
    )

    bad = [
        x for x in vals
        if x not in [0, 1, 0.0, 1.0]
    ]

    if bad:
        bad_values[c] = bad[:10]

print(
    "Columns with non-binary values:",
    len(bad_values)
)

if bad_values:

    for k, v in bad_values.items():
        print(
            " ",
            k,
            "->",
            v
        )

assert not bad_values


# ============================================================
# 10. IMAGE MANIFEST AUDIT
# ============================================================

print("\n[10] IMAGE MANIFEST AUDIT")
print("-" * 75)

required_image_columns = [
    "case_id",
    "image_order",
    "image_path",
    "image_status",
    "split",
]

missing_image = [
    c for c in required_image_columns
    if c not in image_manifest.columns
]

print(
    "Missing image columns:",
    missing_image
)

assert not missing_image

print(
    "\nimage_status:"
)

print(
    image_manifest[
        "image_status"
    ].value_counts(
        dropna=False
    )
)

normal_images = image_manifest[
    image_manifest[
        "image_status"
    ].astype(str)
    == "NORMAL"
].copy()

print(
    "\nNORMAL images:",
    len(normal_images)
)

print(
    "NO_SIGNAL:",
    (
        image_manifest[
            "image_status"
        ].astype(str)
        == "NO_SIGNAL"
    ).sum()
)

print(
    "UNREADABLE:",
    (
        image_manifest[
            "image_status"
        ].astype(str)
        == "UNREADABLE"
    ).sum()
)


# ============================================================
# 11. IMAGE CASE COVERAGE
# ============================================================

print("\n[11] IMAGE CASE COVERAGE")
print("-" * 75)

normal_case_ids = set(
    normal_images[
        "case_id"
    ].astype(str)
)

v4_case_ids = set(
    v4_manifest[
        "case_id"
    ].astype(str)
)

missing_normal_cases = (
    v4_case_ids
    - normal_case_ids
)

print(
    "V4 cases:",
    len(v4_case_ids)
)

print(
    "Cases with NORMAL image:",
    len(
        v4_case_ids
        & normal_case_ids
    )
)

print(
    "Cases without NORMAL image:",
    len(missing_normal_cases)
)

if missing_normal_cases:
    print(
        "Examples:",
        list(
            missing_normal_cases
        )[:10]
    )

assert not missing_normal_cases


# ============================================================
# 12. IMAGE GROUPS
# ============================================================

image_groups = {}

for case_id, group in normal_images.groupby(
    "case_id"
):

    paths = (
        group[
            "image_path"
        ]
        .astype(str)
        .tolist()
    )

    paths = list(
        dict.fromkeys(paths)
    )

    image_groups[
        str(case_id)
    ] = paths[:MAX_IMAGES]

counts = np.array([
    len(x)
    for x in image_groups.values()
])

print(
    "\nImage count/case:"
)

print(
    "mean  :",
    counts.mean()
)

print(
    "std   :",
    counts.std()
)

print(
    "min   :",
    counts.min()
)

print(
    "median:",
    np.median(counts)
)

print(
    "max   :",
    counts.max()
)

print(
    "cases >8:",
    (counts > 8).sum()
)


# ============================================================
# 13. CHECK ACTUAL IMAGE FILES
# ============================================================

print("\n[13] ACTUAL IMAGE FILE CHECK")
print("-" * 75)

sample_paths = (
    normal_images[
        "image_path"
    ]
    .drop_duplicates()
    .head(20)
    .tolist()
)

missing_paths = []
open_errors = []
sizes = []

for path in sample_paths:

    if not os.path.exists(path):

        missing_paths.append(path)

        continue

    try:

        with Image.open(path) as im:

            im.verify()

        with Image.open(path) as im:

            sizes.append(
                (
                    im.size,
                    im.mode
                )
            )

    except Exception as e:

        open_errors.append(
            (
                path,
                repr(e)
            )
        )

print(
    "Sample paths:",
    len(sample_paths)
)

print(
    "Missing:",
    len(missing_paths)
)

print(
    "Open errors:",
    len(open_errors)
)

print(
    "Sample sizes:",
    sizes[:10]
)

assert not missing_paths
assert not open_errors


# ============================================================
# 14. TOKENIZER / TARGET AUDIT
# ============================================================

print("\n[14] TOKENIZER / TARGET AUDIT")
print("-" * 75)

tokenizer = AutoTokenizer.from_pretrained(
    MT5_NAME
)

print(
    "Tokenizer vocab:",
    len(tokenizer)
)

assert "ket_luan" in v4_manifest.columns

sample_reports = (
    v4_manifest[
        "ket_luan"
    ]
    .fillna("")
    .astype(str)
    .tolist()
)

sample_reports = [
    x.strip()
    for x in sample_reports
    if x.strip()
]

print(
    "Non-empty reports:",
    len(sample_reports)
)

print(
    "Sample report lengths:"
)

print(
    [
        len(x)
        for x in sample_reports[:10]
    ]
)

token_test = tokenizer(
    sample_reports[:4],
    max_length=MAX_TARGET_LENGTH,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

print(
    "Token test shape:",
    token_test["input_ids"].shape
)

assert (
    token_test["input_ids"].shape
    == (min(4, len(sample_reports)), MAX_TARGET_LENGTH)
)


# ============================================================
# 15. CHECKPOINT STRUCTURE
# ============================================================

print("\n[15] CHECKPOINT STRUCTURE")
print("-" * 75)

checkpoint = torch.load(
    LAST_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print(
    "Checkpoint keys:"
)

print(
    list(
        checkpoint.keys()
    )
)

print(
    "\nEpoch:",
    checkpoint["epoch"]
)

print(
    "Best val:",
    checkpoint["best_val"]
)

assert checkpoint["epoch"] == 2

state = checkpoint[
    "model_state"
]

print(
    "Model state keys:",
    len(state)
)


# ============================================================
# 16. CHECKPOINT CONDITIONER SHAPES
# ============================================================

print("\n[16] CHECKPOINT CONDITIONER")
print("-" * 75)

for name, tensor in state.items():

    if name.startswith(
        "conditioner."
    ):

        print(
            f"{name:55s}",
            tuple(tensor.shape)
        )


# ============================================================
# 17. CHECKPOINT MAIN COMPONENTS
# ============================================================

print("\n[17] CHECKPOINT COMPONENT COUNTS")
print("-" * 75)

component_counts = {}

for key in state:

    prefix = key.split(".")[0]

    component_counts[prefix] = (
        component_counts.get(
            prefix,
            0
        ) + 1
    )

for k, v in sorted(
    component_counts.items()
):

    print(
        f"{k:20s}: {v}"
    )


# ============================================================
# 18. OPTIMIZER / SCHEDULER AUDIT
# ============================================================

print("\n[18] OPTIMIZER / SCHEDULER")
print("-" * 75)

optimizer_state = checkpoint[
    "optimizer_state"
]

scheduler_state = checkpoint[
    "scheduler_state"
]

print(
    "Optimizer param groups:",
    len(
        optimizer_state[
            "param_groups"
        ]
    )
)

for i, group in enumerate(
    optimizer_state[
        "param_groups"
    ]
):

    print(
        f"Group {i}: "
        f"lr={group['lr']:.12g}, "
        f"weight_decay={group.get('weight_decay')}"
    )

print(
    "\nScheduler state:"
)

for k, v in scheduler_state.items():

    if isinstance(
        v,
        (int, float, str, bool)
    ):

        print(
            f"  {k}: {v}"
        )


# ============================================================
# 19. HISTORY AUDIT
# ============================================================

print("\n[19] CHECKPOINT HISTORY")
print("-" * 75)

history = checkpoint.get(
    "history",
    []
)

print(
    "History rows:",
    len(history)
)

for row in history:

    print(
        row
    )


# ============================================================
# 20. DEFINE EXACT CONDITIONER
# ============================================================

class StructuredConditioner(nn.Module):

    def __init__(
        self,
        d_model,
        num_concepts=48,
        num_negative=5,
        num_attributes=6
    ):

        super().__init__()

        self.num_concepts = (
            num_concepts
        )

        self.num_negative = (
            num_negative
        )

        self.num_attributes = (
            num_attributes
        )

        self.concept_value = nn.Embedding(
            2,
            d_model
        )

        self.negative_value = nn.Embedding(
            2,
            d_model
        )

        self.attribute_value = nn.Embedding(
            2,
            d_model
        )

        self.concept_label = nn.Embedding(
            num_concepts,
            d_model
        )

        self.negative_label = nn.Embedding(
            num_negative,
            d_model
        )

        self.attribute_label = nn.Embedding(
            num_attributes,
            d_model
        )

        self.type_embedding = nn.Embedding(
            3,
            d_model
        )

        self.norm = nn.LayerNorm(
            d_model
        )

    def forward(
        self,
        concept_targets,
        negative_targets,
        attribute_targets
    ):

        device = concept_targets.device

        concept_ids = (
            concept_targets
            .long()
            .clamp(0, 1)
        )

        negative_ids = (
            negative_targets
            .long()
            .clamp(0, 1)
        )

        attribute_ids = (
            attribute_targets
            .long()
            .clamp(0, 1)
        )

        concept_idx = torch.arange(
            self.num_concepts,
            device=device
        )

        concept_type = self.type_embedding(
            torch.zeros(
                self.num_concepts,
                dtype=torch.long,
                device=device
            )
        )

        concept_tokens = (
            self.concept_value(
                concept_ids
            )
            +
            self.concept_label(
                concept_idx
            )[None, :, :]
            +
            concept_type[None, :, :]
        )

        negative_idx = torch.arange(
            self.num_negative,
            device=device
        )

        negative_type = self.type_embedding(
            torch.ones(
                self.num_negative,
                dtype=torch.long,
                device=device
            )
        )

        negative_tokens = (
            self.negative_value(
                negative_ids
            )
            +
            self.negative_label(
                negative_idx
            )[None, :, :]
            +
            negative_type[None, :, :]
        )

        attribute_idx = torch.arange(
            self.num_attributes,
            device=device
        )

        attribute_type = self.type_embedding(
            torch.full(
                (
                    self.num_attributes,
                ),
                2,
                dtype=torch.long,
                device=device
            )
        )

        attribute_tokens = (
            self.attribute_value(
                attribute_ids
            )
            +
            self.attribute_label(
                attribute_idx
            )[None, :, :]
            +
            attribute_type[None, :, :]
        )

        structured_tokens = torch.cat(
            [
                concept_tokens,
                negative_tokens,
                attribute_tokens,
            ],
            dim=1
        )

        structured_tokens = self.norm(
            structured_tokens
        )

        B = (
            concept_targets.shape[0]
        )

        structured_attention = torch.ones(
            B,
            structured_tokens.shape[1],
            dtype=torch.long,
            device=device
        )

        return (
            structured_tokens,
            structured_attention
        )


# ============================================================
# 21. DEFINE MODEL
# ============================================================

class VisualProjector(nn.Module):

    def __init__(
        self,
        in_dim=768,
        out_dim=512
    ):

        super().__init__()

        self.proj = nn.Linear(
            in_dim,
            out_dim
        )

    def forward(self, x):

        return self.proj(x)


class V4Model(nn.Module):

    def __init__(self):

        super().__init__()

        self.vision = ViTModel.from_pretrained(
            VIT_NAME
        )

        self.projector = VisualProjector(
            768,
            512
        )

        self.mt5 = AutoModelForSeq2SeqLM.from_pretrained(
            MT5_NAME
        )

        self.concept_head = nn.Linear(
            512,
            48
        )

        self.negative_head = nn.Linear(
            512,
            5
        )

        self.attribute_head = nn.Linear(
            512,
            6
        )

        self.conditioner = StructuredConditioner(
            512,
            48,
            5,
            6
        )

    def encode_images(
        self,
        images
    ):

        visual_tokens = []

        for image_list in images:

            x = torch.stack(
                image_list,
                dim=0
            ).to(
                DEVICE,
                non_blocking=True
            )

            out = self.vision(
                pixel_values=x
            )

            cls = out.last_hidden_state[
                :,
                0,
                :
            ]

            cls = self.projector(
                cls
            )

            visual_tokens.append(
                cls
            )

        max_n = max(
            x.shape[0]
            for x in visual_tokens
        )

        padded = []
        masks = []

        for x in visual_tokens:

            n = x.shape[0]

            if n < max_n:

                pad = torch.zeros(
                    max_n - n,
                    512,
                    device=x.device,
                    dtype=x.dtype
                )

                x = torch.cat(
                    [x, pad],
                    dim=0
                )

            padded.append(x)

            mask = torch.zeros(
                max_n,
                dtype=torch.long,
                device=x.device
            )

            mask[:n] = 1

            masks.append(mask)

        return (
            torch.stack(padded),
            torch.stack(masks)
        )

    def forward(
        self,
        images,
        labels,
        concept_targets,
        negative_targets,
        attribute_targets
    ):

        visual_tokens, visual_mask = (
            self.encode_images(
                images
            )
        )

        denom = visual_mask.sum(
            dim=1,
            keepdim=True
        ).clamp(min=1)

        pooled = (
            visual_tokens
            * visual_mask.unsqueeze(-1)
        ).sum(
            dim=1
        ) / denom

        concept_logits = (
            self.concept_head(
                pooled
            )
        )

        negative_logits = (
            self.negative_head(
                pooled
            )
        )

        attribute_logits = (
            self.attribute_head(
                pooled
            )
        )

        # ----------------------------------------------------
        # MAIN V4:
        # predicted structured values
        # NOT ground truth
        # ----------------------------------------------------

        concept_pred = (
            torch.sigmoid(
                concept_logits
            ) >= 0.5
        ).float()

        negative_pred = (
            torch.sigmoid(
                negative_logits
            ) >= 0.5
        ).float()

        attribute_pred = (
            torch.sigmoid(
                attribute_logits
            ) >= 0.5
        ).float()

        structured_tokens, structured_mask = (
            self.conditioner(
                concept_pred,
                negative_pred,
                attribute_pred
            )
        )

        prefix = torch.cat(
            [
                visual_tokens,
                structured_tokens
            ],
            dim=1
        )

        prefix_mask = torch.cat(
            [
                visual_mask,
                structured_mask
            ],
            dim=1
        )

        encoder_outputs = (
            self.mt5.encoder(
                inputs_embeds=prefix,
                attention_mask=prefix_mask,
                return_dict=True
            )
        )

        lm_outputs = self.mt5(
            encoder_outputs=encoder_outputs,
            attention_mask=prefix_mask,
            labels=labels,
            return_dict=True
        )

        report_loss = (
            lm_outputs.loss
        )

        concept_loss = (
            F.binary_cross_entropy_with_logits(
                concept_logits,
                concept_targets
            )
        )

        negative_loss = (
            F.binary_cross_entropy_with_logits(
                negative_logits,
                negative_targets
            )
        )

        attribute_loss = (
            F.binary_cross_entropy_with_logits(
                attribute_logits,
                attribute_targets
            )
        )

        total_loss = (
            report_loss
            + 0.5 * concept_loss
            + 0.5 * negative_loss
            + 0.25 * attribute_loss
        )

        return {
            "loss": total_loss,
            "report_loss": report_loss,
            "concept_loss": concept_loss,
            "negative_loss": negative_loss,
            "attribute_loss": attribute_loss,
            "concept_logits": concept_logits,
            "negative_logits": negative_logits,
            "attribute_logits": attribute_logits,
            "visual_tokens": visual_tokens,
            "structured_tokens": structured_tokens,
            "prefix": prefix,
        }


# ============================================================
# 22. MODEL LOAD TEST
# ============================================================

print("\n[22] MODEL CHECKPOINT LOAD TEST")
print("-" * 75)

model = V4Model().to(
    DEVICE
)

load_result = model.load_state_dict(
    state,
    strict=True
)

print(
    "✓ Strict checkpoint load succeeded."
)

print(
    load_result
)


# ============================================================
# 23. EXACT PARAMETER / CHECKPOINT MATCH
# ============================================================

model_state_keys = set(
    model.state_dict().keys()
)

checkpoint_state_keys = set(
    state.keys()
)

missing_keys = (
    model_state_keys
    - checkpoint_state_keys
)

unexpected_keys = (
    checkpoint_state_keys
    - model_state_keys
)

print(
    "\nMissing checkpoint keys:",
    len(missing_keys)
)

print(
    "Unexpected checkpoint keys:",
    len(unexpected_keys)
)

assert not missing_keys
assert not unexpected_keys


# ============================================================
# 24. BUILD READ-ONLY DATASET FOR ONE BATCH
# ============================================================

print("\n[24] ONE-BATCH DATA TEST")
print("-" * 75)


class AuditDataset(Dataset):

    def __init__(
        self,
        dataframe
    ):

        self.df = (
            dataframe
            .reset_index(drop=True)
        )

    def __len__(self):

        return len(self.df)

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        case_id = str(
            row["case_id"]
        )

        paths = image_groups[
            case_id
        ]

        images = []

        for path in paths:

            img = Image.open(
                path
            ).convert("RGB")

            img = transforms.Compose([
                transforms.Resize(
                    (224, 224)
                ),
                transforms.ToTensor(),
                transforms.Normalize(
                    [0.5]*3,
                    [0.5]*3
                )
            ])(img)

            images.append(img)

        report = str(
            row["ket_luan"]
        )

        encoded = tokenizer(
            report,
            max_length=96,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = encoded[
            "input_ids"
        ].squeeze(0)

        labels[
            labels == tokenizer.pad_token_id
        ] = -100

        target = structured_df.loc[
            case_id
        ]

        concept = torch.tensor(
            target[
                concept_columns
            ].values.astype(
                np.float32
            )
        )

        negative = torch.tensor(
            target[
                negative_columns
            ].values.astype(
                np.float32
            )
        )

        attribute = torch.tensor(
            target[
                attribute_columns
            ].values.astype(
                np.float32
            )
        )

        return {
            "case_id": case_id,
            "images": images[:8],
            "labels": labels,
            "concept": concept,
            "negative": negative,
            "attribute": attribute,
        }


def audit_collate(batch):

    return {
        "case_id": [
            x["case_id"]
            for x in batch
        ],

        "images": [
            x["images"]
            for x in batch
        ],

        "labels": torch.stack([
            x["labels"]
            for x in batch
        ]),

        "concept": torch.stack([
            x["concept"]
            for x in batch
        ]),

        "negative": torch.stack([
            x["negative"]
            for x in batch
        ]),

        "attribute": torch.stack([
            x["attribute"]
            for x in batch
        ]),
    }


audit_loader = DataLoader(
    AuditDataset(
        v4_manifest.head(4)
    ),
    batch_size=4,
    shuffle=False,
    num_workers=0,
    collate_fn=audit_collate
)

audit_batch = next(
    iter(audit_loader)
)

print(
    "Case IDs:",
    audit_batch["case_id"]
)

print(
    "Image counts:",
    [
        len(x)
        for x in audit_batch["images"]
    ]
)

print(
    "Labels:",
    audit_batch["labels"].shape
)

print(
    "Concept:",
    audit_batch["concept"].shape
)

print(
    "Negative:",
    audit_batch["negative"].shape
)

print(
    "Attribute:",
    audit_batch["attribute"].shape
)


# ============================================================
# 25. ONE-BATCH FORWARD DRY RUN
# ============================================================

print("\n[25] ONE-BATCH FORWARD DRY RUN")
print("-" * 75)

model.eval()

labels = audit_batch[
    "labels"
].to(
    DEVICE
)

concept = audit_batch[
    "concept"
].to(
    DEVICE
)

negative = audit_batch[
    "negative"
].to(
    DEVICE
)

attribute = audit_batch[
    "attribute"
].to(
    DEVICE
)

with torch.no_grad():

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
        enabled=USE_BF16
    ):

        outputs = model(
            images=audit_batch[
                "images"
            ],
            labels=labels,
            concept_targets=concept,
            negative_targets=negative,
            attribute_targets=attribute
        )

print(
    "Total loss:",
    float(
        outputs["loss"]
    )
)

print(
    "Report loss:",
    float(
        outputs["report_loss"]
    )
)

print(
    "Concept logits:",
    tuple(
        outputs[
            "concept_logits"
        ].shape
    )
)

print(
    "Negative logits:",
    tuple(
        outputs[
            "negative_logits"
        ].shape
    )
)

print(
    "Attribute logits:",
    tuple(
        outputs[
            "attribute_logits"
        ].shape
    )
)

print(
    "Visual tokens:",
    tuple(
        outputs[
            "visual_tokens"
        ].shape
    )
)

print(
    "Structured tokens:",
    tuple(
        outputs[
            "structured_tokens"
        ].shape
    )
)

print(
    "Prefix:",
    tuple(
        outputs[
            "prefix"
        ].shape
    )
)


# ============================================================
# 26. CHECKPOINT IMMUTABILITY
# ============================================================

print("\n[26] CHECKPOINT IMMUTABILITY CHECK")
print("-" * 75)

print(
    "No checkpoint has been saved."
)

print(
    "No optimizer was created."
)

print(
    "No backward() was called."
)

print(
    "No optimizer.step() was called."
)

print(
    "No scheduler.step() was called."
)


# ============================================================
# 27. FINAL AUDIT SUMMARY
# ============================================================

print("\n")
print("=" * 75)
print("AUDIT COMPLETE — NO TRAINING PERFORMED")
print("=" * 75)

print(
    "V4 manifest rows       :",
    len(v4_manifest)
)

print(
    "Train cases             :",
    (
        v4_manifest["split"]
        == "train"
    ).sum()
)

print(
    "Val cases               :",
    (
        v4_manifest["split"]
        == "val"
    ).sum()
)

print(
    "NORMAL images           :",
    len(normal_images)
)

print(
    "Structured cases        :",
    len(structured_df)
)

print(
    "Concept dimensions      :",
    len(concept_columns)
)

print(
    "Negative dimensions     :",
    len(negative_columns)
)

print(
    "Attribute dimensions    :",
    len(attribute_columns)
)

print(
    "Checkpoint epoch        :",
    state is not None and 2
)

print(
    "Checkpoint strict load  : PASS"
)

print(
    "One-batch forward       : PASS"
)

print(
    "Checkpoint modified     : NO"
)

print("\n")
print("STOP HERE.")
print(
    "Do NOT train yet."
)
print(
    "Send me this entire audit output."
)
print(
    "Only after reviewing it will we write the Epoch-3 cell."
)

# Cleanup GPU memory while keeping checkpoint untouched
del model
del outputs
del audit_batch
del audit_loader

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print("\n✓ Audit cell finished safely.")

V4 COMPLETE PRE-TRAIN AUDIT

[1] FILE EXISTENCE
---------------------------------------------------------------------------
✓ V4 manifest              : /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/v4_training_manifest.csv
✓ Image manifest           : /content/drive/MyDrive/NoiSoi_Matching/final_manifest/final_dataset_manifest.csv
✓ Structured targets       : /content/drive/MyDrive/NoiSoi_Matching/v3_ontology_v2/v3_structured_targets_v2_2.csv
✓ Ontology mapping         : /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/diagnostics/frozen_concept_ontology_mapping.csv
✓ V4 config                : /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/v4_config.json
✓ V3 checkpoint            : /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v3/checkpoints/best.pt
✓ V4 last checkpoint       : /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/checkpoints/last.pt
✓ V4 best checkpoint       : /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/checkpoi

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


RuntimeError: Error(s) in loading state_dict for V4Model:
	Missing key(s) in state_dict: "concept_head.weight", "concept_head.bias", "negative_head.weight", "negative_head.bias", "attribute_head.weight", "attribute_head.bias". 
	Unexpected key(s) in state_dict: "structured_heads.concept.weight", "structured_heads.concept.bias", "structured_heads.negative.weight", "structured_heads.negative.bias", "structured_heads.attribute.weight", "structured_heads.attribute.bias". 

In [13]:
# ============================================================
# V4 FINAL COMPATIBILITY AUDIT
#
# This is the LAST audit before training Epoch 3.
#
# NO:
#   backward()
#   optimizer.step()
#   scheduler.step()
#   checkpoint save
#
# Goal:
#   Reconstruct the EXACT checkpoint architecture/naming.
# ============================================================

import os
import gc
import math
import warnings

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    ViTModel,
)

warnings.filterwarnings("ignore")


# ============================================================
# 0. PATHS
# ============================================================

BASE_DIR = "/content/drive/MyDrive/NoiSoi_Matching"

V4_DIR = os.path.join(
    BASE_DIR,
    "baseline_model_v4"
)

LAST_CHECKPOINT = os.path.join(
    V4_DIR,
    "checkpoints",
    "last.pt"
)

V4_MANIFEST = os.path.join(
    V4_DIR,
    "v4_training_manifest.csv"
)

IMAGE_MANIFEST = os.path.join(
    BASE_DIR,
    "final_manifest",
    "final_dataset_manifest.csv"
)

STRUCTURED_TARGETS = os.path.join(
    BASE_DIR,
    "v3_ontology_v2",
    "v3_structured_targets_v2_2.csv"
)

ONTOLOGY_MAPPING = os.path.join(
    V4_DIR,
    "diagnostics",
    "frozen_concept_ontology_mapping.csv"
)

VIT_NAME = "google/vit-base-patch16-224"
MT5_NAME = "google/mt5-small"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

USE_BF16 = (
    DEVICE.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

IMAGE_SIZE = 224
MAX_IMAGES = 8
MAX_TARGET_LENGTH = 96

D_MODEL = 512

NUM_CONCEPTS = 48
NUM_NEGATIVE = 5
NUM_ATTRIBUTES = 6

BATCH_SIZE = 4
GRAD_ACCUM = 4
TOTAL_EPOCHS = 5


# ============================================================
# 1. LOAD DATA
# ============================================================

print("=" * 70)
print("V4 FINAL COMPATIBILITY AUDIT")
print("=" * 70)

v4_manifest = pd.read_csv(
    V4_MANIFEST
)

image_manifest = pd.read_csv(
    IMAGE_MANIFEST
)

structured_df = pd.read_csv(
    STRUCTURED_TARGETS,
    index_col="case_id"
)

ontology_map = pd.read_csv(
    ONTOLOGY_MAPPING
)

checkpoint = torch.load(
    LAST_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

assert checkpoint["epoch"] == 2

state = checkpoint[
    "model_state"
]

print("✓ Data loaded")
print("✓ Epoch-2 checkpoint loaded")


# ============================================================
# 2. EXACT ONTOLOGY
# ============================================================

concept_columns = (
    ontology_map[
        "column"
    ].tolist()
)

negative_columns = [
    "negative__no_abnormal_external_middle_ear",
    "negative__no_abnormal_nose_sinus",
    "negative__no_abnormal_ent",
    "negative__no_bleeding",
    "negative__no_foreign_body",
]

attribute_columns = [
    "attribute__acute",
    "attribute__chronic",
    "attribute__right",
    "attribute__left",
    "attribute__bilateral",
    "attribute__post_surgery",
]

assert len(concept_columns) == 48
assert len(negative_columns) == 5
assert len(attribute_columns) == 6


# ============================================================
# 3. IMAGE GROUPS
# ============================================================

normal_images = image_manifest[
    image_manifest[
        "image_status"
    ].astype(str) == "NORMAL"
].copy()

normal_images[
    "case_id"
] = normal_images[
    "case_id"
].astype(str)

image_groups = {}

for case_id, group in normal_images.groupby(
    "case_id"
):

    paths = (
        group[
            "image_path"
        ]
        .astype(str)
        .tolist()
    )

    image_groups[
        case_id
    ] = paths[:MAX_IMAGES]

print(
    "✓ Image groups:",
    len(image_groups)
)


# ============================================================
# 4. TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MT5_NAME
)


# ============================================================
# 5. TRANSFORM
# ============================================================

image_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.5, 0.5, 0.5],
        [0.5, 0.5, 0.5]
    ),
])


# ============================================================
# 6. EXACT CONDITIONER
# ============================================================

class StructuredConditioner(nn.Module):

    def __init__(
        self,
        d_model,
        num_concepts=48,
        num_negative=5,
        num_attributes=6
    ):

        super().__init__()

        self.num_concepts = (
            num_concepts
        )

        self.num_negative = (
            num_negative
        )

        self.num_attributes = (
            num_attributes
        )

        self.concept_value = nn.Embedding(
            2,
            d_model
        )

        self.negative_value = nn.Embedding(
            2,
            d_model
        )

        self.attribute_value = nn.Embedding(
            2,
            d_model
        )

        self.concept_label = nn.Embedding(
            num_concepts,
            d_model
        )

        self.negative_label = nn.Embedding(
            num_negative,
            d_model
        )

        self.attribute_label = nn.Embedding(
            num_attributes,
            d_model
        )

        self.type_embedding = nn.Embedding(
            3,
            d_model
        )

        self.norm = nn.LayerNorm(
            d_model
        )

    def forward(
        self,
        concept_targets,
        negative_targets,
        attribute_targets
    ):

        device = concept_targets.device

        concept_ids = (
            concept_targets
            .long()
            .clamp(0, 1)
        )

        negative_ids = (
            negative_targets
            .long()
            .clamp(0, 1)
        )

        attribute_ids = (
            attribute_targets
            .long()
            .clamp(0, 1)
        )

        # ----------------------------------------------------
        # CONCEPT
        # ----------------------------------------------------

        concept_idx = torch.arange(
            self.num_concepts,
            device=device
        )

        concept_type = self.type_embedding(
            torch.zeros(
                self.num_concepts,
                dtype=torch.long,
                device=device
            )
        )

        concept_tokens = (
            self.concept_value(
                concept_ids
            )
            +
            self.concept_label(
                concept_idx
            )[None, :, :]
            +
            concept_type[None, :, :]
        )

        # ----------------------------------------------------
        # NEGATIVE
        # ----------------------------------------------------

        negative_idx = torch.arange(
            self.num_negative,
            device=device
        )

        negative_type = self.type_embedding(
            torch.ones(
                self.num_negative,
                dtype=torch.long,
                device=device
            )
        )

        negative_tokens = (
            self.negative_value(
                negative_ids
            )
            +
            self.negative_label(
                negative_idx
            )[None, :, :]
            +
            negative_type[None, :, :]
        )

        # ----------------------------------------------------
        # ATTRIBUTE
        # ----------------------------------------------------

        attribute_idx = torch.arange(
            self.num_attributes,
            device=device
        )

        attribute_type = self.type_embedding(
            torch.full(
                (
                    self.num_attributes,
                ),
                2,
                dtype=torch.long,
                device=device
            )
        )

        attribute_tokens = (
            self.attribute_value(
                attribute_ids
            )
            +
            self.attribute_label(
                attribute_idx
            )[None, :, :]
            +
            attribute_type[None, :, :]
        )

        structured_tokens = torch.cat(
            [
                concept_tokens,
                negative_tokens,
                attribute_tokens,
            ],
            dim=1
        )

        structured_tokens = self.norm(
            structured_tokens
        )

        B = concept_targets.shape[0]

        structured_attention = torch.ones(
            B,
            structured_tokens.shape[1],
            dtype=torch.long,
            device=device
        )

        return (
            structured_tokens,
            structured_attention
        )


# ============================================================
# 7. EXACT V4 MODEL NAMING
#
# CRITICAL:
# Checkpoint uses:
#
# structured_heads.concept
# structured_heads.negative
# structured_heads.attribute
#
# NOT:
# concept_head
# negative_head
# attribute_head
# ============================================================

class VisualProjector(nn.Module):

    def __init__(
        self,
        in_dim=768,
        out_dim=512
    ):

        super().__init__()

        self.proj = nn.Linear(
            in_dim,
            out_dim
        )

    def forward(self, x):

        return self.proj(x)


class V4Model(nn.Module):

    def __init__(self):

        super().__init__()

        self.vision = ViTModel.from_pretrained(
            VIT_NAME
        )

        self.projector = VisualProjector(
            768,
            512
        )

        self.mt5 = AutoModelForSeq2SeqLM.from_pretrained(
            MT5_NAME
        )

        # EXACT CHECKPOINT STRUCTURE
        self.structured_heads = nn.ModuleDict({
            "concept": nn.Linear(
                512,
                48
            ),

            "negative": nn.Linear(
                512,
                5
            ),

            "attribute": nn.Linear(
                512,
                6
            )
        })

        self.conditioner = StructuredConditioner(
            512,
            48,
            5,
            6
        )

    def encode_images(
        self,
        images
    ):

        visual_tokens = []

        for image_list in images:

            x = torch.stack(
                image_list,
                dim=0
            ).to(
                DEVICE,
                non_blocking=True
            )

            out = self.vision(
                pixel_values=x
            )

            cls = out.last_hidden_state[
                :,
                0,
                :
            ]

            cls = self.projector(
                cls
            )

            visual_tokens.append(
                cls
            )

        max_n = max(
            x.shape[0]
            for x in visual_tokens
        )

        padded = []
        masks = []

        for x in visual_tokens:

            n = x.shape[0]

            if n < max_n:

                pad = torch.zeros(
                    max_n - n,
                    512,
                    dtype=x.dtype,
                    device=x.device
                )

                x = torch.cat(
                    [x, pad],
                    dim=0
                )

            padded.append(x)

            mask = torch.zeros(
                max_n,
                dtype=torch.long,
                device=x.device
            )

            mask[:n] = 1

            masks.append(mask)

        return (
            torch.stack(padded),
            torch.stack(masks)
        )

    def forward(
        self,
        images,
        labels,
        concept_targets,
        negative_targets,
        attribute_targets
    ):

        visual_tokens, visual_mask = (
            self.encode_images(
                images
            )
        )

        denom = visual_mask.sum(
            dim=1,
            keepdim=True
        ).clamp(min=1)

        pooled = (
            visual_tokens
            * visual_mask.unsqueeze(-1)
        ).sum(
            dim=1
        ) / denom

        # EXACT CHECKPOINT HEAD STRUCTURE

        concept_logits = (
            self.structured_heads[
                "concept"
            ](
                pooled
            )
        )

        negative_logits = (
            self.structured_heads[
                "negative"
            ](
                pooled
            )
        )

        attribute_logits = (
            self.structured_heads[
                "attribute"
            ](
                pooled
            )
        )

        # Main V4 uses predictions,
        # NOT ground truth, for conditioning.

        concept_pred = (
            torch.sigmoid(
                concept_logits
            ) >= 0.5
        ).float()

        negative_pred = (
            torch.sigmoid(
                negative_logits
            ) >= 0.5
        ).float()

        attribute_pred = (
            torch.sigmoid(
                attribute_logits
            ) >= 0.5
        ).float()

        structured_tokens, structured_mask = (
            self.conditioner(
                concept_pred,
                negative_pred,
                attribute_pred
            )
        )

        prefix = torch.cat(
            [
                visual_tokens,
                structured_tokens
            ],
            dim=1
        )

        prefix_mask = torch.cat(
            [
                visual_mask,
                structured_mask
            ],
            dim=1
        )

        encoder_outputs = self.mt5.encoder(
            inputs_embeds=prefix,
            attention_mask=prefix_mask,
            return_dict=True
        )

        lm_outputs = self.mt5(
            encoder_outputs=encoder_outputs,
            attention_mask=prefix_mask,
            labels=labels,
            return_dict=True
        )

        report_loss = (
            lm_outputs.loss
        )

        concept_loss = (
            F.binary_cross_entropy_with_logits(
                concept_logits,
                concept_targets
            )
        )

        negative_loss = (
            F.binary_cross_entropy_with_logits(
                negative_logits,
                negative_targets
            )
        )

        attribute_loss = (
            F.binary_cross_entropy_with_logits(
                attribute_logits,
                attribute_targets
            )
        )

        total_loss = (
            report_loss
            + 0.5 * concept_loss
            + 0.5 * negative_loss
            + 0.25 * attribute_loss
        )

        return {
            "loss": total_loss,
            "report_loss": report_loss,
            "concept_loss": concept_loss,
            "negative_loss": negative_loss,
            "attribute_loss": attribute_loss,
            "concept_logits": concept_logits,
            "negative_logits": negative_logits,
            "attribute_logits": attribute_logits,
            "visual_tokens": visual_tokens,
            "structured_tokens": structured_tokens,
            "prefix": prefix,
        }


# ============================================================
# 8. CREATE MODEL
# ============================================================

print("\n[1] CREATE EXACT MODEL")
print("-" * 70)

model = V4Model().to(
    DEVICE
)

print(
    "✓ Model instantiated"
)


# ============================================================
# 9. EXACT STATE-DICT CHECK
# ============================================================

print("\n[2] STATE-DICT COMPATIBILITY")
print("-" * 70)

model_keys = set(
    model.state_dict().keys()
)

checkpoint_keys = set(
    state.keys()
)

missing = (
    model_keys
    - checkpoint_keys
)

unexpected = (
    checkpoint_keys
    - model_keys
)

print(
    "Model keys:",
    len(model_keys)
)

print(
    "Checkpoint keys:",
    len(checkpoint_keys)
)

print(
    "Missing:",
    len(missing)
)

print(
    "Unexpected:",
    len(unexpected)
)

if missing:
    print("\nMISSING:")
    for x in sorted(missing):
        print(" ", x)

if unexpected:
    print("\nUNEXPECTED:")
    for x in sorted(unexpected):
        print(" ", x)

assert not missing
assert not unexpected

print(
    "✓ Exact key sets match"
)


# ============================================================
# 10. STRICT LOAD
# ============================================================

print("\n[3] STRICT CHECKPOINT LOAD")
print("-" * 70)

model.load_state_dict(
    state,
    strict=True
)

print(
    "✓ STRICT LOAD PASS"
)


# ============================================================
# 11. EXACT OPTIMIZER STRUCTURE
#
# Checkpoint has exactly 5 groups:
#
# 0 vision
# 1 projector
# 2 conditioner
# 3 structured heads
# 4 mt5
# ============================================================

print("\n[4] OPTIMIZER STRUCTURE")
print("-" * 70)

optimizer = torch.optim.AdamW(
    [
        {
            "params":
                model.vision.parameters(),
            "lr": 1e-5,
        },

        {
            "params":
                model.projector.parameters(),
            "lr": 1e-4,
        },

        {
            "params":
                model.conditioner.parameters(),
            "lr": 1e-4,
        },

        {
            "params":
                model.structured_heads.parameters(),
            "lr": 1e-4,
        },

        {
            "params":
                model.mt5.parameters(),
            "lr": 5e-5,
        },
    ],
    weight_decay=0.01
)

print(
    "Optimizer groups:",
    len(
        optimizer.param_groups
    )
)

assert len(
    optimizer.param_groups
) == 5

for i, group in enumerate(
    optimizer.param_groups
):

    print(
        f"Group {i}: "
        f"lr={group['lr']}"
    )

print(
    "✓ Optimizer has exact 5 groups"
)


# ============================================================
# 12. OPTIMIZER STATE LOAD
# ============================================================

print("\n[5] OPTIMIZER CHECKPOINT LOAD")
print("-" * 70)

optimizer.load_state_dict(
    checkpoint[
        "optimizer_state"
    ]
)

print(
    "✓ Optimizer state load PASS"
)

print(
    "Restored LRs:"
)

for i, group in enumerate(
    optimizer.param_groups
):

    print(
        f"  Group {i}: "
        f"{group['lr']:.12g}"
    )


# ============================================================
# 13. SCHEDULER
# ============================================================

steps_per_epoch = math.ceil(
    1535 / 4
)

assert steps_per_epoch == 384

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=384 * 5
)

scheduler.load_state_dict(
    checkpoint[
        "scheduler_state"
    ]
)

print("\n[6] SCHEDULER")
print("-" * 70)

print(
    "T_max:",
    scheduler.T_max
)

print(
    "last_epoch:",
    scheduler.last_epoch
)

print(
    "_step_count:",
    scheduler._step_count
)

assert scheduler.T_max == 1920
assert scheduler.last_epoch == 768

print(
    "✓ Scheduler state load PASS"
)


# ============================================================
# 14. BUILD ONE-BATCH DATASET
# ============================================================

print("\n[7] ONE-BATCH DATA")
print("-" * 70)


class AuditDataset(Dataset):

    def __init__(
        self,
        dataframe
    ):

        self.df = (
            dataframe
            .reset_index(drop=True)
        )

    def __len__(self):

        return len(self.df)

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        case_id = str(
            row["case_id"]
        )

        images = []

        for path in image_groups[
            case_id
        ][:MAX_IMAGES]:

            img = Image.open(
                path
            ).convert("RGB")

            img = image_transform(
                img
            )

            images.append(img)

        report = str(
            row["ket_luan"]
        )

        encoded = tokenizer(
            report,
            max_length=MAX_TARGET_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = encoded[
            "input_ids"
        ].squeeze(0)

        labels[
            labels == tokenizer.pad_token_id
        ] = -100

        target = structured_df.loc[
            case_id
        ]

        concept = torch.tensor(
            target[
                concept_columns
            ].values.astype(
                np.float32
            )
        )

        negative = torch.tensor(
            target[
                negative_columns
            ].values.astype(
                np.float32
            )
        )

        attribute = torch.tensor(
            target[
                attribute_columns
            ].values.astype(
                np.float32
            )
        )

        return {
            "case_id": case_id,
            "images": images,
            "labels": labels,
            "concept": concept,
            "negative": negative,
            "attribute": attribute,
        }


def audit_collate(batch):

    return {
        "case_id": [
            x["case_id"]
            for x in batch
        ],

        "images": [
            x["images"]
            for x in batch
        ],

        "labels": torch.stack([
            x["labels"]
            for x in batch
        ]),

        "concept": torch.stack([
            x["concept"]
            for x in batch
        ]),

        "negative": torch.stack([
            x["negative"]
            for x in batch
        ]),

        "attribute": torch.stack([
            x["attribute"]
            for x in batch
        ]),
    }


audit_loader = DataLoader(
    AuditDataset(
        v4_manifest[
            v4_manifest["split"] == "train"
        ].head(4)
    ),
    batch_size=4,
    shuffle=False,
    num_workers=0,
    collate_fn=audit_collate
)

batch = next(
    iter(audit_loader)
)

print(
    "Cases:",
    batch["case_id"]
)

print(
    "Images/case:",
    [
        len(x)
        for x in batch["images"]
    ]
)

print(
    "Labels:",
    tuple(
        batch["labels"].shape
    )
)

print(
    "Concept:",
    tuple(
        batch["concept"].shape
    )
)

print(
    "Negative:",
    tuple(
        batch["negative"].shape
    )
)

print(
    "Attribute:",
    tuple(
        batch["attribute"].shape
    )
)


# ============================================================
# 15. FORWARD DRY RUN
# ============================================================

print("\n[8] FORWARD DRY RUN")
print("-" * 70)

model.eval()

with torch.no_grad():

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
        enabled=USE_BF16
    ):

        out = model(
            images=batch["images"],
            labels=batch["labels"].to(
                DEVICE
            ),
            concept_targets=batch[
                "concept"
            ].to(DEVICE),
            negative_targets=batch[
                "negative"
            ].to(DEVICE),
            attribute_targets=batch[
                "attribute"
            ].to(DEVICE),
        )

print(
    "Total loss:",
    float(
        out["loss"]
    )
)

print(
    "Report loss:",
    float(
        out["report_loss"]
    )
)

print(
    "Concept logits:",
    tuple(
        out[
            "concept_logits"
        ].shape
    )
)

print(
    "Negative logits:",
    tuple(
        out[
            "negative_logits"
        ].shape
    )
)

print(
    "Attribute logits:",
    tuple(
        out[
            "attribute_logits"
        ].shape
    )
)

print(
    "Visual tokens:",
    tuple(
        out[
            "visual_tokens"
        ].shape
    )
)

print(
    "Structured tokens:",
    tuple(
        out[
            "structured_tokens"
        ].shape
    )
)

print(
    "Prefix:",
    tuple(
        out[
            "prefix"
        ].shape
    )
)

assert (
    out["concept_logits"].shape
    == (4, 48)
)

assert (
    out["negative_logits"].shape
    == (4, 5)
)

assert (
    out["attribute_logits"].shape
    == (4, 6)
)

assert (
    out["structured_tokens"].shape[1]
    == 59
)

assert (
    out["prefix"].shape[1]
    == out["visual_tokens"].shape[1] + 59
)

print(
    "✓ Forward shape checks PASS"
)


# ============================================================
# 16. CHECK NO NaN
# ============================================================

print("\n[9] NUMERICAL SANITY")
print("-" * 70)

for name, value in out.items():

    if torch.is_tensor(value):

        finite = torch.isfinite(
            value
        ).all().item()

        print(
            f"{name:25s}: "
            f"{'FINITE' if finite else 'NaN/INF'}"
        )

        assert finite

print(
    "✓ No NaN / Inf in forward output"
)


# ============================================================
# 17. FINAL
# ============================================================

print("\n")
print("=" * 70)
print("FINAL COMPATIBILITY AUDIT RESULT")
print("=" * 70)

print("✓ V4 manifest schema")
print("✓ ket_luan report target")
print("✓ image_status handling")
print("✓ NORMAL image coverage")
print("✓ structured ontology")
print("✓ patient-level split")
print("✓ Epoch-2 checkpoint")
print("✓ conditioner architecture")
print("✓ structured_heads naming")
print("✓ exact state_dict")
print("✓ exact 5 optimizer groups")
print("✓ optimizer state")
print("✓ cosine scheduler state")
print("✓ one-batch forward")
print("✓ output shapes")
print("✓ no NaN / Inf")

print("\nNO TRAINING WAS PERFORMED.")
print("NO CHECKPOINT WAS SAVED.")
print("NO CHECKPOINT WAS MODIFIED.")

print("\nSTOP.")
print("If every item above says PASS, we can write the final Epoch-3 training cell.")

V4 FINAL COMPATIBILITY AUDIT
✓ Data loaded
✓ Epoch-2 checkpoint loaded
✓ Image groups: 7607

[1] CREATE EXACT MODEL
----------------------------------------------------------------------


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✓ Model instantiated

[2] STATE-DICT COMPATIBILITY
----------------------------------------------------------------------
Model keys: 409
Checkpoint keys: 409
Missing: 0
Unexpected: 0
✓ Exact key sets match

[3] STRICT CHECKPOINT LOAD
----------------------------------------------------------------------
✓ STRICT LOAD PASS

[4] OPTIMIZER STRUCTURE
----------------------------------------------------------------------
Optimizer groups: 5
Group 0: lr=1e-05
Group 1: lr=0.0001
Group 2: lr=0.0001
Group 3: lr=0.0001
Group 4: lr=5e-05
✓ Optimizer has exact 5 groups

[5] OPTIMIZER CHECKPOINT LOAD
----------------------------------------------------------------------
✓ Optimizer state load PASS
Restored LRs:
  Group 0: 6.54508497187e-06
  Group 1: 6.54508497187e-05
  Group 2: 6.54508497187e-05
  Group 3: 6.54508497187e-05
  Group 4: 3.27254248594e-05

[6] SCHEDULER
----------------------------------------------------------------------
T_max: 1920
last_epoch: 768
_step_count: 769
✓ Scheduler sta

In [26]:
# ================================================================
# V4 — EPOCH 3 TRAINING
# Resume EXACTLY from Epoch-2 checkpoint
# ================================================================

import os
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import ViTModel, MT5ForConditionalGeneration
from transformers.models.mt5.tokenization_mt5 import MT5Tokenizer
from PIL import Image
from contextlib import nullcontext

# ------------------------------------------------
# 0. CONFIG
# ------------------------------------------------
BASE = "/content/drive/MyDrive/NoiSoi_Matching"

MANIFEST_PATH = f"{BASE}/baseline_model_v4/v4_training_manifest.csv"
STRUCTURED_PATH = (
    f"{BASE}/v3_ontology_v2/"
    f"v3_structured_targets_v2_2.csv"
)
CKPT_DIR = f"{BASE}/baseline_model_v4/checkpoints"

LAST_CKPT = f"{CKPT_DIR}/last.pt"
EPOCH2_CKPT = f"{CKPT_DIR}/epoch_02.pt"
BEST_CKPT = f"{CKPT_DIR}/best.pt"

HISTORY_CSV = f"{BASE}/baseline_model_v4/v4_training_history.csv"

os.makedirs(CKPT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 4
GRAD_ACCUM = 4
MAX_IMAGES = 8
MAX_TARGET_LENGTH = 96

NUM_CONCEPTS = 48
NUM_NEGATIVE = 5
NUM_ATTRIBUTES = 6

VISION_LR = 1e-5
PROJECTOR_LR = 1e-4
STRUCTURED_LR = 1e-4
MT5_LR = 5e-5
WEIGHT_DECAY = 0.01

TOTAL_EPOCHS = 5

print("=" * 70)
print("V4 EPOCH 3 TRAINING")
print("=" * 70)
print("Device:", DEVICE)
print("Checkpoint:", LAST_CKPT)


# ------------------------------------------------
# 1. SEED
# ------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)
structured_df = pd.read_csv(STRUCTURED_PATH, index_col=0)

# Ensure case_id is string consistently
manifest["case_id"] = manifest["case_id"].astype(str)
structured_df.index = structured_df.index.astype(str)

print("Manifest:", manifest.shape)
print("Structured:", structured_df.shape)

# Exact V4 case set
case_df = (
    manifest[
        manifest["case_id"].isin(structured_df.index)
    ]
    .drop_duplicates("case_id")
    .copy()
)

assert len(case_df) == 7606, (
    f"Expected 7606 V4 cases, got {len(case_df)}"
)

assert set(case_df["case_id"]) == set(structured_df.index), \
    "V4 case IDs do not exactly align."


# ------------------------------------------------
# 3. IMAGE GROUPS
# ------------------------------------------------
normal_manifest = manifest[
    manifest["image_status"].astype(str).str.upper() == "NORMAL"
].copy()

normal_manifest = normal_manifest[
    normal_manifest["case_id"].isin(case_df["case_id"])
].copy()

image_groups = {
    case_id: g.sort_values("image_order")
    for case_id, g in normal_manifest.groupby("case_id")
}

assert len(image_groups) == 7606

for case_id in case_df["case_id"]:
    assert case_id in image_groups
    assert len(image_groups[case_id]) >= 1

print("Image groups:", len(image_groups))


# ------------------------------------------------
# 4. TOKENIZER
# ------------------------------------------------
tokenizer = MT5Tokenizer.from_pretrained("google/mt5-small")


# ------------------------------------------------
# 5. DATASET
# ------------------------------------------------
class V4Dataset(Dataset):

    def __init__(self, cases, split):
        self.df = cases[cases["split"] == split].copy()
        self.df = self.df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]
        case_id = str(row["case_id"])

        imgs = image_groups[case_id]

        # deterministic first MAX_IMAGES
        imgs = imgs.iloc[:MAX_IMAGES]

        pixel_values = []

        for _, r in imgs.iterrows():

            path = r["image_path"]

            img = Image.open(path).convert("RGB")

            # Exact preprocessing used by the existing model
            img = img.resize((224, 224))

            arr = np.asarray(img).astype(np.float32) / 255.0

            arr = torch.from_numpy(arr).permute(2, 0, 1)

            # ViT normalization
            mean = torch.tensor(
                [0.5, 0.5, 0.5]
            ).view(3, 1, 1)

            std = torch.tensor(
                [0.5, 0.5, 0.5]
            ).view(3, 1, 1)

            arr = (arr - mean) / std

            pixel_values.append(arr)

        # pad to MAX_IMAGES
        n = len(pixel_values)

        while len(pixel_values) < MAX_IMAGES:
            pixel_values.append(
                torch.zeros_like(pixel_values[0])
            )

        pixel_values = torch.stack(pixel_values)

        image_mask = torch.zeros(MAX_IMAGES)

        image_mask[:n] = 1

        # -----------------------------
        # report
        # -----------------------------
        target = str(row["ket_luan"])

        tok = tokenizer(
            target,
            max_length=MAX_TARGET_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = tok["input_ids"].squeeze(0)

        labels[
            labels == tokenizer.pad_token_id
        ] = -100

        # -----------------------------
        # structured targets
        # -----------------------------
        s = structured_df.loc[case_id]

        concept_cols = [
            c for c in structured_df.columns
            if c.startswith("concept__")
        ]

        negative_cols = [
            c for c in structured_df.columns
            if c.startswith("negative__")
        ]

        attribute_cols = [
            c for c in structured_df.columns
            if c.startswith("attribute__")
        ]

        concepts = torch.tensor(
            s[concept_cols].astype(float).values,
            dtype=torch.float32
        )

        negatives = torch.tensor(
            s[negative_cols].astype(float).values,
            dtype=torch.float32
        )

        attributes = torch.tensor(
            s[attribute_cols].astype(float).values,
            dtype=torch.float32
        )

        return {
            "case_id": case_id,
            "pixel_values": pixel_values,
            "image_mask": image_mask,
            "labels": labels,
            "concept_targets": concepts,
            "negative_targets": negatives,
            "attribute_targets": attributes,
        }


# ------------------------------------------------
# 6. DATALOADERS
# ------------------------------------------------
train_ds = V4Dataset(case_df, "train")
val_ds = V4Dataset(case_df, "val")

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=False,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False,
)

print("Train cases:", len(train_ds))
print("Val cases:", len(val_ds))


# ------------------------------------------------
# 7. EXACT V4 ARCHITECTURE
# ------------------------------------------------
class StructuredConditioner(nn.Module):

    def __init__(
        self,
        d_model,
        num_concepts=48,
        num_negative=5,
        num_attributes=6
    ):
        super().__init__()

        self.num_concepts = num_concepts
        self.num_negative = num_negative
        self.num_attributes = num_attributes

        self.concept_value = nn.Embedding(2, d_model)
        self.negative_value = nn.Embedding(2, d_model)
        self.attribute_value = nn.Embedding(2, d_model)

        self.concept_label = nn.Embedding(
            num_concepts, d_model
        )
        self.negative_label = nn.Embedding(
            num_negative, d_model
        )
        self.attribute_label = nn.Embedding(
            num_attributes, d_model
        )

        self.type_embedding = nn.Embedding(3, d_model)

        self.norm = nn.LayerNorm(d_model)

    def forward(
        self,
        concept_targets,
        negative_targets,
        attribute_targets
    ):

        device = concept_targets.device

        concept_ids = concept_targets.long().clamp(0, 1)
        negative_ids = negative_targets.long().clamp(0, 1)
        attribute_ids = attribute_targets.long().clamp(0, 1)

        concept_idx = torch.arange(
            self.num_concepts,
            device=device
        )

        concept_type = self.type_embedding(
            torch.zeros(
                self.num_concepts,
                dtype=torch.long,
                device=device
            )
        )

        concept_tokens = (
            self.concept_value(concept_ids)
            + self.concept_label(concept_idx)[None, :, :]
            + concept_type[None, :, :]
        )

        negative_idx = torch.arange(
            self.num_negative,
            device=device
        )

        negative_type = self.type_embedding(
            torch.ones(
                self.num_negative,
                dtype=torch.long,
                device=device
            )
        )

        negative_tokens = (
            self.negative_value(negative_ids)
            + self.negative_label(negative_idx)[None, :, :]
            + negative_type[None, :, :]
        )

        attribute_idx = torch.arange(
            self.num_attributes,
            device=device
        )

        attribute_type = self.type_embedding(
            torch.full(
                (self.num_attributes,),
                2,
                dtype=torch.long,
                device=device
            )
        )

        attribute_tokens = (
            self.attribute_value(attribute_ids)
            + self.attribute_label(attribute_idx)[None, :, :]
            + attribute_type[None, :, :]
        )

        structured_tokens = torch.cat(
            [
                concept_tokens,
                negative_tokens,
                attribute_tokens
            ],
            dim=1
        )

        structured_tokens = self.norm(
            structured_tokens
        )

        B = concept_targets.shape[0]

        structured_attention = torch.ones(
            B,
            structured_tokens.shape[1],
            dtype=torch.long,
            device=device
        )

        return structured_tokens, structured_attention


class V4Model(nn.Module):

    def __init__(self):

        super().__init__()

        self.vision = ViTModel.from_pretrained(
            "google/vit-base-patch16-224"
        )

        self.mt5 = MT5ForConditionalGeneration.from_pretrained(
            "google/mt5-small"
        )

        self.projector = nn.Sequential(
    nn.Linear(768, 512)
)

        self.conditioner = StructuredConditioner(
            512,
            NUM_CONCEPTS,
            NUM_NEGATIVE,
            NUM_ATTRIBUTES
        )

        # IMPORTANT:
        # exact checkpoint naming
        self.structured_heads = nn.ModuleDict({
            "concept": nn.Linear(512, 48),
            "negative": nn.Linear(512, 5),
            "attribute": nn.Linear(512, 6),
        })

    def forward(
        self,
        pixel_values,
        image_mask,
        labels,
        concept_targets,
        negative_targets,
        attribute_targets
    ):

        B, N, C, H, W = pixel_values.shape

        x = pixel_values.reshape(
            B * N, C, H, W
        )

        vision_out = self.vision(
            pixel_values=x
        ).last_hidden_state[:, 0]

        vision_out = vision_out.reshape(
            B, N, 768
        )

        visual_tokens = self.projector(
            vision_out
        )

        # Mask padded image slots
        visual_tokens = (
            visual_tokens
            * image_mask[:, :, None]
        )

        # pooled representation for structured heads
        denom = image_mask.sum(
            dim=1,
            keepdim=True
        ).clamp(min=1)

        pooled = (
            visual_tokens.sum(dim=1)
            / denom
        )

        concept_logits = self.structured_heads[
            "concept"
        ](pooled)

        negative_logits = self.structured_heads[
            "negative"
        ](pooled)

        attribute_logits = self.structured_heads[
            "attribute"
        ](pooled)

        # IMPORTANT:
        # Preserve the existing V4 conditioning mechanism:
        # hard binary predictions -> conditioner.
        concept_pred = (
            torch.sigmoid(concept_logits) >= 0.5
        ).float()

        negative_pred = (
            torch.sigmoid(negative_logits) >= 0.5
        ).float()

        attribute_pred = (
            torch.sigmoid(attribute_logits) >= 0.5
        ).float()

        structured_tokens, structured_attention = (
            self.conditioner(
                concept_pred,
                negative_pred,
                attribute_pred
            )
        )

        visual_attention = (
            image_mask.long()
        )

        prefix = torch.cat(
            [
                visual_tokens,
                structured_tokens
            ],
            dim=1
        )

        attention_mask = torch.cat(
            [
                visual_attention,
                structured_attention
            ],
            dim=1
        )

        # mT5 input embeddings
        inputs_embeds = self.mt5.encoder.embed_tokens(
            torch.zeros(
                B,
                prefix.shape[1],
                dtype=torch.long,
                device=prefix.device
            )
        )

        inputs_embeds = (
            inputs_embeds + prefix
        )

        encoder_outputs = self.mt5.encoder(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            return_dict=True
        )

        out = self.mt5(
            encoder_outputs=encoder_outputs,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=True
        )

        # Auxiliary losses
        concept_loss = nn.functional.binary_cross_entropy_with_logits(
            concept_logits,
            concept_targets
        )

        negative_loss = nn.functional.binary_cross_entropy_with_logits(
            negative_logits,
            negative_targets
        )

        attribute_loss = nn.functional.binary_cross_entropy_with_logits(
            attribute_logits,
            attribute_targets
        )

        report_loss = out.loss

        total_loss = (
            report_loss
            + 0.5 * concept_loss
            + 0.5 * negative_loss
            + 0.25 * attribute_loss
        )

        return {
            "loss": total_loss,
            "report_loss": report_loss,
            "concept_loss": concept_loss,
            "negative_loss": negative_loss,
            "attribute_loss": attribute_loss,
            "concept_logits": concept_logits,
            "negative_logits": negative_logits,
            "attribute_logits": attribute_logits,
        }


# ------------------------------------------------
# 8. CREATE MODEL
# ------------------------------------------------
model = V4Model().to(DEVICE)

print("Model created.")


# ------------------------------------------------
# 9. EXACT OPTIMIZER — 5 GROUPS
# ------------------------------------------------
optimizer = torch.optim.AdamW(
    [
        {
            "params": model.vision.parameters(),
            "lr": VISION_LR,
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": model.projector.parameters(),
            "lr": PROJECTOR_LR,
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": model.conditioner.parameters(),
            "lr": STRUCTURED_LR,
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": model.structured_heads.parameters(),
            "lr": STRUCTURED_LR,
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": model.mt5.parameters(),
            "lr": MT5_LR,
            "weight_decay": WEIGHT_DECAY,
        },
    ]
)


# ------------------------------------------------
# 10. EXACT SCHEDULER
# ------------------------------------------------
train_batches = len(train_loader)

steps_per_epoch = (
    train_batches + GRAD_ACCUM - 1
) // GRAD_ACCUM

TOTAL_STEPS = steps_per_epoch * TOTAL_EPOCHS

assert steps_per_epoch == 384, (
    f"Expected 384 optimizer steps/epoch, got {steps_per_epoch}"
)

assert TOTAL_STEPS == 1920

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=TOTAL_STEPS,
    eta_min=0,
)


# ------------------------------------------------
# 11. LOAD EPOCH-2 CHECKPOINT
# ------------------------------------------------
assert os.path.exists(LAST_CKPT), \
    f"Missing checkpoint: {LAST_CKPT}"

checkpoint = torch.load(
    LAST_CKPT,
    map_location="cpu"
)

print("Loading Epoch-2 checkpoint...")

model.load_state_dict(
    checkpoint["model_state"],
    strict=True
)

optimizer.load_state_dict(
    checkpoint["optimizer_state"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state"]
)

history = checkpoint["history"]

start_epoch = checkpoint["epoch"] + 1

assert checkpoint["epoch"] == 2
assert start_epoch == 3

print("Resume epoch:", start_epoch)
print("Best val:", checkpoint["best_val"])
print("Scheduler last_epoch:", scheduler.last_epoch)


# ------------------------------------------------
# 12. RESTORE RNG
# ------------------------------------------------
if "rng_state" in checkpoint:
    torch.set_rng_state(
        checkpoint["rng_state"]
    )

if torch.cuda.is_available() and "cuda_rng_state" in checkpoint:
    torch.cuda.set_rng_state_all(
        checkpoint["cuda_rng_state"]
    )


# ------------------------------------------------
# 13. BF16
# ------------------------------------------------
use_bf16 = (
    torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
)

print("BF16:", use_bf16)


# ------------------------------------------------
# 14. TRAIN EPOCH 3
# ------------------------------------------------
model.train()

train_total = 0.0
train_report = 0.0
train_concept = 0.0
train_negative = 0.0
train_attribute = 0.0

optimizer.zero_grad(set_to_none=True)

num_batches = len(train_loader)

for batch_idx, batch in enumerate(train_loader):

    pixel_values = batch["pixel_values"].to(
        DEVICE,
        non_blocking=True
    )

    image_mask = batch["image_mask"].to(
        DEVICE,
        non_blocking=True
    )

    labels = batch["labels"].to(
        DEVICE,
        non_blocking=True
    )

    concept_targets = batch[
        "concept_targets"
    ].to(
        DEVICE,
        non_blocking=True
    )

    negative_targets = batch[
        "negative_targets"
    ].to(
        DEVICE,
        non_blocking=True
    )

    attribute_targets = batch[
        "attribute_targets"
    ].to(
        DEVICE,
        non_blocking=True
    )

    amp_ctx = (
        torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16
        )
        if use_bf16
        else nullcontext()
    )

    with amp_ctx:

        outputs = model(
            pixel_values=pixel_values,
            image_mask=image_mask,
            labels=labels,
            concept_targets=concept_targets,
            negative_targets=negative_targets,
            attribute_targets=attribute_targets,
        )

        loss = (
            outputs["loss"]
            / GRAD_ACCUM
        )

    loss.backward()

    train_total += outputs["loss"].detach().item()
    train_report += outputs[
        "report_loss"
    ].detach().item()

    train_concept += outputs[
        "concept_loss"
    ].detach().item()

    train_negative += outputs[
        "negative_loss"
    ].detach().item()

    train_attribute += outputs[
        "attribute_loss"
    ].detach().item()

    if (
        (batch_idx + 1) % GRAD_ACCUM == 0
        or batch_idx + 1 == num_batches
    ):

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()
        scheduler.step()

        optimizer.zero_grad(
            set_to_none=True
        )

    if (
        (batch_idx + 1) % 100 == 0
        or batch_idx == 0
        or batch_idx + 1 == num_batches
    ):
        print(
            f"train "
            f"{batch_idx+1}/{num_batches} | "
            f"loss={outputs['loss'].item():.4f} | "
            f"report={outputs['report_loss'].item():.4f}"
        )


n_train = len(train_loader)

train_metrics = {
    "total": train_total / n_train,
    "report": train_report / n_train,
    "concept": train_concept / n_train,
    "negative": train_negative / n_train,
    "attribute": train_attribute / n_train,
}


# ------------------------------------------------
# 15. VALIDATION
# ------------------------------------------------
model.eval()

val_total = 0.0
val_report = 0.0
val_concept = 0.0
val_negative = 0.0
val_attribute = 0.0

with torch.no_grad():

    for batch in val_loader:

        pixel_values = batch["pixel_values"].to(
            DEVICE,
            non_blocking=True
        )

        image_mask = batch["image_mask"].to(
            DEVICE,
            non_blocking=True
        )

        labels = batch["labels"].to(
            DEVICE,
            non_blocking=True
        )

        concept_targets = batch[
            "concept_targets"
        ].to(
            DEVICE,
            non_blocking=True
        )

        negative_targets = batch[
            "negative_targets"
        ].to(
            DEVICE,
            non_blocking=True
        )

        attribute_targets = batch[
            "attribute_targets"
        ].to(
            DEVICE,
            non_blocking=True
        )

        amp_ctx = (
            torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16
            )
            if use_bf16
            else nullcontext()
        )

        with amp_ctx:

            outputs = model(
                pixel_values=pixel_values,
                image_mask=image_mask,
                labels=labels,
                concept_targets=concept_targets,
                negative_targets=negative_targets,
                attribute_targets=attribute_targets,
            )

        val_total += outputs[
            "loss"
        ].item()

        val_report += outputs[
            "report_loss"
        ].item()

        val_concept += outputs[
            "concept_loss"
        ].item()

        val_negative += outputs[
            "negative_loss"
        ].item()

        val_attribute += outputs[
            "attribute_loss"
        ].item()


n_val = len(val_loader)

val_metrics = {
    "total": val_total / n_val,
    "report": val_report / n_val,
    "concept": val_concept / n_val,
    "negative": val_negative / n_val,
    "attribute": val_attribute / n_val,
}


# ------------------------------------------------
# 16. CHECKPOINT SAFETY
# ------------------------------------------------
epoch = 3

best_val = checkpoint["best_val"]

improved = (
    val_metrics["total"] < best_val
)

record = {
    "epoch": epoch,
    "train_total": train_metrics["total"],
    "train_report": train_metrics["report"],
    "train_concept": train_metrics["concept"],
    "train_negative": train_metrics["negative"],
    "train_attribute": train_metrics["attribute"],
    "val_total": val_metrics["total"],
    "val_report": val_metrics["report"],
    "val_concept": val_metrics["concept"],
    "val_negative": val_metrics["negative"],
    "val_attribute": val_metrics["attribute"],
    "lr_vision": optimizer.param_groups[0]["lr"],
    "lr_projector": optimizer.param_groups[1]["lr"],
    "lr_conditioner": optimizer.param_groups[2]["lr"],
    "lr_structured": optimizer.param_groups[3]["lr"],
    "lr_mt5": optimizer.param_groups[4]["lr"],
}

history.append(record)


# ------------------------------------------------
# 17. SAVE EPOCH 3 FIRST
# ------------------------------------------------
save_state = {
    "epoch": epoch,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "scheduler_state": scheduler.state_dict(),
    "best_val": (
        val_metrics["total"]
        if improved
        else best_val
    ),
    "rng_state": torch.get_rng_state(),
    "cuda_rng_state": (
        torch.cuda.get_rng_state_all()
        if torch.cuda.is_available()
        else None
    ),
    "history": history,
}

epoch3_path = f"{CKPT_DIR}/epoch_03.pt"

torch.save(
    save_state,
    epoch3_path
)

print("\nSaved:", epoch3_path)


# ------------------------------------------------
# 18. SAVE BEST ONLY IF IMPROVED
# ------------------------------------------------
if improved:

    torch.save(
        save_state,
        BEST_CKPT
    )

    print(
        "✓ New BEST checkpoint:",
        BEST_CKPT
    )

else:

    print(
        "Best checkpoint unchanged."
    )


# ------------------------------------------------
# 19. SAVE HISTORY
# ------------------------------------------------
pd.DataFrame(history).to_csv(

    HISTORY_CSV,
    index=False
)

print("History saved:", HISTORY_CSV)


# ------------------------------------------------
# 20. ONLY NOW UPDATE last.pt
# ------------------------------------------------
torch.save(
    save_state,
    LAST_CKPT
)

print("✓ last.pt updated after validation.")


# ------------------------------------------------
# 21. FINAL REPORT
# ------------------------------------------------
print("\n" + "=" * 70)
print("EPOCH 3 COMPLETE")
print("=" * 70)

print(
    f"Train total    : {train_metrics['total']:.6f}"
)
print(
    f"Train report   : {train_metrics['report']:.6f}"
)
print(
    f"Train concept  : {train_metrics['concept']:.6f}"
)
print(
    f"Train negative : {train_metrics['negative']:.6f}"
)
print(
    f"Train attribute: {train_metrics['attribute']:.6f}"
)

print()

print(
    f"Val total      : {val_metrics['total']:.6f}"
)
print(
    f"Val report     : {val_metrics['report']:.6f}"
)
print(
    f"Val concept    : {val_metrics['concept']:.6f}"
)
print(
    f"Val negative   : {val_metrics['negative']:.6f}"
)
print(
    f"Val attribute  : {val_metrics['attribute']:.6f}"
)

print()

print(
    f"Previous best  : {best_val:.6f}"
)
print(
    f"Improved       : {improved}"
)
print(
    f"Scheduler step : {scheduler.last_epoch}"
)

print("=" * 70)
print("NO TEST SET USED.")
print("EPOCH 3 CHECKPOINT SAVED ONLY AFTER VALIDATION.")
print("=" * 70)

ModuleNotFoundError: No module named 'transformers.models.mt5.tokenization_mt5'

In [15]:
# ================================================================
# V4 — TRANSFORMERS IMPORT COMPATIBILITY CHECK
# ================================================================

import transformers
import torch

print("transformers version:", transformers.__version__)
print("torch version:", torch.__version__)

from transformers import ViTModel, MT5ForConditionalGeneration
from transformers.models.mt5.tokenization_mt5 import MT5Tokenizer

print("✓ ViTModel import PASS")
print("✓ MT5ForConditionalGeneration import PASS")
print("✓ MT5Tokenizer import PASS")

tokenizer = MT5Tokenizer.from_pretrained("google/mt5-small")

print("✓ MT5 tokenizer load PASS")
print("Tokenizer vocab size:", tokenizer.vocab_size)

print("=" * 70)
print("IMPORT COMPATIBILITY AUDIT: PASS")
print("NO MODEL CREATED")
print("NO TRAINING")
print("NO CHECKPOINT MODIFIED")
print("=" * 70)

transformers version: 5.16.1
torch version: 2.11.0+cu128


ModuleNotFoundError: No module named 'transformers.models.mt5.tokenization_mt5'

In [16]:
# ================================================================
# TRANSFORMERS 5.x — MT5 TOKENIZER API AUDIT
# NO TRAINING / NO CHECKPOINT MODIFICATION
# ================================================================

import transformers
import importlib.util

print("transformers:", transformers.__version__)

print("\n[1] Check tokenizer backends")
print("sentencepiece installed:",
      importlib.util.find_spec("sentencepiece") is not None)
print("tokenizers installed:",
      importlib.util.find_spec("tokenizers") is not None)

print("\n[2] Check available MT5-related symbols")

candidates = [
    "MT5Tokenizer",
    "MT5TokenizerFast",
    "AutoTokenizer",
]

for name in candidates:
    print(
        f"{name:20s}:",
        hasattr(transformers, name)
    )

print("\n[3] Try AutoTokenizer")

try:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(
        "google/mt5-small"
    )

    print("✓ AutoTokenizer import PASS")
    print("✓ MT5 tokenizer load PASS")
    print("Tokenizer class:", type(tokenizer).__name__)
    print("Vocab size:", tokenizer.vocab_size)

except Exception as e:
    print("✗ AutoTokenizer FAILED")
    print(type(e).__name__)
    print(str(e))

print("\n" + "=" * 70)
print("AUDIT COMPLETE")
print("NO MODEL CREATED")
print("NO TRAINING")
print("NO CHECKPOINT MODIFIED")
print("=" * 70)

transformers: 5.16.1

[1] Check tokenizer backends
sentencepiece installed: True
tokenizers installed: True

[2] Check available MT5-related symbols
MT5Tokenizer        : False
MT5TokenizerFast    : False
AutoTokenizer       : True

[3] Try AutoTokenizer
✓ AutoTokenizer import PASS
✓ MT5 tokenizer load PASS
Tokenizer class: T5Tokenizer
Vocab size: 250100

AUDIT COMPLETE
NO MODEL CREATED
NO TRAINING
NO CHECKPOINT MODIFIED


In [26]:
# ================================================================
# V4 — PROJECTOR KEY FIX + EXACT STATE-DICT AUDIT
# NO TRAINING / NO SAVE
# ================================================================

class Projector(nn.Module):

    def __init__(
        self,
        in_dim=768,
        out_dim=512,
    ):
        super().__init__()

        self.proj = nn.Linear(
            in_dim,
            out_dim,
        )

    def forward(self, x):
        return self.proj(x)


# ------------------------------------------------
# Replace ONLY the projector
# ------------------------------------------------

model.projector = Projector(
    768,
    512,
).to(DEVICE)


# ------------------------------------------------
# Compare keys again
# ------------------------------------------------

model_keys = set(
    model.state_dict().keys()
)

ckpt_keys = set(
    checkpoint["model_state"].keys()
)

missing = sorted(
    ckpt_keys - model_keys
)

unexpected = sorted(
    model_keys - ckpt_keys
)

print("=" * 70)
print("AFTER PROJECTOR FIX")
print("=" * 70)

print(
    "Model keys:",
    len(model_keys)
)

print(
    "Checkpoint keys:",
    len(ckpt_keys)
)

print(
    "Missing:",
    missing
)

print(
    "Unexpected:",
    unexpected
)

assert len(model_keys) == 409
assert len(ckpt_keys) == 409

assert len(missing) == 0, missing
assert len(unexpected) == 0, unexpected

print(
    "\n✓ EXACT 409 / 409 KEY MATCH"
)


# ------------------------------------------------
# Check all tensor shapes
# ------------------------------------------------

shape_mismatch = []

model_state = model.state_dict()

for key in sorted(ckpt_keys):

    ckpt_shape = tuple(
        checkpoint["model_state"][key].shape
    )

    model_shape = tuple(
        model_state[key].shape
    )

    if ckpt_shape != model_shape:

        shape_mismatch.append(
            (
                key,
                ckpt_shape,
                model_shape,
            )
        )

print(
    "Shape mismatches:",
    len(shape_mismatch)
)

assert len(shape_mismatch) == 0, (
    shape_mismatch
)

print(
    "✓ ALL 409 SHAPES MATCH"
)


# ------------------------------------------------
# Strict load
# ------------------------------------------------

model.load_state_dict(
    checkpoint["model_state"],
    strict=True,
)

print(
    "✓ STRICT LOAD PASS"
)

print(
    "\nNO TRAINING"
)

print(
    "NO optimizer.step()"
)

print(
    "NO scheduler.step()"
)

print(
    "NO checkpoint save"
)

print("=" * 70)

In [19]:
# ================================================================
# V4 — EPOCH 3 TRAINING
# RESUME EXACTLY FROM EPOCH 2
#
# Transformers 5.x compatible
# AutoTokenizer -> google/mt5-small
#
# IMPORTANT:
# - No test set
# - No modification to Epoch-2 checkpoint
#   until Epoch-3 validation is complete
# ================================================================

import os
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from transformers import (
    ViTModel,
    MT5ForConditionalGeneration,
    AutoTokenizer,
)

from PIL import Image
from contextlib import nullcontext


# ================================================================
# 0. CONFIG
# ================================================================

BASE = "/content/drive/MyDrive/NoiSoi_Matching"

MANIFEST_PATH = (
    f"{BASE}/baseline_model_v4/"
    f"v4_training_manifest.csv"
)

STRUCTURED_PATH = (
    f"{BASE}/v3_ontology_v2/"
    f"v3_structured_targets_v2_2.csv"
)

CKPT_DIR = (
    f"{BASE}/baseline_model_v4/"
    f"checkpoints"
)

LAST_CKPT = f"{CKPT_DIR}/last.pt"
EPOCH2_CKPT = f"{CKPT_DIR}/epoch_02.pt"
BEST_CKPT = f"{CKPT_DIR}/best.pt"

HISTORY_CSV = (
    f"{BASE}/baseline_model_v4/"
    f"v4_training_history.csv"
)

os.makedirs(CKPT_DIR, exist_ok=True)


# Model
VISION_MODEL = "google/vit-base-patch16-224"
TEXT_MODEL = "google/mt5-small"

# Training
BATCH_SIZE = 4
GRAD_ACCUM = 4

MAX_IMAGES = 8
MAX_TARGET_LENGTH = 96

NUM_CONCEPTS = 48
NUM_NEGATIVE = 5
NUM_ATTRIBUTES = 6

VISION_LR = 1e-5
PROJECTOR_LR = 1e-4
STRUCTURED_LR = 1e-4
MT5_LR = 5e-5

WEIGHT_DECAY = 0.01
TOTAL_EPOCHS = 5

SEED = 42

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ================================================================
# 1. ENVIRONMENT
# ================================================================

import transformers

print("=" * 70)
print("V4 EPOCH 3 TRAINING")
print("=" * 70)

print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "BF16 supported:",
        torch.cuda.is_bf16_supported()
    )

print()


# ================================================================
# 2. SEED
# ================================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


# ================================================================
# 3. LOAD MANIFEST
# ================================================================

# ================================================================
# 1–7. LOAD V4 CASE MANIFEST + IMAGE MANIFEST CORRECTLY
# ================================================================

print("[1] Loading data...")

# ------------------------------------------------
# V4 CASE MANIFEST
# ------------------------------------------------
V4_MANIFEST_PATH = (
    f"{BASE}/baseline_model_v4/"
    f"v4_training_manifest.csv"
)

# ------------------------------------------------
# IMAGE MANIFEST
# ------------------------------------------------
IMAGE_MANIFEST_PATH = (
    f"{BASE}/final_manifest/"
    f"final_dataset_manifest.csv"
)

# ------------------------------------------------
# STRUCTURED TARGETS
# ------------------------------------------------
STRUCTURED_PATH = (
    f"{BASE}/v3_ontology_v2/"
    f"v3_structured_targets_v2_2.csv"
)

# Load the THREE files separately
v4_manifest = pd.read_csv(
    V4_MANIFEST_PATH
)

image_manifest = pd.read_csv(
    IMAGE_MANIFEST_PATH
)

structured_df = pd.read_csv(
    STRUCTURED_PATH,
    index_col=0
)

# Consistent case_id type
v4_manifest["case_id"] = (
    v4_manifest["case_id"]
    .astype(str)
)

image_manifest["case_id"] = (
    image_manifest["case_id"]
    .astype(str)
)

structured_df.index = (
    structured_df.index
    .astype(str)
)

print(
    "V4 manifest      :",
    v4_manifest.shape
)

print(
    "Image manifest   :",
    image_manifest.shape
)

print(
    "Structured       :",
    structured_df.shape
)


# ================================================================
# 2. VERIFY V4 CASE MANIFEST
# ================================================================

case_df = v4_manifest.copy()

assert case_df.shape == (7606, 15)

assert (
    case_df["case_id"].nunique()
    == 7606
)

assert (
    set(case_df["case_id"])
    == set(structured_df.index)
), (
    "V4 cases and structured targets "
    "do not exactly align."
)

print(
    "✓ V4 cases:",
    len(case_df)
)


# ================================================================
# 3. VERIFY SPLITS
# ================================================================

split_counts = (
    case_df["split"]
    .value_counts()
    .to_dict()
)

print(
    "Splits:",
    split_counts
)

assert split_counts["train"] == 6138
assert split_counts["val"] == 756
assert split_counts["test"] == 712

print(
    "✓ Fixed patient-level split verified"
)


# ================================================================
# 4. VERIFY IMAGE MANIFEST SCHEMA
# ================================================================

required_image_columns = [
    "case_id",
    "image_order",
    "image_path",
    "image_status",
]

missing_columns = [
    c
    for c in required_image_columns
    if c not in image_manifest.columns
]

assert not missing_columns, (
    f"Image manifest missing columns: "
    f"{missing_columns}"
)

print(
    "✓ Image manifest schema verified"
)


# ================================================================
# 5. BUILD NORMAL IMAGE GROUPS
# ================================================================

print(
    "\n[2] Building NORMAL image groups..."
)

v4_case_ids = set(
    case_df["case_id"]
)

normal_manifest = image_manifest[
    image_manifest["case_id"].isin(
        v4_case_ids
    )
].copy()

normal_manifest = normal_manifest[
    normal_manifest["image_status"]
    .astype(str)
    .str.upper()
    == "NORMAL"
].copy()

print(
    "NORMAL images:",
    len(normal_manifest)
)


# ------------------------------------------------
# Group by case
# ------------------------------------------------

image_groups = {
    case_id: g.sort_values(
        "image_order"
    )
    for case_id, g
    in normal_manifest.groupby(
        "case_id"
    )
}

print(
    "NORMAL image groups:",
    len(image_groups)
)


# ================================================================
# 6. VERIFY EVERY V4 CASE HAS NORMAL IMAGE
# ================================================================

missing_image_cases = [
    case_id
    for case_id in v4_case_ids
    if case_id not in image_groups
    or len(image_groups[case_id]) == 0
]

assert len(
    missing_image_cases
) == 0, (
    "V4 cases without NORMAL images: "
    f"{missing_image_cases[:20]}"
)

print(
    "✓ Every 7606 V4 case has "
    "at least 1 NORMAL image"
)


# ================================================================
# 7. VERIFY IMAGE COUNT RANGE
# ================================================================

image_counts = [
    len(image_groups[c])
    for c in case_df["case_id"]
]

print(
    "Images/case:"
)

print(
    "  min    =",
    min(image_counts)
)

print(
    "  median =",
    int(np.median(image_counts))
)

print(
    "  mean   =",
    round(float(np.mean(image_counts)), 4)
)

print(
    "  max    =",
    max(image_counts)
)

assert min(image_counts) >= 1

# The raw image manifest can contain >8 NORMAL images/case.
# V4 training intentionally truncates each case to MAX_IMAGES=8.
print(
    "Raw images/case max:",
    max(image_counts)
)

print(
    f"V4 training images/case will be capped at {MAX_IMAGES}"
)

# Verify that every case can provide at least one image.
# Cases with >MAX_IMAGES are intentionally truncated by Dataset.
assert all(
    n >= 1
    for n in image_counts
)

print(
    "✓ Raw image groups verified"
)
print(
    "✓ MAX_IMAGES=8 will be enforced during Dataset loading"
)


# ================================================================
# 8. STRUCTURED ONTOLOGY
# ================================================================

concept_cols = [
    c
    for c in structured_df.columns
    if c.startswith("concept__")
]

negative_cols = [
    c
    for c in structured_df.columns
    if c.startswith("negative__")
]

attribute_cols = [
    c
    for c in structured_df.columns
    if c.startswith("attribute__")
]

assert len(concept_cols) == 48
assert len(negative_cols) == 5
assert len(attribute_cols) == 6

print(
    "✓ Structured ontology:",
    len(concept_cols),
    "concepts /",
    len(negative_cols),
    "negatives /",
    len(attribute_cols),
    "attributes"
)


# ================================================================
# 9. REPORT TARGET
# ================================================================

assert "ket_luan" in case_df.columns

assert (
    case_df["ket_luan"]
    .notna()
    .all()
)

assert (
    case_df["ket_luan"]
    .astype(str)
    .str.strip()
    .ne("")
    .all()
)

print(
    "✓ ket_luan report target verified"
)


# ================================================================
# FINAL DATA AUDIT
# ================================================================

print("\n" + "=" * 70)
print("DATA LOADING AUDIT PASS")
print("=" * 70)

print(
    "V4 cases        :",
    len(case_df)
)

print(
    "Train           :",
    (case_df["split"] == "train").sum()
)

print(
    "Val             :",
    (case_df["split"] == "val").sum()
)

print(
    "Test            :",
    (case_df["split"] == "test").sum()
)

print(
    "Image groups    :",
    len(image_groups)
)

print(
    "NORMAL images   :",
    len(normal_manifest)
)

print(
    "Structured      : 48 / 5 / 6"
)

print(
    "Report target   : ket_luan"
)

print("=" * 70)
print("✓ SAFE TO PROCEED TO DATASET")
print("=" * 70)


# ================================================================
# 8. DATASET
# ================================================================

class V4Dataset(Dataset):

    def __init__(
        self,
        cases,
        split
    ):
        self.df = (
            cases[
                cases["split"] == split
            ]
            .copy()
            .reset_index(drop=True)
        )

        # frozen ontology column order
        self.concept_cols = [
            c
            for c in structured_df.columns
            if c.startswith("concept__")
        ]

        self.negative_cols = [
            c
            for c in structured_df.columns
            if c.startswith("negative__")
        ]

        self.attribute_cols = [
            c
            for c in structured_df.columns
            if c.startswith("attribute__")
        ]

        assert len(
            self.concept_cols
        ) == NUM_CONCEPTS

        assert len(
            self.negative_cols
        ) == NUM_NEGATIVE

        assert len(
            self.attribute_cols
        ) == NUM_ATTRIBUTES

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        case_id = str(
            row["case_id"]
        )

        imgs = image_groups[case_id]

        # EXACTLY MAX_IMAGES
        imgs = imgs.iloc[
            :MAX_IMAGES
        ]

        pixel_values = []

        for _, img_row in imgs.iterrows():

            path = img_row["image_path"]

            img = Image.open(
                path
            ).convert("RGB")

            # ViT 224x224
            img = img.resize(
                (224, 224),
                Image.Resampling.BILINEAR
            )

            arr = (
                np.asarray(img)
                .astype(np.float32)
                / 255.0
            )

            arr = torch.from_numpy(
                arr
            ).permute(2, 0, 1)

            # Existing V4 normalization
            mean = torch.tensor(
                [0.5, 0.5, 0.5],
                dtype=torch.float32
            ).view(3, 1, 1)

            std = torch.tensor(
                [0.5, 0.5, 0.5],
                dtype=torch.float32
            ).view(3, 1, 1)

            arr = (
                arr - mean
            ) / std

            pixel_values.append(
                arr
            )

        n_images = len(
            pixel_values
        )

        assert n_images >= 1
        assert n_images <= MAX_IMAGES

        # Padding
        while len(pixel_values) < MAX_IMAGES:

            pixel_values.append(
                torch.zeros_like(
                    pixel_values[0]
                )
            )

        pixel_values = torch.stack(
            pixel_values
        )

        image_mask = torch.zeros(
            MAX_IMAGES,
            dtype=torch.float32
        )

        image_mask[
            :n_images
        ] = 1.0

        # ------------------------------------------------
        # REPORT TARGET
        # ------------------------------------------------

        target = str(
            row["ket_luan"]
        )

        encoded = tokenizer(
            target,
            max_length=MAX_TARGET_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = encoded[
            "input_ids"
        ].squeeze(0)

        labels[
            labels == tokenizer.pad_token_id
        ] = -100

        # ------------------------------------------------
        # STRUCTURED TARGETS
        # ------------------------------------------------

        s = structured_df.loc[
            case_id
        ]

        concepts = torch.tensor(
            s[
                self.concept_cols
            ].astype(float).values,
            dtype=torch.float32
        )

        negatives = torch.tensor(
            s[
                self.negative_cols
            ].astype(float).values,
            dtype=torch.float32
        )

        attributes = torch.tensor(
            s[
                self.attribute_cols
            ].astype(float).values,
            dtype=torch.float32
        )

        return {
            "case_id": case_id,

            "pixel_values":
                pixel_values,

            "image_mask":
                image_mask,

            "labels":
                labels,

            "concept_targets":
                concepts,

            "negative_targets":
                negatives,

            "attribute_targets":
                attributes,
        }


# ================================================================
# 9. DATALOADERS
# ================================================================

print("\n[4] Creating dataloaders...")

train_ds = V4Dataset(
    case_df,
    "train"
)

val_ds = V4Dataset(
    case_df,
    "val"
)

assert len(train_ds) == 6138
assert len(val_ds) == 756

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=False,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False,
)

print(
    "Train:",
    len(train_ds)
)

print(
    "Val:",
    len(val_ds)
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Val batches:",
    len(val_loader)
)


# ================================================================
# 10. EXACT STRUCTURED CONDITIONER
# ================================================================

class StructuredConditioner(
    nn.Module
):

    def __init__(
        self,
        d_model,
        num_concepts=48,
        num_negative=5,
        num_attributes=6
    ):

        super().__init__()

        self.num_concepts = (
            num_concepts
        )

        self.num_negative = (
            num_negative
        )

        self.num_attributes = (
            num_attributes
        )

        self.concept_value = (
            nn.Embedding(
                2,
                d_model
            )
        )

        self.negative_value = (
            nn.Embedding(
                2,
                d_model
            )
        )

        self.attribute_value = (
            nn.Embedding(
                2,
                d_model
            )
        )

        self.concept_label = (
            nn.Embedding(
                num_concepts,
                d_model
            )
        )

        self.negative_label = (
            nn.Embedding(
                num_negative,
                d_model
            )
        )

        self.attribute_label = (
            nn.Embedding(
                num_attributes,
                d_model
            )
        )

        self.type_embedding = (
            nn.Embedding(
                3,
                d_model
            )
        )

        self.norm = nn.LayerNorm(
            d_model
        )

    def forward(
        self,
        concept_targets,
        negative_targets,
        attribute_targets
    ):

        device = (
            concept_targets.device
        )

        concept_ids = (
            concept_targets
            .long()
            .clamp(0, 1)
        )

        negative_ids = (
            negative_targets
            .long()
            .clamp(0, 1)
        )

        attribute_ids = (
            attribute_targets
            .long()
            .clamp(0, 1)
        )

        # -------------------------
        # concepts
        # -------------------------

        concept_idx = torch.arange(
            self.num_concepts,
            device=device
        )

        concept_type = (
            self.type_embedding(
                torch.zeros(
                    self.num_concepts,
                    dtype=torch.long,
                    device=device
                )
            )
        )

        concept_tokens = (
            self.concept_value(
                concept_ids
            )
            + self.concept_label(
                concept_idx
            )[None, :, :]
            + concept_type[
                None, :, :
            ]
        )

        # -------------------------
        # negatives
        # -------------------------

        negative_idx = torch.arange(
            self.num_negative,
            device=device
        )

        negative_type = (
            self.type_embedding(
                torch.ones(
                    self.num_negative,
                    dtype=torch.long,
                    device=device
                )
            )
        )

        negative_tokens = (
            self.negative_value(
                negative_ids
            )
            + self.negative_label(
                negative_idx
            )[None, :, :]
            + negative_type[
                None, :, :
            ]
        )

        # -------------------------
        # attributes
        # -------------------------

        attribute_idx = torch.arange(
            self.num_attributes,
            device=device
        )

        attribute_type = (
            self.type_embedding(
                torch.full(
                    (
                        self.num_attributes,
                    ),
                    2,
                    dtype=torch.long,
                    device=device
                )
            )
        )

        attribute_tokens = (
            self.attribute_value(
                attribute_ids
            )
            + self.attribute_label(
                attribute_idx
            )[None, :, :]
            + attribute_type[
                None, :, :
            ]
        )

        # -------------------------
        # concatenate
        # -------------------------

        structured_tokens = torch.cat(
            [
                concept_tokens,
                negative_tokens,
                attribute_tokens,
            ],
            dim=1
        )

        structured_tokens = (
            self.norm(
                structured_tokens
            )
        )

        B = (
            concept_targets.shape[0]
        )

        structured_attention = (
            torch.ones(
                B,
                structured_tokens.shape[1],
                dtype=torch.long,
                device=device
            )
        )

        return (
            structured_tokens,
            structured_attention
        )


# ================================================================
# 11. EXACT V4 MODEL
# ================================================================

class V4Model(nn.Module):

    def __init__(self):

        super().__init__()

        self.vision = (
            ViTModel.from_pretrained(
                VISION_MODEL
            )
        )

        self.mt5 = (
            MT5ForConditionalGeneration
            .from_pretrained(
                TEXT_MODEL
            )
        )

        self.projector = nn.Linear(
            768,
            512
        )

        self.conditioner = (
            StructuredConditioner(
                512,
                NUM_CONCEPTS,
                NUM_NEGATIVE,
                NUM_ATTRIBUTES
            )
        )

        # IMPORTANT:
        # exact checkpoint naming
        self.structured_heads = (
            nn.ModuleDict({
                "concept":
                    nn.Linear(
                        512,
                        48
                    ),

                "negative":
                    nn.Linear(
                        512,
                        5
                    ),

                "attribute":
                    nn.Linear(
                        512,
                        6
                    ),
            })
        )

    def forward(
        self,
        pixel_values,
        image_mask,
        labels,
        concept_targets,
        negative_targets,
        attribute_targets
    ):

        B, N, C, H, W = (
            pixel_values.shape
        )

        # ------------------------------------------------
        # ViT
        # ------------------------------------------------

        x = pixel_values.reshape(
            B * N,
            C,
            H,
            W
        )

        vision_out = self.vision(
            pixel_values=x
        ).last_hidden_state[:, 0]

        vision_out = vision_out.reshape(
            B,
            N,
            768
        )

        # ------------------------------------------------
        # Visual projection
        # ------------------------------------------------

        visual_tokens = (
            self.projector(
                vision_out
            )
        )

        # Remove padded images
        visual_tokens = (
            visual_tokens
            * image_mask[:, :, None]
        )

        # ------------------------------------------------
        # Pooled representation
        # ------------------------------------------------

        denom = (
            image_mask.sum(
                dim=1,
                keepdim=True
            )
            .clamp(min=1)
        )

        pooled = (
            visual_tokens.sum(
                dim=1
            )
            / denom
        )

        # ------------------------------------------------
        # Structured prediction heads
        # ------------------------------------------------

        concept_logits = (
            self.structured_heads[
                "concept"
            ](pooled)
        )

        negative_logits = (
            self.structured_heads[
                "negative"
            ](pooled)
        )

        attribute_logits = (
            self.structured_heads[
                "attribute"
            ](pooled)
        )

        # ------------------------------------------------
        # HARD BINARY STRUCTURED PREDICTIONS
        #
        # Preserve audited V4 behavior.
        # ------------------------------------------------

        concept_pred = (
            torch.sigmoid(
                concept_logits
            ) >= 0.5
        ).float()

        negative_pred = (
            torch.sigmoid(
                negative_logits
            ) >= 0.5
        ).float()

        attribute_pred = (
            torch.sigmoid(
                attribute_logits
            ) >= 0.5
        ).float()

        # ------------------------------------------------
        # Structured conditioner
        # ------------------------------------------------

        structured_tokens, structured_attention = (
            self.conditioner(
                concept_pred,
                negative_pred,
                attribute_pred
            )
        )

        # ------------------------------------------------
        # Prefix
        # ------------------------------------------------

        visual_attention = (
            image_mask.long()
        )

        prefix = torch.cat(
            [
                visual_tokens,
                structured_tokens,
            ],
            dim=1
        )

        attention_mask = torch.cat(
            [
                visual_attention,
                structured_attention,
            ],
            dim=1
        )

        # ------------------------------------------------
        # mT5 encoder
        # ------------------------------------------------

        # Prefix embeddings are directly supplied
        # to the mT5 encoder.
        #
        # The zero token IDs only provide a correctly
        # shaped embedding tensor to which the visual /
        # structured prefix is added.

        zero_ids = torch.zeros(
            B,
            prefix.shape[1],
            dtype=torch.long,
            device=prefix.device
        )

        base_embeddings = (
            self.mt5.encoder.embed_tokens(
                zero_ids
            )
        )

        inputs_embeds = (
            base_embeddings
            + prefix
        )

        encoder_outputs = (
            self.mt5.encoder(
                inputs_embeds=inputs_embeds,
                attention_mask=attention_mask,
                return_dict=True
            )
        )

        # ------------------------------------------------
        # Report generation
        # ------------------------------------------------

        output = self.mt5(
            encoder_outputs=encoder_outputs,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=True
        )

        report_loss = (
            output.loss
        )

        # ------------------------------------------------
        # Auxiliary structured losses
        # ------------------------------------------------

        concept_loss = (
            F.binary_cross_entropy_with_logits(
                concept_logits,
                concept_targets
            )
        )

        negative_loss = (
            F.binary_cross_entropy_with_logits(
                negative_logits,
                negative_targets
            )
        )

        attribute_loss = (
            F.binary_cross_entropy_with_logits(
                attribute_logits,
                attribute_targets
            )
        )

        # ------------------------------------------------
        # TOTAL V4 LOSS
        # ------------------------------------------------

        total_loss = (
            report_loss
            + 0.5 * concept_loss
            + 0.5 * negative_loss
            + 0.25 * attribute_loss
        )

        return {
            "loss":
                total_loss,

            "report_loss":
                report_loss,

            "concept_loss":
                concept_loss,

            "negative_loss":
                negative_loss,

            "attribute_loss":
                attribute_loss,

            "concept_logits":
                concept_logits,

            "negative_logits":
                negative_logits,

            "attribute_logits":
                attribute_logits,

            "visual_tokens":
                visual_tokens,

            "structured_tokens":
                structured_tokens,

            "prefix":
                prefix,
        }


# ================================================================
# 12. CREATE MODEL
# ================================================================

print("\n[5] Creating V4 model...")

model = V4Model().to(
    DEVICE
)

print("✓ Model created")


# ================================================================
# 13. EXACT 5-GROUP OPTIMIZER
# ================================================================

optimizer = torch.optim.AdamW(
    [
        {
            "params":
                model.vision.parameters(),
            "lr":
                VISION_LR,
            "weight_decay":
                WEIGHT_DECAY,
        },

        {
            "params":
                model.projector.parameters(),
            "lr":
                PROJECTOR_LR,
            "weight_decay":
                WEIGHT_DECAY,
        },

        {
            "params":
                model.conditioner.parameters(),
            "lr":
                STRUCTURED_LR,
            "weight_decay":
                WEIGHT_DECAY,
        },

        {
            "params":
                model.structured_heads.parameters(),
            "lr":
                STRUCTURED_LR,
            "weight_decay":
                WEIGHT_DECAY,
        },

        {
            "params":
                model.mt5.parameters(),
            "lr":
                MT5_LR,
            "weight_decay":
                WEIGHT_DECAY,
        },
    ]
)

assert len(
    optimizer.param_groups
) == 5

print(
    "✓ Optimizer groups:",
    len(optimizer.param_groups)
)


# ================================================================
# 14. EXACT COSINE SCHEDULER
# ================================================================

train_batches = len(
    train_loader
)

steps_per_epoch = (
    train_batches
    + GRAD_ACCUM
    - 1
) // GRAD_ACCUM

TOTAL_STEPS = (
    steps_per_epoch
    * TOTAL_EPOCHS
)

assert steps_per_epoch == 384, (
    f"Expected 384 steps/epoch, "
    f"got {steps_per_epoch}"
)

assert TOTAL_STEPS == 1920

scheduler = (
    torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=TOTAL_STEPS,
        eta_min=0
    )
)

print(
    "✓ Scheduler T_max:",
    TOTAL_STEPS
)


# ================================================================
# 15. LOAD EPOCH-2 CHECKPOINT
# ================================================================

print("\n[6] Loading Epoch-2 checkpoint...")

assert os.path.exists(
    LAST_CKPT
), f"Missing: {LAST_CKPT}"

checkpoint = torch.load(
    LAST_CKPT,
    map_location="cpu"
)

assert checkpoint["epoch"] == 2

# ------------------------------------------------
# Exact model state
# ------------------------------------------------

model.load_state_dict(
    checkpoint["model_state"],
    strict=True
)

print(
    "✓ Model strict load PASS"
)

# ------------------------------------------------
# Optimizer
# ------------------------------------------------

optimizer.load_state_dict(
    checkpoint["optimizer_state"]
)

print(
    "✓ Optimizer state load PASS"
)

# ------------------------------------------------
# Scheduler
# ------------------------------------------------

scheduler.load_state_dict(
    checkpoint["scheduler_state"]
)

print(
    "✓ Scheduler state load PASS"
)

# ------------------------------------------------
# History
# ------------------------------------------------

history = checkpoint[
    "history"
]

best_val = checkpoint[
    "best_val"
]

start_epoch = (
    checkpoint["epoch"] + 1
)

assert start_epoch == 3

print(
    "Resume epoch:",
    start_epoch
)

print(
    "Previous best:",
    best_val
)

print(
    "Scheduler step:",
    scheduler.last_epoch
)


# ================================================================
# 16. RESTORE RNG
# ================================================================

if "rng_state" in checkpoint:

    torch.set_rng_state(
        checkpoint["rng_state"]
    )

if (
    torch.cuda.is_available()
    and checkpoint.get(
        "cuda_rng_state"
    ) is not None
):

    torch.cuda.set_rng_state_all(
        checkpoint[
            "cuda_rng_state"
        ]
    )

print(
    "✓ RNG state restored"
)


# ================================================================
# 17. VERIFY RESUME STATE BEFORE TRAINING
# ================================================================

assert (
    scheduler.last_epoch
    == 768
), (
    "Unexpected scheduler position"
)

expected_lrs = [
    6.54508497187e-06,
    6.54508497187e-05,
    6.54508497187e-05,
    6.54508497187e-05,
    3.27254248594e-05,
]

for i, expected in enumerate(
    expected_lrs
):

    actual = (
        optimizer.param_groups[i]["lr"]
    )

    assert abs(
        actual - expected
    ) < 1e-12, (
        f"LR mismatch group {i}: "
        f"{actual} vs {expected}"
    )

print(
    "✓ Epoch-2 optimizer/scheduler state verified"
)


# ================================================================
# 18. BF16
# ================================================================

USE_BF16 = (
    torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
)

print(
    "BF16:",
    USE_BF16
)


# ================================================================
# 19. TRAIN EPOCH 3
# ================================================================

print("\n" + "=" * 70)
print("STARTING EPOCH 3")
print("=" * 70)

model.train()

train_total = 0.0
train_report = 0.0
train_concept = 0.0
train_negative = 0.0
train_attribute = 0.0

optimizer.zero_grad(
    set_to_none=True
)

num_batches = len(
    train_loader
)

for batch_idx, batch in enumerate(
    train_loader
):

    pixel_values = (
        batch["pixel_values"]
        .to(
            DEVICE,
            non_blocking=True
        )
    )

    image_mask = (
        batch["image_mask"]
        .to(
            DEVICE,
            non_blocking=True
        )
    )

    labels = (
        batch["labels"]
        .to(
            DEVICE,
            non_blocking=True
        )
    )

    concept_targets = (
        batch["concept_targets"]
        .to(
            DEVICE,
            non_blocking=True
        )
    )

    negative_targets = (
        batch["negative_targets"]
        .to(
            DEVICE,
            non_blocking=True
        )
    )

    attribute_targets = (
        batch["attribute_targets"]
        .to(
            DEVICE,
            non_blocking=True
        )
    )

    amp_context = (
        torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16
        )
        if USE_BF16
        else nullcontext()
    )

    with amp_context:

        outputs = model(
            pixel_values=
                pixel_values,

            image_mask=
                image_mask,

            labels=
                labels,

            concept_targets=
                concept_targets,

            negative_targets=
                negative_targets,

            attribute_targets=
                attribute_targets,
        )

        loss = (
            outputs["loss"]
            / GRAD_ACCUM
        )

    loss.backward()

    # ------------------------------------------------
    # Accumulate metrics
    # ------------------------------------------------

    train_total += (
        outputs["loss"]
        .detach()
        .item()
    )

    train_report += (
        outputs["report_loss"]
        .detach()
        .item()
    )

    train_concept += (
        outputs["concept_loss"]
        .detach()
        .item()
    )

    train_negative += (
        outputs["negative_loss"]
        .detach()
        .item()
    )

    train_attribute += (
        outputs["attribute_loss"]
        .detach()
        .item()
    )

    # ------------------------------------------------
    # Optimizer step
    # ------------------------------------------------

    should_step = (
        (batch_idx + 1)
        % GRAD_ACCUM == 0
        or
        (batch_idx + 1)
        == num_batches
    )

    if should_step:

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        scheduler.step()

        optimizer.zero_grad(
            set_to_none=True
        )

    # ------------------------------------------------
    # Progress
    # ------------------------------------------------

    if (
        batch_idx == 0
        or
        (batch_idx + 1) % 100 == 0
        or
        (batch_idx + 1) == num_batches
    ):

        print(
            f"[TRAIN] "
            f"{batch_idx + 1:4d}/"
            f"{num_batches} | "
            f"loss="
            f"{outputs['loss'].item():.4f} | "
            f"report="
            f"{outputs['report_loss'].item():.4f} | "
            f"lr="
            f"{optimizer.param_groups[4]['lr']:.3e}"
        )


# ================================================================
# 20. TRAIN METRICS
# ================================================================

train_metrics = {

    "total":
        train_total / num_batches,

    "report":
        train_report / num_batches,

    "concept":
        train_concept / num_batches,

    "negative":
        train_negative / num_batches,

    "attribute":
        train_attribute / num_batches,
}


# ================================================================
# 21. VALIDATION
# ================================================================

print("\n" + "=" * 70)
print("VALIDATING EPOCH 3")
print("=" * 70)

model.eval()

val_total = 0.0
val_report = 0.0
val_concept = 0.0
val_negative = 0.0
val_attribute = 0.0

num_val_batches = len(
    val_loader
)

with torch.no_grad():

    for batch_idx, batch in enumerate(
        val_loader
    ):

        pixel_values = (
            batch["pixel_values"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        image_mask = (
            batch["image_mask"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        labels = (
            batch["labels"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        concept_targets = (
            batch["concept_targets"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        negative_targets = (
            batch["negative_targets"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        attribute_targets = (
            batch["attribute_targets"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        amp_context = (
            torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16
            )
            if USE_BF16
            else nullcontext()
        )

        with amp_context:

            outputs = model(
                pixel_values=
                    pixel_values,

                image_mask=
                    image_mask,

                labels=
                    labels,

                concept_targets=
                    concept_targets,

                negative_targets=
                    negative_targets,

                attribute_targets=
                    attribute_targets,
            )

        val_total += (
            outputs["loss"]
            .item()
        )

        val_report += (
            outputs["report_loss"]
            .item()
        )

        val_concept += (
            outputs["concept_loss"]
            .item()
        )

        val_negative += (
            outputs["negative_loss"]
            .item()
        )

        val_attribute += (
            outputs["attribute_loss"]
            .item()
        )

        if (
            batch_idx == 0
            or
            (batch_idx + 1)
            % 100 == 0
            or
            (batch_idx + 1)
            == num_val_batches
        ):

            print(
                f"[VAL] "
                f"{batch_idx + 1:3d}/"
                f"{num_val_batches} | "
                f"loss="
                f"{outputs['loss'].item():.4f}"
            )


# ================================================================
# 22. VALIDATION METRICS
# ================================================================

val_metrics = {

    "total":
        val_total / num_val_batches,

    "report":
        val_report / num_val_batches,

    "concept":
        val_concept / num_val_batches,

    "negative":
        val_negative / num_val_batches,

    "attribute":
        val_attribute / num_val_batches,
}


# ================================================================
# 23. CHECKPOINT DECISION
# ================================================================

epoch = 3

improved = (
    val_metrics["total"]
    < best_val
)

new_best = (
    val_metrics["total"]
    if improved
    else best_val
)


# ================================================================
# 24. HISTORY RECORD
# ================================================================

record = {

    "epoch":
        epoch,

    "train_total":
        train_metrics["total"],

    "train_report":
        train_metrics["report"],

    "train_concept":
        train_metrics["concept"],

    "train_negative":
        train_metrics["negative"],

    "train_attribute":
        train_metrics["attribute"],

    "val_total":
        val_metrics["total"],

    "val_report":
        val_metrics["report"],

    "val_concept":
        val_metrics["concept"],

    "val_negative":
        val_metrics["negative"],

    "val_attribute":
        val_metrics["attribute"],

    "lr_vision":
        optimizer.param_groups[0]["lr"],

    "lr_projector":
        optimizer.param_groups[1]["lr"],

    "lr_conditioner":
        optimizer.param_groups[2]["lr"],

    "lr_structured":
        optimizer.param_groups[3]["lr"],

    "lr_mt5":
        optimizer.param_groups[4]["lr"],

    "scheduler_step":
        scheduler.last_epoch,
}


history.append(
    record
)


# ================================================================
# 25. BUILD CHECKPOINT
# ================================================================

save_state = {

    "epoch":
        epoch,

    "model_state":
        model.state_dict(),

    "optimizer_state":
        optimizer.state_dict(),

    "scheduler_state":
        scheduler.state_dict(),

    "best_val":
        new_best,

    "rng_state":
        torch.get_rng_state(),

    "cuda_rng_state":
        (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),

    "history":
        history,
}


# ================================================================
# 26. SAVE EPOCH 3
# ================================================================

epoch3_path = (
    f"{CKPT_DIR}/epoch_03.pt"
)

torch.save(
    save_state,
    epoch3_path
)

print(
    "\n✓ Saved Epoch-3 checkpoint:",
    epoch3_path
)


# ================================================================
# 27. SAVE BEST IF IMPROVED
# ================================================================

if improved:

    torch.save(
        save_state,
        BEST_CKPT
    )

    print(
        "✓ NEW BEST:",
        BEST_CKPT
    )

else:

    print(
        "Best checkpoint unchanged."
    )


# ================================================================
# 28. SAVE HISTORY
# ================================================================

pd.DataFrame(
    history
).to_csv(
    HISTORY_CSV,
    index=False
)

print(
    "✓ History saved:",
    HISTORY_CSV
)


# ================================================================
# 29. UPDATE last.pt
#
# ONLY AFTER:
#   - training finished
#   - validation finished
#   - epoch_03.pt written
# ================================================================

torch.save(
    save_state,
    LAST_CKPT
)

print(
    "✓ last.pt updated:",
    LAST_CKPT
)


# ================================================================
# 30. FINAL REPORT
# ================================================================

print("\n" + "=" * 70)
print("EPOCH 3 COMPLETE")
print("=" * 70)

print(
    f"Train total     : "
    f"{train_metrics['total']:.6f}"
)

print(
    f"Train report    : "
    f"{train_metrics['report']:.6f}"
)

print(
    f"Train concept   : "
    f"{train_metrics['concept']:.6f}"
)

print(
    f"Train negative  : "
    f"{train_metrics['negative']:.6f}"
)

print(
    f"Train attribute : "
    f"{train_metrics['attribute']:.6f}"
)

print()

print(
    f"Val total       : "
    f"{val_metrics['total']:.6f}"
)

print(
    f"Val report      : "
    f"{val_metrics['report']:.6f}"
)

print(
    f"Val concept     : "
    f"{val_metrics['concept']:.6f}"
)

print(
    f"Val negative    : "
    f"{val_metrics['negative']:.6f}"
)

print(
    f"Val attribute   : "
    f"{val_metrics['attribute']:.6f}"
)

print()

print(
    f"Previous best   : "
    f"{best_val:.6f}"
)

print(
    f"Epoch-3 val     : "
    f"{val_metrics['total']:.6f}"
)

print(
    f"Improved        : "
    f"{improved}"
)

print(
    f"Scheduler step  : "
    f"{scheduler.last_epoch}"
)

print()

print(
    "Epoch-3 checkpoint:",
    epoch3_path
)

print(
    "Best checkpoint:",
    BEST_CKPT
)

print(
    "Last checkpoint:",
    LAST_CKPT
)

print("=" * 70)
print("TEST SET WAS NOT USED.")
print("EPOCH 2 WAS PRESERVED UNTIL EPOCH 3 VALIDATION COMPLETED.")
print("=" * 70)

V4 EPOCH 3 TRAINING
Transformers: 5.16.1
Torch: 2.11.0+cu128
Device: cuda
GPU: NVIDIA A100-SXM4-40GB
BF16 supported: True

[1] Loading data...
V4 manifest      : (7606, 15)
Image manifest   : (76405, 26)
Structured       : (7606, 101)
✓ V4 cases: 7606
Splits: {'train': 6138, 'val': 756, 'test': 712}
✓ Fixed patient-level split verified
✓ Image manifest schema verified

[2] Building NORMAL image groups...
NORMAL images: 76209
NORMAL image groups: 7606
✓ Every 7606 V4 case has at least 1 NORMAL image
Images/case:
  min    = 1
  median = 9
  mean   = 10.0196
  max    = 31
Raw images/case max: 31
V4 training images/case will be capped at 8
✓ Raw image groups verified
✓ MAX_IMAGES=8 will be enforced during Dataset loading


AssertionError: 

In [20]:
# ================================================================
# STRUCTURED TARGET COLUMN AUDIT
# NO TRAINING / NO CHECKPOINT MODIFICATION
# ================================================================

print("=" * 70)
print("STRUCTURED TARGET COLUMN AUDIT")
print("=" * 70)

print("\nShape:")
print(structured_df.shape)

print("\nIndex name:")
print(structured_df.index.name)

print("\nFirst 30 columns:")
for i, c in enumerate(structured_df.columns[:30]):
    print(i, repr(c))

print("\nLast 30 columns:")
start = max(0, len(structured_df.columns) - 30)

for i, c in enumerate(
    structured_df.columns[start:],
    start=start
):
    print(i, repr(c))

# ------------------------------------------------
# Prefix detection
# ------------------------------------------------

concept_cols = [
    c for c in structured_df.columns
    if str(c).startswith("concept__")
]

negative_cols = [
    c for c in structured_df.columns
    if str(c).startswith("negative__")
]

attribute_cols = [
    c for c in structured_df.columns
    if str(c).startswith("attribute__")
]

print("\nPrefix detection:")
print("concept__  :", len(concept_cols))
print("negative__ :", len(negative_cols))
print("attribute__:", len(attribute_cols))

print("\nDetected concept columns:")
print(concept_cols)

print("\nDetected negative columns:")
print(negative_cols)

print("\nDetected attribute columns:")
print(attribute_cols)

print("\n" + "=" * 70)
print("NO TRAINING")
print("NO CHECKPOINT MODIFIED")
print("=" * 70)

STRUCTURED TARGET COLUMN AUDIT

Shape:
(7606, 101)

Index name:
case_id

First 30 columns:
0 'patient_group_id'
1 'split'
2 'pdf_path'
3 'ho_ten'
4 'nam_sinh'
5 'gioi_tinh_clean'
6 'ly_do_noi_soi'
7 'tai'
8 'hoc_mui'
9 'hong_mui'
10 'hong_thanh_quan'
11 'hong_mieng'
12 'ket_luan'
13 'phan_biet'
14 'de_nghi'
15 'target_norm'
16 'concepts'
17 'attributes'
18 'num_concepts'
19 'num_attributes'
20 'concept__viem_ong_tai_ngoai'
21 'concept__nhot_ong_tai_ngoai'
22 'concept__chan_thuong_ong_tai_ngoai'
23 'concept__viem_tai_giua'
24 'concept__viem_mang_nhi'
25 'concept__thung_mang_nhi'
26 'concept__xep_mang_nhi'
27 'concept__ray_tai'
28 'concept__viem_tai_xuong_chum'
29 'concept__hau_phau_va_nhi'

Last 30 columns:
71 'attributes_str'
72 'concepts_v2'
73 'negative_findings'
74 'attributes_v2'
75 'concept__hep_ong_tai_ngoai'
76 'concept__polyp_ong_tai_ngoai'
77 'concept__chan_thuong_mang_nhi'
78 'concept__viem_vanh_tai'
79 'concept__tu_dich_vanh_tai'
80 'concept__ro_luan_nhi'
81 'concept__not_va

In [21]:
# ================================================================
# V4 — FROZEN ONTOLOGY COLUMN AUDIT + FIX
# NO TRAINING
# NO CHECKPOINT MODIFICATION
# ================================================================

print("=" * 70)
print("V4 FROZEN ONTOLOGY ALIGNMENT")
print("=" * 70)

# ------------------------------------------------
# EXACT 48 CONCEPTS FROM FROZEN V4 ONTOLOGY
# ------------------------------------------------

concept_cols = [
    "concept__amidan_qua_phat",
    "concept__chan_thuong_mang_nhi",
    "concept__chan_thuong_ong_tai_ngoai",
    "concept__chay_mau_mui",
    "concept__di_vat_hong",
    "concept__di_vat_hong_thanh_quan",
    "concept__di_vat_tai",
    "concept__dich_vat_mui",
    "concept__hat_day_thanh",
    "concept__hau_phau_mui_xoang",
    "concept__hau_phau_va_nhi",
    "concept__hep_ong_tai_ngoai",
    "concept__hoc_xuong_ca",
    "concept__lech_vach_ngan",
    "concept__liet_day_thanh",
    "concept__nang_day_thanh",
    "concept__nhot_ong_tai_ngoai",
    "concept__not_vanh_tai",
    "concept__polyp_day_thanh",
    "concept__polyp_mui",
    "concept__polyp_ong_tai_ngoai",
    "concept__qua_phat_va",
    "concept__ray_tai",
    "concept__ro_luan_nhi",
    "concept__seo_hoc_mui",
    "concept__theo_doi_trao_nguoc",
    "concept__thung_mang_nhi",
    "concept__tien_dinh_mui",
    "concept__tu_dich_vanh_tai",
    "concept__u_hoc_mui",
    "concept__u_nhu_cuon_mui",
    "concept__u_nhu_hoc_mui",
    "concept__u_xoang",
    "concept__viem_amidan",
    "concept__viem_hong",
    "concept__viem_luoi",
    "concept__viem_mang_nhi",
    "concept__viem_mieng",
    "concept__viem_mui",
    "concept__viem_mui_xoang",
    "concept__viem_ong_tai_ngoai",
    "concept__viem_tai_giua",
    "concept__viem_tai_xuong_chum",
    "concept__viem_thanh_quan",
    "concept__viem_va",
    "concept__viem_vanh_tai",
    "concept__viem_xoang",
    "concept__xep_mang_nhi",
]

# ------------------------------------------------
# EXACT 5 NEGATIVES
# ------------------------------------------------

negative_cols = [
    "negative__no_abnormal_external_middle_ear",
    "negative__no_abnormal_nose_sinus",
    "negative__no_abnormal_ent",
    "negative__no_bleeding",
    "negative__no_foreign_body",
]

# ------------------------------------------------
# EXACT 6 ATTRIBUTES
# ------------------------------------------------

attribute_cols = [
    "attribute__acute",
    "attribute__chronic",
    "attribute__right",
    "attribute__left",
    "attribute__bilateral",
    "attribute__post_surgery",
]

# ------------------------------------------------
# CHECK EXISTENCE
# ------------------------------------------------

all_structured_cols = (
    concept_cols
    + negative_cols
    + attribute_cols
)

missing = [
    c
    for c in all_structured_cols
    if c not in structured_df.columns
]

assert not missing, (
    "Missing frozen ontology columns:\n"
    + "\n".join(missing)
)

# ------------------------------------------------
# CHECK COUNTS
# ------------------------------------------------

assert len(concept_cols) == 48
assert len(negative_cols) == 5
assert len(attribute_cols) == 6

assert len(all_structured_cols) == 59

# ------------------------------------------------
# CHECK DUPLICATES
# ------------------------------------------------

assert len(
    set(all_structured_cols)
) == 59

# ------------------------------------------------
# CHECK BINARY VALUES
# ------------------------------------------------

for c in all_structured_cols:

    vals = (
        pd.to_numeric(
            structured_df[c],
            errors="coerce"
        )
        .dropna()
        .unique()
    )

    bad = [
        v for v in vals
        if v not in (0, 1)
    ]

    assert not bad, (
        f"Non-binary values in {c}: {bad}"
    )

# ------------------------------------------------
# PRINT
# ------------------------------------------------

print(
    "✓ Frozen concepts :",
    len(concept_cols)
)

print(
    "✓ Frozen negatives:",
    len(negative_cols)
)

print(
    "✓ Frozen attrs    :",
    len(attribute_cols)
)

print(
    "✓ Total structured:",
    len(all_structured_cols)
)

print(
    "\nExtra concept columns in source "
    "are intentionally ignored:"
)

extra_concepts = [
    c
    for c in structured_df.columns
    if c.startswith("concept__")
    and c not in concept_cols
]

for c in extra_concepts:
    print("  -", c)

print("\n" + "=" * 70)
print("✓ FROZEN ONTOLOGY ALIGNMENT PASS")
print("✓ EXACT 48 / 5 / 6 USED")
print("✓ NO TRAINING")
print("✓ NO CHECKPOINT MODIFIED")
print("=" * 70)

V4 FROZEN ONTOLOGY ALIGNMENT
✓ Frozen concepts : 48
✓ Frozen negatives: 5
✓ Frozen attrs    : 6
✓ Total structured: 59

Extra concept columns in source are intentionally ignored:
  - concept__di_vat
  - concept__ap_to_mieng
  - concept__khong_thay_benh_ly_tai
  - concept__khong_thay_benh_ly
  - concept__khong_thay_diem_chay_mau
  - concept__khong_thay_di_vat

✓ FROZEN ONTOLOGY ALIGNMENT PASS
✓ EXACT 48 / 5 / 6 USED
✓ NO TRAINING
✓ NO CHECKPOINT MODIFIED


In [22]:
# ================================================================
# V4 — DATASET ONE-BATCH AUDIT
# NO MODEL
# NO TRAINING
# NO CHECKPOINT MODIFICATION
# ================================================================

print("=" * 70)
print("V4 DATASET ONE-BATCH AUDIT")
print("=" * 70)

# ------------------------------------------------
# Dataset
# ------------------------------------------------

class V4DatasetAudit(Dataset):

    def __init__(self, cases, split):

        self.df = (
            cases[
                cases["split"] == split
            ]
            .copy()
            .reset_index(drop=True)
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        case_id = str(
            row["case_id"]
        )

        imgs = image_groups[case_id]

        # V4 hard cap
        imgs = imgs.iloc[
            :MAX_IMAGES
        ]

        pixel_values = []

        for _, img_row in imgs.iterrows():

            path = img_row["image_path"]

            img = Image.open(
                path
            ).convert("RGB")

            img = img.resize(
                (224, 224),
                Image.Resampling.BILINEAR
            )

            arr = (
                np.asarray(img)
                .astype(np.float32)
                / 255.0
            )

            arr = torch.from_numpy(
                arr
            ).permute(2, 0, 1)

            mean = torch.tensor(
                [0.5, 0.5, 0.5],
                dtype=torch.float32
            ).view(3, 1, 1)

            std = torch.tensor(
                [0.5, 0.5, 0.5],
                dtype=torch.float32
            ).view(3, 1, 1)

            arr = (
                arr - mean
            ) / std

            pixel_values.append(arr)

        n_images = len(
            pixel_values
        )

        assert 1 <= n_images <= MAX_IMAGES

        while len(pixel_values) < MAX_IMAGES:

            pixel_values.append(
                torch.zeros_like(
                    pixel_values[0]
                )
            )

        pixel_values = torch.stack(
            pixel_values
        )

        image_mask = torch.zeros(
            MAX_IMAGES,
            dtype=torch.float32
        )

        image_mask[
            :n_images
        ] = 1.0

        # ------------------------------------------------
        # Report
        # ------------------------------------------------

        target = str(
            row["ket_luan"]
        )

        encoded = tokenizer(
            target,
            max_length=MAX_TARGET_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = encoded[
            "input_ids"
        ].squeeze(0)

        labels[
            labels == tokenizer.pad_token_id
        ] = -100

        # ------------------------------------------------
        # Structured
        # ------------------------------------------------

        s = structured_df.loc[
            case_id
        ]

        concepts = torch.tensor(
            s[concept_cols]
            .astype(float)
            .values,
            dtype=torch.float32
        )

        negatives = torch.tensor(
            s[negative_cols]
            .astype(float)
            .values,
            dtype=torch.float32
        )

        attributes = torch.tensor(
            s[attribute_cols]
            .astype(float)
            .values,
            dtype=torch.float32
        )

        return {
            "case_id": case_id,
            "pixel_values": pixel_values,
            "image_mask": image_mask,
            "labels": labels,
            "concept_targets": concepts,
            "negative_targets": negatives,
            "attribute_targets": attributes,
        }


# ------------------------------------------------
# Create audit loader
# ------------------------------------------------

audit_ds = V4DatasetAudit(
    case_df,
    "train"
)

audit_loader = DataLoader(
    audit_ds,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

batch = next(
    iter(audit_loader)
)

# ------------------------------------------------
# Shape checks
# ------------------------------------------------

print("\n[1] Shapes")

print(
    "pixel_values :",
    tuple(batch["pixel_values"].shape)
)

print(
    "image_mask   :",
    tuple(batch["image_mask"].shape)
)

print(
    "labels       :",
    tuple(batch["labels"].shape)
)

print(
    "concept      :",
    tuple(batch["concept_targets"].shape)
)

print(
    "negative     :",
    tuple(batch["negative_targets"].shape)
)

print(
    "attribute    :",
    tuple(batch["attribute_targets"].shape)
)

assert batch[
    "pixel_values"
].shape == (
    4,
    8,
    3,
    224,
    224
)

assert batch[
    "image_mask"
].shape == (
    4,
    8
)

assert batch[
    "labels"
].shape == (
    4,
    96
)

assert batch[
    "concept_targets"
].shape == (
    4,
    48
)

assert batch[
    "negative_targets"
].shape == (
    4,
    5
)

assert batch[
    "attribute_targets"
].shape == (
    4,
    6
)


# ------------------------------------------------
# Image mask
# ------------------------------------------------

print("\n[2] Image counts")

for i, case_id in enumerate(
    batch["case_id"]
):

    n = int(
        batch["image_mask"][i].sum()
    )

    print(
        case_id,
        "->",
        n,
        "images"
    )

    assert 1 <= n <= 8


# ------------------------------------------------
# Labels
# ------------------------------------------------

print("\n[3] Labels")

valid_label_count = (
    batch["labels"] != -100
).sum(dim=1)

print(
    "Valid label tokens:",
    valid_label_count.tolist()
)

assert (
    valid_label_count > 0
).all()


# ------------------------------------------------
# Structured values
# ------------------------------------------------

print("\n[4] Structured values")

for name in [
    "concept_targets",
    "negative_targets",
    "attribute_targets",
]:

    x = batch[name]

    unique = torch.unique(x)

    print(
        name,
        "unique =",
        unique.tolist()
    )

    assert torch.all(
        (x == 0) | (x == 1)
    )


# ------------------------------------------------
# Numerical sanity
# ------------------------------------------------

print("\n[5] Numerical sanity")

for name in [
    "pixel_values",
    "labels",
    "concept_targets",
    "negative_targets",
    "attribute_targets",
]:

    x = batch[name]

    if x.is_floating_point():

        assert torch.isfinite(x).all()

        print(
            name,
            ": FINITE"
        )


# ------------------------------------------------
# Image range
# ------------------------------------------------

pixels = batch[
    "pixel_values"
]

print(
    "\nPixel range:",
    float(pixels.min()),
    "to",
    float(pixels.max())
)

assert torch.isfinite(
    pixels
).all()


# ------------------------------------------------
# Final
# ------------------------------------------------

print("\n" + "=" * 70)
print("✓ DATASET ONE-BATCH AUDIT PASS")
print("=" * 70)

print("4 cases")
print("max 8 images/case")
print("labels: 4 × 96")
print("structured: 48 / 5 / 6")
print("NO MODEL CREATED")
print("NO TRAINING")
print("NO CHECKPOINT MODIFIED")
print("=" * 70)

V4 DATASET ONE-BATCH AUDIT

[1] Shapes
pixel_values : (4, 8, 3, 224, 224)
image_mask   : (4, 8)
labels       : (4, 96)
concept      : (4, 48)
negative     : (4, 5)
attribute    : (4, 6)

[2] Image counts
10000.10000.0.10014 -> 8 images
10001.10001.0.10015 -> 7 images
10002.10002.0.10016 -> 8 images
10004.10004.0.10018 -> 8 images

[3] Labels
Valid label tokens: [13, 7, 11, 13]

[4] Structured values
concept_targets unique = [0.0, 1.0]
negative_targets unique = [0.0]
attribute_targets unique = [0.0, 1.0]

[5] Numerical sanity
pixel_values : FINITE
concept_targets : FINITE
negative_targets : FINITE
attribute_targets : FINITE

Pixel range: -0.9843137264251709 to 1.0

✓ DATASET ONE-BATCH AUDIT PASS
4 cases
max 8 images/case
labels: 4 × 96
structured: 48 / 5 / 6
NO MODEL CREATED
NO TRAINING
NO CHECKPOINT MODIFIED


In [23]:
# ================================================================
# V4 — FINAL MODEL / CHECKPOINT / GRADIENT AUDIT
#
# NO TRAINING
# NO optimizer.step()
# NO scheduler.step()
# NO CHECKPOINT SAVE
# ================================================================

print("=" * 70)
print("V4 FINAL MODEL / CHECKPOINT / GRADIENT AUDIT")
print("=" * 70)

# ================================================================
# 0. IMPORTS / DEVICE
# ================================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import (
    ViTModel,
    MT5ForConditionalGeneration,
)

print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)
print("Device:", DEVICE)

# ================================================================
# 1. EXACT CONDITIONER
# ================================================================

class StructuredConditioner(nn.Module):

    def __init__(
        self,
        d_model,
        num_concepts=48,
        num_negative=5,
        num_attributes=6,
    ):
        super().__init__()

        self.num_concepts = num_concepts
        self.num_negative = num_negative
        self.num_attributes = num_attributes

        self.concept_value = nn.Embedding(
            2, d_model
        )

        self.negative_value = nn.Embedding(
            2, d_model
        )

        self.attribute_value = nn.Embedding(
            2, d_model
        )

        self.concept_label = nn.Embedding(
            num_concepts, d_model
        )

        self.negative_label = nn.Embedding(
            num_negative, d_model
        )

        self.attribute_label = nn.Embedding(
            num_attributes, d_model
        )

        self.type_embedding = nn.Embedding(
            3, d_model
        )

        self.norm = nn.LayerNorm(
            d_model
        )

    def forward(
        self,
        concept_targets,
        negative_targets,
        attribute_targets,
    ):

        device = concept_targets.device

        concept_ids = (
            concept_targets
            .long()
            .clamp(0, 1)
        )

        negative_ids = (
            negative_targets
            .long()
            .clamp(0, 1)
        )

        attribute_ids = (
            attribute_targets
            .long()
            .clamp(0, 1)
        )

        # ------------------------------------------------
        # CONCEPT
        # ------------------------------------------------

        concept_idx = torch.arange(
            self.num_concepts,
            device=device,
        )

        concept_type = self.type_embedding(
            torch.zeros(
                self.num_concepts,
                dtype=torch.long,
                device=device,
            )
        )

        concept_tokens = (
            self.concept_value(
                concept_ids
            )
            + self.concept_label(
                concept_idx
            )[None, :, :]
            + concept_type[None, :, :]
        )

        # ------------------------------------------------
        # NEGATIVE
        # ------------------------------------------------

        negative_idx = torch.arange(
            self.num_negative,
            device=device,
        )

        negative_type = self.type_embedding(
            torch.ones(
                self.num_negative,
                dtype=torch.long,
                device=device,
            )
        )

        negative_tokens = (
            self.negative_value(
                negative_ids
            )
            + self.negative_label(
                negative_idx
            )[None, :, :]
            + negative_type[None, :, :]
        )

        # ------------------------------------------------
        # ATTRIBUTE
        # ------------------------------------------------

        attribute_idx = torch.arange(
            self.num_attributes,
            device=device,
        )

        attribute_type = self.type_embedding(
            torch.full(
                (self.num_attributes,),
                2,
                dtype=torch.long,
                device=device,
            )
        )

        attribute_tokens = (
            self.attribute_value(
                attribute_ids
            )
            + self.attribute_label(
                attribute_idx
            )[None, :, :]
            + attribute_type[None, :, :]
        )

        # ------------------------------------------------
        # CONCAT
        # ------------------------------------------------

        structured_tokens = torch.cat(
            [
                concept_tokens,
                negative_tokens,
                attribute_tokens,
            ],
            dim=1,
        )

        structured_tokens = self.norm(
            structured_tokens
        )

        B = concept_targets.shape[0]

        structured_attention = torch.ones(
            B,
            structured_tokens.shape[1],
            dtype=torch.long,
            device=device,
        )

        return (
            structured_tokens,
            structured_attention,
        )


# ================================================================
# 2. EXACT V4 MODEL
# ================================================================

class V4Model(nn.Module):

    def __init__(self):

        super().__init__()

        self.vision = ViTModel.from_pretrained(
            "google/vit-base-patch16-224"
        )

        self.mt5 = MT5ForConditionalGeneration.from_pretrained(
            "google/mt5-small"
        )

        self.projector = nn.Linear(
            768,
            512,
        )

        self.conditioner = StructuredConditioner(
            512,
            48,
            5,
            6,
        )

        # EXACT checkpoint naming
        self.structured_heads = nn.ModuleDict({
            "concept": nn.Linear(
                512,
                48,
            ),

            "negative": nn.Linear(
                512,
                5,
            ),

            "attribute": nn.Linear(
                512,
                6,
            ),
        })

    def forward(
        self,
        pixel_values,
        image_mask,
        labels,
        concept_targets,
        negative_targets,
        attribute_targets,
    ):

        B, N, C, H, W = pixel_values.shape

        # ------------------------------------------------
        # VISION
        # ------------------------------------------------

        x = pixel_values.reshape(
            B * N,
            C,
            H,
            W,
        )

        vision_out = self.vision(
            pixel_values=x
        ).last_hidden_state[:, 0]

        vision_out = vision_out.reshape(
            B,
            N,
            768,
        )

        # ------------------------------------------------
        # PROJECT
        # ------------------------------------------------

        visual_tokens = self.projector(
            vision_out
        )

        visual_tokens = (
            visual_tokens
            * image_mask[:, :, None]
        )

        # ------------------------------------------------
        # POOL
        # ------------------------------------------------

        denom = (
            image_mask.sum(
                dim=1,
                keepdim=True,
            )
            .clamp(min=1)
        )

        pooled = (
            visual_tokens.sum(dim=1)
            / denom
        )

        # ------------------------------------------------
        # STRUCTURED HEADS
        # ------------------------------------------------

        concept_logits = (
            self.structured_heads[
                "concept"
            ](pooled)
        )

        negative_logits = (
            self.structured_heads[
                "negative"
            ](pooled)
        )

        attribute_logits = (
            self.structured_heads[
                "attribute"
            ](pooled)
        )

        # ------------------------------------------------
        # HARD PREDICTIONS
        # ------------------------------------------------

        concept_pred = (
            torch.sigmoid(
                concept_logits
            ) >= 0.5
        ).float()

        negative_pred = (
            torch.sigmoid(
                negative_logits
            ) >= 0.5
        ).float()

        attribute_pred = (
            torch.sigmoid(
                attribute_logits
            ) >= 0.5
        ).float()

        # ------------------------------------------------
        # CONDITIONER
        # ------------------------------------------------

        structured_tokens, structured_attention = (
            self.conditioner(
                concept_pred,
                negative_pred,
                attribute_pred,
            )
        )

        # ------------------------------------------------
        # PREFIX
        # ------------------------------------------------

        visual_attention = image_mask.long()

        prefix = torch.cat(
            [
                visual_tokens,
                structured_tokens,
            ],
            dim=1,
        )

        attention_mask = torch.cat(
            [
                visual_attention,
                structured_attention,
            ],
            dim=1,
        )

        # ------------------------------------------------
        # MT5
        # ------------------------------------------------

        zero_ids = torch.zeros(
            B,
            prefix.shape[1],
            dtype=torch.long,
            device=prefix.device,
        )

        base_embeddings = (
            self.mt5.encoder.embed_tokens(
                zero_ids
            )
        )

        inputs_embeds = (
            base_embeddings
            + prefix
        )

        encoder_outputs = self.mt5.encoder(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            return_dict=True,
        )

        output = self.mt5(
            encoder_outputs=encoder_outputs,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=True,
        )

        report_loss = output.loss

        concept_loss = (
            F.binary_cross_entropy_with_logits(
                concept_logits,
                concept_targets,
            )
        )

        negative_loss = (
            F.binary_cross_entropy_with_logits(
                negative_logits,
                negative_targets,
            )
        )

        attribute_loss = (
            F.binary_cross_entropy_with_logits(
                attribute_logits,
                attribute_targets,
            )
        )

        total_loss = (
            report_loss
            + 0.5 * concept_loss
            + 0.5 * negative_loss
            + 0.25 * attribute_loss
        )

        return {
            "loss": total_loss,
            "report_loss": report_loss,
            "concept_loss": concept_loss,
            "negative_loss": negative_loss,
            "attribute_loss": attribute_loss,
            "concept_logits": concept_logits,
            "negative_logits": negative_logits,
            "attribute_logits": attribute_logits,
            "visual_tokens": visual_tokens,
            "structured_tokens": structured_tokens,
            "prefix": prefix,
        }


# ================================================================
# 3. CREATE MODEL
# ================================================================

print("\n[1] Creating exact V4 model...")

model = V4Model().to(
    DEVICE
)

print("✓ Model instantiated")


# ================================================================
# 4. CHECK EXACT STATE-DICT KEY SET
# ================================================================

print("\n[2] State-dict compatibility...")

checkpoint = torch.load(
    LAST_CKPT,
    map_location="cpu",
)

model_keys = set(
    model.state_dict().keys()
)

checkpoint_keys = set(
    checkpoint["model_state"].keys()
)

missing = sorted(
    checkpoint_keys - model_keys
)

unexpected = sorted(
    model_keys - checkpoint_keys
)

print(
    "Model keys:",
    len(model_keys)
)

print(
    "Checkpoint keys:",
    len(checkpoint_keys)
)

print(
    "Missing:",
    len(missing)
)

print(
    "Unexpected:",
    len(unexpected)
)

assert len(missing) == 0
assert len(unexpected) == 0

assert len(model_keys) == 409
assert len(checkpoint_keys) == 409

print(
    "✓ Exact 409-key match"
)


# ================================================================
# 5. STRICT LOAD
# ================================================================

print("\n[3] Strict checkpoint load...")

model.load_state_dict(
    checkpoint["model_state"],
    strict=True,
)

print(
    "✓ STRICT LOAD PASS"
)


# ================================================================
# 6. OPTIMIZER
# ================================================================

print("\n[4] Creating exact optimizer...")

optimizer = torch.optim.AdamW(
    [
        {
            "params":
                model.vision.parameters(),
            "lr":
                1e-5,
            "weight_decay":
                0.01,
        },

        {
            "params":
                model.projector.parameters(),
            "lr":
                1e-4,
            "weight_decay":
                0.01,
        },

        {
            "params":
                model.conditioner.parameters(),
            "lr":
                1e-4,
            "weight_decay":
                0.01,
        },

        {
            "params":
                model.structured_heads.parameters(),
            "lr":
                1e-4,
            "weight_decay":
                0.01,
        },

        {
            "params":
                model.mt5.parameters(),
            "lr":
                5e-5,
            "weight_decay":
                0.01,
        },
    ]
)

assert len(
    optimizer.param_groups
) == 5

optimizer.load_state_dict(
    checkpoint["optimizer_state"]
)

print(
    "✓ 5 optimizer groups"
)

expected_lrs = [
    6.54508497187e-06,
    6.54508497187e-05,
    6.54508497187e-05,
    6.54508497187e-05,
    3.27254248594e-05,
]

for i, expected in enumerate(
    expected_lrs
):

    actual = optimizer.param_groups[
        i
    ]["lr"]

    print(
        f"Group {i}: "
        f"{actual:.14e}"
    )

    assert abs(
        actual - expected
    ) < 1e-12

print(
    "✓ Optimizer state PASS"
)


# ================================================================
# 7. SCHEDULER
# ================================================================

print("\n[5] Scheduler...")

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=1920,
    eta_min=0,
)

scheduler.load_state_dict(
    checkpoint["scheduler_state"]
)

print(
    "T_max:",
    scheduler.T_max
)

print(
    "last_epoch:",
    scheduler.last_epoch
)

print(
    "_step_count:",
    scheduler._step_count
)

assert scheduler.T_max == 1920
assert scheduler.last_epoch == 768
assert scheduler._step_count == 769

print(
    "✓ Scheduler state PASS"
)


# ================================================================
# 8. PREPARE ONE AUDIT BATCH
# ================================================================

print("\n[6] Preparing one-batch forward...")

# Reuse the already audited dataset if available.
# Otherwise construct it from V4DatasetAudit.

if "audit_loader" not in globals():

    audit_ds = V4DatasetAudit(
        case_df,
        "train",
    )

    audit_loader = DataLoader(
        audit_ds,
        batch_size=4,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

batch = next(
    iter(audit_loader)
)

pixel_values = batch[
    "pixel_values"
].to(
    DEVICE,
    non_blocking=True,
)

image_mask = batch[
    "image_mask"
].to(
    DEVICE,
    non_blocking=True,
)

labels = batch[
    "labels"
].to(
    DEVICE,
    non_blocking=True,
)

concept_targets = batch[
    "concept_targets"
].to(
    DEVICE,
    non_blocking=True,
)

negative_targets = batch[
    "negative_targets"
].to(
    DEVICE,
    non_blocking=True,
)

attribute_targets = batch[
    "attribute_targets"
].to(
    DEVICE,
    non_blocking=True,
)

print(
    "Cases:",
    batch["case_id"]
)


# ================================================================
# 9. FORWARD DRY RUN
# ================================================================

print("\n[7] Forward dry run...")

model.train()

use_bf16 = (
    torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
)

amp_ctx = (
    torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    )
    if use_bf16
    else nullcontext()
)

optimizer.zero_grad(
    set_to_none=True
)

with amp_ctx:

    outputs = model(
        pixel_values=pixel_values,
        image_mask=image_mask,
        labels=labels,
        concept_targets=concept_targets,
        negative_targets=negative_targets,
        attribute_targets=attribute_targets,
    )

print(
    "Total loss:",
    outputs["loss"].item()
)

print(
    "Report loss:",
    outputs["report_loss"].item()
)

print(
    "Concept logits:",
    tuple(
        outputs["concept_logits"].shape
    )
)

print(
    "Negative logits:",
    tuple(
        outputs["negative_logits"].shape
    )
)

print(
    "Attribute logits:",
    tuple(
        outputs["attribute_logits"].shape
    )
)

print(
    "Visual tokens:",
    tuple(
        outputs["visual_tokens"].shape
    )
)

print(
    "Structured tokens:",
    tuple(
        outputs["structured_tokens"].shape
    )
)

print(
    "Prefix:",
    tuple(
        outputs["prefix"].shape
    )
)

assert outputs[
    "concept_logits"
].shape == (4, 48)

assert outputs[
    "negative_logits"
].shape == (4, 5)

assert outputs[
    "attribute_logits"
].shape == (4, 6)

assert outputs[
    "visual_tokens"
].shape == (4, 8, 512)

assert outputs[
    "structured_tokens"
].shape == (4, 59, 512)

assert outputs[
    "prefix"
].shape == (4, 67, 512)

print(
    "✓ Forward shape PASS"
)


# ================================================================
# 10. NUMERICAL SANITY
# ================================================================

print("\n[8] Numerical sanity...")

for name in [
    "loss",
    "report_loss",
    "concept_loss",
    "negative_loss",
    "attribute_loss",
    "concept_logits",
    "negative_logits",
    "attribute_logits",
    "visual_tokens",
    "structured_tokens",
    "prefix",
]:

    value = outputs[name]

    assert torch.isfinite(
        value
    ).all(), (
        f"NaN/Inf detected: {name}"
    )

    print(
        f"{name:20s}: FINITE"
    )

print(
    "✓ No NaN / Inf"
)


# ================================================================
# 11. BACKWARD / GRADIENT AUDIT
#
# IMPORTANT:
# backward() ONLY.
# NO optimizer.step()
# NO scheduler.step()
# ================================================================

print("\n[9] Gradient audit...")

outputs["loss"].backward()

gradient_stats = {}

for name, param in model.named_parameters():

    if not param.requires_grad:
        continue

    if param.grad is None:

        gradient_stats[name] = (
            "NONE"
        )

    else:

        grad = param.grad

        assert torch.isfinite(
            grad
        ).all(), (
            f"NaN/Inf gradient: {name}"
        )

        gradient_stats[name] = (
            float(
                grad.detach()
                .abs()
                .mean()
                .item()
            )
        )


# ------------------------------------------------
# Component-level gradient audit
# ------------------------------------------------

def component_grad_stats(module):

    total = 0
    nonzero = 0
    none = 0

    for p in module.parameters():

        if not p.requires_grad:
            continue

        total += 1

        if p.grad is None:

            none += 1

        elif torch.any(
            p.grad != 0
        ):

            nonzero += 1

    return total, nonzero, none


components = {
    "vision":
        model.vision,

    "projector":
        model.projector,

    "conditioner":
        model.conditioner,

    "structured_heads":
        model.structured_heads,

    "mt5":
        model.mt5,
}

for name, module in components.items():

    total, nonzero, none = (
        component_grad_stats(
            module
        )
    )

    print(
        f"{name:20s} "
        f"params={total} "
        f"nonzero_grad={nonzero} "
        f"none_grad={none}"
    )

    assert total > 0


# ------------------------------------------------
# Check structured heads specifically
# ------------------------------------------------

for name, param in (
    model.structured_heads
    .named_parameters()
):

    assert param.grad is not None

    assert torch.isfinite(
        param.grad
    ).all()

print(
    "✓ Structured heads receive gradients"
)


# ================================================================
# 12. IMPORTANT: DO NOT STEP
# ================================================================

print("\n[10] Optimizer/scheduler safety check")

print(
    "Optimizer step was NOT called."
)

print(
    "Scheduler step was NOT called."
)

print(
    "Checkpoint save was NOT called."
)

assert scheduler.last_epoch == 768

print(
    "✓ Scheduler remains at epoch-2 position"
)


# ================================================================
# 13. FINAL RESULT
# ================================================================

print("\n" + "=" * 70)
print("FINAL MODEL / CHECKPOINT / GRADIENT AUDIT RESULT")
print("=" * 70)

print("✓ Exact model architecture")
print("✓ 409 / 409 state-dict keys")
print("✓ Strict checkpoint load")
print("✓ Exact 5 optimizer groups")
print("✓ Optimizer state")
print("✓ Cosine scheduler state")
print("✓ One-batch forward")
print("✓ Output shapes")
print("✓ No NaN / Inf")
print("✓ Backward pass")
print("✓ Gradient sanity")
print("✓ NO optimizer.step()")
print("✓ NO scheduler.step()")
print("✓ NO checkpoint saved")
print("=" * 70)

print(
    "SAFE TO START EPOCH 3 TRAINING"
)

print("=" * 70)

V4 FINAL MODEL / CHECKPOINT / GRADIENT AUDIT
Transformers: 5.16.1
Torch: 2.11.0+cu128
Device: cuda

[1] Creating exact V4 model...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✓ Model instantiated

[2] State-dict compatibility...
Model keys: 409
Checkpoint keys: 409
Missing: 2
Unexpected: 2


AssertionError: 

In [24]:
# ================================================================
# V4 — INVESTIGATE THE 4 EXPECTED ViT KEY MISMATCHES
# NO TRAINING / NO SAVE
# ================================================================

model_keys = set(model.state_dict().keys())
ckpt_keys = set(checkpoint["model_state"].keys())

missing = sorted(ckpt_keys - model_keys)
unexpected = sorted(model_keys - ckpt_keys)

print("Checkpoint-only keys:")
for k in missing:
    print("  ", k)

print("\nModel-only keys:")
for k in unexpected:
    print("  ", k)

expected_ckpt_only = {
    "vision.classifier.weight",
    "vision.classifier.bias",
}

expected_model_only = {
    "vision.pooler.dense.weight",
    "vision.pooler.dense.bias",
}

assert set(missing) == expected_ckpt_only, (
    f"Unexpected checkpoint-only keys: {missing}"
)

assert set(unexpected) == expected_model_only, (
    f"Unexpected model-only keys: {unexpected}"
)

common_keys = model_keys & ckpt_keys

print("\nCommon keys:", len(common_keys))

assert len(common_keys) == 405

# Compare shapes of every common key
shape_mismatch = []

for k in sorted(common_keys):

    ckpt_shape = tuple(
        checkpoint["model_state"][k].shape
    )

    model_shape = tuple(
        model.state_dict()[k].shape
    )

    if ckpt_shape != model_shape:
        shape_mismatch.append(
            (k, ckpt_shape, model_shape)
        )

print("Shape mismatches among common keys:",
      len(shape_mismatch))

assert len(shape_mismatch) == 0

print("\n✓ All 405 common keys match exactly")
print("✓ Only the known ViT classifier/pooler mismatch remains")
print("✓ No training performed")
print("✓ No checkpoint modified")

Checkpoint-only keys:
   projector.proj.bias
   projector.proj.weight

Model-only keys:
   projector.bias
   projector.weight


AssertionError: Unexpected checkpoint-only keys: ['projector.proj.bias', 'projector.proj.weight']

In [25]:
# ================================================================
# V4 — EXACT PROJECTOR ARCHITECTURE INSPECTION
# NO TRAINING / NO SAVE / NO CHECKPOINT MODIFICATION
# ================================================================

print("=" * 70)
print("PROJECTOR ARCHITECTURE INSPECTION")
print("=" * 70)

# ------------------------------------------------
# 1. Check checkpoint projector keys
# ------------------------------------------------

print("\n[1] Checkpoint projector keys:")

for k, v in checkpoint["model_state"].items():

    if k.startswith("projector"):
        print(
            f"{k:40s}",
            tuple(v.shape),
            v.dtype
        )


# ------------------------------------------------
# 2. Current model projector keys
# ------------------------------------------------

print("\n[2] Current model projector keys:")

for k, v in model.state_dict().items():

    if k.startswith("projector"):
        print(
            f"{k:40s}",
            tuple(v.shape),
            v.dtype
        )


# ------------------------------------------------
# 3. Inspect projector object
# ------------------------------------------------

print("\n[3] Current projector object:")
print(model.projector)

print(
    "\nProjector class:",
    type(model.projector)
)


# ------------------------------------------------
# 4. Inspect checkpoint tensor values
# ------------------------------------------------

print("\n[4] Checkpoint projector tensor statistics:")

for k in [
    "projector.proj.weight",
    "projector.proj.bias",
]:

    x = checkpoint[
        "model_state"
    ][k]

    print(
        k,
        "shape=", tuple(x.shape),
        "mean=", float(x.float().mean()),
        "std=", float(x.float().std()),
    )


# ------------------------------------------------
# 5. Check whether projector wrapper is likely
# ------------------------------------------------

assert (
    "projector.proj.weight"
    in checkpoint["model_state"]
)

assert (
    "projector.proj.bias"
    in checkpoint["model_state"]
)

assert (
    "projector.weight"
    not in checkpoint["model_state"]
)

assert (
    "projector.bias"
    not in checkpoint["model_state"]
)

print(
    "\n✓ Checkpoint definitely expects "
    "projector.proj.*"
)

print(
    "\nNO TRAINING"
)

print(
    "NO optimizer.step()"
)

print(
    "NO scheduler.step()"
)

print(
    "NO checkpoint modification"
)

print("=" * 70)

PROJECTOR ARCHITECTURE INSPECTION

[1] Checkpoint projector keys:
projector.proj.weight                    (512, 768) torch.float32
projector.proj.bias                      (512,) torch.float32

[2] Current model projector keys:
projector.weight                         (512, 768) torch.float32
projector.bias                           (512,) torch.float32

[3] Current projector object:
Linear(in_features=768, out_features=512, bias=True)

Projector class: <class 'torch.nn.modules.linear.Linear'>

[4] Checkpoint projector tensor statistics:
projector.proj.weight shape= (512, 768) mean= -6.52116141282022e-05 std= 0.021367279812693596
projector.proj.bias shape= (512,) mean= 8.853679173626006e-05 std= 0.021150389686226845

✓ Checkpoint definitely expects projector.proj.*

NO TRAINING
NO optimizer.step()
NO scheduler.step()
NO checkpoint modification


In [27]:
# ================================================================
# V4 — PROJECTOR KEY FIX + EXACT STATE-DICT AUDIT
# NO TRAINING / NO SAVE
# ================================================================

class Projector(nn.Module):

    def __init__(
        self,
        in_dim=768,
        out_dim=512,
    ):
        super().__init__()

        self.proj = nn.Linear(
            in_dim,
            out_dim,
        )

    def forward(self, x):
        return self.proj(x)


# ------------------------------------------------
# Replace ONLY the projector
# ------------------------------------------------

model.projector = Projector(
    768,
    512,
).to(DEVICE)


# ------------------------------------------------
# Compare keys again
# ------------------------------------------------

model_keys = set(
    model.state_dict().keys()
)

ckpt_keys = set(
    checkpoint["model_state"].keys()
)

missing = sorted(
    ckpt_keys - model_keys
)

unexpected = sorted(
    model_keys - ckpt_keys
)

print("=" * 70)
print("AFTER PROJECTOR FIX")
print("=" * 70)

print(
    "Model keys:",
    len(model_keys)
)

print(
    "Checkpoint keys:",
    len(ckpt_keys)
)

print(
    "Missing:",
    missing
)

print(
    "Unexpected:",
    unexpected
)

assert len(model_keys) == 409
assert len(ckpt_keys) == 409

assert len(missing) == 0, missing
assert len(unexpected) == 0, unexpected

print(
    "\n✓ EXACT 409 / 409 KEY MATCH"
)


# ------------------------------------------------
# Check all tensor shapes
# ------------------------------------------------

shape_mismatch = []

model_state = model.state_dict()

for key in sorted(ckpt_keys):

    ckpt_shape = tuple(
        checkpoint["model_state"][key].shape
    )

    model_shape = tuple(
        model_state[key].shape
    )

    if ckpt_shape != model_shape:

        shape_mismatch.append(
            (
                key,
                ckpt_shape,
                model_shape,
            )
        )

print(
    "Shape mismatches:",
    len(shape_mismatch)
)

assert len(shape_mismatch) == 0, (
    shape_mismatch
)

print(
    "✓ ALL 409 SHAPES MATCH"
)


# ------------------------------------------------
# Strict load
# ------------------------------------------------

model.load_state_dict(
    checkpoint["model_state"],
    strict=True,
)

print(
    "✓ STRICT LOAD PASS"
)

print(
    "\nNO TRAINING"
)

print(
    "NO optimizer.step()"
)

print(
    "NO scheduler.step()"
)

print(
    "NO checkpoint save"
)

print("=" * 70)

AFTER PROJECTOR FIX
Model keys: 409
Checkpoint keys: 409
Missing: []
Unexpected: []

✓ EXACT 409 / 409 KEY MATCH
Shape mismatches: 0
✓ ALL 409 SHAPES MATCH
✓ STRICT LOAD PASS

NO TRAINING
NO optimizer.step()
NO scheduler.step()
NO checkpoint save


In [28]:
# ================================================================
# V4 — FINAL FORWARD / OPTIMIZER / SCHEDULER / GRADIENT AUDIT
#
# CHECKPOINT KEY MATCH ALREADY PASSED
# NO optimizer.step()
# NO scheduler.step()
# NO checkpoint save
# ================================================================

print("=" * 70)
print("V4 FINAL FORWARD / GRADIENT AUDIT")
print("=" * 70)

# ================================================================
# 1. EXACT OPTIMIZER
# ================================================================

print("\n[1] Restoring optimizer...")

optimizer = torch.optim.AdamW(
    [
        {
            "params": model.vision.parameters(),
            "lr": 1e-5,
            "weight_decay": 0.01,
        },
        {
            "params": model.projector.parameters(),
            "lr": 1e-4,
            "weight_decay": 0.01,
        },
        {
            "params": model.conditioner.parameters(),
            "lr": 1e-4,
            "weight_decay": 0.01,
        },
        {
            "params": model.structured_heads.parameters(),
            "lr": 1e-4,
            "weight_decay": 0.01,
        },
        {
            "params": model.mt5.parameters(),
            "lr": 5e-5,
            "weight_decay": 0.01,
        },
    ]
)

assert len(optimizer.param_groups) == 5

optimizer.load_state_dict(
    checkpoint["optimizer_state"]
)

expected_lrs = [
    6.54508497187e-06,
    6.54508497187e-05,
    6.54508497187e-05,
    6.54508497187e-05,
    3.27254248594e-05,
]

for i, expected in enumerate(expected_lrs):

    actual = optimizer.param_groups[i]["lr"]

    print(
        f"group {i}: "
        f"{actual:.14e}"
    )

    assert abs(actual - expected) < 1e-12

print("✓ Optimizer state PASS")


# ================================================================
# 2. SCHEDULER
# ================================================================

print("\n[2] Restoring scheduler...")

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=1920,
    eta_min=0,
)

scheduler.load_state_dict(
    checkpoint["scheduler_state"]
)

print(
    "T_max:",
    scheduler.T_max
)

print(
    "last_epoch:",
    scheduler.last_epoch
)

print(
    "_step_count:",
    scheduler._step_count
)

assert scheduler.T_max == 1920
assert scheduler.last_epoch == 768
assert scheduler._step_count == 769

print("✓ Scheduler state PASS")


# ================================================================
# 3. PREPARE AUDIT BATCH
# ================================================================

print("\n[3] Preparing one batch...")

if "audit_loader" not in globals():

    audit_ds = V4DatasetAudit(
        case_df,
        "train",
    )

    audit_loader = DataLoader(
        audit_ds,
        batch_size=4,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

batch = next(iter(audit_loader))

pixel_values = batch["pixel_values"].to(
    DEVICE,
    non_blocking=True,
)

image_mask = batch["image_mask"].to(
    DEVICE,
    non_blocking=True,
)

labels = batch["labels"].to(
    DEVICE,
    non_blocking=True,
)

concept_targets = batch["concept_targets"].to(
    DEVICE,
    non_blocking=True,
)

negative_targets = batch["negative_targets"].to(
    DEVICE,
    non_blocking=True,
)

attribute_targets = batch["attribute_targets"].to(
    DEVICE,
    non_blocking=True,
)

print("Cases:")
for x in batch["case_id"]:
    print(" ", x)

print(
    "pixel_values:",
    tuple(pixel_values.shape)
)

print(
    "image_mask:",
    tuple(image_mask.shape)
)

print(
    "labels:",
    tuple(labels.shape)
)

print(
    "concept:",
    tuple(concept_targets.shape)
)

print(
    "negative:",
    tuple(negative_targets.shape)
)

print(
    "attribute:",
    tuple(attribute_targets.shape)
)


# ================================================================
# 4. FORWARD
# ================================================================

print("\n[4] Forward pass...")

model.train()

optimizer.zero_grad(set_to_none=True)

if torch.cuda.is_available() and torch.cuda.is_bf16_supported():

    amp_ctx = torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    )

else:

    from contextlib import nullcontext
    amp_ctx = nullcontext()


with amp_ctx:

    outputs = model(
        pixel_values=pixel_values,
        image_mask=image_mask,
        labels=labels,
        concept_targets=concept_targets,
        negative_targets=negative_targets,
        attribute_targets=attribute_targets,
    )


print(
    "Total loss:",
    float(outputs["loss"].detach().float().cpu())
)

print(
    "Report loss:",
    float(outputs["report_loss"].detach().float().cpu())
)

print(
    "Concept loss:",
    float(outputs["concept_loss"].detach().float().cpu())
)

print(
    "Negative loss:",
    float(outputs["negative_loss"].detach().float().cpu())
)

print(
    "Attribute loss:",
    float(outputs["attribute_loss"].detach().float().cpu())
)


# ================================================================
# 5. EXACT OUTPUT SHAPES
# ================================================================

print("\n[5] Output shape audit...")

assert outputs["concept_logits"].shape == (
    4, 48
)

assert outputs["negative_logits"].shape == (
    4, 5
)

assert outputs["attribute_logits"].shape == (
    4, 6
)

assert outputs["visual_tokens"].shape == (
    4, 8, 512
)

assert outputs["structured_tokens"].shape == (
    4, 59, 512
)

assert outputs["prefix"].shape == (
    4, 67, 512
)

print(
    "✓ concept_logits     ",
    tuple(outputs["concept_logits"].shape)
)

print(
    "✓ negative_logits    ",
    tuple(outputs["negative_logits"].shape)
)

print(
    "✓ attribute_logits   ",
    tuple(outputs["attribute_logits"].shape)
)

print(
    "✓ visual_tokens      ",
    tuple(outputs["visual_tokens"].shape)
)

print(
    "✓ structured_tokens  ",
    tuple(outputs["structured_tokens"].shape)
)

print(
    "✓ prefix             ",
    tuple(outputs["prefix"].shape)
)


# ================================================================
# 6. NUMERICAL CHECK
# ================================================================

print("\n[6] Numerical check...")

check_tensors = {
    "loss": outputs["loss"],
    "report_loss": outputs["report_loss"],
    "concept_loss": outputs["concept_loss"],
    "negative_loss": outputs["negative_loss"],
    "attribute_loss": outputs["attribute_loss"],
    "concept_logits": outputs["concept_logits"],
    "negative_logits": outputs["negative_logits"],
    "attribute_logits": outputs["attribute_logits"],
    "visual_tokens": outputs["visual_tokens"],
    "structured_tokens": outputs["structured_tokens"],
    "prefix": outputs["prefix"],
}

for name, tensor in check_tensors.items():

    assert torch.isfinite(tensor).all(), (
        f"NaN/Inf detected in {name}"
    )

    print(
        f"✓ {name:20s} finite"
    )


# ================================================================
# 7. BACKWARD
#
# IMPORTANT:
# ONLY backward()
# NO optimizer.step()
# NO scheduler.step()
# ================================================================

print("\n[7] Backward / gradient audit...")

outputs["loss"].backward()

print("✓ backward() completed")


# ================================================================
# 8. COMPONENT GRADIENT AUDIT
# ================================================================

def grad_stats(module):

    total = 0
    with_grad = 0
    nonzero = 0
    none = 0

    for p in module.parameters():

        if not p.requires_grad:
            continue

        total += 1

        if p.grad is None:

            none += 1

        else:

            with_grad += 1

            assert torch.isfinite(
                p.grad
            ).all()

            if torch.any(p.grad != 0):
                nonzero += 1

    return total, with_grad, nonzero, none


components = {
    "vision": model.vision,
    "projector": model.projector,
    "conditioner": model.conditioner,
    "structured_heads": model.structured_heads,
    "mt5": model.mt5,
}

print("\nGradient summary:")

gradient_summary = {}

for name, module in components.items():

    stats = grad_stats(module)

    gradient_summary[name] = stats

    total, with_grad, nonzero, none = stats

    print(
        f"{name:20s} "
        f"total={total:4d} "
        f"grad={with_grad:4d} "
        f"nonzero={nonzero:4d} "
        f"none={none:4d}"
    )

    assert total > 0


# ================================================================
# 9. STRUCTURED HEADS MUST HAVE AUXILIARY GRADIENTS
# ================================================================

print("\n[8] Structured-head gradient audit...")

for name, p in model.structured_heads.named_parameters():

    assert p.grad is not None

    assert torch.isfinite(
        p.grad
    ).all()

    print(
        f"✓ {name}"
    )


# ================================================================
# 10. CRITICAL HARD-CONDITIONING CHECK
# ================================================================

print(
    "\n[9] Hard structured conditioning check..."
)

print(
    "The structured predictions are thresholded "
    "at 0.5 before entering the conditioner."
)

print(
    "Therefore report-loss gradients do not "
    "backpropagate through the hard threshold."
)

print(
    "Structured heads are still trained through "
    "their auxiliary BCE losses."
)

print(
    "✓ Architecture preserved exactly"
)


# ================================================================
# 11. ENSURE NO STEP HAPPENED
# ================================================================

print("\n[10] Step safety check...")

assert scheduler.last_epoch == 768
assert scheduler._step_count == 769

print(
    "scheduler.last_epoch =",
    scheduler.last_epoch
)

print(
    "scheduler._step_count =",
    scheduler._step_count
)

print(
    "✓ optimizer.step() NOT called"
)

print(
    "✓ scheduler.step() NOT called"
)

print(
    "✓ checkpoint NOT saved"
)


# ================================================================
# 12. FINAL
# ================================================================

print("\n" + "=" * 70)
print("FINAL V4 MODEL AUDIT")
print("=" * 70)

print("✓ 409 / 409 checkpoint keys")
print("✓ All tensor shapes match")
print("✓ Strict checkpoint load")
print("✓ Optimizer restored")
print("✓ Scheduler restored at step 768")
print("✓ Forward PASS")
print("✓ Output shapes PASS")
print("✓ Numerical sanity PASS")
print("✓ Backward PASS")
print("✓ Gradient sanity PASS")
print("✓ No optimizer step")
print("✓ No scheduler step")
print("✓ No checkpoint modification")
print("=" * 70)

print("SAFE TO START EPOCH 3")
print("=" * 70)

V4 FINAL FORWARD / GRADIENT AUDIT

[1] Restoring optimizer...
group 0: 6.54508497187473e-06
group 1: 6.54508497187472e-05
group 2: 6.54508497187472e-05
group 3: 6.54508497187472e-05
group 4: 3.27254248593736e-05
✓ Optimizer state PASS

[2] Restoring scheduler...
T_max: 1920
last_epoch: 768
_step_count: 769
✓ Scheduler state PASS

[3] Preparing one batch...
Cases:
  10000.10000.0.10014
  10001.10001.0.10015
  10002.10002.0.10016
  10004.10004.0.10018
pixel_values: (4, 8, 3, 224, 224)
image_mask: (4, 8)
labels: (4, 96)
concept: (4, 48)
negative: (4, 5)
attribute: (4, 6)

[4] Forward pass...
Total loss: 7.325815200805664
Report loss: 7.217798233032227
Concept loss: 0.04089515656232834
Negative loss: 0.026325834915041924
Attribute loss: 0.2976248264312744

[5] Output shape audit...
✓ concept_logits      (4, 48)
✓ negative_logits     (4, 5)
✓ attribute_logits    (4, 6)
✓ visual_tokens       (4, 8, 512)
✓ structured_tokens   (4, 59, 512)
✓ prefix              (4, 67, 512)

[6] Numerical chec

In [ ]:
# ================================================================
# V4 — EPOCH 3 TRAINING
#
# RESUME FROM EPOCH 2
# NO TEST SET
# CHECKPOINT SAFE
# ================================================================

import os
import time
import math
import json
import random
import numpy as np
import torch
from torch.utils.data import DataLoader

print("=" * 70)
print("V4 — EPOCH 3 TRAINING")
print("=" * 70)

# ================================================================
# 0. SAFETY
# ================================================================

assert checkpoint["epoch"] == 2

assert scheduler.last_epoch == 768
assert scheduler._step_count == 769

assert len(optimizer.param_groups) == 5

print("✓ Resuming from Epoch 2")
print("✓ Scheduler position: 768")
print("✓ No test set will be used")


# ================================================================
# 1. DATA LOADERS
# ================================================================

print("\n[1] Building train / val loaders...")

train_ds = V4DatasetAudit(
    case_df,
    "train",
)

val_ds = V4DatasetAudit(
    case_df,
    "val",
)

train_loader = DataLoader(
    train_ds,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print(
    "Train cases:",
    len(train_ds)
)

print(
    "Val cases:",
    len(val_ds)
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Val batches:",
    len(val_loader)
)

assert len(train_ds) == 6138
assert len(val_ds) == 756

# 1535 batches / 4 = 383.75
# Therefore 384 optimizer steps per epoch.
expected_optimizer_steps = 384

print(
    "Expected optimizer steps:",
    expected_optimizer_steps
)


# ================================================================
# 2. RESUME RNG STATE
# ================================================================

print("\n[2] Restoring RNG state...")

if "rng_state" in checkpoint:
    torch.set_rng_state(
        checkpoint["rng_state"]
    )

if (
    torch.cuda.is_available()
    and checkpoint.get("cuda_rng_state") is not None
):
    torch.cuda.set_rng_state_all(
        checkpoint["cuda_rng_state"]
    )

print("✓ RNG state restored")


# ================================================================
# 3. TRAINING SETTINGS
# ================================================================

EPOCH = 3
GRAD_ACCUM = 4
MAX_NORM = 1.0

MODEL_DIR = (
    "/content/drive/MyDrive/"
    "NoiSoi_Matching/"
    "baseline_model_v4"
)

CKPT_DIR = os.path.join(
    MODEL_DIR,
    "checkpoints",
)

os.makedirs(
    CKPT_DIR,
    exist_ok=True,
)

epoch_ckpt_path = os.path.join(
    CKPT_DIR,
    "epoch_03.pt",
)

best_ckpt_path = os.path.join(
    CKPT_DIR,
    "best.pt",
)

last_ckpt_path = os.path.join(
    CKPT_DIR,
    "last.pt",
)

print(
    "\nEpoch:",
    EPOCH
)

print(
    "Gradient accumulation:",
    GRAD_ACCUM
)

print(
    "Effective batch size:",
    4 * GRAD_ACCUM
)


# ================================================================
# 4. TRAIN
# ================================================================

print("\n" + "=" * 70)
print("TRAINING")
print("=" * 70)

model.train()

optimizer.zero_grad(
    set_to_none=True
)

train_start = time.time()

running = {
    "loss": 0.0,
    "report": 0.0,
    "concept": 0.0,
    "negative": 0.0,
    "attribute": 0.0,
}

num_batches = len(train_loader)

for batch_idx, batch in enumerate(
    train_loader,
    start=1,
):

    pixel_values = batch[
        "pixel_values"
    ].to(
        DEVICE,
        non_blocking=True,
    )

    image_mask = batch[
        "image_mask"
    ].to(
        DEVICE,
        non_blocking=True,
    )

    labels = batch[
        "labels"
    ].to(
        DEVICE,
        non_blocking=True,
    )

    concept_targets = batch[
        "concept_targets"
    ].to(
        DEVICE,
        non_blocking=True,
    )

    negative_targets = batch[
        "negative_targets"
    ].to(
        DEVICE,
        non_blocking=True,
    )

    attribute_targets = batch[
        "attribute_targets"
    ].to(
        DEVICE,
        non_blocking=True,
    )

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
        enabled=(
            torch.cuda.is_available()
            and torch.cuda.is_bf16_supported()
        ),
    ):

        outputs = model(
            pixel_values=pixel_values,
            image_mask=image_mask,
            labels=labels,
            concept_targets=concept_targets,
            negative_targets=negative_targets,
            attribute_targets=attribute_targets,
        )

        loss = (
            outputs["loss"]
            / GRAD_ACCUM
        )

    loss.backward()

    running["loss"] += (
        outputs["loss"]
        .detach()
        .float()
        .item()
    )

    running["report"] += (
        outputs["report_loss"]
        .detach()
        .float()
        .item()
    )

    running["concept"] += (
        outputs["concept_loss"]
        .detach()
        .float()
        .item()
    )

    running["negative"] += (
        outputs["negative_loss"]
        .detach()
        .float()
        .item()
    )

    running["attribute"] += (
        outputs["attribute_loss"]
        .detach()
        .float()
        .item()
    )

    is_accum_boundary = (
        batch_idx % GRAD_ACCUM == 0
    )

    is_last_batch = (
        batch_idx == num_batches
    )

    if (
        is_accum_boundary
        or is_last_batch
    ):

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_NORM,
        )

        optimizer.step()

        optimizer.zero_grad(
            set_to_none=True
        )

        scheduler.step()

    if (
        batch_idx == 1
        or batch_idx % 100 == 0
        or batch_idx == num_batches
    ):

        elapsed = (
            time.time()
            - train_start
        )

        print(
            f"[Train] "
            f"{batch_idx:4d}/{num_batches} "
            f"loss={outputs['loss'].item():.5f} "
            f"report={outputs['report_loss'].item():.5f} "
            f"elapsed={elapsed/60:.1f}m"
        )


# ================================================================
# 5. TRAIN METRICS
# ================================================================

train_metrics = {
    k: v / num_batches
    for k, v in running.items()
}

print("\n" + "=" * 70)
print("TRAIN COMPLETE")
print("=" * 70)

for k, v in train_metrics.items():
    print(
        f"{k:12s}: {v:.6f}"
    )

print(
    "Scheduler last_epoch:",
    scheduler.last_epoch
)

print(
    "Scheduler _step_count:",
    scheduler._step_count
)

# Epoch 3 adds exactly 384 optimizer steps
assert scheduler.last_epoch == 1152
assert scheduler._step_count == 1153


# ================================================================
# 6. VALIDATION
# ================================================================

print("\n" + "=" * 70)
print("VALIDATION")
print("=" * 70)

model.eval()

val_running = {
    "loss": 0.0,
    "report": 0.0,
    "concept": 0.0,
    "negative": 0.0,
    "attribute": 0.0,
}

with torch.no_grad():

    for batch_idx, batch in enumerate(
        val_loader,
        start=1,
    ):

        pixel_values = batch[
            "pixel_values"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        image_mask = batch[
            "image_mask"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        labels = batch[
            "labels"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        concept_targets = batch[
            "concept_targets"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        negative_targets = batch[
            "negative_targets"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        attribute_targets = batch[
            "attribute_targets"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=(
                torch.cuda.is_available()
                and torch.cuda.is_bf16_supported()
            ),
        ):

            outputs = model(
                pixel_values=pixel_values,
                image_mask=image_mask,
                labels=labels,
                concept_targets=concept_targets,
                negative_targets=negative_targets,
                attribute_targets=attribute_targets,
            )

        val_running["loss"] += (
            outputs["loss"]
            .float()
            .item()
        )

        val_running["report"] += (
            outputs["report_loss"]
            .float()
            .item()
        )

        val_running["concept"] += (
            outputs["concept_loss"]
            .float()
            .item()
        )

        val_running["negative"] += (
            outputs["negative_loss"]
            .float()
            .item()
        )

        val_running["attribute"] += (
            outputs["attribute_loss"]
            .float()
            .item()
        )

val_metrics = {
    k: v / len(val_loader)
    for k, v in val_running.items()
}

print("\nValidation metrics:")

for k, v in val_metrics.items():
    print(
        f"{k:12s}: {v:.6f}"
    )


# ================================================================
# 7. CHECK BEST
# ================================================================

previous_best = float(
    checkpoint["best_val"]
)

current_val = float(
    val_metrics["loss"]
)

is_best = (
    current_val < previous_best
)

print("\n" + "=" * 70)
print("CHECKPOINT DECISION")
print("=" * 70)

print(
    "Previous best:",
    f"{previous_best:.6f}"
)

print(
    "Epoch 3 val:",
    f"{current_val:.6f}"
)

print(
    "Improved:",
    is_best
)


# ================================================================
# 8. HISTORY
# ================================================================

history = checkpoint.get(
    "history",
    [],
)

history = list(history)

history.append({
    "epoch": EPOCH,
    "train_loss":
        train_metrics["loss"],
    "train_report":
        train_metrics["report"],
    "train_concept":
        train_metrics["concept"],
    "train_negative":
        train_metrics["negative"],
    "train_attribute":
        train_metrics["attribute"],
    "val_loss":
        val_metrics["loss"],
    "val_report":
        val_metrics["report"],
    "val_concept":
        val_metrics["concept"],
    "val_negative":
        val_metrics["negative"],
    "val_attribute":
        val_metrics["attribute"],
    "lr_vision":
        optimizer.param_groups[0]["lr"],
    "lr_projector":
        optimizer.param_groups[1]["lr"],
    "lr_conditioner":
        optimizer.param_groups[2]["lr"],
    "lr_structured":
        optimizer.param_groups[3]["lr"],
    "lr_mt5":
        optimizer.param_groups[4]["lr"],
})


# ================================================================
# 9. SAVE EPOCH 3
# ================================================================

epoch3_checkpoint = {
    "epoch": EPOCH,

    "model_state":
        model.state_dict(),

    "optimizer_state":
        optimizer.state_dict(),

    "scheduler_state":
        scheduler.state_dict(),

    "best_val":
        min(
            previous_best,
            current_val,
        ),

    "rng_state":
        torch.get_rng_state(),

    "cuda_rng_state":
        (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),

    "history":
        history,
}

torch.save(
    epoch3_checkpoint,
    epoch_ckpt_path,
)

print(
    "\n✓ Saved:",
    epoch_ckpt_path
)


# ================================================================
# 10. SAVE BEST ONLY IF IMPROVED
# ================================================================

if is_best:

    torch.save(
        epoch3_checkpoint,
        best_ckpt_path,
    )

    print(
        "✓ New BEST checkpoint saved:",
        best_ckpt_path
    )

else:

    print(
        "No new best checkpoint."
    )


# ================================================================
# 11. UPDATE LAST ONLY AFTER EVERYTHING ABOVE SUCCEEDS
# ================================================================

torch.save(
    epoch3_checkpoint,
    last_ckpt_path,
)

print(
    "✓ LAST checkpoint updated:",
    last_ckpt_path
)


# ================================================================
# 12. FINAL EPOCH 3 SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("EPOCH 3 COMPLETE")
print("=" * 70)

print(
    f"Train loss: {train_metrics['loss']:.6f}"
)

print(
    f"Val loss:   {val_metrics['loss']:.6f}"
)

print(
    f"Best val:   "
    f"{min(previous_best, current_val):.6f}"
)

print(
    "Scheduler:",
    scheduler.last_epoch
)

print(
    "Test set: NOT USED"
)

print("=" * 70)

V4 — EPOCH 3 TRAINING
✓ Resuming from Epoch 2
✓ Scheduler position: 768
✓ No test set will be used

[1] Building train / val loaders...
Train cases: 6138
Val cases: 756
Train batches: 1535
Val batches: 189
Expected optimizer steps: 384

[2] Restoring RNG state...
✓ RNG state restored

Epoch: 3
Gradient accumulation: 4
Effective batch size: 16

TRAINING
[Train]    1/1535 loss=8.29298 report=8.13842 elapsed=0.4m
[Train]  100/1535 loss=0.51845 report=0.42566 elapsed=43.7m
[Train]  200/1535 loss=1.20209 report=1.07356 elapsed=87.3m
[Train]  300/1535 loss=1.15096 report=1.01347 elapsed=130.3m
[Train]  400/1535 loss=0.74613 report=0.58382 elapsed=174.9m
[Train]  500/1535 loss=0.91630 report=0.70799 elapsed=221.0m
[Train]  600/1535 loss=0.53246 report=0.42719 elapsed=265.4m
[Train]  700/1535 loss=0.81228 report=0.65658 elapsed=309.7m
[Train]  800/1535 loss=0.97337 report=0.76821 elapsed=353.7m
[Train]  900/1535 loss=0.65624 report=0.47306 elapsed=398.2m
[Train] 1000/1535 loss=0.73326 report=0

In [30]:
print("model exists:", "model" in globals())
print("optimizer exists:", "optimizer" in globals())
print("scheduler exists:", "scheduler" in globals())

if "scheduler" in globals():
    print("scheduler.last_epoch:", scheduler.last_epoch)
    print("scheduler._step_count:", scheduler._step_count)

print("val_loader exists:", "val_loader" in globals())
print("train_metrics exists:", "train_metrics" in globals())

model exists: True
optimizer exists: True
scheduler exists: True
scheduler.last_epoch: 1152
scheduler._step_count: 1153
val_loader exists: True
train_metrics exists: True


In [31]:
# ================================================================
# V4 — RESUME AFTER VALIDATION INTERRUPTION
# VALIDATE + SAVE EPOCH 3 ONLY
#
# DO NOT TRAIN AGAIN
# DO NOT optimizer.step()
# DO NOT scheduler.step()
# ================================================================

import os
import time
import torch

print("=" * 70)
print("V4 — EPOCH 3 VALIDATION RECOVERY")
print("=" * 70)

# ================================================================
# 0. SAFETY CHECK
# ================================================================

assert "model" in globals()
assert "optimizer" in globals()
assert "scheduler" in globals()
assert "val_loader" in globals()
assert "train_metrics" in globals()

assert scheduler.last_epoch == 1152
assert scheduler._step_count == 1153

print("✓ Model is at end of Epoch 3")
print("✓ Scheduler = 1152")
print("✓ Training metrics preserved")
print("✓ Will NOT train again")


# ================================================================
# 1. VALIDATION
# ================================================================

print("\n" + "=" * 70)
print("VALIDATION")
print("=" * 70)

model.eval()

val_running = {
    "loss": 0.0,
    "report": 0.0,
    "concept": 0.0,
    "negative": 0.0,
    "attribute": 0.0,
}

val_start = time.time()

with torch.no_grad():

    for batch_idx, batch in enumerate(
        val_loader,
        start=1,
    ):

        pixel_values = batch[
            "pixel_values"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        image_mask = batch[
            "image_mask"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        labels = batch[
            "labels"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        concept_targets = batch[
            "concept_targets"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        negative_targets = batch[
            "negative_targets"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        attribute_targets = batch[
            "attribute_targets"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=(
                torch.cuda.is_available()
                and torch.cuda.is_bf16_supported()
            ),
        ):

            outputs = model(
                pixel_values=pixel_values,
                image_mask=image_mask,
                labels=labels,
                concept_targets=concept_targets,
                negative_targets=negative_targets,
                attribute_targets=attribute_targets,
            )

        val_running["loss"] += (
            outputs["loss"]
            .float()
            .item()
        )

        val_running["report"] += (
            outputs["report_loss"]
            .float()
            .item()
        )

        val_running["concept"] += (
            outputs["concept_loss"]
            .float()
            .item()
        )

        val_running["negative"] += (
            outputs["negative_loss"]
            .float()
            .item()
        )

        val_running["attribute"] += (
            outputs["attribute_loss"]
            .float()
            .item()
        )

        if (
            batch_idx == 1
            or batch_idx % 25 == 0
            or batch_idx == len(val_loader)
        ):

            elapsed = (
                time.time()
                - val_start
            )

            print(
                f"[Val] "
                f"{batch_idx:3d}/{len(val_loader)} "
                f"loss={outputs['loss'].item():.5f} "
                f"elapsed={elapsed/60:.1f}m"
            )


# ================================================================
# 2. VALIDATION METRICS
# ================================================================

val_metrics = {
    k: v / len(val_loader)
    for k, v in val_running.items()
}

print("\n" + "=" * 70)
print("VALIDATION COMPLETE")
print("=" * 70)

for k, v in val_metrics.items():
    print(
        f"{k:12s}: {v:.6f}"
    )


# ================================================================
# 3. VERIFY TRAIN METRICS STILL EXIST
# ================================================================

print("\nTrain metrics preserved:")

for k, v in train_metrics.items():
    print(
        f"{k:12s}: {v:.6f}"
    )


# ================================================================
# 4. COMPARE WITH EPOCH 2
# ================================================================

previous_best = float(
    checkpoint["best_val"]
)

current_val = float(
    val_metrics["loss"]
)

is_best = (
    current_val < previous_best
)

print("\n" + "=" * 70)
print("CHECKPOINT DECISION")
print("=" * 70)

print(
    f"Epoch 2 best: {previous_best:.6f}"
)

print(
    f"Epoch 3 val:  {current_val:.6f}"
)

print(
    "Improved:",
    is_best
)


# ================================================================
# 5. HISTORY
# ================================================================

history = list(
    checkpoint.get(
        "history",
        [],
    )
)

# Remove accidental Epoch 3 entry if this cell
# is ever rerun after partial save.
history = [
    h for h in history
    if h.get("epoch") != 3
]

history.append({
    "epoch": 3,

    "train_loss":
        train_metrics["loss"],

    "train_report":
        train_metrics["report"],

    "train_concept":
        train_metrics["concept"],

    "train_negative":
        train_metrics["negative"],

    "train_attribute":
        train_metrics["attribute"],

    "val_loss":
        val_metrics["loss"],

    "val_report":
        val_metrics["report"],

    "val_concept":
        val_metrics["concept"],

    "val_negative":
        val_metrics["negative"],

    "val_attribute":
        val_metrics["attribute"],

    "lr_vision":
        optimizer.param_groups[0]["lr"],

    "lr_projector":
        optimizer.param_groups[1]["lr"],

    "lr_conditioner":
        optimizer.param_groups[2]["lr"],

    "lr_structured":
        optimizer.param_groups[3]["lr"],

    "lr_mt5":
        optimizer.param_groups[4]["lr"],
})


# ================================================================
# 6. BUILD EPOCH 3 CHECKPOINT
# ================================================================

epoch3_checkpoint = {
    "epoch": 3,

    "model_state":
        model.state_dict(),

    "optimizer_state":
        optimizer.state_dict(),

    "scheduler_state":
        scheduler.state_dict(),

    "best_val":
        min(
            previous_best,
            current_val,
        ),

    "rng_state":
        torch.get_rng_state(),

    "cuda_rng_state":
        (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),

    "history":
        history,
}


# ================================================================
# 7. SAVE EPOCH 03 FIRST
# ================================================================

MODEL_DIR = (
    "/content/drive/MyDrive/"
    "NoiSoi_Matching/"
    "baseline_model_v4"
)

CKPT_DIR = os.path.join(
    MODEL_DIR,
    "checkpoints",
)

os.makedirs(
    CKPT_DIR,
    exist_ok=True,
)

epoch_ckpt_path = os.path.join(
    CKPT_DIR,
    "epoch_03.pt",
)

best_ckpt_path = os.path.join(
    CKPT_DIR,
    "best.pt",
)

last_ckpt_path = os.path.join(
    CKPT_DIR,
    "last.pt",
)

torch.save(
    epoch3_checkpoint,
    epoch_ckpt_path,
)

print(
    "\n✓ epoch_03.pt SAVED"
)


# ================================================================
# 8. SAVE BEST IF IMPROVED
# ================================================================

if is_best:

    torch.save(
        epoch3_checkpoint,
        best_ckpt_path,
    )

    print(
        "✓ best.pt UPDATED"
    )

else:

    print(
        "✓ best.pt preserved "
        "(Epoch 2 remains best)"
    )


# ================================================================
# 9. UPDATE LAST
# ================================================================

torch.save(
    epoch3_checkpoint,
    last_ckpt_path,
)

print(
    "✓ last.pt UPDATED"
)


# ================================================================
# 10. FINAL SAFETY CHECK
# ================================================================

print("\n" + "=" * 70)
print("EPOCH 3 RECOVERY COMPLETE")
print("=" * 70)

print(
    f"Train loss: {train_metrics['loss']:.6f}"
)

print(
    f"Val loss:   {val_metrics['loss']:.6f}"
)

print(
    f"Best val:   "
    f"{min(previous_best, current_val):.6f}"
)

print(
    "Scheduler:",
    scheduler.last_epoch
)

assert scheduler.last_epoch == 1152
assert scheduler._step_count == 1153

print(
    "Test set: NOT USED"
)

print(
    "Training again: NO"
)

print("=" * 70)

V4 — EPOCH 3 VALIDATION RECOVERY
✓ Model is at end of Epoch 3
✓ Scheduler = 1152
✓ Training metrics preserved
✓ Will NOT train again

VALIDATION
[Val]   1/189 loss=0.27960 elapsed=0.0m
[Val]  25/189 loss=0.74193 elapsed=0.1m
[Val]  50/189 loss=0.33259 elapsed=0.3m
[Val]  75/189 loss=1.16469 elapsed=0.4m
[Val] 100/189 loss=0.43994 elapsed=0.6m
[Val] 125/189 loss=0.62289 elapsed=0.7m
[Val] 150/189 loss=0.43437 elapsed=0.8m
[Val] 175/189 loss=0.85351 elapsed=1.0m
[Val] 189/189 loss=0.57169 elapsed=1.1m

VALIDATION COMPLETE
loss        : 0.689219
report      : 0.526985
concept     : 0.063280
negative    : 0.055134
attribute   : 0.412110

Train metrics preserved:
loss        : 0.909012
report      : 0.755409
concept     : 0.059978
negative    : 0.062720
attribute   : 0.369018

CHECKPOINT DECISION
Epoch 2 best: 0.652164
Epoch 3 val:  0.689219
Improved: False

✓ epoch_03.pt SAVED
✓ best.pt preserved (Epoch 2 remains best)
✓ last.pt UPDATED

EPOCH 3 RECOVERY COMPLETE
Train loss: 0.909012
Val l

In [33]:
# ================================================================
# V4 — VALIDATION GENERATION EVALUATION
# Compare:
#   1. best.pt       = Epoch 2
#   2. epoch_03.pt   = Epoch 3
#
# NO TRAINING
# NO OPTIMIZER
# NO SCHEDULER
# NO TEST SET
# ================================================================

import os
import re
import math
import json
import numpy as np
import pandas as pd
import torch
from collections import Counter

print("=" * 70)
print("V4 — VALIDATION GENERATION EVALUATION")
print("=" * 70)

# ================================================================
# 0. PATHS
# ================================================================

CKPT_DIR = (
    "/content/drive/MyDrive/"
    "NoiSoi_Matching/"
    "baseline_model_v4/"
    "checkpoints"
)

BEST_CKPT = os.path.join(
    CKPT_DIR,
    "best.pt"
)

EPOCH3_CKPT = os.path.join(
    CKPT_DIR,
    "epoch_03.pt"
)

assert os.path.exists(BEST_CKPT)
assert os.path.exists(EPOCH3_CKPT)

print("Best checkpoint:")
print(BEST_CKPT)

print("\nEpoch 3 checkpoint:")
print(EPOCH3_CKPT)


# ================================================================
# 1. TOKENIZER
# ================================================================

print("\n[1] Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    "google/mt5-small"
)

print(
    "Tokenizer:",
    type(tokenizer).__name__
)

print(
    "Vocab size:",
    tokenizer.vocab_size
)


# ================================================================
# 2. VALIDATION LOADER
# ================================================================

print("\n[2] Validation loader...")

if "val_loader" not in globals():

    val_ds = V4DatasetAudit(
        case_df,
        "val",
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=4,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

print(
    "Validation cases:",
    len(val_ds) if "val_ds" in globals()
    else len(val_loader.dataset)
)

assert len(val_loader.dataset) == 756


# ================================================================
# 3. GENERATION FUNCTION
# ================================================================

def generate_v4(
    model,
    pixel_values,
    image_mask,
    concept_targets,
    negative_targets,
    attribute_targets,
    max_new_tokens=96,
    num_beams=4,
):

    B, N, C, H, W = pixel_values.shape

    # ------------------------------------------------
    # Vision
    # ------------------------------------------------

    x = pixel_values.reshape(
        B * N,
        C,
        H,
        W,
    )

    vision_out = model.vision(
        pixel_values=x
    ).last_hidden_state[:, 0]

    vision_out = vision_out.reshape(
        B,
        N,
        768,
    )

    visual_tokens = model.projector(
        vision_out
    )

    visual_tokens = (
        visual_tokens
        * image_mask[:, :, None]
    )

    # ------------------------------------------------
    # Pool
    # ------------------------------------------------

    denom = (
        image_mask.sum(
            dim=1,
            keepdim=True,
        )
        .clamp(min=1)
    )

    pooled = (
        visual_tokens.sum(dim=1)
        / denom
    )

    # ------------------------------------------------
    # Structured predictions
    # ------------------------------------------------

    concept_logits = model.structured_heads[
        "concept"
    ](pooled)

    negative_logits = model.structured_heads[
        "negative"
    ](pooled)

    attribute_logits = model.structured_heads[
        "attribute"
    ](pooled)

    concept_pred = (
        torch.sigmoid(
            concept_logits
        ) >= 0.5
    ).float()

    negative_pred = (
        torch.sigmoid(
            negative_logits
        ) >= 0.5
    ).float()

    attribute_pred = (
        torch.sigmoid(
            attribute_logits
        ) >= 0.5
    ).float()

    # ------------------------------------------------
    # Structured conditioner
    # ------------------------------------------------

    structured_tokens, structured_attention = (
        model.conditioner(
            concept_pred,
            negative_pred,
            attribute_pred,
        )
    )

    # ------------------------------------------------
    # Prefix
    # ------------------------------------------------

    visual_attention = image_mask.long()

    prefix = torch.cat(
        [
            visual_tokens,
            structured_tokens,
        ],
        dim=1,
    )

    attention_mask = torch.cat(
        [
            visual_attention,
            structured_attention,
        ],
        dim=1,
    )

    # ------------------------------------------------
    # mT5 encoder embeddings
    # ------------------------------------------------

    prefix_len = prefix.shape[1]

    zero_ids = torch.zeros(
        B,
        prefix_len,
        dtype=torch.long,
        device=prefix.device,
    )

    base_embeddings = (
        model.mt5.encoder.embed_tokens(
            zero_ids
        )
    )

    inputs_embeds = (
        base_embeddings
        + prefix
    )

    # ------------------------------------------------
    # Generate
    # ------------------------------------------------

    generated = model.mt5.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        num_beams=num_beams,
        do_sample=False,
        early_stopping=True,
    )

    return generated


# ================================================================
# 4. TEXT CLEANING
# ================================================================

def clean_text(text):

    text = str(text)

    text = text.replace(
        "<pad>",
        ""
    )

    text = text.replace(
        "</s>",
        ""
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()


def decode_labels(
    labels,
):

    labels = labels.detach().cpu().clone()

    labels[
        labels == -100
    ] = tokenizer.pad_token_id

    texts = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True,
    )

    return [
        clean_text(x)
        for x in texts
    ]


def decode_predictions(
    generated,
):

    texts = tokenizer.batch_decode(
        generated,
        skip_special_tokens=True,
    )

    return [
        clean_text(x)
        for x in texts
    ]


# ================================================================
# 5. METRICS
# ================================================================

def normalize_text(s):

    s = clean_text(s).upper()

    return s


def exact_match(
    predictions,
    references,
):

    return np.mean([
        normalize_text(p)
        == normalize_text(r)
        for p, r in zip(
            predictions,
            references,
        )
    ])


def char_similarity(
    prediction,
    reference,
):

    p = normalize_text(prediction)
    r = normalize_text(reference)

    if len(p) == 0 and len(r) == 0:
        return 1.0

    if len(p) == 0 or len(r) == 0:
        return 0.0

    # Levenshtein distance
    prev = list(range(len(r) + 1))

    for i, pc in enumerate(p, start=1):

        curr = [i]

        for j, rc in enumerate(r, start=1):

            if pc == rc:
                cost = 0
            else:
                cost = 1

            curr.append(
                min(
                    curr[-1] + 1,
                    prev[j] + 1,
                    prev[j - 1] + cost,
                )
            )

        prev = curr

    distance = prev[-1]

    return 1.0 - (
        distance
        / max(
            len(p),
            len(r),
        )
    )


def mean_char_similarity(
    predictions,
    references,
):

    return np.mean([
        char_similarity(
            p,
            r,
        )
        for p, r in zip(
            predictions,
            references,
        )
    ])


def token_f1_single(
    prediction,
    reference,
):

    p_tokens = normalize_text(
        prediction
    ).split()

    r_tokens = normalize_text(
        reference
    ).split()

    if len(p_tokens) == 0 and len(r_tokens) == 0:
        return 1.0

    if len(p_tokens) == 0 or len(r_tokens) == 0:
        return 0.0

    p_count = Counter(p_tokens)
    r_count = Counter(r_tokens)

    overlap = sum(
        (
            p_count & r_count
        ).values()
    )

    if overlap == 0:
        return 0.0

    precision = (
        overlap
        / len(p_tokens)
    )

    recall = (
        overlap
        / len(r_tokens)
    )

    return (
        2 * precision * recall
        / (precision + recall)
    )


def mean_token_f1(
    predictions,
    references,
):

    return np.mean([
        token_f1_single(
            p,
            r,
        )
        for p, r in zip(
            predictions,
            references,
        )
    ])


# ================================================================
# 6. BLEU
# ================================================================

try:

    from nltk.translate.bleu_score import (
        sentence_bleu,
        SmoothingFunction,
    )

    bleu_smoother = (
        SmoothingFunction()
        .method1
    )

    def bleu_single(
        prediction,
        reference,
    ):

        p = normalize_text(
            prediction
        ).split()

        r = normalize_text(
            reference
        ).split()

        if len(p) == 0:
            return 0.0

        if len(r) == 0:
            return 0.0

        return sentence_bleu(
            [r],
            p,
            weights=(
                0.25,
                0.25,
                0.25,
                0.25,
            ),
            smoothing_function=(
                bleu_smoother
            ),
        )

    def mean_bleu(
        predictions,
        references,
    ):

        return np.mean([
            bleu_single(
                p,
                r,
            )
            for p, r in zip(
                predictions,
                references,
            )
        ])

except Exception:

    def mean_bleu(
        predictions,
        references,
    ):

        return float("nan")


# ================================================================
# 7. GENERATION EVALUATION
# ================================================================

@torch.no_grad()
def evaluate_checkpoint(
    checkpoint_path,
    checkpoint_name,
):

    print("\n" + "=" * 70)
    print(
        f"EVALUATING: {checkpoint_name}"
    )
    print("=" * 70)

    ckpt = torch.load(
        checkpoint_path,
        map_location="cpu",
    )

    print(
        "Checkpoint epoch:",
        ckpt["epoch"]
    )

    # ------------------------------------------------
    # Load model weights only
    # ------------------------------------------------

    model.load_state_dict(
        ckpt["model_state"],
        strict=True,
    )

    model.to(DEVICE)
    model.eval()

    predictions = []
    references = []
    case_ids = []

    start = time.time()

    for batch_idx, batch in enumerate(
        val_loader,
        start=1,
    ):

        pixel_values = batch[
            "pixel_values"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        image_mask = batch[
            "image_mask"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        labels = batch[
            "labels"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        concept_targets = batch[
            "concept_targets"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        negative_targets = batch[
            "negative_targets"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        attribute_targets = batch[
            "attribute_targets"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=(
                torch.cuda.is_available()
                and torch.cuda.is_bf16_supported()
            ),
        ):

            generated = generate_v4(
                model=model,
                pixel_values=pixel_values,
                image_mask=image_mask,
                concept_targets=concept_targets,
                negative_targets=negative_targets,
                attribute_targets=attribute_targets,
                max_new_tokens=96,
                num_beams=4,
            )

        pred_texts = decode_predictions(
            generated
        )

        ref_texts = decode_labels(
            labels
        )

        predictions.extend(
            pred_texts
        )

        references.extend(
            ref_texts
        )

        case_ids.extend(
            batch["case_id"]
        )

        if (
            batch_idx == 1
            or batch_idx % 25 == 0
            or batch_idx == len(val_loader)
        ):

            elapsed = (
                time.time()
                - start
            )

            print(
                f"[{batch_idx:3d}/"
                f"{len(val_loader)}] "
                f"elapsed={elapsed/60:.1f}m"
            )

    # ------------------------------------------------
    # Metrics
    # ------------------------------------------------

    metrics = {
        "exact_match":
            exact_match(
                predictions,
                references,
            ),

        "char_similarity":
            mean_char_similarity(
                predictions,
                references,
            ),

        "token_f1":
            mean_token_f1(
                predictions,
                references,
            ),

        "bleu":
            mean_bleu(
                predictions,
                references,
            ),

        "unique_predictions":
            len(set(predictions)),
    }

    print("\nMetrics:")

    for k, v in metrics.items():

        if isinstance(v, float):
            print(
                f"{k:22s}: {v:.6f}"
            )
        else:
            print(
                f"{k:22s}: {v}"
            )

    return {
        "checkpoint": checkpoint_name,
        "epoch": ckpt["epoch"],
        "metrics": metrics,
        "predictions": predictions,
        "references": references,
        "case_ids": case_ids,
    }


# ================================================================
# 8. RUN EPOCH 2 BEST
# ================================================================

result_best = evaluate_checkpoint(
    BEST_CKPT,
    "BEST — Epoch 2",
)


# ================================================================
# 9. RUN EPOCH 3
# ================================================================

result_epoch3 = evaluate_checkpoint(
    EPOCH3_CKPT,
    "Epoch 3",
)


# ================================================================
# 10. COMPARISON TABLE
# ================================================================

comparison = pd.DataFrame([
    {
        "checkpoint":
            "Epoch 2 best",

        "epoch":
            result_best["epoch"],

        **result_best["metrics"],
    },

    {
        "checkpoint":
            "Epoch 3",

        "epoch":
            result_epoch3["epoch"],

        **result_epoch3["metrics"],
    },
])

print("\n" + "=" * 70)
print("GENERATION COMPARISON")
print("=" * 70)

display(comparison)


# ================================================================
# 11. EXAMPLE PREDICTIONS
# ================================================================

print("\n" + "=" * 70)
print("EXAMPLE PREDICTIONS")
print("=" * 70)

for i in range(
    min(
        20,
        len(result_best["predictions"]),
    )
):

    print("\n" + "-" * 70)

    print(
        "CASE:",
        result_best["case_ids"][i]
    )

    print(
        "REF:",
        result_best["references"][i]
    )

    print(
        "E2 :",
        result_best["predictions"][i]
    )

    print(
        "E3 :",
        result_epoch3["predictions"][i]
    )


# ================================================================
# 12. COLLAPSE ANALYSIS
# ================================================================

print("\n" + "=" * 70)
print("PREDICTION COLLAPSE ANALYSIS")
print("=" * 70)

for name, result in [
    (
        "Epoch 2",
        result_best,
    ),
    (
        "Epoch 3",
        result_epoch3,
    ),
]:

    counts = Counter(
        result["predictions"]
    )

    print(
        f"\n{name}"
    )

    print(
        "Unique:",
        len(counts)
    )

    print(
        "Top predictions:"
    )

    for text, count in (
        counts.most_common(10)
    ):

        print(
            f"  {count:4d} × {text}"
        )


# ================================================================
# 13. SAVE EVALUATION RESULTS
# ================================================================

eval_dir = os.path.join(
    MODEL_DIR,
    "evaluation",
)

os.makedirs(
    eval_dir,
    exist_ok=True,
)

comparison_path = os.path.join(
    eval_dir,
    "v4_val_generation_comparison_epoch2_epoch3.csv",
)

comparison.to_csv(
    comparison_path,
    index=False,
)

pred_df = pd.DataFrame({
    "case_id":
        result_best["case_ids"],

    "reference":
        result_best["references"],

    "epoch2_prediction":
        result_best["predictions"],

    "epoch3_prediction":
        result_epoch3["predictions"],
})

pred_path = os.path.join(
    eval_dir,
    "v4_val_generation_predictions_epoch2_epoch3.csv",
)

pred_df.to_csv(
    pred_path,
    index=False,
)

print("\n" + "=" * 70)
print("SAVED")
print("=" * 70)

print(comparison_path)
print(pred_path)

print("\n✓ Test set was NOT used")
print("✓ No training performed")
print("✓ No checkpoint modified")

V4 — VALIDATION GENERATION EVALUATION
Best checkpoint:
/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/checkpoints/best.pt

Epoch 3 checkpoint:
/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/checkpoints/epoch_03.pt

[1] Loading tokenizer...
Tokenizer: T5Tokenizer
Vocab size: 250100

[2] Validation loader...
Validation cases: 756

EVALUATING: BEST — Epoch 2
Checkpoint epoch: 2
[  1/189] elapsed=0.0m
[ 25/189] elapsed=0.9m
[ 50/189] elapsed=1.9m
[ 75/189] elapsed=2.8m
[100/189] elapsed=3.7m
[125/189] elapsed=4.7m
[150/189] elapsed=5.6m
[175/189] elapsed=6.5m
[189/189] elapsed=7.0m

Metrics:
exact_match           : 0.000000
char_similarity       : 0.010695
token_f1              : 0.000000
bleu                  : 0.000000
unique_predictions    : 9

EVALUATING: Epoch 3
Checkpoint epoch: 3
[  1/189] elapsed=0.0m
[ 25/189] elapsed=0.2m
[ 50/189] elapsed=0.5m
[ 75/189] elapsed=0.7m
[100/189] elapsed=0.9m
[125/189] elapsed=1.1m
[150/189] elapsed=1.4m
[175/189] elapsed=1.6m
[1

,checkpoint,epoch,exact_match,char_similarity,token_f1,bleu,unique_predictions
0,Epoch 2 best,2,0.000000,0.010695,0.000000,0.000000,9
1,Epoch 3,3,0.093915,0.419452,0.444891,0.123492,1



EXAMPLE PREDICTIONS

----------------------------------------------------------------------
CASE: 10003.10003.0.10017
REF: VIÊM MŨI MẠN
E2 : VI- N <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33

In [34]:
# ================================================================
# V4 — CORRECT GENERATION PATH AUDIT
#
# IMPORTANT:
# Use EXACT SAME encoder path as training:
#
# inputs_embeds
#      ↓
# model.mt5.encoder(...)
#      ↓
# encoder_outputs
#      ↓
# model.mt5.generate(encoder_outputs=...)
#
# NO TRAINING
# NO CHECKPOINT MODIFICATION
# ================================================================

import torch
import os

print("=" * 70)
print("V4 — CORRECT GENERATION PATH AUDIT")
print("=" * 70)


# ================================================================
# 1. CORRECT GENERATION FUNCTION
# ================================================================

@torch.no_grad()
def generate_v4_correct(
    model,
    pixel_values,
    image_mask,
    max_new_tokens=96,
    num_beams=4,
):

    B, N, C, H, W = pixel_values.shape

    # ------------------------------------------------
    # Vision
    # ------------------------------------------------

    x = pixel_values.reshape(
        B * N,
        C,
        H,
        W,
    )

    vision_out = model.vision(
        pixel_values=x
    ).last_hidden_state[:, 0]

    vision_out = vision_out.reshape(
        B,
        N,
        768,
    )

    visual_tokens = model.projector(
        vision_out
    )

    visual_tokens = (
        visual_tokens
        * image_mask[:, :, None]
    )

    # ------------------------------------------------
    # Pool
    # ------------------------------------------------

    denom = (
        image_mask.sum(
            dim=1,
            keepdim=True,
        )
        .clamp(min=1)
    )

    pooled = (
        visual_tokens.sum(dim=1)
        / denom
    )

    # ------------------------------------------------
    # Structured heads
    # ------------------------------------------------

    concept_logits = model.structured_heads[
        "concept"
    ](pooled)

    negative_logits = model.structured_heads[
        "negative"
    ](pooled)

    attribute_logits = model.structured_heads[
        "attribute"
    ](pooled)

    # IMPORTANT:
    # inference uses model predictions,
    # NOT ground truth.

    concept_pred = (
        torch.sigmoid(
            concept_logits
        ) >= 0.5
    ).float()

    negative_pred = (
        torch.sigmoid(
            negative_logits
        ) >= 0.5
    ).float()

    attribute_pred = (
        torch.sigmoid(
            attribute_logits
        ) >= 0.5
    ).float()

    # ------------------------------------------------
    # Structured conditioner
    # ------------------------------------------------

    structured_tokens, structured_attention = (
        model.conditioner(
            concept_pred,
            negative_pred,
            attribute_pred,
        )
    )

    # ------------------------------------------------
    # Prefix
    # ------------------------------------------------

    prefix = torch.cat(
        [
            visual_tokens,
            structured_tokens,
        ],
        dim=1,
    )

    visual_attention = image_mask.long()

    attention_mask = torch.cat(
        [
            visual_attention,
            structured_attention,
        ],
        dim=1,
    )

    # ------------------------------------------------
    # EXACT SAME embedding construction as forward()
    # ------------------------------------------------

    prefix_len = prefix.shape[1]

    zero_ids = torch.zeros(
        B,
        prefix_len,
        dtype=torch.long,
        device=prefix.device,
    )

    base_embeddings = (
        model.mt5.encoder.embed_tokens(
            zero_ids
        )
    )

    inputs_embeds = (
        base_embeddings
        + prefix
    )

    # ------------------------------------------------
    # CRITICAL FIX
    #
    # Explicitly run encoder first.
    # This is identical to training forward().
    # ------------------------------------------------

    encoder_outputs = model.mt5.encoder(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask,
        return_dict=True,
    )

    # ------------------------------------------------
    # Generate FROM encoder_outputs
    # ------------------------------------------------

    generated = model.mt5.generate(
        encoder_outputs=encoder_outputs,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        num_beams=num_beams,
        do_sample=False,
        early_stopping=True,
    )

    return generated


# ================================================================
# 2. EVALUATE ONLY 20 CASES
# ================================================================

def quick_generation_test(
    checkpoint_path,
    checkpoint_name,
    num_cases=20,
):

    print("\n" + "=" * 70)
    print(checkpoint_name)
    print("=" * 70)

    ckpt = torch.load(
        checkpoint_path,
        map_location="cpu",
    )

    model.load_state_dict(
        ckpt["model_state"],
        strict=True,
    )

    model.to(DEVICE)
    model.eval()

    predictions = []
    references = []

    with torch.no_grad():

        for batch in val_loader:

            pixel_values = batch[
                "pixel_values"
            ].to(
                DEVICE,
                non_blocking=True,
            )

            image_mask = batch[
                "image_mask"
            ].to(
                DEVICE,
                non_blocking=True,
            )

            labels = batch[
                "labels"
            ].to(
                DEVICE,
                non_blocking=True,
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
                enabled=(
                    torch.cuda.is_available()
                    and torch.cuda.is_bf16_supported()
                ),
            ):

                generated = (
                    generate_v4_correct(
                        model,
                        pixel_values,
                        image_mask,
                        max_new_tokens=96,
                        num_beams=4,
                    )
                )

            pred = tokenizer.batch_decode(
                generated,
                skip_special_tokens=True,
            )

            labels_cpu = (
                labels.detach()
                .cpu()
                .clone()
            )

            labels_cpu[
                labels_cpu == -100
            ] = tokenizer.pad_token_id

            ref = tokenizer.batch_decode(
                labels_cpu,
                skip_special_tokens=True,
            )

            predictions.extend(
                [clean_text(x) for x in pred]
            )

            references.extend(
                [clean_text(x) for x in ref]
            )

            if len(predictions) >= num_cases:
                break

    predictions = predictions[
        :num_cases
    ]

    references = references[
        :num_cases
    ]

    print(
        "\nUnique predictions:",
        len(set(predictions))
    )

    for i in range(
        min(
            num_cases,
            len(predictions),
        )
    ):

        print("\n" + "-" * 60)

        print(
            "REF:",
            references[i]
        )

        print(
            "PRED:",
            predictions[i]
        )

    return predictions, references


# ================================================================
# 3. EPOCH 2
# ================================================================

e2_pred, e2_ref = quick_generation_test(
    BEST_CKPT,
    "BEST — EPOCH 2",
    num_cases=20,
)


# ================================================================
# 4. EPOCH 3
# ================================================================

e3_pred, e3_ref = quick_generation_test(
    EPOCH3_CKPT,
    "EPOCH 3",
    num_cases=20,
)


# ================================================================
# 5. QUICK METRICS
# ================================================================

print("\n" + "=" * 70)
print("QUICK GENERATION CHECK")
print("=" * 70)

print(
    "Epoch 2 unique:",
    len(set(e2_pred))
)

print(
    "Epoch 3 unique:",
    len(set(e3_pred))
)

print(
    "Epoch 2 exact:",
    exact_match(
        e2_pred,
        e2_ref,
    )
)

print(
    "Epoch 3 exact:",
    exact_match(
        e3_pred,
        e3_ref,
    )
)

print("=" * 70)

V4 — CORRECT GENERATION PATH AUDIT

BEST — EPOCH 2

Unique predictions: 4

------------------------------------------------------------
REF: VIÊM MŨI MẠN
PRED: VI- N <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_id_33> <extra_i

In [38]:
# ============================================================
# V4 — GENERATION COLLAPSE DIAGNOSTIC AUDIT
# EPOCH 2 vs EPOCH 3 — NO TRAINING / NO CHECKPOINT MODIFICATION
# ============================================================

import os
import json
import math
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from collections import Counter

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASE = "/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4"
CKPT_DIR = os.path.join(BASE, "checkpoints")

E2_CKPT = os.path.join(CKPT_DIR, "epoch_02.pt")
E3_CKPT = os.path.join(CKPT_DIR, "epoch_03.pt")

print("=" * 70)
print("V4 GENERATION COLLAPSE DIAGNOSTIC")
print("=" * 70)
print("Device:", DEVICE)
print("E2:", os.path.exists(E2_CKPT), E2_CKPT)
print("E3:", os.path.exists(E3_CKPT), E3_CKPT)


# ============================================================
# 1. CHECK EXISTING OBJECTS
# ============================================================

required = [
    "model",
    "val_loader",
    "tokenizer",
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        f"Missing objects in runtime: {missing}\n"
        "Please run the existing V4 model/data setup cells first."
    )

print("\nExisting model/data objects: PASS")


# ============================================================
# 2. SAVE CURRENT MODEL STATE IN MEMORY
# ============================================================

original_state = {
    k: v.detach().cpu().clone()
    for k, v in model.state_dict().items()
}

print("Current model state snapshot: PASS")


# ============================================================
# 3. HELPERS
# ============================================================

def load_ckpt_for_audit(path):
    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    model.load_state_dict(
        ckpt["model_state"],
        strict=True
    )

    model.to(DEVICE)
    model.eval()

    return ckpt


def restore_original_model():
    model.load_state_dict(
        original_state,
        strict=True
    )
    model.to(DEVICE)
    model.eval()


@torch.no_grad()
def build_encoder_inputs(batch):
    pixel_values = batch["pixel_values"].to(
        DEVICE,
        non_blocking=True
    )

    image_mask = batch["image_mask"].to(
        DEVICE,
        non_blocking=True
    )

    B, N, C, H, W = pixel_values.shape

    flat = pixel_values.view(
        B * N,
        C,
        H,
        W
    )

    vision_outputs = model.vision(
        pixel_values=flat
    )

    pooled = vision_outputs.last_hidden_state[:, 0]

    pooled = pooled.view(
        B,
        N,
        -1
    )

    mask = image_mask.unsqueeze(-1).float()

    visual = (
        pooled * mask
    ).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

    # --------------------------------------------------------
    # V4 CORRECT ARCHITECTURE
    # ViT 768 → Projector 512
    # Structured heads operate on 512
    # --------------------------------------------------------

    visual_512 = model.projector(visual)

    # Visual prefix for mT5
    visual_prefix = visual_512

    # Structured predictions
    concept_logits = model.structured_heads["concept"](
        visual_512
    )

    negative_logits = model.structured_heads["negative"](
        visual_512
    )

    attribute_logits = model.structured_heads["attribute"](
        visual_512
    )

    concept_prob = torch.sigmoid(concept_logits)
    negative_prob = torch.sigmoid(negative_logits)
    attribute_prob = torch.sigmoid(attribute_logits)

    concept_pred = (
        concept_prob >= 0.5
    ).float()

    negative_pred = (
        negative_prob >= 0.5
    ).float()

    attribute_pred = (
        attribute_prob >= 0.5
    ).float()

    structured_tokens, structured_attention = (
        model.conditioner(
            concept_pred,
            negative_pred,
            attribute_pred
        )
    )

    inputs_embeds = torch.cat(
        [
            visual_prefix.unsqueeze(1),
            structured_tokens
        ],
        dim=1
    )

    attention_mask = torch.cat(
        [
            torch.ones(
                B,
                1,
                dtype=torch.long,
                device=DEVICE
            ),
            structured_attention
        ],
        dim=1
    )

    encoder_outputs = model.mt5.encoder(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask,
        return_dict=True
    )

    return {
        "encoder_outputs": encoder_outputs,
        "attention_mask": attention_mask,

        "concept_logits": concept_logits,
        "negative_logits": negative_logits,
        "attribute_logits": attribute_logits,

        "concept_prob": concept_prob,
        "negative_prob": negative_prob,
        "attribute_prob": attribute_prob,

        "concept_pred": concept_pred,
        "negative_pred": negative_pred,
        "attribute_pred": attribute_pred,

        "inputs_embeds": inputs_embeds,
    }


# ============================================================
# 4. GET EXACT 20 VALIDATION CASES
# ============================================================

batch0 = next(iter(val_loader))

print("\nValidation batch:")
for k, v in batch0.items():
    if torch.is_tensor(v):
        print(
            f"  {k:20s}",
            tuple(v.shape)
        )

# Use first 20 samples from the first validation batch.
# If batch size is 4, collect batches until 20.
audit_batches = []

count = 0

for b in val_loader:
    audit_batches.append(b)

    if "labels" in b:
        count += b["labels"].shape[0]

    if count >= 20:
        break

print("\nAudit cases collected:", count)


# ============================================================
# 5. TOKENIZATION HELPERS
# ============================================================

def decode_ids(ids):
    ids = ids.detach().cpu().tolist()

    return tokenizer.decode(
        ids,
        skip_special_tokens=False
    )


def clean_text(ids):
    ids = ids.detach().cpu().tolist()

    # T5 labels dùng -100 để mask
    ids = [
        tokenizer.pad_token_id if x == -100 else x
        for x in ids
    ]

    return tokenizer.decode(
        ids,
        skip_special_tokens=True
    ).strip()


def first_nonpad_target(labels):
    labels = labels.detach().cpu()

    pad_id = tokenizer.pad_token_id

    for x in labels.tolist():
        for token in x:
            if token != -100 and token != pad_id:
                return token

    return None


# ============================================================
# 6. TEACHER-FORCED DIAGNOSTIC
# ============================================================

@torch.no_grad()
def teacher_forced_diagnostic(batch):
    enc = build_encoder_inputs(batch)

    labels = batch["labels"].to(
        DEVICE,
        non_blocking=True
    )

    out = model.mt5(
        encoder_outputs=enc["encoder_outputs"],
        attention_mask=enc["attention_mask"],
        labels=labels,
        return_dict=True
    )

    logits = out.logits

    # logits[t] predicts labels[t]
    pred_ids = logits.argmax(dim=-1)

    valid = labels != -100

    correct = (
        (pred_ids == labels) &
        valid
    )

    token_acc = (
        correct.sum().float()
        /
        valid.sum().float().clamp(min=1)
    )

    return {
        **enc,
        "labels": labels,
        "logits": logits,
        "pred_ids": pred_ids,
        "token_acc": token_acc.item(),
    }


# ============================================================
# 7. AUTOREGRESSIVE STEP-BY-STEP DIAGNOSTIC
# ============================================================

@torch.no_grad()
def autoregressive_diagnostic(
    batch,
    max_steps=12
):

    enc = build_encoder_inputs(batch)

    B = batch["labels"].shape[0]

    # mT5 decoder starts with pad_token_id
    decoder_ids = torch.full(
        (B, 1),
        tokenizer.pad_token_id,
        dtype=torch.long,
        device=DEVICE
    )

    steps = []

    for step in range(max_steps):

        decoder_attention = torch.ones_like(
            decoder_ids,
            dtype=torch.long
        )

        out = model.mt5(
            encoder_outputs=enc["encoder_outputs"],
            attention_mask=enc["attention_mask"],
            decoder_input_ids=decoder_ids,
            decoder_attention_mask=decoder_attention,
            return_dict=True
        )

        logits = out.logits[:, -1, :]

        probs = F.softmax(
            logits,
            dim=-1
        )

        top_prob, top_id = torch.topk(
            probs,
            k=5,
            dim=-1
        )

        next_id = top_id[:, 0]

        steps.append({
            "step": step,
            "decoder_input": decoder_ids.detach().cpu(),
            "top_id": top_id.detach().cpu(),
            "top_prob": top_prob.detach().cpu(),
            "next_id": next_id.detach().cpu(),
        })

        decoder_ids = torch.cat(
            [
                decoder_ids,
                next_id.unsqueeze(1)
            ],
            dim=1
        )

    return {
        **enc,
        "decoder_ids": decoder_ids,
        "steps": steps,
    }


# ============================================================
# 8. RUN E2 / E3
# ============================================================

results = {}

for epoch_name, ckpt_path in [
    ("EPOCH_2", E2_CKPT),
    ("EPOCH_3", E3_CKPT),
]:

    print("\n" + "=" * 70)
    print(epoch_name)
    print("=" * 70)

    ckpt = load_ckpt_for_audit(
        ckpt_path
    )

    print(
        "Checkpoint best_val:",
        ckpt.get("best_val")
    )

    epoch_results = []

    total_token_correct = 0
    total_token_valid = 0

    for batch_idx, batch in enumerate(
        audit_batches
    ):

        tf = teacher_forced_diagnostic(
            batch
        )

        ar = autoregressive_diagnostic(
            batch,
            max_steps=12
        )

        labels = tf["labels"]

        valid = labels != -100

        total_token_correct += (
            (
                tf["pred_ids"] == labels
            ) &
            valid
        ).sum().item()

        total_token_valid += (
            valid.sum().item()
        )

        for i in range(
            labels.shape[0]
        ):

            ref = clean_text(
                labels[i]
            )

            ar_ids = ar["decoder_ids"][i]

            pred = clean_text(
                ar_ids[1:]
            )

            # First actual generated token
            first_id = ar["steps"][0][
                "next_id"
            ][i].item()

            first_token = tokenizer.decode(
                [first_id],
                skip_special_tokens=False
            )

            # Top-5 at first step
            first_top_ids = ar["steps"][0][
                "top_id"
            ][i].tolist()

            first_top_probs = ar["steps"][0][
                "top_prob"
            ][i].tolist()

            first_top = []

            for tid, prob in zip(
                first_top_ids,
                first_top_probs
            ):
                tok = tokenizer.decode(
                    [tid],
                    skip_special_tokens=False
                )

                first_top.append(
                    (
                        tok,
                        float(prob)
                    )
                )

            # Structured prediction summary
            c = tf["concept_pred"][i].sum().item()
            n = tf["negative_pred"][i].sum().item()
            a = tf["attribute_pred"][i].sum().item()

            epoch_results.append({
                "batch_idx": batch_idx,
                "sample_idx": i,
                "reference": ref,
                "prediction": pred,
                "first_token": first_token,
                "first_top5": first_top,
                "num_positive_concepts": int(c),
                "num_positive_negatives": int(n),
                "num_positive_attributes": int(a),
            })

    token_acc = (
        total_token_correct /
        max(total_token_valid, 1)
    )

    results[epoch_name] = epoch_results

    print(
        "\nTeacher-forced token accuracy:",
        round(token_acc, 6)
    )

    preds = [
        x["prediction"]
        for x in epoch_results
    ]

    print(
        "Unique autoregressive predictions:",
        len(set(preds))
    )

    print(
        "\nFirst 5 cases:"
    )

    for x in epoch_results[:5]:
        print("-" * 60)
        print("REF :", x["reference"])
        print("PRED:", x["prediction"])
        print("FIRST TOKEN:", repr(x["first_token"]))
        print(
            "STRUCT:",
            "concept=",
            x["num_positive_concepts"],
            "negative=",
            x["num_positive_negatives"],
            "attribute=",
            x["num_positive_attributes"]
        )
        print("TOP5:", x["first_top5"])


# ============================================================
# 9. COMPARE STRUCTURED CONDITIONING
# ============================================================

print("\n" + "=" * 70)
print("STRUCTURED CONDITIONING COMPARISON")
print("=" * 70)

for epoch_name in [
    "EPOCH_2",
    "EPOCH_3"
]:

    rows = results[epoch_name]

    concept_counts = [
        x["num_positive_concepts"]
        for x in rows
    ]

    negative_counts = [
        x["num_positive_negatives"]
        for x in rows
    ]

    attribute_counts = [
        x["num_positive_attributes"]
        for x in rows
    ]

    print(
        epoch_name,
        "| concept positives:",
        Counter(concept_counts),
        "| negative positives:",
        Counter(negative_counts),
        "| attribute positives:",
        Counter(attribute_counts)
    )


# ============================================================
# 10. CHECK WHETHER FIRST TOKEN IS COLLAPSED
# ============================================================

print("\n" + "=" * 70)
print("FIRST TOKEN COLLAPSE")
print("=" * 70)

for epoch_name in [
    "EPOCH_2",
    "EPOCH_3"
]:

    first_tokens = [
        x["first_token"]
        for x in results[epoch_name]
    ]

    counts = Counter(
        first_tokens
    )

    print(
        "\n",
        epoch_name,
        "first-token distribution:"
    )

    for token, count in counts.most_common():
        print(
            repr(token),
            "=>",
            count
        )


# ============================================================
# 11. VERIFY MODEL STATE WAS NOT MODIFIED
# ============================================================

current_state = model.state_dict()

changed = []

for k, original in original_state.items():

    current = current_state[k].detach().cpu()

    if not torch.equal(
        original,
        current
    ):
        changed.append(k)

print("\n" + "=" * 70)
print("SAFETY CHECK")
print("=" * 70)

if changed:
    print(
        "WARNING — model state changed:",
        len(changed)
    )
    print(changed[:20])
else:
    print(
        "Model parameters unchanged: PASS"
    )

restore_original_model()

print(
    "Original runtime model restored: PASS"
)

print("=" * 70)
print("AUDIT COMPLETE")
print("=" * 70)

V4 GENERATION COLLAPSE DIAGNOSTIC
Device: cuda
E2: True /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/checkpoints/epoch_02.pt
E3: True /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/checkpoints/epoch_03.pt

Existing model/data objects: PASS
Current model state snapshot: PASS

Validation batch:
  pixel_values         (4, 8, 3, 224, 224)
  image_mask           (4, 8)
  labels               (4, 96)
  concept_targets      (4, 48)
  negative_targets     (4, 5)
  attribute_targets    (4, 6)

Audit cases collected: 20

EPOCH_2
Checkpoint best_val: 0.6521643281140656


OverflowError: out of range integral type conversion attempted

In [1]:
import json
from pathlib import Path

state = {
    "model": "V4_ConceptConditionedGeneration",
    "last_completed_epoch": 3,
    "best_epoch": 2,
    "best_val": 0.6521643281140656,

    "checkpoint_best":
        "/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/checkpoints/best.pt",

    "checkpoint_epoch2":
        "/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/checkpoints/epoch_02.pt",

    "checkpoint_epoch3":
        "/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/checkpoints/epoch_03.pt",

    "checkpoint_last":
        "/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/checkpoints/last.pt",

    "status":
        "Generation audit pending after fixing tokenizer.decode(-100)",

    "next_step":
        "Run V4 generation collapse diagnostic on 20 validation cases",

    "important_findings": [
        "Epoch 2 val loss = 0.652164",
        "Epoch 3 val loss = 0.689219",
        "Epoch 3 autoregressive generation collapsed to VIÊM MŨI MẠN",
        "Generation path was corrected to encoder_outputs",
        "Current audit code had two bugs: structured head input and decode(-100)",
        "structured head input bug was fixed: projector 768->512 before heads",
        "decode(-100) bug remains fixed by replacing -100 with pad_token_id"
    ]
}

path = Path(
    "/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/"
    "v4_runtime_state.json"
)

path.write_text(
    json.dumps(state, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Saved:", path)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/v4_runtime_state.json'

In [40]:
summary = """
V4 STATUS — 2026-09-24

Dataset:
- V4 manifest: 7606 cases
- train 6138 / val 756 / test 712
- max images/case = 8

Architecture:
- ViT-B/16 -> 768
- projector 768 -> 512
- structured heads on 512
- conditioner = 59 tokens
- mT5-small
- visual + structured prefix

Training:
- Epoch 2 best val = 0.6521643281
- Epoch 3 val = 0.689219
- Epoch 2 remains best checkpoint

Generation audit:
- Old generation path was incorrect.
- Correct path uses:
  mt5.encoder(inputs_embeds=..., attention_mask=...)
  then generate(encoder_outputs=...)

Correct-path result:
- Epoch 2: 4 unique predictions, malformed <extra_id_33> outputs
- Epoch 3: 1 unique prediction = VIÊM MŨI MẠN
- 20-case quick check Epoch 3 exact = 15%, but this is mode collapse

Current diagnostic:
- First audit crashed because structured heads received 768 instead of 512.
- Fixed by projector before structured heads.
- Second audit crashed because labels contain -100.
- Fixed clean_text() by replacing -100 with tokenizer.pad_token_id.

NEXT:
Run generation-collapse diagnostic again.
Do NOT train.
Do NOT use test set.
Do NOT modify checkpoints.
"""

summary_path = (
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "baseline_model_v4/v4_status_2026-09-24.txt"
)

with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary)

print(summary_path)

/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/v4_status_2026-09-24.txt


In [2]:
# ============================================================
# V4 — ONE-CELL RESTORE
# Restore environment after Colab runtime disconnect
# NO TRAINING / NO OPTIMIZER / NO CHECKPOINT MODIFICATION
# ============================================================

import os
import sys
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from PIL import Image

from transformers import (
    ViTModel,
    MT5ForConditionalGeneration,
    AutoTokenizer,
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0. DEVICE
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("V4 ONE-CELL RESTORE")
print("=" * 70)
print("Torch:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 1. GOOGLE DRIVE
# ------------------------------------------------------------

try:
    from google.colab import drive

    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")

    print("Drive: mounted")

except Exception as e:
    print("Drive mount skipped:", e)


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

BASE = "/content/drive/MyDrive/NoiSoi_Matching"

V4_DIR = os.path.join(
    BASE,
    "baseline_model_v4"
)

CKPT_DIR = os.path.join(
    V4_DIR,
    "checkpoints"
)

V4_MANIFEST = os.path.join(
    V4_DIR,
    "v4_training_manifest.csv"
)

STRUCTURED_CSV = (
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "v3_ontology_v2/"
    "v3_structured_targets_v2_2.csv"
)

IMAGE_MANIFEST = (
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "final_manifest/"
    "final_dataset_manifest.csv"
)

CONFIG_PATH = os.path.join(
    V4_DIR,
    "v4_config.json"
)

E2_CKPT = os.path.join(
    CKPT_DIR,
    "epoch_02.pt"
)

E3_CKPT = os.path.join(
    CKPT_DIR,
    "epoch_03.pt"
)

BEST_CKPT = os.path.join(
    CKPT_DIR,
    "best.pt"
)

print("\nPaths:")
for p in [
    V4_MANIFEST,
    STRUCTURED_CSV,
    IMAGE_MANIFEST,
    CONFIG_PATH,
    E2_CKPT,
    E3_CKPT,
    BEST_CKPT,
]:
    print(
        "OK " if os.path.exists(p) else "MISS",
        p
    )


# ------------------------------------------------------------
# 3. CONFIG
# ------------------------------------------------------------

with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:
    config = json.load(f)

print("\nConfig:")
print(
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2
    )
)


# ------------------------------------------------------------
# 4. TOKENIZER
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    "google/mt5-small"
)

print("\nTokenizer:")
print("type:", type(tokenizer).__name__)
print("vocab:", tokenizer.vocab_size)
print("pad:", tokenizer.pad_token_id)
print("eos:", tokenizer.eos_token_id)


# ------------------------------------------------------------
# 5. FROZEN ONTOLOGY
# ------------------------------------------------------------

CONCEPT_COLS = [
    "concept__amidan_qua_phat",
    "concept__chan_thuong_mang_nhi",
    "concept__chan_thuong_ong_tai_ngoai",
    "concept__chay_mau_mui",
    "concept__di_vat_hong",
    "concept__di_vat_hong_thanh_quan",
    "concept__di_vat_tai",
    "concept__dich_vat_mui",
    "concept__hat_day_thanh",
    "concept__hau_phau_mui_xoang",
    "concept__hau_phau_va_nhi",
    "concept__hep_ong_tai_ngoai",
    "concept__hoc_xuong_ca",
    "concept__lech_vach_ngan",
    "concept__liet_day_thanh",
    "concept__nang_day_thanh",
    "concept__nhot_ong_tai_ngoai",
    "concept__not_vanh_tai",
    "concept__polyp_day_thanh",
    "concept__polyp_mui",
    "concept__polyp_ong_tai_ngoai",
    "concept__qua_phat_va",
    "concept__ray_tai",
    "concept__ro_luan_nhi",
    "concept__seo_hoc_mui",
    "concept__theo_doi_trao_nguoc",
    "concept__thung_mang_nhi",
    "concept__tien_dinh_mui",
    "concept__tu_dich_vanh_tai",
    "concept__u_hoc_mui",
    "concept__u_nhu_cuon_mui",
    "concept__u_nhu_hoc_mui",
    "concept__u_xoang",
    "concept__viem_amidan",
    "concept__viem_hong",
    "concept__viem_luoi",
    "concept__viem_mang_nhi",
    "concept__viem_mieng",
    "concept__viem_mui",
    "concept__viem_mui_xoang",
    "concept__viem_ong_tai_ngoai",
    "concept__viem_tai_giua",
    "concept__viem_tai_xuong_chum",
    "concept__viem_thanh_quan",
    "concept__viem_va",
    "concept__viem_vanh_tai",
    "concept__viem_xoang",
    "concept__xep_mang_nhi",
]

NEGATIVE_COLS = [
    "negative__no_abnormal_external_middle_ear",
    "negative__no_abnormal_nose_sinus",
    "negative__no_abnormal_ent",
    "negative__no_bleeding",
    "negative__no_foreign_body",
]

ATTRIBUTE_COLS = [
    "attribute__acute",
    "attribute__chronic",
    "attribute__right",
    "attribute__left",
    "attribute__bilateral",
    "attribute__post_surgery",
]

assert len(CONCEPT_COLS) == 48
assert len(NEGATIVE_COLS) == 5
assert len(ATTRIBUTE_COLS) == 6

print("\nOntology: 48 / 5 / 6 PASS")


# ------------------------------------------------------------
# 6. LOAD MANIFESTS
# ------------------------------------------------------------

v4_manifest = pd.read_csv(
    V4_MANIFEST
)

structured_df = pd.read_csv(
    STRUCTURED_CSV,
    index_col=0
)

image_manifest = pd.read_csv(
    IMAGE_MANIFEST
)

print("\nManifest shapes:")
print("V4:", v4_manifest.shape)
print("Structured:", structured_df.shape)
print("Images:", image_manifest.shape)

assert len(v4_manifest) == 7606
assert len(structured_df) == 7606


# ------------------------------------------------------------
# 7. CHECK SPLITS
# ------------------------------------------------------------

print("\nSplits:")
print(
    v4_manifest["split"].value_counts()
)

assert (
    v4_manifest["split"]
    .value_counts()
    .to_dict()
    == {
        "train": 6138,
        "val": 756,
        "test": 712,
    }
)

print("Split: PASS")


# ------------------------------------------------------------
# 8. STRUCTURED TARGET ALIGNMENT
# ------------------------------------------------------------

for c in (
    CONCEPT_COLS +
    NEGATIVE_COLS +
    ATTRIBUTE_COLS
):
    assert c in structured_df.columns, c

print(
    "\nStructured target alignment: PASS"
)


# ------------------------------------------------------------
# 9. MODEL COMPONENTS
# ------------------------------------------------------------

class Projector(nn.Module):

    def __init__(
        self,
        in_dim=768,
        out_dim=512
    ):
        super().__init__()

        self.proj = nn.Linear(
            in_dim,
            out_dim
        )

    def forward(self, x):
        return self.proj(x)


class StructuredConditioner(nn.Module):

    def __init__(
        self,
        d_model,
        num_concepts=48,
        num_negative=5,
        num_attributes=6,
    ):
        super().__init__()

        self.num_concepts = num_concepts
        self.num_negative = num_negative
        self.num_attributes = num_attributes

        self.concept_value = nn.Embedding(
            2,
            d_model
        )

        self.negative_value = nn.Embedding(
            2,
            d_model
        )

        self.attribute_value = nn.Embedding(
            2,
            d_model
        )

        self.concept_label = nn.Embedding(
            num_concepts,
            d_model
        )

        self.negative_label = nn.Embedding(
            num_negative,
            d_model
        )

        self.attribute_label = nn.Embedding(
            num_attributes,
            d_model
        )

        self.type_embedding = nn.Embedding(
            3,
            d_model
        )

        self.norm = nn.LayerNorm(
            d_model
        )

    def forward(
        self,
        concept_targets,
        negative_targets,
        attribute_targets,
    ):

        device = concept_targets.device

        concept_ids = (
            concept_targets
            .long()
            .clamp(0, 1)
        )

        negative_ids = (
            negative_targets
            .long()
            .clamp(0, 1)
        )

        attribute_ids = (
            attribute_targets
            .long()
            .clamp(0, 1)
        )

        concept_idx = torch.arange(
            self.num_concepts,
            device=device
        )

        concept_type = self.type_embedding(
            torch.zeros(
                self.num_concepts,
                dtype=torch.long,
                device=device
            )
        )

        concept_tokens = (
            self.concept_value(
                concept_ids
            )
            +
            self.concept_label(
                concept_idx
            )[None, :, :]
            +
            concept_type[None, :, :]
        )

        negative_idx = torch.arange(
            self.num_negative,
            device=device
        )

        negative_type = self.type_embedding(
            torch.ones(
                self.num_negative,
                dtype=torch.long,
                device=device
            )
        )

        negative_tokens = (
            self.negative_value(
                negative_ids
            )
            +
            self.negative_label(
                negative_idx
            )[None, :, :]
            +
            negative_type[None, :, :]
        )

        attribute_idx = torch.arange(
            self.num_attributes,
            device=device
        )

        attribute_type = self.type_embedding(
            torch.full(
                (self.num_attributes,),
                2,
                dtype=torch.long,
                device=device
            )
        )

        attribute_tokens = (
            self.attribute_value(
                attribute_ids
            )
            +
            self.attribute_label(
                attribute_idx
            )[None, :, :]
            +
            attribute_type[None, :, :]
        )

        structured_tokens = torch.cat(
            [
                concept_tokens,
                negative_tokens,
                attribute_tokens,
            ],
            dim=1
        )

        structured_tokens = self.norm(
            structured_tokens
        )

        B = concept_targets.shape[0]

        structured_attention = torch.ones(
            B,
            structured_tokens.shape[1],
            dtype=torch.long,
            device=device
        )

        return (
            structured_tokens,
            structured_attention
        )


class V4Model(nn.Module):

    def __init__(self):

        super().__init__()

        self.vision = ViTModel.from_pretrained(
            "google/vit-base-patch16-224"
        )

        self.mt5 = MT5ForConditionalGeneration.from_pretrained(
            "google/mt5-small"
        )

        self.projector = Projector(
            768,
            512
        )

        self.structured_heads = nn.ModuleDict({
            "concept": nn.Linear(
                512,
                48
            ),
            "negative": nn.Linear(
                512,
                5
            ),
            "attribute": nn.Linear(
                512,
                6
            ),
        })

        self.conditioner = StructuredConditioner(
            d_model=512,
            num_concepts=48,
            num_negative=5,
            num_attributes=6,
        )


# ------------------------------------------------------------
# 10. CREATE MODEL
# ------------------------------------------------------------

print("\nCreating V4 model...")

model = V4Model()

model.to(DEVICE)
model.eval()

print("Model created.")


# ------------------------------------------------------------
# 11. ARCHITECTURE CHECK
# ------------------------------------------------------------

print("\nArchitecture:")

print(
    "projector:",
    tuple(
        model.projector.proj.weight.shape
    )
)

print(
    "concept:",
    tuple(
        model.structured_heads[
            "concept"
        ].weight.shape
    )
)

print(
    "negative:",
    tuple(
        model.structured_heads[
            "negative"
        ].weight.shape
    )
)

print(
    "attribute:",
    tuple(
        model.structured_heads[
            "attribute"
        ].weight.shape
    )
)

assert (
    model.projector.proj.weight.shape
    == (512, 768)
)

assert (
    model.structured_heads[
        "concept"
    ].weight.shape
    == (48, 512)
)

assert (
    model.structured_heads[
        "negative"
    ].weight.shape
    == (5, 512)
)

assert (
    model.structured_heads[
        "attribute"
    ].weight.shape
    == (6, 512)
)

print("Architecture: PASS")


# ------------------------------------------------------------
# 12. CHECKPOINT LOAD
# ------------------------------------------------------------

e2 = torch.load(
    E2_CKPT,
    map_location="cpu",
    weights_only=False
)

e3 = torch.load(
    E3_CKPT,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint metadata:")

print(
    "E2 epoch:",
    e2["epoch"]
)

print(
    "E2 best_val:",
    e2["best_val"]
)

print(
    "E3 epoch:",
    e3["epoch"]
)

print(
    "E3 best_val:",
    e3["best_val"]
)


# ------------------------------------------------------------
# 13. STRICT LOAD E2
# ------------------------------------------------------------

model.load_state_dict(
    e2["model_state"],
    strict=True
)

model.to(DEVICE)
model.eval()

print(
    "\nE2 strict checkpoint load: PASS"
)


# ------------------------------------------------------------
# 14. RESTORE SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("V4 RESTORE COMPLETE")
print("=" * 70)

print(
    "Best checkpoint : epoch_02.pt"
)

print(
    "Epoch 2 val     :",
    e2["best_val"]
)

print(
    "Epoch 3 val     :",
    e3["best_val"]
)

print(
    "Next task       : generation-collapse diagnostic"
)

print(
    "Training        : NOT RUN"
)

print(
    "Checkpoint      : NOT MODIFIED"
)

print("=" * 70)

V4 ONE-CELL RESTORE
Torch: 2.11.0+cu128
Device: cuda
GPU: NVIDIA A100-SXM4-40GB
Mounted at /content/drive
Drive: mounted

Paths:
OK  /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/v4_training_manifest.csv
OK  /content/drive/MyDrive/NoiSoi_Matching/v3_ontology_v2/v3_structured_targets_v2_2.csv
OK  /content/drive/MyDrive/NoiSoi_Matching/final_manifest/final_dataset_manifest.csv
OK  /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/v4_config.json
OK  /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/checkpoints/epoch_02.pt
OK  /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/checkpoints/epoch_03.pt
OK  /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/checkpoints/best.pt

Config:
{
  "model_name": "V4_ConceptConditionedGeneration",
  "vision_model": "google/vit-base-patch16-224",
  "text_model": "google/mt5-small",
  "ontology": "V3-Ontology-V2.2",
  "num_concepts": 48,
  "num_negative_findings": 5,
  "num_attributes": 6,
  "image_size": 224,
  "max_

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model created.

Architecture:
projector: (512, 768)
concept: (48, 512)
negative: (5, 512)
attribute: (6, 512)
Architecture: PASS


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.20GB            

model.safetensors: downloading bytes:           |  0.00B            


Checkpoint metadata:
E2 epoch: 2
E2 best_val: 0.6521643281140656
E3 epoch: 3
E3 best_val: 0.6521643281140656

E2 strict checkpoint load: PASS

V4 RESTORE COMPLETE
Best checkpoint : epoch_02.pt
Epoch 2 val     : 0.6521643281140656
Epoch 3 val     : 0.6521643281140656
Next task       : generation-collapse diagnostic
Training        : NOT RUN
Checkpoint      : NOT MODIFIED


In [3]:
# ============================================================
# V4 — DATASET / VAL LOADER RESTORE + COMPATIBILITY AUDIT
# AFTER COLAB RUNTIME DISCONNECT
#
# NO TRAINING
# NO OPTIMIZER
# NO SCHEDULER
# NO CHECKPOINT MODIFICATION
# ============================================================

import os
import re
import math
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import AutoImageProcessor

print("=" * 70)
print("V4 DATASET / VAL LOADER RESTORE")
print("=" * 70)


# ============================================================
# 1. BASIC SETTINGS
# ============================================================

MAX_IMAGES = 8
MAX_TARGET_LENGTH = 96
BATCH_SIZE = 4

print("MAX_IMAGES:", MAX_IMAGES)
print("MAX_TARGET_LENGTH:", MAX_TARGET_LENGTH)
print("BATCH_SIZE:", BATCH_SIZE)


# ============================================================
# 2. LOAD MANIFESTS
# ============================================================

v4_manifest = pd.read_csv(
    V4_MANIFEST
)

structured_df = pd.read_csv(
    STRUCTURED_CSV,
    index_col=0
)

image_manifest = pd.read_csv(
    IMAGE_MANIFEST
)

print("\nShapes:")
print("v4_manifest    :", v4_manifest.shape)
print("structured_df  :", structured_df.shape)
print("image_manifest :", image_manifest.shape)


# ============================================================
# 3. COLUMN COMPATIBILITY AUDIT
# ============================================================

print("\nV4 columns:")
print(list(v4_manifest.columns))

print("\nImage manifest columns:")
print(list(image_manifest.columns))


# ------------------------------------------------------------
# Resolve required columns conservatively
# ------------------------------------------------------------

def resolve_column(df, candidates, name):

    for c in candidates:
        if c in df.columns:
            print(
                f"{name}: {c}"
            )
            return c

    raise RuntimeError(
        f"Cannot resolve {name}.\n"
        f"Tried: {candidates}\n"
        f"Available columns: {list(df.columns)}"
    )


CASE_COL = resolve_column(
    v4_manifest,
    ["case_id"],
    "V4 case column"
)

REPORT_COL = resolve_column(
    v4_manifest,
    ["ket_luan"],
    "Report column"
)

IMAGE_CASE_COL = resolve_column(
    image_manifest,
    ["case_id"],
    "Image case column"
)

IMAGE_PATH_COL = resolve_column(
    image_manifest,
    ["image_path"],
    "Image path column"
)

STATUS_COL = resolve_column(
    image_manifest,
    [
        "status",
        "image_status",
        "image_quality_status"
    ],
    "Image status column"
)


# ============================================================
# 4. SPLIT AUDIT
# ============================================================

split_counts = (
    v4_manifest["split"]
    .value_counts()
    .to_dict()
)

print("\nSplit counts:")
print(split_counts)

assert split_counts["train"] == 6138
assert split_counts["val"] == 756
assert split_counts["test"] == 712

val_cases = (
    v4_manifest.loc[
        v4_manifest["split"] == "val",
        CASE_COL
    ]
    .astype(str)
    .tolist()
)

assert len(val_cases) == 756
assert len(set(val_cases)) == 756

print(
    "Validation cases:",
    len(val_cases)
)


# ============================================================
# 5. IMAGE FILTER
# ============================================================

image_manifest[IMAGE_CASE_COL] = (
    image_manifest[IMAGE_CASE_COL]
    .astype(str)
)

image_manifest[STATUS_COL] = (
    image_manifest[STATUS_COL]
    .astype(str)
    .str.upper()
    .str.strip()
)

normal_images = image_manifest[
    image_manifest[STATUS_COL] == "NORMAL"
].copy()

print("\nImage status:")
print(
    image_manifest[STATUS_COL]
    .value_counts()
)

print(
    "\nNORMAL images:",
    len(normal_images)
)


# ============================================================
# 6. KEEP ONLY V4 VALIDATION CASES
# ============================================================

val_set = set(val_cases)

val_images = normal_images[
    normal_images[IMAGE_CASE_COL]
    .isin(val_set)
].copy()

print(
    "NORMAL validation images:",
    len(val_images)
)


# ============================================================
# 7. GROUP IMAGES BY CASE
# ============================================================

case_to_images = {}

for case_id, g in val_images.groupby(
    IMAGE_CASE_COL,
    sort=False
):

    paths = (
        g[IMAGE_PATH_COL]
        .astype(str)
        .tolist()
    )

    # Stable ordering
    paths = sorted(paths)

    case_to_images[str(case_id)] = paths


missing_cases = [
    c
    for c in val_cases
    if c not in case_to_images
    or len(case_to_images[c]) == 0
]

print(
    "Validation cases without NORMAL image:",
    len(missing_cases)
)

assert len(missing_cases) == 0

raw_counts = [
    len(case_to_images[c])
    for c in val_cases
]

print(
    "Raw images/case:"
)

print(
    "  min   =",
    min(raw_counts)
)

print(
    "  median=",
    np.median(raw_counts)
)

print(
    "  mean  =",
    np.mean(raw_counts)
)

print(
    "  max   =",
    max(raw_counts)
)


# ============================================================
# 8. STRUCTURED TARGET ALIGNMENT
# ============================================================

structured_df.index = (
    structured_df.index
    .astype(str)
)

missing_structured = [
    c
    for c in val_cases
    if c not in structured_df.index
]

print(
    "\nValidation cases missing structured target:",
    len(missing_structured)
)

assert len(missing_structured) == 0


# ============================================================
# 9. REPORT ALIGNMENT
# ============================================================

v4_case_df = (
    v4_manifest
    .copy()
)

v4_case_df[CASE_COL] = (
    v4_case_df[CASE_COL]
    .astype(str)
)

v4_case_lookup = (
    v4_case_df
    .set_index(CASE_COL)
)

for c in val_cases:
    assert c in v4_case_lookup.index

print(
    "Report alignment: PASS"
)


# ============================================================
# 10. IMAGE PROCESSOR
# ============================================================

image_processor = AutoImageProcessor.from_pretrained(
    "google/vit-base-patch16-224"
)

print(
    "\nImage processor:",
    type(image_processor).__name__
)

print(
    "size:",
    image_processor.size
)

print(
    "image_mean:",
    image_processor.image_mean
)

print(
    "image_std:",
    image_processor.image_std
)


# ============================================================
# 11. V4 DATASET
# ============================================================

class V4CaseDataset(Dataset):

    def __init__(
        self,
        case_ids,
        case_to_images,
        case_lookup,
        structured_df,
        tokenizer,
        image_processor,
        max_images=8,
        max_target_length=96,
    ):

        self.case_ids = list(case_ids)

        self.case_to_images = case_to_images
        self.case_lookup = case_lookup
        self.structured_df = structured_df

        self.tokenizer = tokenizer
        self.image_processor = image_processor

        self.max_images = max_images
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.case_ids)

    def _load_image(self, path):

        with Image.open(path) as img:

            img = img.convert("RGB")

            out = self.image_processor(
                images=img,
                return_tensors="pt"
            )

            return out["pixel_values"][0]

    def __getitem__(self, idx):

        case_id = self.case_ids[idx]

        # ----------------------------------------------------
        # Images
        # ----------------------------------------------------

        paths = self.case_to_images[
            case_id
        ]

        # Exact V4 cap
        paths = paths[:self.max_images]

        images = []

        for path in paths:

            if not os.path.exists(path):
                raise FileNotFoundError(
                    f"Missing image:\n{path}"
                )

            images.append(
                self._load_image(path)
            )

        n_images = len(images)

        if n_images == 0:
            raise RuntimeError(
                f"No images for case {case_id}"
            )

        # Pad to MAX_IMAGES
        C, H, W = images[0].shape

        pixel_values = torch.zeros(
            self.max_images,
            C,
            H,
            W,
            dtype=torch.float32
        )

        pixel_values[
            :n_images
        ] = torch.stack(images)

        image_mask = torch.zeros(
            self.max_images,
            dtype=torch.long
        )

        image_mask[
            :n_images
        ] = 1

        # ----------------------------------------------------
        # Report
        # ----------------------------------------------------

        row = self.case_lookup.loc[
            case_id
        ]

        report = str(
            row[REPORT_COL]
        ).strip()

        enc = self.tokenizer(
            report,
            max_length=self.max_target_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = enc["input_ids"][0].clone()

        # T5 training convention:
        # padding -> -100
        labels[
            labels == self.tokenizer.pad_token_id
        ] = -100

        # ----------------------------------------------------
        # Structured targets
        # ----------------------------------------------------

        srow = self.structured_df.loc[
            case_id
        ]

        concepts = torch.tensor(
            pd.to_numeric(
                srow[CONCEPT_COLS],
                errors="coerce"
            )
            .fillna(0)
            .astype(float)
            .values,
            dtype=torch.float32
        )

        negatives = torch.tensor(
            pd.to_numeric(
                srow[NEGATIVE_COLS],
                errors="coerce"
            )
            .fillna(0)
            .astype(float)
            .values,
            dtype=torch.float32
        )

        attributes = torch.tensor(
            pd.to_numeric(
                srow[ATTRIBUTE_COLS],
                errors="coerce"
            )
            .fillna(0)
            .astype(float)
            .values,
            dtype=torch.float32
        )

        return {
            "case_id": case_id,
            "image_paths": paths,

            "pixel_values": pixel_values,
            "image_mask": image_mask,

            "labels": labels,

            "concept_targets": concepts,
            "negative_targets": negatives,
            "attribute_targets": attributes,
        }


# ============================================================
# 12. CREATE VALIDATION DATASET
# ============================================================

val_dataset = V4CaseDataset(
    case_ids=val_cases,
    case_to_images=case_to_images,
    case_lookup=v4_case_lookup,
    structured_df=structured_df,
    tokenizer=tokenizer,
    image_processor=image_processor,
    max_images=MAX_IMAGES,
    max_target_length=MAX_TARGET_LENGTH,
)

print(
    "\nval_dataset:",
    len(val_dataset)
)

assert len(val_dataset) == 756


# ============================================================
# 13. COLLATE FUNCTION
# ============================================================

def v4_collate(batch):

    return {
        "case_id": [
            x["case_id"]
            for x in batch
        ],

        "image_paths": [
            x["image_paths"]
            for x in batch
        ],

        "pixel_values": torch.stack([
            x["pixel_values"]
            for x in batch
        ]),

        "image_mask": torch.stack([
            x["image_mask"]
            for x in batch
        ]),

        "labels": torch.stack([
            x["labels"]
            for x in batch
        ]),

        "concept_targets": torch.stack([
            x["concept_targets"]
            for x in batch
        ]),

        "negative_targets": torch.stack([
            x["negative_targets"]
            for x in batch
        ]),

        "attribute_targets": torch.stack([
            x["attribute_targets"]
            for x in batch
        ]),
    }


# ============================================================
# 14. VAL LOADER
# ============================================================

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=v4_collate,
)

print(
    "\nval_loader batches:",
    len(val_loader)
)

assert len(val_loader) == math.ceil(
    756 / BATCH_SIZE
)


# ============================================================
# 15. ONE-BATCH DATA AUDIT
# ============================================================

batch = next(iter(val_loader))

print("\n" + "=" * 70)
print("ONE-BATCH DATA AUDIT")
print("=" * 70)

for k, v in batch.items():

    if torch.is_tensor(v):
        print(
            f"{k:20s}",
            tuple(v.shape),
            v.dtype
        )

    else:
        print(
            f"{k:20s}",
            type(v).__name__
        )


# ------------------------------------------------------------
# Expected shapes
# ------------------------------------------------------------

assert batch["pixel_values"].shape == (
    4,
    8,
    3,
    224,
    224
)

assert batch["image_mask"].shape == (
    4,
    8
)

assert batch["labels"].shape == (
    4,
    96
)

assert batch["concept_targets"].shape == (
    4,
    48
)

assert batch["negative_targets"].shape == (
    4,
    5
)

assert batch["attribute_targets"].shape == (
    4,
    6
)

# ------------------------------------------------------------
# Binary structured targets
# ------------------------------------------------------------

for name in [
    "concept_targets",
    "negative_targets",
    "attribute_targets",
]:

    vals = torch.unique(
        batch[name]
    ).cpu().tolist()

    print(
        name,
        "unique:",
        vals
    )

    assert all(
        x in [0.0, 1.0]
        for x in vals
    )


# ------------------------------------------------------------
# Finite checks
# ------------------------------------------------------------

assert torch.isfinite(
    batch["pixel_values"]
).all()

assert torch.isfinite(
    batch["concept_targets"]
).all()

assert torch.isfinite(
    batch["negative_targets"]
).all()

assert torch.isfinite(
    batch["attribute_targets"]
).all()


# ------------------------------------------------------------
# Image mask
# ------------------------------------------------------------

mask_counts = (
    batch["image_mask"]
    .sum(dim=1)
    .cpu()
    .tolist()
)

print(
    "images/case in first batch:",
    mask_counts
)

assert all(
    1 <= x <= 8
    for x in mask_counts
)


# ============================================================
# 16. CHECK ACTUAL IMAGE PIXEL RANGE
# ============================================================

print(
    "pixel range:",
    float(batch["pixel_values"].min()),
    "to",
    float(batch["pixel_values"].max())
)


# ============================================================
# 17. CHECK LABELS WITHOUT DECODING -100
# ============================================================

valid_label_ids = batch["labels"][
    batch["labels"] != -100
]

print(
    "valid label token count:",
    valid_label_ids.numel()
)

assert valid_label_ids.numel() > 0

print(
    "valid label min:",
    int(valid_label_ids.min())
)

print(
    "valid label max:",
    int(valid_label_ids.max())
)


# ============================================================
# 18. MODEL ONE-BATCH FORWARD AUDIT
# ============================================================

print("\n" + "=" * 70)
print("V4 ONE-BATCH FORWARD AUDIT")
print("=" * 70)

model.eval()

with torch.no_grad():

    pixel_values = (
        batch["pixel_values"]
        .to(
            DEVICE,
            non_blocking=True
        )
    )

    image_mask = (
        batch["image_mask"]
        .to(
            DEVICE,
            non_blocking=True
        )
    )

    B, N, C, H, W = (
        pixel_values.shape
    )

    flat = pixel_values.reshape(
        B * N,
        C,
        H,
        W
    )

    vision_outputs = model.vision(
        pixel_values=flat
    )

    pooled = (
        vision_outputs
        .last_hidden_state[:, 0]
    )

    assert pooled.shape == (
        B * N,
        768
    )

    pooled = pooled.reshape(
        B,
        N,
        768
    )

    mask = image_mask.unsqueeze(-1).float()

    visual_768 = (
        pooled * mask
    ).sum(dim=1)

    visual_768 = (
        visual_768
        /
        mask.sum(dim=1).clamp(min=1)
    )

    assert visual_768.shape == (
        B,
        768
    )

    visual_512 = model.projector(
        visual_768
    )

    assert visual_512.shape == (
        B,
        512
    )

    concept_logits = (
        model.structured_heads[
            "concept"
        ](
            visual_512
        )
    )

    negative_logits = (
        model.structured_heads[
            "negative"
        ](
            visual_512
        )
    )

    attribute_logits = (
        model.structured_heads[
            "attribute"
        ](
            visual_512
        )
    )

    print(
        "visual_768:",
        tuple(visual_768.shape)
    )

    print(
        "visual_512:",
        tuple(visual_512.shape)
    )

    print(
        "concept_logits:",
        tuple(concept_logits.shape)
    )

    print(
        "negative_logits:",
        tuple(negative_logits.shape)
    )

    print(
        "attribute_logits:",
        tuple(attribute_logits.shape)
    )

    assert torch.isfinite(
        visual_768
    ).all()

    assert torch.isfinite(
        visual_512
    ).all()

    assert torch.isfinite(
        concept_logits
    ).all()

    assert torch.isfinite(
        negative_logits
    ).all()

    assert torch.isfinite(
        attribute_logits
    ).all()


# ============================================================
# 19. RESTORE SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("DATASET / VAL LOADER RESTORE COMPLETE")
print("=" * 70)

print(
    "Validation cases :",
    len(val_dataset)
)

print(
    "Validation batches:",
    len(val_loader)
)

print(
    "Images/case cap  :",
    MAX_IMAGES
)

print(
    "Target length    :",
    MAX_TARGET_LENGTH
)

print(
    "One-batch shapes : PASS"
)

print(
    "Structured target: PASS"
)

print(
    "Model forward    : PASS"
)

print(
    "Training         : NOT RUN"
)

print(
    "Checkpoint       : NOT MODIFIED"
)

print("=" * 70)

V4 DATASET / VAL LOADER RESTORE
MAX_IMAGES: 8
MAX_TARGET_LENGTH: 96
BATCH_SIZE: 4

Shapes:
v4_manifest    : (7606, 15)
structured_df  : (7606, 101)
image_manifest : (76405, 26)

V4 columns:
['case_id', 'patient_group_id', 'split', 'ket_luan', 'concepts_v2_str', 'negative_findings_str', 'attributes_v2_str', 'concept_vector_str', 'negative_vector_str', 'attribute_vector_str', 'has_structured_label', 'structured_loss_mask', 'num_concepts_v2', 'num_negative_findings', 'num_attributes_v2']

Image manifest columns:
['case_id', 'image_order', 'image_path', 'image_name', 'record_id', 'datetime', 'date', 'image_index', 'n_images_case', 'is_no_signal_v3', 'image_status', 'patient_group_id', 'split', 'pdf_path', 'ho_ten', 'nam_sinh', 'gioi_tinh_clean', 'ly_do_noi_soi', 'tai', 'hoc_mui', 'hong_mui', 'hong_thanh_quan', 'hong_mieng', 'ket_luan', 'phan_biet', 'de_nghi']
V4 case column: case_id
Report column: ket_luan
Image case column: case_id
Image path column: image_path
Image status column: image_

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]


Image processor: ViTImageProcessor
size: SizeDict(height=224, width=224, longest_edge=None, shortest_edge=None, max_height=None, max_width=None)
image_mean: (0.5, 0.5, 0.5)
image_std: (0.5, 0.5, 0.5)

val_dataset: 756

val_loader batches: 189

ONE-BATCH DATA AUDIT
case_id              list
image_paths          list
pixel_values         (4, 8, 3, 224, 224) torch.float32
image_mask           (4, 8) torch.int64
labels               (4, 96) torch.int64
concept_targets      (4, 48) torch.float32
negative_targets     (4, 5) torch.float32
attribute_targets    (4, 6) torch.float32
concept_targets unique: [0.0, 1.0]
negative_targets unique: [0.0]
attribute_targets unique: [0.0, 1.0]
images/case in first batch: [6, 8, 6, 8]
pixel range: -0.9450980424880981 to 1.0
valid label token count: 62
valid label min: 1
valid label max: 199835

V4 ONE-BATCH FORWARD AUDIT
visual_768: (4, 768)
visual_512: (4, 512)
concept_logits: (4, 48)
negative_logits: (4, 5)
attribute_logits: (4, 6)

DATASET / VAL LOADER

In [4]:
# ============================================================
# V4 — FINAL 20-CASE GENERATION COLLAPSE DIAGNOSTIC
# EPOCH 2 vs EPOCH 3
#
# NO TRAINING
# NO OPTIMIZER
# NO SCHEDULER
# NO CHECKPOINT MODIFICATION
# ============================================================

import os
import json
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np

from collections import Counter


print("=" * 70)
print("V4 FINAL GENERATION COLLAPSE DIAGNOSTIC")
print("=" * 70)


# ============================================================
# 1. SAFETY CHECK
# ============================================================

assert "model" in globals()
assert "val_loader" in globals()
assert "tokenizer" in globals()

assert os.path.exists(E2_CKPT)
assert os.path.exists(E3_CKPT)

print("Model       : PASS")
print("val_loader  : PASS")
print("tokenizer   : PASS")
print("E2 checkpoint: PASS")
print("E3 checkpoint: PASS")


# ============================================================
# 2. SNAPSHOT CURRENT MODEL
# ============================================================

original_state = {
    k: v.detach().cpu().clone()
    for k, v in model.state_dict().items()
}

print("Model snapshot: PASS")


# ============================================================
# 3. SAFE DECODING
# ============================================================

def safe_decode(ids, skip_special_tokens=True):

    if torch.is_tensor(ids):
        ids = ids.detach().cpu().tolist()

    if isinstance(ids, np.ndarray):
        ids = ids.tolist()

    # labels use -100 for masked padding
    ids = [
        tokenizer.pad_token_id
        if int(x) == -100
        else int(x)
        for x in ids
    ]

    return tokenizer.decode(
        ids,
        skip_special_tokens=skip_special_tokens
    ).strip()


# ============================================================
# 4. BUILD EXACT V4 ENCODER INPUT
# ============================================================

@torch.no_grad()
def build_v4_encoder(batch):

    pixel_values = batch[
        "pixel_values"
    ].to(
        DEVICE,
        non_blocking=True
    )

    image_mask = batch[
        "image_mask"
    ].to(
        DEVICE,
        non_blocking=True
    )

    B, N, C, H, W = pixel_values.shape

    flat = pixel_values.reshape(
        B * N,
        C,
        H,
        W
    )

    # --------------------------------------------------------
    # ViT
    # --------------------------------------------------------

    vision_outputs = model.vision(
        pixel_values=flat
    )

    pooled = (
        vision_outputs
        .last_hidden_state[:, 0]
    )

    assert pooled.shape == (
        B * N,
        768
    )

    pooled = pooled.reshape(
        B,
        N,
        768
    )

    # --------------------------------------------------------
    # Masked image pooling
    # --------------------------------------------------------

    mask = image_mask.unsqueeze(-1).float()

    visual_768 = (
        pooled * mask
    ).sum(dim=1)

    visual_768 = (
        visual_768
        /
        mask.sum(dim=1).clamp(min=1)
    )

    # --------------------------------------------------------
    # Projector 768 -> 512
    # --------------------------------------------------------

    visual_512 = model.projector(
        visual_768
    )

    assert visual_512.shape == (
        B,
        512
    )

    # --------------------------------------------------------
    # Structured heads
    # --------------------------------------------------------

    concept_logits = (
        model.structured_heads[
            "concept"
        ](
            visual_512
        )
    )

    negative_logits = (
        model.structured_heads[
            "negative"
        ](
            visual_512
        )
    )

    attribute_logits = (
        model.structured_heads[
            "attribute"
        ](
            visual_512
        )
    )

    # EXACT V4 hard threshold
    concept_prob = torch.sigmoid(
        concept_logits
    )

    negative_prob = torch.sigmoid(
        negative_logits
    )

    attribute_prob = torch.sigmoid(
        attribute_logits
    )

    concept_pred = (
        concept_prob >= 0.5
    ).float()

    negative_pred = (
        negative_prob >= 0.5
    ).float()

    attribute_pred = (
        attribute_prob >= 0.5
    ).float()

    # --------------------------------------------------------
    # Structured conditioner
    # --------------------------------------------------------

    structured_tokens, structured_attention = (
        model.conditioner(
            concept_pred,
            negative_pred,
            attribute_pred
        )
    )

    # --------------------------------------------------------
    # Visual + structured prefix
    # --------------------------------------------------------

    inputs_embeds = torch.cat(
        [
            visual_512.unsqueeze(1),
            structured_tokens
        ],
        dim=1
    )

    attention_mask = torch.cat(
        [
            torch.ones(
                B,
                1,
                dtype=torch.long,
                device=DEVICE
            ),
            structured_attention
        ],
        dim=1
    )

    # --------------------------------------------------------
    # EXACT encoder path used for generation
    # --------------------------------------------------------

    encoder_outputs = model.mt5.encoder(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask,
        return_dict=True
    )

    return {
        "encoder_outputs": encoder_outputs,
        "attention_mask": attention_mask,

        "visual_768": visual_768,
        "visual_512": visual_512,

        "concept_logits": concept_logits,
        "negative_logits": negative_logits,
        "attribute_logits": attribute_logits,

        "concept_prob": concept_prob,
        "negative_prob": negative_prob,
        "attribute_prob": attribute_prob,

        "concept_pred": concept_pred,
        "negative_pred": negative_pred,
        "attribute_pred": attribute_pred,

        "inputs_embeds": inputs_embeds,
    }


# ============================================================
# 5. TEACHER-FORCED DIAGNOSTIC
# ============================================================

@torch.no_grad()
def teacher_forced(batch):

    enc = build_v4_encoder(batch)

    labels = batch[
        "labels"
    ].to(
        DEVICE,
        non_blocking=True
    )

    out = model.mt5(
        encoder_outputs=enc[
            "encoder_outputs"
        ],
        attention_mask=enc[
            "attention_mask"
        ],
        labels=labels,
        return_dict=True
    )

    logits = out.logits

    pred_ids = logits.argmax(
        dim=-1
    )

    valid = labels != -100

    correct = (
        (pred_ids == labels)
        &
        valid
    )

    token_acc = (
        correct.sum().float()
        /
        valid.sum().float().clamp(min=1)
    )

    return {
        **enc,

        "labels": labels,
        "logits": logits,
        "pred_ids": pred_ids,

        "loss": float(
            out.loss.item()
        ),

        "token_acc": float(
            token_acc.item()
        ),
    }


# ============================================================
# 6. AUTOREGRESSIVE DIAGNOSTIC
# ============================================================

@torch.no_grad()
def autoregressive_diagnostic(
    batch,
    max_steps=12
):

    enc = build_v4_encoder(batch)

    B = batch[
        "labels"
    ].shape[0]

    # mT5 decoder starts from pad token
    decoder_ids = torch.full(
        (B, 1),
        tokenizer.pad_token_id,
        dtype=torch.long,
        device=DEVICE
    )

    steps = []

    for step in range(max_steps):

        decoder_attention = torch.ones_like(
            decoder_ids,
            dtype=torch.long
        )

        out = model.mt5(
            encoder_outputs=enc[
                "encoder_outputs"
            ],
            attention_mask=enc[
                "attention_mask"
            ],
            decoder_input_ids=decoder_ids,
            decoder_attention_mask=decoder_attention,
            return_dict=True
        )

        logits = out.logits[:, -1, :]

        probs = F.softmax(
            logits,
            dim=-1
        )

        top_prob, top_id = torch.topk(
            probs,
            k=5,
            dim=-1
        )

        next_id = top_id[:, 0]

        steps.append({
            "step": step,
            "top_id": top_id.cpu(),
            "top_prob": top_prob.cpu(),
            "next_id": next_id.cpu(),
        })

        decoder_ids = torch.cat(
            [
                decoder_ids,
                next_id.unsqueeze(1)
            ],
            dim=1
        )

    return {
        **enc,
        "decoder_ids": decoder_ids,
        "steps": steps,
    }


# ============================================================
# 7. COLLECT EXACTLY 20 VALIDATION CASES
# ============================================================

audit_batches = []

n_cases = 0

for b in val_loader:

    audit_batches.append(b)

    n_cases += len(
        b["case_id"]
    )

    if n_cases >= 20:
        break

print(
    "\nAudit cases:",
    n_cases
)

assert n_cases >= 20


# ============================================================
# 8. RUN E2 / E3
# ============================================================

all_results = {}

for epoch_name, ckpt_path in [
    ("EPOCH_2", E2_CKPT),
    ("EPOCH_3", E3_CKPT),
]:

    print("\n")
    print("=" * 70)
    print(epoch_name)
    print("=" * 70)

    ckpt = torch.load(
        ckpt_path,
        map_location="cpu",
        weights_only=False
    )

    model.load_state_dict(
        ckpt["model_state"],
        strict=True
    )

    model.to(DEVICE)
    model.eval()

    print(
        "Checkpoint epoch:",
        ckpt["epoch"]
    )

    print(
        "Checkpoint best_val:",
        ckpt["best_val"]
    )

    rows = []

    total_tf_correct = 0
    total_tf_valid = 0
    total_tf_loss = 0.0
    n_tf_batches = 0

    for batch in audit_batches:

        tf = teacher_forced(
            batch
        )

        ar = autoregressive_diagnostic(
            batch,
            max_steps=12
        )

        labels = tf["labels"]

        valid = (
            labels != -100
        )

        total_tf_correct += (
            (
                tf["pred_ids"]
                == labels
            )
            &
            valid
        ).sum().item()

        total_tf_valid += (
            valid.sum().item()
        )

        total_tf_loss += (
            tf["loss"]
        )

        n_tf_batches += 1

        for i in range(
            len(batch["case_id"])
        ):

            ref = safe_decode(
                labels[i]
            )

            generated_ids = (
                ar["decoder_ids"][i][1:]
            )

            pred = safe_decode(
                generated_ids
            )

            first_id = (
                ar["steps"][0]
                ["next_id"][i]
                .item()
            )

            first_token = safe_decode(
                [first_id],
                skip_special_tokens=False
            )

            # first-step top 5
            top_ids = (
                ar["steps"][0]
                ["top_id"][i]
                .tolist()
            )

            top_probs = (
                ar["steps"][0]
                ["top_prob"][i]
                .tolist()
            )

            top5 = []

            for tid, prob in zip(
                top_ids,
                top_probs
            ):

                tok = safe_decode(
                    [tid],
                    skip_special_tokens=False
                )

                top5.append(
                    (
                        tok,
                        round(
                            float(prob),
                            6
                        )
                    )
                )

            concept_n = int(
                tf["concept_pred"][i]
                .sum()
                .item()
            )

            negative_n = int(
                tf["negative_pred"][i]
                .sum()
                .item()
            )

            attribute_n = int(
                tf["attribute_pred"][i]
                .sum()
                .item()
            )

            rows.append({
                "case_id":
                    batch["case_id"][i],

                "reference":
                    ref,

                "prediction":
                    pred,

                "first_token":
                    first_token,

                "first_top5":
                    top5,

                "concept_positive":
                    concept_n,

                "negative_positive":
                    negative_n,

                "attribute_positive":
                    attribute_n,

                "teacher_forced_loss":
                    tf["loss"],

                "teacher_forced_token_acc":
                    tf["token_acc"],
            })

    token_acc = (
        total_tf_correct
        /
        max(
            total_tf_valid,
            1
        )
    )

    mean_tf_loss = (
        total_tf_loss
        /
        max(
            n_tf_batches,
            1
        )
    )

    all_results[
        epoch_name
    ] = rows

    predictions = [
        r["prediction"]
        for r in rows
    ]

    first_tokens = [
        r["first_token"]
        for r in rows
    ]

    print(
        "\nTeacher-forced loss:",
        round(
            mean_tf_loss,
            6
        )
    )

    print(
        "Teacher-forced token accuracy:",
        round(
            token_acc,
            6
        )
    )

    print(
        "Unique autoregressive predictions:",
        len(
            set(predictions)
        )
    )

    print(
        "First-token distribution:"
    )

    for tok, count in (
        Counter(first_tokens)
        .most_common()
    ):

        print(
            " ",
            repr(tok),
            "=>",
            count
        )

    print(
        "\nStructured positive-count distribution:"
    )

    print(
        " concept:",
        Counter(
            r["concept_positive"]
            for r in rows
        )
    )

    print(
        " negative:",
        Counter(
            r["negative_positive"]
            for r in rows
        )
    )

    print(
        " attribute:",
        Counter(
            r["attribute_positive"]
            for r in rows
        )
    )

    print(
        "\nFirst 5 cases:"
    )

    for r in rows[:5]:

        print("-" * 60)

        print(
            "CASE:",
            r["case_id"]
        )

        print(
            "REF :",
            r["reference"]
        )

        print(
            "PRED:",
            r["prediction"]
        )

        print(
            "FIRST:",
            repr(r["first_token"])
        )

        print(
            "TOP5:",
            r["first_top5"]
        )

        print(
            "STRUCT:",
            "concept=",
            r["concept_positive"],
            "negative=",
            r["negative_positive"],
            "attribute=",
            r["attribute_positive"]
        )


# ============================================================
# 9. E2 vs E3 SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("EPOCH 2 vs EPOCH 3 SUMMARY")
print("=" * 70)

summary_rows = []

for epoch_name in [
    "EPOCH_2",
    "EPOCH_3"
]:

    rows = all_results[
        epoch_name
    ]

    preds = [
        r["prediction"]
        for r in rows
    ]

    summary_rows.append({
        "epoch":
            epoch_name,

        "unique_predictions":
            len(set(preds)),

        "exact_matches":
            sum(
                r["reference"]
                == r["prediction"]
                for r in rows
            ),

        "exact_rate":
            np.mean([
                r["reference"]
                == r["prediction"]
                for r in rows
            ]),

        "dominant_prediction":
            Counter(preds)
            .most_common(1)[0][0],

        "dominant_count":
            Counter(preds)
            .most_common(1)[0][1],
    })

summary_df = pd.DataFrame(
    summary_rows
)

print(
    summary_df.to_string(
        index=False
    )
)


# ============================================================
# 10. CHECK MODEL WAS NOT MODIFIED
# ============================================================

# Restore E2 first, then compare against original snapshot
model.load_state_dict(
    original_state,
    strict=True
)

model.to(DEVICE)
model.eval()

changed = []

current_state = model.state_dict()

for k, original in original_state.items():

    current = (
        current_state[k]
        .detach()
        .cpu()
    )

    if not torch.equal(
        original,
        current
    ):
        changed.append(k)

print("\n" + "=" * 70)
print("SAFETY CHECK")
print("=" * 70)

if changed:

    print(
        "WARNING — model state changed:",
        len(changed)
    )

    print(
        changed[:20]
    )

else:

    print(
        "Model parameters unchanged: PASS"
    )

print(
    "No optimizer step: PASS"
)

print(
    "No scheduler step: PASS"
)

print(
    "No checkpoint save: PASS"
)

print("=" * 70)
print("DIAGNOSTIC COMPLETE")
print("=" * 70)

V4 FINAL GENERATION COLLAPSE DIAGNOSTIC
Model       : PASS
val_loader  : PASS
tokenizer   : PASS
E2 checkpoint: PASS
E3 checkpoint: PASS
Model snapshot: PASS

Audit cases: 20


EPOCH_2
Checkpoint epoch: 2
Checkpoint best_val: 0.6521643281140656

Teacher-forced loss: 0.593063
Teacher-forced token accuracy: 0.809917
Unique autoregressive predictions: 4
First-token distribution:
  'VI' => 20

Structured positive-count distribution:
 concept: Counter({0: 17, 1: 2, 2: 1})
 negative: Counter({0: 20})
 attribute: Counter({0: 19, 1: 1})

First 5 cases:
------------------------------------------------------------
CASE: 10003.10003.0.10017
REF : VIÊM MŨI MẠN
PRED: VIÊM TAI GIỮA T MẠN
FIRST: 'VI'
TOP5: [('VI', 0.954785), ('HI', 0.037896), ('CH', 0.00473), ('H', 0.001031), ('NH', 0.000556)]
STRUCT: concept= 0 negative= 0 attribute= 0
------------------------------------------------------------
CASE: 10024.10024.0.10038
REF : VIÊM MŨI MẠN + VA
PRED: VIÊM MŨITT
FIRST: 'VI'
TOP5: [('VI', 0.953086), (

In [6]:
# ============================================================
# RESTORE E2 / E3 CHECKPOINT OBJECTS
# ============================================================

import torch
import os

assert "E2_CKPT" in globals(), "E2_CKPT path missing"
assert "E3_CKPT" in globals(), "E3_CKPT path missing"

print("E2 path:", E2_CKPT)
print("E3 path:", E3_CKPT)

assert os.path.exists(E2_CKPT), "E2 checkpoint file not found"
assert os.path.exists(E3_CKPT), "E3 checkpoint file not found"

E2 = torch.load(
    E2_CKPT,
    map_location="cpu",
    weights_only=False
)

E3 = torch.load(
    E3_CKPT,
    map_location="cpu",
    weights_only=False
)

print()
print("E2 loaded:")
print("  epoch    :", E2["epoch"])
print("  best_val :", E2["best_val"])

print()
print("E3 loaded:")
print("  epoch    :", E3["epoch"])
print("  best_val :", E3["best_val"])

assert E2["epoch"] == 2
assert E3["epoch"] == 3

assert "model_state" in E2
assert "model_state" in E3

print()
print("E2 checkpoint: PASS")
print("E3 checkpoint: PASS")
print("Ready for ablation audit.")

E2 path: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/checkpoints/epoch_02.pt
E3 path: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/checkpoints/epoch_03.pt

E2 loaded:
  epoch    : 2
  best_val : 0.6521643281140656

E3 loaded:
  epoch    : 3
  best_val : 0.6521643281140656

E2 checkpoint: PASS
E3 checkpoint: PASS
Ready for ablation audit.


In [7]:
# ============================================================
# V4 CONDITIONING ABLATION AUDIT
# ============================================================
# Mục tiêu:
#   FULL          = visual + predicted structured
#   ZERO_STRUCT   = visual + zero structured
#   ZERO_VISUAL   = zero visual + predicted structured
#   ZERO_BOTH     = zero visual + zero structured
#
# Chạy E2 và E3 trên cùng 20 validation cases.
# KHÔNG optimizer / scheduler / backward / checkpoint.
# ============================================================

import copy
import torch
import torch.nn.functional as F
import pandas as pd
from collections import Counter

assert "model" in globals(), "model missing"
assert "val_loader" in globals(), "val_loader missing"
assert "tokenizer" in globals(), "tokenizer missing"
assert "E2" in globals(), "E2 checkpoint missing"
assert "E3" in globals(), "E3 checkpoint missing"

device = next(model.parameters()).device
model.eval()

print("=" * 70)
print("V4 CONDITIONING ABLATION AUDIT")
print("=" * 70)
print("Device:", device)

# ------------------------------------------------------------
# 1. Snapshot model
# ------------------------------------------------------------
model_snapshot = {
    k: v.detach().cpu().clone()
    for k, v in model.state_dict().items()
}

# ------------------------------------------------------------
# 2. Safe decode
# ------------------------------------------------------------
def safe_decode(ids):
    if torch.is_tensor(ids):
        ids = ids.detach().cpu().tolist()

    ids = [
        tokenizer.pad_token_id if int(x) < 0 else int(x)
        for x in ids
    ]

    return tokenizer.decode(
        ids,
        skip_special_tokens=True
    ).strip()

# ------------------------------------------------------------
# 3. Get first 20 validation cases
# ------------------------------------------------------------
audit_cases = []

with torch.no_grad():
    for batch in val_loader:
        B = len(batch["case_id"])

        for i in range(B):
            audit_cases.append({
                "case_id": str(batch["case_id"][i]),
                "pixel_values": batch["pixel_values"][i].clone(),
                "concept_targets": batch["concept_targets"][i].clone(),
                "negative_targets": batch["negative_targets"][i].clone(),
                "attribute_targets": batch["attribute_targets"][i].clone(),
                "labels": batch["labels"][i].clone(),
            })

        if len(audit_cases) >= 20:
            break

audit_cases = audit_cases[:20]

print("Audit cases:", len(audit_cases))

# ------------------------------------------------------------
# 4. Core V4 conditioning function
# ------------------------------------------------------------
def build_conditioning(pixel_values,
                       concept_override=None,
                       negative_override=None,
                       attribute_override=None,
                       zero_visual=False):

    pixel_values = pixel_values.to(device)

    # -------------------------
    # ViT
    # -------------------------
    vision_out = model.vision(
        pixel_values=pixel_values
    )

    visual = vision_out.last_hidden_state

    # Same visual representation used by V4
    visual = visual

    if zero_visual:
        visual = torch.zeros_like(visual)

    # -------------------------
    # pooled visual
    # -------------------------
    pooled = visual.mean(dim=1)

    # -------------------------
    # Projector 768 -> 512
    # -------------------------
    visual_512 = model.projector(pooled)

    # -------------------------
    # Structured heads
    # -------------------------
    concept_logits = model.structured_heads["concept"](
        visual_512
    )

    negative_logits = model.structured_heads["negative"](
        visual_512
    )

    attribute_logits = model.structured_heads["attribute"](
        visual_512
    )

    # -------------------------
    # Original V4 hard threshold
    # -------------------------
    concept_pred = (
        torch.sigmoid(concept_logits) >= 0.5
    ).float()

    negative_pred = (
        torch.sigmoid(negative_logits) >= 0.5
    ).float()

    attribute_pred = (
        torch.sigmoid(attribute_logits) >= 0.5
    ).float()

    # -------------------------
    # Overrides
    # -------------------------
    if concept_override is not None:
        concept_pred = concept_override

    if negative_override is not None:
        negative_pred = negative_override

    if attribute_override is not None:
        attribute_pred = attribute_override

    # -------------------------
    # Structured conditioner
    # -------------------------
    structured_tokens, structured_attention = (
        model.conditioner(
            concept_pred,
            negative_pred,
            attribute_pred
        )
    )

    # -------------------------
    # Visual prefix + structured
    # -------------------------
    visual_tokens = model.visual_prefix(visual)

    inputs_embeds = torch.cat(
        [
            visual_tokens,vi
            structured_tokens
        ],
        dim=1
    )

    visual_attention = torch.ones(
        visual_tokens.shape[:2],
        dtype=torch.long,
        device=device
    )

    attention_mask = torch.cat(
        [
            visual_attention,
            structured_attention
        ],
        dim=1
    )

    return {
        "inputs_embeds": inputs_embeds,
        "attention_mask": attention_mask,
        "concept_pred": concept_pred,
        "negative_pred": negative_pred,
        "attribute_pred": attribute_pred,
    }

# ------------------------------------------------------------
# 5. Autoregressive generation
# ------------------------------------------------------------
@torch.no_grad()
def generate_from_conditioning(cond, max_new_tokens=20):

    encoder_outputs = model.mt5.encoder(
        inputs_embeds=cond["inputs_embeds"],
        attention_mask=cond["attention_mask"],
        return_dict=True,
    )

    decoder_ids = torch.full(
        (1, 1),
        tokenizer.pad_token_id,
        dtype=torch.long,
        device=device,
    )

    generated = []

    for step in range(max_new_tokens):

        out = model.mt5(
            encoder_outputs=encoder_outputs,
            attention_mask=cond["attention_mask"],
            decoder_input_ids=decoder_ids,
            return_dict=True,
        )

        next_logits = out.logits[:, -1, :]

        next_token = torch.argmax(
            next_logits,
            dim=-1,
            keepdim=True
        )

        token_id = int(next_token.item())

        if token_id == tokenizer.eos_token_id:
            break

        generated.append(token_id)

        decoder_ids = torch.cat(
            [decoder_ids, next_token],
            dim=1
        )

    return safe_decode(generated)

# ------------------------------------------------------------
# 6. Run one checkpoint
# ------------------------------------------------------------
@torch.no_grad()
def run_checkpoint(ckpt, ckpt_name):

    print()
    print("=" * 70)
    print(ckpt_name)
    print("=" * 70)

    model.load_state_dict(
        ckpt["model_state"],
        strict=True
    )
    model.eval()

    records = []

    for case in audit_cases:

        pv = case["pixel_values"].unsqueeze(0)

        # ====================================================
        # FIRST: obtain normal predicted structured state
        # ====================================================
        base = build_conditioning(pv)

        concept_pred = base["concept_pred"].clone()
        negative_pred = base["negative_pred"].clone()
        attribute_pred = base["attribute_pred"].clone()

        # ====================================================
        # FULL
        # ====================================================
        full_cond = build_conditioning(
            pv,
            concept_override=concept_pred.clone(),
            negative_override=negative_pred.clone(),
            attribute_override=attribute_pred.clone(),
            zero_visual=False,
        )

        pred_full = generate_from_conditioning(full_cond)

        # ====================================================
        # ZERO STRUCT
        # ====================================================
        zero_struct_cond = build_conditioning(
            pv,
            concept_override=torch.zeros_like(concept_pred),
            negative_override=torch.zeros_like(negative_pred),
            attribute_override=torch.zeros_like(attribute_pred),
            zero_visual=False,
        )

        pred_zero_struct = generate_from_conditioning(
            zero_struct_cond
        )

        # ====================================================
        # ZERO VISUAL
        # ====================================================
        zero_visual_cond = build_conditioning(
            pv,
            concept_override=concept_pred.clone(),
            negative_override=negative_pred.clone(),
            attribute_override=attribute_pred.clone(),
            zero_visual=True,
        )

        pred_zero_visual = generate_from_conditioning(
            zero_visual_cond
        )

        # ====================================================
        # ZERO BOTH
        # ====================================================
        zero_both_cond = build_conditioning(
            pv,
            concept_override=torch.zeros_like(concept_pred),
            negative_override=torch.zeros_like(negative_pred),
            attribute_override=torch.zeros_like(attribute_pred),
            zero_visual=True,
        )

        pred_zero_both = generate_from_conditioning(
            zero_both_cond
        )

        records.append({
            "case_id": case["case_id"],
            "ref": safe_decode(case["labels"]),

            "full": pred_full,
            "zero_struct": pred_zero_struct,
            "zero_visual": pred_zero_visual,
            "zero_both": pred_zero_both,

            "concept_count": int(concept_pred.sum().item()),
            "negative_count": int(negative_pred.sum().item()),
            "attribute_count": int(attribute_pred.sum().item()),
        })

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------
    rows = []

    for condition in [
        "full",
        "zero_struct",
        "zero_visual",
        "zero_both",
    ]:

        preds = [r[condition] for r in records]
        counts = Counter(preds)

        dominant, dominant_count = (
            counts.most_common(1)[0]
            if counts else ("", 0)
        )

        rows.append({
            "checkpoint": ckpt_name,
            "condition": condition,
            "unique_predictions": len(counts),
            "dominant_prediction": dominant,
            "dominant_count": dominant_count,
        })

    summary = pd.DataFrame(rows)

    print(summary.to_string(index=False))

    # --------------------------------------------------------
    # First 5 examples
    # --------------------------------------------------------
    print()
    print("-" * 70)
    print("FIRST 5 CASES")
    print("-" * 70)

    for r in records[:5]:

        print("CASE:", r["case_id"])
        print("REF :", r["ref"])
        print("FULL:", r["full"])
        print("Z-S  :", r["zero_struct"])
        print("Z-V  :", r["zero_visual"])
        print("Z-B  :", r["zero_both"])
        print(
            "STRUCT:",
            f"concept={r['concept_count']}",
            f"negative={r['negative_count']}",
            f"attribute={r['attribute_count']}"
        )
        print("-" * 70)

    return summary, records

# ------------------------------------------------------------
# 7. Run E2
# ------------------------------------------------------------
summary_e2, records_e2 = run_checkpoint(
    E2,
    "EPOCH_2"
)

# ------------------------------------------------------------
# 8. Run E3
# ------------------------------------------------------------
summary_e3, records_e3 = run_checkpoint(
    E3,
    "EPOCH_3"
)

# ------------------------------------------------------------
# 9. Combined summary
# ------------------------------------------------------------
summary_all = pd.concat(
    [summary_e2, summary_e3],
    ignore_index=True
)

print()
print("=" * 70)
print("FINAL ABLATION SUMMARY")
print("=" * 70)

print(
    summary_all.to_string(index=False)
)

# ------------------------------------------------------------
# 10. Pairwise equality analysis
# ------------------------------------------------------------
def pairwise_stats(records):

    result = []

    for r in records:

        result.append({
            "full==zero_struct":
                r["full"] == r["zero_struct"],

            "full==zero_visual":
                r["full"] == r["zero_visual"],

            "full==zero_both":
                r["full"] == r["zero_both"],

            "zero_visual==zero_both":
                r["zero_visual"] == r["zero_both"],
        })

    df = pd.DataFrame(result)

    return {
        col: int(df[col].sum())
        for col in df.columns
    }

print()
print("=" * 70)
print("PAIRWISE OUTPUT EQUALITY")
print("=" * 70)

print("E2:", pairwise_stats(records_e2))
print("E3:", pairwise_stats(records_e3))

# ------------------------------------------------------------
# 11. Parameter integrity
# ------------------------------------------------------------
unchanged = all(
    torch.equal(
        v.detach().cpu(),
        model_snapshot[k]
    )
    for k, v in model.state_dict().items()
)

print()
print("=" * 70)
print("SAFETY CHECK")
print("=" * 70)
print(
    "Model parameters unchanged:",
    "PASS" if unchanged else "FAIL"
)
print("No optimizer step: PASS")
print("No scheduler step: PASS")
print("No checkpoint save: PASS")

# ------------------------------------------------------------
# 12. Restore original model state
# ------------------------------------------------------------
model.load_state_dict(
    model_snapshot,
    strict=True
)

print("Original model state restored: PASS")

print("=" * 70)
print("ABLATION AUDIT COMPLETE")
print("=" * 70)

V4 CONDITIONING ABLATION AUDIT
Device: cuda:0
Audit cases: 20

EPOCH_2


ValueError: too many values to unpack (expected 4)

In [8]:
# ============================================================
# V4 CONDITIONING ABLATION AUDIT — FIXED MULTI-IMAGE VERSION
# ============================================================

import torch
import pandas as pd
from collections import Counter

device = next(model.parameters()).device
model.eval()

print("=" * 70)
print("V4 CONDITIONING ABLATION AUDIT — FIXED")
print("=" * 70)
print("Device:", device)

# ------------------------------------------------------------
# Snapshot
# ------------------------------------------------------------
model_snapshot = {
    k: v.detach().cpu().clone()
    for k, v in model.state_dict().items()
}

# ------------------------------------------------------------
# Safe decode
# ------------------------------------------------------------
def safe_decode(ids):
    if torch.is_tensor(ids):
        ids = ids.detach().cpu().tolist()

    ids = [
        tokenizer.pad_token_id if int(x) < 0 else int(x)
        for x in ids
    ]

    return tokenizer.decode(
        ids,
        skip_special_tokens=True
    ).strip()

# ------------------------------------------------------------
# Collect 20 cases
# ------------------------------------------------------------
audit_cases = []

with torch.no_grad():
    for batch in val_loader:

        B = len(batch["case_id"])

        for i in range(B):

            audit_cases.append({
                "case_id": str(batch["case_id"][i]),

                # IMPORTANT:
                # keep all images for this case
                "pixel_values":
                    batch["pixel_values"][i].clone(),

                "labels":
                    batch["labels"][i].clone(),
            })

        if len(audit_cases) >= 20:
            break

audit_cases = audit_cases[:20]

print("Audit cases:", len(audit_cases))

# ------------------------------------------------------------
# Inspect image tensor shape
# ------------------------------------------------------------
print(
    "First case pixel_values shape:",
    tuple(audit_cases[0]["pixel_values"].shape)
)

# Expected:
# [N_images, 3, 224, 224]

# ------------------------------------------------------------
# V4 encoder / conditioner
# ------------------------------------------------------------
@torch.no_grad()
def build_conditioning(
    pixel_values,
    concept_override=None,
    negative_override=None,
    attribute_override=None,
    zero_visual=False,
):

    # --------------------------------------------------------
    # pixel_values:
    # [N, 3, 224, 224]
    # --------------------------------------------------------
    pixel_values = pixel_values.to(
        device=device,
        dtype=next(model.vision.parameters()).dtype
    )

    if pixel_values.ndim == 3:
        pixel_values = pixel_values.unsqueeze(0)

    assert pixel_values.ndim == 4, (
        f"Expected [N,3,H,W], got {pixel_values.shape}"
    )

    # --------------------------------------------------------
    # ViT
    # --------------------------------------------------------
    if zero_visual:
        # Preserve exact tensor shape but remove visual signal
        visual_per_image = torch.zeros(
            pixel_values.shape[0],
            197,
            768,
            device=device,
            dtype=pixel_values.dtype,
        )

    else:
        vision_out = model.vision(
            pixel_values=pixel_values
        )

        visual_per_image = vision_out.last_hidden_state

    # --------------------------------------------------------
    # Multi-image aggregation
    #
    # [N,197,768]
    #       ↓ mean over images
    # [1,197,768]
    # --------------------------------------------------------
    visual = visual_per_image.mean(
        dim=0,
        keepdim=True
    )

    # --------------------------------------------------------
    # Visual prefix
    # --------------------------------------------------------
    visual_tokens = model.visual_prefix(visual)

    # --------------------------------------------------------
    # Pooled visual -> structured heads
    # --------------------------------------------------------
    pooled = visual.mean(dim=1)

    visual_512 = model.projector(pooled)

    concept_logits = model.structured_heads["concept"](
        visual_512
    )

    negative_logits = model.structured_heads["negative"](
        visual_512
    )

    attribute_logits = model.structured_heads["attribute"](
        visual_512
    )

    # Original V4 hard threshold
    concept_pred = (
        torch.sigmoid(concept_logits) >= 0.5
    ).float()

    negative_pred = (
        torch.sigmoid(negative_logits) >= 0.5
    ).float()

    attribute_pred = (
        torch.sigmoid(attribute_logits) >= 0.5
    ).float()

    # --------------------------------------------------------
    # Overrides
    # --------------------------------------------------------
    if concept_override is not None:
        concept_pred = concept_override.to(device)

    if negative_override is not None:
        negative_pred = negative_override.to(device)

    if attribute_override is not None:
        attribute_pred = attribute_override.to(device)

    # --------------------------------------------------------
    # Structured conditioner
    # --------------------------------------------------------
    structured_tokens, structured_attention = (
        model.conditioner(
            concept_pred,
            negative_pred,
            attribute_pred
        )
    )

v

# ------------------------------------------------------------
# Generation
# ------------------------------------------------------------
@torch.no_grad()
def generate_from_conditioning(
    cond,
    max_new_tokens=20
):

    encoder_outputs = model.mt5.encoder(
        inputs_embeds=cond["inputs_embeds"],
        attention_mask=cond["attention_mask"],
        return_dict=True,
    )

    decoder_ids = torch.full(
        (1, 1),
        tokenizer.pad_token_id,
        dtype=torch.long,
        device=device,
    )

    generated = []

    for _ in range(max_new_tokens):

        out = model.mt5(
            encoder_outputs=encoder_outputs,
            attention_mask=cond["attention_mask"],
            decoder_input_ids=decoder_ids,
            return_dict=True,
        )

        next_token = torch.argmax(
            out.logits[:, -1, :],
            dim=-1,
            keepdim=True
        )

        token_id = int(next_token.item())

        if token_id == tokenizer.eos_token_id:
            break

        generated.append(token_id)

        decoder_ids = torch.cat(
            [decoder_ids, next_token],
            dim=1
        )

    return safe_decode(generated)

# ------------------------------------------------------------
# Run checkpoint
# ------------------------------------------------------------
@torch.no_grad()
def run_checkpoint(ckpt, ckpt_name):

    print()
    print("=" * 70)
    print(ckpt_name)
    print("=" * 70)

    model.load_state_dict(
        ckpt["model_state"],
        strict=True
    )
    model.eval()

    records = []

    for case in audit_cases:

        pv = case["pixel_values"]

        # ----------------------------------------------------
        # Base structured prediction
        # ----------------------------------------------------
        base = build_conditioning(pv)

        cp = base["concept_pred"].clone()
        np = base["negative_pred"].clone()
        ap = base["attribute_pred"].clone()

        # ----------------------------------------------------
        # FULL
        # ----------------------------------------------------
        cond = build_conditioning(
            pv,
            concept_override=cp,
            negative_override=np,
            attribute_override=ap,
            zero_visual=False,
        )

        pred_full = generate_from_conditioning(cond)

        # ----------------------------------------------------
        # ZERO STRUCTURED
        # ----------------------------------------------------
        cond = build_conditioning(
            pv,
            concept_override=torch.zeros_like(cp),
            negative_override=torch.zeros_like(np),
            attribute_override=torch.zeros_like(ap),
            zero_visual=False,
        )

        pred_zero_struct = generate_from_conditioning(cond)

        # ----------------------------------------------------
        # ZERO VISUAL
        # ----------------------------------------------------
        cond = build_conditioning(
            pv,
            concept_override=cp,
            negative_override=np,
            attribute_override=ap,
            zero_visual=True,
        )

        pred_zero_visual = generate_from_conditioning(cond)

        # ----------------------------------------------------
        # ZERO BOTH
        # ----------------------------------------------------
        cond = build_conditioning(
            pv,
            concept_override=torch.zeros_like(cp),
            negative_override=torch.zeros_like(np),
            attribute_override=torch.zeros_like(ap),
            zero_visual=True,
        )

        pred_zero_both = generate_from_conditioning(cond)

        records.append({
            "case_id": case["case_id"],
            "ref": safe_decode(case["labels"]),

            "full": pred_full,
            "zero_struct": pred_zero_struct,
            "zero_visual": pred_zero_visual,
            "zero_both": pred_zero_both,

            "concept_count": int(cp.sum().item()),
            "negative_count": int(np.sum().item()),
            "attribute_count": int(ap.sum().item()),
        })

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------
    rows = []

    for condition in [
        "full",
        "zero_struct",
        "zero_visual",
        "zero_both",
    ]:

        preds = [r[condition] for r in records]
        counts = Counter(preds)

        dominant, dominant_count = counts.most_common(1)[0]

        rows.append({
            "checkpoint": ckpt_name,
            "condition": condition,
            "unique_predictions": len(counts),
            "dominant_prediction": dominant,
            "dominant_count": dominant_count,
        })

    summary = pd.DataFrame(rows)

    print()
    print(summary.to_string(index=False))

    # --------------------------------------------------------
    # First 5 cases
    # --------------------------------------------------------
    print()
    print("-" * 70)
    print("FIRST 5 CASES")
    print("-" * 70)

    for r in records[:5]:

        print("CASE:", r["case_id"])
        print("REF :", r["ref"])
        print("FULL:", r["full"])
        print("Z-S :", r["zero_struct"])
        print("Z-V :", r["zero_visual"])
        print("Z-B :", r["zero_both"])

        print(
            "STRUCT:",
            f"concept={r['concept_count']}",
            f"negative={r['negative_count']}",
            f"attribute={r['attribute_count']}"
        )

        print("-" * 70)

    return summary, records

# ------------------------------------------------------------
# E2
# ------------------------------------------------------------
summary_e2, records_e2 = run_checkpoint(
    E2,
    "EPOCH_2"
)

# ------------------------------------------------------------
# E3
# ------------------------------------------------------------
summary_e3, records_e3 = run_checkpoint(
    E3,
    "EPOCH_3"
)

# ------------------------------------------------------------
# Combined
# ------------------------------------------------------------
summary_all = pd.concat(
    [summary_e2, summary_e3],
    ignore_index=True
)

print()
print("=" * 70)
print("FINAL ABLATION SUMMARY")
print("=" * 70)

print(summary_all.to_string(index=False))

# ------------------------------------------------------------
# Pairwise equality
# ------------------------------------------------------------
def pairwise_stats(records):

    comparisons = {
        "full==zero_struct":
            [r["full"] == r["zero_struct"] for r in records],

        "full==zero_visual":
            [r["full"] == r["zero_visual"] for r in records],

        "full==zero_both":
            [r["full"] == r["zero_both"] for r in records],

        "zero_visual==zero_both":
            [r["zero_visual"] == r["zero_both"] for r in records],
    }

    return {
        k: sum(v)
        for k, v in comparisons.items()
    }

print()
print("=" * 70)
print("PAIRWISE OUTPUT EQUALITY")
print("=" * 70)

print("E2:", pairwise_stats(records_e2))
print("E3:", pairwise_stats(records_e3))

# ------------------------------------------------------------
# Integrity
# ------------------------------------------------------------
unchanged = all(
    torch.equal(
        v.detach().cpu(),
        model_snapshot[k]
    )
    for k, v in model.state_dict().items()
)

print()
print("=" * 70)
print("SAFETY CHECK")
print("=" * 70)
print(
    "Model parameters unchanged:",
    "PASS" if unchanged else "FAIL"
)
print("No optimizer step: PASS")
print("No scheduler step: PASS")
print("No checkpoint save: PASS")

model.load_state_dict(
    model_snapshot,
    strict=True
)

print("Original model state restored: PASS")
print("=" * 70)
print("ABLATION AUDIT COMPLETE")
print("=" * 70)

V4 CONDITIONING ABLATION AUDIT — FIXED
Device: cuda:0
Audit cases: 20
First case pixel_values shape: (8, 3, 224, 224)

EPOCH_2


AttributeError: 'V4Model' object has no attribute 'visual_prefix'

In [9]:
# ============================================================
# V4 MODEL STRUCTURE / FORWARD AUDIT
# READ-ONLY — NO TRAINING / NO CHECKPOINT
# ============================================================

import inspect
import torch

print("=" * 70)
print("V4 MODEL STRUCTURE AUDIT")
print("=" * 70)

print("\nMODEL CLASS:")
print(type(model))

print("\nMODEL MODULES:")
for name, module in model.named_children():
    print(f"  {name:25s} -> {type(module).__name__}")

print("\nMODEL PARAMETERS BY COMPONENT:")
components = {}

for name, param in model.named_parameters():
    root = name.split(".")[0]
    components[root] = components.get(root, 0) + param.numel()

for name, n in sorted(components.items()):
    print(f"  {name:25s} -> {n:,}")

print("\nV4Model.forward SOURCE:")
try:
    print(inspect.getsource(model.forward))
except Exception as e:
    print("Could not retrieve source:", repr(e))

print("\nKNOWN ATTRIBUTES:")
for attr in [
    "vision",
    "projector",
    "conditioner",
    "structured_heads",
    "mt5",
    "visual_prefix",
]:
    print(
        f"  {attr:20s}:",
        "YES" if hasattr(model, attr) else "NO"
    )

print("\nCHECKPOINT MODEL KEYS — FIRST 40:")
# inspect E2 checkpoint naming without loading anything
keys = list(E2["model_state"].keys())

for k in keys[:40]:
    print(" ", k)

print("\nCHECKPOINT COMPONENT ROOTS:")
roots = sorted(set(k.split(".")[0] for k in keys))

for root in roots:
    count = sum(k.startswith(root + ".") for k in keys)
    print(f"  {root:25s} -> {count} keys")

print("\n" + "=" * 70)
print("READ-ONLY AUDIT COMPLETE")
print("=" * 70)

V4 MODEL STRUCTURE AUDIT

MODEL CLASS:
<class '__main__.V4Model'>

MODEL MODULES:
  vision                    -> ViTModel
  mt5                       -> MT5ForConditionalGeneration
  projector                 -> Projector
  structured_heads          -> ModuleDict
  conditioner               -> StructuredConditioner

MODEL PARAMETERS BY COMPONENT:
  conditioner               -> 35,840
  mt5                       -> 300,176,768
  projector                 -> 393,728
  structured_heads          -> 30,267
  vision                    -> 86,389,248

V4Model.forward SOURCE:
def _forward_unimplemented(self, *input: Any) -> None:
    r"""Define the computation performed at every call.

    Should be overridden by all subclasses.

    .. note::
        Although the recipe for forward pass needs to be defined within
        this function, one should call the :class:`Module` instance afterwards
        instead of this since the former takes care of running the
        registered hooks while the la